# Lượt 1 — OCR ảnh gốc bằng bốn model

Gắn packages/models/PDF, điền ba đường dẫn rồi **Run All**. Không chạy restoration.
Chỉ xuất **original_ocr_handoff.zip** gồm ảnh/crop gốc, cấu hình và raw checkpoint của bốn model.
Không tính consensus/coverage, không tạo HTML, TXT hay ZIP báo cáo riêng ở notebook này.
Đặt gói handoff cùng PDF trong `DATA_DIR` của **1_ocr_va_so_sanh.ipynb** để đếm/tối ưu trước phục dựng.


## Cấu hình lượt OCR gốc

Mặc định mọi sách/trang và ba crop. Dùng BOOKS/PAGES/LIMIT để chạy phạm vi nhỏ khi cần; giữ WORK_DIR để resume.


In [ ]:
# OCR GỐC BẰNG BỐN MODEL — điền ba đường dẫn Input rồi Run All.
from pathlib import Path
if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle, Internet Off.')

PACKAGES_DIR = ''  # Dataset packages offline
MODELS_DIR = ''  # Dataset models OCR; không cần nạp restoration
DATA_DIR = ''  # Thư mục PDF nguồn, chưa cần dữ liệu OCR trước đó
BOOKS = []  # [] = tất cả; ['1','2'] = chọn sách; nhận wildcard
MAX_BOOKS = None  # None = mọi sách; số nguyên >0 để giới hạn
PAGES = ''  # '' = mọi trang; '1-5,10' = số trang từ 1 của mỗi PDF
LIMIT = None  # None = mọi trang; số nguyên >0 giới hạn tổng trang cả lượt
RUN_OCR = True  # True = OCR mới/resume; False = đọc checkpoint, không OCR
WORK_DIR = Path('/kaggle/working/original_ocr')  # Giữ đường dẫn để resume; có original_ocr_handoff.zip và output/
CROP_TYPES = ['page', 'group', 'column']  # page/group/column hoặc kết hợp; nhiều loại tăng thời gian OCR

# Gói handoff luôn giữ ảnh/crop + raw OCR đủ 4 model để notebook tiếp tục dùng lại.
# Chỉ chuyển original_ocr_handoff.zip sang notebook 1; thư mục ảnh/raw giữ để resume.


In [ ]:
VARIANT_MODE = 'baseline'
RESTORATION_THRESHOLDS = None
UNIHIR_MAX_REGIONS = 0
WORK_DIR = Path(WORK_DIR)  # nhận cả chuỗi hoặc Path từ ô chính
# CẤU HÌNH NÂNG CAO — thường giữ nguyên. Chỉ sửa các tham số được chú thích.
# Ô này chạy sau ô chính; đường dẫn/profile/CONFIG tự ghép ở đây. Sửa xong chọn Run All.
# MODEL_OPTIONS của UniHIR: steps=10/ref_count=1 là lượt nhỏ; tăng sẽ tốn thời gian hơn.
# placement=sharded chia weights qua nhiều GPU; max_memory là ngân sách weights, không chặn toàn bộ VRAM.

# Ảnh/cache: đường dẫn writable. Giữ thư mục này để resume và để gom PNG vào ZIP; không cần đổi theo từng lượt OCR.
IMAGE_OUTPUT_DIR = '/kaggle/working/original_image_dataset'

# Tự suy ra từ IMAGE_OUTPUT_DIR; không cần chỉnh trực tiếp.
DATASET_DIR = str(Path(IMAGE_OUTPUT_DIR) / '_dataset')

# Runtime tạm tại /tmp, không phải kết quả để tải về; thường giữ nguyên.
RUNTIME_DIR = '/tmp/vtm_image_runtime'

# True: dùng cache FlashAttention đã chuẩn bị. False: có thể build lại rất lâu; không làm inference tự nhanh hơn.
REQUIRE_FLASH_CACHE = True

# GPU logic nhìn thấy trong cùng phiên: mặc định [0,1,2,3]. Chỉ chỉnh khi phần cứng khác; đồng bộ MODEL_OPTIONS/GPU_MAP.
GPU_IDS = [0, 1, 2, 3]

# True: GPU xong sớm có thể giúp model khác; False: giữ phân công cố định. Không thay phạm vi trang/crop.
ADAPTIVE_GPU_SCHEDULING = True

# 'parallel_models': bốn OCR đồng thời (cần GPU_MAP hợp lệ); 'sequential': lần lượt, ít GPU hơn và thường lâu hơn. Model lớn vẫn cần sharding.
EXECUTION_MODE = 'parallel_models'

# Tự theo RUN_OCR; False ở chế độ checkpoint để không tạo ảnh/nạp restoration.
RUN_PREPARATION = RUN_OCR

# True: chạy fixture CPU nhúng trước xử lý; False: bỏ kiểm tra này. Không phải kiểm accuracy hay inference GPU.
RUN_SELF_TESTS = VARIANT_MODE == 'full'

# True: kiểm thêm dataset sau xử lý, tốn thời gian I/O; False: giữ các kiểm tra bắt buộc của pipeline.
VALIDATE_OUTPUT = False

# True: xem vài preview ảnh ngay trong notebook; False: xem ảnh trong ZIP cuối lượt.
EXPORT_PREVIEWS = False

# True: thêm một bản ảnh theo phương pháp ngoài báo cáo; False: tránh bản sao riêng, ZIP chính vẫn có ảnh.
EXPORT_IMAGE_FOLDERS = False

# True: xuất thêm ảnh ghép so sánh; False: bỏ việc ghép ảnh. Không thay ảnh đầu vào OCR.
EXPORT_COMPARISONS = False

# Lưới ảnh ghép: số cột nguyên > 0; panel width/height là pixel preview. Chỉ ảnh hưởng bản ghép, không thay chất lượng ảnh OCR.
COMPARISON_COLUMNS = 4
COMPARISON_PANEL_WIDTH = 384
COMPARISON_PANEL_HEIGHT = 512

# True: thêm ZIP ảnh riêng; False: không thêm ZIP phụ. Không cần bật để có ZIP báo cáo; ZIP ảnh không thay checkpoint _dataset.
EXPORT_DATASET_ZIP = False

# Tên ZIP ảnh phụ, chỉ dùng khi EXPORT_DATASET_ZIP=True.
DATASET_ZIP = IMAGE_OUTPUT_DIR + '.zip'

# Chỉ dùng khi VARIANT_MODE='selected': danh sách có 'original', thêm 'tadisr_x4', 'difftsr' hoặc 'unihir_restoration'. Thêm phương pháp tăng việc.
VARIANTS = ['original']

# []: mọi vùng thuộc crop đã chọn; hoặc list ID lấy từ metadata layout để giới hạn vùng. Áp dụng cho mọi sách/trang được chọn.
REGION_IDS = []

# True: dùng lại receipt ảnh phục dựng hợp lệ; False: chạy lại phần ảnh đó. OCR có checkpoint riêng, không bị xoá bởi cờ này.
RESUME = True

# False: lỗi phục dựng chặn công bố snapshot mới; True: cho xuất dataset thiếu có đánh dấu. Vùng không áp dụng không phải lỗi.
ALLOW_PARTIAL = False

# Giới hạn pixel: số nguyên > 0 cho ảnh nguồn/đầu ra. Vượt giới hạn sẽ báo lỗi/partial; không tự hạ độ phân giải để lách.
MAX_SOURCE_PIXELS = 100000000
MAX_OUTPUT_PIXELS = 100000000

# None: dùng dung lượng trống; hoặc số byte > 0 để chặn dung lượng. STORAGE_RESERVE_BYTES chừa chỗ cho runtime/checkpoint.
MAX_STORAGE_BYTES = None
STORAGE_RESERVE_BYTES = 2 * 1024 ** 3

# Timeout theo giây, > 0; giảm quá thấp dễ ngắt vùng chậm. MODEL_LOAD_TIMEOUT_SECONDS là thời gian chờ nạp model, không phải inference.
REGION_TIMEOUT_SECONDS = 600
MODEL_LOAD_TIMEOUT_SECONDS = 900

# Giá trị cố định đã hỗ trợ: BATCH_SIZE=1, DESKEW=False, DEWARP=False. Giá trị khác bị từ chối; không dùng để tăng tốc.
BATCH_SIZE = 1
DESKEW = False
DEWARP = False

# Layout mặc định: dpi > 0; reading_direction='vertical_rtl' hoặc 'horizontal_ltr'; columns_per_group > 0. Đổi layout tạo cache mới; tăng DPI không tạo chi tiết nguồn thật.
LAYOUT = {'dpi': 200,
 'analysis_max_side': 1200,
 'reading_direction': 'vertical_rtl',
 'columns_per_group': 3,
 'column_cuts': None,
 'page_column_cuts': {}}

# Ghi đè DPI/hướng đọc theo stem sách. Khi chỉnh hướng đọc, đồng bộ BOOK_PROFILES bên dưới để ảnh và OCR cùng layout.
BOOK_LAYOUTS = {'1': {'dpi': 225},
 '2': {'dpi': 226},
 '3': {'dpi': 226},
 '4': {'dpi': 181},
 '5': {'dpi': 181},
 '6': {'dpi': 226},
 'bien': {'dpi': 200, 'reading_direction': 'horizontal_ltr'},
 'kinh': {'dpi': 45}}

# {}: giữ mặc định. Đây là tham số phương pháp, không phải danh sách bật phương pháp.
VARIANT_OPTIONS = {}

# Thông số inference của model, không tự bật model. TADiSR tile/overlap là bội 32, overlap < tile; DiffTSR steps 1..200. Tăng steps thường tốn thêm thời gian.
MODEL_OPTIONS = {'tadisr_x4': {'tile': 256, 'overlap': 64, 'device': 'cuda', 'seed': 42, 'placement': 'sharded', 'gpu_ids': GPU_IDS.copy(), 'max_memory': '18GiB', 'activation_reserve_gib': 4, 'entry_reserve_gib': 6, 'prompt_device': 'cpu'}, 'difftsr': {'steps': 20, 'device': 'cuda', 'seed': 42}, 'unihir_restoration': {'steps': 10, 'ref_count': 1, 'device': 'cuda', 'seed': 42, 'placement': 'sharded', 'gpu_ids': GPU_IDS.copy(), 'max_memory': '18GiB', 'activation_reserve_gib': 4, 'entry_reserve_gib': 2, 'vae_gpu_id': 'auto', 'inference_timeout_seconds': 1200, 'inference_hard_timeout_seconds': 3600, 'progress_stall_seconds': 300, 'verify_vae': VARIANT_MODE == 'full'}}

# {}: runtime offline tự quản lý; chỉ điền interpreter đã chuẩn bị sẵn khi chẩn đoán.
RUNTIME_PYTHONS = {}

# 'warn': ghi cảnh báo khi source Torch khác RECORD; 'strict': dừng. Không thay phiên bản Torch/weights.
KERNEL_TORCH_SOURCE_POLICY = 'warn'

# Điều kiện thêm theo phương pháp: books/crop_types là list, require_review=True/False. Không tự nhận dạng degradation.
APPLICABILITY = {}

# Metadata duyệt vùng theo key <sách>:<trang từ 1>:<region_id>. Lấy ID từ metadata ảnh. DiffTSR cần upright_horizontal_single_line=True, character_count=1..24, allow_simplified_text_prior=True và crop gần 4:1.
REVIEWED_REGIONS = {}

# True: tạo/cache prompt TADiSR từ asset offline nếu cần; False: cần prompt đã có. Chỉ chạy khi chọn TADiSR.
GENERATE_TADISR_PROMPT_OFFLINE = True

# TỰ GHÉP — không sửa dict CONFIG; sửa biến phía trên rồi Run All để đồng bộ ảnh và OCR.
CONFIG = dict(packages_dir=PACKAGES_DIR, models_dir=MODELS_DIR, data_dir=DATA_DIR, output_dir=DATASET_DIR, gpu_ids=GPU_IDS, execution_mode=EXECUTION_MODE, books=BOOKS, pages=PAGES, limit=LIMIT, max_books=MAX_BOOKS, variants=VARIANTS, crop_types=CROP_TYPES, region_ids=REGION_IDS, resume=RESUME, allow_partial=ALLOW_PARTIAL, max_source_pixels=MAX_SOURCE_PIXELS, max_output_pixels=MAX_OUTPUT_PIXELS, max_storage_bytes=MAX_STORAGE_BYTES, timeout_seconds=REGION_TIMEOUT_SECONDS, load_timeout_seconds=MODEL_LOAD_TIMEOUT_SECONDS, batch_size=BATCH_SIZE, deskew=DESKEW, dewarp=DEWARP, layout=LAYOUT, book_layouts=BOOK_LAYOUTS, variant_options=VARIANT_OPTIONS, model_options=MODEL_OPTIONS, runtime_pythons=RUNTIME_PYTHONS, reviewed_regions=REVIEWED_REGIONS, kernel_torch_source_policy=KERNEL_TORCH_SOURCE_POLICY, generate_tadisr_prompt_offline=GENERATE_TADISR_PROMPT_OFFLINE, applicability=APPLICABILITY)
CONFIG['require_flash_cache'] = REQUIRE_FLASH_CACHE
CONFIG['adaptive_gpu_scheduling'] = ADAPTIVE_GPU_SCHEDULING

# 0..9, PNG luôn lossless: 0 lớn/ghi nhanh, 1 mặc định nhanh, 6..9 thường nhỏ hơn nhưng nén lâu hơn.
PNG_COMPRESSION_LEVEL = 1
CONFIG['png_compression_level'] = PNG_COMPRESSION_LEVEL
CONFIG['storage_reserve_bytes'] = STORAGE_RESERVE_BYTES

# Số worker CPU nguyên > 0; không phải số GPU và không thay bốn engine OCR.
CPU_VARIANT_WORKERS = 2
CONFIG['cpu_variant_workers'] = CPU_VARIANT_WORKERS

# Mask GPU của UniHIR trong scheduler; đồng bộ GPU_IDS và MODEL_OPTIONS nếu đổi phần cứng.
UNIHIR_GPU_IDS = [0, 1, 2, 3]

# Tự trỏ tới dataset.json trong DATASET_DIR. Khi RUN_OCR=False và ảnh ở Input mới, sửa riêng đường dẫn này để xuất lại PNG đúng nguồn.
PREPARED_IMAGES = str(Path(DATASET_DIR) / 'dataset.json')

# ĐƯỜNG DẪN TỰ GHÉP — thường không chỉnh. Notebook nhận cả thư mục models có folder bọc.
_model_folders = {'hunyuan': 'hunyuan',
 'ppocr_det': 'ppocr_det',
 'ppocr_rec': 'ppocr_rec',
 'paddle_vl': 'paddle_vl',
 'paddle_layout': 'paddle_layout',
 'tonggu_ocr': 'tonggu/ocr',
 'tonggu_layout': 'tonggu/layout/finetune_infer.pth'}
_models_root = Path(MODELS_DIR) if MODELS_DIR else None
if _models_root and (not (_models_root / 'hunyuan').is_dir()) and (_models_root / 'models/hunyuan').is_dir():
    _models_root = _models_root / 'models'
WEIGHTS = {name: _models_root / folder for name, folder in _model_folders.items()} if _models_root else {}

# Các biến REVIEW_* chỉ đổi preview, không giới hạn OCR: book=None chọn tự động; page=None hoặc số trang từ 1; crop phải thuộc CROP_TYPES.
REVIEW_BOOK = None
REVIEW_PAGE = None
REVIEW_CROP_TYPE = 'group'

# 'regions': cụm bất đồng; 'characters': từng chữ. MAX_REVIEW_ROWS > 0 giới hạn số mục preview, không phải số trang OCR.
REVIEW_VIEW = 'regions'
MAX_REVIEW_ROWS = 20

# Tuỳ chọn OCR ít dùng, ví dụ {'gpu':0} khi EXECUTION_MODE='sequential'; không cần điền cho cấu hình bốn L4 mặc định.
ADVANCED = {}
ADVANCED['export_consensus'] = VARIANT_MODE == 'full'

# PROFILE CÓ SẴN — giữ cấu hình theo sách. Tắt bằng ADVANCED["use_book_profiles"]=False; đổi token/pixel budget có thể ảnh hưởng thời gian và output.
BOOK_PROFILES = {'1.json': {'books': ['1.pdf'],
            'dpi': 225,
            'max_side': 750,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '2.json': {'books': ['2.pdf'],
            'dpi': 226,
            'max_side': 754,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '3.json': {'books': ['3.pdf'],
            'dpi': 226,
            'max_side': 754,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '4.json': {'books': ['4.pdf'],
            'dpi': 181,
            'max_side': 704,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '5.json': {'books': ['5.pdf'],
            'dpi': 181,
            'max_side': 704,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 '6.json': {'books': ['6.pdf'],
            'dpi': 226,
            'max_side': 653,
            'max_new_tokens': 6000,
            'prompt': '',
            'reading_direction': 'vertical_rtl'},
 'bien.json': {'books': ['bien.pdf'],
               'dpi': 200,
               'max_side': 2333,
               'max_new_tokens': 6000,
               'prompt': '',
               'reading_direction': 'horizontal_ltr'},
 'kinh.json': {'books': ['kinh.pdf'],
               'dpi': 45,
               'max_side': 2206,
               'max_new_tokens': 6000,
               'prompt': '',
               'reading_direction': 'vertical_rtl'}}

# Tự ghép WORK_DIR/output; đây là thư mục HTML/JSONL/ZIP để tải về.
OUTPUT_DIR = WORK_DIR / 'output'

# True: xuất báo cáo cuối lượt; False: bỏ bước xuất, giữ checkpoint. Không thay việc OCR.
EXPORT_RESULTS = False

# True: thêm ảnh vùng bất đồng vào báo cáo, tốn render/dung lượng; False: bỏ ảnh bất đồng. Khác với EXPORT_IMAGES_AND_OCR.
EXPORT_CROPS = False

# True: tạo ZIP mỗi sách/crop như notebook 3; False: chỉ thư mục HTML/JSONL. Giữ True để tải về thuận tiện.
CREATE_ZIP = False

# True: hiển thị bất đồng thêm trong notebook; False: xem trong HTML đã xuất, giảm việc hiển thị.
PREVIEW_CONFLICTS = False

# Attention: sdpa (mặc định), auto, eager, flash_attention_2 (cần runtime hỗ trợ). SDPA không bảo đảm kernel Flash. region_upscale=1.0 giữ tỷ lệ; >1 chỉ nội suy, không tạo chi tiết thật. resolution_strict=True dừng nếu phải thu nhỏ.
IMAGE_PREPARATION = {'attention_backend': 'sdpa',
 'segmentation': 'auto',
 'segmentation_args': {'columns_per_region': 3, 'max_regions': 64},
 'region_upscale': 1.0,
 'region_max_pixels': 0,
 'region_max_side': 0,
 'resolution_strict': False}

# Ảnh tham chiếu original; danh sách chạy nhiều phương pháp nằm ở OCR_VARIANTS, không bật restoration bằng biến này.
PREPARED_VARIANT = 'original'

# Tự theo CROP_TYPES ở ô chính; không chỉnh thành lựa chọn thứ hai.
PREPARED_CROP_TYPES = CROP_TYPES

# Giữ False khi OCR nhiều phương pháp: thiếu ảnh phải được ghi rõ, không tự render trang khác thay vào.
PREPARED_MISSING_FALLBACK = False

# Giữ True để bốn OCR chạy trên mọi phương pháp đã tạo. False dành cho đối chứng một variant, không phải nút bật/tắt UniHIR.
ENABLE_MULTI_VARIANT_OCR = True

# 'all': mọi phương pháp trong snapshot, trừ mask; hoặc list có 'original' để giới hạn. Không tự tạo ảnh phương pháp chưa có.
OCR_VARIANTS = 'all'

# True: cho OCR/đối chiếu ảnh UniHIR/DiffTSR có provenance; False: không chấp nhận các ảnh tái tạo này. Không tự bật inference.
ALLOW_GENERATED_VARIANTS = True

# True: phân tích thêm mực ảnh nguồn để rà soát; False: bỏ phần CPU này. Không thay bốn dự đoán OCR.
ENABLE_SPATIAL_COVERAGE = False

# analysis_max_side: cạnh phân tích; tăng làm chậm/tốn RAM. save_overlays/save_candidate_masks True thêm ảnh bằng chứng, False giảm dung lượng.
SPATIAL_COVERAGE_OPTIONS = {'analysis_max_side': 1600, 'save_overlays': True, 'save_candidate_masks': VARIANT_MODE == 'full'}

# Đối chiếu biến thể với original; các cờ compare_by_engine/compare_across_engines/save_difference_crops bật/tắt phần phân tích và ảnh, không chứng minh accuracy.
VARIANT_STABILITY_OPTIONS = {'reference_variant': 'original',
 'compare_by_engine': True,
 'compare_across_engines': True,
 'save_difference_crops': True}

# Hai cờ nghiên cứu H10: False mặc định; True phân tích thêm ảnh/checkpoint đã có, tăng CPU/I/O, không chạy lại restoration.
ENABLE_RESTORATION_ABLATION = False
ENABLE_VISUAL_FIDELITY = False

# Giới hạn số vùng/mẫu và cạnh ảnh phân tích H10. blind_review=True ẩn tên trong phần rà soát H10; raw vẫn giữ tên.
RESTORATION_EVALUATION_OPTIONS = {'max_regions': 256, 'max_review_cases': 48, 'analysis_max_side': 1024, 'blind_review': False}

# '' bỏ qua; hoặc đường dẫn JSON nhãn do người đọc xuất từ HTML. Không suy nhãn tự động.
MANUAL_LABELS_FILE = ''

# Phân công GPU ban đầu cho bốn engine; worker thấy card của mình là cuda:0. Với parallel_models, không gán trùng một card cho hai engine.
GPU_MAP = {'ppocr': 0, 'tonggu': 1, 'paddle': 2, 'hunyuan': 3}
ADVANCED.update(execution_mode=EXECUTION_MODE, gpu_map=GPU_MAP)

# True: dùng cache native OCR đã chuẩn bị. False: có thể cài/build lại lâu; không cần đổi cho lần chỉnh cấu hình này.
REQUIRE_OCR_NATIVE_CACHE = True
import os as _ocr_cache_os
_ocr_cache_os.environ['VTM_OCR_REQUIRE_CACHE'] = '1' if REQUIRE_OCR_NATIVE_CACHE else '0'
ADVANCED['adaptive_gpu_scheduling'] = ADAPTIVE_GPU_SCHEDULING

# Chọn bằng OCR gốc, đủ 6 cặp trên ngưỡng 0..1; tăng ngưỡng giảm việc. max_regions/max_total_source_pixels là trần riêng (None = không chặn); UniHIR thêm tối đa 2 vùng/PDF. Không đổi strategy nếu dùng Run All hai giai đoạn.
SELECTIVE_RESTORATION = dict(enabled=True, strategy='ocr_disagreement', thresholds=RESTORATION_THRESHOLDS,
    min_characters=8, max_regions=UNIHIR_MAX_REGIONS, max_regions_per_book=2,
    max_total_source_pixels=12_000_000,
    method_budgets={'tadisr_x4': {'max_regions':256, 'max_total_source_pixels':32_000_000},
                    'difftsr': {'max_regions':512, 'max_total_source_pixels':16_000_000}})
CONFIG['selective_restoration'] = SELECTIVE_RESTORATION
EXPORT_IMAGES_AND_OCR = False


In [ ]:
from __future__ import annotations
import math
PDF_IMAGE_VERSION = 'pdfium-visible-v1'

def pdf_pixel_to_page(size, bbox, rotation):
    """Six affine coefficients: x'=a*x+c*y+e, y'=b*x+d*y+f.

    Input pixels have a top-left origin; output is unrotated PDF canvas units.
    PDFium does not expose /UserUnit: do not label these physical DPI values.
    """
    w, h = size
    l, b, r, t = bbox
    if rotation == 90:
        return [0, (t - b) / w, (r - l) / h, 0, l, b]
    if rotation == 180:
        return [-(r - l) / w, 0, 0, (t - b) / h, r, b]
    if rotation == 270:
        return [0, -(t - b) / w, -(r - l) / h, 0, r, t]
    return [(r - l) / w, 0, 0, -(t - b) / h, l, t]

def pdf_visible_render(page, scale, max_pixels):
    w, h = page.get_size()
    size = [math.ceil(w * scale), math.ceil(h * scale)]
    if min(size) < 1 or size[0] * size[1] > max_pixels:
        raise ValueError(f'Page render {size} exceeds max_source_pixels={max_pixels}; review acquisition settings (source is never silently reduced)')
    bitmap = page.render(scale=scale, draw_annots=True)
    try:
        return bitmap.to_pil().convert('RGB')
    finally:
        bitmap.close()

def acquire_pdf_image(page, *, dpi=200, max_source_pixels=100000000, strategy='auto'):
    """Return owned RGB pixels and evidence for this page's acquisition decision.

    `dpi` is the rendering fallback in pixels per 72 PDF canvas units. For a
    raster-only full-page scan use its native density, including when raw decode
    fails validation. Mixed/vector pages use at least the requested resolution.
    """
    import pypdfium2 as pdfium
    from PIL import ImageChops
    if strategy not in {'auto', 'render'} or dpi <= 0 or max_source_pixels <= 0:
        raise ValueError('Invalid PDF acquisition options')
    width, height = page.get_size()
    bbox, rotation = (list(page.get_bbox()), page.get_rotation())
    objects = list(page.get_objects())
    annotations = pdfium.raw.FPDFPage_GetAnnotCount(page)
    images, density = ([], 0.0)
    for obj in objects:
        if obj.type != pdfium.raw.FPDF_PAGEOBJ_IMAGE:
            continue
        pixels = obj.get_px_size()
        matrix = obj.get_matrix()
        container = obj.container
        while container is not None:
            matrix = matrix.multiply(container.get_matrix())
            container = container.container
        axes = [math.hypot(matrix.a, matrix.b), math.hypot(matrix.c, matrix.d)]
        corners = [matrix.on_point(x, y) for x, y in ((0, 0), (1, 0), (1, 1), (0, 1))]
        bounds = [min((c[0] for c in corners)), min((c[1] for c in corners)), max((c[0] for c in corners)), max((c[1] for c in corners))]
        full = max(0, min(bbox[2], bounds[2]) - max(bbox[0], bounds[0])) * max(0, min(bbox[3], bounds[3]) - max(bbox[1], bounds[1])) >= 0.8 * (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
        native = max(pixels[0] / axes[0], pixels[1] / axes[1]) if min(axes) > 0 else 0
        density = max(density, native)
        images.append({'native_size': list(pixels), 'bounds_pdf': bounds, 'matrix_pdf': [matrix.a, matrix.b, matrix.c, matrix.d, matrix.e, matrix.f], 'level': obj.level, 'full_page_scan': full, 'pixels_per_72_canvas_units': 72 * native})
    reasons = []
    simple = len(objects) == 1 and objects[0].type == pdfium.raw.FPDF_PAGEOBJ_IMAGE
    obj = objects[0] if simple else None
    if not simple:
        reasons.append('not_one_top_level_image; preserve_composition')
    if rotation:
        reasons.append('page_rotation')
    if annotations != 0:
        reasons.append('page_annotations_or_unknown')
    if simple:
        m = obj.get_matrix()
        l, b, r, t = bbox
        if obj.level or abs(m.b) > 1e-06 or abs(m.c) > 1e-06 or (m.a <= 0) or (m.d <= 0) or any((abs(a - v) > 0.001 for a, v in zip(obj.get_bounds(), bbox))) or any((abs(a - v) > 0.001 for a, v in zip([m.e, m.f, m.a, m.d], [l, b, r - l, t - b]))):
            reasons.append('placement_crop_rotation_or_reflection')
        clip = pdfium.raw.FPDFPageObj_GetClipPath(obj)
        if clip and pdfium.raw.FPDFClipPath_CountPaths(clip) > 0:
            reasons.append('image_clip_path')
    image, method, validation = (None, 'pdf_render', None)
    if strategy == 'auto' and (not reasons):
        nw, nh = obj.get_px_size()
        if nw * nh > max_source_pixels:
            raise ValueError('Native image exceeds max_source_pixels')
        scale = math.nextafter(min(nw / width, nh / height), 0)
        if abs(nw / width - nh / height) > 0.0001 * max(nw / width, nh / height):
            reasons.append('anisotropic_native_geometry')
        else:
            bitmap = obj.get_bitmap(render=False)
            try:
                candidate = bitmap.to_pil().convert('RGB')
            finally:
                if hasattr(bitmap, 'warn_on_close'):
                    bitmap.warn_on_close = False
                bitmap.close()
            visible = pdf_visible_render(page, scale, max_source_pixels)
            equal = candidate.size == visible.size and ImageChops.difference(candidate, visible).getbbox() is None
            validation = {'method': 'full_pixel_equality_to_visible_render', 'equal': equal, 'render_size': list(visible.size), 'scale': scale}
            if equal:
                image, method = (candidate, 'native_validated')
            else:
                reasons.append('native_pixels_differ_from_visible_page')
    if image is None:
        raster_only = bool(images) and all((o.type in {pdfium.raw.FPDF_PAGEOBJ_IMAGE, pdfium.raw.FPDF_PAGEOBJ_FORM} for o in objects)) and (annotations == 0)
        scale = density if raster_only and density else max(dpi / 72, density)
        image = pdf_visible_render(page, scale, max_source_pixels)
    else:
        scale = validation['scale']
    meta = {'version': PDF_IMAGE_VERSION, 'method': method, 'size': list(image.size), 'page_size_pdf': [width, height], 'page_bbox_pdf': bbox, 'media_box_pdf': list(page.get_mediabox()), 'crop_box_pdf': list(page.get_cropbox()), 'rotation': rotation, 'object_types': [o.type for o in objects], 'annotations': annotations, 'embedded_images': images, 'requested_dpi': dpi, 'render_scale': scale, 'native_validation': validation, 'fallback_reasons': reasons, 'source_downsampled': False, 'source_to_pdf': pdf_pixel_to_page(image.size, bbox, rotation), 'resolution_unit': 'pixels_per_72_pdf_canvas_units; physical_UserUnit_unavailable', 'warnings': ['low_native_resolution; interpolation_cannot_recover_detail'] if images and max((max(i['native_size']) for i in images)) <= 750 else []}
    return (image, meta)
from math import floor, isfinite, sqrt
from pathlib import Path

def projection_mask(image, options):
    """Analysis-only ink mask; never use these pixels for canonical/OCR crops."""
    from PIL import Image, ImageChops, ImageFilter, ImageOps
    gray = image.convert('L')
    gray.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    radius = max(2, min(gray.size) / 80)
    mask = ImageChops.subtract(gray.filter(ImageFilter.GaussianBlur(radius)), gray)
    return ImageOps.autocontrast(mask).point(lambda v: 255 if v > options.get('ink_threshold', 40) else 0)

def projection_gaps(image, axis, options):
    """Find internal low-ink gutters; suppress long rules in analysis only."""
    mask = projection_mask(image, options)
    w, h = mask.size
    data = mask.tobytes()
    left, right = (max(1, int(w * 0.03)), min(w - 1, int(w * 0.97)))
    top, bottom = (max(1, int(h * 0.03)), min(h - 1, int(h * 0.97)))
    if left >= right or top >= bottom:
        return []
    rows = [sum((v > 0 for v in data[y * w + left:y * w + right])) for y in range(top, bottom)]
    valid_rows = [y for y, count in zip(range(top, bottom), rows) if count < 0.7 * (right - left)]
    if not valid_rows:
        return []
    cols = [sum((data[y * w + x] > 0 for y in valid_rows)) for x in range(left, right)]
    valid_cols = [x for x, count in zip(range(left, right), cols) if count < 0.7 * len(valid_rows)]
    if not valid_cols:
        return []
    if axis == 'x':
        values = [sum((data[y * w + x] > 0 for y in valid_rows)) / len(valid_rows) for x in range(w)]
        extent, length = (image.width, w)
    else:
        values = [sum((data[y * w + x] > 0 for x in valid_cols)) / len(valid_cols) for y in range(h)]
        extent, length = (image.height, h)
    values = [0.0 if value >= 0.7 else value for value in values]
    threshold = options.get('gap_ink_fraction', 0.006)
    minimum = max(2, round(length * options.get('min_gap_fraction', 0.006)))
    edge = max(minimum, round(length * 0.06))
    gaps, start = ([], None)
    for i in range(edge, length - edge + 1):
        blank = i < length - edge and values[i] <= threshold
        if blank and start is None:
            start = i
        if not blank and start is not None:
            if i - start >= minimum:
                cut = round((start + i) / 2 * extent / length)
                span = max(minimum * 3, round(length * 0.035))
                before = values[max(edge, start - span):start]
                after = values[i:min(length - edge, i + span)]
                support = max(3, round(length * 0.006))
                if sum((v > threshold * 2 for v in before)) >= support and sum((v > threshold * 2 for v in after)) >= support:
                    gaps.append(cut)
            start = None
    return gaps

def _partition_box(box, cuts, axis, direction):
    x1, y1, x2, y2 = box
    ends = [0, *cuts, x2 - x1 if axis == 'x' else y2 - y1]
    result = [(x1 + a, y1, x1 + b, y2) for a, b in zip(ends, ends[1:])] if axis == 'x' else [(x1, y1 + a, x2, y1 + b) for a, b in zip(ends, ends[1:])]
    return result[::-1] if axis == 'x' and direction == 'vertical_rtl' else result
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from fnmatch import fnmatch
from datetime import datetime, timezone
from tqdm.auto import tqdm

def _sanitize(name: str) -> str:
    return re.sub('[^\\w\\-]+', '_', name).strip('_') or 'run'

def _run_dir(workdir: str | Path, kind: str, name: str, cfg) -> Path:
    digest = cfg.cache_key()[:8]
    return Path(workdir) / 'runs' / f'{kind}_{_sanitize(name)}_{digest}'

def parse_pages(spec: str | int, n: int) -> list[int]:
    """'1-10,15' -> [0..9, 14] (đếm từ 1); '' -> mọi trang."""
    spec = str(spec)
    if not spec.strip():
        return list(range(n))
    out: list[int] = []
    for part in spec.split(','):
        a, _, b = part.strip().partition('-')
        out += range(int(a) - 1, min(int(b or a), n))
    return sorted(set((p for p in out if 0 <= p < n)))

def _pdf_pages(path: Path) -> int:
    import pypdfium2 as pdfium
    doc = pdfium.PdfDocument(str(path))
    try:
        return len(doc)
    finally:
        doc.close()

def _entry_matches(entry: dict, pattern: str) -> bool:
    return fnmatch(entry['name'], pattern) or fnmatch(entry['stem'], pattern) or fnmatch(Path(entry['path']).name, pattern)

def select_entries(entries: list[dict], patterns: list[str]) -> list[dict]:
    """Chọn sách theo tên/glob. [] = tất cả. Pattern không khớp -> báo lỗi."""
    if not patterns:
        return entries
    chosen: list[dict] = []
    seen: set[str] = set()
    missing: list[str] = []
    for pat in patterns:
        hits = [e for e in entries if _entry_matches(e, pat)]
        if not hits:
            missing.append(pat)
        for e in hits:
            if e['path'] not in seen:
                chosen.append(e)
                seen.add(e['path'])
    if missing:
        names = ', '.join((e['name'] for e in entries))
        raise ValueError(f'books không khớp: {missing}. Có sẵn: {names}')
    return chosen

def list_pdfs(input_dir: str | Path) -> list[dict]:
    """Liệt kê PDF trong input_dir: name (đã làm sạch), stem, path, pages."""
    root = Path(input_dir)
    if not root.exists():
        raise FileNotFoundError(f'không thấy input_dir: {root}')
    files = [p for p in sorted(root.rglob('*.pdf')) if '__MACOSX' not in str(p)]
    if not files:
        raise FileNotFoundError(f'không thấy PDF nào trong {root}')
    entries: list[dict] = []
    used: set[str] = set()
    for p in files:
        name = _sanitize(p.stem)
        while name in used:
            name += '_'
        used.add(name)
        entries.append({'name': name, 'stem': p.stem, 'path': str(p)})
    for entry, path in zip(entries, files):
        entry['n_pages'] = _pdf_pages(path)
    return entries
from pathlib import Path, PureWindowsPath

def runtime_path(value: str | Path) -> Path:
    return Path(value)
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED
from threading import Event, Lock
from collections import deque

def gpu_worker_environment(indices, environment=None):
    """Select logical devices, respecting a mask inherited from the platform."""
    env = dict(os.environ if environment is None else environment)
    indices = list(indices)
    if not indices or any((type(i) is not int or i < 0 for i in indices)) or len(set(indices)) != len(indices):
        raise ValueError('GPU IDs must be distinct nonnegative logical indices')
    inherited = env.get('CUDA_VISIBLE_DEVICES')
    visible = inherited.split(',') if inherited is not None else None
    if visible is not None:
        visible = [v.strip() for v in visible if v.strip() and v.strip() != '-1']
        if any((i >= len(visible) for i in indices)):
            raise ValueError(f'GPU IDs {indices} exceed inherited CUDA_VISIBLE_DEVICES={inherited!r}')
    env['CUDA_VISIBLE_DEVICES'] = ','.join((visible[i] if visible is not None else str(i) for i in indices))
    return env

def validate_ocr_gpu_map(engines, mapping):
    if set(mapping) != set(engines):
        raise ValueError('GPU_MAP must assign exactly one GPU to every OCR engine')
    ids = list(mapping.values())
    if any((type(i) is not int or i < 0 for i in ids)) or len(set(ids)) != len(ids):
        raise ValueError('Parallel OCR requires one distinct nonnegative GPU ID per engine')
    return dict(mapping)

def probe_gpu_inventory():
    """Probe in a child so importing Torch cannot freeze the parent's CUDA mask."""
    source = "import json, torch\ndevices=[]\nfor i in range(torch.cuda.device_count()):\n    x=torch.ones(2,device=f'cuda:{i}'); value=float((x*x).sum().item())\n    devices.append(dict(index=i,name=torch.cuda.get_device_name(i),\n        total_bytes=torch.cuda.get_device_properties(i).total_memory,\n        free_bytes=torch.cuda.mem_get_info(i)[0],capability=list(torch.cuda.get_device_capability(i)),\n        tensor_sum=value))\nprint(json.dumps(devices))\n"
    result = subprocess.run([sys.executable, '-u', '-c', source], capture_output=True, text=True, encoding='utf8', timeout=90, check=True)
    return json.loads(result.stdout.splitlines()[-1])

def run_gpu_task_waves(tasks, function, gpu_ids):
    """Run one task per device per wave; multi-GPU tasks reserve the entire set.

    tasks is a sequence of (name, reserved_gpu_ids). The caller owns publication.
    The event lets process supervisors stop their children on notebook interrupt.
    """
    allowed = set(gpu_ids)
    wave, used, results = ([], set(), {})
    cancel = Event()

    def drain():
        if not wave:
            return
        with ThreadPoolExecutor(max_workers=len(wave)) as pool:
            futures = {pool.submit(function, name, cancel): name for name, _ in wave}
            try:
                for future in as_completed(futures):
                    results[futures[future]] = future.result()
            except BaseException:
                cancel.set()
                for future in futures:
                    future.cancel()
                raise
        wave.clear()
        used.clear()
    for name, ids in tasks:
        reserved = set(ids)
        if not reserved <= allowed or len(reserved) != len(ids):
            raise ValueError(f'{name}: invalid GPU reservation {ids}')
        if used & reserved or (wave and (not reserved)):
            drain()
        if not reserved:
            results[name] = function(name, cancel)
            continue
        wave.append((name, ids))
        used.update(reserved)
    drain()
    return results

def choose_gpu_job(pending, active, timings, *, preferred=None, failed=()):
    """Start untouched models first, then help the longest estimated backlog.

    A helper is worthwhile only when its share of remaining inference exceeds
    the measured model load cost. Without measurements require two queued pages.
    Running pages are never moved between workers.
    """
    candidates = [n for n, count in pending.items() if count and n not in failed]
    fresh = [n for n in candidates if not active.get(n, 0)]
    if fresh:
        return preferred if preferred in fresh else max(fresh, key=lambda n: pending[n])

    def worthwhile(name):
        timing = timings.get(name, {})
        if not timing.get('page_seconds'):
            return pending[name] >= 2
        return pending[name] * timing['page_seconds'] / (active.get(name, 0) + 1) > timing.get('load_seconds', 0)
    candidates = [n for n in candidates if worthwhile(n)]
    return max(candidates, key=lambda n: pending[n] * timings.get(n, {}).get('page_seconds', 1) / (active.get(n, 0) + 1), default=None)

def run_gpu_page_jobs(jobs, function, gpu_ids, *, preferred=None, cancel_event=None, reserved_jobs=None):
    """Persistent worker per GPU/model; idle cards steal unclaimed pages.

    function(name, page_iterator, gpu_ids, slot, cancel) owns one model load.
    jobs maps single-GPU models to pages. reserved_jobs maps sharded models to
    (pages, GPU IDs); one copy holds its reservation while spare cards do other
    work. No framework is imported.
    """
    queues = {name: deque(pages) for name, pages in jobs.items()}
    reserved_jobs = reserved_jobs or {}
    pending_heavy = list(reserved_jobs)
    for name, (pages, ids) in reserved_jobs.items():
        if name in queues or not ids or (not set(ids) <= set(gpu_ids)) or (len(set(ids)) != len(ids)):
            raise ValueError(f'{name}: invalid shared-model reservation')
        queues[name] = deque(pages)
    active, timings, results = ({}, {}, {})
    lock, cancel = (Lock(), cancel_event if cancel_event is not None else Event())

    class PageStream:

        def __init__(self, name):
            self.name = name

        def __iter__(self):
            first = True
            while not cancel.is_set():
                with lock:
                    if not queues[self.name]:
                        return
                    page = queues[self.name].popleft()
                started = time.monotonic()
                yield page
                with lock:
                    old = timings.setdefault(self.name, {})
                    seconds = time.monotonic() - started
                    if first:
                        seconds = max(0.001, seconds - old.get('load_seconds', 0))
                    old['page_seconds'] = seconds
                    first = False

        def model_loaded(self, seconds):
            with lock:
                timings.setdefault(self.name, {})['load_seconds'] = seconds or 0
    free, running = (set(gpu_ids), {})
    with ThreadPoolExecutor(max_workers=len(gpu_ids)) as pool:

        def launch(name, ids):
            active[name] = active.get(name, 0) + 1
            free.difference_update(ids)
            slot = '-'.join(map(str, ids))
            running[pool.submit(function, name, PageStream(name), list(ids), slot, cancel)] = (name, ids, slot)
        try:
            while not cancel.is_set():
                with lock:
                    for name in pending_heavy[:]:
                        ids = reserved_jobs[name][1]
                        if set(ids) <= free:
                            launch(name, ids)
                            pending_heavy.remove(name)
                    for gpu in sorted(free):
                        name = choose_gpu_job({n: len(queues[n]) for n in jobs}, active, timings, preferred=(preferred or {}).get(gpu))
                        if name is not None:
                            launch(name, [gpu])
                if not running:
                    break
                finished, _ = wait(running, return_when=FIRST_COMPLETED)
                for future in finished:
                    name, ids, slot = running.pop(future)
                    results[name, slot] = future.result()
                    active[name] -= 1
                    free.update(ids)
        except BaseException:
            cancel.set()
            raise
    return results
VARIANT_SPECS = {'docentr_binarization': (1, 'neural_restoration', 'neural_restoration', 'DocEnTR released base/8 H-DIBCO2018 ViT reconstruction', 'learned_prior_sensitivity'), 'degan_binarization': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 1024; Keras Torch backend; threshold .95', 'learned_prior_sensitivity'), 'degan_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 1024 deblur; Keras Torch backend', 'learned_prior_sensitivity'), 'degan_watermark_removal': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 512 watermark removal; explicit region review', 'learned_prior_sensitivity'), 'original': (1, 'source_observation', 'source', 'canonical RGB pixels', 'reference'), 'clahe': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CLAHE Lab L', 'conservative_probe'), 'bicubic_x2': (2, 'deterministic_transformation', 'geometric_scale_interpolation', 'Pillow bicubic x2', 'conservative_probe'), 'bicubic_x4': (4, 'deterministic_transformation', 'geometric_scale_interpolation', 'Pillow bicubic x4', 'conservative_probe'), 'denoised': (1, 'deterministic_transformation', 'denoising_sharpening', 'NLM Lab L', 'glyph_sensitivity'), 'grayscale_luma': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB luminance projection', 'glyph_sensitivity'), 'grayscale_red': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB R projection', 'selective_information'), 'grayscale_green': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB G projection', 'selective_information'), 'grayscale_blue': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB B projection', 'selective_information'), 'lab_l': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab L projection', 'glyph_sensitivity'), 'lab_a': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab a chroma diagnostic', 'selective_information'), 'lab_b': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab b chroma diagnostic', 'selective_information'), 'red_ink': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB red dominance mask', 'selective_information'), 'background_normalized': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'Gaussian background division Lab L', 'glyph_sensitivity'), 'sauvola': (1, 'deterministic_transformation', 'binarization', 'Sauvola reflected local moments', 'glyph_sensitivity'), 'niblack': (1, 'deterministic_transformation', 'binarization', 'Niblack mean-minus-k-std; equality is background', 'glyph_sensitivity'), 'bilateral': (1, 'deterministic_transformation', 'denoising_sharpening', 'bilateral Lab L', 'glyph_sensitivity'), 'unsharp': (1, 'deterministic_transformation', 'denoising_sharpening', 'thresholded unsharp Lab L', 'glyph_sensitivity'), 'retinex_msr': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'three-scale log Retinex intensity; percentile mapping', 'glyph_sensitivity'), 'retinex_msrcp': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'three-scale MSRCP; common RGB gain with saturation bound', 'glyph_sensitivity'), 'wolf': (1, 'deterministic_transformation', 'binarization', 'Wolf-Jolion global-normalized local threshold', 'glyph_sensitivity'), 'nick': (1, 'deterministic_transformation', 'binarization', 'NICK mean+k*sqrt(E[I^2]+variance)', 'glyph_sensitivity'), 'color_pca': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB principal-component projection; luminance sign', 'selective_information'), 'color_gmm': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'diagonal Lab GMM EM; brightest component is paper', 'selective_information'), 'mean_shift': (1, 'deterministic_transformation', 'denoising_sharpening', 'OpenCV spatial-color mean shift, no pyramid by default', 'glyph_sensitivity'), 'anisotropic_diffusion': (1, 'deterministic_transformation', 'denoising_sharpening', 'Perona-Malik exp conductivity; 4 neighbors; no-flux Lab L', 'glyph_sensitivity'), 'background_morphological': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'grayscale closing surface; divide original Lab L', 'glyph_sensitivity'), 'background_guided': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'closing surface self-guided filter; divide original Lab L', 'glyph_sensitivity'), 'background_otsu': (1, 'deterministic_transformation', 'binarization', 'hypothesis: closing surface -> division -> Otsu', 'glyph_sensitivity'), 'realesrgan_x4': (4, 'neural_restoration', 'neural_restoration', 'Real-ESRGAN RRDBNet x4plus', 'learned_prior_sensitivity'), 'tadisr_x4': (4, 'neural_restoration', 'neural_restoration', 'TADiSR CogView4 RealCE joint segmentation decoder', 'learned_prior_sensitivity'), 'difftsr': (None, 'neural_restoration', 'neural_restoration', 'DiffTSR IDM/TDM/MoM and TransOCR text prior', 'learned_prior_sensitivity'), 'unihir_restoration': (1, 'generative_reconstruction', 'generative_reconstruction', 'UniHIR BAGEL MLLM reconstruction', 'generative_sensitivity'), 'docres_appearance': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt appearance', 'learned_prior_sensitivity'), 'docres_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt deblurring', 'learned_prior_sensitivity'), 'docres_deshadowing': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt deshadowing', 'learned_prior_sensitivity'), 'docres_binarization': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt binarization', 'learned_prior_sensitivity'), 'docdiff_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DocDiff coarse predictor and linear-100 residual diffusion', 'learned_prior_sensitivity'), 'restormer_motion_deblurring': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN motion checkpoint', 'learned_prior_sensitivity'), 'restormer_defocus_deblurring': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN single-image defocus checkpoint', 'learned_prior_sensitivity'), 'restormer_real_denoising': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN SIDD checkpoint', 'learned_prior_sensitivity'), 'restormer_gaussian_denoising': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN blind Gaussian checkpoint', 'learned_prior_sensitivity')}
SURVEY_SOURCE_GROUPS = {'39d1b637eed99a6e1981c326e876537ccf96baa897e16f1dca4f6aec01e32800': 'G1', '3e9a2be8a104ad044cd054dee7c0a98422aa57d2c9a4ceaea985b8d9a56a0d0f': 'G1', 'ba8129a51d43a81a2ab566153596ebe482c72a570a0c9f319a69d79c95dde7b5': 'G1', '68b6accd0c037854e41e65ed45fe1ebcd27889999a75ae3df2bfc597275107e7': 'G1', '66fd3918224508aa4dffa5cecc5abe741094c563569a61ea9e976f893811b38e': 'G2', '4b26a2be1db3c95fbd4d4387fd72c417525a813f5ed2c6a6787e02a08163121a': 'G3'}
UNAVAILABLE_RESTORATION_METHODS = {'lwtformer': {'state': 'not_integrated', 'reason': 'Official repository has architecture/test code but no verified released pretrained checkpoint', 'upstream': 'https://github.com/INWLY/LWTformer'}, 'gladunet': {'state': 'not_integrated', 'reason': 'No accessible official inference implementation and pretrained checkpoint verified', 'upstream': 'https://www.nature.com/articles/s40494-025-01965-8'}, 'gatos_energy': {'state': 'not_integrated', 'reason': 'No pinned reliable complete upstream implementation; background_otsu is a separate implemented hypothesis'}, 'single_image_bleed_reduction': {'state': 'not_integrated', 'reason': 'No verified dedicated implementation and no established bleed-through labels in the surveyed corpus'}}

def variant_spec(name):
    if name == 'tadisr_text_mask':
        return dict(method_family='neural_restoration', algorithm='TADiSR predicted segmentation mask', perturbation_class='diagnostic_only', provenance='neural_restoration', scale=4)
    scale, provenance, group, algorithm, interpretation = VARIANT_SPECS[name]
    return dict(method_family=group, algorithm=algorithm, perturbation_class=interpretation, provenance=provenance, scale=scale)

def preparation_applicability(name, book, page, region, config):
    """Explicit analyst conditions only; never invent a degradation detector."""
    rule = config.get('applicability', {}).get(name, {})
    stem = str(book).removesuffix('.pdf')
    if rule.get('books') is not None and (not {str(book), stem}.intersection(rule['books'])):
        return ('not_applicable', 'book outside explicit method applicability')
    if rule.get('crop_types') is not None and region.get('crop_type') not in rule['crop_types']:
        return ('not_applicable', 'crop type outside explicit method applicability')
    review = preparation_region_review(book, page, region['id'], config)
    if rule.get('require_review') and name not in review.get('approved_variants', []):
        return ('not_applicable', 'requires explicit region approval; degradation unverified')
    return ('ready', None)

def preparation_region_review(book, page, region_id, config):
    reviews = config.get('reviewed_regions', {})
    return reviews.get(f'{book}:{page + 1}:{region_id}', reviews.get(f"{str(book).removesuffix('.pdf')}:{page + 1}:{region_id}", {}))

def validate_applicability(config):
    for name, rule in config.get('applicability', {}).items():
        if name not in VARIANT_SPECS or not isinstance(rule, dict) or set(rule) - {'books', 'crop_types', 'require_review'}:
            raise ValueError('Unknown method/applicability option')
        for key in ('books', 'crop_types'):
            if key in rule and (not isinstance(rule[key], list) or any((not isinstance(v, str) for v in rule[key]))):
                raise ValueError('Applicability selections must be lists of strings')
        if set(rule.get('crop_types', [])) - {'page', 'group', 'column'}:
            raise ValueError('Invalid applicability crop_types')
        if 'require_review' in rule and type(rule['require_review']) is not bool:
            raise ValueError('require_review must be boolean')
EXTRA_CLASSICAL = ()
from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
import io
from pathlib import Path, PurePosixPath
IMAGE_DATASET_FORMAT = 'vtm-image-dataset'
IMAGE_DATASET_VERSION = 1
IMAGE_DATASET_CODE_SIGNATURE = '3a454bd5aa9586106fc34026efd24aeb0f3d97e2cfc5d2a444b91abcf88be492'

def dataset_json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False) + '\n').encode('utf-8')

def dataset_hash(value):
    return hashlib.sha256(dataset_json_bytes(value)).hexdigest()

def dataset_file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def dataset_code_signature():
    if IMAGE_DATASET_CODE_SIGNATURE:
        return IMAGE_DATASET_CODE_SIGNATURE
    root = Path(__file__).parent
    return dataset_hash({name: dataset_file_hash(root / name) for name in ('image_dataset.py', 'image_preparation.py', 'pdf_images.py', 'restoration_classical.py', 'restoration_registry.py')})

def dataset_environment():
    import PIL
    import pypdfium2 as pdfium
    result = {'pillow': PIL.__version__, 'pypdfium2': str(pdfium.PYPDFIUM_INFO), 'pdfium': str(pdfium.PDFIUM_INFO), 'python': sys.version.split()[0]}
    return result

def dataset_path(root, relative):
    relative = str(relative)
    if '\\' in relative or ':' in relative:
        raise ValueError(f'Dataset requires portable relative paths: {relative}')
    parts = PurePosixPath(relative)
    if parts.is_absolute() or '..' in parts.parts or (not parts.parts):
        raise ValueError(f'Invalid dataset path: {relative}')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError(f'Dataset path escapes root: {relative}')
    return path

def dataset_write(root, relative, data):
    path = dataset_path(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_bytes(data)
    temporary.replace(path)
    return {'path': str(relative), 'sha256': hashlib.sha256(data).hexdigest()}

def dataset_read(root, reference):
    data = dataset_path(root, reference['path']).read_bytes()
    if hashlib.sha256(data).hexdigest() != reference['sha256']:
        raise ValueError(f"Dataset checksum mismatch: {reference['path']}")
    return data

def dataset_store_image(root, image):
    image = image.convert('RGB')
    output = io.BytesIO()
    image.save(output, format='PNG', compress_level=6)
    data = output.getvalue()
    sha = hashlib.sha256(data).hexdigest()
    relative = f'objects/{sha[:2]}/{sha}.png'
    path = dataset_path(root, relative)
    if not path.exists() or dataset_file_hash(path) != sha:
        dataset_write(root, relative, data)
    return {'path': relative, 'sha256': sha, 'size': list(image.size), 'pixel_sha256': hashlib.sha256(image.tobytes()).hexdigest()}

def dataset_load_image(root, reference):
    from PIL import Image
    with Image.open(io.BytesIO(dataset_read(root, reference))) as loaded:
        image = loaded.convert('RGB')
    if list(image.size) != reference['size'] or hashlib.sha256(image.tobytes()).hexdigest() != reference['pixel_sha256']:
        raise ValueError(f"Dataset image dimensions/pixels changed: {reference['path']}")
    return image

@contextmanager
def dataset_lock(root):
    """OS lock releases after interruption; no stale PID/lockfile cleanup needed."""
    import os
    Path(root).mkdir(parents=True, exist_ok=True)
    with (Path(root) / 'prepare.lock').open('a+b') as stream:
        if stream.tell() == 0:
            stream.write(b'0')
            stream.flush()
        stream.seek(0)
        import fcntl
        fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        try:
            yield
        finally:
            stream.seek(0)
            fcntl.flock(stream, fcntl.LOCK_UN)

def dataset_validate_cuts(cuts):
    if cuts is not None and (not isinstance(cuts, list) or any((isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (not 0 < v < 1) for v in cuts)) or cuts != sorted(set(cuts))):
        raise ValueError('column_cuts must be unique increasing fractions in (0,1)')

def dataset_partition_qc(regions, size, direction):
    """Exact geometric partition check, independent of OCR/text completeness."""
    w, h = size
    expected = 0 if direction == 'horizontal_ltr' else w
    for index, region in enumerate(regions):
        x1, y1, x2, y2 = region['source_box']
        if not all((isinstance(v, int) for v in (x1, y1, x2, y2))) or not 0 <= x1 < x2 <= w or (y1, y2) != (0, h) or (region['reading_order'] != index):
            raise ValueError('Invalid canonical crop bounds/order')
        start, end = (x1, x2) if direction == 'horizontal_ltr' else (x2, x1)
        if start != expected:
            raise ValueError('Canonical partition has a gap, overlap or inconsistent reading order')
        expected = end
    if not regions or expected != (w if direction == 'horizontal_ltr' else 0):
        raise ValueError('Canonical partition omits page content')

def dataset_plan_layout(image, config, *, page=0):
    """Projection baseline, no model/pixel budget; all coordinates in source pixels."""
    from PIL import Image, ImageDraw
    analysis = image.copy()
    analysis.thumbnail((config.analysis_max_side, config.analysis_max_side), Image.Resampling.LANCZOS)
    options = {key: getattr(config, key) for key in ('ink_threshold', 'gap_ink_fraction', 'min_gap_fraction')}
    mask = projection_mask(analysis, options)
    w, h = image.size
    aw, ah = analysis.size
    manual = config.page_column_cuts.get(str(page + 1), config.column_cuts)
    if manual is not None:
        cuts = [round(v * w) for v in manual]
        method = 'manual_columns'
    elif config.reading_direction == 'vertical_rtl':
        cuts = sorted(set((round(v * w / aw) for v in projection_gaps(analysis, 'x', options))))
        method = 'whitespace_projection'
    else:
        cuts, method = ([], 'whole_page_horizontal')
    if len(set([0, *cuts, w])) != len(cuts) + 2:
        raise ValueError('Column cuts collapse in source space')
    warnings = ['heuristic_layout; geometric_coverage_is_not_text_coverage']
    if len(cuts) + 1 > config.max_columns:
        if manual is not None:
            raise ValueError('Manual layout exceeds max_columns')
        warnings.append('too_many_gutters; retained_whole_page')
        cuts = []
    if not cuts:
        warnings.append('no_partition; whole_page_retained')
    thresholds = sorted(set([max(1, config.ink_threshold - 8), config.ink_threshold, min(254, config.ink_threshold + 8)]))
    found = [projection_gaps(analysis, 'x', {**options, 'ink_threshold': t}) for t in thresholds]
    ink = mask.tobytes()
    top, bottom = (int(0.03 * ah), max(1, int(0.97 * ah)))
    projection = [sum((ink[y * aw + x] > 0 for y in range(top, bottom))) / (bottom - top) for x in range(aw)]
    evidence = []
    for cut in cuts:
        x = min(aw - 1, max(0, round(cut * aw / w)))
        lo, hi = (x, x)
        while lo > 0 and projection[lo - 1] <= config.gap_ink_fraction:
            lo -= 1
        while hi < aw - 1 and projection[hi + 1] <= config.gap_ink_fraction:
            hi += 1
        tolerance = max(2, round(aw * 0.006))
        stability = sum((any((abs(c - x) <= tolerance for c in cs)) for cs in found)) / len(found)
        radius = max(2, round(aw * 0.035))
        left, right = (projection[max(0, x - radius):x], projection[x + 1:min(aw, x + radius + 1)])
        risks = []
        if projection[x] > 0.02:
            risks.append('cut_crosses_analysis_ink; rules_or_strokes_need_review')
        if stability < 2 / 3:
            risks.append('unstable_across_ink_thresholds')
        if not left or not right or min(max(left), max(right)) <= config.gap_ink_fraction * 2:
            risks.append('weak_text_support_on_one_side')
        evidence.append({'source_x': cut, 'analysis_x': x, 'gutter_width_analysis': hi - lo + 1, 'ink_fraction': projection[x], 'threshold_stability': stability, 'warnings': risks})
    full = (0, 0, w, h)
    columns = _partition_box(full, cuts, 'x', config.reading_direction)
    group = config.columns_per_group
    groups = []
    for start in range(0, len(columns), group):
        chunk = columns[start:start + group]
        groups.append((min((b[0] for b in chunk)), 0, max((b[2] for b in chunk)), h))
    regions = []
    for kind, boxes in (('page', [full]), ('group', groups), ('column', columns)):
        for index, box in enumerate(boxes):
            regions.append({'id': f'{kind}_{index:03d}', 'crop_type': kind, 'reading_order': index, 'source_box': list(box), 'overlap': False})
        dataset_partition_qc([r for r in regions if r['crop_type'] == kind], image.size, config.reading_direction)
    preview = analysis.copy()
    draw = ImageDraw.Draw(preview)
    for region in [r for r in regions if r['crop_type'] == 'column']:
        box = region['source_box']
        x1, x2 = (round(box[0] * aw / w), round(box[2] * aw / w))
        draw.rectangle((x1, 0, max(x1, x2 - 1), ah - 1), outline='red', width=1)
        draw.text((x1 + 2, 3), str(region['reading_order'] + 1), fill='red')
    for item in evidence:
        if item['warnings']:
            draw.line((item['analysis_x'], 0, item['analysis_x'], ah - 1), fill='orange', width=2)
    draw.text((3, ah - 14), 'RTL <-' if config.reading_direction == 'vertical_rtl' else 'LTR ->', fill='red')
    layout = {'method': method, 'source_size': [w, h], 'analysis_size': [aw, ah], 'analysis_to_source': [w / aw, 0, 0, h / ah, 0, 0], 'reading_direction': config.reading_direction, 'regions': regions, 'cuts': evidence, 'excluded_regions': [], 'uncertain_regions': [r['id'] for r in regions if r['crop_type'] == 'column'], 'geometric_coverage': 1.0, 'text_coverage': 'unverified', 'warnings': warnings, 'thresholds_tested': thresholds, 'review_status': 'needs_visual_review'}
    return (layout, mask.convert('RGB'), preview)

def dataset_validate_page(root, record, *, check_pixels=True):
    source = dataset_load_image(root, record['source']) if check_pixels else None
    acquisition = json.loads(dataset_read(root, record['acquisition']))
    layout = json.loads(dataset_read(root, record['layout']))
    if record['source']['size'] != layout['source_size'] or acquisition['size'] != layout['source_size']:
        raise ValueError('Source, acquisition and layout size mismatch')
    dataset_read(root, layout['analysis_mask'])
    dataset_read(root, layout['preview'])
    for kind in ('page', 'group', 'column'):
        regions = [r for r in layout['regions'] if r['crop_type'] == kind]
        dataset_partition_qc(regions, layout['source_size'], layout['reading_direction'])
    if [r['id'] for r in record['regions']] != [r['id'] for r in layout['regions']]:
        raise ValueError('Image dataset omits canonical regions')
    for region, geometry in zip(record['regions'], layout['regions']):
        if any((region[k] != geometry[k] for k in ('source_box', 'crop_type', 'reading_order'))):
            raise ValueError('Region geometry differs from canonical layout')
        if 'original' not in region['images']:
            raise ValueError('Region original variant is missing')
        x1, y1, x2, y2 = region['source_box']
        for name, entry in region['images'].items():
            expected = [(x2 - x1) * (2 if name == 'bicubic_x2' else 1), (y2 - y1) * (2 if name == 'bicubic_x2' else 1)]
            if entry['size'] != expected or entry['image_to_source'] != [(x2 - x1) / expected[0], 0, 0, (y2 - y1) / expected[1], x1, y1]:
                raise ValueError('Variant geometry/transform mismatch')
            if check_pixels:
                image = dataset_load_image(root, entry)
                if name == 'original' and image.tobytes() != source.crop(region['source_box']).tobytes():
                    raise ValueError('Original crop is not pixel-identical to source')
    return layout

def dataset_open(directory):
    path = runtime_path(directory)
    descriptor = path / 'dataset.json' if path.is_dir() else path
    root = descriptor.parent.parent if descriptor.parent.name == 'snapshots' else descriptor.parent
    data = json.loads(descriptor.read_text(encoding='utf-8'))
    if data.get('format') != IMAGE_DATASET_FORMAT or data.get('version') != IMAGE_DATASET_VERSION:
        raise ValueError('Unsupported image dataset format/version')
    body = {k: v for k, v in data.items() if k != 'snapshot_id'}
    if dataset_hash(body) != data['snapshot_id']:
        raise ValueError('Dataset snapshot identity mismatch')
    return (root, data)

def dataset_page(root, descriptor, source_sha, page):
    entries = [p for p in descriptor['pages'] if p['source_sha256'] == source_sha and p['page'] == page]
    if not entries:
        return None
    records = [json.loads(dataset_read(root, e['record'])) for e in entries]
    if len({r['input_key'] for r in records}) != 1:
        raise ValueError('Ambiguous prepared layouts for the same source/page')
    record = records[0]
    if record['source_sha256'] != source_sha or record['page'] != page:
        raise ValueError('Dataset page identity mismatch')
    return record

def prepare_image_dataset(input_dir, output_dir, *, config=None, books=None, pages='1-2', limit=2, max_books=1, profile_dir=None, publish=True):
    """Prepare a bounded selection; repeat the same command to resume.

    A completed source/layout/variant is independently reusable. Only after QC
    succeeds is a new dataset.json published. Existing snapshots remain usable.
    """
    config = config or ImageDatasetConfig()
    config.__post_init__()
    if limit is not None and limit < 1 or (max_books is not None and max_books < 1):
        raise ValueError('limit/max_books must be positive or None')
    entries = select_entries(list_pdfs(input_dir), books or [])
    if max_books is not None:
        entries = entries[:max_books]
    selected, remaining = ([], limit)
    for entry in entries:
        wanted = parse_pages(pages, entry['n_pages'])
        if remaining is not None:
            wanted = wanted[:remaining]
            remaining -= len(wanted)
        if wanted:
            selected.append((entry, wanted))
    if not selected:
        raise ValueError('No pages selected')
    root = Path(output_dir).resolve()
    stats = {'pages': 0, 'source_reused': 0, 'layout_reused': 0, 'variants_reused': 0}
    started, environment, code = (time.monotonic(), dataset_environment(), dataset_code_signature())
    page_entries, manifest_rows, configurations = ([], [], {})
    with dataset_lock(root):
        marker = root / 'dataset.json'
        if marker.exists():
            dataset_open(root)
        for entry, wanted in selected:
            import pypdfium2 as pdfium
            cfg = ImageDatasetConfig(**asdict(config))
            if profile_dir:
                profile = json.loads((Path(profile_dir) / (entry['stem'] + '.json')).read_text(encoding='utf-8-sig'))
                for key in ('dpi', 'reading_direction'):
                    if key in profile:
                        setattr(cfg, key, profile[key])
            cfg.__post_init__()
            configurations[entry['name']] = asdict(cfg)
            source_sha = dataset_file_hash(entry['path'])
            doc = pdfium.PdfDocument(entry['path'])
            try:
                doc.init_forms()
                for page in wanted:
                    source_key = dataset_hash({'source_sha256': source_sha, 'page': page, 'acquisition': cfg.acquisition, 'dpi': cfg.dpi, 'max_source_pixels': cfg.max_source_pixels, 'code': code, 'environment': environment})
                    source_receipt = f'sources/{source_key}.json'
                    try:
                        cached = json.loads(dataset_path(root, source_receipt).read_bytes())
                        if cached['key'] != source_key:
                            raise ValueError('Source cache key mismatch')
                        image = dataset_load_image(root, cached['source'])
                        acquisition = json.loads(dataset_read(root, cached['acquisition']))
                        stats['source_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        pdf_page = doc[page]
                        try:
                            image, acquisition = acquire_pdf_image(pdf_page, dpi=cfg.dpi, max_source_pixels=cfg.max_source_pixels, strategy=cfg.acquisition)
                        finally:
                            pdf_page.close()
                        source_asset = dataset_store_image(root, image)
                        acquisition_ref = dataset_write(root, f'sources/{source_key}.acquisition.json', dataset_json_bytes(acquisition))
                        cached = {'key': source_key, 'source': source_asset, 'acquisition': acquisition_ref}
                        dataset_write(root, source_receipt, dataset_json_bytes(cached))
                    layout_options = {k: v for k, v in asdict(cfg).items() if k not in {'dpi', 'acquisition', 'max_source_pixels', 'variants', 'variant_options'}}
                    layout_options['page_column_cuts'] = {str(page + 1): cfg.page_column_cuts.get(str(page + 1), cfg.column_cuts)}
                    layout_key = dataset_hash({'source': cached['source']['sha256'], 'options': layout_options, 'code': code})
                    layout_relative = f'layouts/{layout_key}.json'
                    try:
                        layout_data = dataset_path(root, layout_relative).read_bytes()
                        layout = json.loads(layout_data)
                        if layout['key'] != layout_key:
                            raise ValueError('Layout key mismatch')
                        dataset_read(root, layout['analysis_mask'])
                        dataset_read(root, layout['preview'])
                        stats['layout_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        layout, mask, preview = dataset_plan_layout(image, cfg, page=page)
                        layout.update(key=layout_key, analysis_mask=dataset_store_image(root, mask), preview=dataset_store_image(root, preview))
                        layout_data = dataset_json_bytes(layout)
                        dataset_write(root, layout_relative, layout_data)
                    layout_ref = {'path': layout_relative, 'sha256': hashlib.sha256(layout_data).hexdigest()}
                    regions = []
                    for geometry in layout['regions']:
                        crop = image.crop(geometry['source_box'])
                        variants = {}
                        for name in cfg.variants:
                            options = cfg.variant_options.get(name, {})
                            variant_environment = dict(environment)
                            if name in {'clahe', 'denoised', *EXTRA_CLASSICAL}:
                                import cv2
                                variant_environment['opencv'] = cv2.__version__
                            key = dataset_hash({'source': cached['source']['sha256'], 'box': geometry['source_box'], 'variant': name, 'options': options, 'code': code, 'environment': variant_environment})
                            receipt = f'variants/{key}.json'
                            try:
                                variant = json.loads(dataset_path(root, receipt).read_bytes())
                                if variant['key'] != key:
                                    raise ValueError('Variant cache key mismatch')
                                dataset_load_image(root, variant)
                                stats['variants_reused'] += 1
                            except (OSError, ValueError, KeyError):
                                prepared, operation = dataset_variant(crop, name, options)
                                x1, y1, x2, y2 = geometry['source_box']
                                variant = {**dataset_store_image(root, prepared), 'key': key, 'operation': operation, 'image_to_source': [(x2 - x1) / prepared.width, 0, 0, (y2 - y1) / prepared.height, x1, y1]}
                                dataset_write(root, receipt, dataset_json_bytes(variant))
                            variants[name] = variant
                            manifest_rows.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page + 1, 'page_index': page, 'acquisition_method': acquisition['method'], 'source_size': cached['source']['size'], 'source_image': cached['source'], 'analysis_size': layout['analysis_size'], 'analysis_to_source': layout['analysis_to_source'], 'source_to_pdf': acquisition['source_to_pdf'], 'region_id': geometry['id'], 'crop_type': geometry['crop_type'], 'source_box': geometry['source_box'], 'reading_order': geometry['reading_order'], 'reading_direction': cfg.reading_direction, 'variant': name, **variant, 'qc_warnings': acquisition['warnings'] + layout['warnings'] + [w for c in layout['cuts'] for w in c['warnings']], 'layout': layout_ref, 'acquisition': cached['acquisition'], 'code_sha256': code, 'processing_version': IMAGE_DATASET_VERSION})
                        regions.append({**geometry, 'images': variants})
                    record = {'input_key': dataset_hash({'source': source_key, 'layout': layout_key, 'variants': cfg.variants, 'options': cfg.variant_options}), 'source_sha256': source_sha, 'page': page, 'source': cached['source'], 'acquisition': cached['acquisition'], 'layout': layout_ref, 'regions': regions}
                    dataset_validate_page(root, record)
                    record_bytes = dataset_json_bytes(record)
                    record_key = hashlib.sha256(record_bytes).hexdigest()
                    record_ref = dataset_write(root, f'pages/{record_key}.json', record_bytes)
                    page_entries.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page, 'record': record_ref})
                    stats['pages'] += 1
            finally:
                doc.close()
        manifest_data = b''.join((dataset_json_bytes(row) for row in manifest_rows))
        manifest_sha = hashlib.sha256(manifest_data).hexdigest()
        manifest_ref = dataset_write(root, f'manifests/{manifest_sha}.jsonl', manifest_data)
        descriptor = {'format': IMAGE_DATASET_FORMAT, 'version': IMAGE_DATASET_VERSION, 'pages': page_entries, 'expected_pages': [[e['book'], e['source_sha256'], e['page']] for e in page_entries], 'manifest': manifest_ref, 'configurations': configurations, 'environment': environment, 'code_sha256': code, 'geometry': 'top_left_source_pixels; half_open_boxes', 'ocr_text_coverage': 'unverified'}
        descriptor['snapshot_id'] = dataset_hash(descriptor)
        data = dataset_json_bytes(descriptor)
        snapshot = f"snapshots/{descriptor['snapshot_id']}.json"
        dataset_write(root, snapshot, data)
        if publish:
            dataset_write(root, 'dataset.json', data)
    stats.update(snapshot_id=descriptor['snapshot_id'], dataset=str(root / snapshot), seconds=round(time.monotonic() - started, 3), manifest_rows=len(manifest_rows))
    return stats

def validate_image_dataset(directory):
    root, descriptor = dataset_open(directory)
    snapshot = root / f"snapshots/{descriptor['snapshot_id']}.json"
    if not snapshot.is_file() or dataset_open(snapshot)[1] != descriptor:
        raise ValueError('Immutable dataset snapshot missing or mismatched')
    dataset_read(root, descriptor['manifest'])
    actual = [[e['book'], e['source_sha256'], e['page']] for e in descriptor['pages']]
    if actual != descriptor['expected_pages'] or len(set(map(tuple, actual))) != len(actual):
        raise ValueError('Dataset expected-page coverage mismatch')
    warnings = []
    for entry in descriptor['pages']:
        record = dataset_page(root, descriptor, entry['source_sha256'], entry['page'])
        layout = dataset_validate_page(root, record)
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}: {w}" for w in layout['warnings']))
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}/x{c['source_x']}: {w}" for c in layout['cuts'] for w in c['warnings']))
    if descriptor.get('restoration'):
        validate_restoration_dataset(root, descriptor)
    return {'pages': len(actual), 'snapshot_id': descriptor['snapshot_id'], 'warnings': warnings, 'partial': bool(descriptor.get('restoration', {}).get('partial')), 'geometric_coverage': 1.0, 'ocr_text_coverage': 'unverified'}

def validate_restoration_dataset(root, descriptor):
    """Check every extension asset, including rows not selected for OCR."""
    index, report = restoration_index(root, descriptor)
    for page in descriptor['pages']:
        record = dataset_page(root, descriptor, page['source_sha256'], page['page'])
        for region in record['regions']:
            for variant in report['selected_variants']:
                row = index[page['source_sha256'], page['page'], region['id'], variant]
                if row['state'] != 'ok':
                    continue
                if variant == 'tadisr_text_mask':
                    dataset_load_image(root, row['image']).close()
                else:
                    variant_region_entry(root, descriptor, record, region, variant, index=index, report=report, allow_generated=True)
                for extra in row.get('extras', {}).values():
                    dataset_read(root, extra)
    return (index, report)

def package_image_dataset(directory, archive):
    """ZIP_STORED PNGs; include only this validated snapshot's referenced files."""
    import zipfile
    root, descriptor = dataset_open(directory)
    validate_image_dataset(directory)
    archive = Path(archive).resolve()
    if archive.exists():
        raise FileExistsError(f'Archive already exists: {archive}')
    references = {descriptor['manifest']['path']}
    for entry in descriptor['pages']:
        references.add(entry['record']['path'])
        record = json.loads(dataset_read(root, entry['record']))
        references.update((record[k]['path'] for k in ('source', 'acquisition', 'layout')))
        layout = json.loads(dataset_read(root, record['layout']))
        references.update((layout[k]['path'] for k in ('analysis_mask', 'preview')))
        references.update((v['path'] for r in record['regions'] for v in r['images'].values()))
    if descriptor.get('restoration'):
        index, report = validate_restoration_dataset(root, descriptor)
        references.update((descriptor['restoration']['report']['path'], report['manifest']['path']))
        for row in index.values():
            if row.get('image'):
                references.add(row['image']['path'])
            references.update((ref['path'] for ref in row.get('extras', {}).values()))
    archive.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as output:
        for name in ['dataset.json', f"snapshots/{descriptor['snapshot_id']}.json", *sorted(references)]:
            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
            info.external_attr = 33188 << 16
            data = dataset_json_bytes(descriptor) if name == 'dataset.json' or name.startswith('snapshots/') else dataset_path(root, name).read_bytes()
            output.writestr(info, data)
    return {'archive': str(archive), 'files': len(references) + 2, 'bytes': archive.stat().st_size, 'snapshot_id': descriptor['snapshot_id']}
from math import isclose, isfinite
OCR_IMAGE_VARIANTS = {name: (spec[0], spec[1]) for name, spec in VARIANT_SPECS.items()}
RESTORATION_FAILURE_STATES = {'failed', 'missing_assets', 'runtime_incompatible', 'inference_failed', 'timeout', 'resource_exceeded'}

class VariantUnavailable(ValueError):

    def __init__(self, states):
        self.states = states
        super().__init__('Prepared variant unavailable: ' + json.dumps(states, ensure_ascii=False))

def restoration_index(root, descriptor):
    """Validate extension identity and index rows; pixels are verified when consumed."""
    ext = descriptor.get('restoration')
    if not ext:
        return ({}, None)
    if ext.get('format') != 'vtm-image-restoration' or ext.get('version') != 1:
        raise ValueError('Unsupported restoration extension format/version')
    report = json.loads(dataset_read(root, ext['report']))
    if report.get('format') != ext['format'] or report.get('version') != 1:
        raise ValueError('Restoration report identity mismatch')
    canonical = {k: v for k, v in descriptor.items() if k not in {'restoration', 'snapshot_id'}}
    if dataset_hash(canonical) != report.get('canonical_snapshot'):
        raise ValueError('Restoration canonical snapshot mismatch')
    rows, counts, aliases = ({}, {}, {})
    allowed_aliases = {(p['book'], p['source_sha256'], p['page']) for p in descriptor['pages']}
    for line in dataset_read(root, report['manifest']).splitlines():
        row = json.loads(line)
        key = (row['source_sha256'], row['page_index'], row['region_id'], row['variant'])
        if row.get('page') != row['page_index'] + 1 or (row['book'], row['source_sha256'], row['page_index']) not in allowed_aliases:
            raise ValueError('Duplicate/invalid restoration region identity')
        if row['book'] in aliases.get(key, set()):
            raise ValueError('Duplicate restoration row for the same book alias')
        if key in rows and any((row.get(k) != rows[key].get(k) for k in ('state', 'image', 'canonical_image', 'source_image', 'source_box', 'crop_type', 'reading_order', 'operation', 'parameters', 'provenance'))):
            raise ValueError('Conflicting restoration results for identical source aliases; select one alias per source PDF')
        aliases.setdefault(key, set()).add(row['book'])
        rows.setdefault(key, row)
        state = row.get('state')
        if state not in {'ok', 'not_applicable', 'skipped_by_configuration'} | RESTORATION_FAILURE_STATES:
            raise ValueError('Invalid restoration variant state')
        counts[state] = counts.get(state, 0) + 1
    expected = set()
    for page in descriptor['pages']:
        rec = dataset_page(root, descriptor, page['source_sha256'], page['page'])
        for region in rec['regions']:
            for variant in report['selected_variants']:
                key = (page['source_sha256'], page['page'], region['id'], variant)
                expected.add(key)
                if key not in rows:
                    raise ValueError('Restoration manifest silently omits selected variant/region')
                row = rows[key]
                if page['book'] not in aliases[key] or row.get('source_image_hash') != rec['source']['sha256'] or row.get('source_image') != rec['source'] or (row.get('canonical_image') != region['images']['original']) or any((row.get(k) != region[k] for k in ('source_box', 'crop_type', 'reading_order'))):
                    raise ValueError('Restoration source/crop identity mismatch')
                expected_provenance = 'neural_restoration' if variant == 'tadisr_text_mask' else OCR_IMAGE_VARIANTS.get(variant, (None, None))[1]
                if row.get('provenance') != expected_provenance or expected_provenance is None:
                    raise ValueError('Restoration provenance mismatch')
                if variant == 'original' and row['state'] == 'ok' and any((row['image'].get(k) != region['images']['original'][k] for k in ('sha256', 'pixel_sha256', 'size'))):
                    raise ValueError('Restoration original row differs from canonical pixels')
    if set(rows) != expected or counts != report['states']:
        raise ValueError('Restoration manifest inventory mismatch')
    if any((counts.get(s, 0) for s in RESTORATION_FAILURE_STATES)) and (not report.get('partial')):
        raise ValueError('Failed restoration falsely labelled complete')
    if bool(ext.get('partial')) != bool(report.get('partial')):
        raise ValueError('Restoration extension partial status mismatch')
    return (rows, report)

def variant_region_entry(root, descriptor, record, region, variant, *, index=None, report=None, allow_generated=False):
    if variant not in OCR_IMAGE_VARIANTS:
        raise ValueError(f'Unsupported OCR variant: {variant}; diagnostic masks are not OCR images')
    if variant in {'difftsr', 'unihir_restoration'} and (not allow_generated):
        raise ValueError('Generated-content comparison requires explicit opt-in')
    if variant in region['images']:
        entry = region['images'][variant]
        image = dataset_load_image(root, entry)
        return (entry, {'state': 'ok', 'provenance': OCR_IMAGE_VARIANTS[variant][1]})
    if index is None:
        index, report = restoration_index(root, descriptor)
    row = index.get((record['source_sha256'], record['page'], region['id'], variant))
    if row is None:
        raise VariantUnavailable([{'region': region['id'], 'variant': variant, 'state': 'missing_variant'}])
    if row['state'] != 'ok':
        raise VariantUnavailable([{k: row.get(k) for k in ('region_id', 'variant', 'state', 'reason')}])
    if variant == 'degan_watermark_removal' and (not preparation_region_review(row['book'], row['page_index'], row['region_id'], report['configuration']).get('watermark_removal_approved')):
        raise ValueError('DE-GAN watermark-removal region approval is unverified')
    if variant == 'difftsr':
        review = report['configuration'].get('reviewed_regions', {}).get(f"{row['book']}:{row['page']}:{row['region_id']}", {})
        box = region['source_box']
        ratio = (box[2] - box[0]) / (box[3] - box[1])
        if not (review.get('upright_horizontal_single_line') and review.get('allow_simplified_text_prior') and (1 <= review.get('character_count', 0) <= 24) and (abs(ratio / 4 - 1) <= 0.05)):
            raise ValueError('DiffTSR applicability metadata/input contract unverified')
    image = dataset_load_image(root, row['image'])
    box = region['source_box']
    scale = OCR_IMAGE_VARIANTS[variant][0]
    if scale is not None and list(image.size) != [(box[2] - box[0]) * scale, (box[3] - box[1]) * scale]:
        raise ValueError('Restoration scale/dimensions mismatch')
    if variant == 'difftsr' and image.size != (512, 128):
        raise ValueError('DiffTSR processor canvas must be 512x128')
    mapping = [(box[2] - box[0]) / image.width, 0, 0, (box[3] - box[1]) / image.height, box[0], box[1]]
    affine = row['image'].get('image_to_source', [])
    if len(affine) != 6 or any((not isinstance(a, (int, float)) or not isfinite(a) or (not isclose(a, b, abs_tol=1e-09)) for a, b in zip(affine, mapping))):
        raise ValueError('Restoration coordinate mapping mismatch')
    for ref in row.get('extras', {}).values():
        dataset_read(root, ref)
    return ({**row['image'], 'operation': row.get('operation', {}), 'key': row['key'], 'provenance': row['provenance'], 'model_identity': row.get('model_identity'), 'coordinate_mapping_evidence': row.get('coordinate_mapping_evidence'), 'warnings': row.get('warnings', [])}, row)

def resolve_variant_page(root, descriptor, record, cfg, *, index=None, report=None):
    from copy import deepcopy
    record = deepcopy(record)
    kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
    states = []
    for region in record['regions']:
        if region['crop_type'] != kind:
            continue
        try:
            entry, _ = variant_region_entry(root, descriptor, record, region, cfg.image_variant, index=index, report=report, allow_generated=cfg.allow_generated_variants)
            region['images'][cfg.image_variant] = entry
        except VariantUnavailable as exc:
            states.extend(exc.states)
    if states:
        raise VariantUnavailable(states)
    return record

def ocr_input_asset_identity(entry):
    """Keep the existing pixel/config identity independent of new report annotations."""
    result = {k: v for k, v in entry.items() if k in {'sha256', 'pixel_sha256', 'size', 'mode', 'image_to_source', 'provenance', 'model_identity', 'operation', 'coordinate_mapping_evidence'}}
    if 'operation' in result:
        result['operation'] = {k: v for k, v in result['operation'].items() if k not in {'method_family', 'algorithm', 'perturbation_class', 'provenance', 'scale', 'resources', 'threshold_tie_epsilon'}}
    return result

def bind_restoration_variant(cfg, root, descriptor, source_sha, pages, *, allow_unavailable=False):
    """Do not put other selected variants into this variant's cache identity."""
    index, report = restoration_index(root, descriptor)
    states, identity = ({}, [])
    seen_pages = set()
    for page in descriptor['pages']:
        if page['source_sha256'] != source_sha:
            continue
        if page['page'] in seen_pages:
            continue
        seen_pages.add(page['page'])
        record = dataset_page(root, descriptor, source_sha, page['page'])
        layout = dataset_validate_page(root, record)
        if layout['reading_direction'] != cfg.reading_direction:
            raise ValueError('Dataset and OCR reading directions differ')
        try:
            resolved = resolve_variant_page(root, descriptor, record, cfg, index=index, report=report)
            kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
            assets = [{'region': r['id'], 'box': r['source_box'], 'order': r['reading_order'], 'image': r['images'][cfg.image_variant]} for r in resolved['regions'] if r['crop_type'] == kind]
            if not assets:
                raise ValueError('Prepared crop type contains no regions')
            for asset in assets:
                asset['image'] = ocr_input_asset_identity(asset['image'])
            identity.append({'page': page['page'], 'source': record['source'], 'assets': assets})
            states[page['page']] = {'state': 'ok'}
        except VariantUnavailable as exc:
            states[page['page']] = {'state': 'unavailable', 'regions': exc.states}
    selected = {p: states.get(p, {'state': 'missing_page'}) for p in pages}
    if not allow_unavailable and any((s['state'] != 'ok' for s in selected.values())):
        raise VariantUnavailable(selected)
    cfg.image_content_key = dataset_hash({'reader': 'restoration-ocr-v1', 'source_pdf': source_sha, 'variant': cfg.image_variant, 'inputs': identity})
    return selected

def plan_restoration_variants(root, base, config, *, applicability=None):
    applicability = applicability or preparation_applicability
    result = dict(schema='vtm-restoration-plan-v1', selected_regions=0, manifest_rows=0, planned_output_images=0, planned_output_pixels=0, by_variant={}, resource_warnings=[], storage_estimate_basis='PNG upper estimate from raw RGB plus row/deflate overhead; before deduplication', runtime_estimate='unmeasured; CPU crop timings are not full-page/GPU estimates')
    storage = 0
    for page in base['pages']:
        record = json.loads(dataset_read(root, page['record']))
        for region in record['regions']:
            selected = region['crop_type'] in config['crop_types'] and (not config.get('region_ids') or region['id'] in config['region_ids'])
            result['selected_regions'] += int(selected)
            w, h = region['images']['original']['size']
            for name in config['variants']:
                spec = variant_spec(name)
                item = result['by_variant'].setdefault(name, dict(planned_images=0, pixels=0, not_applicable=0, deselected=0, method=spec, parameters=classical_parameters(name, config.get('variant_options', {}).get(name, {})) if name in EXTRA_CLASSICAL else config.get('model_options', {}).get(name, config.get('variant_options', {}).get(name, {}))))
                result['manifest_rows'] += 1
                if not selected:
                    item['deselected'] += 1
                    continue
                state, reason = applicability('tadisr_x4' if name == 'tadisr_text_mask' else name, page['book'], page['page'], region, config)
                if state != 'ready':
                    item['not_applicable'] += 1
                    continue
                scale = spec['scale']
                ow, oh = (w * scale, h * scale) if scale else (512, 128)
                pixels = ow * oh
                item['planned_images'] += 1
                item['pixels'] += pixels
                result['planned_output_images'] += 1
                result['planned_output_pixels'] += pixels
                if name != 'original':
                    storage += (3 * ow + 1) * oh * 1.01 + 1024
                limits = classical_resource_metadata(name, w * h) if name in EXTRA_CLASSICAL else {}
                if pixels > config['max_output_pixels'] or w * h > limits.get('method_max_pixels', float('inf')):
                    result['resource_warnings'].append(dict(book=page['book'], page=page['page'] + 1, region=region['id'], variant=name, status='resource_exceeded', pixels=pixels, method_max_pixels=limits.get('method_max_pixels'), reason='explicit pixel limit; no automatic downscale'))
    result['additional_storage_estimate_bytes'] = int(storage)
    result['deduplication'] = 'content-addressed PNG storage; identical outputs can share objects, method rows stay distinct'
    return result



@dataclass
class ImageDatasetConfig:
    dpi: int = 200
    acquisition: str = 'auto'
    max_source_pixels: int = 100000000
    analysis_max_side: int = 1200
    reading_direction: str = 'vertical_rtl'
    columns_per_group: int = 3
    max_columns: int = 64
    column_cuts: list[float] | None = None
    page_column_cuts: dict = field(default_factory=dict)
    ink_threshold: int = 40
    gap_ink_fraction: float = 0.006
    min_gap_fraction: float = 0.006
    variants: list[str] = field(default_factory=lambda: ['original'])
    variant_options: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.dpi <= 0 or self.max_source_pixels < 1 or (not 32 <= self.analysis_max_side <= 1600) or (self.columns_per_group < 1) or (self.max_columns < 1):
            raise ValueError('Invalid preparation dimensions/group limits')
        if self.acquisition not in {'auto', 'render'}:
            raise ValueError('acquisition must be auto or render')
        if self.reading_direction not in {'vertical_rtl', 'horizontal_ltr'}:
            raise ValueError('Invalid reading_direction')
        if not 0 < self.ink_threshold < 255 or not 0 <= self.gap_ink_fraction < 1 or (not 0 < self.min_gap_fraction < 0.5):
            raise ValueError('Invalid projection thresholds')
        if not isinstance(self.variants, list) or 'original' not in self.variants or len(set(self.variants)) != len(self.variants) or set(self.variants) - {'original'}:
            raise ValueError('variants require original and supported classical names')
        if self.variant_options:
            raise ValueError('Source baseline has no transformation options')
        dataset_validate_cuts(self.column_cuts)
        for page, cuts in self.page_column_cuts.items():
            if not str(page).isdigit() or int(page) < 1:
                raise ValueError('page_column_cuts uses 1-based page keys')
            dataset_validate_cuts(cuts)

def dataset_variant(image, name, options):
    if name != 'original':
        raise ValueError(f'Removed image transformation: {name}; use original')
    return (image, {'method': 'identity', 'parameters': {}, 'warnings': []})


Dùng worker riêng cho TADiSR, DiffTSR và UniHIR; ảnh gốc không qua restoration.
Giữ provenance và OCR gốc. Output UniHIR là generative_reconstruction, cần kiểm tra với ảnh nguồn.


In [ ]:
UPSTREAM_REVISIONS = {'TADiSR': '9d910f8f67224fae2f57e3d63d68396eebc0b671', 'DiffTSR': 'f9839633f5dbcd7cd80ed0a7609edd6ff5d3b5f5', 'UniHIR': 'f67fa3f36063bf79b957e617f915b584da543953'}
# Lossless upstream files, including licenses; standard library only.
import base64 as _vendor_base64
import hashlib as _vendor_hashlib
import json as _vendor_json
import zlib as _vendor_zlib
_VENDOR_SOURCES_SHA256 = 'dababeb040c596794eabb389d39eb0b99aa4c56bc8f30177da177ba7937762d6'
_VENDOR_SOURCES_B85 = 'c-riJ3wzr(mgrx>IN#n#twdXjlcsLnZym=;eJ2mE?M`>D?T@A<N@gRGDoOd#H2?jc2LJ*fD9Mi7>FlPv6N@Bp02~|~oEHv0t{v~ZOpXrMwmz=C+27rJceJ;*wf1)ZSiDIFaazQu>1n!~&2IB#d|8Utpe_0v{iou;l4+8T#gB6lr9&~u(lSpj=4F-_Sm7Yfr%6#HSt^o3T*i5PaVy4ol$P<ZBSv`~i)<tYmr*{BJEF`0$gP;gc>xu&i!w@51Req0+3jfxMaxToQ)Hv^I?7{eGAfE}kVF7b470&}8mDDcVuPb(5*MOXUdG~d?MPNRU29W2!#J9p0&sz!)kkril$Y7O6nR|0Fa`*wBa(D5nGdla>hlD~EE~b<gb^4nfSngG4D4)2OtWD!!oM+%Z8pD{B*kS%43mOU1TPDGN!ab+Sk|*#6!8Q)1|SmXJPo+&9~Hvh&k(JpL{x!qt}nBxJ(dJ|G@9orG#XQ-VFrk!Hh+l+CBDJ(qiizCu5olQcf$k+RcztZ9mB`yBD;!dJUr`ZRzi2V7dR2KYI^0T;xd{{#6>Kz1}y`QBXdYO_Mj+%1d?bXW?4>6x<d`93qS6OqxY|lf8IIV6Z=Qv;PCxF_FwM3g!$Y#f)}T29r5%2@sIC+Iu=moaOd6eKgIi3V&~mI#ozbez3hm+|2{a}J312Y4^Pwmw+C<b_u%FJyWKZGz1)BITD*XY@7@E6?*pL&+~fBG+mx{O_l^+S+r7ixAK`K5#r~W9<9~Ke(^vb)?-1&%_lIIf9PAt(@9+NfX6H~G{B(Hm{%8+6@Dkv@+kf}!5SrP0yZ7!mfM(&H*!u@Oh@&5O-n_xKPSc&AU;u~MMX~$-;Gc*4uYWuiKfZtSat~g<*n__9ym+(6ZNa#9-|Xza?TD8<Z+BkrQLXm?28M^_xP#*7ANTMbwzvcTzk9s@{vD2F_x-!$LwM}KKo5^~-Jkc5_BvwcaQ_H#^6K#YTNoqa6so)@C{Xd;9)m%A3u|Ja2tNOGw5K78mwP*J0N4>$rctWWFt2N!wZpxgmv8ri=@3}VBf*SkOB}~HWp{^&_Yjt6_vb9148{I58UtO-f!4Z*agj~twD7KyNW4r&qd8IE+iVz5Xnl(Rn<Z%}j^c5}30}qn_|6>Z`x%g9GETQo*I?n5aelh?!)YqS_mQ~F;}N{REX!H3wZ0zZH_26y<>U3}qFC=g-FOykY(CrUL52VGsMmkq`}Wy);vj;4eZL<4Ab$5l8AG4u7r`K#t|OxyT&?qPHI(Pyd?$9Hdp;nKu99L7m;in>p8$)Ibv8FQzxzLro;}@o^6huxA5;V|Mnx8N<#IkALl;KTAcp?H<FCZKpZ1=}K6MF=s+n(}Y&`jnF<XfL6HRB|iI-6sVGDr9_1O=L(5bhD4(uQ4B!l0tB4DQfOm3r8yqN<d`se&Ig4h3qBLAI4*)K^fUgP`!f{G~=g7P$ui<}@HCsFzfFq5Cs<O=38DQ}4nT_sl;bn*h2J)m(qou|owXMHyd#G7)syszIU)3GS>0lbIJBrc2fQ8J$AaTw);%cKNm2@k<9v+?N~kl3)igo58b`ww`ECMEo4<O#F10KS=DCGoXtLJs14;C(;rSOtOaA>Btr_k%nuif&Yv&_5glj_?;E=)xrEp(N<w`*j3Z4N-OlSaSp$ja(hwi>LrbJj$bKe4XVVfam98^9FV~Agya!WC<;@f)-h>7a446mr*I=n;Gm<NjZV7InS`;umuAKW9m6Dwi5Q*H13x36c{{maow~iVXrTsJ=g^w!}@c_M=SPCcDPfG6k6foF({)?SF!3S?fbaBO(tD_gFO+`sQ3_wW89gdt1KB7;{EPn_aeJNyv}AMN<dBS8tjC;15S~`Pg5YqAqosgs~rK;2m4tx=?I`Ks5r-;SUJh_cs!p(IX4LenE@hZKsbTgHS>591IE=B$m5tHqj+41zh@JWjkr-B2N3G`A3!Rb-DfYvRg?prmpF<T<t_dqizf^!2lPyV7;twbv0?xcR6O`FBSOK+-MxhQC?r-c;z2Y=`7MEYLsF2?E+d%6GKb;DPK`0lZO)Vy;qc2_8r!2s;%y8P&``Wr)LoGrLMty(pc`cK*(63;a5@9rq$SG}UGddd$BmR8eDxL6!F)CZS#Aj#2~m;IB)Wx_ASW{c>0%a<WC0`t&Eqs6wxPuW<SdYHK%ha<2geCBiLyt)o%_EF)*19}M+}h_1IFTLs=lDmRDjG`z>73XaGCvMXAe6#iX&X#1<f%mo8f#QM}_ONqGDeCNV!0Huvq{R&jPVa*x|MiT7v8<B7qb|EQCuVfD&`&BeaqjF49NJUuX(p1;c#Z!kUdHu^jBHW9UY43ky6h5*pttT*4rl!=GVA^wq-Rk)UJ@$Q<%}Tvj|CIHNe<fMNIutzLq_*!>kCAfvvB1|P<GHcyAecQ}_U)<fH=m?k%b7nr4|8y^rB_X<`Y%tguL0#P2uk1E3~O0ad34|exN0oxKRTD|mWm2nHm0)fd*E6E@NI5(&!RAiOF)RplJxp04DL#i;-3tS7Q=?qH4s(*8^e{j?xR>A}$;vq`(CZ&Kt2GTERG}%Bg3dC7=JWpVio~F)}Okj3puVGQyJNSKeswW{LA6N&d0}D2)S5#O$*%CZ45v_)*D$bi?1WK2piT3Sc04xws0`XsW0~JVdPf-%EqzvW;7lxwf?1Nl3%>Q#D93+^E+G#<LBA8X&18!^k;9U{8c_##JE<#|QMMF_Q+eIVRHWy(H>vyrfjjkD>3YacnMxhn8W?%`({191(*%XAPNnFw!z!)q9V~4+(%<~$WfcQv&HEHkIQVONyNeeuWQCmYv0=Ag?o+iZ%*<ZQAm8U5#!f2c(BPo~#;vCn~&bivp;E8w^lwpQ77|da0x4^cd$vk3)f(m5LJe_F4Vc@6f@8U(A4lbt=2sposgQItc@aO1g|1JEXB@BOG?Z1S7sdW^Re;3_uSJ1y(i~nF*WSQOLJ$ToD8f<<Ge>Z|}o;;(!y{CQn`=lTAp3w8NXML_D8C4fXB3eX}@vw~`HiL~m6@A`kP(9#30OEPD(c|~e2?TGugv|e!Wjs)RgaKEZ`S0Rbu~pbVBOH42TMU18wPauqSkwp4J(j9~lv92eCx5?=68X5!f6iJBDn-30j+OSB-vwwo;_VBB2puXLVKk~G>lc&kVtopuTrZ<xQsm(WY9bt!@cU{1*~Z33F#9lR3u_=+y27d0#pyfw4SI=sogsKNcE?jzkLhc@5B)B_0g8Co(ZPa18EZ4dRXm8p-ZxLx0Do5t6<G%?`IwAdA_W1z0EzV$UKB9>MMuiJlL<?RAgb_EPN4qDc_lR#&@!x0V9CKRz=zJyT8-b@f!G;Fvl4}$s?2GMU|}H11&WR_Nbyoo$mV4Mo1l1|l;4r~*=35AhOdHbhDkanu`HU+CR*6shaEGTOd?iAPSf-AbEtoboFE-WAbfy`6^kw~N<x^J-GXqMZa;ygB)Ye%7)ni7SddE*Z1H}EK%>bPTI`4k9TzYwm-9#*+#c+MR06wBPK*v@H?o%v;~9$F=^&w<dzQ?w=dgW{m@<ahizYz=C2qR&9HeVG|Fhd!5`bnD1*L2uh==oFkggXA&K>vX^}$b~wfob{9nk|G(yqFdjeu>UID2q=oPpjbt}kQM1h5_1i-gex=)>>y?^QzLO}AU`&oH|NbvzmdfGsRa29E2?$@*a4>-TJkUED!_orA=M+jkL^APL|sV4q(BYCv^4_^F$vFxhc>m4GUN<vO@gOn}~L0IJ`pvLG>{B(`PE=0lEQHrE~e2{3rx?V^Z>HrwaVHa9lTSr9xonmorncMbyDB#AC2xT4ZDCMm6uN*Hc?Al7F2Eo^n?E<(;LH4g`n&r@s>Byrp;3)6aBl12v#_3JlpVTd2%6qaaS(bcUuFHlxV2k|V42l2U<YPA}eq$3c6p@1OTsT392E0lR*Cq-F{ZDS}3T+EXR;~j9LKpj^KRsSV%pz}cdj51h(`)x_QWf-$TAC{1a#wb%8Z~@p4+ZzGwc^l{7RTGF6L1e=&z#GJ<R)<HxjYKv<!NedJP~d<uNmIjMq7-RZ|1ae<OB<BJhM~eDK`HBt<Vg`}dLU2Xl2?SnAT`B3sxVps+d}4(XE&fj5&L@56R%(35+kkx%2S@qfEOx|A=V73CXgYJDfdV*ZFm7xH=CE;A?T#nF}VozCc=wek8c^n7o$9zipH%2Dfma;X2dJpXAt-?Dp#QPC8JxBpb=n%A~W(CQxmC=4+ND2HSn-v?l`{FQxsvlUBl2|?hs2ypz2HehKVP3wRELMji_<EM574x4)&1!;aTJ2T~)lx#d(%?d7B}71njgqOhHnhb}E(IE)dOh20HfiA|4J|W~RjmGWV^t*U$nOGM&{?4x&BKYaBA5Py@0CYV4dYHR&=;akcO3MRXMp6?o!2MVMxT(Kwh@3>}wNmM1_(u-i!K&sb_`Q?u3wh^??b#6Btqa_vDG<@V9OHLk6A0B)#cCq;Q!YTIb9VJp*jTmu+Q$z}+)ZXqni_tZ8*(4oDGIm0C8W6<abhDMESxDqKOAbdWYv+<6sCNPo|kq0m@lEEDQ*PYMUyg(DVdDCHo1>j&17dV}?moT#9VdpeOo9`rIJ7Y=CN0eyL5D6S9n(}^9ZG?G(cCPax4`%839I*j&foe@ic1(O}vn_{g2q`9sBs73^z~|+9nXTVu^E{m7ArBA0IdnptN$Pj&@eK|2!PeC(<jMrut?2!3l&j&8T}vP<L9B~!o;}Ag@??On_RJCqKxRQP3m9Qw8D7&k9u}r+5=fk^4AXc@)O8s#K|cqwC)pD1etDNTD(+fRD^5`N2B{4hJK3wvefm3+>mtzp00_E5CrGov^AHLPn<%crcyJW68z!R;mCUYTg`M*lL5{h^q2g{0h-1Q%s~?Gn`d5+>tVqUFq(H#o1sikG3k%C<oJY5Wxem&dFwhI~@j@f0byJxkYV5F7YPvS8B&BZFScf+6NjjCL1G%Dcknzn7I6Ce9uzblhipXh}3LQDLOHHse3VO$20BfAK`$;rUL3~sTgo^RMnqtlM!e2%tkrdUsA=R#;(YR!rDDU+l>R4fxhgf<UX@{Rp+B~Nru2lXhcYgUNGzE&gAwn+QyW#^WP8O+3NX=T6kxYTXO<=QhORh^p2z?CoY`Or@{If!REM`^zH<`1L2cx+YEULq5H+vh+*w}M`orXX29KOZEwc_C8IkV+;OJqAo;}583MKp>{{}3#ySv=6*MQa1D#C;2HN!|bn^F1?GcH?yP{zt58XlN)F1pQh9IbgSy?ceGm&s0u2c4h3ZzZW~8k6XZ`*+&YD65`$sc3a9VM6zUnSAH7HIcq@!MgzKE8W$3f4k}px$If1p1J+T$Y-~vexPI<h<<Q{Vk;Z=v3_`(I+GyC(pZpeaGz<fDclvWO>!ssKEWMt)Z}w{#w-^1nvQ=Rtj?qeUY8VGl9B=VzESE_Rlpf{f?fPt%4RV5nmRe`%^McH?uv?<Et03B4@#_>c8h{dmATlsrS+aPA*aeCJ0r_zUc6k~hn!dVd`eM~bg#sw$)sV)8GHS^-nr&eMt+-Nr1saz|Hp9Lq+pjEa%4{nm=cv%m;6s{SPtbKln^<MR8^%6qmY$^Ez{Y&+{D~2ech-_|dTrQ~4^>$yi}SXNyDj<+kosBlAQv7=*lrw{HpgWn)Wy-0T1o!}O2-1J0YqbvSvu_ZoG?b^qLUD)QF2h{Xt?T<>lyW?Lb~Q10wVJS>98l-wosW*jqn-6lDa-rCEFl(pB(JHI$m037LUqyN3)!c>^4u~!9f>jJCNQF^74?7bS}L_=gI-Xj@%(@<T!ZPb&-Rnw@CKn5K$7+u8a{>Voh=Za@1-1pM=Fp{PA+$-{?O>*{XvZN12q9c>AN_B3~MxK|xTJH`8!+E{GZVM6E@0&?WNij~JjMV|9o_V2q9m*ufVZsY5|K_=v$eGHwSSDPBhf@96q6fQK9U1+)+l3%5V+46mYez-WCJ&vOtJK^x2RDR=LH)hg`B(QQ%2Q`ylg80!@9^5i)`&j1m{_Qz+vr~Q7n3x77B@P}KV^6ig^(od)9C%LPxb=Dde_S)9kN33|d#=QtBtO(|7ZA<hz{ADTwqPD~dz7XOg|G{s`5Z|A!H5d=pI#p3=_2BZVpjD!>CUGH;+rcWYIFZ?fX0^<23V0+9>6_<IH=jP){PwBwvAB#-V<M>E4C2A_;l{VUZ=P&C-FWiN;366gE_(2PqvySiXZ>h6>^<u}xp?}lH+nk!ChiY=@zZa=9mN|^@M*kZwORlj49p#X+gL?HTDPW<Bsybc+{CBOviV`Dp^MI<=DFcQ3umV8<61s3en*vmR(jt)eg5p(=F`oFS=qdZ;?eVq-Y6PA>3uthM&AsdZ9E%1?>~L=?Z%U*gJ+|`x0}zO44(H!-)=tJ=so%7o58cq=e;NWr@i=_(KoBi3h<y9SFIdi2$E3vCd;D`C5Z6HI4Y;nEX0VB@TeEQ!k~u`6xI-UsAsaw5zE=aKWDT|VTHOTbqWfV&L8bRfCUN?9B>CTYp@QlZ0(~YFUl>$<5D>V#&OvNIOw%rpI|p9x9g^+^uu;9=m#68soqeOP5U%8HjfZC6#UTt20x_p>FkzjKEp@cSny+S6Q2$ebe+T3jbH<xQT#4pU(6WvX7Ci=nPz^1izSU1Cn&lZY(A&AzYeGP^o-g9Y^`E7L=66IeGbCt1;~JLdL?~huobQ0{~hFU7q_jsG;DHQOrt@QWIPb>q^Vsk&wzp9{zltj7>?#Bz=ff7f54WIX1x87yrbUcW9T_Wr16X5wvZrDLz^TQsy^yIWU1Rbcz?8i{QmHt;o<xD$6}klw4gy)ZDH6BWPD(&9UxACj!t?&Udafh%9c~Rjj`&u3qi91Hn_z^sU8Cm72~{YZFIb~+o!an591M8DqC&Q{Xx9T(pZ9_q2<s;HOycqA5;9x0exu=W1d35%61qIvjGgitQr7c4I^2nb-HF6(=g&)nU}YZ^EoJBm+@q_eY*Arsatrq4G<7=kn=#Mg!x+15Od>NBTH%x<Ew-=JdiO)(VQY>2J>OGxH-v#kP9i@+h9#X!!k;8>0MwuC^QT@;z!UufgEw^6lI5E&M17ej8GZvh%y2d9mtmtk?ck(2pW6ukHn!E0E^C}j!+z_0)KfsBd;985VOk+B;+m$V{G@IMmNcHK9xl};z|Dlij}}9#p|3D*p7gX=aKmFrLO^CuE^|A|3hgyn->xk)hs>$WzI5krt6p7Ck*_XfudUYphAHWiu3A>$}rZlRng%XuBu8~;(6uyN#ilaDlM8Q*uy6CJf{u=&6RZar6}MJ$grSiN%n#Y@OAY-QOPi(>j-s(DF4u*;3*28?1*1xG0YqY%)o_Fcji&y)B=#4Ors#Rb&F}10jF5AhXrFPU>(ZQ(H|UN(V``cC0L55Jvz{K0L6T?s}zv8my+;&Aj_!wF~5dJ78Gt_KnR+Hwz2V`k|%IDxN`VC|884=1I6UYEkF#pMTL19IyA&UO-^XN<+Uui6>!3k&0wL}-i9d+Lu6iI2>U&^$h?ep7-`1{L(t{RFy#G&dA;YTMs*0oe^SHUB?8kHB{8kv98!o`M;WP1!364uA#C6{0+7bn+)BuQK<KMiTsy;>Qn;tur)#iV!*;r64tH=FmE*}Y)Sh`Bx9lP+40^xH3jH&W<*>?I^83iwZxZB3D$1mThZbazXerBEGJvXfBwrrjOQ{iMxoW|X5A+6^$H#2yJXYQIln!7bCc4ffBe|vndnj%1M<q%^<6B=vi40;RAweO(w*tkKJ|xdsN4!80jb2H=z-h{!k)s3MJ4tQ0dn)>B74c-$VeOSnv|G6Ewznz{^9W5L<S4HOslK6tPE#WH?gB-ERt2nWLawS|n?W(-g$N|!$Gn7tAh!*WYWZ{DUciR3PTLA~Pwc-eoOaR>IN-3@!Z2=V%W6Bgf{ptkL~~NB?bN%-iZB`atMRv5m3Tpw*k5H94I8V>(+^G+iSOnyAb+vHf)E}Ond?zj;PMOvI9##`s1R&X=jHwV?**9we*b|N_V>S|cwRk0KN51)Rj;xOjF2T-#g}KsI6V!Vyxlqa`x)0g8JiD<@la$l{t@GkGx4>r$2GEYTKo9PDM2Ak;lgs{O(c3kN!2R3W$*o-Hc#l!nVL>Xf1YZ-j+~fv5C*;WRF*}=*W04!46u%*FuuVMX4)u_1OI-y7NC`?b&|&cNW{@BZc)LuJ#-Jc&ol)g8&Nz#vXO$gM@R0UPR;FC(WHn?)&d)&x1@t(U}I%S$b!yq*xzu~oSfP3a0sNa(h%Zcgw9kDa9Yc;14C##t;>!Yo%o)i2c+A!eC4TU0V?w8uw}1D=Gr?2@w%c_oK%3fEU;(ob{%jTM6($IJ+b@1Pxz&M=Ba0)&?sw{bcZ>K)0P(>_ZVAkIC%_GW-7|+Z4BhG<t?<h8H*yBpd5Mxe!T_%WVqQ76#|L6uSk_d(cuJm@!3|5>4J;_VB6>$0GQQh53845Xj_zyeelq(vDYAjEA7?cty4m|6Jx_M$iG5rg%P$rON^9T)fj2SFczZ;GKh<d`DjFfHOh`1YkgVlq1>v`PsKHxIGlfZag$9;2}2;hcF`=y#WmahntCKIkEnt5GGpfCc1;hJRxPRJ$;{NAf8Pl2HZnE4@r`GKWuMhNGp>2EVUC+7yGFcxTw5+JhO)c|gDk}}o06TD8Hz1NKTH<F3JKZ`um;A=GRd@1?4*1JKt4lhHbC<ji8p~zicBM|Eu=D=k1U*HFUui(9B2F0%+?2=3~}W2T@IdTsy_H?>?s!!Ej(g4jwJ#tjJSyK;DKS>Va=IC^wK@OJo7hVa-NPUDtAnY_6)<O)IPG|Z{Hq>7w`L^S`qGBj7w9ok<r{*yJ>m!F@=1Vktrz&Y2qaCovu}{rN=<4pVHv>Nli2>AQa@M0i2-5i!YL4!E&lT7)w!v{EnQ;_=-xMnUv!wVO4MNs3oa~A)emYK3$u{Bhs-4lQ;rha28s+VKIwfwQj4Oty%c$c(~+OUwybnn}tcIEMdYhX%8@w%(eupT-9wTZD7<cRmI$In<{Cms?c64$-8E^?}6}sz?rjKQ1n<U9{Y>O4aH#&e8ri-tJ1E1;l)36-}~OEi>8}aJJ3X7k%0NIgbWsa?D)4yQJKA5y79hU1V+=7N$IGC%Glecjo}Yt=t_RGfLGJ$j7*G9*FMtkPbli4w<1z1poD=k&_}&QJ~c71rY3X;x!Qlu@(gCaysfZ>ihs3_cy?~hK);8gKU8bd0}HA3!C|TWy}jihx;fo?A+;Ak#wEC7FV~C3oz$T2*hyv%i#ll-S{A{r=3T4)_Cg>yH4suTIwVL1HwLszxkLtKP5$BU+maANpAeF?w?snJnvf<OrDN{7VKQoeazF;01A@x|(}{;F)>4|q>(U_Av-UN|EWlcYZ=><!O;_iNe#X?yYc0+b{tIMN|EPQ&sY{dLzgC=F+;xk(v_R(70&o$Ac)BPZ-$5nuH*YIAo=L{gU|v=jAa|>@e|)PQupJ#h7|!x|l-z8i-?)y>>iH<TiYrtRWQw!0c7uXX|Ked2)7fhXNjpIEuGP@$nP{B@7S4T!mSal9qRPQ+USB43Mw37|oeLZ@m_1zC3arCSdPctTwYZZC^<Bt{IJV(X#VR7mp~@IXG1;8#&l<DjVh;TCV@ys%gm}J1Q3Qp~ghxkq2wc!SBOoJGz6QWc%SNO-9!oM+70K>dAGb>^j%X{@h7((7a7+X3g|;lKH+FL->L`Q|ww|=*bfF_2=NFILxF?K~)B%pUp=iAUZ1@`FaWu4z{S{mj<muY=1?+kVl+HxC1`2)|(4UrW#P%Zanl-`JWIv`P(?ns;mGxB{#q9lAdkNYd-}X(ef_7bvw?Lb{qR7{<pn&cWUFFeOZC7X^rU>x}<Hp&M4qw}Kn;@1XoftYm4ii@S&{-lCLggsoN5e@1#KSE)?wAEg7@?o#BbNGvE|xah3td%)O^+zUs{VgE#lRzjH05xrmi#c(s119Os$h2XYX@H4R@8ucbosJEEz$n#JjR80nPBdMTjI`)p>2gxM_rI5QA{^NfYg8Lt8M$!3cK^3q$HR?oK0ENfrqCvV)D{qxf2W}r580WfW{IoLDRc-#!i1lAhd;`sUgGIR%{PTdpL2`p!)SSGDMtB_=P_4U^t)7ik56mLOf$ZAuYEzQFJOOmtHYQ66t?&I;-$uS@gY&!yZ(Jx{dS+t54AyQ#q|s*=N)0e}}es39|W_=wcxpAHQ&%l$KY-lrk>R86suda5kA2trHC&S11(?We_1yBE{bT$$`u{NsxenBa)P_g0);aCul{8?AMV~TXnadl#;OVaKkXwo9}yDI=10N7eH4bE&>^Y)8xx=H`EuDo<JTlrjBx04ye2_;2jUPP^t;c{DphIemkOUx$=SRan*s3k`k6((19jA!<a^;3~@IYC<iy%qEsMu#EZSd;|ek-wli}gabVS!xdC+rQ;)8;CC*QF-yEHZ{}v}ldk1IdW?6SAg)Y6rrNs+fYoQQl%oNC08rxnu4!>ff`!8j}V3OicfU;%D>x2{dlHwIKwb!%STPD#dVu2B3QWh~W`eIEZVrC^Vu4;QrkBnCh0zGR5!TJ~vT7gXZh+cQ0*KL+S=-39HWuN7kH`=9uW>R&91ij{M*}~ng<OFRy^}ja1){RgS&C&qm*<Ebd=F`^}gs;8Aj50Qy`FBgzD?=-sfjH-37mjO+BZ4H$+2X*b6O4O0AvdTQF?8mH44pZ!rqN9Z`okC_f;79qIhTyaDX#64GM;J64RE5cVe3us@_qR3{c(7(^Rf;Mwa4jvQYN4<gxP4NK1g0|8{v(b=<Ri?5~D=>$^sq<-By@YJhFvp1mjV;&Y8J!N1x9qH+UYybkSjJMxLNpgd6oF^D@w;(_X?~oY|}|?3bJ*9o5pT3dN{8Y4RlO2erDWXtkT1w}WPKP%*e_gd=&h53wIh{(>k`TnSUxN`5b^z|Idyh=HZM?cjeuBNHt1Ic*+vvN8HV;eD9o9glp?)6&B{gS%2Br;O*L|NO~fe$|%Y67HM#?_Td69TR1!6MUG-wJ0lQYR>d3FyHR{clc)S-Rt8Yw@kPO?X+a9Z+jJR)!Nz;FEip|Fh;+Dp>oZrX+kdpA7WJ0tnB8dA7aagE$+@^Tpf?M5Fni{Box6bZ8uPxZzBL&Sq=nVZHqzpct=Z7Vq5I9WegcTCpZOCn?*wlA{%oQ1l1mFSyz}(lruC0K2+lQ%0k4FEP?ESyhd}RPgu;9jtK!F%>m3~5-G(lMw`R<-u+|$<^B!`Jo!-Hh9J~2kwG+*xq$1<p8v2dz6qY5i`GA~Ng25d%Fh(*U<@1`#dI*0)P*XG3IifRPN)B+%(;p#w+N$9p2bPR0?34B_<+k=ABi^@SxO=&*OIUbf6SvveXLcnW7IZB@ZkyOo7(TD_$Y73HFxgG#+ezi82>7{9n{AtC>yB7xsy?i)nL~ec*a%}TV1IkqqrfOaF*j5-Pek=jOzd4tMIpNvC&@C!<GqRecc124H8eUExz)6GRHy*U-0`yjG_%LtL-`(|F%eGpdooshPA3*+R?)MnB){s`o}I219FK+;(_h73@ohI{rc(<)50y<p}RU1ltoT@XMn7V-kb?{!w2ryuoHrSqy~F*OF4v8+ijQJ>a*muga`Lz{sR-~z`vK!ax-R^4Q_SJZXjlj9)7(-Jx>pv3Ak7P>JzX^?`n}F3)*PrYTA@&dT(ou+jHeH@7nFzXE%(z68f;pL6&m{!j-e1PzHxPA8VviXnP_fM0J2DDp0KNRG$_da`YS`*-39Zm<ML2k5l}o2GxS{zs0Gt_^5Cs`$&wW3m(r7Eb*K;58<SkgC@vgd!!C_8fdI=z!Hr-l_5i!_{g;JhE6Zj5dwb3WbZS0G1Q75%Ji!C1dkh_DkKj(VXjpuT%v1+O9w%4W(RqP2&!_;SAa80V{umZ;4%8S3ZpMP-21Pe_7C@7hOZCb|8#Iv1@xGw@<CWTT`Mp#3AGs-2V&%zfa=>-6!W`ZbUqq1Ju0B;IJpaQp2npg7S{b#oiGjY*h`Dik90y~mn6Z%2O?2pFD-+yBqBNsE%AIah-anPrw@CS;5I54`zH3H1Z1v5K3h$tY9}z&nQY2YB6A&r&?VPaoth|p2p{LJQc;3(C2s*p3Y8QuunsA*P)7YzwD3rr%cM*p=A1dl-SP_0`+}x$k-$u09-CzFf$yIH35`MA#Dh5}O53Llf5m*psmvABgtKP=)yT1ND#8(T8SgF-AL4jcINXB0Srg8)PT5zK{%21FO)*($@Km~<&<@`85aOY9ItdIEIGZC)nBy}~-^$rLzMb)`+_wb@PD&d|K}ptt1Gfl@W;pE{N&Ewq4kn1v>Dt~6&NU{S!B&WERTJK!EVKzaW<PS!I>N@lGCqU&sXB_R4B;hZnp1R-nKo%j5ToJhTXn*B-1i^iTQ<wd8r4womAm>+%te3J4w3@1|3N_n)Aj}aRK=G8RttY6u&S_ykkhq!iW!4a=E97Y`sv1&X3c_Ag>@|MOu>`Uyv)$S3b<OhLy#gAnY$%k;&AX2C#aG+oLCX%^j&hH7gZ7`l|Rf1>Zky61~z$mpN`T6@y!gtqjRlg9OJiW)Ts{`B^Ph2PNJ{i^}DQmg%?I>O0C9B&_?)iOMG;oelh~D)Oup>>lAFOU8!XH-}Vx#ECrj7T?*|OBQ-^=Kl1dDyIy@T$V=9jLbpq5OBU%*spfPQS$tGLKLtP!877s(E_8HGUdlJgL>U-ZQQP!H1<x4%OJhubGsc93qsI*&R3D5W1p=zhEI^kJ(E>s_s6WcmpYSw@Nd<W2PsXWQKPu>-%%1wC595Fx+S_0``Y?K;uwzOTWN$&vf>TQlN2qO|t#Ftr)$lS?@2KA<ocf3&)@iAktBS@069<hNoKyYBqdkF-XZ6ErD!G33)-H}p67{t{8hX8~<3~;}$@(;%mA6)R>-0>NZ^&GR%cK&wEQ~Lb$v%FvyKhW{QL>s2B20>_O>Y4(P<q6A6$P9lb<mH*x_D~wBw`SpOlD|~fRUoJv8EZ{awRu;dooq7LVMJvWVvvpvQ)z;aX@@juRy`2MX8s<3y=;z;ND=4Spr%%-sbzV#4wB5@FNE<@ai#{pY}42hSj*^NsTJB&|Ec-@^UjVC~0~DZm`IGrXTAq!-GtJ_fz*>T;6BCL*zu;jt;?!zx_r^NxV~>u1%Bb<6<H*8d}J->J)^i$VvrtHAI-&b|I#cvmpgEXQgZ8YDU)`?&b+^^VdNZr$}0{A}M6K;*g=~;8wql`G4H)zhiC-v_Wkv`ZYe0LcHxtE;uD1<`g9bK6hNEyv={C7GNvb#wlQAcif$Go0R4jKNx9Ul*Gu%^+}Kk3H^#@?kG4K{%0NX%NaZE6!$#EI_oN>n<NbpLxW?=(Q0gLd<<Ak3VAQs2E#sjPG#9uW_&w|t4?$yZ&BbJKhIuJ+DhZZ6zhDcw6c*;1XRz$lx<0LqBnT`^t9TGVfjlnM<>bw#!ym@N7tEfZz=cRN7@v8KeZO;R3l5I(A}Ekrp30u<)5m}f)C8+S^!l2)&eV(Hsy(B{If~e;K5rNjy{|9w(TydIbFnq8~JZ|H%(m0j`q^y9#C8XqFXn94~cAHmL!q_%-fdD%a-e#Qe*L!76>f1ZnkabeO4WK8x<e+tLrf(Jl>0*ZEn2v_hVVDmOY_lR0%7Y!<{N-v&vjk8K%v=*OgtGv}cw5smr%W&2uu#R@s~}?X|jSPm7lN?nHU3#RFNYSF%v+oJW-Z5KV{aOqM^`f1}D$5~xMVp8TnumfN-^3U&!wl0vFnT3fP{HYYaU*778pA9*UOosZMkRof^{zgiqEp0O>bPwj^Nv1UDSXH%^yWf`4jS1d81Q57$$Ft-`Lgb_o4>$V{7S76%Vl!#(3ZJYlvz<{mti_O4m`%*VGF-)Prr)%E{mC!{3lp;y{4w=^6^lBP0+qEJF_txpvNLphQRD@y5Ymx*i(T)q(D5Gs<x1$tovM4_}+4E$Vm`6~fEdzRmtF~LAi23${mTe228E7!bFCSmjgf>NUwy|Hm!*p!hydncScEQ4kJR}A4dHDK;CWk1`Biu*RSr8TU&>{%Hr_hAE4e#+%lk!QwE!IW<>C<4t>NOjmah^kB5G{hy4rXyaotJT|-w_)f(Q5}Fe8CPM{}#74OhFY-qu>x_qqUJbQmE=K-)>Vu3@~lt;1a|v>Mi^o%;xy-B%01z+ym96UPoZB+5vZ1B7*US17R%iK*U!(W)R=dfCvzyf=t_hmpM%C^ESx!Fou@S5N3=EUAry(C!{-`i{-<kh=6Tj)CuHGE~r1q(XKSu#bcv(mJW?`s9}~-AQx+x)mB%#l|k=!Y$%|X)CR0F%38XE)F15&-Q*>B9M@FJD=(vgp<|dz0l`(fCS7+>Q%V6KJArJS01qtiMSMEEi|);o^))LhGGXpqjsqE#oC|gLfV?5f)lo|83cEn1<h^qiUAfkov9dc(ZqGz}l@E$HeU;xhkO}?KvJv)&Yyr}o_@|kagkIq5{>#&}?lfEqc6H!YCXF|Ei~GAQpT3%>B$2h4pP;%j&ZA+V-YbsxpD1}HgDG7|Upb0l?ysX9)Tr{NBW{#8ts^dy2v%J@vpg_RC_um7Hh!pHhaJ3IIZa_a9elu)kvD;xp>G;TDSf~n$V$*hI83Id&#p2OP<Tq~Ms%qju%PU^)o(AWo8n1>35Cg;wSoV(G}y0cRKOE3370)K;yr+o5Pg`0mI`1;cT6FvemaHU2PigRCo81%IvinT^zUmzaZ_1+q}@Mjx_>lnINza6U%KT(M!oNNLP-vIs7}fl{!zT<Y8WD1C}@C(Avj~~_H>Uc+e}nN^0iD@T@wLc+22g$ES+aJ^wa?4uZkqV3xwv~AO=a(ZvQo!TK4eGP8RcN3tPaA1NOxyZ5Ig6u#LclbtDy4`0JQ`=~OT~^4ySz9rS@_)8K73oKN)WM^D}wMQ#*=LhF1>(3S|@?Zw?^R%RJD%ZZ!yMdJ!uK2R1P$&a9)MN75Tt|jbb1bOQH7U$W5iUt>|^D-Xd&6b>Tj@l)G!F*BoEZ2ByK-Vj`ZaU^TJ5|>^x-)i`qa9+{cb|wmVSNxW=8h-?8e=4}LAq)>xr%qL#s^t80dnYpGy)$m<v@Sfs`!k-aoX*NMB@!bM^X=4cKWB6^JsEZ=5aKAjrl6P_*vL`N8I|lLvc`cIrGW%Ox^%zxareAG`O6nA5hH3^=LA%TPSmjWp10xsIm57u($C27JuCI{iB5`YFRUN{{e&eOToc>B<9Y;M-AWxHCcTGmrZoH_OaS2O^Z4~;CKySIUpD=gNx(VgO`rDZNkND_0{neHv+4pR(hp1LV>ciWul<^O6QA8|7lcerAopvz(VR6oowA&>+0?N^*$l>eV+WmT!$po9w5o4Zks|pv+X%wX>3`V&wH{|E#h6Mi6Q@5sTvoypPuNnZQH$$@+z4{T+r^b!_t~RYfPx!q1xM)*ICNH4G`{(MQR23w^)zx*{*}gKNkeiP5<t6<EN69h~o37>T}S3@%d=KxEI=+m=qqnarqRT&iYJVg>Xa8KqUdnCw1VLSw;)X$pUSscJU#Zk>h!U2@?!}FqMV}n}N&1e6Z|myTz|Gz23Ed{g&=Y$7ra&vsE3g_0+7`(6fs$w;1;_uS$)Xv+;v3@1zSmld8Hl@!%GIe(qi-b$;$dsn<4!Rd)it(m*|W+JnVb{nLN?tX)gqu6w8q9NX1?jt5vBHgIC|2y1tfO}9A$M?RsTqWW~!yd>wd72B`xCax)_mCmQ(1bI`j-N$gbP_@5JXuwz@`wd<eqb^A2n}`-9WUOeeK+6y)7~80ob<_*MW;ntK<szBP`T8MxFE_9!{W}0he?>?NbXZ&_BXoTbm?|#+3nCsG5wJ@BHzI<Dy#x`BNpJt!m{`DeblhM~wXi|Jx7P?9r5!?|jU{|jjX|h!yB)z6k~crKJny9}L(qD*?CVG}m6yW2?YX+$CsU5^K{lE2@i1+*exCy^#6#oi9{ZG5?UEn9`-k$(@r^%pR>82`AMC&J`_gU8zWEfSDHYN_d}VjnXT|1aCbMJy{S9>GU0n8G0)mEgeD5t^H}&%6!P}$3WsLg-NI<V9+4WmWz=6*%lVSieL2MM@8_h*^xm6X|+J8wf-R=bBj>C(n_yQf(y?;b8xt`oRjcWb_+*hw6YXZi+2;^%Op@6cLx@;-SBX+?rxG4z{q*Fe3#Ouk_9Dwvp*Y1F-Zm6;oeLuX1<zMer`NRDx?>b_)BYy0NpZyM%`HJoo?F*T&>PNXXsz~R_-TWrE<Z|$MN)WQ+LfEbRBe$+wWVhyBkYf1lLwHCS=Mwv7+>?cQ%Y5(5fj!Q}UX}4OUrK$jI87nd;~66>WNp^m@-%y6L}z<ZHQg_~V5S$;JS1|)(NQx7Qb3Qsg5Q<DJkATiw&OCntvn@9s-OMp=SI8Z^^w#|yGPl%?#j-PrS>pmTdk==22?i1EwGa1@fbF>I1hCIZ0j*k_T$b9UXoS7;WF_9b6qF*@tw5)THcZRY)B|eXTswpsmVoNpGlMS<hKM_($ihAx{|X5#JT0zs^iFluoP1=Tt=cx#m+1k4I2I*(6i$At(aYF-S*qk(_+28>9;@M?*f2p*`<+@-fC}&PRv3clbb)yV{!*~N$pY)hYnU-Gk&khYTgnz9|;~Np<ZFUbdO>?jR0=^a<XyOF_i}Q_>eq%*fCHns|A?OwD8{MR*i#h>6Mr_z_b_C$(Psh9Hyakw8I;-pD}1FbZvUeotVY_#ArwW)xL9QJr!47zISvv9g>2qRsMy<xikr)VI$yx1pBjTF<2?9AO>uL!Vc2@U|u@6$T=cJp3O2Wqk<ncf{l*vkd(Uavt~Zv&mjR>p=-5B3wNW*(h&9SC{OTAsEV&Hu9M=@x<#5$`5~FHjUpN>9w%r=FdGI=7G+rUqLBhmbQ>OX6_xIj5VzWFEweVe*RJMqh|xE)^_r16G_2CbQ&rCFm|cdF)SF)|-?;h3W7;u5E+~l;VBwQ<L|4x)cQvOLJr(A&!(%HFY{;u^6xfyzHS{8CdpajZ%@$=JxD%Fn%D^NG&@mQ?Mwo`MuA{KD14Hle<{{-O3y6tG)LU}+K}P_$pYYj{VT=k*WxlF4K&XnR7ie1fMJHPJjJdz!m=Sx<xb0r9HgL&s%tl}GtCsv^>eikmrRc_e4ZF_*^1~{93*UC<eIW0Lcwk_fe<8J5#;BIX-s+VdHYK*}i~Da4<A-(N*0xr$^U+I@X2Vrx?&{!It%JUsg<U`u%00Je8bt5SCSRYFdNFgBdY<ok+UlRwJZ^ZtA-~_QZvma!{TWRX1}8ORGl=73Ht=?t42N;b=fapRSGCMZa?4drt7Uy^FRfD1XRYRvy2Q2aT8Ws<-7DR{my(a%4K2hV7PO=mfKA=a$+pdDG~3X1TV@HXDYgPtbuCa|+ug6puU2Kt?C^5`4t9q>GA61iAj%f02hfX-Ee>j*){=N9sHtLLRt{eBc#Pa0>%gV<vXV_ZDf+Ff9dsp7?9N)DpfP?=bk04(qEu8(M;*#9D8$?~pspMvwFKS*MjB4|kHtErS-}jlv5MNF?}WuFxuRh7UsuMbNLPZlx85+CYJF)))uAo;m?!a798U1=h|;_!Lps@{ZPyJz8pkOr$hbjhO@T|iR$sK%AlO4pCk748h+bk)i$2>o!aFFU5_Dm^am*jc9Sny_QPLescsE!DR$lueL2fl<)Ku^S16aV7T6#cczLwAHPLz+Q(ajyZgVkGi=|f@P)w8ID!h7<Z55Vp~8_C7I)JI@<v-w0Oj~SpVQ%D$iXiVqlwwsX$?wPcv(T6zf6=ggN3-q*Vb5EK<d;AW)uUrP1u<5kr^OZBCPr7B86XEdbvnW|FhFuuuMTTJl#r$GK*$I)&s`~o0r?)jakcyfHJ`}jb<>^;-AbMftXflII2?z5j;DPL}D-;V5uyg*^ThGAYEvCv;cR_}u>!H4E4=GLIlb@`;I!Rly3>TSH@lz_1*&{luS7iyqCmHRa^3PYV%*4A2mP&_XHDvXts&5y1B&Hw=sN1;g?!y;f70(z^JVUx$S}+<cFH?s@pXPZ^3$sE5Mmx%%c3MYfJ%-%UoB-lJE&;z3^GNoSx8+dfr!;k!r%yLzgkSF-@qH)Fp|-!av}D%X=x;8JhW-=E!Vpb|+1ctbgw7aiVNo8D%?y-Us2pzN5IjY!SgM9HtAv@ku~dyE+~BIlf{UW9mIhZXmRi4H_J0m$k)!;NMC=3O(vK&bNc+#1p|UBq@~@3cn;SoPa3$jH;esm(nj2i{=%Guuc8qxcvb}%V-oI=@UoiV+d;hY%f7#yu9k+MR%cK*qeg1x4PvJ%u&|qC=OJ4uGbuj`*f1`oII_yw0iQ&!T3gD-8zE#D7Ag>qC18J7VP_~*)33`gY-mqm<=vcMGknj?QZTp5Vz$4$~hsP!d8B80WW<-lhVRu9&%c5uqvbb0w3rd*Lrx8upXp3y`+Ka6o$<4{vY+JUq<|g}|Ch1JgAJ^m)Pm@H7e_WHBO-<5JAF#<}oMw5fj`dOnlRk7Ga2=qPjqV3$GV_pw4nv0Z{j_~wF#U%C^W<T`Y&L?4a$IvVK}E=mov*pS>hXO^<kzi!0yi-pbTEW|Wqi!&{j#%iuvb*bmsd1ZXmnpZabIk%)I^zuAGjv``c?nB(!Zdu!Nv-p&MXoPF_C@9X(s`3)v>5n&L<a>l8Z?Gid$GTFN!2eS7^&SN89x&bCw0}=rWaG%jsiB<@lppfuJ||9dT(fY}z-O6p)wK-UnsYGW-*+0>n25_|C&E**N%C0k<R0cqRA~_lB&MXShbE4Rl^7Z7)Heu3>zbUKR9^sam5^0NoAx*3$;2KmEnrW6$h+7Zy>~=2EJ1>7@yYZO^<}wBvTgLm^T;8-dtIffjx7)mNg05!;$pRC-tx!K^r>w&;3H1<ql{?fN;!mfP;Ow>K_idB<!houjz#D8~zpT=vuWY*Mz7$Cf0f6CmQQyr_35zseh_&&-+SgIuz<zQx|WHLL<$zdX$J02)!ORqk90zNt5HLUdS_I$b-f(v><JDf{X1nopeOVTfnm#zl2cpRvN?n+o7oL%TId(DsQ_*X>MgMUmv6+@~2`v3bCsypf*df7MgN!9T5&{stZ`a9y$i-uqoY&7i!eltk8bIlE2j<F4Daldem+q^?>gjVgowvb^u^fiAwTFH1unt40Jrwwh=WFNx$5ZTId-eqB>a^m>Cm#M^Y+zu&A>wi^GUDpfyqHt6PVn#rQ=FIE6f*kU==(|Ioh{ICiDV?+v$-F>1FfB)~(xOb+giMD-ODeZ`GrXBQD4n>M#b-=KGtcGrRFs}aboMs7UcF+8w`ifzrFzSxPA2dx=w0s;`><=A=r&~T1#6PALs`l?+^~wFKZmu>7=JHf?VNf%r?dDwW6kGzrf=$#z<32flabjN#a;A24qb1`8iMCU<5^!ws$N=P#n;al{lzx!+VPUOJ=X34SEV>l$!<M+>V`&}uVXY#wzv_6JOL8ADR^L#^=-sF0mdPTspkbWVJG9FRgWIS)`i+^&i=mu7bAGE_{d#XxQAAZwXGVs7eNzv8vc)F1iWPQd$#w2EBGpgt?C6U{Ty{Lm@xBKp8XAj0ylioLn6hD{|4sum;37S&3YKZB5$Zi}Tw{zZymrm#ud&=0blbe7%)LLxo#hH5Ubj)7VzN}CvZN$V-w{PvEy*}ZBersN#30XR1#rwP9~L0Wf~-#o?Ie%kGIa9;yd!y8+D1`bWQ%5qN1dy4B_<)6ox#{J^f+4WW2S0f2jIQY1B+|V@XDGYjgw?|Y#uX1G#n;1^(ZuLQ;*d#Dp~t0KF@1(wv8QV9<DbllDZ?T)-_v`)%S}8903nIYm$beIKpiIg>yKe0p{Pfy=ZVY{n{;2brj*fQOFZyLj_*XMrw>eObFdqaTSv`ozc<fF({lL{aur{v_?jSkG`&c64y}ya}UFghk-c0lougjKreP80lepV%y$Xn+=&Ca9)fRO5a;H!R<ns+4Ut&SDdzaxn!<iiUG!qi<$2)o-87Q0t%{i+&(sYm^l$wnyjj*<wMgXSc{0S4<I-55$KqYU*KV|VHFTr_d7_t0=+qTeoF4QL<Rvb0%`QlPjgyy`v_h_f;xdA<(V_RM#5E?rfko8AjEo!BL7r$bl8eCVq*z}UePiWJFN}?5+_I32PysCEx^HQKwwcBR0CcI+8-W-!j&&V-exScys8$1VjanKPIoaYQEN4&_KAv#6{??hl-2o`}GRVIGDFR`u?4pCLpy#uC>Ihup7H0lK^a;IN=tIIo7U+?44qF?orfD#eiA$nM07bE(j^ImQ+O=7Pk35IihMG@GRbcg^PQCEiBw7%bKpF8JjHu{T-r#eAsm0_*f(}c1U4%1cs?%Ai_ZNAqQgwQwVOOc&m?xWR+3Vt9|4qO*-R$#s+<)5bdvVEgYT}&OQ<>rIE_gc(E~s;}(esWtcbYv%`4n3w#)y|y)1^8Dul;Eu?~&LI<sBpHu_%YWlbq@(gk$A2&O#6%eIUazhFXm^pMZ1vWG&T2Alo`0)MhOo)M}lFVK3`DVrF^Q8GkGF9E$onZu)uOk47YU<6fgqeBg-A;}oy*liI~#HV-o{c}`m&e;qa=fo$_6C&_BzWtM%g5K#ZTVDPLt+WR59OlI@it-F$K&4*Dyw|Iwx2=-kk3yQo~jBnmTrpXpbwTkV{(6|y;^8=`CJbdNn4vs)1@A)FO)jLBjwplx(iJk~~8b@}|1mekY@`!AsqQI-8&}ILRojtx8O2}Q3WRni2UQ9}nUBx-Pzpe`Op4E&oE<u5M3)}mAHi(MSW1F#&DXqi}60I~oGJdVi+`_X$Sj8nZUzMDYC<*uV>U0-MmerhiI=TY4lpOP)*&GP^DwdKeh=jxhvr)z03sau7J0j)C>K0hMH+VEcvF$46F1oDKXj>3Pq3l@==2IZvTC+@_xFlUU8}r3tvOn_jud1(sue}<^=}_7Nocre9rs|C|@x8e&aFUG2|8NV#64==CV?t4tA)wGJ6lhuoOpwNGpzV&#LRW{&1N@Bs{;j#cD-xHS<XyTDO$_I=Nix7><Z6%3M&{gpwA=rL+qXkW+*6ZQL=$n<CfvhvP8vq%o+hmf;>3|zBsfVxf(Pxhhm%lALi*Q9a@KHT6lH2cp2T8rH5yO(v!#QmXvX~Drwb2D^*~&0{Qgn<INKdrY>ju;VLCg9@n}|Zy3r&zF?U@_Fj*k$;p5WHwFwprI>ytTAvq|qXaRIEOTbg-tBol^brw|$>5E@C`*I%=S_1zotGpvQ7%Tk;du}yrs<SR{F%QRmB=~A~yTiC5D|mJdWs3p{PR18#k;j&ckU9Ihv5<C6j&FQH#=oH9`syZb|8SV|@cz}*ua$YduUTUG7I!CEoyS}R3qp52RJkIeD^%9hIfu1)QbammJgF0K@Y|l;W|S7oGA31N*`5Zzd^ldAyPn|8mu9CN)KW_-iNhB+gQ?EsgJktJXPjdQ!$eEg3tRK*2jXRnNvQ;Un~2Ig+MsOo-0PfGM_qrs*~rsC9I&xNP77Z;c`-4wv^X4Uv2}!v74|#BNaDn3do3m&)y|5oVOyV@6wti1x+criB6#XR_2%Jdz|V0W@U%V}iYHk%a}gtTV0C$>m2RQ2lpVhkuw~RJ(yG_$IH5C6KVe$PgM)HCB^SV;{$x0hCT2NKJAh?P(|yZlO0e4YsoSa}Zni|Jfm=C$wk>44SqSs2D8i_eR~5_nA|D5+1tD$wgQ=Wq%z4NsRXcFerroBI+KoAav9>Dwxzo`RT8gYS*S2=F%Q!0WKp!&LK_Elg!to@#fL%*{Y@M!M0ATSeOkh&pb}p0gB}`}sczYgC<H@bSYq66FbOiamW#j-%rm_JAPB(L%<wM;_$oE;d9wUgy?Ev4yYoC=aGl?5yoyEdRQ+-`)mK9|PV*@fnpL)wE)?Qj!c^*~1i+feBvQ604y=ZiOp3bq%mMUt-JE&L<QkoXUo^%$5mo&yKFA6MK2y*kx4#-J5qNUDIhUiJq4|rLX_Y@SyGgFbaoig{gWY(&+tj}kz8h$OBVptvyQvHBz+Y@w@fJo?5=Lwfds#1K?EQ9Lm>U_B`i^$2=R=0QNS9HCLjh?<%ZMtfmQK(~5tT9I|Z<%%@9H?!OL&NHF7KlYM=5jFZIe;(bVl-w$pBVEkq*a)JP6%^<K7y%Y{I!cbv?RY;)#irXsx_ez*r1XcRoRZthljfB>DtU@?keewF0JrtCC*cq*#KYPV_Lk!nYNBTtF~>PR8v4PI>Qany-Vu7WudRNNK@$&%ZdO4bAqIRp0!jL{T{DZurt2P%2(Mu9kS+ydX^jj2&9Gb$O8oO<=;{hBP_oqs@Pv&0B#jWY~8nW{~%)1aRE@(wLQxK8nrM)dZqYVilV{We^?M(p5|MrOmy$lNm1r}_Y?j)Ye+#ojc&q%4t;`V!v<&NxA&Ys=1$g9$?@xk7NqCjt%7eC<f~R80yUg-&APsT8#n(tbDRyzq5xQEbIerTpF=H>^sI$RO#>|-l3mQpv#J;ln1YH#2V6kVM>?G&{YE#^oL@&jzu)%Egwsfw&r^B&!E>k^qdJONJV-z=<%0meQc~QNcjris<-cXM!ikDFsgM-t_r{r7xqnKx#kK=LyUNS}s$o^S)q5toqUAvO57B$pZqpx&#~4-X+dh6YgJeeY$;4yLX?Z({AqW+8UnbB|GMR)|XeZjXDFve*8213rQ&Bp~>{fjWa<$fKrCrt8nKb{zRi+_{nvPS*6A$85!_AfGqOVfQ^{!~El@v@ESnp1{{j){7e_idbjKq3ltL;H@+qDn*7>tz%AWayb$yQw}Nj#3wol`eod22WE%#4q6C!+)pf9G+m?NN(v2xzdNNp|iXmBdbsKg(-~{oQj0c`q1$>{fEO?X9}H>hty!lRq(#W>1S{4@8eM*H~=wwJO<bqcX;O`k@w-|Mmj!N_T}1HEQ2BB`g<?V@&-OOVzuJ_q;7)H+Jznp2fhvdX0fwtAnB#n&r_R9o6-Y==R)9-3!(yIUxKCqyha-VNAr~yrE+Su5+MX%^K8+^WCLl=d(%9&L@w<vrgSKYzilnbgVwoI|pLBF`VNVN-v4#ICp@Zi{g?};9hVrMG;M7j)yBVaW2bn496Jy^U>R(PvURei=sF_nnU~46o65SPB3QVZ@kJ^#>6UtEj3htHsrkeL~(+qW6IYH8RMd`wY3lxO@OTc7a1dMMl(gF3c#sPp$vOzY8RSy;$*Us?(J&fFbzwxf!ozq#~s<8&HDUp*LmQ=el`HP6KA|n1E0+=ebQK`p*3G*Z*@3Y8az~kbryza<lG}f%N%T`Uxv*ud?7+@S6^uWx}60grdNY>2eYz3=*>#youb+eC3XOYUkR*^t$p-lb}$doZ;3^Iqtms98HajJw4l$Vn^S9xYIq(AxY7|GgIC*<+HF(uQCIt<bDrBqw`NNl?YL{4V+vr6;vCn$v_xIZlL_q0DaI0JgJ`1Rb)UTuFZEs6@eSUH9H=hi*%CT6GQ-2?c%Gz;U_%BtlUwm7JKRzEM+)9fq4+OUvM|o6t21NYF-$Bpn_o<lf$*8kU>7dtv)LqxF;o{F(<S<&%F8^Pk1xgfxw28610OZUz3492W%foRX#@KO|8xtaw_?C1B|K)w-C$8IE6pI|qzI#{D4C#Jz2!Go9?nyAB&z)TJ7V{zmph8hU_iAF73E!(K1*px!wxLVVkM|k3XO$gx=gZR+ZR1)Ai^{ojK%@lcm8_T*;+qF(2AMuW7WQRV{{G-K7eMq!s?OH3J5ANn|h}iWMkazHZ49x0D{@A7$td8E{f6lUzrPz)M!?w@0hggK55`MgYF;DSYraXKccbC2h2A2CFk3hoNr%pzWo()zAbj>RSv{FN(*4_Q_gOroWIrE{>zm-idBWBN#X1!Yq{ajkKYR_D$vg#oaZa&*-s%{pzi?eI{^C*!2TkDPHH$Ocif5@=e!B!4AeAJt|N-9kY&zs$ShAeXHKV24|7S!BI$-fEULaby{?`$>)y$xyh*}dO|t8^Xm;4c7cY}y081fmzIe(Jh-(*2xr;*u17co2_p~?y8IANV9bL#HB2|E4l@zV(h3Ymsm}CWf2$SJW1vVc~leEQ+d6P1IegZ+dzBn$e!<-~_@_}+`@qPYB<{5mbbbrq2(1*zO_W)_o%thr*CuSD--;fl|;xfx~ikZd#?A&5sQl9-!OnGKzOuA#rv)V?wYUVTV_FW}FvxK!JdC+QQ8?us_3umS5WdA-H&>k!=*wRhJ$dYB^bJgr#_FOe)`Ff-iXIT|g-mE`1#aPw2-1i$BT`f<Tv%o$#RTyX18c(A_&bo0(gA;mfs-vNHh5?frsJ`bE2)#FsntC_;rjw@%PG}~;Z4v|T+lZB1cU53!Cozd`lETc;RpTcuU#|gl8*e<vBtCNS!rH}EUg9FvCC?m`;&=&#CuTD!KGFFflYL%aCKJk7(4rbDJ=6C*hi(QC6L>IBi(lt){9D}W_B^rESf=^>;Y5d4FAjb($;A%6kpMiWSudr5=sFpa$=xa1?QdDpHg-C>ZPBvo*DXxTiuu#gZBs@i^JHu_`$n{*;j!(sJ2f3d(ty3PFm$znG@Ho&MA1akVl<X_ZDDJCrbbRI{j%;<U)__t!VyC0H#c)aIk|u)%gzH-he%aXtGlEM;mtgi{<hJ0bj?UUpn>_Wzi#^=w5oIPZllZ3!rT4<b;7hdo2}^peFXfnXeN3)E1mUKr={~95N(X*v=@sF-uQMu;7v=+ZpK^cZVfgg%zCa<WplViuNg_z(|rLpA;$auu8pjAC|4`zreJo*?X$i3IIRKMvGLh~%S+GL-u)P@>g;yY$y|#rw(V4@Uh5*2Gn1v(&h5_R#BKXCt={z*Zk-QRY)EbbUl~aM6Ar5W`DaayBL6O@Nxz&YZ8%T*<wWV16QzII6QwJiDP8NVl@Umb`Ffck-wbBAYg-)7wRRLo({6N)vK@cKb7@0yacf49Rbg&0n*3H!oLW#tgrUzxgyCq88KT2bBvXuU6O<Asl9Hh;HOM9tjx8yI=whIX?nV=ef5LL|6$xARlbuW!vmL(T;5`aQgY7jhi<WVm1FZ&_V1|BTR>Vm(qbEuyjOp$udyHJmu(Ywb_<mcLHPBm&zFwok(>dl%#ndaX!{!*yoT}Q|qLk<yg5Pg*k80DVcTSGz5L^y+Oa)7R#1X1I<(x$gv_1*E%XPj+g@wVis}6A06JRnj<h-h3+s~j>5Y1+BI&4{`Bq&8@RM0>^yE|J3zST({Gb+!Q@VP0q^EK#@*|nCs(R~&2J8GpPKz>Xz5JaRLA{OKg105Iiq!8U|y}0|t9!ra%n#q*q8<kJFEq~GHYS7q?VJMTfNJ)_vpOgQZzsPc1n$}P3^H4ynb5P|@>=eOP5p}9eV;QjX%`<gBtx4eM6nu&|&N6nF0k&dW4`db^jBh12MM0<WP1!}3A|}~50V14JG9B6G41*e(1Sm86BrdGzR*nSBW+V|56d*gv<7jx>RhQ@v#o_B0GQt|$qjd!+E8_er7T91hi|_(kHFZTcAvHe`?=ei)JOhx%D!vss2QssG5aDTC<Y+?*!o|=w8B!^yEtQFDU~X;&4s8;5+2CM@zgqb)Q<>_16YbmeERsF<H7~v=i`mMaOj|D^*$=`DE1f`?&}q((-Qw$Tf9!~#+f~?T*unpIO;koFxY(`amK|{uYObO<sH)A&OippTqrVeW_p>B&%=60!Wv}T*ubY-+WD4j;yJNnVur?}AMIaD!1jLiN>lI}YR~fZ8WNQF}e_dh&KyR=~f&$9f*ev6|ckFb&*ElD~dvY<KY6gT>(xN4?ZX5j0oEMZDQ?eszwX&F!ZOPJNkiWIw!?z}5Q~sRBfYYU@TAN#h0;=lzOnYR9&dBAeW^Bt=QporKpKt1)nHG^HNa9ECzSSXsTlDYOl70!umQZ(kz&j^#AxKmNNlH`-s$%=}Tu-{GYRP2wbRS0b2%WEhRYl|saK~uVZ9E6&M4R&+87v}7{xHm94sfPJRJ>Ukk{q5gKy4?e<<{1ue;1sIuebG>VFRT}chB51iQg5ffpsrjNz8WzD<Mh2Dpzzui*Q{5H5RAb_O`2~>Xgy~xa`T6;&@xc<IXxJoQ5)%td5><*Z|hm5eQRkKb9Z6bVdXdgTbz+GJ5ji6fK{7Kn|laD&n$WkV;(7$FnHMh+z5Rmm;gg@vP84LGT=AQz_76&NC&&<8S#ImNLo*^20F-Ackx|xa#ZQK%SMDpU$VVTNLcm*&;>YD#0zhGSn4ywSanYu>VGNVV~2ibLWWbf%q^f)Mvy9*2KSNDQ-!Xsv2x)pu>M&C6f|e{tSnZA1LGl)5F*Y){9i-L6Q&VphEz8T;plTkfXgj64xNP-himdFAcFfnU4F?91TBVN4ps3{d|1EDLzzkc=o}XnS#o!s--LG^3GKjP)2h9KBbm@!pT&*k|*I76d$THHIE78v(3lel=5R*b>ko^=p5~1+5Y1iL&AbUGr{1h+{JWv+;4od`~J=Q!|=uH!#+q!ruU2LNY59B1QKOoJ*FDavEx>sga+#x5*UE&#g?xVSihG1Uaoxjrlp6GefbAYjhkIvX@1KlgY9w9dO}SopPG{fKti*C6&`36?L>15*OUb>N0VeG$tIUtdqqXpGLxe+|Dq*(u8tbWUJwLKS4e-6M3{C1{?kIQ??&*H3t!2?s;On=q1?9{E0#Ct+!&)n)YMD5!sar|li%>j-Ndns19f2`o)!{}#x|i`X&4{_mmp{^gs|a(K*MSzw2DxBRK35D=F9<yIA<;V=l;XS+B9wgV;90#ati}h=c{ee?cEPE+{nhVMtZ(Re8|H(cnS{wEJUAERyK2j1<%fESjHm8(yra@<6GYN7WE2a$*NZtSvD+hwiyv@zRju*8vmFY%SgyeKfK9B>iwqD{#zdy5OkG(n7Jm^2M?vuNC>HX?TA%Tj)z{?24q5}zm_AZBK1NtBQFO;iZHp;md|k|p|ofnbeK*%Oi7DVtQgfR7E0n$h{<X}Kutg)2-POv;`~QC2d;g3n13H&?E;0jDR8hZ9ydlAY(6DHj?A!#QTBPwa}=i{G+SUp0&0?A(wZeVefVGZc%ENm?LgUFfC!XGgu%T4Z$?+=N%-7#RItP)G!CjbCXLSKg-|09*^I5UioOmB0}8E5SPb4p)3}&LXz4>s54{{V42pYa;y`v|yMnhP3lny7waq>fbiX?3gm?u?R#f9NA@TVK2?F$@Zg(7^x3U<J;hfVR(v|jfqz69^CdQ$jMNqI&50okV0j1)K&gPA$SuzZZrSK-v^kNu=V>%V0fx-6Gp&TXKv|X<TV=_x-g$d<8F#0BdUa)a@5Ep~^vti&=6v@;YObw1A2&uy;I$hiVQFS=VCU__X;S)&AZL_2v21ao;r`9U%DaTGcvq=WCdjE#muF?7awAa6DhwceiVeTrLB&>f34C<$)wYXI*hUq1KHQVc}nL&rjQgjSHgajlRnlPm4p1rQy7{7|=Q%ekiU$%rWl7wx?1)amtgO%O~NrxL7Xbzps0QkH4^dhDkKV>~+e`pv+wt@-z5zL$rbEnoZcG=&0|Nf0HZvHCyP0z>TLBGm5J<TmP4caW`BL5BfJMvLB7!|?N;nD<Fx6IO^NRzruDrDN1`%+lqzO?WNr|c;19tc)lY;_I$$Ayp$BC;kKbUF(R>^zbQ)C&<^pcl8)u7(KozNCbT^E_TWvg)Xq4CKK+gc+H#E|=n^mJz^@l!l8i91OfdCBI77A=AcpXHMDzOf(rUpE6(u@DNssMLwr%Dz~1rEvdt?X4(>%A#PHjzP(ESqRAj;i77-Ow7Gw)K_VvgnSj{bvj9_K2?)A-V<E1WLC|SK7c8<B#zN;*=#&XxVO~{!zB1dcV8}B93dE)bLM=uuI?7eK7`TjQ7$)+_1Dy*Rn&$<YYSLYgs^Xn+Tg36u&oa5Xr1l+w1&K3LFz;|aNG?D|)+W4F7N7J_`kSMS4((NB2;+JtKd)cDcwbb$(bfzWtqg+H(sVU%AX4EG59rNG8=$Hf#<Og2Su_lFV+kb8p~hpSfXAjSE!@CeZ5e53euUjx0PpDvz|c&appF|226GVeX>(rK$>noVi2?K3keuNhrwZItw-Sqs80IF%2nRGaljbKbU<bw%4(nWzlI8{2k5xxu;RY*urLLv$CUKP0r03*VHKlU{y>a*1GB|l?oW}Nx5-MBAa`~aPIQs7~N*C2Ja+L=X<gY;NUS?T=I4`f`IK`YEeDA%=&r!gW`8z}>d<ZL?V_-O)KsLEevuQ$N=;I}Q5!S$rCx#4@Je`ANf3^TldjUR{<DFYrwmj}Y%S^+HpD=hjLWH_{ULk%tTY=soXBpfM=wRh%ud++bm4sm=WuXL1=um>0jp}o0b@a0^Jj<>Ps&*C!>Bh3B2Ut+ZlOVE#6*0M(TikBNz6podS^mM)b}b|>=YAmKO+#1|QLkwn8^O07Q+lVu)h%b59(Di{sp(V7hQ5>rbj$q(Eq_~YyZ=yavtJhjQKQ?&NsPOF-GiU0f}jeDudLP-tea$FND1{;HlFKd_PDU1B=ZeJT-q#dw{-X#!($hiY;>z{qZ^?%5rLIf7}{i#U5Dl9NxylY`8<XWIKPZx^f|^-aI!u@t%~DUPx=^Pwljt8GZwE8eky>+M=3^Wl2H{(7BiIZfJbSGPQ1dH4BT{+6!%=(>oI6PZc09DB%!@5%UQ9tzCOEUhhvtH*N54lSO>NOa^pJmFD}+8k>F)HojhU}9bviqq~Aq6b+fb^QIESUo)wGQ{S5XG0&SLK5{yRmojP}s0pom_Z5LzffPTo{L`ovN(l6<DJ7|-$T7q_uQcxg{bllzyWFd043k**_ce-}IbenC-LA5(rnd}Krjaoa$O?QeX^<l$455!LeH2@TZC5&DYRhdI5f<I<kW)J@?j|WRt(7|Pf1hNg1N}9>EN3-5D{&g|JUuR3%dz>O^hBti932(>+>c0)+(I^?P(5vq9yY~|Itx>disWzAM5ll;htOt)H0l#87tXXWWobTHE(%=oFPur^9%8z6CZQ5QhQ)5NlJU#`My_ww`&dZ>fO_H+pxbxVKMrLEI$HRn{-!Gn}KftqeH%G!3hrmNP1pavA{TF-oeeBs^ETH$YfHr}6j9b`*!ucPYqezK;KmPjh7whR4>**Kk>Hi|@>D_HKe+i*>nQ%H!-2kYElS>usN;2I=>)`91cmIZ>>59VXFfImpGLsZ`l%T{T;v15Dko+UDi|048$-B6$de3!iiDE=fWsZ?0#(ZBMzuCVdhyOQ>-1;-)wvwSGll1NpE``hBAz;*F$a)+1iQj@Dc|bKraB0F&8W>t;Dh9clLMH1PI+-O5t*%E&1#PU=sJjSNs1Dt??wz)T2a$R<|3y+yC95nFc|hxX2$@G}2wpBkn$c(JQ6?WjD{q3KI#IG-47*Z8yo07?i_t3*tf6P!tA4Y3M0Za7F(D#)77UYvwEm7!uKJ9ixBfD!u~nu@8N16`y_@y?vxDmH$buwBKUZ=$AXVwQ{NTEG651J(X}GDH3PiW3zL3eC$}Z<AchOmz*nxDOF*&ZZ>#Q6E@x{&h#m(xtSvMZe&w9rgnwrn<UlkT}*ASR5;VyrCxXZsTs^wv=NCuN<pC%@ZH+1_ikqiH(kqdV<#og08ty!?u4B<YyiWN;o?45~kW`Geg$rKYJdSz%-7RG-UH+<#1Zn6Y6+5z#_(_xkLzEyc6eMZ1t<n;M`fN>vHLIsw+X7buqx<7J=Yk@8$oW%pY;dUrf0P($yTm!_A%WSCabqD%34%u>7RJ{=`)QYym4y8vUPYYViF4SolcqBVxtrfpDu-lZ2Vi(hK6frhLnK=raVxg{yCVprQn<Qdb<B`0O6`qvz+n9Tb2Sm4))}6eDY4;ejAk@0naxVKaN*oe2zEP3VMFQXEYyd`~ee(n6o(bOUyxaYM?~mN&Z8jZarCsPG0yhhuX{&VO=)(T<XCWIgX_)<|&u-v<A)VuWkO8-#g^qxK+U~e1qsIyZJf!Z?09%-R4KuK{mzMugpTmEJ2jW-3OG46^Ld;Vc_)|UK(7Uq-L5dp3Vx-)$&oUChe7>P~57K8MT_Um1>@(3{nuXlUMO5~N?4K<*gNmCiabxq#WAl#b1M+6PWV>a{$VC&NEiI5Atb8r@P0O~)6SjJcB<dJF$23UNg>IhgH~9$uw~zVs(85RJ8T(jtFw6yoDNvfNEE=*aW>}3GUl>(XP==8y08`&ooLj+F(NAXTQgYl3)V2UED&GzFtGpxzS>*<lX|7asRl{EKn9%arSWh^Rf=Ue{mIXx%^UTrtx}vgea!bsDJJh1Q3Aj+EPm6g+W=1NFc!#IU?$HdJW3O#kytCL-tTryx*77V<LsvJ2>-~!>bx10q6oQ1d!DK#+mnTOq<`-RsQZmXXBgz$4GJOHA;$R$f#ACL+?mX5%Up!VWIPN@^H~Xw6pFc?Pqs6r!qg3)3qnIAUg4JR!MNQDC0E56%^8^ho=?Z<v`I7TXCnXC?nwX>}iX;N9Uf1OW3SQN+04L-nT-!nm%I`4CWRZxY!KE<sMk+x;QUwF8h(!B5<tsInUIGo8m-3?P(lHwyA)fddA>HC-)%AYlCYH3dt#9GRyXIRi&rxAfJGJz21WMXk?*{8d4VFg3iV!5G0?J&rW@Yqnun^-h(FZ^RM&CUU4iMC5p#T9iVt_J-W}vOK6hzv9p$w)y8~IRSG>zc+Dy>nwb}8hTEY~_Ou~j$t5Z{9K;8w!@bAj66-W{md%XYPPth`6zBu?f2xM;_-1$deGRjo-7!7%@vvQ^_%X2ND03l^Z<==B|(YW!`xo8!1dz<MiBhBr(4n<w#AjH;4iU07d?ldCwTw73bVU^ju-BgTWDa3h6jAp1|0nwuF+cs1@4x7-}|v>9ICg60w4Rb@$w34zp-4Uhp3nT(uG(u7Mwg2Cd5yG_{!gu6V0tZ)CU$Elbd-e}VpIc|gr$1VN~Joh5Ax?xBKYTGua7<WL9kDz-as5lx}QTmbQ5?BM*@r05G`dP6|cpM=&zDkC)O-V{zWkbgcUR&r3liLc;nzD@1w9%OZo^1<Won$$S^bY;SL;3lwD+mBjm}J-Z1go#hW^5GJ+?+^VAUMgTX30%F38`oSj1g6H&N!^BcX|e58`Vq^RGkgxg>YC2Ui(U$`bih_o1O*rPUtIy(4}&xt{7x^m!51ib%ZAyn<XlaZ%Tt^-O{{QBtvgEs}<lQ`gd7rMqnXLM1yh;EXbyHHZ`{zDOcqWBgYyZlL0=1PHT_>fd)C9wY@sgF2z{j6-hctd6{SPF)yj}iV>e1uAU(WWTAYK=;j0Jc}@J4)*K!N&^3s_g%W|!^}0D1(WJ<#1r?PJ^bX!12Yk)g4lc9#1T%V*Wd#bMWkvHOa?}@`%$rxJHl4~cNwESiS36pDKZD%iPGA+AIn8+bDPJG!mbAZG+jLE)=xFffrYKk83a~7x!VgSwtZjQ6KT&1R?7O{8b;npE8=anw5`9a?aZ8W9T`%$qZ)~rUP==*->7>P_jn;rQ^lIoZTC5P%_7*c+Rj->$ss$A5k{#H8qICOYBX~+TR}T57b-{*S?E0E0MaM0##71e(L%u^{+pZ9BL`n6%<7Td=L@yYkri+lbBcG=OGc~77c*o-|<h3Big-k)kq3TzKe6$;F+zK?3-0};#HcfrKO2uCln%p85Q`HYfu}^hl%F*G2nNzj|`A;u%3>1+M6Z4^FZLzK#eF#15az&kiyY<<s$E6GAk5QH$y0^>J5P*Vzp|FQ>-d!PCu53Jv3h?i``8?QO@z>r3{yg5<!0>*O(eQr(`|*y}Nx5sLu~-M~o7Q1A3YXWJz`IL9FDu0`n*tw12{)VP19{gB-BxINLNJ^BfMnS@3;h6}SnXAU&ng-VY8D>~SpFZR;jUX|e2WX)?AQ<BD4((Hy}U1`&_j?i%3?!yhi!jk4u|1N;XC1E^VPkx&E`XCVa5q$os$FhQS)GJfb2O!TWwh2Cp#nKu2VNY4?j~`^;@4`z&n_@JVqP+tx|I|2~;X*F~3m7Q0{L@OJPSyGavjQvzh#BrfqkO$NPAVFrD)$8_dXo(D&s?oe!0unrV57rT2fCp_kX%!&rLd)uNwc>0MpkxNFB3SZD9r@49t%k!iASbzK5~B-1}oI>{&~Z}5J<r50MQC21%u@&P8$4)A6zAQ235n6=uCVjD=+vdThrrn#}t&~P47%UPi5+z<U1_d|a<F7Jwdu5usrL(RV``YUB0Hp8J2%&n~HwXI&ea%sM>!v5RWWWh=c`(AMxCLOg=w0m-9UkZg5I}SGZD)?nK<~RfV8SCG(RR6q~&eYFwGO_~@JnaS4r(p{(y77p|7JEWBY}u|U8WI@K{GZo(Xm3g9Er!_8`3KOF{&g;yY8u<(>)LBT&@x30ZT5?X@+vL|m*lHK*_Tc!r=VeswM3`{VDB1Gl5jN1KsHb|O@Lz<_AG}@&A@C&RlJhX46-FDEYdBflzQpcIbJY6l&*-xa+A~5qU1ojiW0i4Qu$R3VY7<17^sUP(PK|mTnTb;nH6ywh<)kkp!f03AdY1qMm4$IP3N{h6pr=TRUQYog!enj#0r5PsfEanY4xsZMU(63wvaIKsz(V05JR}dyJ@d<&JOA@YC{4+qI^rQE6`faNh3_n;Rwane7r!Hm0_s2+!)aS=2p(CH|Eh1MbD4VJC-vb9tO5j!)k;UFOj>D(@c_?dH6e4K_bIC@!5X7VkK7>&2p2o%bLcAa(Y#WBz2Qg!l>Ib^+`o}e7aNl3oJjG&pBo)=EyiuIi~~Px;=9^OX>rMsn?e@+fd&Swp3MXULB<J+%~(l%r1E0SSx6e!^`ZE)%fGWfvgE=AvmLBRv%UZSM%SDB*}g6W&ZX1mgr2UDGOJnPYBZ4!n<J{2=ZDg&eB$E-8O<zHI@aMtMS%kRZAwaF~eMEcB4&qkjFU&nIFEk^%ezyoAm*}2A2RRW&5VZO&R>E4z_ExxX}f*u<gdSDw~~t-)yw=H|&C7L3#;@Caa&1KU6(iSXxqi)rkgIi2~c8f;H8W;2SiDMxTq-n)P-;9kVU#ZIjdHFpegBM~AO>-YF~Qik8c<l241e)1?NqLp^FRm5|U{X&68nA5YgRncEP#D~X$>Z2F}|@1RUZ>g9EO0dGG!GBXFTvaD}Q`_lFRLPa$$_f{-Sv5{pZ-RjKIj8Ghe_v>i#p(TA8p1TzZJl?RJ$RNlqf9xc%N~H;q0R}MA!t(8t-dR8gT3T(snasx5R&zXQHE4B@yrpaKXsC_46Gz5_!}fSVcQOZfLu2p_{?j}2mt|3v3iqCNL=P6OQ*a#3iz12AFwK%e1J_pz>T>*9!^4Z7H(XdVC{y6lL&vq*_6D^5q}FmC-MX!AI%Sa1DnL&*I^x@>XEnqWF**#wi^)8%Ks|59UjI($>)#cB{bdO3-vxvH)lt}Ag2VnoBw~|Lz^x9ej}l#M<Ea5VHUbfj;plh-tUB>IO7a4aA?tfw9e^}Vd|9i4-0wMM<Y#nIVxJ!;MTQa8Q7V6o<224$vJXA4bU93t8%OiddF1bB{zj|ywfPi@qfh_#h*~zDn7_|_Bg14lktVQunYfI`=Gp-0o4!IbYq9m+A|SsHDq`=gGBLbN?l3d`B~#P?m&{H7b0?=-{+{)0%uj!%3F<GMp}ud5BDDYEX<sOX48JrD<5Dnz@1Z@<&uhz8B1Y3nUwkF5#Mh$786<FfqHirgPOsNK!?w8&=dYoU()98Ts+{J8KN839U%uZ`6}S;fuXas`sk?)0h}MBB@J!kORFblo_A-~r%|T0$0<OM>__sLEidG*H*=ws?gs}%GX=f8Uqs$?USX)ICUgSjKX+|UuYHvIm#e;G|gC{+VZrnJ7hI@CkXgw0$CI5*xSu_NBMOiYImVabIKf-IIw?yj_l<<xiw7(OB%P2*}XF0{*ym)>1owN<mRBXvmO?`o}q4ojWMzKQaY$S5j)HRf^N%F@lXOk(}=4HjUx_3IA1^Tk(b<uzN)TU$auwG{B0%bPH5C@W$hApA573Gi15TiS3TnrncUeb6gbKCABo{2<f=PaUNy3LJ^>MP=!Z;J(x@*?kzEpsi?iAt!d_9}BzG+Tip_GQALVB2+}3)k+;ty*kbh;ufLZdx$4cKxonY}bolHy6LIUlm?W_aK|yzKuS_FNnA_QUMHgLjStvtHfWtKYY1&7#{Audb78C9QHPPZ5PMxz32u;&%_UPWLQOdwIU8`md_OTL>n&$oT8)5_`dB)JVHM_84}TH?wpj&W|frz|0F<Yn6d%TX!XiLDM9sByk1gK&jtTDfevh8C%(38b*P0cYNqxn2!gZatC7~zYubU!ShDg$v{bB!S7d98RallS#Cc3K=H_I(;(@Ek^7AwyPYK>oZtt60?$lC*vaw+*5eK&<WHD-N*d7Io;HGx+4Q?q8nLrQB&VMg@PdrLnHjT$oh{lzAcj3(}0fq?MD*be(*JB7<72;DrM7amU;!X0wd=lj$XrLIbE+w{>gxxVm#!I-&<Yn{=4RyK!?pMfnE=Zg<`CScZu(P-b&Zwt|+1UmFmE^WU331Y$qB=cG&;v|A&Kj##vfiR1v&JHZPG{!a)*FHxoe}F)?IR%T0E;k&58Es3ukPw%Qh|Cce(s1?tKtBfxxj1B+jjdw*h5`$_)GAI{?`AHMcx0yS9SmX%eue9y6%7O!tSq9<ep2rzrxz?ueP`!etGxZ_1*tN7P!4cdCS+@r(`s%)qS+;GIi8fFInE5HGJj@rtW;+PTKD+gvr|ME7X{dr3<I}zIdtHfNGknJ=35IqB3*%`b9_l`13CQ-u?0C4#o`3C<aqi@(G-M>0LVEm!LpijPrgzzL46KRQOn#Ix(%33&vR+VnyLih5Mf9@o<RlhR5Ub@s?0Z$zv|wd#p?m`c+@Qf`#+gz#zOgp3nxxdM&({zdqGGSgHf}InO`}cH8vhyg7Qge|Se1j*h)cCNXtV9;+e2pOhjD0AgG0swZozFuxjfKui9(If75QJjp>bnyH{Y_McGJ8jk0~q*O(#eddXWx?7GKTFacNW=N@m^td~G8g9EK?RFcBQl-@$Id@OhWGBG6J3VOJ-Ti=zX}8f}Ry*LLiu)YYf4~`(QJNXS!P-ZrUSRGSJONX6%Vs7`j^h#M`rShKe$VQ%b09~D$8k~Kckxw6-Bw*=zVlvA?)5t#$uhxtV-5cuGMC+N;SgT_q9?=toRzh7Y19^-?jbC(`z@N?<HMud^z#=DhhC={=BkJP9&!y?Ww{kLf5MMD5Svl`PDCaN?D5A>kAwfu-n+K9aU6-le`O+{91dh)NPv=TaSdmwo6SVGP_&hV;z9#p2u=iGU<@D<_Ud`$t87QMd=lI7HMSGqHXF;{bvBVDUp~D5WRE~fpK^Y}TbJ&h?wJ7yQj{%w>CIA@neOW9>guZM>gs9^H^d_{Hm0w*+sbF0M?tgJ?R)guT5n5vz8;I6+7Y%(Wxvve|HqgKYV2+mW06Vo?XM5B))@}VZ8n3^p<(G5yfgLq97vyGf)K+(r)nwH)ON(sX{R-5%&V07S4%5ci<iQVD#Mjh`zS(<G*9I;#eav)Mu}gd2nP+-(m(PjOc9a;=JKJr6I(W!eGG2E`<BGC-hDggG&P2Entt4l8IQ;4qNcWP{_FUEU!_6l)i!`22Z1G_eFw5Y?*P}+wlrn0aAE7#!Xkc}IWl{2`ao&==r0c(JU;he=5`8`2q~5)xGzzng}N<1sU2^p+YBG+dLEc+6W_g{p%I>v_zw{^`}h6OC5`b_q*E_pygigb?_fOmgHli<yCHL+oA&!BtUB#p%qws^#Mf4fF#NCKhG#~Hh^5zu34PxfSW#g;5qWE3@)ks;B2=QX>Ze})@IZ>B;XMSz;I6MHZAX{5vA*eL<?ErH(JN?Q64<vr4vuanZx0wEm`4=O>k4Qtz)Uu!`kPuvG2@ipQ?ul`#MmrSM*NsSn;Nj45)+E;c!)*1Y(y7xvfN!skn>|m$uSJvI<@2A?Ci1G_&HnU#nl?d<8-Z#b&&p{`?NwU5p>hWn2kk%+HPw}IylhkF>LkRl=YZ*E-?Xgajl~o*N*v#!me%oxUC<5aQgB4s>SXc74*GRC)-NmHYtgF_T0OzBi7XsW_lU93z5DVVvfUnax(--z7?AxdfRh>*x=T8Lz73jRE>g(`Yj^1?GW2)ZMpTD@>_R_-GX-1aW(mNxHl|THlZ6OO3@*ZSdElL8odhIexh(;ozl&g<E^yZW)(zDBg6<Hx7~Jowk(p#TJDM;7IXQ5_1j*{1J;%=vlw0!)zRlxivGUMKDxNR7Bo#Qhsdk4_*?hrrD;fALG8{LwiVR3tDv@Mw|?st)MgReR#4jtYFj~VE2u40P^sHcg>4nJ?PuHGhidaRl1=#KRkwGews)hpcccDiI~v)1KOA7rht3JV>Xp=bfbTRNqAEoSRFP6QegHvBlFB7;-Jn0q;r!<iPwH3rO0%djEEpwyEYbOH8)VoLop{Jv@N-%G#4L=Z5Rqrd^x8m1q_T+@w07dj0c)Jj%Oh+CtI@IPEyGLri$~3|1XLktTv9$673fqt3>p2r8#DjJ07l=J65qw3%|{I{n93Cs`$w9UFO;JsX~lfu-Ynk7-s6)Id=^lnf}p;UwjhnjJ2_E1pQxY8VLFNAXl9H<Zpe`;PeIDi=k&%g9@&~m0kAwOfV=<@%TOkQyf{~rEofuME1H-ju05t$u%mMaXG@1?jvkzz&7?Py$d*8jaf$?{hVf)3P1t|z=!v=MqjPfPZ2Ct$iGMP&^RFQ3)Yv96Dr$J0?d%Wgfx-yi?LT(-*lcOv1G5ECvMK%HAT&3OqR**n71MnbMTDA>@xw{NV4eXpxY6x+c7ZydPx7_M+-X|rn2T=z!L9Hm;c5cUOwfs@+h(EPnj<JWeq~@gEPc8xs1l}&16C;~NMwdAEMc6_!WW#IjteaesO$g-bP}L-$JG#>vFQUI*m7nwtBAz02LmLX^)SyE-P+<IKiO!QCi72^KX6OShTVlxAa2sg&n59)-^AAfWu=Kfz{pL>5_Fqw@DY4E8}jRXzisPnv-vyRpEtBtVOwo&tF7Cqwl=DyO5!So62=-1*Do8@RvQ4lVYP*&bJ$3(+zE5oK<;c|BO4S$mDgyB%hU%Zz({EssXs{8X!UsUwBelzFqzZVaHH;5tXtjeSqz5uk*xO!3}=0q$}fD^I|u9OyK^ZAWMaDl2`Mn5e?zz$HnGLIL8nMeylYU|Z4f;V^$XODv(T*a{m^fQ9k*Gb1Nd2CkI~<FR_wOxhn_VD3qVqz_TtzK-FqA*_H?b|GlvBT>j8lN7-fgVmd}l4%xk5pc+_gi^J_~Y--u%-O9cF=kad4i6q`!9i(Ij+LBMgYIy{leu<G2Pj&RdkE-^mrsG;0K70<8R{8l)9D1px;=2x98m>!-|)K{arc%5E9!MTSbPX-K@9l^~;7$)2oNw&$1;Alg5)D+3B80+i25g$t`zlnj|{21P;!xtMog+x|^XwF#lyA8H8z;b<z1{0<>O0H4~exXz?mCo6w>-DiDU!RNf#bZgvIF~_NJ1!b$gKo1rSSW6R0Q`XqwUHTdvkdrc%8h{wp56KFCC{C*<hi9FvxGr(rv#a89b((7yW6w6`<9n>+p%Zcv1i|-*fT?2J~VyLo_n_y<_}Y0-XX_^whpl$p2FN(9hna8j>&kn70Tzk64M(OttW6XUfq_hccyHOFZM@~Kck6fefMS~LUGsLJsWVEZLb-&*9^CD%>ZU)P5KGxxAOcX7}qfzwUJbJMD^%}a1&)MV0Cjw3Lol(>1`4bfu=yZ%UXTy)BZ~zUcd7E%?lS+-~GGcdDHH<2{_05gmcdTk8La?JFMaN#J1cuB=90^Tc?Edw$hxnBYLS1nPCyz(i0Z#u*tSYk^uuky9LzRt%5YdP6W_s4{lRn(=J>dXc`nY8bDK6H<o>b)#hE+jXyo!f9}r~Opc1?^j|48+!gY98k}wAv{@3B?j=eKhRNv@DQ3vfg@X~Ntv7=VmS^<tw<hTJsGfX^5ql-|@+YbrBU&%DWG|(pK7bL9m<rvtSD*^O2U)0#lAe4P9T_ybwOa<I4ZYg@-XJ)GGrsyu@}hz}$gd*z`t1RNJCl{+;nD2X`7UeO#0zqub*4+k417PlINw1Y$}Mx_B-a&pUY_`^K7_8>`&x>5v(?PE;6kCDM*G|Jo7{e5elebMUK#J}UXrjVleKTg#OKOeVFPba+OFdr7=B@cgtv+1zcx*8&9k~aQ}1wR=k}!iAx_%u8N01px2NeHKTU5f%x8P1-r<oq+mrSOGimQ&y{g}Arp>jHGUl`Mq*TQlJN$Y9M&{eRL~N7`#5S@Ilhz^L%+Tn&v>|{g05*GR3r{_J%o)R3xAogA(6L0drS_e6g^R*cFUT*q_zIWxwkq(sI8RCe{w!oIx7qRCy6=X3BqtST0fygf1V+>&1w_2s2$b994qR&EchRd&UrVD2%EAvyQtC;qzSj~mnVoBsl5r@Y&FA5|LuKR^g{^LlY}`ne`why#&B(k#m?bt`2iLp3n;ZkAT6~jZG`Y5GTdw_}<yw?x@V3jdt-9&zhaxk#EUU?PA#yjL?d1C*qT~G%RgF{acesGx+<Er_dSLgq9{8W92Y#TE`VJJhce)t;4#eS5SvP{LycKD;XIt8BOS{cwzTTF3+cNKaDf4if_Io7u5^%o1g6+2KEv`35Wf-x{xGOf2+_2?UI;A%53hPrVWiR<q$`<mznM9gn%vr3p1CJ%Z46l9Q+%mR-iD(ILX(&!OTehUqkw{Dj9P-reW9cceBg6aNd}W88!z-DWG8zqyI#X5(xoDV%5YfurZ?rBr@YD|aKl<JO?3o*1|E~Y^2WxLU)qnHr{*#ybU%q?u!ll)V&)t0Gn;Tz$vG$h_`!7AacJ1~4<u~9>|LTYRFaCD@o3~dlKHk6l)y;o@ZS~WaZ+!FSjrT6Defe*zFMs`i&p#oj!gRY9PQ`OG+d4v-DT`azoBqd_7yDoRaqaoPUB7bGviE1L2ea16jJ5jgtCqblYsnYXP_#^!RHY+309RoD##3u=US56o0;6u!G0y_p2|5Ch)l0wapZ{|8)i3&AB3d`Ey>{dB`~6E7*RFoEdj2E+s&#mlcueoJL4AuU)PL#47_kAgpy-S5^gsHIqN_qzAF;cv)jyqIedhx)Y}ft@{o*hFqR4w)_3;o?v1m<B>1PELEu)u}CAZo1>Y+>>l(kfbpeXA)7x(H(_`6K_+UP{@%Xq9NyRnvxLqnV=>0!)5&WVyv?MP)=CPn|7r*FP@ZS}QJtw~~;T}Z^p6uHY<d+#!^a{sH}^`CoznWX>Qi)-(E0zVWP#o3+vi;U#dfOJR^*A33H)TYokTVqg}BrId?WbpD3ORV2Iq(LninVHg>5~-bbTJZxecUjlJ`eyCrkELCr&tWbRm+*&TsqEQyNzLc95SR7or&*&AlBl6bq$VZ(NCH*KenCpy3b!z66);*WXm_l|dVuvXh1GXEqtxs$W5RwY6Ya+3_tq}|{^ncfZ+!STLU`}r`%ga&)#J15K79xcy<b8*#o&G`r!ws6<vKwNPnF}q+N$2TRWQwnQaR{!0*r>E^kpdpV+4sn8?`57A60(AAzfGs+Ww>QKEw_WV!67v*fILR6_cH4iy5!_5HFb02?1QMt~w#^guvtWvS?QJJR4QXYBX5D;l26Z)&A8>q@(+P|1fJ^zw&(6>R<XCfBhz-k%;)x;NSkIUtGWX;_7o>05>pWp>jR{()BBU?q7Wwgc9>gx($eS{Jb8n7ql@Q0A~e2`4CIIEvvFo3y++LPIOy{3;YvZkI?|_UmOEF88i8H#xesSA~KHK0^_=BN6id2U0H4c%*q+t0Fpj%K&Ka8ff>gX8VFVPiDdG|dSysq)5MiR7*BbZQLi9`a<D?1AEpQ<Gjp!ns(bS?DQ)y;A;sogefNvi7cQ^9_-||PKF&g-|Ng71FQS-1lkJs@tM5HC*o;I35?18(s~6EwBamrUWMnZ8veX85z*0!{-+l>9Nz3LrLi0n+4W1X7RD%uH$|z+J??|8*&;caym4XyHfufbr*HDh^zURK~Kl|CuxBqbS&A+WZfwnCp#NV4iv-GIf4#d${2cGfK#*0+2Z$|oM%2vtcoo2V;)fuY^ll?w?V)YMi2_&EX7zhas16^-?^K$>qKdip;SDddg<`m2{K{=ZV55Bkuue#A5WfX0%)bN{XYQS$>qBTDSAQmzXE~Dl8Ummg+rQMaGrB{Z+&L>L`fgLX`x=BrBxfkMCwKz%}%^*B}%(JS5Ct#5XgAAGY%mTiGDK|BkH?^t9c$u~Jr@<3Zu{WQC7n=UJ7kS`s*Lp6td`p?r3~7`(tt=p*mSqp7)`q9@aA7fw*dnQbvDPz48Xt>6hyY7sx401UQcIKl^e^%!)SKe+S%Nf0Fc!>OL}iu^^Fu2L?lxM>8ek&!2K`s>m1=G`41Kp*D%ZR1boM^uwZSZ<W9P-)xO%8H+$oEKQUR&FGcl68;?pk#G09HAW%8?^u73R9+UL)IH;xCbSIYCiNWFWd|Nck)&n`i6CE-^;`BVRc3#+euX22j)ST>PDuMeC9JlenTO#knfH{fMRk|ybK7~olewej_D-q!krNmmGAr8e{q|JW+TwMNPsLa&id34PU&#Gbz(W3>&lCxz!sF4KMsek;;nQ-&LWwbiG%gf1qzV_X$ua#NxW{Fsu;U$bYyXyaCk6Y<vw)p(1@m$ZHoFrd!vc$wsMJJr;*$7b|~ZonzIisnI@8bKG=wWoo()?Rsi_0<cjZ~diz@oN8_cW}-nMjT$QJ@q%>THPrDpm<sEebPu~dUzoqZhZ9Ajf+qAFTNV{PX6P|XO%;WyUG(6-oX2ReEFOKI^q+83!{c33E)Lf2>?#}Coir2^<8#UvIBD>35UCDfDzps6-=BHW73euRhNkjV|Nlpx-=B^BWcL>fsb!lQOnUt)zt}BF-A0gHGu{7EIYqWsR6<_?^v<PBqSeNHO2Imm=I;^cBy@$fVB_p#>cSgIYYb_94mExrq~zRxZyC+l4aYKtyQh9NheJgJ^NYGG)fooX{JRCLtWg*IZ(>{h^sW*b~&(|`nUdnyu#gQm)7-bAKV0Oef4GK>r6GE^he_~4e2l9mMP)_;h>Xk?snu3KpO2w*-$eml;E-rZ?DNFMZeNo)cd-L2C=b&C+0LF>;a}=*_K9xjc)eE5K>+b{+=;_ie3(encBw)q?m65ohDhmA$_BK-`t5K@al(nB$AD$J1;7z41FyPN@aN~2YFSOvsxPzr1R_6ErzLwUIWjuRZ4ZY?A23R*vc@VZ#AB0)JUE5@_Hc*iMrpKP=^^cuPEE=+GJEO<iuakqE{wlaZrQ|+248qkNG)q1$pfI0MmZ3d7oxj74!MD60GZX4M@*@i_s4;+71aDGO0dV<j?`1HO4I^O|A+cptq`iH<SyZ$vZ!s(o7{`b_0<x-;5$?*{&4Uu+>@ez~=3^b*tHJl%ZpR*j@~A&ji1Uemc56x`hIp!5;EDl_ihSA(F`J;%l^TZFSmR)VG+;lDh0w*<l<<YWs+1nkKl*IvluFCN7Qb_w)4UUDh-j{fg;<aj8$HFiFeu2?7+1Pn#M%M8BOQ?h601)23ED*XMplN=^-Im~_+tN~R_?86M4=US-(fsq`ADv8{KB6wQ`fuWNk(CA0$nziuD6vTe&y)XeC8NvS$wpHgb3_U-#{-gO4KC-lB2Mm}Z?@~ZYN=&d*=M5bE-&-Bh@FheboQs#2FqZSr)<)JVrWBvXC>mYV@Z08!-&NXny$=)BTXGN{4zV8{PbUCs~sTKpV8DVpHGot49$&2qw3Vh468D{H_?2ZoD+hKbg3_qAq&Dwa?tAPnlwm1xZZ^zUQ5t&;#3x=xrjS;)*fihX)okGo)ddHU)s7Kn>In|_S->~9Caq$7Y*}hxXKVb}BCzJ&m-FnAA>$ZJZbC#N5n5x1A^Q<2NqK;;^E_Rz0MBc4S5batSeah;NdKOZ5S-CCv&$??JNaX(!J0YQUE08vX)4Dyhx*fl6Nn)*w8OkSeVu8fywEdNW4Ns*N+QljLvbYTFxEW(NdcRw*bkS8nYs7ZYA+G^%ATkik?koqenry)Q;}PTc8E3#2GPS98AI<l@l=p;Hv|Bj~tnN0ghUbRewg;_toWYxu%_?1k0JTyucamF#RY4iP5rMBzn4f2ickaZH!ij~fH9sLpM(-Bn?C+*lhM$!nw3)V9D+kXZNMU|PV8)BAWt3O2<t5`Sry6u@*-}=%sHzw8lQWmeM4=X-VwPa;_*GkTbDAUlfY<cea-gxih2=4bAGe%v!_)0p6wNqbT$4M~gU4kB23m{_3@P1ohRsssqGcO9jK+f=slBqY85kOXUhftF7~!XC$lYemvvWEn#>cG!81i|?01~~Hv5^?n*kdqAd>ShZbyRo5tcb5m9NlcMte{RpTx}g!Yo|%EJTCWoJymva(ZR>$Ht`=cn@MYPz<A1FPrN^dD4|6ORToK4DXEnu#vCN3jvqLL-d7yRvQ#<N17d?4#1vk2SMH*?93r1oTf-QnV6r13r5CS>`Bb&aJ!=YO^weoJ4Nu3Y9f$q1to1Vl5C2#`J9L?DWOxh_c%l^J!H9I5Iaay211k!oh(4daD<U|ASgr1ZIIn@Nc-9yI(P`0?ynDvZO*p%OrBIx88@3QkEJF;~cvOmC7c%DXGej+}7l*z&(y$|i_D4gH@%KQ1>Igg_Dfdta)bX2Ec*a|XUmIeXYK&b%q&osKHO-yFCO}yo3DIYGo5C3Fv*-j)T5vY1K~jhF`A9G3mCB-?7A=DjswN8|8bxTpfge=D<Og^gF@qY(k{lX49*Gh+Wte4(m@FK4Fbq2FpygNC>6NbgT#aE=1uJM<ZZmpRtfA%-!<yl(o~R%m2gzDNv);2#6MCm{l)AuZ4!`J~rp=hsge`saYd~E1z5yM3JMF4eu7xhO*f_t<{A(THtnX8?eD4v3?Q#&*hZ!1%JS1KpNfs`0j{zradCDY2)+)jOa8jgNRnfUdiRK{XGLn(GVj7vT6HGC9>s$9R**!?R#)^p_<5Ifw^Kj`MO<3j;qWMN9Z~2y|u-5RvS$q2Qap1-Flyw};w}uC{u5~c%_zm71I|Ot8h$j222hu<u&~!w2`t<(*flxMz@^o%V0`kyd478lpJn06h-aA9OY>SW$iI6R{7hA!y*S0rgI5LZ6wpLEH(3Et$w^1~S7IJm}jAz&sF`<-)F4+0ISN_G8-IUz9=(W9O#hbF0I-OQHH9k&`Ha_y|t>DS@n&(u4#`q&$FC=+1UYMBNQ`kH1hdIE$?d9B@M1v+uHK$L@ZP~RgyS8Q5Fxl1iDn7>G!3yY>WEZ-BwnBcfm%g;l`e;4x4tIK`L@ulmOZP@BO>Scj3mT<(Gw^|IRF<XddKHEV-&vv#2rW4sH?hJCM_%GZB#cSNh+0BQnl#x0aTzsj+46wV{DiY7Ywcc$7woNHP~vCBOXhIf$Z7dmOZ?D7H=>c_3Wh)=A403{R{-!?CC{i%Y={A!dv<57yi?e{uz6WGFi$|r0n2>W8v9u!U;)>#1Je}7$_lDg5!UI98DI>eW4y+Fu~&E^f9zyCXx7XR5{`H&#0uXG-fjAH(^MRIHUT?D(jpg84g!rV2C}^*vb{mb_Dp0oSSIf>u-H2dn)yWsC)yd(|6Zd1y?Fl(W=(^Xz&4vic9?eQDaEpq+M-JHqNMl;t}2A4x)dR#8<OgmcC^O@UW`p6nm<~~2{i+e8Ja;SuENC_Y6W=!^%vUc(Q7L)bo@Gv+$P{`0<Ly$SY3b~o5D&D?$mp?wMuebpTtJCLpG*7H?K%<L8%(28NmG5M70g;<*m@sLtHr9YI#Jp9F4aD8Z_*^+sb=ed2cIk)~z8Z6Hl8gOq)FXzxFP_5b4?kwizf3^J*&QYe?&u(M(pY1Y?~TSUsOLk;)7)LPm1G4>U;F)wOMwd^=`InhSCrHM=3>V?*}F#!QUSt%{8r5}UOh1TnYGg~8^+ctIHq<AwE11`s;Cwhe}DgJIiXAgwod?7*=pkR&h+4Y!B<gL<t6V<XQfe2<KTL3>}DvJ1AYg7vL}QS5?-zXCR!P+xLr@PNxOjVX!&AC^ZbMm;R#hm|x$QPE~4lUct;Zv!>VIa;(6%h{o|kr!)GYq+pow|r{@<>Lm+dL{^sTf*!}w*+uFEgS!W(sV6)q%%T+%miq4dMz)E;TLvWbS`3k3?|+f%mCWIz#kaQhyQA3KR${HUZPSaEcK0HEF^G5UQ#$#f_B>rTLEXm5-4H`x9r#bPLI+>dv0aPY((qJw)5m$w}k|uk@^^M$x^Vg6!;a-6fF9XM;UygtdtC)Xc5~@rAWiU+mUJfU@981e2A@5;_DD&I%>ZYQ5agX|2X!{>31&vdM4@A4(N;ek-L)W2^ie1n8X>f&K$O^9J-3sBzNq~AR@(*35`7BdfzuN`afint;=M(&KQYW(u&-&siJA>Y|?MAf#Ek*+#TK-ePB{7L^LHaR;g`Q{j)w^IlxYbF$y{XX056>+8MBl-^Rld?NU8hZh>Jf>`J@R?F5U9v4c6B!ml`C>XO0%=wh>CaVD4{L21>nH8dn`_q~%7`SC)onv+=7d%3XJXn38rU&#T6IYcpMHWV|U81X2QOFS>OJ!q&|>EUUPHU)x(mfi9#TBogKhGv$gn?8ajrgB0D51vv6t<7$Ntnw2C)v6P}!$d-WCp}aPGsINA&~MY>!x#{ZI}u0fZj<vu4_v|nxALs0#?*DUQLefc9=lRB+$qoWI={}_>K*dkBY14vm=N}gl{1dZ!DBu{x1~_*mUXvf_l6hA;BUJiqCx%J+f>$?A9NsX!HCD2PXRNI@oDMr7cg(BAQj>QJ=!IEw4pdJUtY{)-LE4TyKsgYFIu)JnlsKQ5<)354v}$XydHGKmXNXvCeSDC?U18dw|L-r{nOXHe!soPuKB$6Kp@fH^U-0*$Wz;VuM4jRqx8De4_o99z+8g)2@E23d;CR+{q)#j)jkz<2Bzn(S@uyr$}a*wXY@pHBD!=bYb|H33MK-gg$*Cv%5v`V$$~R5aXxpR4&Yn1h_0=&oJBZ=-Mcf|swF=$<Oo^}O)dbyuz$Vb_zHAm=^YK+6XC3q$Wy6qAyL_mJVf|U)82`rB`;K&DlEV>K-ow{8OQEAso=bo7AC)}`i+oXOn8ph4~qpn6Wj9Kjy<W~q+d+8A=Cz8B@d0WTS{FgQLV1Sl9jsOB7+J|m~OKgGH0&GtJBr$Hk)pqP34%cFE{jo5d0H#>54+2>)kOYD-I^vGCY)8*SHSm8-OCVjgLEtdnSNQ>7v67-7$pZGWd!NY^*d87(7wcyl#>YKp?f@G(LuM>q^&Bt20EvPOQhUmVLZ8hbc+W^22l%nV~GKt}Qsh3{ZyA&sxX^F&;=`S>{f@F5+J;1xn*xokyLaOsZ-R<<EiWjmn~Biv33b%xn(j#L+Ljf+-9u1U!CS@od4Y+(*h~o=G_P<aUd5*7ufe1#<#0%=7>Boi7DC8q~q?frO=~9L}uh;eOo6NfEpQ$}@K5I8Ldph&$-Os?%mEHyBMyqf%V((<cQ<i2I1#^M$Ehz*1FQ+ZORP{el<XS-&FQ(9d;rm24(@(7Q=&qJ?<Wm)%~7h>H{Vm7or*7fsff{ZC-MIfO@~nCu1AxAR~N>y*D+@9j?Q)5etKI+A6e*4!33w<L1b(Te7%Y^h>oKCZy>c#D*y4)FFW$id4P-mBWcYDU|QT;DKJTWZiTl;AWPxRVfRUhT+LY79U;0w5XxkiMg?ni?d`ab~VB^0iBX1yMI_j>YzHhnm|u!JHVy)|Ds{=M4^z*2w=<<8e5}OqsF)o{lu1afz8`(u}48ZIksupP6$5b(;A`W?V^_A%x6EQAmNN*)sZMi`^12e98=!)$7ih!J~4UCQe#|TRC@rkox5~kRS}2mVII==wvqGENd|ULw2EIQiNe1*;yul#GDW*`xG_5v(zoq=0sTW&-k4jnJ?x;xgM0q8?N6R56|!$OuIPFz+1h$4o}{bJ6y=`{nnfvni;o_^_HUJM9LiN<dTQ!0HUB3vX8k_5g4NK9qq;A1g({`1*dYhvmX~F%r*FC|FOfzW=s1Xm@PaoJN-))n8vqZ;MAl!%2#1w)Jg?DbTF#rEOeU#T`HY=Par@IDUFAZqs>?XR;0wiP#cVvN4mt>(xooG7jWAE!Q-v-cp^Qh6>R*LF+@$OOHeSpK#D2>3M`v-5K{6XsGOzJK9Arb6@E*4gV*6sQSQm6w~TH~VAHrmQ(so$sxHdeDwkKoXZkJpAk%06_N@F?oR)(|b=zk-NXbQZMyj5h7@X(I?NpX9OweHXq(5OKcS0KTCl~M5EjkyDitCZu;ydR(k<DcslkG;BHi_mYa(Q(L(`c<skmwiEB~K41ceLsDeQ>>g?|q}|`S~zczHnH^bAj{pfl)8Pbi@4V7;}Q5w?!S9&FcMF<A&L;-j61P7>#a7t>)@M(8BPLn%AN8NNgD(4qY!)QM6VFE!DM{S;Uc}hKtrb8WoIDWLFSI^FHZl{NF+*L(qJ+0|t{|VF17(RtNzh3IQBdDF%qSlfB7c9mur&s(5AyyXeap&7zSMmSzq9JHVB#>>7HpzQ3C{Y7v{sO?bR-&612W(uh2COZ>Ag?~KScTf#CCyKIP425?C#gABHJZ`*!C$~G9=KLfYHNO2a0icIhG2CeBdlVTKudSV@88o5ogJH#Dmo7}+Iwm9x=i(|NHZ?f3-#><EsFPrs`#a)$ymRZC6qRL$@1Ww8pOay|rV6^8uVGCx&#bt^wc{B9dv>$nRW83>L!wF_emW?p<*6qI-e2@Dt#>Alg7vs$xy8kk|#9nX8j6Nt^Gi%b?+unTH!aJ`<i@g5n@q-V*>ZQ?Ydm);3-_G4vapi~H`0G}!q#?r0@krOj6Q{+!DTeD$kDHjg?T)|bRyyIg<`EXyMLyv;N}_-x48nS_?7@mkyv6NidLKXi6&2A8#=V<~VddqjyNEgO0k^H%s4@0QXq@!9(5wVinBJIg85fVR8nSV#X6DGr*_pY6CFHo$&mWpTJTv#pj2dAfU`tu7AMN<yrDF)r9&>;{@c5HjMrVzdGBMQT4hjRQdIW?))0IhS6OdMqDROccXG%f2m!bzIG6N2dyLS@58i$?dsn`OncW_BAo(8jW>=+}R{2I>PCPNQcjVb!wF-$J!cA6D!(2?wD^G!Z>soQ!?lu5nfu^xn>CoXrSs}w8YS$SFF!FC(vhIwzJ{^T^9`W=gIXdOxvpvxeM;ZyF8OuH_U?~~*^AbF!ay3)}BJs<GtK8a+h+wgjB!$l}HuTu(p4NSVNU0UZiOJ!7u(A^UH&uG(hO3@B;dgB#iLKyHvP_Hua@`ZayfrIpsw+#i&PS9<KuqyV!<bYoGMAb#VS=}$b+6;#`kY^9h&6W=SV(#GV(doltPx-;=nWLrY<Hu)@{i1YY=E(8G2cv}%y3}Md89b~vp`Y4nqp%y>=T<dTx*>WA7u_?SRW1iBj^m6aFB8@Bq{};(U^eK#;E%k334TciYq%>V(G=!ucUSDF;WJ*ZNW13zX1eXUiY+T0K^PbU{4k5e*MNzFfUhVAzhGtC?1VN}Ha-LH@S$bHliGmC%ge>Bb|t>ZJi5^w$}w{CT;i0qf;KmouX~G#o2)iJC6iXQ{o0c8r7*Q%R1x6uDG6UqNxM_eTIG6%_UjirCTszM9*4rfTLjt&gNiEd=+tpFaX-L^e|6CnHENgPXGc2;*lCM8?RXv4h+T%Cop$1;NWn4)oC{+k3;K!RcF-!(M76_qg)g?ni-6@E8yg#wd=#FU;yLfAs}AUjcD!%RTj_wH%32Se+%MK+3~Wk>682Yt;04a9cv3y>fAD0i73P=ZtS6LY=R!~jP|B_Tx*K)`^29QtQFAVGd2vb#=VQUR&{+TzyHa$o;yk}y05d{{G~{AlST3QPr*T{~Io0ZfdDG^Yha#jE@ZY@nw;=wVRC9OPuhQMkIBTQ(3Gn&`8)0G|q#Qv<#>-l1MGC6O<bsR<S@HxkuMc-}5Zt^fQc!T=u@LL0l2z0d^Eaa*M3#71Q1XFxG^FYfcj)oABjk`&%7XC8QsZFo@wH7}iM`%hMu7dmoLTFn+&M*V;16@=B@}pQ_avFCxxBy)Up=Tj8&}-%tt0XKBj&7s20Z^c{x%b<#9W8!vP3hBuY3iGMtd({O@xnhJ&(Kw4tfg2UX36xybJ6Up8?&ucgS%UL8p0qJBPTlhO3`IKLlp-6Ewjw1}mf11lbPuQajY?re???2O*VsD~X!b(aVafE!Kyluaf25)WC=(w`2p9PcZ=eg94CHnKI{O#1rwB+@eaEtbC=tjD0`R2!hTMNEsW9p(8=HTldm3=YZAS#S3JltvTIlVX7w?Kp#pd7TWAq9&h-dbN!Q5i+8jf>kwrzz%qVlwY)ZRHKqgbTIBhtM=6smv2%^vWt5_l?gi)Dg~?EGt*Y1REG?r81#eLg>hTL{<lZ;sBLK0;$q+DWk=zPdjLL}XGN~DXZs=IYXcrwMP`z&2!%Oa2j}zWCld!;yTlVU8XK;Q60~=&S*?ui{H=A*u)2t;g_C&VlWKIl2b0)`uRsDu&UQ5X|hd2AEPN?gv6-laY$4y_3@CGXmWx=w;79artV@m|Z!732J$TLt(CJ`x$plHCPC{FI9ZYh<(+V7M~u|x==S6_^+Rb*3mhdj=dcuF@tlYl5PnU<1fidQ|D>d@I8D*(vJx&`sc*vaeyL)#fgA+2`PlR+v51!9@M;=EHdGB%)U21Et{7IaM95Tys|&(jYDNDt1KOR}T9fT`66ApxdwoZ;?@D#$ezmZABq7|Th?#*mNR$vdAzB_!1?(DVE(E|+G4u@l);2qZ*^UVbwzsRb=)MG;`S4oIp7U-Q(o9`#!(nIE$eT5(D?JQuTJ+D3+r)P!$4i+r0IGzj#j{A!fhqadGJJ{@1@&d|khwrw+JIci5G2LTWnnZ?5#wIjHG!c*Sxr-8&5K@iZj_+Watst&EZGdPpuxE7}r%h$-w&GN<8oZBTu9K(`&*^5m7ls0&upEvn;$^?@X(KR^3q>RCpGaYK8DYjGQBXNM)><o>DfiDURs9W<x&_bnhcX1JB>8Tw8!yQsGWnwjI?V#I2ew6LSCNqjvOh+@76iv9OJWFN+{>WFO666XOH4E$3EDWeQxo*u#qo%Rh$et-2v004KX?Wx<0fHRE{Lgtb`EcW4QdbHtaTaA<#9Yv@9`^jRpu!(J=bYN%pgxCjf$|aIQFz1qWzbbNNo!WJ`aIIDj5JFl#oFL}NXT~tEvBGYjobRFOF0EGaw8R%A)1nIHIf=s61p1)eMo-bxI1TX--I=$2FfN}vtpVQ={^4@yOugfLXM0}WW=Jzlo{QgJK!(Uxen07<+^vUTJsKg9auNQm?jK6L#6Vtu=btA5ScJ2g8)238hUbc#YTvQjm}bO68JA^UTXS1<9HyM>`ZE92NPHMV6b9nQMF{tEMU6iIe{pyUH*Ollb7N!2zm0jG@bo4d4zn?4Ub8b{B6t`mZ#)CxXQ+#W#0{rv|};2y)B2!q$KFJMtEZa8D(g+xXhGJl3%R8`bGbx|3j2nefNsijG_p$tq3sV>&kgaxBd+_^93$k7^$#=ypXP=4YpB)O`=ywaYm;$$Scx`k;>-s*pK1pC21#q=?V)5doq*O5%DXfY&x@wDbKv6?q3${urh!XhgRxkhPE_liVbOJ_^eEd`4ycz8AO;E;|Zs*cW)Lg?0fIQfAV?C=)jQxD{@V*=s1}Ol?7e6BJ_QVKE!cbF_!4j_Z3aM+AC55hlvuHCk#G|VWj4>bWuWatnDq<LBQ&c>XlNtPEKY`p4lrIoS!%U{k7FsKVAL!y|vGuzxnzL{n!7jvjY)zMOEa{vKd@PIR`TH`6*;y{JRjV+%qfpMy-68wfgsK{pa89e|Bl@ji<a;=+}d0|NNKNuYGXy@o)N1zp(nsXZU#Kg|&}A?_YjHGbp$$TJT=(UhO^To>IfDfzaw!wQzSdcSS!%mO6pBcoOhpqMor^{o|+os~_I@)8qZ;{*0Kt_MfYt{E=U-J^Res6Ia%4p1m#RN6poe(9tn4fXV!W*^>+UF_un=3gH%133n|RdR3_1to4hm^}t-#+6SLXxRB0TRFovnsU7vgsU32Xxb=Dj3s+`hfXd;c@bn9rdP}^e{X&X9jY5*cNA(F;E+Z7x@H~}XFy;;Vg`rqtdnSp_-GGlVZznKD1MHJu$Qim2TZ0D}^xZYqhWpzdr36QBxpMPJ2RHe=DSW(*vD~o8L~pu<#yXgtTDQVfDk++jks#2iSa-1XB;1&+DAA|5OT7%ED5#e)E7UpzYdkQQUS)-_hZ+W-*xYmyd}?9QY$o2r)jyqIedmK4Uw^Uo<!ku+zkaiJ?RTqxcx&~wPx~)@xcb&}tFJs8Gm{s^%<#je+q4<EjN>+YAWeg|@ljwSzD^~zEUzTw65^<P)MAV|-x&3TUr^q0c1G^VBoI>bgOb(O^J6JZQ5iuSfi;GvD-6q_9Ro~jrWokLdZ1G_h@D2ryeH|A6+@bZE}>5ZO2Nd&#d@N1DXXuDZbyU847!wY+2d6u6}!zd&0x7nSW&`wYwR)Ai*sXX^wY;ob%sV$Jvn~ILw$KaNbNmSk%YZ|1=P|@%tlE(dDAumxls<FKH$!NmvaFpkT<3_N*vXeHpvIcCEl~rL1Y{p1v};i*Gn6)WN~d%(5+WWL8Z;(jZ+SH;Q~+xk9r#x?GjGh07?kbK~i7r+=CQrc~uesy8M&xYaQ+7`bq-Ljb9{G0|8MIY|B*2ZPN+s2>t-CAMcn@*$Dt(`F<?+TTno^=T*IGj4{jl<`{~1UNZ28erz$5?3cRHz7^DawYb)41tCMn^v9tMazzX}?z5?f!UCI}s^KoQd@os*OWEaYCEh+yR)C$jtc6B`B0cK$<A4wr%3AZ~GPGb*l!*M3N$uCjT2$Lp2kRMH8K^zAlC@4+QG@Fp7&xz4P1l*xJdnkb+Hysn*D@#5-CAYS{(Qj0%Z$9LPvSkaD6j{K5Qzglw#BOgLFu1T)2p1p8E?uC2S4H#J>p@ni0dZ8rC=G9e7RdICjA}FQp@$*GI;_4kEb?<!BGeFj_sc%X=Kbz3FH36%)CogP>1|zTw|Q7x>Q=308FCi`6&T=VPFlCDOwFtVQOLX1^|duO8aayk~7iV0%L%i8EOj{C}d3xbi-h*88p4IZ{I916cFf4*)L7y=}~xE>C}Kbj$RWHcgc`9{3%1*yp@&~H<uPGH)>2CPdeTJ>UIKLH_ApH#%l`7`Z+-9+O?PcW)-HDwv{8*9~DspJ15)`yz7F=)wP;5(K0fZIiE8qs0587Z96KVl}OQQ)ibduM{aqX|D~-JBN7P}M~SPO=!@M4*@062fGCEMAgLYBJG=Gp3F$T|6||ZW7^l&&wWp4qQHLtb7>uMlMM0B8VMlGqqs0P!@}L-4>ZGC|Jipcrx}hCQw9dsMN|m6)4UE9ZQ<zaf0*XOb&EjLp1yca@jB$Ntqnp<~K@3sCk{;YRJ1A$o92E(x5Q9~?1z7n~_y`6lb@NUOxS<d=B0m&jAMz!}0Q<BPziWBf1T>?gP-v6=#VtufP-~%(Y@xs{6gJyJ!E7O4Nbf;m^F2^R+MlPb{)s$IY4NVW?<Tc($akfN+d}2To411OcLNAIHKl|@o?ePu9c(e6cgq1(<)h)bTO%ab)H1*796XiWQ4(k=p`M~#Qj6v@@qeR9cH+d$k(4>cTOIlQkXc1zV|v~gPQ!2F*{FuQqLNY&pCrl_Qp?&UZBE1QQig-s@<tvIC6Gptr%~n*gyM&#O1J9ju4@TP$TR#*eR0tL$ICV)tm1`HBbVHe%A4}+R^*p9LdQ=Woh1VVR|_iEvQ2@+_>Z6`9Xx?GCM<lHK%Q4xUHHq<b~0iPEi|y3M=kF+s*8)F5%Ru>-Zkh<TNK{V6l})jWz2#gIhQuqih@4*i3x*Vg$WraQ4ebQ2|Mphz^p<5f*L;VXyS!|r4oWXwWCzRTy>?=sU4Cm?f{GCtRLbDgGb5kHSC%ojBFFX;c3Q9@^nB5qf@2L5g-6xRL8IA7Q>>tyRPhl;n;0sJj0kX&aSX<>{Jt0YYq1dtTWo7Ez4xVgo5QkDL5m;2<5<I6e?ZECGtg-Kz(<_ezZY2@fOBNgyF%gvRNd_7w!s$OBmmWw1hG$z<A2E=1dzoRx|>Nlo5-*jtGlJGKm|Y=xYlGTRa01I;F+BTa&pOa4^CSaH@QPNUGoMU^$F7uJr+fk;Wm$yoD5~cb0yMLzE-YU!}HaH#DXdG%gMs79%p18>R+Da4HkuoO0ZkB6Y}61(H7cF-i@6_>Es3ctEP!s(dEBLzIPtG13NxgIS}xsXAp)wZ*Z=BoWVzKZaPGb6U;XSVTfKs2VhUM`oqLCyOA&yKR$G)@O>1j=u=|L=PJO6pPc2->G}_dMs%BHH>7&dqiQH<Y4B|O{2`^^1{#!F_J=BPrZTy0~Tsyr7<)KI&B+EIrs;y^0wEiyA?d_hmDUZ#-JoMK(&>u<VW<pgVd(zT0uP{f{h$`<e^AmJ}1@Y*Ee4=(^SiAm)fNA?$!4r)0td&N|~vn%9%~EQ|U1tnR)8-!iLyu$GPkdjF*`^F}q{xu^oqJ_8&ZY;vmTV^@0bi=@yX4vkrsSZ-(B`XoH724UAR5n3|uu&92+-fgYRKHMkb$W1SiwUtV5zTx!S(+O_dIHx`bIk)As^d*p;Qee{5}|Jcz3GjlV?j-Id%9h<ctI&m;-%^o~Hd+fkN`|)*_N*$OvF*iH2?;(66K;-GnEQXb!Ps@p%J*RfauI<1Rg7rFwII`0Z)Gd2t@zPu24FfjohO_}9;zIFOYzYkugRbqg{W1jv;+(5uyI?ewd)5h0_|)=vge~aSmaM%TR_a5!ph96_DaZxwBzBdc)#C$7!E%#x&|oZ7r)Rm{4qazQZAwsSRXR)P1rBjnnv+&_qLDVS@M><|I!NG?m~@*_IJV_h2$Dnvvt&il<N@Wlq;I|#a{IUhZU-PRFxE~y)Cu)0_6gs1f#ET=R-+NQUqwmjZU#Wyq+=bztvvKUciXK1xF{m9Vt{PaH30{Dh5f<k`Hb*hFXOfc2sJzv$C8GA<ADmR;$ku`EgG1=5gpoKwl!fU<5*(bVYjj*kON~^<oN_<pq5<%ZqR3$68Hfa+Xq}|q{1b?g`gJwMIb6jkqW}LcTfD7wz?WY+lxgf2i*?FMbi!d3>R2k5+uMZdrcq(Uh`*y*Dy7P|Cd1*=wm~D`VGvO(Rl#=hr~JSSGx$YZ5iy+l%fc*_USAG;NS2=4zMKF<H<o|l*WB0==>O9K&OIY%m?=PqSr>FoW3p+`Ojc8jiBl;($&k*kQ#BnS*dq<W2D=mj9qoV;Ufw#tYNU&S*DmzKC*_#!?S8)3OBI~e!>!2DIOO6TDQ&T5*B?B(nbf4m46M)k<tOT+2eO?71N1O@+o?_4#Ep4)%2C8qFl*j^ro(kLbw)_iQr|;Zb~4r-s0}0k5erm{6ueMUXH5SZ$rdYXp@OY5CHMCZh!^g@}1>I?@tEpGfAPb3~vc<5?088(c}O%E;<uU4W^;!UBj({q&e&Qb@aXo`Jf4}EXrNv$%-qO%Z=oQlz~7ED6NDbQ;aunQSNlmHef<Yd?gGU`rxh5nJ7Bh(+==9)~Y<bGfi7D{tC>qIw*UC7zezzf0l0pN4kV2VUQoNp>$FR8c1MF3W+{i3spRegwwQ&t>BaZW@2%MjnUBI1X*74D@(exfRO`p5#>SKW0NsN&Tfi(Y~jHJN(XKE1Te&y>vK#(L(S-g9v-}bVQ`_jpiVQzlJ;cE=qKg25@m~qXvkV|8VU+h_B|d%f-m$-+rwqCnzkP1g+Qi3+E7oxT*RYTb-#Irh*t*2L6$+ID<g;B2PxHF1S=HPNmf^?in0l`kV0OtXpBFeHkZah`q;-Og__BFql%DX8cOY}kU?n1;1Ew&g%zVf1xzEV1_cInA*;=j4k`#->_&M~!)_UbijXSOViD#!AWsO2xj`+FuuqsHYP?B14pl0;iGyOB+DGOudqB`dpvDlhXEX}|#!7W73TS5=L5XcV?Sm1K?Er*z76+q@2cmI`wXtTCjFxUw41$G|!l1nukqfES38TqF!w_bNDy&GT>8-&3BCJ4G`gN?0+pb|&*KEp$3VUJ4YlMb~2L;{rP{~wCnTjIJ2dE+0#!+UPPW!AbS`2p3h|Z_)!U}HW;Q7WT?QfwZ6?W)Jl7pJ|@>V1v%<f9ogx-WLzk-`7Fr6E2`;1prl4{wH7B6@$bFJx$Kph+@qHZcvq69m&<0w$wwe-2`M5q0XX&txJRrlqjT;EdCNr|$v5o_970-Va8hhb&nI11pdH>Ax=K0?{4W5bmID9@@Lt!8aTN)c6XtOw8_!<P3eLP(p;I?-h%ELb;neWXu4U1(FNi$*33B@5t91|V4xU|B*>!8mMrlvhB)0ijs0F8fvTdNp&z1|f7CA9G+QwfV~Ro?Gwqa*J&b&lQ3}cs8i;E@@ovi`6AIDc5iw?P9l(uM;9r6Zfrd87c!UfpJ@P%!Q!dh||4Uq<LK#A^K|2SS~0zO(u7u%@nOq0!NP1q&7~+U6jZ_j6t%Yp4aN&q=VH<N19ZCV#uqjjMd`qYJ-UGmMkApM!Qg<M62`w&j^u@@anj$K!4do#^a-;gD|m$AxY9lbq>kELnYhAGgY{R4e;z3ks4*1K(11EeIPoQ)%ypeBq%zGl|*gAYz{-W?bBpk#B<|vVdVKrJL;0a4l}U24w|0OnIIRzc+pQs`|WrI)h8LIMXU%tV587PPqeX&;~|xyW6dlg8$=7L5TqCKw;Gy`Ut=WPnv0)F@ClEA9qF^kCT|B}m?NrLGKV&-7U|i?)Ae<CIqdo!>}}nvv4(Iv3VqFTh>6N!LXQ*%qaKR&y9THd4N*__OpZ!}Y+HcCR-4I^%D9qCTQO_oA|;w{LM2N3i?s&MR~$pdGp>F#U2&lzF@VDbXH|-ZU3f}ScR@MSdc*BS(k?ETKsNf)3Nl615Sx%j2CYnJt=q^lLx~K?L8P_KRXFSU0n#lSt)z%58k03Vk?VXhsKc6yjQ}amrXo#jyBY3o7r3s5I3tR@R)GQbp`R$obZgm6w3q0uOPV9T2>B;gv#Qy$)-3ymA_E-PVdxTIzfl|636>sy6M2T$prMWd3O>c*2ozlbHKK{ffZ~l;w8q<B2c}$Bn%kOBfYud&RAL3HwXNEWm;<wW)FbVg72J-gg+SP%m6g>^4&vBOGyz2iqaeZ<MhRJKT6!#2B@jZd8jukTDhPW@p{+Crarbi3>`YRd$4OVspfppwAeN6f>R|L8o158x5X?D>+eFIaToetVt*$qw&p1sqq)eU!g=ye)SaK2T!f3ixTC7Gq=%tburIm|bW*v@@cO)^mGt`UhDD>6gq?0<8zQlL{r0%&PF35}x15qcM!nCauPDw;viDg9ah)O1hg+sCUCtcJV3}?)B)4zp#C`$gIR;@)kGTA`T&Zdysm6oCAMHB0a6iO||Cl1*eu!!U#1BmTv4*S_tBgEUSm^rf5@?2QT&MomW4#iput+in%8-v&6%8v+Zk7x~ymP`yGG1o|r^^DyCrBpD5CHjPMxumnE4n|@xs5Hu?$np?`Fd9a;XhG&u?nJLy?KY&DYjBQ~HN1qBgPIU@L}lp{!N|+BiA7#8SZ=Yk(C#J}k%=>S_av22(Q1V3T-qvNJ0a#%(?<lMh|cI}qrXocg^%pw6tiD*9HwkxF&u5{Dp*XxC#%h&MOtn42CQLp*PohK1Q6TSeT5MUiS08(Q7cOw9oJ-iBSV><9<{`Ytsi3JvYXaPVi$r}yOGrxI@Uu?(6b?powovQReW4R6HINBCi0i`;%2AjCDYtv16^X;0s-5M`-a#wEbE@`4Xx{hO-6`noWn5K*sZFaQ3h}{=wL0iiA0*G9Po;c6JMdULj6L>QL2Q{t9ra4g>yn58POD*Anb$%QmWNd4OWCS33_6-&~naO@hZCLCwZ)h()Ma@n>V=Pix#y3u?Hliv{OQqmYVfl4M?DN*ihGYrI7q$dylOgwSVF^&?BcTA9SaAZ48LB#3P_3vjtIM$-olntme!T3te%{oS6d@eh5Q@DxZcHEQ6qd+vtc7&>C!$3UpEoNVHr*Z*al~Ea#RSKA|U4bb}<<J&tw24{71W`LRyoA)qGis)`w*ur2p^4NU8IT%c)!frhin*-E5;EgKD-kOpCdt&QlS^C7Xy)r+Hh)EMOq<z8!otu}qanmGZc+P>)%Gba?0PtMFec<iA$>*Vz8?DWyOnS&>+W3ze;!LdWu^wD2hKbtvv0IVXPcc)h<8L!qIpQKGybHzq8ihQxIlwLhpDigtIIn_>)y+G5snYqIUvq1i%xtXJfW@nB*aPY{%qjOp7$idnD4+5Uk`(`i;HgVITnYp6}PjIxuv;=Z|dKQN6p~KU&*71jCj~_d6ko7U|T-I?n1G><HW_{Y}pdBGzSjIR<I~CTXPA`glbHt00DAs(k?8yv6FrBc+N|5^@3H>mr_-eJy@?LEHlFL`$`%SKhB_H1Fz#B<etajLU%RZ+*pF!npftl3o5Go7>Ue(E$3TVQLU3cBceY6fnMQQGbrdO-`HOeoSRh#iy!($xzlGf#Zo2?1lL#X>@aaALrS3}=Mv{fP-?O>opNW1sxvxTLqp;;(twggnCg9p;NKtt@(wM_bs0alk$3sL++*z$1iU*8gk`2}_`Z(X3F!(L<D5EM|Q=tfr_V2+OQHt*)6o{zLSrE82WmWiZYC3m{~#&3!t)x@pgd$5OY)l0k<#Zg~`UJHUMCNI>zl4n3`2dx&Rp5^1Xfcv8B*D+}r1q;+={3genGzUbj;HCreu}%|i+6ytv5Ezm|6=VKA2}C*V-RfDNwrUqeoC(YZK?oUwA>hl|u-CDsE2u7z6jCB%<I|BU*CxoxCA9X<*_+s~9J<*jP1ed%5U`Jx+^A*<0Qq#m#9Q=8!hrZA95ELkccjHmRv{64#DQL;iD59BpN)xIm*`tTxh`B=WZjITAmUsq$d{PPtl5zZGe0zUnP6G+V6coUC|-vuq7hX!$Wa$5{-#;iH(!*IE_PwaODtZabd6-I&l&BJGu5VzCg>yAlg1Ci>x_#nf03m&&VQc%M4!coK2>iKRykY+Gn}V5=-qaMM3Xee6(zOV-EFs{ojT#121)Hv0&d}>%_cq5<({xdv_AEa3K3zI`M<2OrDkC&Y$PuqJbD1NZdyd2C^3EfI24-s#T1TZl)j7uEA&`I{6^?3YdcsNEjVJ8tcdY2of66t4`3FxTXepZ7lYZzAF$|q^=gPWO~CwTbx_8gQm=!%h4Ty1hOvlU73#1jS&QVYST<_wamTU`1kEuO$DmJJ*}#u77OnbesT+b30&D>*1BHuNRcM-B-zCP`6ZSyXuc!@Kat<&GAUvREJq+lqFP0XrSS8`9Fmn+w9R=>vqK)j*mQ>erkGJeaVKuZ<t8fXi=BXWYa2r0Nd<OH)4P$$-B1VVL%7=awBp?VV_kfkZCvur!o`2jymz)vpXU_NFzXfZau*Xcc=9aI_J|K9esx7)qZp!KruNJ0T)}u)?|3pyaf(C^IYdPVumrDS@Db}+jITedV7Ln#d2+HJ$a}9r><ZV|!$GuKqtq1_l0pg<4DE9pzQ$<A05gg59myFB8-3q<H5cB;OZFgz3GyP6@o`HsBeX@jriJ*VYS{LsNslf`j-Hb;mN+(QY?T5+EIds-$_Tcn^BL|&Eb;s0>`>Os~PL5r~fK5+K8_bsx)Phqx?w7-LAKvK_wkz;u)@ubE2|g|Ywo85mPw9YY+5xsqua5aPDC70io?R2I6$$3PaTk8vHx6yyPuE4AY9at~HRL!tY+oHio90k(%+T#9Cvef8oApAGztpn#KYV<)Fj3f*0Z^{B)b<vOW5`P(FjLU-nkf3i(1C7O&SFN##<*Bjj?0?kEf+lj<G;dfi&6}Q5}{B+6iS3bDUL!3;gk?ii2ymR>f+e_ia-h(PUuc@*ztq$vReg;Vci_o&6Vq2&k4`gpeoSC`rplR*!m~_yZN4>O~Z%N-TM0di#np$+faW_L>1e;+z}VEiM#dO-pXQXzfLvhf7yK$IFfK%uqEsszI+tl<??&??saM~Te@XzCUYO)c7Ogp?Ara-zgqJ@_1$I9j{^w760Eyz2Mp|5-DBd&iSQGUYVG3019PpJ>6vNx@23p3pnyK;VpLe{<@~Qeb*=a=7_=K}>8B^FVX6Fct))o?^l%rEe=z9A8w0Icdjx0ouY&GI8v9v3cl^-OFH^fUf7oC00OUT9+#7+pXV1)`BL@y@Fa`Z_0QKj6p8spac}DH9qxZn8EHwj=A7IVt2%SI?H0OgxgSq6P@)O?$b<kvZsOeL@ATKo@#4V1W1x<0g)m?X4|L>jO{n0w=ErVH|gNHZH&xsR*pggOd9(RQMJLnPdn!gUvw!`8V-+}cArne6oo(k<E=~>3rA1OXskj?-6e@`#icjX>&Gh{sIQkC|V)n!&}Teux)tYbxvVi-?&S31tS*$9g2<gjE8s^mnKTvR1j>-xAdZ+aO*cjr!=nNGg2b0_um1R`DT*4U*QcI0w|!r(%7M^fJx{S}OXI8GE$ZbCE@L)s|u>T;6>9r6`VZ*5|?Xl+J*<v6^dgq3x7iMq!YMd&`5mO<!Kd;(BSC<~*?BvrZi4vxJq^J=icMD5~Q8ErJ#ESi(;PQ8}^^51X2|Bow|coE9Bfs~1k#yr3Miv=TlswrqQ>$#wEyhS{mFxN&?mkQs9r>^{Zct0_F1HGH~L*)w}!~cJ7UBB|l>Z=zJ&Og3<qyOm_*RQ_VfB&QY<-hfx{=EOqlWT80b^Xfu>sQ}gyY|eD=byOw`^z`~_}V|dym;fA*Kb~czi-}n@8b2V&-B0g?b_APuYdifsC(&&8|R<2?6oJp=wG<j|BqMtU!KocI4`()L~Fu<9t$EuE70`XTkrHQ|Ka+TSJz&-*njiuwdXFt`_<3i>|cC!_54TBsK_)&Z~D)E2K}w)PGTICbpq7A+xCO-A78$+bEmMdcJ7>>0$PeT`fq)F<MJnK*Pgub&4sm(e*^sn6mPu#e*h44KyMqddFiROk6yTW{yek<4X<7MX!W^opxC|>vky!k?f>>7wSADD0FtEJ3#)HG!If73>v3oV@&Ch9{cBhM@A)ULUwQez&%d><MeyI}--cRLEVuUNe*zu)PhMJm{Eq;<|Ji>41y(O!xcS>>W8_+W>(6WNe85d3`dTe$npq}~1fMx@1cw|X6HM^6mp?`fU;7kqdjP95r<ALO@85p%=7ksfFa17+rq{2$0M!6A!h^C54_;lr24wgv6B^#*p#A52t8YEmfBxMYUww7`+O_Lny>j!9PjLZm0m=v{luz)f>tDPB1m>d1dsqH!MR=`#{9gYLzfmm$9#G`Qg})+!zj_9{^6>OQL_{=r^%<*PZ4io!yXYJ6fprYbj~vq<lhR3iRHt#spZZ(>r5B;_>RW%g`Q8_6Z(c?i`{!nJz)Lxj)7I)MpRK-d`R1F?0FY=Ze0}-GXHNnI{(%_a#bFZ@6P6F=1bm>pW+KJG9zcBcllS}YBb)UvKE3*zzgTlX4WNK^BshZD_Mdqg5u=Xx-@X7tef_I%VCW(;(kHC{_J!+LuIfW@<Hf)Czy3Y6apThuaR&3t%U`d3b@|4pe_wm+H`e^k*Pp`dr*FTs`s(YER{fR@w2VBdXXTw;f@2GsfQZKZ#@D~=Km9@fv#b42uK<y5eE9h4e>?$wzxmdA_}%~P87_c6Bh1W^&b|>B9iFwg0@q)=_73oqs{HI{)@ic*i@WoM)7H3kn(x6W<_q_pHb(BvcY*SXRjyxqS_qdHUIDHE0SlkF9h9Bi1(I+A?Mr`#vG2e1@5s!uyqI0c3CIxu^XB;{Za)2j4ilSx|BYA^*HMsv36o;=m1i+O=X0N|f*|?r#hWj`e&f?m<ve>!ka-fx9EsZ(KVEz3nVaX|>0kZu=G%YhU%9gS^dE5+zxCYpYv-?j@$8L{{?vcw^ZxUHSpEE!bmFeP{2w3;6)pPjT!lw{3O|2-^{r14=BJnX?>`02L9hE?zRN=E=96DBO?X0Cr{_*vtABa##y6i_zxJnpeEICnYp>n7{64%c7A7W6|KrQ&khy>J!RnKruU>ko|H1{m14x}qAd)`d!2ktEc+wx#xBUy3`k!6GCU)+`DNl&syl`Rl-M{x={}=|;I&fg-2tomwK`jC<zyI6yYcBvfZhZY@|AVh?y!Y|ii=PoGCD2d5uuQJyp4^1SF1|_)0a8Lk#swP}!XbL|3MgO;7pFRKp81AI+6S`?%`CQs2>y-NzXI_UeL;qJ>tm$y<u^bp0c9u^=L^@rdJz=sjVHf?Iv_m(EvDhx)eo(L)j$72j3E&9Fa8FK45bn%`uE?!%*}6IU48x2>sNk*viQq4R$qH@^|h~A6UE4)Q{%>OuiX6b5@Eag?5oTpH!fcVHf9;cf`9eRzjD%O3ul6obIk1AFdz*bfVRa<0uDO=2GB@+C7xP)`JL5gU%L6kpI~4?x+`faX233*0n$9+rhfl7Z0Ch{`tPI8?SBZ`<rS8jq}AThQ|^)6qAd|s6cjVC1s9Fa53?B_QP2|Cue^^vrY8Frz5>c7`1Npsnn9|)L6u$v9kBXezv;jJ=l-7zCY1U1w4$IEMCl7x(M0&-Kl>NXTc=n41(+JxfiwC7W&5A}UP7Wj{ipwa{j1-tzWu?CPyZ@0%2@&xQu$xL3u2Q(VsuspD}x(eed|07BCvyX+9suqQtHB00KylPH{4d{G;!cJtkj{!{`(iWQ`f)z?8fEyV1)b6zZrE<r4xj)26N6j{b;Fk8VLK|+U4KF!@t;_T%KU|pLqfFAIb4Y{{_W`(CZ+@*4}-bUs|#Rk^<TQ9U{mg1obo5P+*c_fUG!&Ve5#2;z!o95iOCyh>(`oD0`GH3QG}sJO&8oG-kHRAtU8fvC|~(PuDt$_t(Gp&CTEc73*>EwudK)T0J5I0FNm0$G1TxiG;ggdz=;*e@r+Npuu2fufF|`F}`dWfRWL^`~^y-j~=(?XK)b~83V$7fffL{JAS1+8`5Pq&dav({A5N<s@*uLBC|?kYqAnil?=AkrQqc1*Dm#6e1>Mob1*}0zV^4(%m2mdLp^G<2rSl%dUK@jz@UFNF`c4`NYjK&W6gFVW>ivufuT+~eYgZgPqV!SIvZZ_WDtWVhvKw8AqB`e(binNtvPwzm@LGUCg((xtT*P5(COu%f$LJGM4}OM+bjNA@<4-eHD1V16kshpdC%U+&f1lNoud9Le}ySeh%)nAnjwNS70es10VZlXx@fx=Euxflw?*h%JH-_Ur(^O3G|DH&;J|-Bny0!^k8r`i;@~P8WNt{j7@n;;<hA(8;nHJJ_+0UDsc`p?ckk!RVt-tifcG#O=d$oqy{4I<B|pWM<~GvO9Jchc!#^(E3otQZRK>0T_?~_He|*mYe%JC#kF|oZb1o`UUW5{rQU}TiWVE*vRC6!=4}R~*|EV&!3LO4W_AdOYwX-)y0T{2*DFB6$)<p9-y^%U5y=A^PrAyZS@?rnd-@z8S{>5{^RA649=6~^HG+ICXkOa`3wkWasm!7w#>$RW_qObwaa;iv&osfDY$zX-nzYNR2r`N81;y4b=xckN<H%z)u=KsBO;V<z2A6W;Rn7MZeeKJ#Ybe0`_Ttn?|ExR1Tim6(&6dH(|J+39dkyDJeKPg`W`M&H*-LKHuBhHzbN%6=$KylPESDEJM!fI(Amq8-D<DT(C>nxV(;x-r5q%&*K5lmqw+>d?}H=HYU0?Dd&>F!RMHVtD6pkIC`il;-L+KLW=tzMPKf&DY&Qq(*&4#=st7f_f2ZyM70pW1=DYT{v-PX_ma#KslHml8!o88ixu#f~_sPu^4xRiS-FnMhafa#kGt!n+PJqJu&LIy!+uNm@s_Z4Lw?dXEewz#8Q$`a%<q8*$6bPvXOJG~n2E9RQ-xrc(?|y|B`zbnfH?mcg3!z~zIqAVzYsxM;=EB4X-2CN9QoS2|rzkRn=%!lkJlI;f#ArV$(q2{SF81Yx1S2HTg{LM9Mkc&E|D!|t8alDv3?POreb;kxHXy18B2b0<%86UukB(x$sdR*)git#*J%shF;^GO{AJjgg4mii|<lJwSr<i9uYEKtB|BwC4Bdyh@Y_weT|3qKGbP*JYgX6grSPhB97K+-4gaKZeW)^I5RqF0<TxfKK9^!!+aUStf=@exEls$Sk=*c4zpF9OX4SDUWzYckU8}IY^f_-0A*5wJ$A&7SZJ=mYQx{w<-4E$#ai@s)^}^EO2;MloxRVyg7lDLs1*p18qDG7nR8rbPp3EsQZ}W6AT8k;u+k>kxdk;RYjqM!wmv~*8LQ*BJTf-dczw%d~%LYF)<80)=JgHRr8LuF8~&u5j;n{*Y?C=e%!mzMIzsKYBtSbR1LIL<M{ASh=^XqspX>#<s+<gV6hUkDG!v$Ug;=L6>Y7k-5kInitOW#0cl|J^sKNREMsh*#t9tySEra=4roq{<B24J^;|EJ)aq#4f>07Up0AB&DhOocm1g9|(k8!M6;~V!KUT)UC@Rj!kmz4l>3liNQ|3Zu))PVJ<jKm)s2;vVyp0_|KbANPiYH)@_~HOP-zAM2h#g#xg8K6=##KKmm9H^Huocb&MV|xO(K?G3iP|LN!`g8<Z#}|L-RLrHZ0n&=HY2DG`XKJu5|SMcDxqkrEn#1fDq$Vx3r#9IY%6+(;XT<Xn4RLYR_NY(C|dace$T|sbz2Cxj(!NAW&;f)hgKr5XwZ;T8(@1I8d~(=I2RLnu%M&J(m-dGA%vEhB9D}4#0v3j8H^ClF{(w;);LWtU4?1|x(o)Md$uP)*Znh|IPg{%j3$o15uHWwQ#m@}&e4G6=fW62am_OM3)Dr*sM(fL4J!NqtDjg}d2uI4%;)7t_Xi|q)5$nIkVDIX*)ZSv3bssjYJj(E%6cdidQ@JMfzjmXJaSSzURrT+Ar!KfaZXKufl`$^CJiCajT1G0LKAIMNrV(Zz^GG!m)NJShzG2iOdnFFX}=LoCTS2=aFe^Uj9bxUnDO<lG*Ojv4ELsi6R|3lLGvA0tHFVi1Bk%+St$CNwvQM6Hl7bHO9od?GGRfX15F&GrCe5`%*5%e$eN)gHo;|KRTZFUWe$^LGLkN&GSD>p#4rMnD1*j6w)iNtqx@*h)DalPYs&p)nZT1CFfk%ct<Qq2=5WWr+NnOcj5G-;qdWd=cQHsXUN`Bb12H}=OjB`u5oHSMC({k*ha#Va%62A4PGiDDGSQ$|`Qh0o@78>Cj`GYIdNDR1iHl=vu~dV{PzljI7jfg_EEdLEhaEmBLDAI0BGncJu{0VcIx(p*r%4l>>nJ*@WnMs-Mld%;@PyY+t-j>JbLq&&AvXuXXl;&Ln~PdQ!f9T+VGYkw!-l@x0ta3SmWUIE>9*C_ND}E59M3;3E*_T?+FFoSCKC6djy#Q9Pof7xWy0Nd<NZM0ctS3zHDlOol!LmK$1Rk?6C_s$H7M&7>H8oNcIPf43Hf1uK1WHxxj-CJr%UIm%F!2h^@&)mi|>g0;VC2bfG|51Gk2;<rc+m##eT>#R$s;u9B~{WC0mQW%$*W`B$V0BE@&${YN}3_g}fL<n-N3gVN@y1tl&h)9;iSWQyR26Hwq;K#MNV+qSs()b|Sv;MA71q)rnwSTrPt7h>XDCLiiaj_N@Pvw@#g^dyl|hWcYUDF)DP<J~{qiMtr*We>tSMW}51-OIE5VNk=l}$Zc5mLUr7gj!+xa$l|&IH8A>4l!Z)YRWt0iy(rZYEa-8d9-Ng<J<1w|`5F9>qRd4WphI2qcZua2dloR261g^)d=C?4p(0<bdn>s9%vy_X*zv^CB>E(+d0rxuN$)k6aM7VMR0~FW`2%b2*nwjhF*v`nU`<CyYSg5Rg&wj2r%$<tm~xJSQmtyMPKs6JGiP4qOe^rEtE5bxQVmBD|0qFdCV+szr&M_TPU!!i$NzEVlJx+sdwLW>ogRxn)2N-O`=z||>lVi0vybG|j>7Ie7=b^2e<UrSI;;RmL>&3`FDH)7o!OnShC?lE9O|#9`PKmH#GCJ8C+4sNcfpGT9_F6~(&3sHdyN6aZauo@E@vI@(e(h<{Y5M9+=GWea@PKb4ouUFJU*X3eHx%yV9x<3qc4W0eH3$gsu_J6-S|7ztW+)a8ynyoYk={W0aX{>lrFH^Stuxf$|AW%eDd2tlX$S_HtMnzCLKSEWfW*eABhnxH_2yA9(%<=DoVAQ{#~4kk}I@_W)E{VUuhsa<4~|OjuH+Oio;Z$pMQX&z7Jsb-DvEp_^1bj(~EJg3Wo5{_MF-MtIF9!lgqjMkpo9dwO{;vt{NV$9r{(_#F2X+>HVY|wu)4zgh^^M3zlqdpX*n<2mJ6%)L6@HI%Ry#3U)ldwCMi)(V151;XM!DJH0gh$nNT5``&2)GJo6+!{wk|owClJ+1q5OOf(grEC|`^b(Sb$g?L?((Z;PFjUkdvY>8C)bd9!B5HJ|BR9uR|fFB&bnZ|^q&e5pzY0-2R`ivP!j3dZFAu7>3C0(Qy5Rr>r-XuBe1CNPyvT!UfiOe`+eP!nHMh0^%Y=VNrg;&og6Y9QLmF)}nk2o=fJZ?zJkxxeA`6PrYXd<QPi`KETm`VarjP|%rHGecOelFOW*r0<kIq@aOo*IxK87A>`Ii#cPr<${1X#rFw8E*Lia=^p6c68t;5}veZt>CG1uUR9#<%xsh!k@Z-33O@b8CA$Om-}GQY;Z=1oDg?SJY#DNFAGfHRXT`@j_<0K3)@AqJj4wGbn1yJQ<4J2v)!c_c*JW7Z2#ONc%c!VC6AN=WhBx1>JUN5XOK;}L^#4<s7E}t?{R{l3N_fYns_6Na&O@Y$|cb1p>W|xJ8spkCq#tb<~sy{{doM1#5QW5kPjM2NlgVc=UK&VsRE~(GtKA}1*{y&J4xp(-KN+ojNF??Xc26KF`Z$DDB_qhct}-r4OW=SI2?yWMj_K>1YPPWi-Xo_n3B<z2_{sBB7jUQquY3ppA4PYb;Ou3jE~(Bf=2$rQ%&Io$1hMNAQv*{@D!WQwXRFa7CkXV$yqNGT11<o(&G~C2^v$CZmgj(X`@-TEcUY$k$F!xLE<?iG0CU6Lz36_6<1i&>>LPGZm_hO#M=xCK^%^gK%s7%&NoJ=9PwA=rlv(}cN7CW1>i>)2E~T;R7U1UMIzwTT@I3*UkPA=L--I5<P-^H(hN#TDLjN+3oOMJP1+T_MaQQ!Jy73gI};1g<Uj`C$r0_evt^7aG8*iSk|G;r<R%U)3JI7Y5)b5XQbjP(-H7H9eQ|SwcN{`47_l#>SVNl^x3O^F$ZdL*T-uS^>|we{&G5;iFvQdt=sS`UeYhM4b&?n2dtcn3jmt2(0j2zjk=Lr>fsU>Y9|4Uvj&%%kdr_s~^h5c&zED<owCJiJo1zDJl_3K$#BlWFl^I4Xie+ihD8KtVZAD9(&oUv*kP*v*K^aDCzWFYPXyzo*H&m&xnM#wJskCb|m3D8Y(jHN1fUDre-|d&j^Mu**&#>gjJw^{DXZRSMEJas_5)yQTn{KNrKDOa`eU55#hX3MYw1&u9e6AHaN4yXRRq@SXKTBs!@%KskD9@JCayjT`#lcXjj*sd%D89k-o0y#n4|8(c<sdjCus<I4Q|gkLg9i_e?|=9Z=E{&uRG4)8WpbH^=>iP`ecbK9o2E4zDFto;^Bsi6#|pdeE!>mK!9SCG_(Kmw4!j3qo;%e%r<u+;5dR)s!!;M%5>`j}qGRdvB3l6_ClCk!7i@9xKO<Mb(-jN}f>k<aHAcX0^Nb88LRGWuQM?!0926A^>yE>m@HoX!tgQf%9zY6m`!zc`#Tyv{u@A_$gh|urbg8f`i77N%rNTYbt-8&JXU5AKgj|Io@CJycf-{4n+fE=yqnM9LkRQ$C{U;y@!>DobCP!zoow^&8eaEkK`Hc2Ya41WDwK!nK8=F@<vOD67*d0?lFcZa8Wc7<nP<gK*XSI6b)^L0fJM|%67Z@$`qmr;rD^5E1I=jUizf~+uIFtN@ce!?gpnk^h6nUAi!7mC=0KU>L4=hdqv^>A28&RQLey>wd@P(H>2ev{v%BLC@q2exX7(b#DwQ@EmFf8tJ@=n2-oRFVe-D<fAm`cd_j^vdez-wo(GZA~6TU>0kyc)Nns)(-SXbT?g$zn0De+N#kp_5?|f#$he)L&43xlXt2-4F2YVdCL?&|@A9k>8>BlTO}}Up&7Vw5XGl5pLYLRFm<K(e6x$1enyCcX6jar<YJYaw})t8ug@*kH|SjO^K`N?}H&48v-lW4jTF06T1}hE(phZiCj6fU0hOohOz|uP2F}0<rgKgENSp$nMarK9w_;g^DGCrn~ciw$UhL>Ei0H<JP|9=WwI@K+x;WehPu2Gev8+%^9DRCO@7M!>*LK>v72eNdsq?fA+vzE2~%~0EWhXpXDMI3MXPDB)cMj(`YT(;JV_WXf>$hK;SxIovT{-;K(HwuI!gG-7dlkBE)s~}C{ctA3_=OhV>U2jA8ysF&wO<zSl_qM4%m%uDa4Ddx^+tB>4)f5Izw5Z$!$==ZE<m%Ym6#$+!TiDsg_C&ZRb$5Bp_w2dQdC5)oKY=2$+2>1g0`E*c5n#nlAyDmt0ONwo(F&@omubmIBc(|Hdy=gbf-_TO<>!fbc>-xB31vc~K}{!&)lg{!yu9hhBX#n{t0;!(Az%qXg{ZqE*<HRs=8nr1t1C)jju2?nx`9&8rd&GL*b$&%J2{!-`u6wgbA-#k&(u{@y}<cUno{h!So-3-%OOF{hOTikA*hXheJc!myq!F<-4;_TlM+B^t^RaKPQL<06m-2HdEiU*1Y<Av-p9bfw2)#!&;N-K~mW9KNs|@192GH%3>`^2O`$h}WO1Yvk+<-SrCvTivk;N(`Pc@X<r8lsWd=Q><Ka^kZ*g<zq~sKWfRJxU6IYyf8jpaM>|V$vB<B)~J?oyt8gSK~jxthQLnp)`;LEJ{tl(!HgpUSByCXcAR-fC5y?(!$_0L@`Ia68Q5yI)lgEDc;8m)p_reTn1G2PzGAv5(&ndjWUZkW_^SVe_6O|S6eBbNf=0dso0EbaCUz^49v1KQa+8_SV5fG*k9US%;k@A<6|h6}4in4ZR{o>dFA=~D9UHA}f&lslI#A&2+Jev%w`V8FJU0rNRW;NaxR1n~!+OJ_Ys7UCh6%(=IP5snNep2f8_S`lZ-Ex$jCMtx5e7_bs#}U++R@MXF<EGAA&XWlyo+Ce9-qrY!D;2zff&uqp#Y;|>%dd9aVR)dYA`g)HE1uMk{4kM|1Cn5rxhN<fjD<+hcT)HrW=p2;)ZCxWsT{h&zoq%=uCmu7H!5BrD<z3Ysl;0t9Yksk>4!>x}C{F8cU#I6_Qel+j(|`nuf12$p@iaOvApYx{-p0TW>899(e~BPzH7bi<c^0j8?!))SE3wJlL^9SzNb-#fi+iU`nmFca~wfbQ!Cicd~;D&rjv@3+Yl;l2i_nZGIWr{bU{flo^ExgDT4+qmy7gNL0?~y~ZNN7%7}^$!&Rq)DennRhiVECM{E@l%GDOOwm7F?CTOe66iGXrv@n+9mSNzl^wNLV)uA4hC!ALI<r<WhV^LrD7j!DLPk)B5I8G)EB-ackhA*^;td|g`H2OqSR6KnS!+zNGwqJ@Hn+Uao^!HdtdA%@mMYRXT+dpWvQ`|57O^u9I>Ue?$>1@?THFLm%*wV`!yM4<Qn`y8ANH8y&9UsjX_^@Yh)JSUXt6oC39vNVMu5$;(&+wO78v`LZe0W(EE=1%kJzvVw*J*xDLvN7<<DiJs|~|ZkPIJbcZLuE#3+^+gscM#COeZEjULn#umP^$yWt3pnrh@0S{6=$Q5uG#Th}i2m@c0it+(S_+-f7gCG7$=@|ieGqSUjb+scaER#BNq)cdst>TvZ!D)rb4sknx^#cV1I_-P_C4e3qE;f?(l(*px^&9{IEYC)+EQ<TH{ETaw6ST;UzqeI|v7)uQ7Mj=R9nf7`E{YjHI0M!sLaZEEN|1t$iAaYCc;`N9rr70mvZ|xK-I|6SGyPY2_Q@CF+3x^fT0CamWD-EeRczP>cvmp*Ksi6!XF<$Fa%&-UNMp|O&Cf%HFYNvUY4a+mNmcGccqc4sd;(E&-Bb#K;uCYuE06Dd>C@(NjSInhZ_p+=cvsMQMSGQps&{Ap&C%34ly&17)yyY|qMwt=pEu&=^ZR45kID36&O6j2nOMAOvh(>=jn8y1};yDO*%2%z0h|S~RnZI%Km+qoy>kN*5YGgxtk9In<>m0G`lqmx6CP^qJeceBjh56Y&wF7<Kr*>qfl9B`K0cTKkWsS${8D#2U*(B<+pZ_ej<AM@K{75IMT=cbA3V5Raqoof1!65FeRe`B4_H*Eu*Px)ZZrdy79Ub#>*4kOZEjQuU-w8l?L6g-z{!s<5)X^8R=oGV8ggd1IV>GW}v4D}Gjbc8diJfAHuZQwhI>m~)%95hQRQminUQF6GsfFpIM&&f6m1rlmlm>)kaIzM?PTSvbOOaa=gQjg8Xr1T(WwN3h^E5yUasY}67o{6{9)<#<V(UU9Y=`5;9a<k!j9SBCrjb@Q5pO|vF?!X~$^<~RfvHd#CmP*}{D^AaMYcr~!GKWp+ocsmjSUk)%$%gd&YiGE=2{FXs>uhmi;7}9war~yn<I>N)?%+WS}W01i=eH@QDh9FHf0y@mZ;v%?dY9odjPB{U}R;LxC7`BQx>d9=VM1~OBdrCfU<{Tx80`atne^QiNS2c%D2g#77%suB`~v7_PV3hZOUqxuoCc?kVp&h!n!*tSJ!V~&{8zLDeCJ8LUt}GWKlz{5L6+>bei%$7Z!oG%Tib{6dX63Y8NF8Llz2X5im^fqd~w*j7bUPtcRx<D{kF3n@kRlx;#GKE5c`lKG8?YhT1eb#2ao*S$nh|VJXHJYwOS+!SES8wgv;s4GKqk)VM&i)X~V7?5^2FYR?VCH3F16gGjJKA~PinLg1?7A*z!dO}`urTDIkb^c7~UTt1t+Ydx3)#Ey|2$J8XSAnhTC7$#WkG~5+Cz$`(mi&II)#4-t`kBLquxvQaQrjzO}be?r`dGwy)%$opfi(b(S>@&XFsg;^Q@hwZ}yAgNs4c^Jx2EdGCc-w^#$fG(!ABzrd)@$H#(WY19d-~40^)5zLQ@SFNRE5$bxXs?O%g1yUaj_*5UbF|aL(+qAwWIg<Srb|D=~1s8ge*kWcQlFAZ*u?-K3sZarGqvw={|jqC=sHhy9PrO?XZ4yS14G*&Sz<rTlboJO$QTHfjv!bD7{<dcHmYkc-=g7j9C}SzEgtCs0BGp1<@%D?(nXdl+v0iBC!(OrPq#m%8mvdZ^|kiJvLXu*$HLxTI#)}U|AfPZa2Dhm(m~fS|XVhj_{9u^rILwPAlcPA)OV*gY3&oJw6{q$rHMsW9`HIS`;5VrS-Gh#B1b1Cj-hTf<qxHjK1F;A4XDG(kQmIxuo|~fMF;#U|E3Gkv~{j7OhjKOnsTeCtf0_Hp^>x@9av;PSg>?B5R%TycYfxyT7TzV>pT_VkM?|6f?(N5{Z>2$zfp(*oI~_ijj0QxSbULs0;hl22zR+Y$h&{Bc>4Hdoibz1_L-{K+j`7&tYB2Nl!{VdP*a~pur?;)AVK<-^R6P+5oOIaB)hmH(2X{(+Z^m)u+7#wbME{ztUL%(v!1Lyv|s8P)tAD+*s=4ILi1*RJLH`MzLuJ*pg~E{f^-t6g$L9g5#<{iGC~_6|G?gih4;dE^<oJ*xFAC?`|va<t9O+%OBeMlCVPsW<b1KV&}N+8X7inn<nZRZJ8&cS#AE9wT2K_qbZlxr^8^v%2-Ncxj^YW_^eP?=7C7;sMjCi5+cST#6%oDTdSH$EvtTxTZP@bWoUdFKuTm%>jqr`T7FN58^v)ua{;@^>`YWgI^%}o<aC?iBVEq}D`R5lK6EAWC&dE!9AT-aVav7P?=aAhmJZKK*iqb**F;+`4X7xO|2Se0wXbFMNtp1-rF3G=jOCRNg(@c;DM`l(n1*}z?p4*a-kHHm^`KcZus0lY${!$?Qj80&(AXhhE`p6%m4us13zpXlmndNjbJ=3uZ>4S%bb=1<R|vSOL<$`_hbG11;*kYAY9dQ_;ugnbu-OT*`$3zp0KTX-VswdakIT|ATRed4!HlQVxwdR8?#3mVL1PkOowIy%EkWH1(?$*}Y<TEsXAB*ts6_Yk4><%;WS?DO&ngZhIav+CD_9_FmGr?=&WlH*y8C5Ilr0ClTcp34QTq4)v-fVxjburJ*snxoS{ovmK_mfY)z)Bz-mR{z>MrZ7te&duX<IR2(E%Vxpom){0wl;>%1knuNe?p9t5&uq)6<w<wEL`w)&9)R?&>e(?&p3B1VCn8vRN~oL`0n9=jO+cyC1)Bh4KLdoSu`6#c7~Nc}_e}Rt4xb4ZenGyHGUU3!eo3jg4HWDfq;DK#gj}{!wN!>ll)T%=DgnjLVO3ZS1lZVPZmN4BryFB&IW!W<GQkZ<pW-_!$foA*2ynaCO?r5u6M@Xyh;i!1(nX?BC6FQGK46`;m&nH#!l}SDwKJp)%@16N!%%ffBak5;w^f{(1C`5P-!<?{SY6;pSvHC9pQgT7kz4)I0@xQTGLVr=dlU{6pAlOS{rThY-<g%V1yPq}i=nmmrE}Hf06>{JN;f3Rb`|`phO2-5XK%?nP&(Fom7&=lXmDBQKm4qt6UIkoMu!);DUOx*8Lpyv>xyyWW}O=yUzyh5pbrl?ebK8_t%aOK3y;#2Y5BA_H(I0P9%qywN1KGAZ&gH1N5OQYAH;ZB`grs7Hdh#+Q~bnX4R)e07gHyO@Cj#R`$8<0$MHa&$gBF~O@CMmE$L`aa(DnC4$#l<TGuKq}2UOQ6!rn?*jKj><Q-Z=eT}0>~alc;8n(-<{9ur>g7!Ebu{_O5B!Z@ea49+L!DeMmV6+f!f26`6i|U!<64(qZpH81JiV(^Rs%pXwyIQ;EG#@?XwsDBI|;Z_YP-;TtbC)qXKi?XB;(p5i;yiBo^86meI}6ja7z{;(v_DYnNO0k}|kRj;yosp{P-?-3FAYo<8lMQXyJpm)J$H1lc&TU)7MPT0-R^S019}5U3aSw;g0^FJje%WcdhJ_j6x^Ve;iAVkRzXgs3q}GU;lqsM_zo`u@#V<PWXu&ucJo!Q7`US}eX1mGCUV73}yS$6u2k?jTKSuBzo_u>b&Ti>BKSZ1rpcQuKUNE%`*Wzfiyb_$o-xX~=9PwT&|#JM;x!E;m55vRijSZ$sNpZKq!JX~#fqV@n&A*mkyFjwTMOjVA84vbL19osP}rtKGeHH0b0YIy1<UdtUH^*y|Ly`!J>*wNg3=3y#Ctis*N)Z$5gURdpWvJ6-X4_=QY{U%=|CCa(GfpT$%Vz}E#2?~1XiS4A$|>-lBQ3S{l}Y%LA`)}r1kvIlte{m<>vZ@Jn(oO?aVa!LuBVvem0L*=fjjPmz<R8IR^TVkp)7<eJ^gou9j-Iw2I_}=H|FHl}O{I6#1R{}-9C7>UT^S=nBmr=Ac6z@-p<CBjdP5>4KGRLFBXBd99rt~+=ZlYo7H+z*PzDQD?a5*4Oj7hyLZZH6WP~f(Fg(p_wNnK&0`Li5v8F+E^##>=ITAStB1H5jzAr}166n;H%5R8)celVV`vi8Cn&%v3nmRWzaN?L0?372#h*nDA?wpZ79c1*fIU<ic2@sJbdWi3UOR#b(ef8xkh8Jy)%NB|GHYaSsW>@#mRdDc?D!;bS7a3T$CVxUv8vGA#n@TO6V>x&hGK+mmR@hHw(lqe{pge}~^AD~TchU@7VR{^Gi?6&O^tLc_(49l{$%kxS?!z?}-aqkYqfu(D2Fe4f+roQt9Iyo+wh3@&nGc&s;#@k0<zYuM8VrunMVYPql_VZgic)`2~V$A!mCJ4%bVMP4+_4FxsxVF*|d1ix$J-jUl?=uz~4v31$AII43G(v@0_tVds&V{*Sm_P{wdrktC3dPv2wpHsen1pmJs6O*seojaWM_GGSO<GC#7;95iv<sh>7{_Ux@o@6`jVkFc@IKY5tGSfg_p0=%Em9_u@dR3_ntCTQAj__vI3ZUWRR}Kb0V5B?I&>zK+~gsHsYn#VBHX~wf|M2K(=D0zgD7~hSgwf$DjkD=#)0qzX1fCBynQlSZ76h$c7OZ!i8|m|eq{{q9P5DF*mRuOW47AXAJ0sv10RiowzZL=1twl4;{dI8=yN~ZEweN(=p_yVzj|(@$#{SYXs!4B`w_Z$*(T~%Vw~rAdYSaXI~VyyV-hY0#%T4W{mF)FxBCf5HZ6*MV2n&J6B^^9xXv!2XP0Macx=`f_^TMxLG=d~rnFbfReWjCuKAQQv!>BzfpOC*4Ir=NFl#y}$()I3RZdu=??sGzEtC<9;Qbzl#2ZfTL>FkD=#%OSeAW7evu^Jw9aQw&=)@)!qBr02M*fH^7rp0I?tI|1=IbE-S#^v9RTb@+O$|67i!}YzbV!GJAJStupWz<pLu0#H+V<Ng+A%Sm0o{k=L$>+;&OAge)9A|Xb$ePBbN8^Y$x3+Ji7of`iFR)T9gCjvU0AdKtS+nVcV$?(972UH(7!Z7tiqTfjlwO4LjU{AJD&1UqDR|6LLWW1C-HJ{D5bI>eH_H<tk*pm+(qqn`vXxetASrl$hn_C+MF9`&J$y9S9rPF$Z}1C$~B26XGBzc&|qp>tgnwH%-4aqc~G3N2MZDQfue)`S^|UlhW`PhfgRl3AtAu-H@3avemy`_&4Yb;sG}L->oG9TV_=@Yk-$8U;VAAEjzUKF(E}AQahnRtDb=P^pGh$dhxFS=V0oX7si~slgTXsQ)op;sKyf@CA0u@<M(TKs)bTKpI%+~?JVv+pIip)V29($@pu{Co5HIQ*f$sP$8YrTxC}apcy_6mO4$(p`H9laVGc$r2`98+Zw7=WFagG`<&iCUw)dgZSASpI&VdEs9or@MUUwg-J%#Y!sd!{ffrdIE?W{nC{Z&Xgrj<^KOHm!|C1BpN4gnOQQV}+>Q(S)dkq6O?azl~O!h6{L%dh!_c<T2{WW7LyhdDIgzg%<HwI{}xWqkeYviM&MH*jScoBl<A5O}xSsnX+cmMx{}J1-_g$3&wI$pd7l|2)Tido76BTP7Rac)G`WAO}k&?xLM0^Cj|CPhpOGKPV90Me>v5#ENS*dD=RxCiyGZDt{k;~O7EXoyiB9kFq%Zy(I`6pEpSI8d}z^K!n~L-*Ebm@VXs7JUl~D$zxRZJnfTh9JbZh`2w6hpaG001h*~2ey2y)^BggjL7|IU7ll=g@`%KP@njtxyRCPedOARNPV~)Ufn4;@^E!m6aU<`rKpmcacc9Pa#bj1?9l9m%+a1g~5du=lDk_GL#;cT&0KiG6IypC+%x@p`}m7CgdIQq<}m^iYzx~{(QcU2rQf~WCQVtpBEsIdZO48>rJb@hwYxUF*$pV4L1Cx&=cIu1;s*0DM#8gnM<+vT@Nh*AA9rrgNdGVw6wU(#QHu$#U<SGU=0-PbVt7QSn1?d=2N6jvgd2%_{(%u}EyoVn44Y=2i|b2coyKW}{f5tHfdlcSLIO}!S&#@np8&@5;|m|LHbP=;I1lT9?7)N0QiyR!v6$gl)rH!t!99nxnk7|iLw+Jr)Tw>_~)X{~PCUQ&Hyn*GpP`}U>QS}?FqFf@<1x7KJ(D_?ymUi;tJ?2}G$O}HGIvkcFCtsX@kJt~G(AAhoC2_wK82LYG{rAEMaNJr{N*~ACamiO5gL+ipEj21I2e{4D9#tsSE)wBuEX<z%C=y6L*w4948l^H_UY9=0DM-Gwd9vSUE<KgD<6Y{yNY`s=RDzU5)YfWAa#cK1$pGUE+%28~Ro}tD)yHWTMP1>Z0nmB29py#|vV#q0(4!X3HFw<@0$lUWEURn-~hH8f)q#Ya1LSiXJmZhmxfbGSJ<9k!AY$-Uco(9yCK3k#%C?tN`Mc$|)suSYtorjUrT2o!rnKS7xttk(<I6{B2>DVyn%u@XSJ(xIwM@r^l;2$(emM-J;9Z$`NZENDRHThs|nUYrD<1hNK?HS@)-*q@mOoUGjw5Q^N+24oFx1DT(z_8Go0yyTCfS1F3O`3V)^)3qWzymj_r((4nO}q()kvP14axq&DdH6(h5o747-W}AMZT%`07G>E?CvcU|rem_}dfbYSTXC<gnEed<knlaQHTXAsCn{WLKGmCbkx^(D3~N*p!kdaLF_wTIKZ{c&O!@7E<hqZWeiyz{ngkYC=8JJY|4Yn(;dTN!&a?n`;{{j-;M^*=#p3p~qywHT&z;E4T6*+>kRQ4iOyV;?fBWQzq9PZ8%}R#FQEk|U?RE#wT-<%{N9UHT226-(C|;tMGvOM(=&N?cuTXl<7IeN?1MvfBEwVWd6+B0$#9}eJX?enp0rz8hj&K5e20a@n>-n{7*s_v_EpHcyoTh`N=09Xa#(w(*|L?o2Vtu`yg6yKZr;=|2WoHmRyl7~R*7LyguM5U=J;&$-RE_V8Wx-rlFyhrkfoVSG%XQIa4Y4xfq7u6FRz-UPX7M=-%kx~1?d8OLdVtVP%|j%ys1qFER}1bO|5BtX34%dqk|B$xWj+a5FQ@qh_UQNdrYxuV;-yyT@E}lnC?O*#+I`~(i708_pfNyK+1Gw3u8a)5)<5-qMh~|P9cqFAbYHi#fwzP_LEk>1-@p_3np)04>;++eL5D>dY;nGX8CkA_-<t-ZVPOrl7%HIvDtxV=1jn=Z=txYv4R0(1iwG+yZ&~#Fk&A<_rXD{senysM4gFR?632-{G<){;iHMqKI2aTBcN+kU&{UQ^*q2|zb9Wqfk2Dx6z8*kHqwbMJoz$>1BCx1Z&1zw5wZn}#aAhsL3l_hPX|>U)#!-*Y6B&Zr)slW64i3+P*w({upNOi+>T62+_%wAo!*c?fh`{l=rlU-!z>D=o1#{S9qwWG@Io8X1rq`<WJK5=$?^Y*<$#8=&$Bms`!!Gf<XJt8dohj=aDVsV`Hg}*De39up>2sZIvrD}AG6^_Q;!#$P4L5kc`%Tx#a(tLKq~#LGFY&k~wLZRyV2L@+i}kqp^W_|vd#V0a+dH1<9TB%FNCRum(wKvDWm*YA?=x3p^IWfbpKnEnETR+Nnwn8jjFV{NGjAC_g)bIKqD{G#g1R(v@|oxppv@)AXku~x7Ay<XaAfkECn*O|l%lo@GAeQ1ECePY^{+v|tiaeITX{8a1$96x)vYjc@AsoqtLIp8dj@huO0;WWg50*@AfIol0<UFHiH>&fbzLv=itekcOd}~$+WHcjnlhvWvG@(6E6`!s7#kLjCv`B0p^oXP2{USRgbZ!1DSgUD#P|F98OQu*AXH!FRy^Xo5DGUYlxMlL3Zzg!tP5#PXRVp>=)LySUjVOMv53cgIic!ZjOj*JvuOqA28&eVh@VO!#|DQ7v>4|7^?g1yhMc;Fii{mPJ`GR#2zDSqCx8H$4iJQ$(He>`a)Wmu+=oxr>f@`QdHoN?fgl!$VIm%=A%9Rzz-AsC7vwk=_tEbD1k~ywY0JsI@ToZ-!9-eE^iaE_^Al1;cNE$_fRE3iqnX$X+OnM4Uk5cxTUl#&8JsYWLA+l0;q|vrmtmT%jg6H;S1Pk~ZSJ!s%c5ZLWA$Dz6x(_)Fcu4J!fJ$Vq@M<ofsqXQ@k#qrOs4dAmWEqw0B2(5j%XXljnR*WNWZ%yng*7RT+y*XN@4O%zE#^NHX`ZRDAp2ZFI#S^HVZ<t+r%2tn8;;m(%$7_cEc!ZDailP(Ito#7p3LIL9zEn29vtopFA^W*@=mvaSf;)(0XOF;cJ8a7c_nA<}>5W{8?uZ{np{;c`ymC|6m$xREvN}3a8<Z#mF>>vs5zJ2?M%YQcg{^%+?n}r-NcFb}@LBIM(~9av<pnDV`-EId+$>M3#6VT4w=+jO`n%4h7jd@b5m8bVEjA<M7DteC}+uCdR}niwl&si9gVTb#ZK%<Yit}!i7xNXeGV-s=ySMaOD@`<~Fe~_G{X3-Ej<TBinK8gcE?R;@$R9yglBwfxN)%KL@!I{Q?sC1|7xPepyc_;lCMUww>zcYm71b0|qmTrHXQw8a_J0?VCnd?=rewI?qQ%-R7pD=!G%1nEnV`<;{05zw1RmOqQG37-Qz3!C5rC5fK5R;RbJRiqR!r^-#L@b&+32%Nc5$%L-PaUh5sw{>Z`jI_?va4MoiU)#-3T@53el#O0_ySv@aLyk%Mgp#1EQ`gc&@A5+KEbUgOm7+x+tCu1{lPsOgs#N`?+()6leD!EBjt;*i>=NCW|H$y=2`SUNA^VL60%jfL+{8$2cyAS}(yEEq|jZ&*D{OMM9UCQm52xX7#D%>!4HXVwR{LMUHz=m7zOa;Tb_HH~EU+}ly<EtsnHC@9Q2~#fC({ZDuEo9ue1qPyo<NJc?w#hSVh!Mz(El|2rW^hTPX*thVX@rKR**Pp1>i9PcRH3u${owUq;6k#P=W>dt)mQWUK3A9MbxdZf>3+AEZsH~Z(WR1UWaV`FfH`fr`7<56%^Sj$z49{m<u~6_w%lUPi2twa3SRV=JI`ex-7}6ZUc*_s?gAf(hpTWp5givXP}b;Lw(cswems@qBcqOR+>V|>Nh9IS%RhZvb2$yn9TuC8s34mUfov*fSV|Wz<Q7$~a^Xwf=wf@L5<7U)mYBiWJtYkM(qpy<Zpt%?1qbE2feY#fxC^HX9b&$5c${abx#MQyI2Y6d>0^L|Fy)cwyB<M*=MB8dg4jQ!c<%6zcNF4|)^9U_6cy7;-eeW7jTgXdx)K)u<6!5FFpr($_(e@24EWeBj=O%#0xhV%uK<g$z$DGk+5#JT0mAGoD`vwb4^7%XY5R&^X_rHoh#2+?4~C1EkfX_RNdZEYjCGZ-Vb=g32q$tmy_n||+~fEPH!fi@kj^nW!d~$gzIlL!qst`TwNBeV72FpIwIzrUq2Wq_eR?ZyXEFczlax78qrFbKnd30dr`uwzh9EXv>CHT48kz-LXBQMiAp&#U)X0MiMfPxzES<?Ay?wGhA$x<?XtQ3!i8s3;2^k^dF@nS7g(>DCE8nth9_%*yPP3lt+V@m9V@_~c2EP_iCCl!5(f6AL@oCW4_udxnj#aKccFa7nRdIH#W}afoE~W2B3<;z^dz5>8dvm9F{$!Q2NDq{?-gqr>korhd9D1Njj2pz<tAAm4o6{leu`nC4GE4C>zZ3`3U)C6Jbsg)ci?b_?FiDELixo(a{fhN)Sr%R^a88wZ)~8~g`^`P!gWu2toqJe)#uI7{CxOtwe=Ld>;>uUweo;ota~9rM=>;XsHj~*=9hls;eL7-0ah1Jkbpw<A%vim~rk)m@E@=c6CFe}LoHok&Xf&p<^1QZ@+N7R{*ybDS=Is}5v%}SnwCa76=ctMPD5GlX=b`pHwOA^Pf!AC?rIYkdRqw-ua&|}*=jMn&@!u{PgCj=ju+KNM8AcwTWx#N4ioSLN0h5g^kqXbDhG#~8bSIJt77lw6W=$KgwKeRK!wgyZgiOr}lWq0IwzYjCF1J>s*7&3*j&}%G4hZJm_|B{@=(id0+ZtM3pzmOi?<N7h_X_SE4(z=vp0|nU+LV&<oO<+^EXX5WJvhwNuE^l`3kd$fqjx)zyBkLB4#z^?H(K|ONZmhMl<tNRy7vw%eOP4fy_-2CBKKiq*((zFJ(_49fx9;Pb^vD-Q-wG~*g|+i@n}zcCY_lS^ZG1qmd&0Ci50!|eG|-(8GWh!RNS{FBHE+Q1s>TX8-&u~@~Y@X#k);DLo!3t63c%@j4#oVYTiM}A_&)WE@<yu9Ct4CHF3Y{%l#A@nJuq-bla;f2gzHj0`GlBIdITqujNq>0GoIc<rRG%LdTeR2;4ri?-CRMWTEMz`8aUfh$b@L6VG;mKXW+r=9{@^lqm-D>YgMGuv=9oCHiPtoOkr*d@RLL<flCfoetZ~p1yR$eW^icyMwiTa-}9$b<h_I2aCm;UCoAEB$2jTe2$KyZg^wie_autz_Xu^?Bj<*l*ug~KH-X4{UA_NQa^^1_NUtXvD5qwi=S;bi{tCIF%pQ3x*8~1%crh$4Da%KS$%^N9^Q-><FD51C0=FEHzbO2%}+5lR4VM6HkvH2k;0HBuVA+TFl@~aMIRw?M&_H&VCPY$6$gaWAJb?BM>!T>xLyF9L&FiU>6w;`e8J6fG$0AA=uBM;|4aK`PTm#Ik!xSBgeW6I?bz|__fc!K$v;8UON3l$JyoGAH<>VQ@hL>e1BgYo4hz+$0*;rz>qS3YPb(BraB;u-_6x4HH8Vj~>l+K&&9s<}Ep^B_>KkS^ZaTA?S@H~bUF<VgdyPLd2hx2+7{eR0aG)oUp^soVdzf!flZ2egGss3minKL;^e>_tCT~WzW?n~z6&X{pB9+y!*MVWL#A6D|eNs@qWBiFYt(~+YItuBR$@t&zi%Qu4OA7!^NIXe2L#ZB1@XG=BsNUJ@Xd8VA|7*9~gNTXV$aB)=z^o`sZb|Nf;Q$sG%$2Gr7L{o(Vv^L*CM^0xu?8mcgbXW64Dd}wbeQZ?yBB>CRT~WT8e`EkQXHtx$0jLVQbC)1EUl7VmAAtah44_W2`2K`H+hO4y-LYVKJcCPiJ#N*z&GRG?f@)rZ1zaW;@CUYO|*`sKViz2516jyK~lA_O>5_+kQ>&GoNY=Xh=zM~*(Xx@CSw1TK0X1-JH^Hv)anfyw53qZXvqiIC%iUm#8)%epzWEOu|47Ia(xTNe(_UvmNP}af$c@Z<#OgG@X_+F&<nJlt&;nQmsSFu;EvuJ;VETr?k5+j#w9@WzRZ{tZ{7B2md{sNe2m)mPTS0uO+fcuIO*-I?Pd`oR%ZOcu&|*_KZiBD+u^{<_vAqv?o!4~GTFkp%=a*jj<Qtc|2X0X+i1CM=l{@$lBHS}DSNVP%-;58(j}vV##7>^LsRB|vFsM>lsPOzL|-g<{p(4wh*$<19rOaDTV^8aC2A5_=*tu7C7ht(g{@(JlcQMAvm-s2g@!e_+R-z!EdanY9qzOxGAKMhPnxbhKv@c3KR!LIAIw~O4!ajP@X}p%x!g*D<g7MDN#=Ju+CH3Ln8B!HuQ{RaXQ=fABS8U#fhki>QEWof-gu3+B!CMs3!i?HdTx{w3bI{}`Yp2TM6SFdrCpWJ`kvyp5KaT&Xv((`yi4_qU|4r(WNHn{yl}#p-c5X&OpJHUZCnM$H2RIl)CLeVqPg;3vmrkEq)mPD#Z4rP);uJ9IO8|7r-?GkIXeJ~>_%XNbTRi-x@7Ns))a#cb0N!sNHgD27D9%_kl*>s+*YybS1JLeTEmncO?&JNn)$YQ2k3igv6H{RoRS|jpX7t&k|f_hbkXG+sx@(W6iAPm@jLN51mB`rk@Tb1tkbg9*gRl30|pxqVe<gt&Ys4WJo}TwFoIXR9F*0QZmYK^d{4l&_M8XXj>Tk9_e~HU-T5I<aZit#8(hByhhot>TtB~ATrQT^Ebu9XZi-s(<^0^WEQ>ZHN~(T-;v4Cz`IGD=E0=+jl0na1Ai7~QHzBrPhKx^wA*1i^BNlKJ@7wb30ZhIGwY6<f_Ckd7w@?0rO=>L$Y-CAOc~jU|I_5O#ezt_Za;7a}!OkIn{C_9E<@pqy6<ef9Z^>zwignVNwlFlMch%aP)fi5J`29IzxBz%Z-T>S<vDMQ_$DI{DnHMtYi?{=<v2UchS2N`Q+?%7b3ZtB3k;I``&AJZ2c)#)Ioo_Al-ao6c?L3lT_MSoBg}V^B2!u1=$6fF#3Rj~vnxxUSl3UQrjPVpcVJi*uGEc?r-UK(707Gi;YyC|cowp`ze(2kDTuo3D%G_|~o3cU^toNe$U8_kF96R^TcJG`wCZomaFOypBIsAphoKk6=wt@Uj9=H34j7KHMj)u`A5Oyg!MRtx9&O0d?UR~)k!-~S<;iiJg;3bUIy$B!QqHN|9%-9r2NQq|-`Q#Irv*}>quqV3~qJA+BS&#==6l;cr0JU`|kS`Fw9ud0;bLeRGgQ<;uKxz>Ilktt#NTBknTCf-^jSylx@^J*@mxP`oc*ThKn^dpoyILQMr~<9oI~MSw<yp&-L<&)5;4h?>bkt}`MZM8|mz7u-V>~Km`9UKJ@X9;u@fA^@#prZuDsEh-C37$y(R|V3GB#r5Qj(E8yfnFcl4C+oJ4BRCOlx9}SWY}U6E+x|W%v2IZ@Q2`mI!^EY=pj8NISAXgk<?WLWCWZ9>V92cX+~x{d{y(P$ez$Qo^|E>|(v#tk9t8tTkJ=27y3p@GS!&7|oW}>BIEgBgQ>&q6>)xAWCuHs~{qYYGjPS!x9TKDf@IZKsSVI@UXnnd!$m&q82_1?pb_3BdS5hDU5s@U+;F}Js<M@k}vFP<K7LZ1%@v~CRkQ0bWt$FveZQqX=0#eT9;`q>qrsT%j=Tegh}r@5fj#EaDm|@%4}7vu?lCuL^)j%&+a%dUWIWS@}h7kF2q^Lj>Uz0Fb09>ALw{pC`d6pB|BlsU>X*z1q`Zhj?ouMR#iDXg4XN)=g=Ou(C><=mRDP(G$l(j=2`V%v9-eI+-sbc=|UHQqSyN66<=_>%)*)^Z=|lX-s{1R_)xk;h^iE2RRi$#u!a1!z(2CY)AdopC1`urqe4ddTqGxzV*eZ5tH9x1cX3heb#YM<9^(z4;qw!pRRfSWTOHa9aKsGz_Ou$ZtUbMsRB9{uf|@*!yz&4FJ?~;kF@rT=YnCvczg}DmS>~fqMp=g+pe{R6V+pO;<~5NjbnLx-@~7F7(!;nh!-&8NY9>em!%21&vT4(9fnEnLfgGkP6uyl=c*<$*@0vGN(w;Sq?6{j?v1(JV%VJb@#>@vB+7{NN2GL3k=eZNl#OXoc8b}+wIC2Z*W#X^~TH|-SLd|`!EK0SO%=;!#($nl{;oV+-n@Okt7(>*UQGMnZ#D|LNW`K<X)95EdI6jRQc%f0D2KiBBS>k>$fKfPrj8GVfm_|Dwa<yhKAV=f4A$oGzey8pAU|vnP1<M)9Lhi7WyT0I#fFU)UpC=%OJ`$9+tylVPS3dgvh{0~JmMaW-C_bPFN7qFfiBJ~cb~Wu2Ry;G}>)LTloLRSD_*AF%XayRufr_Vam=K9vd}i#|;hJhTeL+6zTqq=4FkW6SO3p01`;3tcjPS*u;lMzZC|ToZ7fF=!c2^T8)<BH3$o(jkz@7(W(4xGlZhE?$SjlN@<m*Q*Tcw8@y_P==glbjIa_mr=`pZ?}R`7@Xs#vtNYcb{|Onr<999^%L-XI|^=`dHGzuXaKhNzcSW;k48()h~)6F;wcxEhu`&2pI{?h0gR>4f4!eDEL}i8maouO~=!gG9)pfau_EzVTevT2*RtMyaATnWG%q+9Z@kopsAEG0KIYsI?(Ztd0hQQ|Jw=HWO398lnu+eGX^ZHsq~YCV!SOoz4gdjH$wigYnMdJcwc63WR7%f6R$P<eGG0==_^aWAIZO^C?bYcaWs7Hm85GG=dhIt33i*O>Tzk=~&#Bc<T7a6o8^GT0D?;uAHMn(j<$JCP`R0f`ki8F)C0+Cz7o52)fKM&STlB3t0Ar-tAbLTAD54KlUnWpoB=O1;#ee`*v(g+V@OWD@I+3;THYciAa(isy9=d^UoF&I$%VjOQSql6_dp-OW#bQbQ*n`Mt?}7zwDuhqiP?SLQmXJ>YIte!vc@Pk(x(~)xt!Wnpy2u%PCqP$~N(&zj8o$GiU8=R(CIN)Xd?QOb#fD>A-B;C_AA$fp;xw*ECfS4R=J{;KE>TkD}Wqi<jjW7jSUW_6jYeh}KS?q%8a}-;mNWc7@c9Uw>~iW&)sRZ{yo}9JE1LjEk*mnoJT3VB`x_GsLy16-KY=erRQ{`P73_b!auMM<)k-GhNSvpwzSpD!B_K4g@`a+|nYUW(^rK3#sW-pjLn)i%}X~mZTlOEGb36EB$^=pccTDzhxe{7wJjbNxSK(A)R0!mVn{1ysj<(o_QBwXc+F`fu8>85MV{wVw0oX($R~fjAM?z)ZbHy&1c=g^XD(r<0(ESiF1XoO92{UUt1XMY#SR}c<0V~X~h2jdI~-2!M~lsAbJMlKVzVK?EeP7afx5g&0X7wdhoiAl{M=B3SzpXkSlmO-&LGPh6EAKBh)|zJwt`V0>VMay3}I9uAsLw<ZAt;dCH%}@0;({X#i1bp$`uXj#<2<+**V^4FV4QBOgg~Vg}$+se2Z>C3O3cgn|UHcB*2&k}eMZ@x~)WnA0O_ugCvBJ7qXDNW#!fipJX7pL%-x%YHanD*f{)C=RO>C_XYwuSNl@D^JFmQ5$a_;so=}pncj=Xdz(ffkx(Na)S*1dQLv>66g8mfSpCepyhFQ@KbZq`}#Eu48mHY!<UDdR{0i0nR*|r#FAUAiqDkwSZY{a4U+b>EH^_e<eUERR^#x*6xK_CYE!6=3~Y+sY_es`_n39P_v)c`V;DU*sx>xC^P-x8V^o=)o&WTY|HFU&KmP51{m1|QKmCXQ7&&6_fBxV8{XhM$|Nfu;?|=MH|Ih#VU;g+1F^&G|fBrZB^l$&K|N0;Q&Hwj5{V)IZ-~GS;<v;v8Z|?0KbUkoRSW|*yfdl9eK(Qw)-N-)J7<zh6w~QfK@Q`~rJm)5~u;@ArPP3AJzw2A#OqI2FMOdSu?F^%--Fw5T!Rfnr;h`qABRbtNqPnw{Y!|R3Y(G3&tww@0<UaEy+G|KeoT2E)oA{74n*yEZcD}RI?_}@wqRRU&+KXQi8tS`U?e{sz1s7^hqw9vg7jNWr=h8!YlC-a;#dSOp3C_6PN{8G&I+n$cUHh8X7NtdP0ZkWf^0jGNes%*=$AGQyvB#_&%A1+HW>uaIm+Nt{4vf!^iNhoLOnTA83Y*TqT=$rRi6>Bx5G;Tf!q`I;AArOyJm*+1bS2?+g?1(L=|Yw6vv8NJEV1_Ud`q8s*ucEv;6^C3uEXP)Ew53hg+l;T146DQQWzFZ@DedaRa(>;s70Yf<Mt9TD+YloG>ITdxs4b}Wbq6+ud>ZbHjNfy08lI-2*E|Mg5TWI#qhf)01P(XU>YzmhCx#}#rOh4a%qGEA!s<I_cXPeRmdiP7|=Q5$wus>REp&q{(;aBc>HQ>XECx>0DRwTI7}1oFU$uto33I_L&c<7%#q98DcFVb+1n>{H3>5z+~J-Kb}O)VLCd!qv}mVdLtxwl)fW~a>EI;6`&V1iE65-Vu2k*qPc<#F;X1zrjC^7rf3LFoNoI?a)K$_ZTl>)T2B5*BVE^}6n8wRvz;!Ik%t2~TI&ttWi(xM3TWqY8d302K|7c$V`$sBgi-tSNod~olH&LS(aocMT2KORL!Ry&6_~oQyRMO|wt!RHe>D+y|9sf|9T}P~6(3b!8?e-6_(Lb&|e!iTIv&{nkC^utR%tbNd{=KMG7{lMqc^`T=hJTYD<w;tOmy}{IS|a;|T79y@CW{7)5QKI<!15aQUjdeH+atyJ@|?1tMDehczRZb_*`w0H(635WXOTv{n$~@$?bn_=y7tlT4c1@qcFA6zYwC@FPcpzHYhR=sRApWv-yS1}XLba*w&#2{(FXUKZbkMAx%K$ks?#y*ba)+OPj1dpBi12)b<BP{v0`C`B@rga1g(v}vDV`$ixH12y~@XUG1os4D9~qrENo=BqX+2tGsC8%Bduj;E7+C3#wy%1mHH)|8M?gzS{&BFa)^w{{MDm_I>xrrZXc6Iw<-LRHCW^N0+WnkuVqUHAJ6!7iV(&nlaiFTOV}f`eAQq}GiV&#?m;lBVN4Fm(10Po{gomJ|JL<&%UV_#{pQ@BCVJV#IK)4!-SzmmW3{VUH`I=$9wd9>zVzhj3V`CpsTh;FCA#<96Q0TIh?gH|cW~&}Y&aikpVSX5v}49<$80|09RnxJ*xfSp_NU!5yd!GUus+;p*Km;9w&CM?*nQ)f!j63Vp>_@jf6a&V!S>FM!FhL2b5kUb$2E>R<=F2S_&6MF*tatj#YK8U=>|v)gG%8D%<TR~%r2?M<wZtw;O561vJ4!Ap9RdJuu*obEu7F_<R{AQh#gnaOVhr8Sk_pXv-l>xVL90018UmF*&Rgc=Yj4T{P5kvJ{Z=TwtRQ1?<}t!pKE3XovrfGB(8Th*u!8f++V{GZR=UHeTQQJd?1@P<c`@jh&D5>dX8OgI9KRltkI*C_|0Q<)c4%#8csMi#83KOZ|7bpv51TibJiN=D;xpdYmS!7HOwpeXlCfY5xipCHi`@5hGt>Ok%O^WOwbJg9%Rl#EjW}Q9GZ!f2+?j5-sYB#MZ;@j3HYG?h9d~s{oc!j#oy?ja}j_~u(z(mEv9P<I<WyEG!GDb^B{ICni0r$loE57P=sFdz0ZVC=`N;obiRsKoAL0U)4%N!Ee_WCC~Wy*1S|9z-EBw7VRlC~5O>_X)|i=WN6wPfo|CpEUb8hK+2FzU4t;0s;03a7Yg1Q2^oG^Z+iqyYqnfoCg&jBCM(cs9I}l)GRr3@dInFk6Te8<iSTr=wxP?7G@cGVr_&u9lS{EawqINw_+%Rozm=rXjI71E}Y7e|ot_^;$=K>MI7e?MNZM#`rPhT0&dz;D(DWmDwVs8yk7dP1Txlz?+5pLV#={%;;dM^UT<-+jbVxDi~TNZI%m_~ZTV%!GZq}rmVL@;o;!{zc4=xepUiEnu*`o859EkG(~mfJwYH!X&n$wb|<e9h16eHuI_!+8DV-V(T>DrzrLR}`Q^rcnYCdG*e{V8Q<`Wjt9gpi2r<0mQ$!Xh(6U`#Feu-OoKFn~C4Guvift-e)h;=(86dXs9@;fpqom+(?EN(7@l?8L73oBUAq9x3G#BZ~_hb;o&+!b7+z<160gC049Kr{1Mfs*CEp>;lNBWrCClc6o)N0WqnDdT7!2@sWrmglXFj9lps_$C6Il(DwfOJ)n>xBA>3$JMYrffVcZRE&b`gxE)sP}Rl!{`U=LA-^P{87GBG_d6u$!7@B3QU3B8qDuFJU9q9BoZisAAhzG5<Ty+4j?NaP*P)2Q2yfN0NdxHByV?<ajr1X2TrN1D5ptgxp6GRH`He5+h7;Qyy}#`1ySsrnEHzCqZoGe*!*@|!vCG<Te7;0$wsQ;Y$Xuk6xV*=1lYp%aWa9M$#rDPS*V;N<ss)e>-gqFYfZpGM)wLTJ29>WYY8maB_U(jh0xhOJ){1w-p_X!}It*@0a<^>zC}*@rXnxcJ@rbYf>`Cu#I)rwi+L*8TJ|`uz-k3<icPU*qGM>0z;WN(QBM8l9Dl3q|RA@U(>1lNyVWkfDr%MEI&b>UWN}x^DQ^(GBm`+3sqFYiYwGu&ov|e9&%%qYA?A5cOow-NDfd?eP>Iy-sL#%ISx3P&4{(xv3q!V2)}>FUVC*&fpn$R!gbTfqB7C{0d@_@<kMs`!?VPhy7<Ace-l<-{bVy@6-^{uxpMB>zMFw1@pvs*puR6+$9yQ4WsJm0jT;g=*W9lkBCbL0UIAA9xr+6AIZKw#jp-Vl$TLH1k{Q^dY~wJC?sQH7{^jex{VS0igkf8C5Avk9|g)sf$~wHd=w}j1<FT(@=>6C6exc=0;R{aKpHGt+{WYKX`|C&hi9lpOI~ZJ)KPRbB<!t@yS?3K@%Hg7{$-xUTk|X)pZCY-N$p9z-TNf|g&o7qPE_tDy#I;%K@QYLCuZH5`8Rc7?miM9;85fUvQ6%BBx*4BKN3AU)@_}R_4)G`T+K>RrP!Ni?9tS>?)Htjm!4H?{=;cocUa>7+}hTyrET3R6&8#6rF2-KpkF_)zEukgwvVoLtLF(3BzRk+LTjpT-HWPkKa2{M$1f9|(7NP0vM=-ib|JgaMmg5}(dl8mM|&lrQATuqU)&gPT3civMoBQ8mbdFNLyI|vsy55gha1ho_Ol|smtMp#W_Zm9nN^w3677>ctOa))*o|t#ZSB#m{-NF48RclyAerkZn9sE)h3=>pr#@s(+DEMVm1C8_sSFrr(%$C_SPmk}GK<)7xy3No$kAq+r8;w&e)HYS?|RW3Z)Jez9_LkF7L|VagLZ=qGy;n8A14Fu37SvN^7$&J@5p5zJMA%dWA}B}KHY9?a2+qWjvw5q4NjyYBT(pv<?8}4H_*n`4ih!e>z)q$?XkKyc3qK%6n5*)2?eK^cpQw*%-K!oJRjnD&em<9uQDlRd6+s*Jy?n?!VknQl{iY=hv8y{f=#m;%y1iVpIMSWT+}wEA!sp7k48D(&QW>|pc=(?RTU#ly8(DNguO@ou+!bHFhSAcJ~0^gvx{ypD9bb&p2F7Au2nP@4;x0)>`+{i5pM%Sgq&$@4$$93CqUw>cr&=GmGnG`bO{SWX8OEG)QHtt=5NyyJ#FnePShH;Skhi&ApVJ<L|*uC_W-w%{M4clZU8c2aSBA>h*7?XmW$a9Y_)vGvb<sH6@3^{Ud_EAaZ1(Bi>zc$521m*pZ3)vYg}Mv#KTaKx(=@QP}&LNq*QlKS9FaXPxsecZ`VX;_kr|vMiV}398$yWrDY35Uq)v$l^6{${nhuG_EZ)Z=)%h6uqVyq__Qd|5KLNYp!gpD##OOZMw4Q6$+M-S^K4Ztin8>@JY=+C0fa7|R>hk3n@9neh!l&CYo1`Ata?}ZY*QFD>5W^L??#e>M{Ij1vh&P40#SvQi#GUtxd!B!(9SP_C>8TSx^?7fc-sYY8ulqNM+%!q!Nlzh7e}?YIJ%$im{3R{<@*s^#Iuowor60YKSy==>1Cc3UAJ5yd%@N5Ue(AvJ1%8|1Mhds4f?JAZn^!Q*d-)7z(dU8+{T|ifPOtQbYU~cDm**@2F>BI4FFy~yUuS)47!LsN>$2$0@}xpCwFQ_TDL+^xq^%j3!WnwUn9(p2RY6+GaLX22C4deT|@BwS6=>61F?>BK=tJcb!s&5MFe95JT8%v!;V8vY*Iwr{0H;$FkKyQga?`={U~f9_~g}|!%R$}Fas@10}+RY;b`UDE{hpXDt5%cb#VcNgm(--AqGX7DC3!PEt6fX5^YUlGPJv#Q5@>h%$29uamI%8j`yQ32wG7rn+aG-AU&v-s|e#oM(9CJEdWw+jKHbKIO%Y5;@BuKFO3feD6_n!_;OvGBQd)6&MRA<r7?fijI|PJdyg~IC_WU`bx|y$_$2jVTQy@ytIIM-EMjXSjv_sf3|DF|*3oZVtVbkQh`Ucjxlo%Ov(4wvcQiUlnm5Wad~wLlufd-3O0(%42?Xw`YQ`opWszWsc(7=(#>xOD0E<)2Kpdoh16C0Rcw7*zPJ_`oQ^_v9_FWktn9jhQYnnpUJ4&qO!Ky)c64|*<qg<>+OkjdYxx?YdGh`h*?IcN6fg4faMlY~OcYGZZdy%M;v2JqDE;kE0GDt3`bTSjVuv~!{JH0BzmakvZth!v>l0e0Ddass?3zifo_N0GoXQXf7l#D4~k~37c4R(9ehw+}M56q9=F=B6sWmr;>>kGhSE(Ng&!7Q$wwCm}5Z%;=XpocU<nn$3BEGq;ZCpupl<PQAm2@0}+@sBsqK_GDWlGD2L0&}{-Z=(91gcBp?^gov3w=Q)KV0WCg+ihTpj=P>o_4IR`uPUz0v+l&tv@pV=)ihcSgjFyjI`+&nl<?j0SWPKIU;P72&5W|BVG1<(l|+lyZdmntQ@!v#`o+5wjj?a6ljBwz&H6f--Oaz+`i_D#u5`}nO%58~m>b|}5Os1IsYy|Qz|?WD>5=iole#!RpN^&;N#LnrJvqG53@C}`aeS_w8$qVxlO7aI{d*+4jW>cjM9YAVLfc(hyC_jKp&!)-WWGdKm{gjwgd%`I0X}Z-)y`!`nzjX6c|;TPFfZ*{JHWtXhdJgxW+flSq@<BK$riZNk&3akOk6vWT8*Bnui=ia7-pE0g!RB2LMj$m=z^6-%Wp2i>zVP@gT^Jiw0Rk(skD8DB940dP~SYh2Yzzpv5`x?q2_omt?>hEjGq!}pqQx%e7y+f^XQ*__IdQrnP|MnBIuySV0+Q;j=ntl!_i;h?LUux$1cc|ekV$!FXPb!1V6yS{%IQhA!dwp(j%aMiHXBbdK|hATg^@{dVTb}qgO{S0rqPqJ(Nkmn`#ZgPK_CllP%Bbjt&aENTZiA9MtMn=d%-i{TiuFe+4E-y#Ko4k$CmfnIAPZKoXN}njx8Q5$&v3r-P5j7|HAS$;W$d_fOUY5(J9?N%kvDS^SN|VnkF&&=;+m3el5_8K(SuLKWF`(yc)><+mP)*LpBN)kcn}>U`IChKcVV&U;jx@io~tL!P6m<KkBhV^hZ@WWraoSd}be_?9x#OD>_JRl`#Ji#`etIR=ux&MC@Uwah9_<}Tl@D{OYU%JSK2l9$<NGsk-;f(p<XsZU^e;2}CT8iFysed{pK4ceM_>RYoWV7gpq!<Doe%A8@^k>h`ls!5=4R5kkJD>xz7bE(a^%GXm`160;x-kxL9Z=}(@$ji-|WNOXQ7R1!!6%>Nwv8`UeL8hL-f+)};!F;)@K&ZVf08(B?Kw0>WD#ny5QR$403(c$>be|2YN2ShQN89L2_#cSp1FnQX9RzVFW!n>kKWA2YSm^Ba0LRQVOff1%VGK!1_dGBXUXtJCv}*$J>t{6YpOg=LR$3yTl53(xnUw&E6i8{MB>+3hoOe|QD*~)}mf~;B!A?uZ(nC=j@2G@)9dlTKP8yuc8N(sGT%#?RUCv^x*Xhe905|}l7kWE{6N_&Dq<tdF(1UBZMEy9Q<y9=2BUxtEhFWo#YABqR22KV>3qHTX>*i@$IN@mNE!otc=OA##^bGyV=fiQ{(=6o{K+Z1ZZ0JrboOG)B<H@Xf(Y~6%zU&75tHEFr3nS_RkZzu1OD+b-g9E3ClgPMh%s0@L^C|L<EwV{d1{L&(Y#1|}0IY8Lf~(RdDlm$xV5a(j&)X9?fK3Vo!^4{kiO@b18$U2W<1jRuqyy8yyAxDWHf}>#1+Qa$=2PA_Ls4$<)W+ZO8R{t0=yc#QcU(=knBWf^p_JgY(P#IZ439)EPA{5jc7D-JX-J|FO&?y+da?0Bo4wIt+lmV2bAq;fq^sSA3eZ>E{x4I$E}?jZycf0xXSB{VLv3wkVg)r7vNGjmgeomtS*qg;vLqa;S;({h+MCrFxuEywt@mv6-96V}fvBuKF}zCd3{0h7GomQQz@EbTdG<Z4%<^Z|psor{kv`1loYZ012>~&Qo=NETc9N}e09=5C;|Ra<u{nG2YBoG{1ER`1=9@4{yl~lQXYjy#Q^G9p9$}G0cB}cyK{c^#@bk`|#Al*ad3j#UhD&oYsOdf0<n5V=q4^l33WUkmiX$-NiZ-GKkLTgzdH8r9KAwk<=i%dd_|cz-+7*wv88v}iJ1$K@S9hSoV+AuFjte3i<6bmeE@#pmLWb1R;)IEqTe40~Y88D9+4e+$C`Ze6fw_)hF9Z7VoGS7;W1c$BnT>iO)E9V;Pw&~8QVklX=HB5Mnv-(F?B8yOu3g04tYq}sYzoz?)AO5*Uh|cb#c9zh=J<AG&z3y~&oFy|ZjgF{*-goJ&G`b~>DBn;@}F5rb4J2*Sa}YrhhQc!7l^S{ysfs$11pP<J8d$sGU_{9#}9+FR%aRgnJ!@FigK_!2A=0=WaDRADw(H7Q&6dQeg&EQuJN?IhuaJuX!Vjk(M~A64(gjgRSQ8Rg?ZSvq*i_RE&X9l><$dE5>iDS)fRM>WdUY;D@>ISpzRI`_kgX6Iax_>)<qwYp8#az^>PIpuz%7%nCuUM++{-L3&?rI9d=a|2>Yf=@qxCE7jbsfA91o{PNMPD+z!__ot}5i0lJIeyxF^VuyFEM^UC&uS<jxe^PEci5eK5hFAF#^MnvWRlCR0M?(LHwHY>_`gBZ0c#%&6i<eq0z?av;4#6t#eXi7#JKqhBi4^)R<?1ZJmarG5!S22}@*amD9nA;z3-LxHPEQ7H#qu)j+ZRawcH6}AOo^KW-cI(yV)r7)i)i<CeMuO!fhjcwD77=BjJJCn9z$@7dQ)-bNM|2#;tWeTEpDptW?`OFQT8u;hjVEwibdkIjw~NP7CrS3_ejK_DLXXiUXy>H;Dg1GlC}B}+;hRt!5nhR)w#;hP*Pq=7)O~ufJ=yQ1I=rn0=hk|80c%%IuLRY?UCa(Krkdl(dsVE5OE|9HtT%WV<g?i@A6@dZVSibJ&-nZzvq0ep`A;B*?8nX{YgapaoM@?be<}Krz2UzYW6Ym+HHzy(`Pnl!>m+wm){69O6^HzyDGtOdP1cgPuYFI{3^RMBy6O!JN<)!mj~5J8W2#B_qL^lzj8Xg1nW~CjVQ794i_8BS{I|N$_p}1tf<*}=Ohgk{J*LPaHt28PuXFMFk`o|kNNae3_5dIy35O!HQ^NZ{tX}ln|A!sZm52BlW>Y-t>2vCLcBg{e<DG`(0;i>z=c}0M(W7mTQkN>wscMqDo^W4X?yKI=3Y`ql%|4TNt$;M}PKoL{>xEw5mUr2PteJCtgP>Q97Ey**)=z+{pkVEYKkvr#c+Cu~$1yXEI!P6{gksQ?qkij8iiAeL=Z=1uI5SKis%4eW`U(_|gwt0xY}u~Ob-BGTrUTX6u|_S)N0`oNdsb8#a@J9gsF2g>hh=3&CU<<2Y>=hgb?8ME11kViH1%yec!xdhv(gx~Ckcdxod73UEHY+wJem~a&8!H!-quBDz{oo+QisQfrWzTURe=jRt4dPPl7n{82TAufaqgA)%~rn}3B`$XTSZqzj*AG;9X$Nd5th77rI5rMpzj=dxsM|v_Sg`m*pLsCry9#ZnruVN!+tI3XTSEFOc_l+z<AoTl?RoeI0h51R-Xrx<1Gj!!9Y`jO+p2>i)u&<%EVg(y|hKWBi5N`z`eJD5yuDQ5#hThO2ahzjmdrhr%#Db@t0xk<;(1;ACNNHqQqr)mIV9PlfUArTS{jXd&U#n4O7=zi@n#o(_D*_p^2@J3l{om9qbNXS8<QEEVP{116gVx-|S(p8*#8c&h-QEu2NIPTtka0FC9hE@ZtHtJz2jR<_}7lol?8Q-?d%b`Y(>R`2n{Bg)L~M56re0W8Fb38_-Evgj=t6d#3{=SO{5vrA2Wd@nVuwRDn55E&g!qP7SByL$&*z@DFgRH#o@;I^urqXBV{dArCE|_Q~jnUpeGE+}t9&4?N@2f+3uy{^)#Gd*Z31&6!K{s5|&<^Brm`SybwaCpr~g{C)~7bQ7^|rOq}CI_3cx<Buxn40&bQ?>d&D(L+l9!%{d!lANx`mV&+v+J3Ez=m^yQL|35n>e3sWzkR|r_r2)%(DnCF{mzJFP*OTG0(UHwK<^dwJ4d}APzUfilAXU-Cv`yg?5=4lWQHi!nL%2!CPD>cHky?5ZY|$Sw$tE$jlhb%$N%cvPh$`s>h_AZDE>oRbbFFCB|50LW`u{PR@;x`@pKN`1ASFlXZ3w<>^dxCrvs#F)B>_PFJFH1?GNmU8cPD?;Jze)fKfYTYBP#@m)e}-YNOLezMs(QVjaz-Sbs)Sop`!|JsRpfSH!ShBmBN|Ox$k(-d#ZFhPUfthkCoK?2p6PhwK5ZVrhmxf8GB)jTRVmssAG7@cH?r7OUSqIpNPBuFz!!?>`dg8mY?NZVdEvvC3fx;zV7|mX#=iT`7$Z`ehA$4C_-Epy~PfrUb&Mm5%(IX@ja07#5mZ%h8(LwqeqU<>36%jrDHH<^esh*9HFAGgSOusKX~+hrusn5705OjG7}Jhv)zRyf@OukhaPGU_`8ZLl=4A1PuWhjeYZ)%ZOSO_2{*pCxb`!v&+zX)0`FKb&``J#$xU3$t2iEGR7&Z^=8DPc2XgD`2d00t;o3g6cH>88OqFv-SD_IK}bP3IBVe{R*bDM7(9AK6kt#^*JC*f5!O_BAvXxSY-o%|9t<$Dw^69e(<5rR$NxS%jkvdYO|UFnAfuzSHY2w}fQxI`y+98bdKhuCp7=U&NFeOcVn1ADY&i-JjnfMXycLd-?bNbt=h>YEk2sKpX4G!8@dRh0CYx!y7ry9JOzSRYiH$BC(%Y@US|I!Ni>!jTc}XGl$exsZ!iO;7lE`3$dGWhDNKy?_^HE|QHbT1}9ytszKDQ&^@-oN<9XBzRx#uji^~DfDSnR0h(CyiN73ex6i$qYwVqf*zIEN?!IsutUyN8jV&w5mFV8^vyN0o}_;A$EJq_K*A<EU$;3Wa|8y}kn*rJ~?zXzLW4zT*`xbE`v1oMBH6`}QGYL#b?E32y>wzgd)a-Nq*_Q0;bypV(~!Ke6Ca3UyM*UxqjCUIy9-p$FYCzES9|1z7{(!tj$jjlN8yKcvxLLQpR-cYz+{&HxI_v=E85S<dZ+$hv(1K>S2GycY^$3{Oczm>8jsemgK1$hMmkE?5}HcL=w*QIu_M6QpBTe~)wI0TQq5pLE6SK4G$-7BOMNr9>3;j@m5dIXXJ5iumo5KLLrO_$^-cN=jw7x)5UuktFvsK{C}%tw`+e3pwN-qOdNmX<|0YYeI1Ir$%hhdC_shNGuJ@u!-$Bz#FOYoZ}_DXv@EGu1NrMjyOk(?D^6+vd?^LQExKDF0CFrvgrsi<|GN7oLU*askBx+>Uu{ima+LXZK>a%B8|S-R7)Oe<&UotU(G;v*Z{n5pcB=!y7?ME{eWM5DE>48fqV(nqFl}L8)h=ghk^HG+Y@*-ofQn5ypdq*H&*obd``yEOFk+7z4pu5Z_W1HeOvHrP`vR~_t<7OV-ZAcUOwBNWTGcyXCJ!ECR6Wt@rdHB<`H-B9HBqcIw}G9+G7GTlzBk)EFZU$Z&trgJq$Xx-DbQ4NY!Qhnn&o~9cTGzL=m|FZD%tINDP0i_(^&*D49+$midd`X>`PYV$bC(_7i9Bq#f+40LAWnFhtVc8|fQ1_iXp<*gx4pdnURke)~5K>d}F*e?3T*ay|KJk({Hg643oejjD)!WJT$hHwHN>=x9OqUC8f5#octUW*w~~+g!<ybs@!^`&QBwU2XkxZy$~@((GW%fc6b<t+hs!|CiSbVRYXkgI|mE54_#EfAc!Z*{R=P$ogSj!+Ys7#tm~|+PzlC&h;ik2Ww#&Du^EVjS%oY!;@QW)g~slEJjsl98>q832{Ab+=e9Tt`~I(;ndy$V4qCj%p@KMM*2(5I9+S}&f}VYp$DhIv?bGv&AP}$Ousd|5YiEMrlLF)>;^hRg{~gr7IW1IJ&mT*mR`^}Ew19*m!Gp2&X;wt3RGU8zh_96jb?dSW*PiiK>xDrt`~iQ{4AdY+{d!PV$H|*iTBoAj-CH_rcJ^Qt)?+M-ikP=3`e!2awEj~xn`kh*P&U(onaT+UySqlUt*CUYPRmr)*dXv<4ah@ul*)-Ef%mtOF$*cXKV*px4_~n<#<9dszPCw5hGS1_JVQva5m>}pZrj;(_yoUUXioOtM64Sz}GpGs(0bco$vR4qTG_zT0PlN43mkomoYb&zG_$e3eor$Q8^5})C16PnupvuTIcKH-3G+a8z|{XymUA4+Hajf&+yO@9df7@eBXVQ%@xty<=n{BjYqxk&U8$7?c|c9<!reI3h18N9F5s$%c}NN>Fql#Zgq!6+fK}Wn0~2%E?3<#MwDPdSo%g2R@H7BogKHByS9xNlL-dgjNx~_jT8OtanxzM!FHKB0B!c@rBLtvWsR!Xb-bFjUttH3u70~DWaw!jX@LixR$0d92$h38j6cKoG_vWkK6T{=TS(dxN@9f&Qp!da)j7P|oR)1+?)l(!tzn)3OvT9LEs@6*f63z|4IDdz+U(!xzR%BrUc@6)bN5RH!}wxb^ZEPgweHg*%vdu_Axx4zbM4wED&2f)NpQC4)+k0NvfwoD(P+Oe1$Dz_ue51RqkXy~g4dyXxjz~u_e)HC|EYVZagFK*c4(9tL8e&*o+s%OYV7$pFTZ_0Ct^}Qhn}KBak*%(Zl3g>P$V_j<(N+kBq55CFf9dTe7oSU=6N-d4>Wr7$#nC_DAZrSU9fb9_*p<+z$6vW+Gftb!UN`L;h*L0jEF{nqo8t(QGs!^`8d>siWi$|Iun2>{mp7QUGUhCNCS#3Z00hC(^G6^K3i9qXVEk%Vl_64>AM(uWR`!A$u3aI$I>BD&NC=<$E6KbT?B^&_<%|R$kf*pvo}WX;Amy{!(w{lG510ptES-O$ri;0%k36d)&P&WL%?)#zKkoLZvsy%1%M8OnB9cR1BdLc*gKVshkYw6RL^(d+(XQu0P7gIfFm)l1M`PD7_;~oilg&~Fc9GVV#Qb|o<`?IG3InyoQ}W$>i4hy90O`KI0@$ztcx;<Y^uFZJ&-tmWDLQp=>jk?wt&H5-%V;x6NAG^BTNdSX%Kz{&I5$}&EhidGKylPAP`IeH1c)(qL^)Bte@yS!q+e{U-Kp#&V}r|2wwnri1vdQvyCv_Q*jEKPS$h*Qb)$H0=a1Xbwr@GLSoKTY*hY+x}>LJfvG7*JSZCs>jz?3(F4Z1ATib@RlTs8WGMIhuK-=NHWI;F!noO<VHjlrazZICSRm1wPO09zy?DuEoQR+%X=psM2MTYszBj};-8LNhO?<A{)hsv%;Y&O>&_)lq|KCC>I;tpffs_R?pno2*YIKbyJUB$)ejSF)vB%0ABBj>ZReM}i`Dg+dMtzG-g1{=P{=XJUV_>Q>p(wM6n>>Dgzg})uScCG%Uf@&NKZQMqkD_v40|ZAAl;{-4pQTZL4%D$wcQ<11v6cEloSDt{-=xULC2l!fS$u%5>_wFb0AL~g)zTqN$KztbccFldHyUO!VG;E1@`_>dmrTH6eN!-wqEnKQhSMAyW1%x$Fl<LFFbV=6Zl@`_4#^x>Bz0<AN#R&`way!2+q*QnOrtBqj04J$?FRl}lMsy(Va0*(hT?4$!`r0Amqr@8;X{fg`=@&52Kav2E|=$(2+4DdNzILHJ;=C8UXXD$acejkFVf+pSRDTrexgkVb~!O(jQJ`RDb~bF1<EuWJao*ds>V)KwPg*E*~7&8RERP6^$}0{xCs60#7|1tqQ{ITsr~Q+b5i^PitM*3JR7*tHqh$Pu_K>cq77T0WHlenu*$yE!&7mb)+s{+75kpWFN-x4<qm93DzI$K&C2q;>irXYoR@zi<GlJC87HzGnNC3t9`0eIK6LO0%z<Yr8XYi`%ryR=$jetG5b-3_4-{!tX}-S{?=Q{w@^HFh@AYtXqvd?H0nT5_*ZP?^N+&5Ec<-D8&pgbCoNH!{jG~WPR^L>OOS^_R=Jb4mnS0<&e@urzUczxRy@b;N4_0G1R0aMy!X!>rJW7s^;^UL{r->h|sY?~7=5@y8ovt-gG(%%_Z)b_3%=uMgyJ|W%RRx=bI@9P!R^!KLs+XSZoC#?2I<spi_>+wHNjqvs;)v}}5+B-$J!_+kko=xaGW~@864gojovk3FtZx`I#`y+(;G!Qp`9Ief(a&DoSNaR{om%^v2&lc)dh5zv<tV3I3kv>GglJdK8hf~2L!8&9N0Yda<L!s|-Ae<2(fy_%uJ#8}!*@53jy0%iig!z=%_L5xskGuD8l=}?s`c@_)46(sr&oNmg)B(cJro%iM-=f<W}JELyH9w~BFt4iZF5)L2psnIIN?U%lZ2;F$K9PqF^6Ir#FvEZlA-|lK~Gd5K4V59fE)=T3p++(x@F?P)q-S?V~2fZ=4Yy}%(WTXYPZ|`sDNLIZL4nR1p$De350GNx<GG(<R>9p`%z6xoY1IDs>f%e&7IcJC@+iuJ67H3i|<H4dI#ipl2_4nfyoU?b5m4Nu~=>{Fu|a9W5jYaV_TMxW~2k9?V~TukJiHA(lXus)lNZ6qJwZbAVhKIpn(+iyS$u^-T>?p^`pi5ceCZ_vbk<yk#QMrOh0sL?mGccp$KbnE+RZh-8fTs>?4igRCQo#tV2rDo4%&|P`wlfkKJ8Cf+zuYiAJZ;#<7QiVz$xesG!hvp-lT=!eS~;F=Ai$yF0tD&!A4Xn=!2i4qgm>!9!9IQKVZ5J*1H2V>|qG$#04^>AXzksw~%SSnjkzIfXcQnQPpzG&(grT1y3m*>7YV08m<0IZ3S)4rQFAc9+b|t&RJ!m8m}vvtu>Ca7dvE&^A!CtBQ7^sCJRZ`y8lyey%E>x^4JO1h~pYBNr~E&{eTo!Dxttg}DQx4dF5Rn?>Wi%DD=coC;E*%_apfOG6*$mw8mohXqMr921xlm<6W*?tMr%m}3s=4Z0k#l+b80Fox-mDy=Iyi3hS>5EN!NsSZx_i`0ixuS;@?p=Xzli&Ztj{PT5m@}aa+K7N_^5e|RW$^A4}J{^F#H91t*Ay~y<lBUCEe--j%gvTWtGQ<2=XwGIfaOOW&HNH4_aLUsoK>IxwcDSj=%A1vNEE#yXX`k{&znIZiagSwr05#72lsN~hbnKN+4mv?FK|odX*bSe4VqQ$I3QXICXS`x2gm^(=^+J3`{pG}#?K&S>C)JW|VW#g^=c8gSw+)m&h-{3O3xi2_ci5&a7_Xc*z}7vT>ifbw;HWq3`+}{b;@J9bwx7|=d5uwQBaY{S;xgluW_8fFft6?H#t&b7`=?jmy#9Uk-RoEXsx8ivfc45^Ghf}Hy~<)G3|8Ka=faKze;;atLT<q-$P$_+DGnypK5OBxRDKS_NKlIwwH)F}T_YiIw?SdU*?;?FM*4xbPs|`Yjn6t^3P^u#gTe7?o|%sKykIEezhS0J8O+#Eod&Fniz((i%7&Zs^FkPeqD?djYir9CkHU-7dkUvW67Cx^Epctvu^4w#!BB6kFnf;QJ{fJsx!*?w>p^d+dbN!G^^BRTG+S1!>-sne1;hjh0S_?(=Z<3pJ&&j52(}Ya=(kU}RSmAEjQE5MjUL?4SYWnLQ)`i2^ovzn8DQyTIqqAnv%7S&R^RIJA>Jtr>Pm+iK(EvaBd1L{#qcPz5p3x7QaTSaWHi4yhNtFd!$Bm}kEq2eMU(uBal-|!DQvYdvkS{d6B*)8i|t-5GiDLjkF+cY?a>U5H`s;JJVGKFSxcXGl}n_Ez``9Pt%bJtPKyE<Q<z-VunvU2CCM%j=Je>W%rE;w%2w;*D$q4g$;7^K$!DX@j2w(g#*WDVChy{HRz?3B4xabbap#V?VJ|_vS(m%Ih^A~=c*KhpGj_93X0+PGNwRwa2s$ivHlHpwWrKDoy6l0CVbG0+ZJ|fveVU?it&7og)oe`9@88h8bDyR>iJzVlJ(Vu<nkkfK-bUy7^kSY1+RT6(4}`4-jrDK<41=IgW#8Ao`+NBXJwhEKYw#c+q+P?2HB7?&ORmj%E7ez^Sxq*(En|iDsVS(G{i<QVeH$L28T(uBnGTjliOMV$#7|<9ILO7cVuBPFga^Tzt4P_=nC{9$R?&GSP=NBKg)t&f3jhWi+h?`SkxLc>W^eb*B>P2~cE<RF`>HOWXdt|<?wAk=!)azha6T_cDO6@9^XDo8#oU|<P7xWYBAkYkvywFdubodK0JihN%8uq4*sg48-t=gmDfXpHs}KF{jv^Q`Tu(aDIq_t_+$`iG0}@_rreh59lF<d9ci?7)1{lKKzgOstM6TC?a!;dbGA%{wfYEY}%3RXv0;ZF<!aT1~)WuhNyD3yH+81s7CZY#c%S#ZUD8_};kPsuC4`xUU#?2Wrjwvhjf6zo2JQ<?!(eTt0x`xLP6S+8a!g=dNQ|E;Z3v}|oZN1EadNMUMa*-%)D<%K~byJ9L3ab&lhkx!`Cfi*bqS!e8V{_K;5nNgBb@AgsI0oI&9}a|96Tf?()Uhu0(PDWY2!>VyeUsIRK}gsvCuBfyeXvwdKC-F(J`mcj1X2eg->$4v2g2PaoV@#=PN}uu2STeQ>vbO=q&?$Q2!A?w(_04$B{s3%9SA$wBPM!1duCEjH|y2z%vOi(_@5^OmceHPfjwxJ^k6HySel(IOn_z4wUTC*ElnNu%g(iRKr$_lj-2O+@Zcs-VC%}|mfN$@3z_{^1RD$hx1%UfY|yNrWRb`U8OgeY5NCEni}Kcy#%zHiq+{7RIeOKzp6b(}F2~vX)|Q2}Z1q^avovafcsT|pL%g{&vk_->!9|x{R)R4t{nyTg0jd3bHeIP%N_5R`B@63Wop!-=C(dC}-#+<%v%u}em{Xblj6z?%*BZXllB7N{!>w9**3_Jcq-qK@Osh<*reG&d9x`<4EmmewdC`u2i9*5PX;Z3MI*z6u6!B&-W3hwxPGa^U(_J!}WhV|pN#)IVFTd+SXAx(o3!r|}k+|26R?ynNc);vpK=(^KQ5)$Tt!!;<0s1^Efag#Q;Amqupi$1%`*OE+{Pu$}yAQ|Set0JG!?KO;G48xcJV}B%B?h!a?0on%bsWB(s!miSN)t%9$Ec7^hIXdiJCo<lAe@2EQ%jslvKD{ga|F3pW?0(s%4UWv^Y~t8HryGD49}hagG}+>?Cr<Hkq*=7C`%P%dc+NYFrJj)Nexag+@@d+5#{Q-j+oeKdy?|G0$M+GdKD_2Ex$QM8%#5LPP2G)b&#_B;$mG~&}FND>Y3DC>AbiJl^Jn|h1W2U$|+X)ijACNH8d|8*)jKFycw&I?wCc2213Ces7)Eq!v%=$1`;fs-?giM6vbHOSgYcM$Tv67?qsySV2{&VS~o<Zw@-*Q^!3|n3V}?|Z!&tV-OSTm$rxH_Q{>4xZd+y4S!2*v+9?RJI*zpmziF*E9N4D?s-P)%eb;wpgSA)s&6@Utu<rrQR(D^MHEfzd7<%?%hPL2ZAP`#Ma)pSDIhU900$~F$7uD<rdCYP;F4lN!z@-muowCZ|=P4*33XmyZ#xpvq24VR;7Yn@u5Ea`AjmBKxFE%Z^Az2Go57W2Gq6xbN!_b9IHStwbGNUK?Ynz?ulA_?bC@`Vhu_N5Kc-IRfXLrPA1V-oJ@7WP=IMbEkIcXxJleVBOplJ+`W@a%(dV#|r*A0$TTO$ekbe)k`Ah$|k>whek?&zqS@IkPFX&K{qnjNvoN19P(ERS>{Lh8&Bg3{O-D?X*h)<nOJPV7_L%<f}4MDuDfl>zqH6kK$JT7l+lylRAEb)SiedBz%at!=5IWWk#m(DLH~Eoh%cs~Z}EN*INW!WnV}+neXLh*&7_^*+(z$i#NxH@gkm*DC96Ud;-}qap=S0vb+yGU>OXr=FKD$)4ZmH-nd_lyJRZSEN%BYLA_sz-zn|LBLhqPK+-;%!kt%6Z5~tF$hK5*Ep)LbJ!fkJARA$yUlbq2Hb>AxddKdrn*(O8+U`nP3y8hDWw+1)k0ET&M<iBjK|hw_RN!lG(79+N&xtQZ@<&_rx@kw?Grhp(KluE#cUS+tK}woU0^N=Pp~J>YRn2vUC1^TS>SdA^_t|fbM66S%sysX=&~a4o(!d`2{S=Ef&a)LMb<r*Q59KP?!LcWar?-ed*<`<rdCK8Pr?>DhUx|&_!n&|IpI|UF^D{GQ-0*JYefzS<Vv9kk5AegxE*EOD>KZQE@JFNDn2bsdLEd8Ceahvn62MLU7FUO$a;ak6B?VXwPw;CD25HBQGAki(v$QgF~e=aH?@Nn%LOwOpH?@qjHm(Z!1c787}d5%VmudfKw=}}nAeAinGMT`TzduZQa|q($m`E5uBY2GSW!w!Rum<!_OhCnwJ?x|`0YlJm7n|(ju%(Iphz#LKTV{U#-YTv0kkI_%4#`7b@UeX4<Ku0=Q$}Ky6qF@BGG^Q1d$!KwGrxj*Sc)AB%@becNfEPV|l&!RiPa^RRMXwnFHs)hU1K{2^~6QwT^W#oMZ_{&N7--(Q*T)+w98D!Sa#dZ}&q+D%!Oe01z2qM!W6M!rK~}4@kd4??#U2l=e8sL^UakxMReXs>f=d4kyrZesz(pmdhCnqpnYUH<k0bW5ax3l#8Nj9uDEi+BCpUs}n*f8W&etIRWl*w5h_%OnHj{62MTw__Wh~Xvbi^k7gOf2;-|zBa+Onf3aG%K-QE*g@7?5EH1lx_J-B#G>xl=pgIq;<@G@{t#i<B-}k7}&new5K*t-D5Pc^)72@%R<s9+&JqCb%t#9<%3dMm?D_B=r$hcYu!y$v@qIW%ue#i>`+E(4!^q=(IEg49m<LpIW>QXVm`i6a&1NGkX{!nM_!}ycetW(XM2iqVpfgf~wA7FOx=cWH3bK8i)VF#!k6>RGX62`);7%}rOz=JM^+8@;!L!TV_TXQ{v)hE$2iwws`bJTN`?RzQl4kY30%@}>}9?*B82@DRb>kjNglNf{flLZYfux10P5D3~P;wk}~(|b0tLTi6fRSU-KS&BVVKUXc8LM)QyqfLpyVj72uspa8N`C@R9EQC`>3*KZ0WD#Rf19*&q{1^lI=N;!`4CKF!7|0)K6y$ep9t$j{f=Abh{T-;}9AL*MX*b1S{LtJmom{6QJ8ZuVhzi2bYj5!4;d~yyOD_{X0s=X?rb1`KbejG+=$EK?j^5GZXmELk(0&~JBEltK)l3`jR&e5Alrdf1M(}HSj*(5vf(%#S;hjHjGS1djdY!51ibe#8fq0idVLm^ICNOc+=!#xU2hbNjqZa5%W{x7gHs;`fSjsZI3I^*l?;<FDC*2XfeR7pArn8wA^F;&%5oIX*oK_C=5~_GQZ=X<j#wUX`I&Y~u@8!?CXtTInEUy<WdAFmAkvWcp>;^|v%!PZ3lM{diiuJ{X1L*e67<VIZr9MaoU&reh%vp&0q07#udb-g>sORu^Mj46X4q)Vq^z($>?Ent&SXMsbNTY=!%h`rPe(2%+PAHOKK3`>0naZP`?jY_89y5v*H+{VJh71RAxe1Ykkz_3j?PBS;S&jy_r>~@jU{3r^io0ot$zy6G$;3~NUf4__n_4k%7<r~x#C#1Be_<`+p7Fx*-WOoWvqC%^%)jbO07dmjf1X+K)w#!zP~=@|Kpm$C6^xTWp*HLvGVd_s4|)>8+FCunC=E_~eKmdIKfI86`FnyN*l|$Ay_GR-N=VeqF7L^FHJ_gQDVg@?y4>qXGCAF>*SH-5=uUG$`-)7L(&SmS!P~#UX2W@Qkje`CH_c|n70WWnEFFA%+V-|#u>xx#3xc|<3;|;64U|O<?~HeA;499Ub0}Y5n2k<`HFQVEX^t-*8+{Q*_4=)mhJ}66`F-?V#to7j&OQ*#3#2`FseK<M!$c>6IbdreJL>UBI9EuEnJTSQwCC>=78BE}_$WD{+6gFs*<G}eJNA5U3q+;Ap6?t-@sDdI&v%B<hgo~#*yq(jKqYxYimu(!tKr$qglG0ON+5qKyC!Fwi^;^_7PXyE$K#nD<QrawCQp24lb7Lr+r#Yu5B~2{{UMOwMt#r73F$f_#8v~`49;Hn<DWep*EXxbxxQKLa}b$6wB8fx10P8Z&zSOZx{}A&!yHG?I=%ym=n)@d-^WS^2(8h<o#*S`N8d9J|M3X?IgUVcQ^n5Ly5kSNHaz@JJOhB*e5>ObCygKPeZn1i4>)LTx<JKA41)M(g$lP7TGlhQ6{c_@ACVGug3)qKrsFzQf`Xol48F(JWrpaz&j2By5k7zg<VuQPuZ?6HRm%3sA3I&Ryz`I)T?%z8o=#QZTd#~`<tOo8w?0Kp+GbuvuyPj|6IW(+8et4F^wy6!fr#?$v<#V;q4~6#3Pd-RelTqSpN}dW0f4ItLVgA^A4%e^W+@WRP6q6<*<0AXlWqjF<G}P>+48y~&;2fue}bF;Oc(s)t)ArIU;*_$vY)z5hd#@LlF`yL4s;MtO3YTz%@Mq30&2lLaFE<49s{l&&0{sKbx)I%QXD6f<UxcP1<=zUCFJm_NOp)FWFbK%J+=v&xDijBNi~Ww-rR5Epz+-rm|YC&18|@BWA0@CH>)7N`H@9)0)g?uf~}UTSb2OjLQlQX#>i<`3Zv$3X2J_puosrWiglP97BiX8^re}>k5iT3Q%v-85Q+0FM^|)AV?~qa$vI`fY+o1XFs0vWe8rsHKAwAxTkY~|yxa_DMRt3@{f4iZqa!%o(QSOcyb>Sm3gqNhdG+zAaQ~?k?nZI-@Tiu5RLeiA<sa4Zk81fxwfxVbmj6X5z`94bhD(~@8JQc}?Z@UuT(2E^RG2&}On%V{lb>GPmfr2fM=U5~7Ez3)gQ19pSX||c3C@fjMbVX>Lp^uoX(>HF3eS5B&qg_Ytx!j=J=%liD~Jar|LM11S0Yua7<vq^Et$%^ie}4eI5)8WhX|ym3@2nVKP?;V5QvGzJvxg0we7XK%~-Y*>Y#@70<`>Lk(yh|bI7~hSWR&V!>~Ws@-wR`9>s-6apBR5=R;ZX-0$}0Q7QFkDD!A2^JpkzYXNS3CNa0%2&U6hRxq4j!gtn$(ru$+r*4H~<Gr;D0anWt(x!{}wt<ZLc!A&O0)O(+U&aelDP2IaTClIxKd4ec^hbzdo{$}kOkshY>WCjGQ0hKPiD2k35X~vihs<=jSGWd>W9#~g`7dtGPIMaQeGc(4_Y78JurysvIdeaFCWCWgsBG>%*8BI*819REgZrSPeetlP?fx=Gt9C{A=ppun<ly|lG+AKSL2>|7wIl4^gNCFyFksX#`I?1i3?tJlGi8_}nu#YB3VGn8n)q}O2tUNEy|yeObMvT0yjoI!)NPv{b=xNT^Q|u=@dF)Ms8_d?=(={>v$RqR-%Y!TxqTa+cQvYrpIWc1Nw{83aaeB(r+nmPLbcfAy5?HzOGv}P88n`pX?OX9uLWgy@Y=D~>IcaE!zfvzXOWl3mL(olEjv^#79xrHGO<H`Pxik13}iI(A*E>BqxhhcavyESFI-Xakr^%c2(J}t5qI=usuLD<=zajvJ46`dp{j~X5@d+qp7Z-)WY@%G1L4msq5CJjF@Ke#Vh`vdhNN$H;Ip{(*1<oYj)QE#xL8$_ez*Av4s$sjZ}J(VRy+kE3!k4_AFW-=`UbPWegtm>59WrAaqbzA?({<iqzkhI9arcNoq_v~Z0DKy5CCm~?m#d)A4$X`Q)Gj%(~!St3b_Jbg`Cod6P)kv<9s)t@9+t$w&&i1+WTbPKWgGW*+4A2JM$!z2bi^KJ?RbTgGZulnsEe2N09*6%MCDB%FXsjqnk-WwALzfYL!WRA|_q1H5^}e&+(Zj+O7;I-xlkOV$4speLSa(^9)5--4IKdiG3iOusTs#8KkWXBXdlL!=raBL0QJm+D>E3EiJh(jBjDfWrm#(t%FFo)|XysAziOESAAO(Fq-sr!#iQzv~vgE>dl$m*SyKNL7ZBMe!b?Qei8fORh4NvoRzg->~5A}S>M_NV)qb>1`}`bY#bX*<S@|Lou|fhU8ldE={61^u}2sV`Vez(%E__K*z{YCd1o`Q7fIMhw{`3Av?f)sQQE~4ziDYW+e@wC=)>>-3R+>?UJdmqy!m;SMRbPYv&b=4OpdlUIq39|y|fcOckeX=|5@ybdU~|8O`A{!XbYoj54wGC1A>;E<HPL8<O)6DhV|}A`IY0(&J8PcKaCG=<Zsv(Wxr~b(rg(U%cI;3LjaF{+ZQcrDP94Wq)ruCsKpAiM|NXT>-lqDCrO@lG%kr&J0oMGPWxFT%SXQvjU;+TT?Bpp_6g<e#4Kp<X<D<4rY+0h*CL-6S$0SFO9DhMde48|MaRd#RYQ3%e%+BJF_)g-%U<0Xndq7!vaD<$K_MhIZ`lq*<(iwY9W+aHN!!c{jc%K9Hr)dlinD=_L_xRqnhb%=XTrkU61EgxY7*B^yO#k!P_Mct2hQhXR^%X^Np}aHJ`Qu+GJk@DGZAvY9y<v%{)In#B*hcE^7XR9t>)d5cjzHeie?R0E!uk5)7dOyz891p_k6mAV(+mhcdS28EIs5p$Mnw7<~Jf6Z!!LAy<V;z+gC=vjI3^<m_i?Qw#64eO}51yk+5tQ&d$H6f5z{AbExe>2@x%=0Bt9U_$iBw1t<<vwl2k<GdiR{c=vs?^xn%=vB;-dl5a{2oexJMt9Cww1%Orn4JXU7KdZNquv^dVjqj?R`My2#gC)MMN&an*qYbotCc4bS?TjYH=#rAibJQWl#bV5#7~$d%B0r`2jJcFK1IFJh_{~(V39fFqFZhr&&sqM%3ZUh)H*iLjfS+?_)BASuG#YQ_^BYV-Kg&l2%UOW^2l^q{!fKhFcRo|;m{&kvM0x{H(bKM5R7+LF`t>K4qtiJe^lVCBjPn&xIF>;C%U9q0sTWPEYE|~0KfeINycq(unRCQ?j^Cfdxi@@1&!>y$quF#NQ_7p5J!g@5CTIkpT;(H}Hq3zCi_`+s9>j(4eZT{lk~gWmSXK$(HderJh`QYNmKWITd|NHcDMMt17)+V^r+f`(5bXW9nzUin$CUP)^*GhXg@y-Rx;2ZQt3lyg%pGE<Yr4D=nIk!pgM=GtfKOR2a`EU~Q;h1XtC@*_tEmGdXIbQR7dhSC>37xqd~fulzyISuNR*6CxJ`K{$~_VtWhL8TkqlvIGEiGBxd>J`#vCY5qt{=?fB1{7+~u*lw(WV5FDNtEaR-yx7AyQsSK`nK9=}Tm^sHln+KOazK12}-<1uGECZXT9DQkTVI0Fa#JffUk|L2igRh%T4&K~~%WS|p_H=OPT@|zjtVsR5OG8A1;3Sg?lq9sw4UjirQ+Q64c{&SIUCSPA{kg00>=*7RZ&!^yNZ)G0TyGY$!t!6jG@bS)8%jmD$zmmh~BKj-T{;RL2Z5N_DOJ8B}ABuMyT!NgkqNu0ToJ{W&!c>UjvU`)fB03(JLFhgXW;6l$bPXfG@(dbKFFJdGWyEP-M8y_n5GWpO@pgzN`=&D-Llk8nY0QUJJhpq~lrm3`ikM0UrXgt0=UU@a+8Y<Fd&KU9Edq)jU@q81m$GZWXyzMy7D<HwE45s4_6W%|7hw64^;USYMCxo)BF2I+f!4Pq5r>iU(yS@cLH1zh4S@}SWk($|oNy?0oWonKQIuP9*fG;$Y(fFj2W^Y01;q%#)=S`Lq2Exg^D>RP@IOG2)0ge&n{)V+Vwr`9!PjPDIU0e0qecUq@o{y%#1@MMvVvuMCqu!oS7{H#n2BVg_Q01UTuBrAumGBK`u>TzadLJf>E;=5@TaY2fuX=x{Nn@$@CePid?-$TtecwSNZn2YTcKu$|806aTS109%Av{~w-Ec;!1~dtN5=Zvkf=PnG>x>Y)>MyY41AV)enHQPRB)h-08~joDfq?n4x#UB=i*nF-nwoBfw0A%6*F#PgH(qfK}hz1%P7WS>eG6~bs^oYzTuGQpb-FH*;a%7<k>x!W&hKkYvTkQpi%iD=cPN2O+0teT-RJ&LIJLj2&4Ko6j^k$;x^?nOp1ytQ-3I!daA$iQ^A;vpX*%sD2@e&%ka8O!Mw1@CqQ6XG%pkaVd&kY@iDSoX@`g>l3(*<g>N$U$2n#8A;$cqv1Z?sCOb49O*h03Mx7vt9vU6~5T~F{N0v4n`MxNbDAquhFwHiZATiR&2#7)0`UO*egWx>`TC+Ttm=_`~W@#cUfjYWrn_bV9WdcC>4K=HeX3JQczcfo4xzOowT@|D0IZ&z8YJ3^&vZ8}fA4PpZG7VL}1{f-!2+M>I&C^7umLRDVIVDtul5?1O=tXDSF$#ekr&dFQW6^O;<%0DV#mFW9`;uRG<WiL?$InK1vChX+j7OkK5KYET0azxhTJTmD3@XWR5#>h3fmnk8qF-NAKNi}DD;GU?8#=;juU#Vv_n1)M1PVE!!0-d(4$H~(yvk6~>=fb$kp)i9x(jsWf`IukUOw>2rOy&m|AlmYeB$spV!wW(GtitfQBJ<_l(yG}*Rfx}^@JxQCXIV`Y04k-m@zIDVtI3JU3MXa>8n4-_Ip!aYbvT44zr1eP3B>MfnVmk>aCgLWwY`f{*w%9>Tyz3<NN{(yXniKy4|-Xsd|VY)bvB+t@rN99(6q|htUlKuAXk$oq60b&9Ekb5A06M!+8Y=nD`I<z1@w+nLTF<5Oe6Fn8lQtIOc5c19pegWYcW|?_V-H6R)Iqo>fM>@AJ%`M=@e<uU$K?c5+<F-z0#F;`$Jcvbd?E-^DUx*VM7^E~H(&eB%k&hhyK-mj*5GxcF~;6~H;<CGcl&K$t{pK)jNmE1P1BY)<ZBShYI=&bw@TD5syjj<(U4@V|DuJ>US5`8zLiVC{?2tp^}gO??mR)$4C1dQ5Jp?SLuy`=*2oAL^)#JxiAHiAGf$n`FK?D{<*r#^P1z;N?<{<kohrom7>rJjcP5481C5&cql4^Kd@vQB-5=rD1I=ocTB`L)~UL`VqZ%V)kr}6Tn-Wch0El*%cF~_Xae?4#rYhM+x39CvChzj3@jiQPsMhOlJkT*z@{K5>2B2hF(tiqtzAF7XNhy2*n&m27WE3eTG;7arfeDrf0!<j{5ddadt9Dop(mmEkZ<W2+Cng{LXf=SzL<YkwU}H5+`O?^a54_&cnEam$V2j!5;|FUxm=3!QxE-4DUu@W!1oZ-l9IT2q*whCth2QMt8H3x^~yp0U(MluS+4lM&g#5ZWXBlg`_L;(Ii>|om_Lu@!fiQF=ZMAykQhJB6#bYMngcDoSTmD%gNWr<7kwZLO}FbkxZ69E*ClUX)#$U<0WQL(kV{HqhfeEJbm$ZpALsb=kGe5&+<=;&hYfpzx!-7KKW-SpN&3!ajNQN)j2ahDW7i^Mb%#2Jbm@qXAdhXN<#(1g-yIw!r^ge4_OXAFN*6-u2n|i|2~i`2m99PF7hBw)O5$-L;h}}kn?d_qP5@ndvIcNQPKI^C-2hB^h#BIFMiz#+sg6uYFbW*K<9=x9=PwFS9kVpk26qNSu<?O>ZH=fGAEZY17e|T_G6mt=(hE|heybUbzBv$2cMkGnB{6Vts-n?Se&ChEPG)s@c|Y*HQ4b~Z^UdDI<iNruZoL&bQ7{3Ql?t#<&XR@(?x*VNyQ0ms^WDyCzn_9^GRe-L^JZ{yO-bf7zhmzEMtCw9u_52NQ!dj(=Acg_9UO31AU#%KKbMm&8eUFXjNt5PHoBBXmvxTnRN`A$K+~%iz`)tVYEQ*SZ)b8=X}8xlvNXy_7xDT>pM?lw8EMsR(C}AIU~x9DL0EMJD)9AB@16mXGj`lz>ePdb?j87Sd&^3*W$0$WV!k)h(n{x{DN$Ys04jmt}yBb9j_Eeo?E0!w3grbLG^8sW6dRO4&2FOkoqt6T{O=}B}!g^OC_;D%N5SU`P7kY**I*T8ASo~P{zM0moD=8%8|B5o3dKYGnkeBdwt{ejBVJ~X7A3`3Fy=2;Li>4d^!r@HdbG`a}Nv+u#qnshx#=7OA*1ep}kLpmx1%Vns0y_(v>NxzG0*3ivN|6eqa}o(f_)vM8N-&YOrLG2x&dtvWo+5peBk50AzQyoDYE&VhhN$?N|8>cIj1t1^}Wp)-Rkxx`R84hRAv9<3ux1WJgE50@==1%PMEJo)M4xl6-BMKo_4WpabKQ$U_}>Tslj;y(w^dd_D<|LeF;eZC(K(L5~UqX|w_xjPU6Ba)Y-)jHcs)$~(NBm$C~^xc6ai7K$Sg{78saBi-1k3NQ!kNa+AWiP!NT@W_S#eVM@1n?Y1bG5hafJN*%TQvqlGp;ksGAQpi990`iA8+NCM6x48BP0(v+aZw>Pgg)BDnDb#&5;(5-R44((lv^wp#cN}wf$edNYg9bN&_Y+dSxH|@t47K^+)qU}lQy06F?lXqG8zeD=$aQjC*Om1`EFgs>epzZ9}s-6(%vYhi*hr!DBs&B!%>wEN9FNvhgFIBR^DOfnDBThernVT_w~2Lic~qWl0-<(JD;Jvv>+Cft}1Df5WPx&XUgSyHP5#(d)Ef4m{)NkErywx&Oh3+dTx&A33Qx~U{v7<ST&e;_o&OYMZgnR&bTzAg<)0Gk!?h)(Qfadr!zRUhh&HhJpcis?L>&f&d&(NJ&-s>ko5M)N@#Ec{^60(C_=+DJ*PiC@i0Hfl9VLvt7&oVCwy`X+O;nat^KQ_QT3;S9NsUSl>Pa2QGJUa>**MAl$+U%$}<*@8}AuJEO?W^x4-6h88GDII(@SmwWrk_=4BzOb1_7SDU7?UF`ltoWj6=jW=+rRp2lpO21q5+b!EF@S&JwYg`R7K%noNWel@f_*Q%2#5M|(^LqkJNIoIUji@qljIoBZvdxOFhdYk}m&Zc-}fHSgD-NFC9M&G9`wUROu8W6qFS0D#;k6tiyMQr)S*04oMRomK*!%lDsuEwxQSW1ADGtAN3y1Mb&Uy;wbT%QP^rf7)CL{QSyvPg+s7(q<50HMZ%1@D*@mZ6PsmevfDO3PRWG1xRU37GNR#})~3igsRM8FEE2b7cXUFyZSU43KEA(OhdM48k~;S9v+#j(^qQ|25)2>aC$_jG#b5Ul-%e2)0!*U#@SMzQ!F8Q9auXPGKwKFfcwRE~0u*7X<-mL320m*RFJ#NOy!C8jNLk^aSn?FA$9#(<_}}pdN=hCL(V3qU+`Qk_+UjJ1}Rk2Ms2U&B^Y7P3L^3EE}@uWwBZpBOnT6cO{)`DQ>p+_4L)Z$Ea3O*S)x+^8$xGE$1dA*fcEgEWd;#AU2U>-FhD07$K!B?~<AjHPI{l2?rL!UM(wTLNU8(<|sd1jMhbt`lI6fd^!RI9^KsIDr7nb<YvJ-tb~Wo6^VHu1+3?+%M)*SAn)Pe+UFV36Loso=N8UDx{Xj!1}U_|!a0U`8qx5Y{MFy<oUb~cIUm#|((xX*NbW9kZ<%)58CuAE&|Q>ljOnnlO8MT2SO%mhV;aOr`z)LjwYXU7qX?;$nuy#C4)^IpcVzCFV*l3+&Vo7W24EO)grg;GPdEyuid~tOu=y~RD3Nwj2ak*4a(PMiAf$W*Huj>3uHLcjqB_9TY&ur%nH<)$D*hWfZOZgGC27?9dbyl^@=5fsmKz-0#gf#~Ku~6zF|!(%&KEeIo8@L5jn+#zauw~;32VFEZdq1Pfwuh}4U8((QD!<^$m?x=y>YD?_SXo5jrZKkIWEm|ao$E25`RoDM4$(ZK>H=VaXev3p3=rk;;<^;^ID_H$>|fnYSg!;(j^Pt+gx8U{YKXh(+i3(i+k7lNuMOP)>l)s8qlCD*PhaoyyVv$z!S6rJ*dB;wtDh#Jz$nN^322Q2>EberegD0<iU?-+5T=q|37<g+TB)iB?$f%ROZYOR6>+Q<z|)XZkIE&+NbO)kL~K2X{8PglAr{OBAHy2l5?^T7J>u_68l0BAVGp8xC0=#0R%vt)APMg{{|&u`=q}x5%Cth4O}vvu5wvfWfAYaSa008d))Xzw=Wy=RN?+a{rg*nbt`#q+l$U0jWQ5R99Z&kOV0-L6Zx+=Iq~TSIpT-J8?&dL2^NqCA4~L;acm(5R{Hj^_lhhS20;<3GRjrFE|%cxJu1J{xwtv+>6ds+zg;w0a!#gBL8V~4l!W)UQ+R06(R9c3q4ri6xyChNy<(WMMh?{Uc+=osu`zY!s~R&5xK$!iXMQ8z>o{QJ={kg2el8hn%!)R2{Tmz*U>Q|9-2S5O^|77<bjmDYEc&Y>XVv-tG;450(6T0Qm;LvobBuSt7}Y9~$4<md0SEq-0pqc{@Dq(`xKmRcs~>2xP%xv1h1)~acU&lmkI|(Tn$&%5dTEqBLJnqsY`g&S@x0Ee*_Bt#Xpp;e!RynusuR^Su-xgakQoJi_i-%hTt=hws2{JXr_)Z1UUq2109&6Jg_tn}GMcvG)t*DrUB?DSNaJDLSTxbVmDdoGek-#j8Q1BMibN;M>Ds$uwgzLlVJ&_IDpN|X7}ER7F+KP`khns95xDUUM=R`A>29OWBhkdsUTsvl_M)ioyDQYz0QuJqYqWQ|1lJo3(@4AT@cxR{y7^8Ro4R_<^vQXy2Xq@A78zZn%*i@yfkmbGD}G-w@*C3^9V3^psuO#@BOqUQr?%8g^8>aA49Jp}V@kaa8eAAGUymeQqo?3`9qvXF_=Cx~>y8sP2HEYTlp8Y28*R56Gt8SYO&#OB(e>sx;g&y?HvCYosX80n=>m+aH-?C58pC1dvV7w!4p?5l;Z@q7MFCf%Mfr1V?CuVNL#G~Yz9mID^-iCu_ck=(dze0MB2yU15x15dZYD2uk`b<w3w|6~K)R}aNV@ioHh(ihbp>%Uqy^U!q0WToD$;RVa`AmhgjUhMRv<D2!fi);Ghyde0?rQ;YW|7^8OuBTy^(c7M`R@n>lG%N$4|Piv@~$5d<Szw_v1(G;HlA7nw_i;o$k-H4+=*}wftv%Y~0-aBN-}KRQJD)xk77!yylv!|0w1PqiMf0SG+S<{MDE%WY5GqyT!i;yM=5Q;L%@fyK90qja$0XG;kYSeW&IB8)^B|<9|sVUy;t~mOA{+QCZZd@7?RHP^;v)AH_na7aVVy_3Ng-PNNgb0uZ%+yem=>b&!iCPpFZ%*vJiPAL<v2@@u|-$aJrsis+|ihpsJJ<H2apug0GKPCT$69Q)t_SYpP+F(HhnpN@*|{@0w|vGbxVD0g&h^sY28c8GTDseNkFz)P&WPKDj0GXCIWex&EB(>%Wr%JV2AcG`1H3cD8Kk?FX^n1U+~*BIJbM_!u^Z-ME)!oG2vdb+)GMa2%_dD#2s0?)(VKM!~w_I^L`=<D4OJP&)X&Oap`^pNkGQ(U89F})W4-nsLf)_OX5<4Y?9$nhX}b|zzh9m~CKDk|aBx4xD<Vs`Z6!QIb<)|J_(3@VA!dZZ4b9v_*0`jmy8jGA8Bt-=Onr)+f9UyCDEj*rJ)O!Ctgx)}aHd(tZ@QORp%L<o=0Gk3@jVh};1pePpF1HZ3L2*`hJdd)Lvvplk+Bd?`{5W-5blO@^-D#!MWEAa}AaY~Fx(cegAmY2w46llgSTpXuNt)C4-2+vigm6CEPw52UG@9YP}(TI?JEu0kO5S4Nu-+yB1)@7R#GEGUB=Ewm!=GUq*Jmc9VcsOM{giB+k5f79~H=!#Dh(XFGk<~s^Uw5KS9VH7D&x}$~C~3_VEg1|>MQsd5CtXNH8+|5Pe4cd9ZguTkc#bLoe6Mt{yM=V1Ojo5860XaUrAlxurtD}RwG;0+zr~3=p8n)}p15O2y~Ok?lQz|9sdr~1ZgV!`haZ6WYd!zKGRF6P`5w=BW`Ai@of5vo_`330Wc4=-=l^koyWfQ@{{=#p|CU0L|JEXk-$nPni|&0F-TNOjy7zB0cvSc1FXxSU7mN8W7V}*!=DS$Tf0wbC??MUxLxvLGgx&qXn8J6FdAEwpQ<Y$r6}r*bAC)OYc;9DI8eI%8DZJJ_dlR4KRSKl|?dRe!e*W_5Z|K;1hs?~Trgp$y@7ZNUS9`ljhYr6O;b^BDkM<521*GJ}RPWvD4FA6wSY^r~Le!Xzm=!1U@7<$BzT2XT!0F&9H()+DhNN`U-w#2#(VkxkrU(5_u$X%9fB0z!X#LjK841M}S#5)>a&A4p{+GS~22)fa%@u&SYB&A>K<Hf+Ry&oO{&{TbmrM$JL8ogQ`g~$yY{ESJ<$M1+YI`+q8)Wxfe8b@PEr%k@c>LbqOi;n!Mr^=MT>O8H?t88G{GV>;H}BvxzO7@rTV+<IDhJh+wO;QZ+2yO(;bjS=XL8CqF*V7Yie?4%H?kj!Lzlh8Y9DLQ3!8P6$1a(BEvARv-8Xb18#!lhKCbUSCXM%5Y{+7=zOplaT=(@(-75&|l|JaJoWx&_zAxnG<ZfaiH|gMnbhVm@w309R_qJD4Ux=LNonQH#U-_@eul&w;{LXj$&UgIIcl<Z%JAUUae&;QI=PiEcE&i+X7QgfVzVrXS^Z&l{|GxA8zVrXS^Zy#3K)un<y`h)bNPEz|(`{zjEI;gTobE-fPAGD<Fov#cyt~`TPbT%ynOk4OcXj<dwm<2>C~m}gujaT0MEy#k?m9mEefaAS;j7&`x86Gk+&c%{I|tnNb-?wryjB`ROxrd0&9^_6Yi`6kIn~d-L$`L>J?mji!hH8ZH`l#*L-MPR?m9kEiAW2TZts{gXzG~?sA7~2;~$E${ERVsr{WmbF(73F9o&77C#NQq<tXcrA<_Lrnv>Ka{i-P}%};k_qS*eIqwiVX<4APxm!JLmp0F^J_t@Q4)9f}F>BaPveWYjn^?Q%rd*5Uo8XIS^66^{U=EZw9HAT0{y?amXQ_rRcdIrZ{+#92iv)b?3UyM(Ty|fM8mD{^V;XZKhh1EVPf9!96d}^JZoU~g<+plP!tP6fsoncQS(}VV*33i{~y&>D^n0=D&l6H4@$6r&5IQUHH_LEQUe>^ZWJoxeON4=j6fAs0l!-vD4emL;Sz(>8kwvQj!KDIr4@bQNO58ExZ2huOKbj7!82P=Z*>9l=F50v`@)=xkA)b`PXPX>BF`FQZdkA@yR_{m4Mf!?1C(*X~E();m8Ke68bXh5v40NbbtCe|yvIrRDD-w7U8@hY3KN%<$cLREg?gH}KFjE+(ir=pbnn#$HJ4+niW{^cmYHOFQjWfz%?--Wa2tT+BqxR8?KQV&C!N&c@?lwDhAWy;l*t)CU{9~AG`VgJW?!VBwjx-)H4Rw^hwvvC?B{a{%|B;scgjrRTAbKdvBcK1X172UCYy|%l(Oz)+uM1_sMPi=P}O52M$-o+M$C@G<uDng14fYC++aZZ2{tur}p9i*FBF${L^`~9Aelmn4pDTmIfYM;;$=gS{|*v;>ZH}Ej<h%WVbg;}Y_EAxb_#%vX>;qpjWe!>o;xo1)plK^R`0$4}JpK)llM{&<&exRsEXGd|TXWULd^cq#<ozv20;Z=9f_}JIxUZXz5ANRx`kMBRxui69CUFq|KM^D;iS^h?SX*#gM5&O8AKU+-J{ulPqzWe-#^;O^fo}c{WCs(ZoyNtO13KY5mUzCV)J%(I4f*-eQ5sRe}s3Jgwu)6<h9htVt(RA|oy1tQqe%oc59(_JK_Vs8F52;h|AwJw5PGrp88cHz4lCH^kW#*6kBDQ|vu3#ZNR9#AXH?Dd|Wzm7LNg+x>K?-y&GMIKvkJ?#cw=Tsz2Ss<Mx?L%Kpn3!fmzL_ifNj=E>%@fhwV7ud753zP<1r7)K|FlfjmP@cv~@^1P+ga1GwoEy0@l^IaD}$PbncjXKJ7LFYPUe4y6Nb-c*rhoy#gN$!z{ag|I9XF6H<)uVM=%EiBX~^aaL4A9HGL+m+yU}oAI|_zSkx1#`h&rOg&UNDT$OM&uDR&48wA`@Pt!OZG!4FY-iWXa%y3E$uN4-G{NuKHuKmdcJnRgaN=cu{DyK`eaJdxH4RNquuM79J)t6Sc#r*Jz=3*UK9F3UbY(^4RDf}uwl+mU_B-2%5SH4m$izqvJ;9zk@yPUz+|9RL>b)g@FxpqrSo^uY!*2H#3Gm=Fvkhx^72T%B+vt5Jg3?h+#F-6{$~sgp;8*HW(7m)yP?>JfCQ~k<qME9+ULUJFeWEPKvXZ=1Z=uKSiua#9p&;$l5U2f5dZxzsrt4Os(BR<M#E>*ljs<k*Pmfqx(y^CJ%wiI|B*|GPUfEyvj7>bfXC0Wl*L(lL{hs^1AKe$3+NAyI3qBtTDuLzkeP(VmBtD03fj>&`7<W`W%KS<w+kG1(Q7nKOL#T2<(M0(C?cdW!^AuC0q*uHv8F!RTe<9v7KKP7wJu2)9rqFU^;J0<N{rnAm{MKUX(uy9#UHnF8N!b)HARitZ85#SU?-mBI{;4)Y(~g{?HixMWulZYxIAMztVr@+4jHRD8+0Q@s4_GGznkP%`3hb7PSm+LU#YIq&`5S~3{$hd(yzGWwR;)dGDtS+}X&R*#Y_?DBFWC*_y!k2CfoXVpl;0{Un!W@m(gE|UwK=e-83M9nC}QTjDF$-$gS)J|ZWC3RCOAD1s1aA~?^_-`(cj-}FjMWN)q=1<shF5!Pji*K1Qlij8L_??7_thpWLL5C+q<n3Pp27`r^Dc*xzE~qpuSmL{h!#;jix$1&YELAc5ICa-lDZjo-S*P_Mo{3Tv}HF1oX-NV)})!gu)G@f6uQO6)8VOQ3htJOws8~jdkdsVH}%1QwQlpChSvo`TB7s#z-=mWP>qC9B!~!N|V6U17;03-RrclijuvI>)Uzy@tc$v1TfLoL8ad|G2&Zhs<&IBNy77k?)Dz(R_{wjn>l6XZ7QTW+pDPw>)=#ByEwOhirdA^l!8z}^i@CS*V0r6ET%jCOn4R8xbRY7pw^zJR(1oZncW6DrS6!P=%HkWRWY+B*s(26SecbY>VmoKySo*mYMZgmY*Si9TOf;?q6kC(4v$!$8WDic8IlMjK-$ebrkD+M&^q{x5jWbZZQ_M($j-~rGB2${;4OuvNdG7nNV8yDVo17m-YUo=>`cn&q(E;O{6wy5lL+a6b&v&;4$<Liah;7yz;k(g6vu7iu+?RSNeiXr?L5idpo3h%Va>C@e_(C>L0wsZ6eACE4LaPmL?KpUIg=@RO;Rb!G(dqgG-l(O<_xzLo%e`+&_2cNyp+x`(}OfF?KJ87JYvjIyrP;CU+Vy+#7`%zLv}We_6oNHx|It_tMWD!s^-K~t~^aM-?{*rZn~9(XJNM8I$PQ=<ZiwY!7W<B1+<cVP3%w`j|T1_JPWO{O6EIa1!Y1teK%%Nx1XLGV{`72-b+aPpQ*YxWgirzXT9ltaO#!CG%-EeFY_ce)~76&^&M>@OIo%^S(E}v`KT@^n1zLnJ}F&z%vyZ>Na#R$6+^p%0AE#kq4*=UFOSFXi{`4ael~Nn)_f{v;%g5<JAU&W-9YsVt8PUaLI<cy`%wVv9kDJOoeq#%_&k2qGek+|;4^bKrDI=B+gZROtJ!bnqBdt3uh_=d)qY|K7TYM>2*#j!-}Ht36=UCYD?X+22h+*~{fFr_W8<7JDOdW%7-b-Kfn{r(5}h2Av|D!hIjuTALKRJgbjT7)a+1d^#!SEyC)zb++?LW!!o})mymtTK*tp#$0_&xnIqWCRuWksL8)AgIE1jR4FQsf*kvAO&@>P9lb0_N%-~-(ruo$+-2cY>RjCgvK$}U6N5G2J6n6FINai`D17}Q_2TS2o^%t9tyA>Zl;N`STb?bYrq9(BzGi*eh8eQc=j!F~FlQL59<5;+yarYFbjL+pll-FY4j@6h@8+U~l4ro$hy53)OfnTddoM|A$#1d~QjCu#Ke&uo)Sw0zC(A0K8Ck21}z*fE{wBT-(Zjd6KZ6kr7q>!nr!AUrAW)Wq0`AUab?jSH~D69VIXEw!Sbu+}$8J63up$HykcbhY}r8fMA%vSL9LgoPim>SIkaqnN_X%f{E<G?ce%w#r@Rk?0v2dunb^m5FXDtF>U~r`W)(`>ZPiOXWi@sIrfW4M25%^N{T&UCh3=g-CeOt!Mgedx3z-PlT4uY&?7(kNI1b+g5;A+l<Z9yDBW&jEh>tZxlU+7h=v=)zLd<;iJI&-%Y)bbjfs$`jB5st+RXtll>$1=QgwajZNgygD1KUHG+q-3t@Sc8?$S`boXbb0n<<yZKY4R8e6gR<;v}iu1t;aqT|WD|3*u3*`8soLS&>UX_frU>}a0YkA2DjCmHqU3`;2Aqp)Qu+1_}Yz4Eb{)l(2Ex5qC;si-Dk5ndY2<9C83@Gm(Dsx|kUg#leT5q2oFpGvrNJ^6MwJRr$rY)(;VVZpjUc?wUqFHk+1P8ksev16h6_90Y-b^C!J0E;CARZOLZX%kF{p$rB?{WVtyOw5Tm#1tP@O(Aw)O+BT?9w|&7Lad1N#ehc2VPt$^RDE;?z&}omO^^RGE&H(7{E#^xrD;e<U4QfX_iUS*9`t--VpckK7Rkbup|5T1;2(F8whMjKBT4=I(30+mt3d=!w_cTaB@<F+GNYW4N=!<PLOE*Mh5UHCUZi7-Qpr~m+3ePy3l{Zm)e$D)c%(_2Jm1jZ_TV#%sh9qbuFJ#Ecst^G4r%KcaxK;m^9DT751me$MNd5#GHZvS^t_1mdpp+ey;1##9qT{rRNu(IHa#k$3l*r_d-IX$8`jUaV#*2u(&E{++kjZ8(ixuZf&a?5@pkXlmrwiI0rx-0T3abyVhV<snBxbmx0u`Khb(zJJ<#`Fb+L&bW8sCZ{a;=WOxTBhA>BnlH>iq+7#vy`HuxY5%AgX47@NUgNEbec@2)$ttUV|zwk8dPhxHl;f|xa(>)KKd<Yvujhu%2?QR-VYW0m4A7LbCj^fQyk->s@8H6~Xynuc<)dPBrFT~gCsk4%rH-&CyYQdAQ^NsmvyRq|MhPJ(dAIWW7$j0wS#ex+8eL=m4?V%8XL`F|vG63W$wMtNSWJA$kA!J{YLQhR-(Y~3S@k@5`@8k?-fA}gc}>fD=Z%zY~PT@Gh!582!;oTy=M#GCts*BwILCZ{Is1H!SudlDWCv2Hi0pn4K&8AeyB!l>HY^ydSU+%*1~q8SDA6%YAnBY-=**;NS6#(vwn(!f5m(_tw`Wi}eBQc%zk67EP*kqU<Uf(x-v_^KRhr<p3Uhv@1U&(Ekvdsfz`Ow;{GG0;#o!Lflq+6Jetm{MCXXSm4h`}FhVlzmXWecE~N+WJ7BqOvx6nhMQgG%r<yUs@;Hir%y7(dTAmuf6))LJTX?O*C#+e6g7BKD^&^pC#<-xv#CPnhI0mmCu^8lT$+~W&VE8M|4jsdg|6K_=Kv0aZxEBj;d>;!x%zyx}hkPUm2!K<zKTLZ4<z?2>42l8)oX!2kox!s40|GMAUF@(zjMvSB~cSh=_Az%;&}Lmr{=ECh(F}Gd@be++%)xzvs!-`Fv>0yh~=fHMj>z(Xg=eqIxm?Li9zj*gXYW_jX@>KvD7E2kjN6e4*z){a1;>H*SY*kjFyGnukxcHxn7(YujN)9nR?J=ShdkfhJGdbWeOiM{eATVz!F1Ml+{Hh3{7aLObg=DvgdvxTHD)N(=W|Xwg4ER?%@zeLXgA7ov5TX|I>gNelwj<Nei8mmB*)0~5tE2nk8<npAj4!74#m12P6G27>Ne@e<aq)@adBw<&wiY#e-C^ap$Hm7aTzRGi7N>4`zxqgw}v_kJ?id*Ax$uyx=kg99Ia(mOQ#(a^^OLqi`AemeBv!QiLEA3nT4@RQyT^#S4|W9)h_DxJLls|TWO_7|cvnOT1R=~omcRKa@69k>IwsjqD|y2M|PnWTVx$xiY+M%<V`mTG9zQ)4ffAIv&3^4c^oF=ibav`$X3Yab`a1cNy(IcQ}ep4@d!D-Fq^6wG*KWgN}=VvMSD0?A?V7qkt-5DLM5L{X}JMH@5EP@c^!XE$605y@HjKGPTmZH(tzRLHFLmdX9nlIkx5R8xH}hNuaVh+<jAEov5?vDGwUvohD(sKUd}uaz5MimuRaJChhsi#FH}o8!)RT2gY4nf^h7HB*G2*+#}Er6Xe!SY<HFPBp>QWN*9PO<#{qJm(;6^X6_5?lKfb@TpUyywZU1l?H%fiX#3>jP-cM`JrfoR3C}qzB{wFN3qZo2_`C1p`Z|VDYa}reIQj=6p$vPEY+T|_)#vVWLF=U%mP}`uN<AkAXu4Lu?Fs{>veZHPh)#wHPQ`)2rh2&lhlM!rMUVl`_$(ztj5+qjcm&MWMb){^|cVX_7jV6=6U)>|MWOjjVD!Mo7XAt%!0w%GX0c7{l1AiZrS6{7=toQuBtFrbbZFlN;oc2l>6Ki{-8^rLFq;RC>4)-C;1=Eu{2&_!uFJMBils(!1VAi9ds8TO_xQDXZ)qHZ>ama1_d|`F+gdXy>C&r$bz4xlw57#UL&69_r%=?x>j}P#TYa!vTwjP(!*X1^>=U(NLjVjAp{2E&ifO#7h^AZWKCC3*WJNqR;Jh4CbaH)JB!v~&@q^uyj1KDQ}>zM-%Ojcbaiz~xuRO-9s$_qZ`i{l;oY~LYHC+YC~jegHz-x5C@i~1SU9#GzXr!fgdo{=jLG9C9c@7z)=J0<sNW|^NA#i7O`0MGnSR!1y8o!doug8j4^;WCl>M+Uymns}ew;>IU5F#TZ}deqe+nOVsAYPxUus>DFm=<oK~&8P0esxiJ2++Tgp=L62#%q#uiKF4b%jCmsv?e=)#H}-z;oMcj{idaM}~=>#E*|@*(cXxKOd5ayn3|cZFH!loX2C{1V4dk@R~b5ScHmlWZgG!mI}fH{budbQPyWThPx^N7-c-_p!X<b7q;(3Q0sxBKAk@Px}S1$1=~@g$?Z|(Ht2C<#JDX|s0uS0(!=Meg|{)fJdvo*4M2WdaI?rNo6i1p)e-sMSzp@3=<10HCTc3oPz-Jq3baioJC0ClF`<ZN`T4|C^(pRl%}1tR(my=9a%|Z22kES6&u^p>fB1W?3!F=>T49<=IaIEt8Uvr$&~+&Zj0iXXB~gCCdoD(@D!NxucVWu61LJRnI@(zY!xo7p{qzU^;t%5KCB=09b<#FG%@Y*7w2s&rVOFFq;MG6>_0N?pOA*9YitdBb;TZ)TpRhmWCzQs<r|d5zr?(X9Zu*_kU<Yi&;@fsRe-JmYJY>_)hZ*k|N68vB3kUt_I47+y*>NT=Nt*a+4rMI-gaSnD+>C8P(fMBzuLQ=YSQT-V3`e6~XntTxKrc%hlm28EZIxOK?1Vq^yp-faY*(TW#=3!!&-zBb2f3{H;tGd@_Bgk2c8$e4j<Faru@pN%!P1TEk8evoyyBS!CdKs?tBE<dEvDbMzx`hm>=3~OS33XYvwwNSb^otfoZAG2AKmd2Hu?_>)R2$tn|SJPrW8!lhUr&!ERUlZ3;L!N7|mwt@_#bjz56ptJ|?G(oG0RFZlEXKEZZ_avcTIEDMkms^_A|FFIAe{FWA34!`RIRYB4Et(T++NwDX`y3u7F8UHnzot$eYJ;q<ZEn)k~1FFLf(<?u|oW5sZ;aG`Rek1#v~gEeP&hDK#!T+$HR-`*E+!7lJEMuJ$JujJoUpU&ldb8ak0@2Lx@XKEhQ8zKjl3#LG9dD%4hrnc?<Zj4Hl%B>>qRWYZilIj?&_NWjP%YN|k7SqeILF<5c%?b;RoV0z_KVlp0`{WbZ5&GqOgVRG+^;H4aB7snsyLs3TZ4PF|1o{kiH$SU1HTIn9uzj6IsQ7(zt9?6Jy$FHHK51Jsd#wzgA=~)Wvp%s*_Z7$}3{rK5iofALBle?56k;WZ48yk<Hbu6YKglV1+F&pXbuA*!v_qLi>xvGk{?w7Eyy{?S*2HN2!{@*K{qI?)6wcS6Ad@*5-5jT_G5Sh5Age9RmsK|nyf!_XeoEztr^8fT?=k&XwuC0OgpW)Q?|;<WYcl_*?rTt0gM1WIzv}0womi7X2s%dx;phg%*PEiJU=3@-T4qxJKqY1w1Ya;ys?1|J!x&?n;wy<)X#K10+4t#N>f!MsZGf4-d~bl3xjSH;wB4o5lKvTzQx@EC4=LfnT-}OCt=UTk6I1%-hbxfU_DYrSnp3@gj#dKsR?pbX%F|jCY$i;!(=U%^95P~ypRY-3YV1hsXHlfX!YMb2Scpfoq=e*OzNeV(dvdSh@dEmJd`bm&dGD%c{P@fFgrf%bAxdGNXt%U(e+kXV2YEEsjt(1D3v_ube7Ewi>RTGdru2x=$&DtANAtRUn8IRutb7p4t8|+o+Z98V!*gFvX=%#MxEiM@Orlm25=lE?f*ad_lRp~k>*GX8#{dlwQcIPXbynU7BHGKux?jHYr04#vm?v-b`RJeeheySG9OORq_CQ5pBU50z9W{wpkyCNoC^oDD6#t=F`>orjt;p-DTj}nb)70yiPelNvbrcA}`<+RUfkCJ(k?;-r7ZN<Q6MkPjdBikuksE#TiLq!R61z4f`JHfG)o*S5G{TKE8-sNnWqFC7+SGUav~6?TPSMvjU>Aauy9^6Wk4)LeN9=U7vy@u~B@}mM^gT2xpvs)e>8(Mb0lRh5Al&OZ!XeF&X;|V<T~0xDLFPfw6%ljfP&Mdr7Xt-Vh`TIZ5?B-v>YAFcD_##y)Bn+5w1&0oiB<}cK_}j>AUx=jgHoC_J~m>V;_h(KaA1}#HqtRl>xiPLO|TI63FFrL;9gkA`$gnaUlF8qqU*P&vZB@mBV|Pq5Yr@KCM9Fz<NVY;lQO?iAj7vzfc!?;`M2`%4U?7I#weWae_}_-wXQAotwDnQqfc=vKzTolDi`kq3r8gOPiNAk(h!FEF<bGFt#~X>i7;AQlpMIGTsY>e7fv@(?n0mYGzcK>?OVWG?nVs-uEbmw;P!n~A@K&dUt7m-Zt)SbI!J~TPBw3~b2&G9z#^_+_G;Ev+pdLW)T94@-(q^oj?z#%(|c8O{R3j!>VIHDEjxHao0Z0RRlNz@$h1(CUnEOMG9u|-S1>x3C6um?VRWOid9zgTnsRjbs%17V#Kx*BrqEt2ra_B|>KtG5%(IG=A5wF|QE#(Y&~(T2b$1V4uNTu(JeOH7yIPqA;!oM1PLEAbnhm>Yqo4Yl4h;<ebPL`n)iyNTs;sDj)m9LTCz;LvE<Xl2c1?cMS9g`l@x7@Dn@z~1s%l&ox33R1v9MI0lhZGBHe&@bXO`G6uC*TX&n(FtO9Uom`<!eihv45uOFF-=O&SC9@bTmt9de7Zd@`d?<#P(cSvtDl{J;lS<UafSi=Y4NuYTYE*)M<H|6e};`DYYsKK_t>_@yEP%cK~nAoOMKQhn>PNa0Og<19h_*zgpSbNP`9i}Yb^a!UQ>%TXcB@>dMmzfWJ!>{R_4os_%a^08f2Sn5Mz@MZ_zY`>9@Y%6TFHz0-b8GA{AxD6o`a#XAA!%B6))IJ%kkQr!8YP-7dN^6U!t+Ee2VpyeF>agc)8%x1Qnby=0Q`W9bA+Bn$ou@(Il_Gh0{NRaf!FtuvOdxe{9_<#@Z+4LSiHaB5U)hF!z&hMG4r}yfKRbKQ<2Hnl5BeXAL^G@AcVArH-fsEO`aRyywSiAk4_y|yv#S%evlZv35G<4`4o}*s8m%&qbOOCt&tL)Ma0wA>#8I%wXWoHg3Sx=}3aRn-xdKkb=uAu&(;wMkb-U@0T|B*%x;<^a_4aXkv$9=&P`}%v>WiP|@9s7@xZgeouSb>ld3dHxNF5_2!lJOBG{3Ztw9i`c>hz6T<dtgB|5>Sc#;VZH6}e0GO{tqxH9sVriE5K%^N<*)IixtRIr6pjwW>v7!?M$9S9!pUyX5th0n=0cwy&nC;$+-`@$u!q>$|Sq+qc<wZJUg&joO~}^W#IBqw`{oyJB>qeRNm~Y?W_fw!V^9|GDk;*JBey?WkW3I*i)~pZ8lwhx*6I#?7?wq+*H1M08>@Bq~^##SDD3+#}Oak65Lf``acpnb>#?4Rc`&jS6ik58ZS5w4W90A9&qQOR*aph7{e4W(J0reWu5}J&yhCuOf?`7{fBe9Czwy#1}kyK%Z%V3Jkn)K3WxJ8{%g-r_2oNd2!vgV*Nj+1)p%w9<yvI;<KdGGk4l$Es80!lU<`v`v)k!8=dUu*K1SuK5S<cY)ZSH^o{Z!ZlRzKUIM84g|BjJ8kiZtR#KMrL$a8}u2?gg?gpfuIEv7!FZ5js+mScbFMSl=QDFUmfoR%hx~uSA4(w_p<f}&}^Q*`ApRi<I6679HD*uWyTbBHUi$VH<_SmOvFBnnP;I3h}(yXMNCHkdhGZfQlZ^U}}R-mH(?PMR#u|@U$Kup#c9=(!f8qrPmni+89OB_bcuXN|Ym3aP2I%y})f!9{ahrhM4gK#&3!z}?IPg3<(3>c7ub<B#>c;*M)-G4!tJT-y`M+>G`-&{JjQLX)f?e0gnf{u_3T1bCHC9}J7y7ad2%#(4e5^r7wL6&X$8UU)KT{=Zso?#CRYJ#M~*n@4H{T4FG-BFC#fH~gdZHWg+SuT8O^m5IoDuny>@4gp!KmB(AUQsyycMf3Nq<x4=I5)}wW8sEMR@_kV`9S`x$^rA+fBi@6)Q_OK2$Pu*4OO(8U)V-Q%p41@4kPL?kEbleuWc=snm215tD?z(tx(qhRRo8aMqsADx=lk}TDMZWxYVU#tq!+;OAKEzGAR$Oke*OtSHm{jkT^*N<Ujw%PN{wLP^T}1zs#9+n-v(J82h8J1Y*J#O=G4zrW<S36r^mf67*Tq>};2CPQACYa>S_g7d=mzC6LwE7<Fw!Ux;1an2d;H7rZpwXVLc|9oxvX<S=bJcyF#~QyoCOs-6j@oY?TDm1z`3nDxnenfOL){$QZ7@r=jU+-0v?7fX8-2G`gB<LB)9dIc&<!3K2Rf-)yyl%^rwVVF_d)HQ=@k6u5D&jiStCC93z(|qI+wDHzp8{II)lsIn1mXh!5lrv??ca&a(M|u~I{(!Ll4ZmQtPit+|loKv5{ZyOr(>gtA9T8DebE3vN`P|6YIAtAq&Lni9L8F?JRF>!Q2jhkfFz+>m231BMd}LUkdHY}3Ui3eI*>_(%TY2+n^N@~m+o#Z;c;fNq=LwZVGCV<Ol$7Ml_juZ4o+wbLtjMO#GyhNsG3&f&oK<}NJAL4mLXzO1?I2EYsI#l<Y&?Of+S|t$ljV=)KXegYlh1i9*W`1RBTYeE+I>sy>jKvLztU1{#fG*hebD-<&%b}wm*2Chh%Z8IEqzc<?|lic4I!(-c_wWspH-No`eGDYqqO6<8K71S6jFBx*9;3I>NfK4s%-zaZB03ODYLnWZCY^yx9OM`RonZU@!R&3rcJ&l6mNKC?n~f0o3!Dy8&F^vU-XFc(sbUguPCAHmW6$Z>FSvY=$>d|(sCM^#)$VP2@mRMxptYK8K0TtUvB{Oc1sp{lNl~3EA;XQG^hfEw0!JjL#MUm;1r1`3v>{E={KOfi4>f<6GP>4n}y2-Of2BSfYgPM^r}NeL)LpOPkg22X=9@-7|4kAbmD1ZmHtlOCu}4AbgvnGYG-z4-f^Q6>?#g{IT?+}_M)#HNumm%r4Y23`neZcMO7px1znFjq65RCU=9`ol0`+A9&O~^^OyZX!qK+GD1w0rI!aXfN-n6_jL+qlQbX-;FAct_c9!TNpp)0ehP0bixIkz*i}XcubZFD}yzE%<rM{xkVAKn#PfgFxLp~dum=b*179M&^w*n<IUyn_X44H(h7Ohh(%3XB6i9RCnjtRzG#zw_6W0S&t9P-r~8fiwT*C$O{bW`2!WhZLC(!NvIZLgN}>jMpvn#B*YlT7&2-DxGBcWOsk5*4zvL%sXIl6q1!{ao?=xuOMm9TneSnqM0AKykss%PB0qQ;!r1^OUG5o1dA6x`h<c&zR!eMK@F5Gl~diu?;oJ6vAdiIY}4ZWgUIZP0Iqh&^A~>i;8q~>+Z_^N@`BpVnQW#CP8lVVUZ-G`fP>VN71cY8nIQec~RLC5m_9O=tWbGuR|-+*n}4yNB6vqaW@>NJp1<2d?ofl*T*ib4D<idf4Vx&v0e9hyY3(5y7d45sI9A7NO2H%#1=cHN9f4t`3H+h;>=o{6LON%K|Pa9Gi_uW=o?_j))o~k3NEi(R2}JJ<)3$?clzQl8Af*~-llVH;22$4tpeQ_J>%9PMFnXtO6lW%x9GW0$9QorKE^NdmVDq0E_Tz9g#+l1ZCD8|ake8nnJ8J`{gR_Y?E+<ebRg(5-__1j92fudL7P1*ThT__h{ZJ>T_x-rPX3Q9RP4Dh@#o^iSwV}jZlDbyGn%C+nsk_>`1L9<b;bu)iK~saq5fg}$Vk8WzbkQ}y(-<>Tp=S0JTlsFf`Y0|jKD%91USo-=8=ZEO=u%H?8doQm6)TVA(U}<al#{z4uSHReyS<)$94%L`l8w%;>Ms=*bLg+L934%L-YoqGytL`vS_riZsEYPR!94(3Z@U9=tBL+shlZ6p4HGbY+yg-G)SOX<BxlKdMqZ|a?kYWB$bJ5!Y%MD4HfebTV?z+03t`!pVFp&1V|}U8;g3QS=es6>FA>9*u8EVqM&R}v>}+qq{ufW3t}7BlLc1L(m~7bOcw+-P1m}P;tt6@yzJK2Zf^x{J?;|unNV32s=73SlRCyw6<4S;;HoKWpZ0}Td(tjUpYF}oE)azVYvxvA^>9@|jFOVfBPS<+1Y<2bNAyDtv;3HOKUe673=6OeaD-&rhM$?Og)>;@?(0}$;>G0E-?>BI@UzHtcnY$C>A~l=DHd+tp%7a{mZ_R0?CX2b`>B5Uxhh40KFltXQdoJlV40qr8harf<1*6JDJ}ZiKKQJEaLhJrsHj^f`X`uu*4P9fIN`1s0AH8u%e}A;S|`Sgzp>50!ESK6mHDUS<_&90(FnJ*e-97$3)QsoIR;TaU9@nQRIXc18s%2{uWEpoR5qb@VR7?wpR&E4r?x3`m-LO+)H*u!ZY0^7tviJ{&d)RRzecARCIuUpu<3urqT3!n(KpX;^rfgPglZaMsMgbIw<>CM4A5mJFrFvRs7XPcQ>&#y*zDMpb>g+6eKA^+C%RDaFX5z?!S*!$w9jxuQtIEno`x0>FT|D18&Yj4o8pL#-N(spre#JBeR<{tk(%z(1^@EyWzAdXD;72e1GxB8S4Oq<I3rU$8(rAYudQM5R@n>dRmeL+%gPZvlKonyscj=C!>NDfLp5wdj8Tyq5J|3dF@RK1Q<>YorD(Z!DYXGrU78N%mqU!RO}&=;;^bS=l`?{AfU<u|CwW~VaLcHgcAaPuC2QicC^Jj6*^ySi5bO^910!RDOcbQVp^M!c!TuTUQk_pVYWo)L8a1gq^4eiI!(CeWP9)&hRv+%Nut0Iu2V+5A#k;g>7=PQ(B6#`TkjkPO5J`%!#K#PhK=rF=Al0{7WTa|@8o62|8V{(9F1O3bh1?be5&dlvK8y+Hk+}HSbv1m1;7cmoGZ%*<2AcugqAsm9m9W0W9owGLy?gBe8c1FJgIu_6m%n4`Wpy4h+b;b@f3zJY9J&<LQ)9zaW1W8bl-)Qt#B!AU=I6hqZH(I{^dtJ(KExqwZuj?rqHl40Twzc?7=l~BDiX|jFBInDSCarvp8jl%j(g&%P0BkbfTBx3P-z9!EAoMav3E@7_H^zV48>qIM7G^-{Z5DZsgYT(s_k4@BB67sP<OX6x~t?mEy_y=n3ZLikx$T+Rd0VPZVC!q>4S-V7`6}cE1TJQxRD_}pW~Ifu9_yNB;2^A>QF-(*xaRgtLLNykLt^n^h`YlZC9cUWF!U2breqq?o&n91Xh-=6h99j?r3Q@TiI?_G}NunCEWfhC!JJO(5KH-l<(^^(H;!!n8FV3Q$*fyC!?Tji9Rt=#}fXqSYCzHsY{uu?|1O~_IulohOg>K8KlO$x^qVs=({^t6aI>h1CgN#Y+VRgXPF9wLN~mpf``d`{%Vk-6`S=iJG{m6Hi|v*$$ZnE*tcD}l)CEN?P9_e8IFQ&OWg9ZfYN;hWaL6-X&$=%R4SF>?Hi&b$VH)cJgPdP?#k;slnuxsv%0Y;R1kJSI~8%v8a+hcwkWoljq;HQ(7SRx>W@#l|J^#@U)Wwu8f)bIY>jwWwNvDm_*qP=-N|C5{3bgDbY4LIMV3QH4RjSYR_3Fie|Vlv;c#2q-90IK<%H{cC3lC0nri8zogIHNxAzWxU~6v?c)vxts4Uom?qk}ZZGxpRmve<G2&zSiZ90#oeUgrXA9LWRp!&pH(G)fK-Dn1z=_X`ZTn!1wUv;8T%)KPRksT+yuj^2nv=Eh`8;nY6G#CgwEhI$Cf8VKa<0!`?Mr+tgu()oH#Db|>6B(j41X$##5~coov)n@MkP`V9_fYo9!27=Z%Xn7Ou9gEt6w_2~U*&^EJIwaZ`Myj~Fi?xb(k<L8rOr<YBcF>LuYdJ2v#UNN_0!ldsvsB=`tp}I!k@NdMrB+u=txO@c>6tC1Nlu>w%wnBNuyJ*D$3TWZ{1wwoN}^>MpFDmQgZX%$L~f|5_K(Y(ok)Ef&+6$lG7*tHP~SK^agzz7lE?00VNVn5*RoZjMB_V&ITz_jK=A3#rXoGV`-v((<KP8T>{*tlZp(Rt)s#MO^M<zvjH(tPw2ht3KOx)1);q~*B}|KoUHAKc}?yZM1zYN#<cQ8r;Kk3UTV=@S<e#$7>Xyacu)gNCZ`K8S7vb2eH6nX#wK1%N$QNk=Iyq2X0(o86_Gb^^Z)Iv_#z@GBxofRw5~0xlXYp#9c?o%yA5h=SVB;C!Omwk30qXwddW~P3a%=Hy9jATc#-%<Z>W?LeTu6o26>222Ia(M63dBC1{$iI#IF5d5YkRx_ZwsoQQ)048QlRU+2npz`V)EH%Je3T)^)hWPk{$ADOG}5;eHUMNzvec-yknazhMNt;zPHAsa-f?Bf~_D)m5z@7i+~O{yX9Pqh$4#OwGFUUTTAZO5jy5onYrz8qm##IHO78qKsD_8GCA<n$({fXs@b#MoG0Zg;}%%yrxY5S%W8H;+LK&vnJkJ3_2H)&djd-G(8?1nS9bMWOn<t@csA7G{PEFqK=X%wvX-`3pO#1YFE|Qy)%k&SJdbn>aK3KQ-r#~I92Ca+W;G(Olf$UrDl`UvOXUj``XlYIjKNu;d1@BZO}e!w+-p!5V|cWN=>NAUYBk(hSHsKd)*L*-z#sB7KyIO9&~H%Tc3gRHYdBp+Fai@_UrTSogLd96!p$-X(7g4y6{_svMR1!id(mIKyZT`Z!(ZIDd0ZnQ&DZ>1aS(=s1Z(IDbJRQ^ieOrN^>xbBjx&uaQnD!OEFw+2iZu`qk08L8{Ph@yM6Jpc7w5tFz-I2yK=<F7oE{JA#J3fz_4f47V8u<+9_JpC=gDBz?G4qT3J~Yq$TV_jWW$~i~VEK+I@rs$_9xT<OMU(Gz8Ew+X1_i;gL9Io*CqkjEIk(=tHa2{DrnEaS`3t6FA8$WBrEBL<7&Zp-Qc5itjZdr|s*0(bl9*!ZU8QPdrkf344}<f-U^^7y*uA!`<B-Y*e@Hm~l%n5>P9t@x)W|O?Vc$6|E~t)RVa4E|oC`pn)FCk%x|_)7W}+fnye}TCY<CuNt7-pMj<~CLDQDC0v}IvX4yOvyO~Evkus%#0zqyM>=L9J0i@6T2(g=Ht?y3lo!F;0@}-hMK?&U!v!Z^-vY}X5LiUPJ>vG_lOtU65LXQxB+=<PSl@--c`}zlfqCHFf$;;d-~j8FcwrTkE<qs!b4j$k2}?1wTYXz!C)I7@biz~-%oXw6CElI~wKQ3Y;(0$f^TT53%P+qiC6ztwuR%`<?uAjR0unA54@1W(Xp~SU2M$AE_82zT;6)Jb>|oasKEG%+F0iKw(ig-PMa!!=dk7BWBp<+M^TbtZ)z`r4F7U3Q{auu(!j(h3>;xwT7&!#9HIgop*ePDz2BldP58_e~`wC>`j5w>vQ%0+o@OTXu4`KNb)z(qLfn#A9x!?mmbb;+<&|Cq&0InXqt*_zmF)lhtB!jk`Fp&g}>T5J#L8%22IRT3vRNDZaGnhNXO;_t;8T!3=HcyuJfV+UwAzUevy(MtD3rb5Ql>?;%95`w<D=-uTD<QOch<xc*H3()KVCM)fFQHlqr?cRA9n5=?dmqH-fiDcU&Y=GgF3*rmnN%})r@}^xPfoxg1zM4$)<NEl*B4v0=4W`(2Ym(T*v6hZz6^tlZRDAU*;VLsqPi0w?cmEQSXn{Q8YrCMlRSwf$^Ic(S%sNJaOr`kbKu~FWSpQ-0ijjmna3{ocg_>w+e3kK;A^4{KiW#*#RxcGNA(Jh?Sn|Xb#eZkV-6l&g46<9o`>ZoSlGaUdE!|>v-51QD7;Lz&PgPW=FdqrfubAO5dgbQyplrBeLOe+w(k1>&T%sOw&6#`0xmglX$x;;!ORBPI)dIya5+a-!ziADGy5p*!RbvHUqQ(^Tx@{253WVu)*E1Z75j>~xCBpXAeO`PNxXc3{DoGf1iXtd>cG(%NYS$qhnWMoI|rB6S{Ji;*?~d{d|JZm9yU~XnZ}o)xAhqu-$BJo<Ub^3Kl07tSOpyKf%z=(UJ#E9Z=QgS0NG2B)B>pIaHE2oP8d9cdk$3FgUct_A0*iWJbMiGj!85_%5$Kc9K<v8q<V-BOQ^Yv*UwRG3%S$4u}2OMp@WWV2h7cocml00{Es8BxQ%khV1EOZkHA&}E@n}p0+PodSOvvdHdYWSVCOuFIN??vW{Uq0^agOEK%yHUR)iZ4usaW<0Tged(+s|xA#`Z7Rk*Q%>IJgo!{IbcxpC2nH#WhJ7cN(S0<+#$J>04<!i)nZ3uw&=z3U_zg1dPX$wF_I<Y!tJ3()69ne+cqLg_NxSVP%k95}~`(svFQnms_d64=}W=iAVq!qGaO3G$IPk3gn~f<fqyp!_k36hR@4mUiKMmUy;EDvlTS;mX2yjv%h=!R#8W?ZEQ_E}X;c7W8ejsxfewBjq#XS%4ISFRHED2AoObVjS)iQDXg#BZ>n)w6+Y35nMR})do0dqP0A7S4pghw<CmZ($zg2TELkMnycc=h7C~yl3jZfaS&G(&N#tN2CeM#!LGW%#XM@pVc7+j=HPM}cy`c5gG7?8%QW=o5e0g^2vTQn!rLewz$Gu-oq+`xsK()ji@2j`#s`m*|FaBFBX7c+Xl@I|j)`jzB>b)Be5;xVj!Uwi$I(!$86e4hoSsM7E#yA};WSxTN88)LoyY!#R%3%WqiD4W7cb!HGNy!hX#pm-Kw|~27CE*aonco2hZ?vTCayGeE)&N(NMzquH^FuW#kXMglsFb)c8TORQSB54ym&bSHrK&&9nQ^z$}-7h;K4bZ-NBhll3M_d1H8F~PtHNWk0O51Tq3?j6gYzUL$r7ZinTYMP3UZr$_^-><H9M4L}6;1tVckdvXL<K?SX&~=A6LmM&45t&f=UO1W!qn6D^dAfjp(BH4v(y@C@9DqNP)uibC%>@HpN?TzKvXIuGG?6Bc}ED}(0GK(&sd88~-(AI&e~s1L5@NF)J5Wfbs}+7gN7QEK&V{fIb1xVVh=PFfVzW`oe<g{3lH%Mr(_1uV{^g$yokpkNp_cj0jZmO}7&9?n-=6+bRH(Q*pLDmYyQJ7JjFC+>9)+(Z-?&rrRL5?NSqgNrCwY+}z5%58w-8N9d&lV@+dXE;Q6<vCbLwHj+Mat<mXSlYo0&i_?`JEveaPEIf3+5+~v;Nd3NNPxs8+TOtNc~Uuq?tSQ40O=&AEMoTr`Wqy6@g{x*yajl%h%bWRatQ`!T9svx-+ALbg-6>UQ$or8QP7C=qcYt*1yG5z!M5sI5;+CU5Lt}?Z-h7-t&126ufp&Qu6l9V4|9|kh4FlaL}x)|j>IA)dI<KG;mjIJW$?lgi8*0<4@OGxxDFh<Aee=bRS=Hj@GQ89wHk5cFX6>=5M2hI99TMpwPkn~C!SMqz5`}GXuAZWVHmhXvoV}m!mBR06UG}3yig^_85pMhQntNHVw}w4U>W+(@M##hJa1;)@OTMr#DT{PN=G1g4mYYGxCT6ayuAUBTqNp5dv5HH!Lw7autIis@%#$8aDz$$2g>N8f_9eiX8KKX4sERhp9^fBApaVm=tP0*U4fB0m<_|V9H_=nd=G9rVWo&SBgBzJjVj)uj3JC_6qW(#P2y4-&Bai<ihN$MdW6eaP7+R1IO0N$5YD)Xr-_!EFq%Wo3Ty;HJ=?m-!kG<lkcQ<rD!Som9nH=l=NTw?h?|O5OE7r|nyFUp99J5s?j?~mytoERIk-3Xw(dr$ecU)~RpKDJ0`@Mk=MWudfx85QyEuA*7L%|%Lt<WVyomyFlwJfUF}QI`j#r5zj4M8PMt5YYb-9Of3%`Yd11`txt)bl{Ow6GZ%H-BTW){@yq@2J<6_C!Mx&xJ~X!DZ9f|%}sdL2^fE}lilYa|*5<s4op!ubZ+ti#ee%3ro>TVQ(&M5(w!iOW3luj07_c)HeV7I3T%8wHs3fK@+sACj3O*o?!4EOCcnx&-Nt3TMD}7`y6tbq9x*TGb2;&O%onti@ZD$gk%(Hk_uhr-m+<QGE?<#Blr)t!=<c8B{NTF9tn3Amc?bN^BOu+&)TD;bIjXQ=Uo}zV^1U3Mj{lCR!J9(Ab6L6ih}?wgNY5Aia)@1+sNP9C@@A#iyCKbq@$P$jl6MpTXoV*(yO#7*v~NZytn>NSeMZ!t^=bnR^q9;h+m$>;uX|UB@Ug2fbb#3b5h6t%h2aO%&XKml+VBL5oW`Rlt=XNJc<;0p(8MmJ6M1<6HvNz4&6k)ucTB5UzVbx(UNe=-30#gJ`>gPh-%10%JvRk%608d{M+_RkRmD+e>hzgj{uUxY(+cQF)J~eXv?a(eqa21g28JA0m|o2-6MZ!l!#^^9<x`xD+6j9LWUGVFV@0;K<2|SbY~ZDT{N%^JAFWfzCa2wD|wR@&@$f$l1(q;8C0;OJHXGJI5kAh_x<fkhcaiA$+=m1IM^I2R9w?tblWL8I$C40lRi#gA$N)7~F$}SrC~;1rMU);@kp?t>cXywC{)Olqe@~Z~;3btwtKnlyESNTuWqaAE$EgBLB}YxeXGvxAiLV%(iOIR;|I=L!%5%!YH{%W|u+14;GKm(J_c@0#6)o&7jH}m`{=P3~n@`V+YleAb3FX34EDA&SI+)#^<LfcL@WOkO#nFm_(!ad=L7HFn&&!ny45Bkr|TR1HmFVj=&R2WfR!71s0Q}x(pUBN%<6(v#7R%=Yy@vB`MU=ObQ-Rz&CMj5k$+Nu>{YlIL?^^4)3;VaS~qu<rMaw<J{r@0rRUUT?hUZRE`t(C9H%=EsaW4GOUtn9$hp+G=-LqLBjzq{opVOsBp66!W%J?a^b2U?X4jHIz01z=kUSp1sIqk#~~cZpmYTngJhZN3juh14(H>@dro49ARNLDN2|I7j=X4R4c3oQIn!zc@Oc(29K+*dROA={G8s|}gTOK@dq~cYvzK^%A2&;6i%K9XWZwazPLkOm(MzzGgNG?_TEvxAwC#ufDD=DF`T|G<@%bhwG)QO(og5JN3|`)WD=zG<!o&jdEkd849MoD3s=zpL)%Ug@BI|XWI)W=95Z}W^%AL=Un<_cwUU)PQmZLbD1+E=fS;pI5<UB=Nbw1MIB1utoHHBPpxE=z<b$qc3^L4bk1yV^cyV+`N;*&awl(8d&OPjC~!p>cAetrj4Yh-T=)>d(40WTIwd=96=aAA#{@3rbw7+Ar}8+h&z#MW>jkE^@javmm%ARB|J7~Iab8VAr_#2L!Xs$ek=@(EB#BG)-u^^=WYtGR)8j$z4zLUdQTIDK_^aG?rnXVAMu@<F&A!iyBX2e7;jt3}{Vf%AQsxghR+6xsrpkya%KLx+S46tiA%>4PT?d~%507bxRHp;_#Yg57PndX9ZdXpIU-Mcl~4xCd+npyw2OLL?9ab3x<^fTaMa=CD74n^D*}A+ELW90^jL{mzkuE3^2df)}Wekj36Cm_HU&gec4$VoKi1O>~%sM-DJsLfcg9--NkSuo^)-eq1~T6iDF+cKFGv2kj+5&i!{Fbc&KL5~0*60*gtKqs?EU5CzmcNT0w+0q)Gh#u7fE1hzm{{lFCg$0?B81@r6R#D$AnWc`AacaZCV#Hj$d4MItfkKj}lR5x*P5uctx=L$aJy9q4LVOIrjIMHF5EN{b^CY~vy4JTgD!BZ#9<;l$P|2%13_`yXMolwPi9iF-|l|v(QU~3t~LL{01!3uQLup<FHzE*VsuJ}+s2omQwv;yeTpIv~}nO4<>qD6F&#|zu7>MmZO^t%FfX7K3@**(K_M>v<UYlGDNU}F;&*5TgToA4GqKEkKleAlOfFq<WgBBatlphQx;z)OMWBIy8Jtby<n-pT;FJC{-@kb@_FP@v3n3D2KH{}%Qhv>HL=oh6=ikn(|G8SGPL7{PHjPM7fJ9z5H_UMk<OliV8GZ@jH9efb_q?YylwF=es!JjxxBT!>^=@k$b&grUPpoLeyEgoSflng#I#ke%aX^x^=ngh8b8opX;Y1@OrVoS#8QHBzH2<^Y81D71;fPL!y_^&+@*z{(<A^nhv&B_kx_1BVXeZQ#S17Uis$N$Af(=O%J5!KEF%G6%y>Jiqm3$%8XnaD5HMGx($lXExE^B90`1cNd;)lfy8ieExhBmh!NYBuneS6aEOUZsXDp*m8j+7XeW^3>S06wSwnnaiNLnUO8DJt}x2%qSOXn-)+?o;rw#zqCnz@sI~wW*RXq*B<uJxLzc4Wq6YH?yl@D-ha?w6%Y|0`0#cG1r4mo@JBJqsk3ixWc&KK#j-3mzyg*W?C^my?^C0L0kpMnTknLGWNp@(aMOS=_F4r<Du9HjxRAOW$fg%nz&^Iv`Os(R=G1x3Z|1olBNh*yOJg~4#s!RCXkAh{q?fk#-UKErsP<;ihZKEvJ?-$@w7I;fAumG30@n#HA{I1Ty*gk4(fgP%n<>CAR^oPLG2?(x(#UOTX;k5s4eHk3)L3kc)&A?m~E$pLM1?6XOZWA}Xz~zNiIzSI8uVGIV)#_k}Q%|xH!KZ%U*n+MaxTI3u2|O#3^e#A^$KfPeJw}BS<jI5T*>{cwGJ6awCwMW0x95o44OjO;F^kvN@d;Hyvglw3=3Ow|M6L#zrEHOM<T+ILf!Q?_al>>4`j^OV6{Tm1dk%Szz~&4*ZIbvtK2O795$qp<&^`!L@y-c1A}GFqPu5W;4;}l0>1q#DD=0`M*9zIEg!ByOT(G&0)-F*ngceTFYKfGoP}@N6b1>^9a{-(_1cfG9*#*fmO2%L@1w#=S*uWbn$XkJ_bu_bt(q6FUBk?8-xk$3uYGy%}Zh<^p^~1vr*jy#)9kj9s{qrQYgFTlpeDNmcAju;*GfNzGklBI02u{qA)C|bFx$Lw)+p4bOKme{#PE<xSM=0)vlnuuGWUJV!H&AUAtggRpxRJk%i$%1$hiI>}KD3&}=QChux7Dbjy$hH<f@=;G%b;y4cqd7ms#vSIQO3<ctG)>uIh>gz%PEpuBGq}gxJy!H*l>bi3CB0#DOD^IZz2oia)zvK;7d0Q)X~8#NX(*gqgCN%OB~w5?rl6z8Lk6XXGl4Rya#aR92Is^E`UPmH;H5D*m?-hk5Ds!ih1IW!)yj6=RqTiOF3LULCG}S@Wab>7`lM*6x=M4-3DAaMVSgXTEpHmc#;Ntd%!z~&sX4X6`$<jN)C9dXeoya+i=AV=ffbh2nr>z?<6ZZ5_OZf1D5uQGYuBYe8<;zk>7<<m*8X$o>0wZ4b{(pCjux}DHqW45m~RnKoBexV0e{WRA43vq7FQFf?aE1I|kO=AiIqAizIpiD}nDEVVFNB$8J<xA<j7*I6-tkB?=Qa*(xJf1<mA9l4{FU5cT4v3ZBitvt5{^SQ&tO2e`ENW;P0D{G?t(baC@e=ySux4Gzy$H!NjwWdm)5{?A|FlVh+m2NG_QapA-!iDb#rJaN$B`C1oIOy!!@G~RHvnoDSjep-NgyQmoh#W^s0j?X-B=MZL#Fhv1fA$xNqQirKa;yA@c3hPz47s3lWq`C_7Ww4Sb*$fDtf!aA(brWX<C7WR75Cx*Gi&I$N$1{s?#)lV{!6H{gz~&a7rIJh*F2-S~f&K9}iFL5Eg@Xx@UPJy}LKXTz1W;+q*T9)*t1=JP_JBVDmi^Efe-l{-316$`8vg?bEWH4pS+X=sb`QyJ5qR7!+FxiMuRGAvCR(5pQ65rZpg7mMSVZ-UR&5T3GSIgOHYo>K1?9wlLx%~nx`-Djq$y9?1epyOj`3k89H<Z?yXUQH1o)euf}=2=FW}2nxG;w@bx=RWn<cR5B~<^dxZc+HU@ZvGR?%#VT)5y&;B9jY$G!N70`w9jwn>Bv#AS3)hZMM_6m~R8g$fwEpmc&uezYCLm-`@?go|e|n`>1ZIJt|Gi|}}lT&&>r6xnJ%fc_(Jo=22Rx;D`MDi*B&AfLjUDY9~m_ERLf2iN9EdW|e6@j4YWR>A%r+zgPt5XvQCtO^cO`1}An&akHdW*vZP_UnFd=0fu)q<#t4y(C#6nJ5VqNHm95YcLT;dw!DIBbQ;~rnIJxmt9~sjhFmrw@H>`D7gw<MZm-eKR(UE*<F<1#@^7|dKn(a!I}dEcCldiMKLGZNutIwh{y5HGTchSB;9a+v~U253FwT#NCh5Ik!l5&_MpoLsjQseM=Q&q9>?xYkT^nH73gyi&m3IeK$Rj|p9i)1R?R`m8Iqa<^(~O0#CR92MX)o0_GUmJ{}g2pKxGNeM@ZR;jvBC3fk{7D+X00oR9Gj@WUEfK=sYNgNj$~r=ZphWKo^e5#`fEK6a*8v>1j2>$hm+HeTdF|brEfy1Lp#8<Z&R1T$>~x$46CM*asyaYQ#|O420sSo+X|GFjs;rQ8crRd{ll<p+pd^9)Q>!Sy@BQLlR#^{s;;caApy2(4}#c*cqw$$kq-_))19Fy!*IO!~PAj><4oVH0J>R3wV+eIMP0dY=EUR6r(c95j>^(jthk|t=bvd*}w~Nlt|)@V;nD&ats&fGPpr`9~bsuDF{~9TJ;*(OT!f}Xl|n>)!dHXG8?)dl;W@!LuYwhDiYrbbkoLtaB&-7c;GsfQP*K4jvBfD4WDJfc>($h=wb`3MoBJ>y?ZDcK~#yo@WW#&gB1mdDTru$tKt8SBNQVWaKQ)t6j8Pj6)ECnw9q8EBycwId<tDq!8-vr58*`yM7Gg(9?Vs7&I=X~;L+-v*a8UGNS#va6rvJ;jq0`sc%^{0XUSF-E*^qL4N%0*xo|v%XO^Mw7@wVx12?Fq;mICaZNLImr4L~!gtiY`)ni;LgWXxQ$EgyDY~jm!P@2b%2BbKipZf|#Lhz&v^BL%9!tDlFO2bSXZ!dsi9aJb)^rP4jjOVduzExj_=XvO=L)QjQAA{L4+S|h09_-o!$ql&Z21ix2*92Qha=8q4B5&)tH%UL-X@G2mY-LIEoH)<$`2tBE!tye5dESKcc)o}h53p+qr{{i(ibv2dP|vjz2p#^c)ud|iJfK=*D%+}Tqs2>D@WX5rXCly_fjjG<o(9E*R@H-pyR8c^*v+9@4M!HS_Y~}}gJcvq<1l^FYVII^jX35}J#i1MQqj%xwtj%t3)thur*t&?aP5**XW;n(yzr4^4Q!sjtyf5L4(Huqa}z}KX!jiD9Uzy)TeC2bfGdKL-*tdbw(xQh`l7_)Lp3U&RIzIT>@T-!2@*|$xlQ6cgt;A9N&t5a&&1H)5@rJ5Hc3@LG6o!3m|npF58hZJ(KEOk0+9qxUBHDsv>b-Xb6BN3lPU`<aCI5axbSHZT&{!V1F#!M<uIZGnBPxgTW?~NXL!NJ@c;;Jfx~R8nu6|GP@Lh*z8r+7Vd!tdYy@0pS{FMY<U`p@u;9hX3~<EY@hK_0K&%Olci>S7oaf#)5^w7Pkf*{>mDJbZ&JheA!psTIu0e`y;Ulnm3D44CegmbCU@A{4M<kjivnRwEBk?m-qkJ(>whqDe>_36VA*=_%lD9>LGAaqpJ5YKHxE%k2J+qK1sZqK=b~!jRS>OynKUKTEpm7eJ2VkQB5_533PV#<4>1^ByVwdRf2v=)(`3OY|z;l2uHt^9go^#{oF^R3CVhS(D!0rVu)Zs!Agwm+If{yBl3eTrI=p@%_G+{6Z3#V`<OWc%2Zlhz$RS!}0gaiv<<s8tB7CFU^JY1_|ALaeWWY&+<yYO@u%*_ex$O&@3R(<>B+d3V{3Mj0izy-WGfek9up5pC1bW{DCuEHfuEyHG*9MoI&B~socnMIf?fMgaoD0|u>u?@Id!6z3WU4^+E@Fm`aPw>Jq$t_|^qRL08)&$`^sJme*j*I85N*dHCx6H$%GKu@(QVJ(_aL|D$XU>KpWkZ)fP(Q?%HNNiUFs_zhXcPK<Z^CurK0%Fes}h635O8e~cL<)RTD4P@aKp<Ob{w~=(Klhbc5`Ui3p_F0C?V%1@g(88gET^5v4R)epq|Ix2ud{Yc>|@XyuS~;RBS(js~KDhVfPMktiAEZ(e@(Pp8?GX%%tI*7i=D(0M+qV(43Dfds>wMJUt-GmpCH0%yawXaE3%3aLtbzE?9Gd^D+#$NHqx}$GCWjFFi0_ep?Nr<{HRv!a7CS8C;@tFadMhWc>`KU3ex$av2<Sl5iLYGsL-w8^<uPgw`@RR>SKhT-rtZYsgnXE*~zegCJEq&tWkN&l4~oBkuWDa~Z^fD4W5#bCNnkt{SNvgNr06FB4aR545;Xa&!{&WX}tNdtf$)<|D9ih<qz#V+Z&*!R8|J?6exYps<B%RE0eTD?z*+0D&ENN`<B@Naa90@y1iamnB?{5Lc08sODHkO%HY-LAs8|3*^v=-5C@*BkmQrafB-|bm@f2#2c!^7t>_D0p`<i=?JgPz*-b-I$QM;imc*F7ObBBuT_qqks8U&f%G!TZ^5;5;+O^dltNP7It_fgq?W;j3o^SvTnDXY5M&Np&Eg-yxeu&*fG30YGSGPeo3!X+i>`YxjQwd`OaZF?FH!JQ{EL9*EUE<n)z9MbH;x+IPJ<N}JYB@6RMV`YdI~MO@HwT$epHX)Q!l3DjvkX}3S7D%W!6rL<ufqLT_R{JKn|(QngdR%ER|Y~?N)6cP%Zw@NA^=)(3b|Y2jC<ColZh&Tx=Utd49J99jR8u3(H$Lu!gqgLF5RR=kZ1k`eMjegyDHuXrkH@+DNx9(lF!zt{e^@0tZ#uoUpP>+<UEBiEJ){?J7=}$(j!ZYJd`eVgx4hxN(9r$yOr;vVKCj^nzR1_XT)SM9n3zT?54xyct62G}>vjYJReQj^^j#`3$KAF;$sdDR6#C;xRZ+Sy3L$df{3CSMy+#3Q4D^S%+&L*th_x6!iJv;SyY0!Kr0b-fCU$;L9Z3TE?y-a&Lo~IWQl^jcv4&2FWcDNg<yL#g@^<8aYm)?Oo){gJVCcEt6vpIBLELuaUeU;_XmX#trEXyj+CU0NLM1xhyEVNb2C{sMa76%4|14eGR!?FhAG2Y=Y1_JX%3iNGYCzqm8%CHC*)LW){tywW=k2c0}?~>|BEJ6YQy?umf%s(0&$|V}Q!MlwJj+(Cx>C3aWYGsRuM0@O1St@XWxxmmEaVN)9gsz+MHfH;^lfBV|0l14`S1How!l*hjGqau|FQaT8DOjd%HNy+Gz%(2)Tvbv)w(4XRD0K{n8;dZDj`QZ=yXM8{6(-UYE)6wjlj6YM{Sr#?~(U@Calk8tRy)eNJc2ZX3dy^AiJAClxHK6Qh<3)3B%-)~j6@L~`pwjhP#MwslKfw>BaJ5gy4Z5N3%hw>LFG7IX_e|=le2<FgDnAjzAK6E)xmVz)y+2ks`xPTOpwF^|pgS9!7*aWk6uuuibBG{*5zz#mmlX4pEpJ5jz&%0p5L*@&x76!!%P8aaeY^xrCXOtMY@J8jy+xj+|-9do_E>r!WL^4zin}KI^%u67&3&K=;r$|=^;W`SHV4?)GyMMy&WsnMx>@F(MJ?P`KYUK=P9dIQ<s!bF-fIHbY392)lwd!fIw@h}ANb(H2mtmqz(jGDw1Qg~CC-BXKxiu6BleFV){hU+|U@Z%_VrYqKDNW!y0PA6}&_q=EOc&v{8_(~P9m*ghXs3ymXW?lG9Z(vX`yEK_foK%eBcNEq-ZXF@5@(Y*9Ast<FYc4$1i82bES`ru$>B;JHP_+l269x<+6qiX$nFWC@<aF-y0UM=r?BRI<5(dcx;#{MTK@mpd)KbUk}Fa0ugK-)g_K272~{pv=a{~?{V2Dm%Wk(_ZqJ&m99;n-Q%({v5}9Px)GR#^An_0gfdoQ;0Pz-xx87Ro_J>)sX05yK{DrjQ^(p_t?1=NwIY&o8mVNu)neOs(a&%6_j))yQ_A7S8X6NVZVvDYM>B<}_U!jvUz20U(X&ZhvIs2{*7Skb)9+1F6J%7Z4A(|?)%rz?e=xL1|Ezr;;+MT77el(vVsWkDgkqh{lM8OP-T+rwmDQ9R1Xk{HO?vc#{wB<#o^LR6lB(FGLOt8!`o~+SYhO91<(h^%Nv*0{kI6^CH_3Ad-nM7+Bc&C6blGMLL0=w@(EEgxyW%6w(gRU#65G3(3N}Zxuh=o8zT%f56O3l^tfWT?8yuuDaJO$Cs8I3QY;u5a;(NYGT?b7WWDx}##j#L6P=_S+iG`)>e8)!Go7QrT8!ahIot)tlx4NswDR&dzOOPnpUi4&GCbl`B99Zloh0h--HAcvNdWWJ0RBP_9iVlg^(h8JhaW{!sENO6m-?IZ6ji_Ej7Lt0%y-Za@hCL2d&;hKg6Z{u^Qx=$AJIB`vP7EvI~Qc(;dFtkB{O$0y_OQX{wT|GdVNmAWI%LTMkVCVBFIgj&Ul1-51Q+%CbfjzRdN~XYm*ca#xY@)3VTsuY=C6@HDg9|hR#{EJa#N!I^<1C8opvls^@-7~qBB$dBnh9lSGWL#3)Qi__ZGzTTm^VVheli_L)hn_QCu=J#HjW}bnvUZ{3{5Q(SZLJ^5-Om|Au5bh|1_BgQ5eT(Ieff{Lg)B;3RO>VIzYfy1*<O^#l>rUlOQE8PG2MM3Z9xFWyy7Vc1SjhD4W3tU{?oFX&Ei$==2<|B`Fvh%ZIqML`qc(YSsKPo1Db43<>R##39-9k+m#2xF&%qGIu~Cv-o10_>bu91S)1pa1AeQ)7lO8p0bN^l-aBo6F9wqfx|6VQLsWz_Hp6}Ws_`ti{6})jpT<kd_gv9czK-c-{9&Tz4DUc9*gec(hdSsdr=5oRJeQ#ihpI66nALAM-CTIb{hpEI8*))1h5{c(ZDhW^LHA|npx^Sz+f!Y#&OxtN@*NFqZ0{QsNva4J@2EHBfL|iHxp=T;%&@_cJgdDN{VSxPNS)7bbf^|a&#|*y}Rh-2yaEu)g(T-pfhn2DWZ54O-`e&bvjoerK5T|&bB65aEDDC(uHF-u`B3zJBTJ`==3a}I$^#DtrhXw8QZuZ+xu+Ai{|I+l{kv*;;Bov5<<CEy0c2BGh_w0LxrtH>b3PcOylYtxmaQO12jF2E0buxN^6Jg@P^FKyXjhtZl+Q2=s(}Exd<!l;91abg7xw#@&!mKiLdvmf0-R7(cCyoPSDsTtyJ+w4P_76T9!_v@LY!FC7TfR>-{*r+QNwpDbKOXDYm;$eTUcw+;$fwtK=g8BXr?o)d<@Pq1jbbTw%$JdUX?Tuh#Q<<SWtBC_N3LXar5f+1fbv1(?r|&yuvZk50DO(mL9mqOl@fS!da4lq=v_;0JjMt9U*@XO7Wsm9C%S+#cDVBXdV&W{a(_qNQE7S3u`^mN>^HSol{coe_jP5g|uKTAgIOlQa(6c^U7XvFOR?=*W+9d9)TLdlfPsrBmxP7e-3~yct5iDKvA4uea-k5K4LRLV?B4Nh*V93M8=e79`4HoW(Eb!WHwD*i;x-cF^q60Nag`nK)ZKA{!^<e200rQ1Oawf!a|)Hx;xO#nBzQUSx?ScIYRW3>jZT)hSxt$Kf~$NnyZ+AUj;E*LG~|u>7t(K{huKu<fmVTB*>}1G+sy!W(4zm=(Yt$m26uE(i50D2*qq5J9tGon6#RrMF<)6=qp9jPgEQ+9hQ$-!f<lnAsBB*rVB9I=M|wfWr1kc^-o*o({5=d9(=PxmvG*+Ml7(3O-$82St)utyd#7x6OPf^(ttUTki^oB(RT4`;x~cf)jgh7Y^{wB|cB0vuWnvW!I-@YY|V))@!?{I>VOyD3U<g6ke;~!*P_XkV7AZ_QF{<IZxI$Q96in5d_*``k0jVSaFdhF6i_ny}2OaaU7|j$UfWJ8)5Mxo0=pWWp)@s`#xImzV%-Gghax0{1lJRVxaJHn9e3pZ5hvO(}l~o!A%?qkw}izq-~YN622+m>s_2$!xJ$E`fVVKc8>6Bl|(m*cL81Ga5lzbTWorQ0C$=W(6bOtAJeG`651iDNu1dvYm>}h!GTGXTmG}xOR8XUhVa=M+Q_~ugN-?jiV-@OKr>4y<iphz4rS@-e!YB67HhbCN^SypCQYZ--W6grK937~Xst*Nx7kskUOcC{EDiwc5Z-HY8AhOZ#JA`PD4W3FGI(+Uc_VKZw#d{Z4xC^R@Hdxud5w8b@NyZ4mkH=l)p0z3j1LmH7(nxzcs+;139?&au^Bv-d{@3^(F0aEBI#wEI!DnX7A>KjJ)AhDr(t|OgNtC4@8X0X&m7S(7)!@AcR`mz=yC%2BBXNgHVSrc9A9pbg&d7+kiF2miiovG;VnA5ji$lADB^h^z1l|U1Rjr|ofUL6L%q?_dM!l~^JMWF7mjcykH=RD_q1%G>zjW$Kx@a$A4jQKc6|P>93@vL)HjXuMYI*Hm(mREs)QHqU$Vnlnp>kcc^V0^r8K(Sz|{+Sd5)4v8cx(JDFnuFZi7w~@nspGFOp=H0=ZmFlBq+qynsMYT@;~TD1U+q7ie*xrB+bcOMGBt9?+{f7A?}NMY4Q>!bfE4nob|RD_`M_aWWqy)8i<x%obK~Iz|f%cqa3%eEKQ#?c-(OQwumTgVsUeoG0U_=<Jj&&6A@H8js>~m989-(?e48;lm^9-A4;v6s^$OJWCdEIfTKsDXx>_Ll)h_7pKTuW-C+p?0_vT(CVV#V$*4QvO=cIWZ#RUpvkB4K^iab&@(VkS1>5dn>Xn40B<MgtQT$Up#2TJvx7<r>;>KDfYeUjm3PR|GTmE7(?B{olD#C6b&{S(`x7L%hjO58#89b5;vrHzp_OA4+o9D}l%J^=FVI4qj!)33AXyzpxfqVjqGXCC&q*YMO9kxRWJ_zba7|_oSyoUkzOJF!wRhzyoGsv`2=am@2(z+CLd#@s64f?w$xqTC1#)D69i{e(-;3sgxOju6lGq3K#3Vku#9-po)@UGrHX<mvhXX}ayQ01rii3=bqR9~6Kg8ula<WthWhh=@>tL2f*?Ja%{Wf<&z4NGegKtXs6lC-T^KDcAIXyX{mpQf@r`J`M%945E@1nkIvb>7JX9P@x`4HN^rc>#9^_W2KXHs~2m6Q+g^fHDio;#$cArd~u%SWV=uIJ%v9QhBZcL`T7X(Gr#N?ctMu#n4>bSF(t;`lU%XFyPeNHR>r6`HJK;5)|`csGeplHcOXIGM>1kfDJTJv~PAQ)GPxoy?&80-B$|OY85-+x5yKiLT+a(8S0|3@1ProkW2PT)G@Wn>%!RjRaz7sY+vuq<DyPC#)Jq=f~volr9v}oF89i$o2|~#@Ny#1^vOlfv5M#(hklg>D3HA+oz|?$P30mku4mNQl4zzpp7CaW$4)|I=Er0l`qKnGFsin>)Co?qh7s2)eE*2q*qt?=HOj9j<0U;E-=?JTU=%98MY*J1r$0Z+hH;jW`Qi81mk(1&K1~U4#yMq(geLqlV}2^(hTgX(`mGDMJ|@e)*g-?<Etq=Qz1JglA5Pm<7hHP7I)DcFsBW?pP<uwED%C_i)1!Rb7{N*{N0N$_R(>O`X@<X4TJ0e^)SANecQNrNGkFFbB>f2sP_it_tEYFEAFtxD|9|jm)Gz*Xk;SVg0Ako^{$cZG~V%{_%aKgk@16n#*uBD2+&Fqt#30x$oviitL*CHZG3{n4p{J#USFWe2^0aP_OxDEqoIp;<sv#;N4fEOF^iH{Gy)o378TCXdY-IK(p-RMkJ0QEEvN8Kv|df2d<ubqQ}v;#DFUkc&H>qvk<bcG&Y+zd$(Pvi2Hlvz$rThX|4(9lc_&bOmnFcSzh;FTss-wmYnC};u>$dDF-WUag)A+wK%7>}=q!S#4_IjhO<d!6gk(3__6GL)S#lChfMK}I#{FnFj`Jy2Il&icvR7ikLl!(`#R{GB(%=HiZ+{=BYAhB)kt8|XC0mPRvy3x0WIB)Ml5}C4O@vVRiq4j3IQ*`hM3FNR5?q$}PY4*or6McO(TN4R<EMLmG`EDd!2&y=XEXl-THPKxEzsB%nYkegadH^1=M$uOfiv6qB9CHc%<pA^Q*^mZZ}#!}20hKAlPQ{<z~>hvc0<!ub^;Rp49^r<eiL1svh7u}y39bqI;!C+@Sr^s@#1g>XJKh<;g}cu*YV~Q0b4SeM6)MkepQNP*vHdJ8V#|<B05cwT88EiXdVc50>|?-ea-^AZ{e$d8)ah@m@NpJt4&tCtk;g&S{1J@kaP-Xj_^$ikMF%JgSzNPwL?-~BAKareir3?^q|1}TPSf>FD%ntAIa?@KybW7Gc}x;dj|wD3zq9C+CKiXcaAQc;p_~H`EemhbCakH>g^^??@N8%dmB8jmq5h|puj3#xx~k5w3?$g+XR%hB@k6UwmXlfYGfXiYA~iw5RbPl(5qE6dx6fgICz6HTP!hEuLQ_cj*Nr3c!T3UdU8pFA(Y9Z`6#ZG$>{{XnWl+7wzo!tJ7oS4pO;DTh8$1fsZ|74*UAjJ2-gd9bbE=+U!fvs;8%Eki6vIaaZ(WL(iF~bqrDm(U%=%{avVURH4;xy|0VX$;Zr}Y?BjeI#dpx+#M@{X`&aS&COQKSc2F-yS#lOlUXXH@WUtWrE-9_xwJCI(p;NQJB*8g41=F#L=6&d75ra-1NwDn|Qi_sl1(l05Jd2O_5OBKq3_4w>$upEaLz7;XTw=ik7ER-+c{;vKvM209oEO5UL3$n}rzy0u^IyQy1<U#xC#v`)j8`^rc>*QQ5tuAs>#vot_Yh@sXflBPo20f%=EJnQO=@AZw!sbqELfzq1w1>CQc09cvBeNtJfS<+XlH|NgIrl9)j6~b>fr)D&*R<8dVZN~`S9ft_8rlg2rGg4TVjb4y4c0DGi2tBMnFy8L6I1XCeh`d<gQyFkxjI}hz>IJa*`ZQp<09_ymWI3pZe;RDSGC^S9z38AkZ8aCQvfW!bwuxtrz$3c?n&vl3gIIJ$hN7xf=@h!^R~F@1vU<t?km2EZZ6<$rHK(Txu3APLc}DSP+Hw>eT?fo@PFf6_<1`L-*!MCeAKq(Z&XeU$V>+*||h(huD9PHv-rjMeEY`La2iByLj!87A6t5l*S2YC8Zk_onhlEs5nW13M)$l#O3}hInJPy9b62uXbx}Z*wO?FE~D8fzF0)_b2y)6fjl|}rFpJiinCITl-B4vOw=Wdl~5@}R~PZ6@6X;vdN4(=$MNnsUZ2DVMFi_1QYI099V{g1%;h>;Pq4`u1idU~NM)SvfpIPN6-hOQ%4JmWljt7FR_RudErEU!!z<S)9mAW;Xn&J!Ezqfbv=>Hu$27KqulMS;B8sfw)FGV>(A6}}=iZes=|+L&H|ph;cljKexWX4zmRNpQUS<bXJaxhR(>Q&=GI^3NlkzHETxJ(39Nt9n6j_}nr4#BMCx@FXb%{!gBn<p(p9D|X_}YJ2Ayemc>rzl|Zn9q7!NDyI6PP}tQ{xQCDLe6JpAY3~^g53->-B0Jr$Mt%vc(`voYBDHyYdFwOw;NtJ)dFQ3B2fM5io0fEPsMmCp~mNfUgd5=$K`LGy;P>!61kyi#WH)g6p(cq_Z1%cY?;^-Cuo0=XdE^35~DdaGB0rlZ7n3Ss=+vl$xQZev}7e;sUQrw!+*Ns_fFeHRM~x-UzzL;t7~LAKqMG6Ujd2OJLst*$UCXDNSYm%vHHn5?jHmC-^E%qVwc*fh<qI4Q?^tJju<V`7m0&sTXT3dW7Sfcq>4BWd=%lZGpu>v#QpsF~%o&vtA0~y*YYvj<z<@R31+q)=S59u|_gpa+1Z<<7|DCZ2HOM9$O0In@LiRvy||OvLq}JAju_q8h4<bB8t{nbpy|ZSn2fbLK;tkoxMm;Q#3b@PY!WDk7_%3We4XoxI9${?PFq}`a#?*lDz<(-X+yjJhOm<ODLJctIOngoG#9yjE@GV@p*+L=U4%3!bR-Mzboeu=-|~X*#o^kfGcY_xr_ZEw)ft86DYYuPcKO#!~B<IW1C(cldumJW&{Dx&7s%@%3iVES(G|L`8~2<!0|Ztk7F==55V3?5uo0~D>PlD{;hf~jW6>klc8&?_3}Jj1>@ozhmTMg7+sOXr)VxlV!%|Z_1Xo3m6j<Z(6hkAKi{L#Q=G`slN}n}M=NLSV419YnSY5KA7dXFXxAtO{5_6NLUeu>U&qM$CX3CZPyk&`qsbbINXl(#>jlki;>|e8WN~f+hYIB67-zv`+5dmW@kI&85_IZ<Mq@a;OXrGY{sP?u>%|P-si4bCvR15@0;B}8VjidWQ1KcCmhkp5nFmQ1f4fjaM<E&mt;Sc+uVZhDL{7-^CC-)*=-e|Ax^~HeQF^obXWt5*o}t%8w6TPutL$P0%`DT(Ddo!k5{ay!%mUe5rg@1A&rRUUIy;PzA{aw4wC}Cg#z|(L#BvM>;3|ffSD3eo{WHjSiB^u$p6_jRf}ZZO*g8A7BJ;Z>0T5v-j}YiE*%00?q98E96PypCg((uO;8cdJP18^gZx+bxJWI^uxp`8}ldS~X+(vUX%$<kJC^E%XbIcpVi$SumM8PyXE4-^DNfuPlJPt3S(;%7?ele5@v7Hli?ql0?v^r5Qh0*RJoe7bn!r#-I8@jPZPHzZEvS^je&(Twu^<z3&X0chinnRZXvbTW~moyheOA}-Q)>4TT7SKtAt%Vq9Hl+xKHL;$d!6G@@#@CBzw#Y&!=mt0%Foa2TS*2%NxRl0MNj$lRK<8hHSn<*fnOw!k=kzL!rhIe-nmj}Y*EA~VnXc7v{1`1QGJl$`hVfjkUiOh%4i`brL=c!EfhDr!Mbil!4&llc+DNg38anWzn^SzS!iuGOIfOPU*jKLS3-ol8`B%`w4$EAz!+n$i$x&gkg<sIQD2uM543PLXTG?lb7~7hnAPgd5>|I9dSrVIlSICfKFi2-mc88?rXe3AGPEqiZBsWkD7>bWxUe)trPmLX%u%i+?Org_#yaK8vki;q5NaF$+KVWmsvg#DwSfJAvcr{Kc!FT0-R;iKN1*xv1vpEVnYWCp2Zem}705_hGkk~$sU8BVryaw7(h{Vpx@g|v`BB^~ew}H1qC;@iUF-jCj1}u_QGPg^Xx6xUMMQb>9fr4k~ELg7{AnzL8TR_EKHV*dnMZE&@DNj<uBh8kMadDp1N@O}k18L;DMiD>rSJ6&{q_cGPkZi_KaRM#eAmDY`E#~!;%`7SJ<Eu4Py}>oG6e?`)0>!RLW{;*q6i^vHuU9V6^*Px}vdJ($O|yiLq`ahjfkSinW`fkJsGLC2Lv}MsV`=7{rppnu1+p}VLwjf?hBGldw}+>1=;AJ2y2MivG9RJ-DjPo&<WbuqV6JcFNOgx+&q!jIPA|~VG@007)oE0%;7khb!F&c#^^l%y(YTKVBJ^s99D+8O!z(#fn8k%E%1_j*yEJgk;+JT48SQP;aFpzB;e$i$E1~5W&4B$hOHZc|s7t4dbka+fkEp+b6CpIeK-R!Gf%RErz+eu*il4!yX><@G)pL9fcEJ(4@zd%CJ;_Nt`UGu)`8dtaYHY1Qj@MCQgT{{Wag_#Pr7V!S^L~<EVjH{cVumcN(C{ItF5~k+y#%@w(A*6^g+a{INSalBES1Em2%epNS4rXYC5{JZ?iw%U>a{9~@6g3ey%a;yDBC!q)hhD>A6)sfH_SG`8ji8WIp(`!v2!%Bg3qfscS$z$B;Z5YHSEvQxqY-chvrgJDAN@K0=!Ps!Z{13Xmts1lvpf>mdZFZNtPE#3aE9K1*a)kob#9O$_u2lOh7X`1&*7db7gcg`K~;JLp3@-PNwHbZI@ikpy)K(TzFSr!Tu<VAJS_dIgg^#TD>-pV^y?1kFph(oMDS+B(;s=Yk0XpCrZqJMj~L2ETLK!UrnG|4X^LBWQ5EGQ0frR-Jrk~OGq|;Zibze@%lAdPvQI~_4)DTI7+S~?>?QGpl4a^O_RwbJRQXTBNhN596<Xab^!|g9$p_uy9ESFYw?V3`B*81L5o=i<uQw6HJaOH-c_=EMH0z>0ShcfXD<H*ByEg=T{M%VMIW17AS<8%PvLbRj$G2KS$q*9Yd*Gq^fnqqMUcRYI1t1)n=~*(3*&SuhJs}los%|@Z}81JS&zOA?cnka3*L}Shyp9Q2#^ez#9MC{_L(<{{VB9@jr_A@w?Nm<XmAN9Ch*)Q_FvLDSgtGd`idq%7fIGj8}%YEopn6t{Su`P=;=m14^w!FBWb*KLDsI(d;#Aay$x=onG#-@q^nb;cty^usJM(WQ-2`I1PKQ5)E3<vM;XwH_RwksSMqo*g`<;F6z(qFm`3p#k_C$x`1TpPzDD5`DaP1p0$-i5{0-Tt5HKh~L*B1qUk<0{(5as+71%_PBz*s}NpfHrm(gLAEuN6g3|Xi$7~fop=2qzT8J^xpyCJ+8p(j<g>&1aRwt7Kw5gG==?id#~(B3?XAF}O5I#s2hkNHk<d>mg!QFxO^Hc2dtQsK8#CwOb(&mc7Yd6L-_i}K)-lmiqj(!+DKbp5UzL2H3Odza9}EIRYzYyged>V<g<1d^}f<QhBP#?vu&a7=1I8c_nm_+%2F7t!n`n!m&|J9N5$qsu5)M4MMAm?vjGQp~-LM&C|E|LX?YKBAE<nOVYXF_Z}5<N;n^XMRw`c35<uPJ@NEfW1kybb^*EWOa^hETW?rx;a4CD=41C(`B~5i6;b|Jjcl@31`{LGz-qs@iQFQVCgFqOQQX2vR%Y8%cML`XR|06d<!enAH%Er_#{d<3wUY)Ut~xTl$=wVj@3&k5-#ACIL@5lWv~#p>-h>=&DKlf_0l5UPO%KoasXGy>&0twl)~GfDV_eD9893K5S~AzD^;{q!o|HY<{f9%B$^J;*bUxT#rYWo>OgD(?M;)VGWCjBzIvrh*N-qL9;b6SG(#pXNFqUC-1%j8xP?F?Upu9hD^fimnMqQb$Dss0E22uA`X}(r{@bZ3dUZoVkb3i^aze^M5;&&Ir*w9m6nu2;hQ?M2=<FapbBF(T810;}#3cfIqY$IHGvvELixqMNYGaHY`p{w-?}k`x4Hd^JXxQ;1TwOs&QMNll5>+}0GAxCqxDJ}}k-!Sg9kcQxKA)$4;AJZ;4ysm~1lQ5|8Oztu$`MMfqTLMLIYN~)oLZ;3S?r(18z9iY2#CBZhgf2ko@D6hI=-62=lispB)b(7o@9sXBnJjT2!+>aVTG<uk?ak<^3khH42t2*2|n9lhdX#~9|!k-fae$SW`!)nfYz{YmID6Q!Zff>r}8LrfRA_ZbQR4AYlf89>ZLgpU#ORN$;lB79?<FKdUXM>C(ywaJ}%<aKHe?i&4YJkKzR~RfEbO`t1xAGwse7}64bwnz!0e9NT!IE=J9d`FKsbE)Wj*?y2hdPcV!<2^q(j2$ujfau*GBMFQPKo3k4Rv!09xtRqE9YDQ*6Fe3_I2Y<vA}VuA%%Ny<lO{HU^nFRxH0f_YrnK1%Gcv=mM0_0d9@fjkJ!&`=R4B4}olEN-HmEDju_e3q3qX}pGJZ|KSCTW=0cZ{X=;a<oAGMZ9_V4f8Iuqf=IzMGIRL6zpo5o{qC9Fo$Vcovv3v{+IC$*vD&V76jTJDP53AmW{`$H$eSqvap4(rt#U~TkjGsO`=qZ?FQLWoE3a1c!I7HG%$-maSEN|SQK3y;Drf%aUu!LZL|bNaQ0nc87~~L!#uhH+kc8|Y~giKvn~)&G%)${7F{ZkKmZp`h`;bFJORqUHrh+!OaT|eX#I$;`N{GGT?CSwB*{2konc$6G%`uIgDif4YbV&ZRj-_)i*3BOi4ImtB3`ea)=MjNdIO!Ek$4c@Wa`yD65B=L103F=xj3o?QSuBY!enlfMGL61M-q9uw=QY0C33uj5;eLKBWp=IcZ60BSinc-vJ}S4gFXG|JV7R|@YNy$<`>>V<x`x^khvt;$UemB3`<v0u1qEaXljS~uj-YFw+q`ORwbbs6yL`eThw1;2W32a!@MAvj>!TvbAa+mQl6lzd-!w~FYMv7IRXq3RIl14o{b<dfzCtJ8>Gn<wz$M#iEqxj=xmu(rL(<@ng5xc=TOLt!$~^3Mtu{^w?IHjo-E?x7K=@=G#LJ=w+mHTO`wHE8s25A1(d1Kz#<M+>xDCxxqKU&$Jgg9y7HK=XYsj@PL^0Ah^AwtoWQ$#Wby=0PvTS+2j}Y55UytM>>OTMX2ES57$@go`kvxhe?5PQ(la!-N6ydLIf&Xx1V&-Zi|0Y|&;M^|c@6CcX;_3aq2Ox0SVS|k=w_A%^7VWcPetpsEX$`!s!CRUXm=dVOrSK_oI$j+j8bv5zfF!;P;HhiZIg>Q1LJ)yO}wjU!;jB*NhnGKUIZ5RQ36G>XflmzX%@f2vnT(N)|O}u^q*DgouMfZR~uwEh=VJnSgcoO5iG6>=xVTHgKTS&B~DPhLiQF|>;{F8|1aPkTll;{#!J$*HCZ&fj>^;c7#M$sPVUjY3I;>3;IEfLC?BEuE4&2u_$&?`(R>(lOKhGUrAaJ-6IZCXfWkGF&#=WEnwn#=6PiyUe~FBT(P9XDy?8xDK_?BLl0XVw9aHZFt<EtJ$!i64HA9O2dSMfvrO8Yc18Sn%$Tx`!>o{D)lTo}Aq_r5@F0jH3+KJ(tG_Lw-LBjWC7F}e}_7Yo}Ac0eS=0nRqR6Am&ag<&~Q7^3>QLvYiF}4B%dXH_C>cs*&NwL+nx4|??F4s%vblk_1RTkeM#|tRoMIblMGI(j3Ozz^+0Vzl6)*0HJK)|2VM|8Ye&%^4Cp{Y1qon?zP>W$N#Fuw7=D^J&JTlnG#&%;pnN&1+~t>E+`g}PIK_Z@mEkWG#wII@N}E?D6LAA=>Gpi?;(TfsLo__9z3@}Ahj(G-r)kyMn#m)Q7ry?l-r*YR1F6+-mfj}kz?`!u?OujlC55%Gs<Erk{jS)f91mQiT_ZD<okw&>*+0SRBu5nqH&hv*)tO4Dd1OLofmd`i0a&-Ajuff2jA_qmf_ramIw7wCF9JT&6rH&hP4u{hk8!C~!f>|co64*%d7dHMU{pANZQgU;V6)#b|)mByq5G9Kq+e%-X?wafF|;;{5M+)n#|xOum;<56=3lRN2OLtFOIVY}Po9Co)`pu;a+gVJBenE31VI7U3u<=1`AUdJ1y=qu+)55Fz@jdbJUE5|_hBi&`^BIU@4UDtx8Ti;@HanT&V0aAW!E&&~szQcpbSI3zA)iGxJRXYm#26D}trhHm%565V?<xz*>Q}g(tRP#&iy}$}B$haRgPYIx8KQU>b`Bqwfd#U9It$PF=e{2~Zffv^PuRiD<9T^d+&|s*KE#C}&%Wq@t<!=T?9=;qM@VJHtTtI7<Z?vQT_NxzU`b%;xKQ^CUV+J#hIIDJM9_r`WX8EWEojia>KGYXB&A6mY&%lW3MW5Q&CMljcp;-qtz0!4p_s-gia$41CZUD&C3JVBUq|F39TB`mKsO*P4jhs2kpomDYXMXK`CsZFC?&uwITRj|o!yUse<MpTaN1B#l^g0FxREE)lXtl|@RWcuJH)Nf-lgo9PusQeWFz1<W`R-+BpT*^Iirk3&*5%%z!G71X(Gf?_fYb8I<q%iLy6?pxJH88t?3Wm1zr-K=753Qsrgy-}ugZPqd43P(s7VuPAh~WS6<@zJ+l^n2G0iV|@yNe*ZQK4b4&1LY;C@s7+b^@<eiQE7FEihM6W*)qBTs{>{cS8e`qD1Gi>up*`EU2=OY4(IPl3e6&A`ndZ8kH`-Tzl&-2JY>KKP)1Q`UlxZ8v~D(&zu`9Z>TkHiq`3^YK$NUjAdijd?0NvA`O=Yf7xc+y1cI()s=pNAyVHMNMPXW_uFZh!vKk8I1Vz^nX|x)(?f0CH@`$)3hp~>KpOr{&G>qk73;Xs&o`9MlPgaS6+Ye+m@12sihH^`kJjI0B^nDl>hht1u|39U6WO67-Fl+?;%6`RkGy_NrA5yq~_Rd6#uIU)mu^eSCXifTSTgqC`A>f*4qRvZ|dLq#J?h+__xO?TCL6ZSe3W2kipxT%y(U-CT2YUS7*LUe+FDVyn_5rWNc7>=?r|4KB#wSV8Gccv{t#k#nIEN0U*)j$WX7-?G~9n+?JP0WzX{wr$glXkPCk)UdUhnH0m66DqsGI|Byep26?_6rN0)JyMJ_0bV2_9n_qsReEa1~$1`WU^yg<Ij@R<z=R<=c6M`r%@X`Tz=@5A7P;()u1Cn1ztcf$0fuU!1&zQ&VaNAukpV>JWyHd*f`gyx$te0on{My<5LsdamzQfY)bB+niOUf3;m0jhdJiXtyyyl9%NJh`qdaImY{kh1`qZS(LwMdo$FVpK7mZrfoWEmb9l2zOuxq-1>xfXO=O+}%uU$t)nSR?VQzRx@{NpFW|NU5u4f^7#A#MArS?$W1u)z{Xs+~<xFV_TZMvQD1d6N>1;$!*x0`RBvVA4}zb+ycucny1<YC2gis^9ld*^nQtyn}D?aA8p#~cUrV6MbtWSL72pDeMx0jI5v6&PDdPr&j111p~E~vJweSKgU;91uRi$gFMjgX2VmJf=jP+U&`9@JAH04JMx{=l>w`t0)Z+`Hga2a%U^eY6463mCO6l;7j5>K5PitGpM@BA!mkT(%T`$$p$@Ift{Nx`xzWnm@HktVOKU(qGE2WXKe)5QO!Pn2Z?emeP>xZ9em5%`s=x!<T2q5Bvjz12$IQc&5aXr<^8_&oa!;J8;b?`*5R_h}v(T{)y_N;4Q0I<<B;OKq9QN#z>&HvFf>l8`CEx#Xh@tj{$mcgHoj0}w!8j&RNS0DV;0Z5V(gn=*~;Y>_qjURgU44ykIzkA^t9_CL5F)7U(s9cVb5yu;WdBCp&SlW==Y3+i3gF?t<KxfyFK-+uR@Odh$S0SXvq{(X<fP&!beZjxD^`E4~J09ntduT-aSr!reO@q6++&o8(x?ts0LYf?ouskJzUkZRG`kf~2wfsgTY?c#f2^;?tDR;W^?n&oUP>w#fX}jU#6!_nxBk=ZPo?%k|s#DWBIWdviHzk)U^iFhANx?0ZR<h-RLf3BT(~L(31Mf}xt{TMh?>;GScSGJ;`{blZeT}AGIvs=d{sD)_>e1mu>koSF@eI2kfArBam*@Ft57+@OKN<p(a=1QvHtOng_C4%#_4kjup@oln28Mb*>iSV{*CWSg{f?f$>FxRCM_qmWKkWOor?2nR-p~3vJ9|It|K#IGJ%7{niCo>@s6IFd8pij(`0P`u;V&I8oFMg`qL-GTS3Kv}zyKg@@S))DJRubHnu{?LH<m)IVq6^@1kaw>EyR(;Av@cJu#-zjQ*5(ormc5qbkJiT<|zW*R>8^?ua;9v9;cwoGvsj$$OXB=H~P{#%=5fC2VfHTLLcUQSyCN_m5hUKtHQ`&nq;t#wTaqgCH@<!9#~A=%(2_yX%HaT?cefzftGJ{jXwYm;C%S$<BqQWZ!N#FNN+n-rmi4GQ$@3c#6t1#q*0QRBKZ}q(jUXR`sxGFnxBE1^8p{4R7CH<m)?nF&UU-YZSQxD@Z41%SfzHmF*CN@F@3Ijkr%oF=Bp0`93bbR!9F=fsonn72loLr;&1`w|AX_5;L^a6e?G>wLT8`lh4YQ&KS^WxPh1<)V4(?bP{uyg^G69gz{h3OJZT`~sm=P}f%LWmUO+=KmOtS6%X&c*(h^t`JV`c)wCY2SK0T4W6bXqzTn`8u6D2!F++e}+W;%EaoehGLAZSn{RR;}pn9Ef~dGC;(?9x}O^hfK_Q42}{tqm|us1d0vZ-?9t6xsU%C_v;`w28rJ@L}&85>yFR!hpTc;c?i9N1XjmSnF2K#o8_Z1XzUE?G{(xnE1!;8M5~e>JjyQOYaDGl3V&lU%q@JD)8LViX2en>-Q}qqk|Tu0YIjTI~A^u%W|yqURb3bydD~P;T*BM`nsXGj0=O}XQ(3=`lpWJQIGYB7y*=cs(by_2S4L<`+){2EfAv7z@UN07Vb~s8vqK12JPIA0+A;uQ>x6>GEi*>`-D2dDW-RLRCo`7jlOcqc?fy(-p1%;nb<%sAjk+)N!k5G2m7lJgns(LQ?+ViCH^x{swl5T%}0re|4c&ga~JS<7f&$jcEdvDmWnAV1RI2}Jx+@gWR#S4Rdh?Tc1A36r#ckb@O>o<x-#JnB|%0Wuy3i3B<`CAEZC7WyvIf^qaB+l1A<#-Ha(*vU96DDmS=#pM$c3dGV2dm$sNDsJ3c*)l;o}Y&e@F3Tn}wBN&4IV@OQuc^)G+@i^pGmz-t3HhmnHd0(JFHw8PXmL~D;NzZcf}(BQxu3r|@rQf9gZ=K5Uj5$7|<h%nrN*Sb5}nsU+W)N*J-u{im2hXl)N)4Q3?CqyN_$EHBJ9_^MiY0Y|UVX+xI(qf5=8UGS`pe1IFGwe&yU!-~QJa>33pk;sx1C%N|IRL>5cALo<w4Mebayo_=3Kg^qTH;gC>luamZ$J^?MwjU7w;fs!o@!BWX{ufCnsNLScQ6lk2*-)F^U))%kC}H>`urQI)25i<5N@lkcx$Bf=N@*v9#oOwR{MAVc~$6?v)Ctkv_DaW#jx0zbQyS>!OJImC3soA4?(fJ?2?ex@>MN4J~hP$_cwBvWz#N|Fe$_|f!!uSvwWzEZVin$%Kbno_dVUln$l_us5Tu5{43(Ena@1oJ$vGMY;ke7D`4iSRJ-A>c+}n)Z@VaBd)ksw$X`_Q(Kv<-U}a2b^=c^0-&rM6>u#-cv~yg!yH$ieaXmGVwbb~B(E9MxhFT%*9vf&|Rp);ve^u3vj*bTRSNy18*eLLny=eZfga4yA)1Emvldwxl#*-%!M(s^Qm+_)^fT%W!+DoIr=n(Dd^TWT;@9Tnq9&p0TAItRi*qFN+(91O(uw{)JI`8s72?e>C?8R5U>LpFc#F6$w-s8Q8&#i%^C;b19o<0=`tleOwD<hMAIX@SFxLgJ{+-!((i$N~%*uu3Nr|rGa$rJLr2KnmHp;VF|vZ2xs1%Q*k-#iJAm(9d({7r+zz=(P|JOD;#hvmsP8bacG4t$!wwYZ(V9ZxmRrxvWZP3FXHdL6{zXwe}3(Nmo=Wp1j8-NnIPK#oAr6OEi6k?d#T+--mbppG$@yZez2m){2w*~hiNQ4grn?j{e3p_v9a4K!5+GaWrw2DC$$@@X4=(uU3&#td~joMRsDlN+&_CtWr*B_u0Lg{s-4`PI;``2TggGy0tv$?Y{LNPBRw<z5E=SKTKucLr!TT%k?70AU>*?D&;rIX-R`Lu9g{$m{=p5Lhs`1zQ#Pl6yM=s~}ohmUDyc1HA-IXlSJSQAg)zU7bHPE5~!9i>^D}-JkyG<4?`Xz%aQ}*e>$XcYUUd<{TY%j#zCS>cEWgHX8U`+id(Y1_=pIh)A>U<S}S~zMm^O*R7gr?b_ZL8{?Q09&&T1je=ZbH4Q%RsI4h!1foSa_eE%a>OzKwMqY#FFK?*#+CioG+CCu9BRtA8Z;}3LQT;Wbcvvj1MiO`{uPdtAq+IY?l}V}apu4$;nw=!OjP}L`zGfz@@L2=$mmC>tNsdda+)BF;#$rRc+oMiWw+dYY<fl3y8kEY7HLY-|)pogJ<FjsC_RF9BO4?d+f24X(?PBkAaOkDWG0@Bkc<{34wyjwwR2^0~F+o)y5n4nO3+YhISH}Ra|G8RV_>CxHAK;sggYxF>@W4HHVI}ZGAYWg0Xd}jz3UYAAZ~t=MB|<%5jY(qPE<DM6Yk;pipei?zl}w0jZL+EQyU~|!;d*X3oE5EsyQc*abtrTBcc3$BD0%F$McQs?w+IijMc$d@yIt)T(EYSqUUC;6bA?{?s7G>UsP1mJ#VYR(ba$%M^IJhVx<>6V!rL0Q8AhmVNOXW?;f1Xal+BFCN~b{PKJ2u0aNqVr?cN68-R|Tu3wDRqVc6i1W)(_UJ3-ZOfZQ7fr2~u|=fh7P8DKd2`t1BWm{{u8<x7yuFGpXhYyf^s-*v4hr9RZOCp`&x(6U^@rWWBJdNd!t^Ii?C`w?I=yjL(|fOoNrAXn{aw>+@(&P%wIUP?NHqeBwqfJ>;U9}m|MlzHGFtk%wU5eXvpqU0v{UZ2jF{I9rc#HWj^QO|}(T)=0o{P&m=1<Zedw%TLBpSF-p2M-!$1z9OPy|Ev`H|0G;c3upy{(Q)zEL2%Qs?icsDDjve5L<`K<9rG5e{9jo(*8DSht~3qzTCH#zwNesqbd5WyeP)xMq#=>?|?_UNe5)(4wU;*U^bHe2}b`Tu~;~{@t_R-eiN)LSb|n*tI5!g^4K9mwCW(s<pn0yHNmM}t=2y<1d`I$0o{8!Y=uSH4NL9gM>aiy2>?x4{&x(k7*^K2Tvki>(i@I1fjEh*4YV%ab$|iEk<h8zP~-uK_8un~6>7Mls!IT)7O~oo9(`uE4=Ytx!xWWy;ie&v2I%8fhq718H)uzkuUyX89iX>-1S-eq%m3i&>wfs_pZ_mE`-c~wJ^Nox&KSs{@oIlKqn7a}IZp?;6nFFrXN}Th&4CXQmg2Vef^IP~)F;QS54o*Y>lly6lVbZnv`A-4I;AH$o)rE*>i8l2d|>Hn0nG5?Bf|@@G_@5ui5LoWK7QI<>+!?Rr~1GqCE$L*H7r8_^N`E8Zay=e;)i3byR*aA@c8^5_@nzzBO<cO^W4SpG9*+30f0};&?p%4z_&-Z7*W62MI|}9&8a~9g=5t1b~%6<E4m$r@b^O&Cyy@cXznYABKK8Kz|uR7jBvTR@3_wnLd!KM>*ZXN5<5S7qWi~J&XLz6uz+P%OH`2#k?<ilwlO?IAVyyR0!Bxi7E$UWCyzzt)nr>>$SKLr-sgOoS{#FKUJi}$?bW93dfp|SjoDCYf2Lr=Iy}(vg`#DC2fI_~@OHlM%T0Pmk8?m;%Wb#WgK}{ZWMW!SioAp}U7y@xdqSC3)@Nhwbz8U_-Ly=@h~SDDl>E`BpPBY($nBPW7nZhHJs^9BU;^A8SFh;+<iLax*Vp1)5wA4%$h2gib8yJz2HkqluI$cRTG{5@ijg<{?<}io{r4%Vvp{8|U#s94?RGOtY8UioZqdsU&xjRR16F|-OIx371WcMXofRjA*eawTJs7@f$xnSY9{<r%o#xm37~;B;%~WG}(=-b1`fu1_-OaP3Q;3bx?7G^IP2!MECfoSR5HPNxXNjP}Bx^I>=xqDx&`UrpH$pkD<rOWXZsljs&?uPXFqa&Fy|LzuJCo9XZI&-NyK}k4F0R-&0|+p^E*@Mo0w&oIkD~&H_C{=Y_KKzjv2>WwMq};3fK@{L!w3)48?iR6EL$^SD9cH(<y(9>O)JmV!TGZFE|4;(?>0yagHxI>i^h>IBM5DflW~UlC$Mk+Gz!bdZGSf6;L=@=GWd=TH7%!8pG{zC*{k|&DA&ve4R5r)RHg9knw`zGjxNkndfNf)S%l6CMgG%I+H|q5+)5S-g`|yv;;$3>N*P(V(Uz5u52_G!9nx6vat*CbudUNu!`POwrtZ5Q4f63%jp?s8)o$#Qt;5j+q$JEljrGA}-F|o{NL?S=TXnuw6J{+~%}@#!dGm;G?YDO8;-^D{ubjfqW)XA$ahKpGBhF`Fy#xR48SU?f<zzg_seHFf->q$Gf7;p`1f$r6w)@o9puMeWBHjy<4iz#i-MjD5FEf}XU+xY03y&SxJ1oUFZAy6X*Uk}s)=gc!_kkfoMQ?-9&`&jfTKfy-7_H(7>1;f9X*uT$&grdcDb<70cqNmcR8#LzG5CC5=+5>2iSKv!al2M!kD5M>ai_g5oJb`qJ2$5l|Ch{$)%EL9tIL74E!mAZ7)=3FVA`dlfO%l~`I9k^I9(;&jnZqIr4!7)f6%wjvE-K7mBK0v7ZXSg^3&Fh0k=1o(U0Vz!kCTX%VrXeWAGS=J~hvbZZc#aThoN-Xx|*81}^B)fxKmsj2ynjRNW8!cbf$%1mZ)pk+z%>eYgLr^TKDv)$5VE)wEXB{UDppA@7mLY3e`H?K{cUl|7>--AXz}VpUo(t0+HpxE*}k-)ehs+n!icE$%k5svyKJz$4-PZ#Ok1ba<1z+XhY7YQ&6*9>AU6!fW?P9J3H>UHaL&Xs0M@oYL0iZBM!$Kh;6Eaj%e(D8=^R=YqD~L;W7^#2ez9el6IkQpvD%hq#Dvfo5m2e$6@S_bq?2j(Hw-TK;#-7~f3bm=LeT3yyyN>Vc-zBkoF5N@y{~j|4hLoXxd$0Bs$*vqpT~jJoZeTs7s}3tDi$1p@~RpF!s{(1GDWipp%aympBb86F<BILe)nj=?vt9dCq<Ai`My9h_W8TC^j?rRb}*@&3pBSd&JQ+6P=OG~aFQ4HyQN_49wW|7py_*Z<ffg$`vls%M~@`uhDh$C$7+v|BDZMkYCy#qNuCNIfG%j=o-pIHM@9Add0}3P=u2-s2o}$^+8Cy?q#ZENPnR=7?&dENyMs`P`VuUyJ)-?7#lUFYSC8K&ehmsP*%q*V2|ItnSePhj3QD8RnZj18>Cqe(!tVGeF}<SKe5h4)+_bySklDfc8dm5%St2qs|V?Pk5-Rh*o{98R_WYAZH?de1AtP3Us6YP&<iwEk(16jFJUFND!U@u>u^A@}NZ1@w?Nsf}^=F#u5)3`z@ebaZ%M&M`jW2mKRRvFn<(Y)fQsq_Act=O_LtbO3X+Y8G#iys8GLNSShu2VGB)ygqISsMq!4^6sSB+Lv!7lMpe$%-Ds%Q_!|u<1^%NPV=yF~0*aqP!hkHv?tuddt|GWqG-#MW_x-S-5oJOYh#u(nJ@rks2<B>xq6HLnOfHY^risx(7hi_z`Uk1or!o$gN2|I#<B<Xbb|G(}{KLl1pypA5C$`@|eq>W0H0F)y9cO<!h#xAwQ}(oslCUu{;g4$Gwv|4%TW(RmG^8Pw*2~jo0%)lWOZ?+53DBzhw@tVX`hEL8I55!MZEc@>WWJ}9F<S6_&R4Epr=gVeQacE@yZPQFk1oQx*%4;+&Ux(8VFMt6hPdKU2vT$1(jOtsY=EV~XZmnf)l`*LSMCaR+P^S6deY27q?u_|($g)^rA!H}>honS4Z>yutMSp;ZGpcJyEWFa_N{391?N`=_GixC+&Y6nt@p1$21C+$_}eDWWZpQ$KaG8<W0|I_rrT=bOy$ZDPy}zyu&SOr2Kp7C8dlWg+%;E@lpfjiQRp%@m3|uhDcXwd`c5>j*rq@*1-Q;*qoLzlm3dW~aEyB#z0d9Z*s=a_qJkN&7#Y#6ir1D#aJN~@-Hje}H>*O5oV`sJ%5}}8p>798GpfhUeblw2f6*b;G#n=wxMd1e{InZD*^B_1r%>05Y=78+4Ra#AGj_?eB|VU)CN-aRZNGH5UkFypRrO|r#%5F8luc}0)O?_8m%Ft(M9iA|1z&D@Mx|ptjr~tfzGC~vHTcY~@$HFzt2eRqjp@nL&9A}b`zTy>R3=H<2D0<_QYc~5b%TICamHD#+RlSho&l0Z<fEnh!V9PJV`xzHsPW4=Jk<NVS!q^)Tfh3?*}zZ_&<L-_&tQlzns?u+J!yFwDE-bo%*Mr~B4`zOHQ(!!c8@@R9U2|xhaaA_4UDvzUMdB{fp2z4kDEKONB&AZzl929bw(WgkZC{>4DFVePS5k9zV0?zM{F~kF_k`xZC5G8mLKzIGr5`!%t48{D=u-%-&#5!f!+u@kgfHNj*@#y)4|)iW)$4elnA3yR+xx}Q92>}_9b1nQz*SWjOS6ODGYwaOt<=@^C>rR8*wo5Vidc+QXtG1ogIeFO%LBwZosg7q&!JmK*qi|daYrBRtbs}1}9;~-QB|viw_StdY$|%TJs{2TIai!qOuZR$~ARR{De+!5DwZnT_8V?_M12YqnULD`Hx$5pBpE9=iOAvY`4IK+U}u-sDm(S10I!ADmT078n|sx+Ap^r_%$BhHkxi+Bxt#(&cEBP@uP~bPxP@KPZee*E}P;f{bdep4ENA5Rdo^Yuth~3Y==jq&ezPj6qauo;+T4~?U-RA+a<i;wh*;T-$Z+EIO@$!Kvl?^*noej0nu)`zinV^>4}yYL6?Qw03E|audQ9SrjJ~8dZl!)!CzV~H(dyr^R->Wl6VknM<)n!-X>>De<MAMCTtpwsY!Tf+I532!?z+0EdxSEdb{-vwtytqv<ioB!<fP{u1M&p&1B~Es`;G;lN>%d{F+}?C+R+dyNWnUr%-XUMPK#UW&H?f<Reb9AF6G(U~LcOI(qKbwD|8qzxSZrzjzt__ay!Q)|2<%GiNlTwF+npKJ4jxsZhwnhEggxw^UXV!#n_)pNA4^SQl5OO?kGu`2__ufql(pq|$7=<(oEm7DH`&tiJwMJBo$`&}rf?bmBynA}{n8#TsuT``F0fIctB?(b3T^!t$Q#L8_;^I5D^NSe{#6TY4=I|JKq2f8dAqHbMcZ!!u-+Lfh<I{5w1{8{j15dNw*V>XzS|7&1x>{l8ly|89-wt&!G#{#T9xSDy&7yDxRvHKbt&h|ZUezkoty2%6!GLA4ma{H>W({%ve$=*FNiRZQ5jscx)MdY|gOpc-H%-cNm~QIj$(n<{o`%fh1byI8p(HcIh!9d9>UDSBtx*LR>$-?-t12<ci9C)ks2n7iAY@+P;|x_;34=#hNCce6quU;5DYJ82g~fn@o{_4t#%Z(&V42f>8?>Z_49TPq8$e-vzV$xd?_kN`i$&0&fIckr35S$q1rL7N&?<ZCTmn^2|Har3Pdvrw6@ZOtTT?|t4l&GK)65Wj8$W@d8t=)pD2?f(JjSs~Y-@`?NZ5VcC!f8u(2OEA~h_n_{to4j*M3LmFkBf!Ps6s;=MNR|YeoKT|peM_45lp9DJ8DbhySY|hm>MF?(y8JS?1@>!&i`Hu+4)On8pX22ptP)7BQgjcD*xpa4k)yZTq?=gU)WZhNr$kK85Z!Vz!W4|n{O0E0IX&VW$>^|p2SK9^?bS_Y9eC2rslPGLtk9-b1G_<K$O@<w`R4}2i|$6d%9Gg^rvRs9gaBx5d~LP0D%j6YVmJqT-$2RE<}%+tHMPy2K+m{KOX*OvuAsQy8b|q>4!gY1F+On?Qr?@<-$#3|n^`osWZZFTrn#NgH=5{_HoHs89-#c>+cZu0nrOo#^<A=>0PDUDwDyOou4Z;yh;$39`@3Eh1duu;NZKG!^^Ojadgbpt9_Jue;Wd|?sJGi#v}Zei>*V6gs@^Pe2Msh&xCG#~Lwo|$O60QI&~FqbX4OhjaS1(-`?K9P<*m}wNR2$OjA=*svp0i|m#$v7_THDKfwmy{78zJ{`a+8`h5+5}7Ux5aMV~tzeeUkgPw!K<&pGUQ-rd=CpRzpzL%lBmaI3OB?Sk9+C!Ttt+x+mXK#{s$q>Fc29gh{F!{Y(N&+&$Ll%I|t_3*>n_vtHi6|8p&6QA9`f}ovG?o(dUms~vq@)hE(>hL^0u2)XCUC~B_?b)KRmKX5r+@IYJ+o+C7tI32|Z(CYsCrkOp*WFCO#Gxtq+&Haj>ASg{bhX<(3i1+$WnfZDn&+l9U<Oand(#RqDLJUnd-D=7DXpb_&9fO?tw!B#SPdrSdmW>0$AIFb;KyV}Jwp#ioX<vhZk<<7aWC?a$K+1n{>Gva73E)2Ot;h7Cv(K+nkLJx`ISjm8%>OGeZb=A=O^AcaXoY>gk%vwTC{22Umj1>0AMy#(&7~$nn^*!3QYUlxSmOrv^2UZ>IigM|MbH4s_~`642}qu?sW|E5PBtx55FwS)&E90vFsdk^m>d9n!z&m!Qqjv9TvoDwvV8W|20@JL$4)9X3pPCk-*v1us9LW?{W_GDZYOLM$9(;RW@kO9Zg%~X-peMEQ3xDzjcP|C%<*d97tTUVb-3WoeFQb{*~DpGV)c;VSM#n(=D|oJaR;6Do|V2c?=Osf4O3*xa^G_`LCr`Hns$mdvjo|b+tELTdej$8nJcEW@<l-nO|lnT9zZ$`Et>!sb4>buDvp*8f#o0r)Z&rgZH?z?pCdKs8>~KHR7wOO7p#Q(oya%w+93#-{xtg0(-yIVZ4)2%&t!*CaIKPauoeWMbUk(-ZWDaT`!;gefP(v>KlCeLwDB?P1Qg(^VF~d#G#tt=}R@iQxBMu&tE!5UUYcIJmwdwyclj&=RZDDklq12s_ibb&Ul#yj6>wJc=q?rU;ZF{`Gc7~pbSErPHWiFbXf=QAYb%Kr<*#U&RMr^%B{^U+01U_u*0}jn^bEi@SQ##44bW^ceKx;$AQ>-^(Y%-ipR^*XH`3IYgPE07tS{wa!8T&s}Cg0utWYMrEgGw*d9v<zJyGJdhPO0ej!i;s5sy5w%Wcm>|V8jr43d#FVbeH^0<{g*}#h!kG%oHs;T}&3;OBp_xZQLIjkR^ZsIrc4d7O@Vz$4mm(5gbtZaHOhn55I1Zz5mperfoty8WaGC8X%GXhD?7HnU)$E2-qfT&IGtX9?<(+%F*A)HRbLj(MFWld>0znfKEu<eZ%s&yhtw_J8>8&XbbK6<Kks){p7TGL(HA?#Fq-0DJUur%($>~}gq_=xa${hc-P&RR1>4Q5PsxD8j$YswmTXT@8)N2K0qYQ@lN%ZK8U{LZE;(?<t9IsxCHA3d@gt+fkW2?75~8GW}N7oZ)0MWJk6>K&kpjFE0khvuKO@t4LO<Uvpyb#uZ`2+5(C^6}XQtoAM&=nXb;{kp!jp`Is?+l6QS{!ku0g*p$LVQD%eZVr~vpmbEKqaM=iwAq^Vm?nx*Fu-xy_EdLnR#95aa#YG6hk9Bju+(I1H11Xzu=~U$|H2>dLw_<K7Grlkee%brJdZ@PfZ^vj9-F%ilwK&=1RJ1i%@tZ)#V?llh|}@Hu&1L#pPH^vlc>+Cv@T9->sL#Yy3NzX@oSyrHLc5sXo5731r@v*-gNLDbW4OPMs7L`d7_nJXlv_FW;CvyyPNZfD}kgkU%=qy^o8$7M=}jbS~TC+;kV6L8yc%sKMyd(b`x#M*53B@^M}2I#*I16aGto@E&P;yBi?n`A>2N8sgQ8Ci@cix=gm964N%6gsaAj_;U)$EBYA(xroiRGMx4DE(<~-_uZqYLC`0GDLbbVRoxLyYvPId}W|-hs>s<k6+~$|bp-KFUMB`RhbFK`j#@}`~<q2=<o8|`y7!y|j@2^6(QVQOCCD_o_`}9X2|IpTq1KmYe;-#%As!heO<!?2!)^vARpIree{yur!DEstQ^-o*%^A7kWb);R~B&#I=o2=Y+8wMkjrwYf^er+i5I@?uu_xkPoEy7ZCjPa9D!#ugWl;zN=ZlK83{*IUUjaoqUt;zrfgcsJXI$-SYwOymNH-E?PNf>hViPoT*dl{VE+bRW5NU;bYJjLB3lJD{po*(M{cJE$_>*!vZT<NA<=O$@q>YL^}n>0O8i|gLrch^U_LA1vZB$y9O8ZrqQX^!3(#}lO?s-ykrBjp}kLC!CG1Y42=R{73{xcS4XR@GleA(e03bS^zPB<Y~kF*(xv5Gpo-83T1m3MPdT@al<_vq5`pincwWZ@~-D+y||&miQyT1lT4gZFp#Mg9?<<lH7f^5bi^T>ZXFJ2c)r;VEVb`d*Svs`=U7hO+RXV`<52h!)k(eOiD-&-EPZgt-32*)ykErW@V%u0p+qqbEFJ9?Jr&_kXo11IAL!q*57aMeLgz)LR@Fts>HK_(Yt^s9({h%t7)0$ZH7C_-dU;#SZJSnc*vt@1)VJ>-MEKf5tp)<UdL<ZZ0C@>K5KzwzB(Gb-+jlFG<mqCa8<LbeqX$luq7hN#0BcMmIXO!^2Cj|^i;^1KD!WG18;8aCZJXJ^}D6W8uzUvD|@5ATP?$OsFsN<7ddLyrR4`LrpP8=w9^KKfklgO8K-jJ^UG1UxcsQk^~xnJh@LlM3q%57k@9?ahIoF~-cjxv(d^^A;3uiwmfnF;e)7BRJw&3m-`!g7(sbSIu+V)C8$du)_Eu80@lAd*N?n=Hap#%**yP|6DN&4>jWqmNhTQBKd(RygC627Ah?(<`?+2>sf1qcQoUzJ%dC+xBw>WF&`j}l3bxlf<T(Mox$D1Z?U|4GTcY=sK{H&+J+Bfz}?Gp&!PmIxC9MVv~XBf!GYVu7Fi389=Q>XEPL>#Sd!jp_ZMIn_D&`SMmbX%$+PO5b4RvD-^=F6<T)$(iMKh<rmYdw;ZV(<kIVr*?BE5o-vzCENtE~fWO7#gwN>SB<vYhpUJXzB@fPRY%gdr?{|NG&R;3QcFbUK3FymbA;Q)gqL~_SO(peL{L@8IEL71k~sDhV+*n+HVEW5ZR3%x709I3GNk+&_Yqr{%qH4K(}sNx6~-~FK(%6s$?|3yahr_-J``)Yz^tVZKRtj9*rOEMnyyW^Y$6i>mB+Rw;{w#p=f@28};9&nza6C!p#PyTfR8Q=Ggc({Ir@R{FA8AL_#zy8JltmL$m5AayUWdU-ogP;dx{ItH!l9b|y4}(yHBjR7-#m)PLYL6sas<ov1nOu3S2*j#j6Gv}?2+9b(6H8UA!(2=~<)wBPm6?ue1l7{X(EyqVd+y52Ta6wzhv1+wC;MA^at@u7)ihDwsNcue}El}yhqCRfnt!0H;5lRFDE>$5wz5L-qG$J=`jz0@^fA}HG}I^UGGB>~t>az59mudwoV8gwL`e}&Hf@oCdx4teuaJ5tp(%-5mS<aDpH`oy=EZa)#gaPP3{`({<-P9Hvq;WxMBZj}8yqn;s=O+haA&yEon&);x&hQ_Z3zi)B$4Gnu>4RV0|w~|TfnalHhl)G?Ve)RchpJVW!f7yuf01yv(eqpEE`H&;`A-@Fj*`QDg0Vz)BM=yCcAQ?+SP5H;4{`Mmd{>$eM&;Y*IaQggY$m0Q(#M%2o@7uZCyVLASY{0rRHk@ySbV_WM-@xD1r?pZvkDbPfJ%e(!pYYul&a<_3o86B>&4J>-b$;=CtEt`ds_P#8_*ig6liqaQ-kW>Iv%6h=qY{5<ajCPwmyVrkJ_1hoP08jTISS;jEg^Tct_~wKt^HxQ+U@%{t3S}HlAy!w>ir4lIrnuh4mbw+7P|5m)Tcq|G?@uas=&V+O&Xed;T#!s%5mo(^E7NDu0AKfdRE@b=)NDpO5pCGR?&K|(-<Vsf^PpBJ?*-0Pa7ELCX4#x|EKh>XfB#je@}9QHRV-T2WF^XRX+W<Tu8rlx_|BT+>?)UHQzNTYZ~oRn77;PtPh82ws!7ZJxU9oe)3maH=llDwq`o-xoC!mhUCz*UpdBp13v(wwTnlzv|FUV_Yn|$IKm<OS6X2_+quc0x>^s)c%K5Fx=#DOMrTq{y|;m96O;@?t4tJPm^>s^CXXkNo|?npCg>|2mP9S_PYcm0SL_*V#t2kvQTlccX;!D}yW*wm7Q8eDC*!5l94}qp6)#=yg_o}HfR~TID_%an3ok$H>{9X4WsaARoADxQiGOax%g3g8fod&Ezdv3+`L1~R<QBX%1}Ed?V{^QGVsB<QqFU#pu1{{qj9k?eKfLy>mF|NkAwo1mlkbR;2H319HH=f8990T(0V+KiM5s=)xXooRJKI}oY6`O2O{<xxIPI3(1DSO_(+9Gut~G7ck@QmWC<}a8Z`EB}JIy<__|!)MHMh~a1<8G6rby@!{XkX*-Xk!hh0->rRoP5G2>EUPi7!N>J4Ua?wP>oVbyHof0BVG&d2Ok{ZQsOVy0t*h;H{s9s&`xey6);<*KObK>R;FG{p-5Be;?n~zmISGc31yCzP*1RoAvMehC-j*)x%G2`)0p|><nD^9)(Q0+c>7BrY;{4m&r><LfS3AHl#e)?}$C&4y%qnr*oJ;$dzn@o7fxN<kz;RHm%v9*-zvtuA0JpT8EoSVesu1b$3RN3he%oX{cFqLwDNbjcp))U_|<19qyK2>u@*hG(_~uol*O#y?aIOiCV*<UZW$|vUQ9Ox`87)MZT-%LkNAYS9WphV#|Q9M;ZimHSW=U(0Xe_{P)nR4>;=(A!o`Ev}17c2(d?xA9po^_j&M_I<^ODC!1<@>T2;gvAb&V%^{}-Cy(yT&N%FB10m#mzwo{fRrjgn@0PmzcuYw=_VkvWA&V}#t$Ht>nO9k+A9?lZlAiy))6wVbQ#X0<<&t{dycW72b#%6i3<mt%cOxRSIV47tCfm^VJ<?>$&02oI-RZh4B+@wTd<#V>^wCEa<2gIATVPZT#XgtVPLr5VWACdmN|_$pElrvw-_&f<knXpb4VJeNcVdIj`0~zLc4<7iJ-p|V8r_O8fyPJPv};b>wOiEfw|0wT!1ZjfX)BIj+|75aG}*Zg%9^~P#sBD-izX48I6@%Ky@)O}a*;I3H1*9StJV`P;iN4+1*9gApmO^)5Qk0LP=duTluYBr%R+L%Y>#o_@wIE%s-+LGhMM|fv+0iLjI}q;qJ!-)tn3tvh1Ckl{Gawan54>u<;A8YFJ*$nzfH=@m~q6KOI3k88!_1$iAvxx$D*0Du$d4I<RBk75y2i7H;3D$#Sh1qbmOeydahOo6&3WS#JLTmAK>k<ZeKxGU2&VXEy>vGYSosO^~%)4qiujl4FuBAd)%WvQplgQgDY+cQkI(tn-HjxG6u@Qexis}!cbdE#_bt#jq(iAf~PfLKp$Ro1Dcc+;G?cV&u7X(8|9*5>GC^%@<nU|S`7`@Wag{%{%$IlScyWvW{gE<PJ)ALxWU5)<8t>7jcBqCf!^?PbU@RhhSVQ+aoV*t6FBG=Lf;VC8zfr3o13W(OAj5qsI$-B?;04eO86Fmo07`Y_RuxhuO?JC7}moM<&N56yDT_9R}UUa&^3GaYTpz|V%|L!lp^M(7iLo`<4h``FtK(Sf-X2-4H3DYZrnKNDK1TNKK!&<=BqI8zDmn3JB5r_XbD<Y+G=u4^|Y!cDabABNoO_0L5h3nZakb;@K*jH5^x$aF{;o#eNs{dPv33DbQgTV7#>)xV>|;7Ko{pPIG7LBuG*Am7_OOJ^V;Qhe(rSkeJ&5+eT^G?f4@$xYi-yVTDo&uU)wu%YKgAR=E7Zu!u|bbwoPkQ%V;&d$2DuvxK*b)e=coIDZ>C(F5lm#VE-4+K^{#qH1fXqel^tV=xN5W3hz#=4v!4|(HyN3=zGQpya#r8iw0Xl4b64m@8s4Uqi=w_bNB<JO%dP%Xlq(gFJ+LQ?uh{<cuZ*(FaRsHAx*TvO%ESQDM;n}RT^PMEkXgl`hfpWQ_dm^O0Vxe5+{e+_A7_5<h9U*X=6w3=FO6<F*P2acWauKQPHh!zm@UROv9p5bA$^Ip6WD-V${eTBk0%t{ZOG5cNBFcESa^{^<Hi9(Y-ehk~Ab!53?Ap&b6jSk#WG*#%ffjhQ|VqHK6h>Ok1<p;rZ<l0cF+(3Iwye6^7Q2Ctpb^C9%2}Gi)YGx%NG_;YC!XNNObql((SEPa5<nZ7yjAoDt^efVAnO5L;vR6&0de-eYR6`ImI$76;00UF>?FE)L(i8l*0DJ^JLcTU-429xXOg`u|JYoAykzZuebZB?P1#;{!xV2yldf@8LuTgF|?c*bqCIDX6Nut9q!Os^@v0=ecWcX>S|Lv1A{A1Kdxo>DBFzz_o{Syobk;C4}@yd*An6U3Cp>{nzi`b=6hhyt4AUyM%miZSOa$SHJTh{a3&F_T4Do-FxGg3uS-cn11m2&uwG$tKCh$I`%Wa{><$E<l8^~`sc#!+duyF@BHw;zcTi}_RSK4@4am6zyJCRv_JS}`S!1d;)mY%yMO!Pp<kQyZ_lJk&>#HTk9JG@8;t26{O-T{?QfQ*{`_n6_0!K=;vaYLyO;mrPk+5q`)@U*|KP8O^oPy){A7J5{C+SiDyF~v&1diS{jw9EvEDx_#(ztC`?}@7^UcPf|IV8Fw~(;!o=bK2yFdQ;k-2XO=jUG?q>lX#b$xx`{72{eQ8D~y-%0-O+yMEdBz=9B?<vyvZgKpuJ-_qKn+kv9S4*+JUIO*i>Y%Uh&Hl!BS0jFNll3=#^B?~7AN}+n|MVaJ(;si2^z9n^U!L=qCFF-k{PSOu@h?Z=uhIJ(`TE=_{7stl?NbNeJVo)faD4m2*Q;88u&2>CUt4^=lhL<No~n@gDwuxoCC0a|{d*~_tovuaBmF^^e|J=W^5Z4s>YU$c=-+V4Z-f3b8^C|FsnE~9pT6He?BD!o$DzO865=;L7lc3g>%sc|{id(;>_;E|)dn-)IP&jb;oBEZ{c_8lAH~OCU+U*yy%zK5-?;<!)l(3^`Teu~^vA9Er{Dj|_s8e^8}#tonEc7-KJdMA_5-E;<!g1nFJ<44-_M-%$8-LBzyG(N<gfS6`q}3i@R@I!|DNilI{16N{^|es$ob52lHdQ`-~X+D@txZK&HcN7{V#v=orwOm|M<6m>-T@_$2)fY-p{}Lz2CQ?^PTg3F6F;|GUeyL^?N`0%s_m7nIG=9_w76RKC_S0*W1+ma>KLlKRWW=#(!?tnqPMJoB#Pt_rBM~-~74}Kl`RBzy5cA*}1Q`Ir>aDe)HRg{r#W){qG(9nMXjs?cDdz{5QN1@81zrziR3K-=Xxi0sgu|fA2w<&usqrcYpSG(Yc5*z>>y|jP!I^=3;j&{RSS}eA#ivq4wa*71Zl+@~C>C?vgV*zU`#@giAvF0dy0R$Z=vqx{V!|9_=`4!eNe+63^0jnc`|7^#`4{w6|q_q@xbU7p9+b8PakMnGRM5aS-xs46hQO7MRn(-OBkz#fBIgS(KnOjhzNmx}UTr+=^rsF_)A26wHw{pIp1+b%gUHHO`WmqPL`K4)SBH7;<iMrHzRl={wGzWqrf$m2Z34uW4#Q+Lq=WavAbd*lnX`(x?Efk~C~duCP>sy2+;sWPE5=xVVSh7_>d#^U^(XX!31=rw+^-v6G^H8q5QkHL8{HF{VbJiUEYGog)hNFs<^gOWP@w4>G^eF%6DQLG4{LFyD}<0Fg!B7@Q}+oVT`wO$YP`HJ4Hv!fD0T9k~Ok_V_qQr_bd)&IdTPcvPp7$LkT@M&ef}dxVN6=_!`Wu&+x?qv(i7J@zd+o;h4nu!d3=GAr&cuu>*9+kTEOIh=*koIu4FdkWJT-^X0PO6|(W3nyMu-_yvVM3JXAUiE2U)4~Te8llfa8!tPs@6cX}UXof1^&B-_d@Q6t!0Jvads$B;S;S2DZMHZ{e}ris?9ubkT1&M+eNE<1DqLWjkXiNJPeI4w+>xy<AM4WfA!ADCMEV1>I}X<z1W>ra%!AAmx)D!wSXIeBad{@yDLz}Am|^<_m*)S5Q;$3sj1?^okj~58k*+1<7Ov0GUqdd5mpwgqyb3TsgyT@gPhPM2-hxdROCeQ~6wN+@%)3$IksCW6T$fT=v1wz*=2}1_P1+~60t&{I@yIu@5x~;theiD<HFBKE;v*|H7iL?^w%M{}?!mc4(He_w$X(bvi{?YJgu#^(6*3m$taz~H+w;GX`I+`*xO5<2##)KBG+kcMzENw8g)t>+P~TxN=XMwG8_+wfIh2@EI!Bi!9xI@2|L+xt7no<DyQIqudv$IN$nKEcrEy;RU39l_J>q=<jXq=x{In=@iG7dGI$GI3uDx@0m^ZNN#Nm<?4bJyt|HAHr^AFB7C6S<X53Lt(#?VNTRbkcX$c1rECTVEB$RAR912c=08ZSb~t<YM~&Bci+caPl)o-R}laQs65g270#6L_RCt3$!~2ztD{@~npWG5Hzx3NRnZx<l;=wwrw0$fgO!1lQF`*3xfFU5S=K%^VK*G%d4kk{<DSi?<M(Pgu=q<xA}(dK(T~N*(E0!NwRT3*0TCdy2!MObl5V(6A^yq-K^Lp>zwdEMPqYrNT$3VM4{o7Mu(?%h*4W>GD;h?T}X+$T*NKQa6H136chmHr&fXeE`k@&Leu9S!<)$KyM>0{jE*-UZ#!*lOBu=349o(p+68KkIw^bbJR_8t|Hn(+-)3o=zbvm5c_=|ukc!Zw-eY{!tzeDEe<x^-NSQ^?b}DN5j~9E5#J|qG!iXAbw~7pRF<?}OMOep5W*&_$$y<7?9=4|N|U_^F59=+ltmSLJ4*II&*F0nnOy92=@dbKjDswc64J5Ic*0_ZnlI@V=?;`0XkB@I;qZ*wBKBXjZBwlSyHgz4vN5Q?h}|j;im+5&*rWb{?3@h?f+NhH=tQyCqSZ+B8Wd(S=}~4Q=`8k7bR9wJ#m_N^$9HxX`&Cq7Jz79DCF3P(Bfcc)vHxUDV{Zqi8)haSVH;0ZD4k+&%cV1Q&JdM3Q|5grW}ftX?4DWm@!p2&f)jIGwQ--tc9lB`xX!+~Ks7^|C-2gzuW%K`j*YJ!+{<Xr@HPGjZAm2mCm5e$W&gtCdqTn~^jc7I>2l|2|8DQ#k@`g_+5|NforAy7BypC<krQh^4zax9^aig7$~ItDqo#pRpSvYIYS1ss+JRGB3Ks26?2c%-$8{=x*>VXX6LKMq=>?2yG>l}DfaVpf3NCZJKgqf&`z$P1klvB4F4X75MKUjOub|a9J!N}AdW*s)gbg-3FuK1RPrQt=euw@LRnb{2p_t{eLmORsRWLV{9K!4R5v2dp-BYJQy`gmLoWJ0-B8eSdO3>f%JSFR#bQ_`#(F*yX;aJ1Nm9;0iA#IQFvOlE?GM};^emF0_Utw^UN(g0(FKd2oI9iF(#Foa*7cW!j^(pM&Fd4fJReE5g#K@7UKBcli1^g2VHZ)prXi-m<`%r=+I(a(WAefV#f^L_>F}qnAwP;tDoiAFMUn5!#xcU-1#V3msq0~~bzu{(`wj(a8>eiL5FVi93r=WSXI!otFt?d^FK9^-;%Wwhn7ETZN=*n&N5e_()K|>e62H7R5D7LTIShGLGK}ALhncpFCr$qlh!f_RcRSYg{>~WChYfrQX7AJfv)1gW8Bknv=MM>|;eE09e=?Tj!-p;t6!>udVAqPoZCGc>QRyGc&arAgMRjefN+LU^aYC}03sI(Th2f>mDg*ZCK(F^k%c%5PId<09%-eLWq?Sj)9)O_fy_}Ieq0aSI<O5pg^(=h#lNT*_o5(k*9sAF<@g{M#wRjl=~reSiA3t#F9P9-qyU`Y|o0JD3_u4Q3j@&uj(3zbM?SnG7TV)BZX35yC1n{3!HzPwvU>fZ3Q!9foP6SB{I+CZU+^#x>KJRCtyWmt{J4KDV?3-M_|z9(N%<|Z~WxHwbkfVGFHkjggRN@ZRH&~LoRb5Uu1h1-L7qrevfoFr5#tYS_P)uYeHBKb`*H*uv-*$}1=D4N`@LAnGl6U*KwO~+0X&7CysSj<qV%(tnue0r{7rDiLlbcS|G$R_bL`I8!1E*^4ldQoNYZfe-p;oihr4IHH&WAGDnT;rpK<sAe%cscBkNguF6HI=6-%bp5xBOKe%AH5qFX?NlE!bw%OI;}#U*6=<8Bj9X{3r8OBaG&OU5nFSaG_YMk#fRPJmm~5X&@4hG3DFjE`(FfP&ZVQu>W(sdE?GFAesWA~=0$(Sp-;^%Sxsp8{3^5h#I?=!=pRVCkGmA~iyS4Xa>B%o!|iXu?unU}m=o-k@s)sTg<3-@+|j+@X(`hqE_=}E@i`+~<2}$IIK!nY{f=k@%()+7hNoAkrl~rkNS9p)7CE|&MSF7k`d5q5i)!PKz{a~ljXY>$T$j-b{_FzBD%}j}PWb6dZjXCM_9-&z;Pj|5hA2ykI*x*OttU16Q{pLu7w;ERe_`^-`iavfr7Pqm@S+MwN#+kKm&NRocY(YqeOvTB<d2wn@xtI{8B3L4JU|QZZo}j%dz~CrNg{5aX;zM%IaU-<bY*#_{+3=YJ9o%v(%Vy`L0Tj;g+ex32b@>g^kkUFOB<Hf+c?ohV{^o>G=?4-S@4uOIK{86v~2LR_|kvjLv~Ev1TK1VIZI>Ack3g#LFDo*2YLxNKDLUS%JN~xwI@suoaxDAi7J&A9bQ;CIOC>FDV4vfoFAduq+A8|9Z5974ybm8LXtCi^sA6kaO2WS<Ls3-eaZEpv_Qv@sEg~(N0^7oRSb*UB{8zx?BjTj`Zf0F@7{@SsgIydn+0TAoDSah0{k0ktN7KVLydMeop+oYi(Z96S2km*<*{CZ-Hd`1L{HLGpkFX5!(uI?6D3+u$jQ}^p@o5o`4#mZSkF_X_%^O^D#7WD=iZ-d6s)+L`%AY=>3!@aV3goU>3<8WMK-T8F2Qxdm%Qwjcz;rJg5?Kwid3;a>3O-8;b2keLfVpF>f;Tmg|QMsBQKthuR6Ar-SaR?v7Ub`vl+JN+Q3!oBX~fj4a$r-b9yZ#>abOjSsrg~&{}`4=s@9*#m*G<{YTKG;g*}(kFZAhCl|^vwd3#s{jPM|lsIBB3xP(dvnaCY-Jmnz;+E1rFEh}*;8o|LvJeZb_vu=uMu#>kpK2T}FmdE!3s1^91$?_me-37bj3w(KmFDqlLyZ~a24GES7f^SGiea}MY(B{g$-m245s)7{p}1M-DIPy#--cOUE*aQu{t;aQu-o7|oZQB~Dxf`ix;(6WggH)av1MXyiweWW1{Cg4U$W!!=ptGmmjaFMwBEwuMC)7Zr)83-bO#P4KHf#;lyXaFUGzIe)Bl}QEgnqpT$6qRmoLmHJJ*nFDt0DPRdSic(iBP?+Pb`RVVi|Z{@pC{DF?G26_0owW4i>oGxpjP9KLu$Ziv$addI)?3y{)iSs*pyriNhvO7+WoRyfq0k~@a9`BuJokQ7fvu?Nux&I7PDC=WiI`(G46TTuBS2@l;QlpEYlOI=~hlvIFht{kPn*oksMRRb1%yhW5bv2no4McjKF&GC_zwT}%o$QDm3v3K~C+(>f4ZkqHtlrFrk#DNFtER=&U*PsY#K7wkW&ZA$vK$UoHD6W!>!n;;}Hy*KHki}K>H7#e*8B$&8-VnknCSC5fWgYOTAdcb*3pGPRi^CNM{r^g`23kcJjb&b;;`%2LrO~wEH4<loL4zh!3~E>z$Z*ZM1uT--+w*in;S-CB-}mSqq11j??RRDf-5hP|TnS~OOC^glL#joX7_>E5Us0)zn<m%S+|AN*N`oOL@?5RPP8ppX-Yy@r9T_=3T)=C9!iDh)&WirVcvVb2MO&xIm2WwiC-5|-q)y2ec9tA2WO~6?O|%Zx*1SGTRn@W??M$c^rRvBg{}J|B6|+$jn<ktswyIE=px=>s0LEU7gG@uVTzKASsYbz$l_@lOA7MjACR=k@g^-x=`U)!k5)CNkdF0bdh24%ydm1W<uCSj2WsNr%nrGhGIp%H3Z!ucpB_XvAM;9)Ja7(_83%Mnk)>#>jl*iWvKDJmbya(gh>w%S~@ea%@9!oUX(DU@kRQ0(-dy8*#*f(J4pugi%l_omQPTcFl^UOydM=p;eysTwhfJ#EVhz27*R5>zmcH?6cyc%`SP}$<sLw!p7JQdXJEZ&1OD3P5N;FN^cTB>6{JwL(%Ty%Qc&?~TYgQW}U9yFCD%5p~G=uC_<U6uc6#jhj14DPjf?aE|=$su31a&e%!fl8h-4z|)@<|N@z^9AcVW(wG-(_zoqBb79o)}f?O<BYco9P;?sV0#ezJ2dj#8%z7ngADi6=pJOF65+(l1r{~fH7MJHW{<aNtlzNKLbrv^9#(6aEU3N{qeiJV9!!XAI$3czg5{3eA?H-(JMnEI>6}z2P&rcbjMfA@8S)C!D^YPsK}Bk}INZf<1nGs$PqZ2GbWB&DMiD8%O=m<;kgemW2(1bAdobPcJp=uK8kcuA1N$m8l$+V`EP<Ju=qp$*;IfY0E2WE|9GC)}>5&>++V0_;VC66difhx*-SOaxrX|~rG*|x|Z+Y3a;c)vupz7iEgm+W)1Njx~sSq1O#Sndp^(nM2R4)Ba{BopIkY>b22D*iR#gna!bS~#0>GLVc#tWSpo^>i^(4SCah)QMJJs9TWu*zBm&x5z}<JFpYL%dYrHl$LCuTzOqn7hBV1{|7bd$d_|#(=#7o-);3X**od@LH#CnRoqns=>zsUvso7vbUu{{#~8YXfLS*cz3BUKEe_!oKFOhc}Vikopl@*Ie(^!DeaINO}Ja&8lsFvHOtcywKSO?-t9TgUA*Q&Ptss1%`sS>C})>iVBrAG1KksGXo|n$Qd<0^B&K{as9cD>4Aotn?qI#()0_)6Xt?NjTnVtC2CFCS3|M72=P;3#u9C_bA4YJwN^J)IgU@+3YEYX<KP8Q|%qo~`z+wRVP3-wlN#S|+Yp5`Bq{lCh-4}KnSk|~OgK`CCXE?n6a4XFdUHc!3L(;j}ZDYF(Wd%(mRA@8vVba7-3c?dERy<#@c7<S#!>*hznCOBxz)Io|71W2U)Uq8>cl=i1*PpRbhRZp23Se8bwCUiGHNsX2TX$}h*se%>EoX;ro!Fa5GlY5)D|;$uKY}MNZtsB(W96C@T`6augu;mqXB;#?<)&~}DAJc*lC8m~WC<TjspP<}e}r{@onTmmswUMJboTU^%B=N;&xIs4j#9aiS>m(@*Nt>dxRkLl<d*XhDy`acP{niO7ZEzOPg+eX9tJw)yOOl2cH-0U4-2sD;aQX2863rkX$lW<*pqCYZw+p&@KK}54fO(_vlJ=Yq>*hve??iPs4CFbQr_}dX|+j<1<qgO&7`-a!vbEF_rQd4fr1GI8*D8>4OeYP{sZQdcczWXaVv;hyRevxe#1<J)e)F=T3K)H6o)04A7W>~(-3oY%D8Z;NpeDFj%#zM>@eriq|3zsk4@T*-u)Tg*L+&hxIx_u6-QFdkW!n~kUMIOs-md(zJefw^)_8pi4IUxu4oOpwX|}4G^ufCC6$#KY-_MmhT{nvMZi6Hu4%4|Z^w=U<q`~skS+7<L5Z6ba@?GQQGrT1_E+zLBEx}Pl!w1_t_1c?<{K`k_<E7`fLa969(9X%+B>}j1%BQZDtkz~(wK5}$P)_^K1^M>Ds$V1VNyB@gr_uVigS#^q_obcrp~LdXJB5{4kfdBJk>vfLF}whC12EnbB2<8NtF2I$+bkimCO%xY*SJ>yk_j&J_3tJkB=bt5$xr@L))e1Nfu|B<e=GslhWiD7jj=Z(0@|hkgW3N3M)2{Xj8F@i90Fv+w7#WmQ@KoNBkqV(tI3?o06&G>^%8ftUHj&fYW-{lF)AQXu?4jN5+Q&ps~U2j^qy1Nl^Yp<%w7o^m;hXgEOT?0S_v$M_4)0WR2rS{90r7!JGOY6?ikCg&s#GJkPn;6W0ZAkJb<uO9`r=QgNA$8+}w9`cM+2!TyB39NH~NT;tacl6Nt;Qm^2&1MLmyPrNnJnR2&?{uA{bw)^l{vtL9z2dk9mI%E%08FCO|!C?D=tqANDq{g&q(P2rc9KX7l>`?Vds}6Nbc+tT=)7+E%g!@G})H!h`ttI6wXf`ZNN!+DvNv+M1$qz>=eH>M}q`ciL_G;jmG%v?ti&c~qS~4j^cOcyX)+dnAAUu<=e8>U(seJX_osL?A(g7XEeBSU?mmne~)cGQm=4eHbchTrj?un-{d3$)ZX;7fmlRXbNHCirdqAt;aaL4NfST7vb$Q;68iB<w%5mkn$>Z>)tyQ<9@s5l=%UM5`%6MQr%%JWjE=ps`^Djn+2{}W1Pu+gLRQZx&dLTsAwGHB$`_y7-&UI$jht7iGS2dgf<#=GZ}HHP94y{u$sP+gIi=4$}D4~q;R6Fk&tRDpbi$}lur<acP4K;w?tEoUE}Qg*E3eZOOK1p|YX*LdDB^OVg94^_Dqz)awNiIK*Q6R6ZFt1xd%?f&<rJr?84&wT96#Ggy5jvEKuiwu2n_wV)?FI&u%u#u+7{**dHukx-XV4mjJj3UK^5uP{bjAgi^xrV#3ENW1)ur<cvjXU|@$7v`#lWX={yKwSKQOM;M%$Imb@?kBT9JIPv^}v3jmqh=D{VXf2ys7zJyiJ>*CNCOTZ{f0x!x#0kygkR!LUIP4LzpDd$jDk9zy6Cn>y4jNRXGIrpZpUjFZt%7v4T@q(k0v|>`YO^fkA>c2h_%R_u`jwdjY3?v6VIHyweu=RjEI5R+Ns5qctZxylUa)%<BQ(EVR=&IKnG0t+KT9($&DJqnV(Q2emU?Y+MdSbK%gZ`9RzOR?mMm>bpOaWQi^cbRLk1a1_3K16inKut-%iH$&4WN2<&#`KiRN1zR11Iuwp{o<em_n$5KhR(AYa!}&$o1(em5FR`!xafP)cZ7uGeIJbwB3yI9ztBbPS9Sv1yj(tc6(8y6Vpsr2J2s4lWkY6DclF+HjTwSFShxL!wT%O7b9HE`ZuXttt7fE(<?3};2QMjh`0`mhZDxdhk$sXEMOk6OfzcrT+C#jiqX=6VEqw^7zxt9>*!JQ3et}x#6RF#Pl7cbbic$vUKo!fToXJKT%8`s!>Leqv|Na>_>AODz!J@7j0oFIHczD_lp?@6-jeDA`nB%LNc2Jjk7Zuey<vko__Xys|7Y=w(2o39gUEyO!=E<vpshG}lul<MHi`>#duUit#3H0m>)rf8>%go~F36b3T(#Z$;$mg@4|I-#-T+LTU9X+5yG{-nV5X25O6XP0xT{#~G~4E+-`WzOH|alzA#R~l_bGS1Wf{@c*jp`7CU9TkjwXI8?IpHu22sU=K3JdVLmQtyb(6<^foUdT_QVSWUP1uvA-pzp&x_2mM~y*MaII*+b`h8EqL(5cfn0YR1aUDooL^Ep+9>zeB~YLrmdMTvZ7i{~+RCbWKudBaEM|H58U%B|S*ps;|v6|0o3=Qy8`RuQKsS?{H6NHT;dL#4!9^WaqEM(>N$A4UAKAv%a(K&8#C1XucSK62$HS`vpnyeK#w#$J(YRp=a9dq_LQ={@#eys6^pDTd3N3%EBhOHi4S)uwUfBREKFh?O;2YIGdz9VzEhc7oRso(8rbd{mU8|5-poRpwUGssDSRu;EhUZG)WL7h{<p>9B=nAFE?X`(T`DaKiRb+7Ef$Kn3YAC7Uu7dQe<{1RgdYuvRU4QB))aceNK(6*Ky=Q-|J3Oi!GU4hhUDCRK^s!N-x)33k`eXv=o-&#1Neq$lC1ih4GVRv4Z6Wd9E|9Ej%8wM7kOj%N6Hy_KcBPuTP&GGfP)oQp+=T_2WnY?e5);c`zlN9rD=Q{(W&$1VCLSWie%>V5`x4wDPAC$LY-sQj)rKY|o%I&P+1TGC)n?E^bm8l51UgVGsF4!*Xu*<dk`dGFn~sPFP>%DE)>>R_nQEn=z!2c7Fao~mGFv0RgFf|d^V6(=?L5Xd-;!}dpDa%LO99#T7Sc7go{`fCYQ`sZn^;iyf{8On9(MtC~nx{YOBs#zK+7}euYDNF$efec@KDoMQtJvHhluzWa-abKoj1m!+=uTtGm+mZGRT$k3x_|+xX1g9)fUJ};3s$AiTo4&NK&>WL<poK@530D$usJ?487-@VBvHyhRoKETgoC60`&?}mboOgAC^C!6#+O6=USZmIMDZ4vVj(<Gk>x|PYDDN<cKIv<$>O3~3uZZ{}W(T|_?P{Rls(%M%y+&hJH8h=t*&51Cs4Ve(;g`V$N3<1%E>zZWR3Lv3#RCo3;^wHnkjjYDL#}6O7{Z{+t12&&V)V&Bpz>Px8}^o8`m}wK9YNE?(S-at90#HY&^mHP74s73Hkf)*aHGPFRf#eh&>czR40(gDCn#h&ITx)&eigp*!bN=UsB;5V%;z`Rj>(_CGgB$<xUk_>#Px<uGn7qZc|ge`G?WyY@60lG(tlipeh0^99L{9Epuqd9X*!F?>PMi`JJ0<CJ#>yj^ix#3<H6!tnQb)^O}HJvxM8x((-}`pE*EgT#@7*_ZLp7IDerWKWuLvYB*)Ml<8=eo5Q|-$S!5JNSH!Ro#q>d1`eRt6$tqK&Ova41N_G0Ag!`)Ms~Q=DS~;2Q#q&Ob5)=~ef$^JCHQ=z|&J5-@`Z`y9C{!spf3YXCEfW(C6Y;8;ZIj=Ve2TigIP5}t1a=1d34F|GJ|{)9T^FibX_@Rt^wRnHglYOCNOIZc&4x}bPDk{-QGbFe#O$V6LG1WgoJz-m=L*dtmP2-Jws+9o&~*vb0D>oPRSnO?K}I%d>N=zfp_S%z608=N6C73FjZEw(V|R$BgJf!)S0lCtUv;n|t)`5eFORWm-6qAa0h~h7Og`6Tl9KUJdPVY+I9A3ZL(LK1Hk{1=rBfhp36&QNO!VuJ=~MCY?k92HhptBr8;51E%Y4jnb-|^IbPQ;@v8!U?3Ww__&4S?*{jqFx8JiTna9Wo^L3R^v_OO*_RZ{PB_AP85A#LHYhSn2KHEtInYeA5Z!Jd5oivSJ-$Tui|z`ZZpfpQLn6KphjJ>mPElTDZxIg{YC2e&jW6?>$ldE}&y7Xxoan)i8DW!K`K#zzNK^vrTHTghgQ3Su7zdQM@lvDt-r9hLj6&uKmae}p~nPt*LB7TvkN!pxW!Lq46UbD(A#tq#{(xXVFt1)+(~Iu0hB>+^Fa&HdZHaq>jDH0D%6+DdVPyNM`k5M;qn<Wr(U`9H%{7t0CCDkfj>&4+6R`sue>k!ZrD5R*GJAAHSVz9@ylANQ$bWBo$sBsM*E16eBuze>}i!<xJe-L_I2elh*>&et0yoH(*^J!H+INKs4y^-~-rVcHVo$oV}wMZV9lcZNcm?<p|m@Rc`I!l98HVY^O_La;*oI#9--c7qE;ww}1ikbV@s8NaUNZDn)9r$*TdjBTo`=^b+MjM*zR5-^L%FF<1@%JntpAI_DRS(sk2Foi)!+P&DzL-7Scmr?=b)ljeS{D88Jxe)4eY^V{OV77>Q0*gtAcAvDmEHW}O;ClU(wBYi>XCJDGkFdy^8G98dD^loCGD-ds4hN{3)EnR;g-+p5#^`jw(B7FfB^p%Q!YK=d5fpr?*ErXP^bFQ2O-5L6QfVMch@2x^Q@qV#{RHC$g(qkojCy=fgS?=51M?kDY{V@=xk&3Yj%+$sK{4yOE7zMiFG(x?G~=%lRwrG3%3?)8PgR;jNgins((ERi6nZ7F`Y=;m;L>Kr<0+j($=PwZ#!D4#lZPheCs5EnDP>-4q-#ifh}RsB#?p!4Y~ZZI?Xon}ly827Etu+LramQ=5+3N%lwp-dPt2w1Rf+u^4Sc9wF=@tbO_Yag&S>UIU4z7u?@Gp3bPV}2mez{~h9qvB%f-Pac63RkXzgMx`UtDPIEw-^)kbHDdl%+eeAMBz<F;aTm&apC8c?cpa!i*4WQw5eKIvP$5AmF%TtQlQNS&oRg4UGhLu%(?UXx>pr}mdyP#HDNLa@h6O*TtjCCKi5gef{^`Lw2@Mc$APA*`3uuxOO0z<^vf4$89Z%4Egi0VjI=nxS#vPL<jU#~V0zIO%ZI!;><kHyG6<wFecD<F?F{ORK$G0S}dvxzpX0y%M1t_*c~W;5{JiU}Xyv4a1tW#(d~-!33iQ;Tn2VS<I<6kdacU4f$(GUHNdwo+iq?+UnAET;@4xv#NIPf;tyIT7RfhQyH=xH^w+pRp9upY0%2zB1@YJ*N6D*kfNQ&5PDZJcf9f>(Z%5ewt49+IpzM_3Qku4HYsxhoIR=tUT8E((fP_M9r7Dox1cmbtH)E9dsSX{K}E)HN1Fm!`!8MW_OQ7?X8~RrMygn3A$>;WtJA8enPd;))S>bT>;ylP9PTMl-F5g9#LDWd>S7k;<YP{eRSokIZvvXwvQO|XC+!W)I$vBt8L@eS2S+~Iv~^^mLBRsEN!=lq#<Xd`(Wl%I)+Q#D^KGIr{V!l}z(xpF1&WQ2z(=DXmAN#AvbS+Cz+#JfqhDNkT}CGb=QFG>n5|K>LhAzMAGADjQIYo|j`|c%q!QrkfY%8(Hqu^Uy+MJpxB>2OnA*b9g!EYY4>}Ye*}+QsQ&KrUb;1IA2EVpAEpWw(Lr=`CG#ea6n0(-PB3p~DJ-X%O+Q4fHry7{SN08-S6QUM(E#B>LoFRpY_L8GDWbcq~qNYPpDVK!@l|u)-UEi4ub}wv1I8Q?N4Bi%RcR1Jiu;RlU(yv%aLcf4#Rb=<9g>Xo+B5@Tm595=T_*5KoX9wpQ>UX>xq<s;;#}fnH6ze(crr?mqOIfxRR=(4?bE^e?8y`C{ZL~6E4BojpJ8Nt=WTNJ93&R@uZK`k3D#KG1+B2OT_QsH1qpe~(1+@|Pls#EMYYCG!R_d~?QN0KIkZ*mwq&|W*1y#{|c<+3g>yR(WQs-h8`x+faqS=3syF+;0aXQ870?sb>hA{8^xk<|t7M5~*eR0O_0+bs`RB7Ab_D*^|sy?9e6vgX_J`@@<XrWO=r50Y2E?K&ddD@re7AmVx&n}xKIi9FC7d1ueJB&i9cUkeG-NIsm`!~q;c{HV#PMax|%bZ@)aQ-Q&3ib)-wb&b?ca~NI^$8y$?gvs`(`$_34Si+Hp0qH)viYd7eTRf67n9~qTw0&BK3Yw%W|UGjqf19++B0I7p>fAboAnNta@1DIsGP0~#{J7Z<&{53lcQ6@z~USq325(78K0IS%z-H6pA2Exl+A;ehM1=hcg}~Kbc$j`bXA6O|1V;<BhLQaGND?a$vO6iv`NWuNh9@d4$BG*UD0xwdqK9wjUy|gn9e{$<Mx))w@=CSr&OOt0a!^GDRqkQ=};q-iHCWY%SoyCA;@v20Yi&+ZD=T2&hldlv%{zM87C9j7pZZewMYH3_+8E?In@xqB$opZQ*8CIsbf%uxeM(8YdzRy$@94DW95m1D=dz3*wE~Xn+e-lvJTik;^8jy7c@FhxU#Xv$?<L9IqUIZ5Bn}&kLXsR*x_-Lnl))p(S3p0VcR2H&CXfo7v8n`=*n9CQ$}t8W+C>*d^$lVfRjVh4zzq4Z(**^Q;^As%oQt$@9p7mk9`|8O`NgRE)Wgi)uL0#EsKu}+IjJ-#X(JK+Pl`0c7Yp8k*{!U!RSQ$EMKzFuxXLvQ4N|$s1&&~1vRh34t1MgD*o$AHSrf|>M2BQVeooq2ApzYZ^_C~G)<^0Pm!izNp8TKnV9NNii1`k!Gg-2_;qJ%fd!K)89Z%8)8Mv4zy0x=La~L(9tA2Ddos%7IKk#xv@ACFA5O^k;yab(*#^dU$}O?2knPHek#uc3SlCHO=JtnGO4#5|F|<E|@c%^b$yG}hs#+(heCONo-H#}zv+CvJ9(!w?=R_~l>KX?_neU+yegw)hwqxJtM;&z~`o}m7C|#%U%#ka4<x|o`<&~dTtlcFwfkZ*dV{syESa|cW;88szS`8XY)(032(W*&f0<8>&GioWxu9LaLhpIAZ9t>hn$HA7gsSFm<>Pb~$<6Tx;RK98P;7CD9Rf;}&E4n{$WO23sw&$QEbv)o@DfUAO%itL3C!sLJVT<!7_ODVOz%i2k<RfgxVVMTi_@%t{fO8>c2bjH+brRi{lp5P69=E(uxTu30Kv|E20F%l_6tOsji>d^NKh6CvPZB)b@g_&rBi%GCE0vtZVS!I=HeA?La9Q{W?zkR{=E$@|`3>E#QagUJd#6qxq4_qRu(KfVqd)j^C)x${ysW#NxWUchTNek<PsxBMDO51rp78Syc8~r1r`rabN!(8{yW!d8t+g<(Ea-z%XDsAVtHWW0W2J$2@l()K++yQA#Ax=m*ECu3dK<e7$c<#|a^{Jp1BShK)0AY6)F668Q9vApklEns%&8nK%j7G;RAq1u<qXtwP_3esdS@OGR%B9*g9lD8;J*0O<a{2LWxM#4iQp1~zvHOP+6#A)ICoeyNa5tU@K(`aW+~am)D(jq_eW?=Dd|B{0pXcuZEP>GRh7COSJts-@puwDV;Jw)+i@ZbekAogL~FLXcz9B2^ydi}&DgJS)Rsh@GgmZDaG7w?{e=VVGixombaA`@tMEz7g5t$R34>A`bg*fCd8OkF$~D%XVodo|#6tj1=OY+?O4c~pz{eW>1Yb80-DTvnQJ3T%T*ZD}$|`zuC7Tk}NTY-I0~elgbP(nBRJGbD1)Jc#DEkf{Q(T!z>jw2P`2iQEQky_;1&OER7jm6TO0m&SN@K7cu5HEAIcdtqk<JYcAte`Bnd74Nw#RU+vvGydlWHj#M?9|b*`(8px&~b~6sDxr`8S{|;W@x_2}*ghhc7o+Zpv1pV;}oP?DsJ7!2U=Y712^q-pcjB$t0#SwC_VNBi4pf8R_I;JN@!P-7cJR(5^zE0QV!94{0a_Ug7C3$tKoZ8CB3)$>mAb0MZYN+V7gmosi0N94l?E;>DCvm9yzjT85l%94-G>KF;1+KYn%irA&I2GG|I(Kf(?)7qF<ldmXwhanzux{x)t@KS9Ca>*yowLVt_ei!wXuu25;2vF1$rf01c`-86M;|1)+^Vp{a-fZyT$kxxr#4N;Amy9a+F$t{kTv7)-C4dIfCTUd8uw}o5(BXD5;z>W=GidQ)(t!R?{a`|p4|JTI*26hvg1f1`IvfQmKSdlDt^mMV4hFgPN9ZxnG>)1VE=fI=kC(WklhG&zb8Wb`RwrOzX>mE`=SVUa8fOEpop_fX_JkKgrQ-IQ@`vr1ancLVelYOLD4qiQ&WO-_m`2x$LTa}AdtkzI@Ry%z6+V5(Of*~}{vD4y0_T8Lgrb0K9pDy&96dCLsp?u}j4(|@;=H#iuGWiSrZ7*Z5i@iG)=dwRaZ5q2}HWqMP$#p{a6qR>;EAiz*856S>H8-@>_>sfbh|_aihFlu|2b}cqF%|nlK?gE6mrAi}^Q=zJgjx|ed93Ehd19?1t5fWoSgmkRflK$l<yDc>$|GAa9O1?J*B?$K3YpvMPg?H(`D#UA3wl26uUs!fr^MxwRN81exWCe<&!ryBv`>18%n?sAU@ReZky7u^BDf_mxx!;8dRN*;><zFmq1TAI6}$wLT>hzrt-AE9px3#v_~azt=AG)Lcl|=wJ0CU}p0F{KM20#RmyC}v3)?Z~6CXjH+EdDev}{4y#Dgc33?DOeJF<1;QiFXB@@+WoK4}e3KA^75QwK^q`6uq3@zBEN5?;;NGqG5LiY?ud99CF1_@<&Vi<dBVlC0<?NQ&`7E#P|x>qChw>Gx=wgY@>T`BcjCOoQoGc0)8CyeYtbO4l}B4Js;Q*9ENt)q`xe=o~p&07V(aG}&#)RIpcrVihvV(X3Eer(R#GCiyE`DXj=`Jd(-_$`4G8=&DQF=Gy*k_;^dPo5FIH4=?F%@G?iO2)P$JWk}6q{~_B0SgPbI5*^}p4q+Lq9q1?caAb1<r4FBFP(R_~fL|>-_TOezhJD#QWwXM<1D_4PET!jAvy1i;caBWd@jWO<qIfRt(dZ_H4rg+dIm$7I+b&%rO1fh9pjqP7g|dCj*>C#_-4`f&uAJz0#fFW^u`JSbufy$zl_VBNUrsr<llqBNhb|H9&s<jacnymwI1W^Fx%WBBvVW4avWa8qMKC&&yOC)FP64NEn5u&>&>ON=$6k{ZI-b&8J%HbpLLIYF>^C5NgvB*}tsry4Y?g}+=p`|0%kYZj66)pnHN;jQ<|VpZAg^KnjQ1Y+PjqgOpK_!)!{OHi*IRyuJU*Zsv993qi0v6HeC%wYy~W7Hc7m*sf{au>(gUjY_|@iJ@ekMxu|I@#jTFdUSE$XU-h^_2j}>ejKimnoYq(C+BEm=|SR2xN+y{_OL7~R440@BdJ^I2U74nxB-X7AQ;#?(9mx2>psn}bgcf&#vs|l`o&|iQT@og#XbnM-^yx>Hi5+2_MRBKVek;?SXZZz!5bilVEmxg?EX};%7K`LW3x0J|AyUfbd=AK;hxqFsL0Xtc^?a9k?_VVt#ad@Zw2r>sUR&3jlJb*iff)1?%Epj+4fmYznNO~ck##q{et^B72{fD%a!*7w5!RrcYGuA8Ywz00LFC(oDSUQ+9G*>VWxznSz!j>XU9#mLCDakigt2_8S!y&^?`9rbgd4io4cXZj!rKh~YTHKkeH?iM_W{8HxojMsIEc5R{3X2{W#+)%(e?a+04U>x|)N0ba(j$w-6Pqox=9tX!agS!1Ybq`R$+ft2gzf}0N=b)OzC&XO^(}@43U>G!g5SVgN4ljyE|WH7_x7KHvxQe#qTJgoWAjYjgp4*B6-*j1+rxGtsR0dkoZDhY5lvne`rCA6+=bZ^?^)5t(#e2%$E&hpQ<z@jsKHB<YfTI%GHu~w2#+9kHeBgpZ;OQlezm%9F2P7<21XgYrMa)l$P)_#nBJtl;N6Iw0hISr=!#yDZWczHx8d;C!FC51b@XSFyOJ_wmuKu`A=J>YsWyeceP<WE*pRM<aSDfxjBKfvadJXEDN6&dS5V_~X+x@olO`!gyIA4f!lyy`I{8byWbkRTGVUd1K=(K<bACs`3|bFL&uN=Ntp|ppsvHz8PNeu=Vy!IGF4+T0gfiU!(H2H^(gxz5=(M5Il(H_&+LCX9k^LX}q;n#Yo(82ZxXKw_aa+Z!&$l~N6*(F_^zjkMOhK!%8yP%%bgQv#{>HD1v%o=)Uo~2k_^5-X#&{RI4LFv0{De*smQ5U-u$u?>!HSCJFB#tPR3epT_Zxa;m>j5Q$tL)hP<gQCbFzrnNV*l8tJ*WB`W;$LXch2Pp>kI=L(-O{(@@QS`S@^c$R_Z*#%>>ud01{^-;n8wyAj;;*goKS6gwwY{2gp%(&WMphc>#(507wRve_fQMe8T-rrh&!9emQ#yh~8+DXK`{YS=o<rbDYHKU28cKw*iQ6PJUJpa%KAWV>{0%iN_=14E1JdueXs;D(Jh>)W?IgS}0a7HoYi-N^6D#gj`~s!xh?n9t(c;!GF1HxBBMPE*$WVu52rx<#?K_|mY`r%{jUZFGmY?#eL>XH_5Opex(r(7pk$r5IyTXj~a!(_p`V)3c=3T%Sp>i-Rp6UsUvHxA^a1HmB<lr>;0t7+5@Myc=6kZQF#gn*{GknF74FXeorc(ksu&GnH~QbpA_pA9NZ+)Wk#$ycs;7;*V(I$~u&Og-=zfEM(HBPMenw-Oo~0{Zw*3BgGk$f!uAlrzNq%iE=R$OpdUb<@=r|1$wGR1Ttx1p$>YHA1f~SZ><e$?N8cb%%#7^QUM1JUKUszz@ZJsm5tm-P+;ZqpB5%<R;gTAQF8`E?~C;>PPjLzw}8P=o=x=U*ht`WhRH3g9`UP859KOaR4x6#?7eAs97l2}`d4bGW3wA-Gzd`AXlOy#+@#^Xltv~sGaiF{%?7#vwAkow^+FKT$Orqr@B6+J`@Z8G&+&QZy$`<j`v(t1W>tOS{RuB3v-YkAKuTl#U5}1Pc2#9YMn*=)l9`dAEKW<)M`Qtfpd)2<f*du`HgF1oKZfb@tOCrx-Y3u3%gq%^SF<!x_4epgZUO$@q8mf<{2^&+mMSLbX%(&MA)q81C*`dzvNkO(?vaB5>8Osh^-3GZw6$01s*<WfGJv{UrH7}q`%vjxBBKjrb?0ojS1BJqtJyw3?4v7nbhw2!4V)jJp7oB?iXPJ0BdwOpm37L&kyO8}41#mnAgxTuCu^i`NC5+Sb4lK=kY3kIRc%V$h%&WAj={FBKHuw>_ovTKhNQkq+PCzl8PW#hHZILAD>b#WcTK8-*L%uPwKNI8m+0=1yuYE8uhaD{Wwuv=>2T7g4Azp9#y1rW^2RP%2CBSP1}14uzcMpQC$|;wQif~E)S6OV{q|^8>4D)GAiJy5^z7NlsIs>}N9x}m*PPYtpC2rfu0_&YOLw}+?gU*MlNXoBe&zXbpR|5NS0>18i_$YEbu3HZJ=Sf?!=1FFgKjjF+Gb_3Qd(+~>JP~v_?9zr)wDF!DvutVAC1ZzUCP8LIoTvrJ@VR?ymxSZ&_n7Mq=O#$Xi?r+B%Mv9bxrB7q1~%w9!%C_x;y)(5)}D}v^Ao1ZphnHbgl2KYl8O9(Aq)L@>*(ICoL1CvPQ0)phr{k+$i1HCT)iV45rp`X{1)^IF6I!X}PIMS?(tTt<ui0QonM3*e#d$DFd~nW|yoU(1vDd3ktL;8y(WfsZxI+PjxCib;{5tty)nk*QD`5y4-bB8m#{32?~<2u0d*_r5*L8{(#Q)(~T<9T1_hI$P_r6HPY@O>8&P9HDt4z4sJ>PlkzIqf;D7+l1xvYH4Z3CQ*;Yl{2tP{c~-tj*T<xeDP?m+>Ti<QCzQFKv)&1*ZiO7RNhfn>!#)3Us0@_Tl}>V6C9e(1t5Z_zrqobRR~p_PgDu@e$42SwG99Uvsut+_ywo~GhZ>aiLuCROuua|t!Q3D(tSEak^2#PzJ2@NPI6s`0nun#u4*7UXu3A?1MkN^K1Mr?&m5L2Iv_xuNQ_%T?Qv}9nxm)R4rrT}OX}7%HEl&?B3oFvh9_<?^!z-k@o;Dtl(+1MjK~@)(gGxHoFSj>It-Ew}C|fG8As{HX8|9wGKkd@01Lb5`p4g`=)6&WkIX$G{UG9OKvnXx!NOOmBO{=m1#{Z#wI-=BeQcyZ8Af-p?(1NryB)8AVJuu5F<<8afqmlEIZRuo9Sy&)#ePp;!Y1t-Cy>t}R>^>bjIzMWaPlo8sGFhG_?O;sr(am*bd*e+N2!<&-S}oNyk<M;1-%tA+m3i=;7o^H^d1IXHca!5yY3x+#?jy&`QsusMv?sN6(*Ah~I0XiO`b4UnCiCN@ripfy%iH60eqRRLq^4IHUOPWolG<0LgC$b6N!BX}7|m<d%0#!c(k7p*|H~<@JEH6DWN$=1K9)AXlWwO&^-_DIvRrdk-=XX`%M}~)$_nXjlZH>|&W2R6CY7(zo(>tfWa{LscTZlQBp{UM+oh=kW&e~emecjRv(ZX<be^`()7@Rt&`fq;%WIQT@5!5r39{aybacuaBebW7&NnJkZDhVmnVFHAXXL4QI=&~(4AI$9QoE!~57CV=W#Z&bc?+4}l3ve9Gj(!L6{+7>%17kYV_H5W*N)KRS$Z@}+qdM-eWkx!nwccSwaU;isRrM?mewyyGh?K(Ls<kpcpx43%V4ohOiL3Zw0lo(>6A{!XhT13-I9T2+M4CzdfK`qkAtVbC5<deGxKD*|9r2UZXA;OacTL4RQ4+i8*eK2$=(j>UXsAPo2@)Qnw5`_&ib09!=AI617)#;937FG3c0U>pr_kH7PjR5E@i4uo>)8IYg9mg0*_R7(5)e*4F+mdns1S}>i?g0X<&x-O-fUT=O@(^%&^`%`Jj!|){wn6GWwb}tdf--WpRYm51sWco^>q#%epjwMEf@7##UvZQmzFnZ&aRKRQejF;R8B4LTeY4-Wh2F?7KE)Z<CJhNPY9lP=&PIMwdW{ZIS6(+Eb$}Y|5`s<n=*$s#4moBWsoN>N+i-mydVknsvE%pDxUvpH`jkH%n7BZ>mP6>Itc%N$vv&Y??N8DvR}U$AYv_c{Va4?Q}?c$7E)cZk);!r(~^#EH%rswbJVydFzyP&Ysn;kbbbc7s%A8)YD9>T9vtWr3#$AIa&d;Y@Hsjkct_)c|glMWtg&kV{&s1S*s=u$E0)d{B(j=EYhYS`FM$rjM8yXmAn6QOgTEH4X5&2v%J}*ER0EY;6>EP+dK56Ra&f9R(hn<Rtf%Y?xox9v}fo|<sRu>lz=va9i(UF9Hzkb8lB%$y4%hVM=AJ%^-Xg5wshPf_iQM$)wE$jUTsqbc1hK~+&urMb-8|r&TqddZ>Oy*(((3LFZgy7^3H%Vu|p=u&i6<Diu5)}H4CJ@k=AWX{R{Fzg}gkhY<DZ=+e*zEojz42PL--IX|Vo$Z;<rYD&yT`q+FWcKO3&4ZHr3t4w)X8#wJP2f-*WKpAIXPljmSYOoAiVMa$=C7YO?P^P|4EN9*T@)l%h>vauo8kI~^{(gy8Togc5A^)Au6O1Z6HIt5pKL27My)7qviEXk__a(T5}Swri~>A(SP-6S2;N_oY<?vtJ4v+iaEqz}xA<3qXPluXt9%eFLFrwo86F+pl;-yUr!;}gn3pEP)?jE>6_Bl7E6I^RW3CP@2)vU?y^ttzc;Wa#yq_I0_UlJrbV+r!e<kW@LKboG$cYTCT2Z0^V%o6<lf9dDx>n=-g5%j=}3UY=T~4g1P;)%g+V-w_2oo9?;ur2}OWSiOeK^qn8}{Oh(fGDI=xc~#zSAl<cc(=cgYkj7Tz#u<5IRM{LQE#M6{)0sX}IZqDy<l!v|#u2=-scosMi?of=_Fk#J{rq%XZX1_&z&#(7xAw?!^IH_vb7Z8E)Pne&qRSg(c3SCQkypn_f3wsuEuFTV@14jE`*gZRUM`nc8qY?dxdSq|MO$~}h5GZ;O1Wu5>6}#h>&aR(X>K8Vy`*c5bPh<%D@x6he6p(Sfmc5v9WTo5$MR_f-Pk96y>w}u0KrxdNaqRd9a3snX;TwjKar~kNniI_U!4S$+g+4F=IjlTwoTI0pv=t6GaIyRkM^v+saPRnJ+!l)_JNovKi?aW)~88Bh14)c=jY{<UNY4v0oN{1N)v-p<)$>$D;*6;8|(7vm<%Jfu%nc>D9x|wM3r0x4NXYH)ns(?e^x0g%W~_yRM{vuwbA1t>0n#gm?X8s%Ekg|Sf#CVq-jjro|PwF%QZ9S2kY|2vT}S%I%cIq@E;dReT&@KB-IR)mR8cTPU=s|a2MUKI0p}XqgSb3r%lW9#=J7vPwQ7`M>py2P?msOPsn<^G&ZGF?a{6+rJ|8mgFEz^R={ikQN1j8yq0@cl*P95qejwMONX|k$|DKv|N414GbjV&cY-ZFtn_WrwQ2caRXJ%Vy*spePHJgYUiV9#bIQmO9hjwa6|{MT^zM?q3T3fNsoGM8tLa{!bUH&;=g9u1yuPc{l`DOxq-#UzT=~<HG7E}xhm<#v-PyNCgGy^JIb0;5#HTCdjZ>w!OX{mvD#rh`MS=H5M(7-vHk(rYkkY>)pH@nh%SvbG+r1syd`wUFrG;jBsGRJ!kjX>2ZG-ew)45iuvrgXGqRr!^x{377OPh_fp;N9opr_-~z=_l_C!O|?qXSykK?V+`!(I8bUumnSr_*G%QhB{BH7$|KR=PYbpY$j-gKv+k=+=?cxuDc^(%ns^XGvL|r;V?rwic;#ULILgIu_|j)%i)Sbh=5ZI>=U^GC6Tpxg<BNNR3PK)EM1u{%g6soHTSO6*FXWjjs17gTrK{LFqb?Pg~{Ft+UZ8I?+v9R!D0N8Q+rI7U<@-+%hL`9uRPrK^~1(k@Y3Hvs&rg{u{Y{Njj-l8akxr6LQp{Y&D$k4U<8zAREs2>y^%vv;G=sXHVYVmDjrI<|sX^p<}D0ZRPx6gPfF01Fz})guL4;wXTqp8fmdcYM+<OXO-UR^SuS7dxCBp(AApry;G^WmyRq;>tKcV$!qm*_ot<e4%&I7w4BK09a4FNJT*+4>*R_PI?$xdwv+ifQn5@rd&%@3soi*cTuEEo<;o#x_LzVq>>vHp1lb*-r$c8o>u+cFN#{D9oFhFaZz}7gp;4u0Q#w3VU~28o%X`Pl;EvRNLKeUYnv|Os>Dui1(VntcE1!VR(kAUM|MS3`_Dbn=OxoN#KbR$ZGfK^#G|@xaI>>y3v@?EIdq}ow>G0{>eZYQ7TAd}sz0XPIl-zWv3~nkF(90IOGk?A}M*Az3u^rL}o!(F?n$89mm7zMh`8DZZmU@q*kqX+nD6bqSb;HueKAo9Wn&;?t_n)SvLzo`jWNSgL>zA6U<+)C}w?w)sq@LI0c>n+GS9-xW-jcz2YwD7_2jzh=r4MxWoKoH=H>}Ccdt|O#p4)m;KB`oW$TN#_Wi6@dk%m^K{e7uv@BFltjIS#d)pAX(JUAj(bSoS4(pC!vucvxd8aSjwr^?BIGSx+nXO;aL0yDa&Lar~DMvkSP9i?JVSvVm*y|iIgsoJHTE6P#3G%~NWR!S!m(qNZVyM0#ME4Lhx+THV`b+SB0N5Eldp<{!}bUVdcmIlc}qdd~1bRQ_=`*ePlln+Q#^W?Zks+vAOt(NQerNav40949^T+vT@wxyaH0z5O>N)J1flYUY=AnhE;oqPXuEVotA`c}EESvo8yjoZ@b=^L=CSBB)u6{%-S0;$v0Kze57xvBqah_0L}eS37Si@b(OS4T!Wl}?c7tI}~dZJegtjej~Iwcv78pH)vvn=R5ouL8ojsZ#D=ChIfwc$qA1(fKiDb()N{$!p8XY^O9=McNuk>x^`;CRZNF#|?C%Rj#g;SN4?U7V>)GO-0|E%4K=9TIpIN%OkXTTG^j{Q}vp34=H<-a&I?X-zVjZ=f|tGzWJZQ>^P!5yY#S$Hcv@2M`XQK>IGZ8S)PC%9g)U5rM#1F?$PxH0*Y#JR_dN7<)F3KrI}XwbrapGA*+?Ne^{vkOSes)D3`ZZq=R9pdygLWpVzlaH5+v6P+kPLs7LN<Cp9Zl^A_pv|0fWvwQ|c0nO;-|k0cPjYZdb3usl^mb{5G|iv;%OP_1;lc78BUddAKV%azJm+OtF^8|CJ<KlPEZIr;chUag{)>&noST+>WD59#47>2H-6JC&{>X@5}a8&x`6m5Bp+vH5(zO&$U^pO*)E=sHO4Nu_O)9G*&x2eh$In(wCDby7zksjoOcU6vOc<kkauzfx)$S2p{|K(_=0JQ^n}HU9@a7?V#Im8ojEwMN;jQhM9u?M7*`MV{Z1*UHJ(F*&T4pli$h|5mFE4k?}MWNl8`>5|v$$SP>~E@iX%tp4?z9&l5JN%Mg;aU?a4(b_q(JRmpqkiBC%vrQVOXiEb*>5;mJ<@H9nep}iFCw5L=9hJ7${;iEp^^ulk+P5NA?<gai<fvadYLM%@$y&d3*ht44<QkYX>q^zAyt}G&b<0N!-_fm6siKW`PfPv%bgM$yX(Qz)%Ivb-xOdjEBR6%E&OzF6B%f|d;9&P{NvkW;N;zqJEj8}Y${GnMyg5Lp_Z7Tmt(LqVmyW8G$_cq+f^^PE>n(KYn9lCfh9S~EB<;MG4#w%;p3+-IYSyGV7~6HSHAnlmq~Rmdzx8&%O4<SAXj7hOCiBh8{;||BBu|V=Ez8p3WC5w_mFJgfYv=jivfKm&uRcGVl8=U@ifwu@Lyqgo>Li`mk=v`u_5zt+qV>I`W}gnv(uS&kt|luda@`WC1zU4NE^m?6x6V%%$V`(mG$<_sKCNVRg$@sq{R4SqN2*zoS0>2fDDCT2_QA^BBHgP>_qx*CK!y)UTR#Eoc6U&2E2sTCw114&57Mc*^ZijW)FD@Q$yLKj=b^j}KNe}%q|()^w9F_|t#Zv8X<U?APNboFy1Gga=V?njZK{*ErX_G{_j}LxK*N{Q?JnhHOloXAD<39J0|dm*>Yj42CpEN?cJR$+<>6gv<3t{8B-5*By{*#Zfn2#NgY(nYAdQ^J?X6@JW>-I1uT|>WmGU97cO)&<%k|Z?Z-EXkl4j`L4ym6e;PnpGD&TSMjnnaZdbCLD%H`uRWqMrcsy?f)p$9u;t@f;DM;baJ;7?BV6EOF?*2zJav^XO-_bDS=WU`C&m(%V=X%~!<HPQkG-XU#YmcVQrZX`7wv|&m*Jf@Q)%HZMKqy9IQU2pfNq~<00bXRU}kic2rel53l%Ex;iX>?gSJybS!<P*?qtIBK(scw*G#>sS@ygDcK?~$fGx#p0Z4A9LZvaw9+j>+(xG7siso6@&`Ht?FPPSd(NscVYto;<lmW@hBw`SXzmdeEY*A4yx2O84>`jOObn%iE;lSUTJzBb)Sil{5}1gNt;tMj75$_N(O9-8WT@w0=yg+9i!6^4N?tc|ySCSsNxDlVoUwOoLxFOzJC@r7qgrC9f`#ZtzKZXhWCMu|})FDcm8$O;X2>RQXz-sgNsQleT&?zfK3M$Z3l_IZeTMuj`dN2G1JCmHAydw5yEQ%T+_?d-Y`KSn37$W<}}Tqz6aRLajW}@@BqL>K`C$?WA@_I;~QUC*<04T3ICzu9NOvxpk48bdvpMdQ?doTfB6&iR^CDsd0JrgbaaSxvO-y%dMmTvL{uX(ybMFdWLqMlFg~NM-%dP@A=UHY44JU8_$n-pOV!hsqI9WJ3T+%JU=+4ZG-1WGjB$K{^N3W8yOg&coE;!S?vJnu9N3x&Z-+|L;3m9xZKw*4ev=ueR6NLbTBJ7&yo2W>1bA|X_4x|NL*BE>u6Pv3@WF#QQB^Qdo)Srr)c#UnQE76z|@(RYG#xQm?vHG*8IQjO8Y&^VJq34lJ;KzTcunzCzp3iN99sm4QZ$&W0hoelU5v)o<*giM;YEwx;Dx3<p14FR%hhJx}em)DUZ~Xx*BP@lQe9Q&SNsr_ND@84>aB(Lw$06{oA8W(l-A8?#)Pb70N~hsn{l66Y{`;G}K6!cjdkVrE}<fzlDrWNy7tj)hKB^A=6s~-2RF~val_8Hq+jAvOXa<?8;*&^2)B#SWf4UDX`;6y}Udq*KCoKX7YNC);22LJ9J}?)}F}qqhxdDO?fMwtWmlv==d(FJ(e3z=+uyM+)nn#<oZMT^_&b&W(Ca725F{2I^3j<OJ}v+w5gZ&f-l-7jZY~XHFUJ%tP9MD{Wk++WTIah>7a+ra@Ud4Rq>{xTACY^=Riaq(`8`!YN>aMt}VT(sG%U3XNRQjLwT;9)Q!lEEz;DIa@={=F+~sR<SF2VeF-KWxDZFZv>cShDBannogEY$`I9+m;7Fe8p_Q|weUB_G%VQgI%dynga(>iuzTZg>W*zPc$G^SNvTL`l1xuolSc2al|0JwF6+}WIu>h|sXL<jMLXmubNbrZEIhe+xpx7Vt7bGx)2fQlw=ZgvB-IH)iF+ByamJ_oGDrhxb?1wMvH?E0)x8~GL01mC@+H*jpR>wq@m?|L>6~kssE~<lf)?<qGG|;>;%nsxB#uKq%)aAYqlrQGTWH))&AAfuM(YMz<!u2OlUit&qP50EZTlN!T30|(B3cz-;zH<v+e-yeGa3PpW(a{QDzyI~U>zC2N8HA-04hod~;YVAS`2GA?&zd$M6{I`0lL#~q3*hzt3{*;+c0BQ&cB{X7o4*q`X)Kx$!|_Ng2L{o_f2z!9VN?wOw2+Ra)%2{u3oO%zk)Vzd0lUt@Thm7ek$GJqjVA%7J2Tsznjj?+3@6R0wRtli8UvFTd}`@3FJVsb18*Ke$<W<M_=On5yr7Z%?_lm<kfBfm?h%4T;YdvM75L+#LpF0vJmE7H!Lc%)Qu8F6%cfy$6VG`=ky2|498)371a&B)bNM&7EnLoX#icf8&%_ffNH{U|MGS2pJ{^$3$mU!;Jk<@SIHuCDB^VFJgW-6>A1>gDHas|v_7+$M4lZG=S-rH6D0-fzCiT$zpw)NC-hnB6uoR};4Tg=rc%O@9ahFrR53Nm&@<#XB7(8{SR&hv`axo-ENE;w?o)JbWw>2A!6D>{Z&-(zTgg-wd+WVx#%|pO(H=|r!BimC)Vqd@R{Vo#J(Xf~B2tI3~Bd3BXDZF^*7cNCq`~#TaC6|8)h7+HB<o4L|njnYqj47rBbaAZ|{NQYpGN*xB*j=?4%XOW7D?P!nP~Ht2F#v`$Q>h)&sFY`!j3CL(G83@1k8~P#NyF$C865M6i$quYWHqJ?Q*lJ(Gr`Wjx6-GnPvKR)OBy^U=C={U#t9C6<#~Xd=Cien`rx@8hahg9R-(>_h^U8gI~!HNTH_M<+wFGDmjmETdE67RzstGfN^%{fwH`~O8<Uz8Qp=l}FdPpCKurK<xtJ!36?I$1qfvzMXim<jpW3=>DI(m=u?Yp2`TOU&!Y3c+e3~uz!vRah&+x;|Uw-7i0C*;={L!bD^m~S?E+~$f+p?x6kC_!V60ty6(;?K_TD!cMkG;dGhyU^~+N->v!3*r&Q{9G9O|%rlNkcf_pD55IC~s8-ZOvHqEf+yd9tPd^*4&`4!$mG@>M~1COU(1GPC>0@BLSE##T7%zBbakqI)Jw^KdWRODg6GOZE_&8z}#*jQ%JG660x#00)`_A_42+GJ`}Q_8syB=Am)Ad)HXN8mjz-p@$L|>|06mG>cGQ*=@i7AVLAnMK>kq3Mm?jFBiU^EzJ3Z7Q|3h>6xD27y;Q1gYeqg#OU?D2g$U@MJiVF*a2&SFn@B%$7mnT)aQ{a$=4uhs{6d;&18nP)Di@scF=}mkEYiS~Cai=d?~G@KBJnuL`Sk8T=an#cNO2*|L{DKXQX(Y$C76m(1>;r0y!bvU#*h?A@Yz)W8G`VE@a=Ez{pOZh35*R?Dqz->31xsnIP%{RtXlk4F^V;{6)_l&MB^GHaHeC7uhg%sC*E)GedT4p;@$)%6pa+bv<jvKqn0WUpbA_2(ABTrhHmdeccGZiF#2sx^7{C<R@HD5Jr)<xV+d-qv-R+s2j4vYeK1jcFBmU~1%WawH|Leo!b1%oV->K{B1JF!G4?(8hWCa?cpmbViov4dL~b@dMuRWK5Dt&|tplSnm?-kq1f)0BW`bc<0^u;wSASOsDj^c%@Y#!tK^r__!7LdUeE<kE)oOV{88Dz$Kux@=zE6iPb}RSEwc;T1#5Irbq6ip08jQx_O98Z2@I0DeuD6<!6uxJF;`q&)l$58Zbx!UMLkB}?w2#6r&xFI1YcKraU?}8&a_y<jQ`AQWroaB%o_Q9B%P3|aJ)`soTBcB)whWtG&R4GVq)7WE4SfKl<9oMPeBlqdQd8$TBXoIjqto_x=3YmV2{ZwK?FHcbAbM&xkbFk){PN#``BEaHiNxg?3^k~{;43lL<#vN;GR>aUjKcx*#KeV5mdo2ZU3o^*a6ztpvUt)v0crmWYm?758Q%w%d3Kq8zXv2YdI6y_7axnbMy&Lt@&$46PqQ2hVdYRq;}JG}d;xVv6`H13E;<A1>|zQ~Q0JFg+6gF^Pg>jj=|Y5c)-y>nC%`7rq!l*3GCR~TRSJ?Je*y>}@JAED7orMQ72)9!pC6zq)PF%KB=~si63Ogo>t8XSF)p<@<TOU0MKOQCNCN9mCL;L8i+{O{Mh$S7jXmK9#)T5m;@odzNzt9+rWaiZw&>k~I#_2{N1o4ey)gj%Gj`$~fE(Zg1_!D35==Fh-Xph=CAdr<inP~Y%!`>Hxi0_&wn9nNrM>nDOvPu5AGyIDPxuRpUCx#DpBd@dJ;jg==pWLi0Y_<vD?%c$Js3JkzA(mfNQ<#x1Z0&E0d_9|5esr4@X|qa@hu$?31T#b>zS<-P=CgwAuwv5Tm!Q%7zW*%3Q848YI)ubUm#eJx&RFrYqCZi{ror=-f=HmJTFQ{l5rQhdbZA4KD$#_s#7)C=fVveZ%4JZom8X~8lJ0SVe6az^TTw=WvIxQ*;kudW&l|#oy3<>mr1R<HOTzYl*yZc7Xv;MFzu&>XzG|<AjX_PskJgvK{}P@sF#TTuq)*=PR5j%Jg2%vWH^H}L%E=`#vHnt_qm-Vp6k$!xdX*r?sUaWNIW-ic%FiS*;ookN`*wRD13SQYawdUz<{7-o}!+$qxgZK=U^9-(Eup36bBw>D~_bYD4L9A9A`%*>W~UNQWOLmqtO3c)YHma46b*W7ipFNRHS|`gj6~0Pq;3`jY~E+`Zy)I*>+Tmv{BA94YR3fw>=5D#$aZ9=X2B!@3iJ&(ijX{daLC^d&T(aPPaB9JW2vBO2qrISR|GPn-5a>g-L$4=6MPBmW>baF5OpCd>3%Z2Wi0rksZ2uacOILtC8vtz)Un-hvq@`Sk+~sI3~u6BcVX<g=V+gnOV|HBhKk2No$s;AKKm53|%aJXtp%W-O5n;#BoC5cIlIAKm3@Lh$K`c_~e=w7iwS@F|F^u#n1us2Ly}DFn0LFaHE?I)m0soQq`oO^qT2?C<f|zIy5u^>x_Y}j#BFB%E4aoVmvhcn+j0-@{Ry?qb>!Y_T{Q1GCNZ#z$`p(DtQ^uqBOlNddI&0=(sO!ScGXr4qIA*&$bPGry_aR4Q{=TE4UXi=w+Zdb22ba%z9b%=O#4LFz96<H2!2T(lP60Ks26YK+-YjWk57OWI$4w^3vJ{uQ32=8SyeeMMh7GuF#J(YU2VS`oD>!{ENu@uOsp<VZEOy1ux*t_Z5t~R7(Z-lVw~g$e%3nQbGPC`Iidrr%J<Ai0Ncw>baha7*X50<eo&N$fGyycy>fDXKu&Ih0RWJVdEn6QKPu!hP%_&Mg8}4YgNZACbDH<2cOVkMMtNY`)77~(*aJapf17P<j-Ezjf?%+pVM=`pvnJOw<@4H?N)Ve-P~}Q2<{Osa_P<LUa#(?!!FeKjuTsxv>GB#HiZ81-9ch^(t^Z~R}zpvzN-+qM3Y(dOj?R^srznfEJcN-BP`dDX-RuU$WHV%LB1M=^+kS1q3bAiO=Zm|7uc~1-*4_d;tYP36AKOlZa$N6!+Rs!u~4w2=-0V7bJBhLeQwUjpE!Qh&-1fA@9XxMmwYi?P+a1VJ@+PFCaya|Fm}wZvt_6(e32DN6pOKVmUhy_8%-2nw;XhLI67e4NyI+k32*o(^~>-1m*1y3hOX_%Tb^xY5l<+h<DarUuG3vL-)!4HPG{dOAq}Y**|&okd3-=0A0s>fFdfBV=e;a2+{T$R15h*;3@2RIA4dEEJTzowIss4P+!C&Pj4oR$^B(!sv-w=ngaDGuT->?v56{IiZ%mAa`~{-x$u)k$Nj>P~<tIG!2as|v{Gp`CEDNaTQ6Gb0Gf*!E+khhb(Pq6|2c?n_S&{3uDo*hMHlnir_zVy(Q}sME=%*K-6;3l?FstwR6Mj5E5ekVRms2tKPs1{GYS=3~9vQ*Upq|;DN!ZwN4SVm#fHUrg6R}9N>>)~~)CSB)x}d7{gqNkH<v422N9p%Ond#ySmiUWAuqqNh{Vc8lFzpGwqJ+nOT-P|%tHZMFVrp%fJ6@O!7x4WHJmN3Lq7hufcUdswhhB66<S`_U+tNTh8Pc}{Y{Ov%nA#~59QHCA69ddkLYR5i@zH$Nk%=BWt(WUCt}Y#dPAYfGhP7RuWoRJHqlI8WIf?}d^CmX$g;{c22HeI1=JV!yW!5$IM54>|-XmPk7m9^ap+Ly|T*!xi;K3YNWQB3rC|V$2Z4|Xb+NP>rtm(2IicrTV^(=L^<56I)a{-pGQ1k=+S&RV><}@Hi@-IJ~T?jHEVTNiuRctEj7*FC$lti6cv0*KI2tYp+vQt7MF5qwrv5nR|0_Q}WoLcX5t+YTkjVJi&5ZmTIR1LC`y<jr19p^}?&oU->3ZDOpM+y`8WWe&|fFqRS&dYvkZ!MCacgque`c6=8JcT-$sjy7QFh7lxK|_?+@xBO(rLN3uU>g9IZQpycbBsm-q~&$XeziX3eHgN53t&+{M)I9LUUn?qMqQ`CnD4H_f2!hUeP?IEg<tTa+E#N0;|0>JzG+x!ku?UWcME#U2hsD~A;eE7pVojS9z$y#*yh$#>ugDJM<bwyK=u37Ii2mjaUs!=3OaZvx>?|5<Z?!3J$LMeI?!4-%(X43h5{vAP5M~u@8!}5<Ka}QJOl!8%DCT+msCFDXVtx`%63`egw12YqNN<HX3zCk#hC@+>-I23YnSqZ9^sb0Ac~ea7Kr)-0o`u!p*ifsMj1n;YDybOF3-46r|HSd02{G^K3jb0)ikFD7Zs5j;bW?psmEk8=pMnm$6in@7CiUqfs>}e*8-ZHWPt`OB^-;hNs2w_Eev8{eCpmgukm^60htusOb;Jo2ycqdi}>mx_z3oji+1LEpZxOX$8NT^NoFhQ=rCFCla^-5L6`0op6@NwjR`v3Ag|8A*SAN_0;wO6y5|L~E1Vxr(V<hzBvR>USuVC4pK83PK1RalrjfT#5HR{J0cd%*jItM>12tFu!V6Cv5s%=)Xa*A#xm9l~mk}g|<#jmMe)L@XZ8-Sr2am4Z`r+E6`?v3Xb>CYOxOVH>FY@?%Fqpz&JXsqGVT7SS!J<@R{z5!wUf>S}U-_A`6pVWW3L>!>4-58afq-DUMwyle4F`l|Ri3&EgGE`%aIiQS%jB(OzR2-L!$nMpeHIA5fKdw;g>#==D-hA@dvfh_{PKaoQDeV9_&~Ua*m#8BfpJ_2ZssG_gm_DM3|yS~NGxQ>WQ1P_@pTYT79N-odSoCZlm=n8`4M_y5QNxQ4<CMoZRBNS+z%Iluqw{Tc<R!L<QD!Yz$a;?{QS5J*{ZkB`0l&=H5Osjf01|l?n5ExM$RX~SN<UMyEu`E#&2b1fo<xI7YD^qAnpxDvhw|bA~6ff`mk&!md*V1cFvvqIc}qlzaZofCyFDXNKu(L5-ZBWvR-(bi7LyRo%8WWS#H+oH~y#?yQDEzCKH>ytpUxvb2Ix+_MP`?@+&kHFKLpM$z*>&E{H_M%ujMYy8Xo$ruIrpOVy|){O&CXXYr@3SFc_czA7ur;=`}@vJ5c0jHbL$=1p(*jm(?B{N*RN@BBCDOe8nT-v2fJz6;%p<(3wMk7YxJ7h)afO+E|)JI2LJ+Ehzr-o5L)^M}mLukXU2d%lmnH!_Q2q8K(cQULY}o+m4al$78CKeiO&wc}a&WtriE%y1wJYO3vj8BF{-na|?k;%Sj60$xt$XR$(A|L_lg|3!ATO1J+OPn3nk+<2rQ=nt9D;59&Fmsra)CeFzCL$WL={6jG){?Fpc=;wd^!ACiN^H~=D6#i*f`0vSZNE99hlTIJM5rZ+{lnZJH{oxS&DMASN;vb5|aB<Qv`~ly=^F#PpDu$t8nAJZkALq3*G7!7RktEFAyT!pUo4sIh#Dj@4AyO!OD;5-o8RsIHd!iT>9)e@aBm&B&-v#41^Y2DVz`4Pf5B>Q}`q;6&|2@X-;oeIun4e_g;;+N`@#x?1zXdB=01#wgJc_3$u_Zt+!aVQKvOugcDfYRLk)ev%V4R&qEeyuMdQYm-rvyBKATU0(8uG`$9fnC*ltjOnFGHa&2bGtZh-AV)La`y<(#TtHMuzYp!DV+`2>6+gjWV2VD{)C33W-I*d>BdWD9Bt;BPdmaBL5ykVG6&403cEtHo0$ck6M7qUbR2TP$)Bst)O-W?GP?t%YWb&>%A<*7^Koj?75dqbK!rDb<YT2i>2%kb|%n>$-*1I%)Ie2;D1)$Eqn{0qbRqKZqZ~u3TA-hPe5Udpv}ne3VC)hcqL)>6zkpl;+<-Fqk%&A(=<3g2p5Vm5&O?V-H}v4JGOQXsH<$Co&NU6L?C2H^vA&$Qps{QJTLqP#sd9G#z#>n-6#m3ffx|tu>yE+Xjfk}7!^a%6e`yz*Pteb4ydhaBXKzc2^tsxQ5p;+iUCOWjT?W(n*J;+%Gg?ksxA@(BlzRbGvi{^kC(gz9Kppk#T$pY64LlPQ4AvrLx&6FakdM|xH1fu1;7EhRwTyRN@BJbD9&tv%rBF{0DOFCGECF~z{?>Lp)$>85w!_`7!~RKukv%;JP|arCJsZ9gH8Q92<{fxf{av#jUkr6IH;{b??vn`G^iRwaiJs_$Nu4>XdoF5_;GT<J2r~AaDdmMh$ZkE;X=h=7`J_4PynoXq9LQ_M}S@>{(@M9p<NQhUm@`Y2Kqh;-$X2L8&l3?;^_E+eI4Vol+WpaHsitGD|il#CnD@?AX1RTjtQU{ff|5E9GMVg8cys?)}{b%V}u0&l`(Xfe(`G7+5ZsBEMilg*td1YLo*AdSqYx5M!v8~wksHJE>u%q1e=x*7-km@opys*jOk)rzKln7HGvLMreW85`85A6H2VeH+Ahn;@E`*x#As$|5W{s##1eR&DHaw63xWtfXzbx<CXx&?v!_K+P1tlXO#VzDZzROE3qN4MT1OcJD14tp2@+QuhqmwFP93&ffJ*?(FgK*SVVM`3dmYm|%q6x-G@Eq=w(&T@w6=N5`U?t@Fdg_fzX--rC!%6RZHto_^so>Iz6@fop{}6<bY?CnGemvYEDmEN87tryp!`vP1l#&yF#H_;d7SVkl5ydW@HI0tQ~!g7nP^8V9L2tFz`mobN4wztggvx;`-MLk!fTHNj!=qB2jmP>_uWn0)Fy121mHgl;wvyypFMSXv$O|?sv5_@m;W`^(Ewz_*j3@z0FPq^h-K$wdq0Bzf91`|&bjrg8#ln{v_MKmkqw|?kQ)vNS{=>7a^dfiL69Dgc|<Fp{~(_4;@JUJ*hC{>n!(J#&(EGc%lF5NpM=5m6xaa+;WA^?3m=|@3j(?e2r#skKsw(=Q<^s=hF_r7iJWd}3(v{;g#unMGc0fwDR3)47hMD)vrH=nlRNkV3-VG=!l1L*OvcMXGGox6?~}n8n@(}uF1O$Va`+xtNw~|$FfwU;HriOuQb02R*mVj&jc=GXLk-LTbB!7ZqWA*ZqY8QERm(nOswWzYya1WWEIZ96M5P-^78MK6GBX(jwleojm+T*TF>*x+Y8+`H^WERxyY2hkgU1i<JiPC_cmH<}?%sc#dqX9&pmC}2<Oy0WnVDd=@cn!7rN1PKw{Lv$*q5E-%l`ORz8vp&(IUMh2ryqkBncAr6a8J>|3c)|y;*z>J+6N<Fc-8g>!XCGBzj#;!~+b_h@}w$Z>MB;7d+FUJ-fxV7UyAR;|2tED<p|(oqYi<RTK;}0}A8t5l$G)+rPn@ety9gyGIk%ArIbR)!2?!2`G4EG^WqRSdraUG7JDO_a5?o@%3-Mz5f({{$6CepCu)zF8yKQ>)$=N_u#g08;r9|ek|q7Z+^>-tc;9LZwO!B5q|gR_E%sG@GB<O@$(8_F{NeUYA`d(@HPQ1-+7+}lbS+&u@m8BNj?bUNTEr?XzEl$8ky!UCwC?qeqiYqG3!DV{<qaf5IFb5LO<?B;V(nyo0PyUT?;%jxaS#^f0mAe@Z$qEFq-L?#^mLF)L`rU#(;l<e}Ap_HH~UDbzFy|^C*L3a29q^pO`ovR{r7^0OK9A5vV~m%}Wkeoz$p5(FhP2<EzG8s>{IzgCWA_7IZD~4qgUGMs{e(;$G<520Z{FAjSdCMUI1B9t_`5G9YR;e>|BV7ZdSYI%1jkzI@1!t1?@!h}(KrlSeQ6>&K5&>&i4x%$l<sLVA_kyhd8XNL_UFL%+m?aJv9(7d}Xjn8P1uZo_Zmq9$N;sStoFC~ib9;F;@MAgJaKM*^gJS&SV&g6Rh%rMgBI;i8Iaa7dUdj0PEcM9i2Aha(C0q*F7X;t9U8UsvMdOCWI^%-j%XxiHwIOn$_0O$hrdBC+_I04M~$7!^Q1N@~k1#+ray7}a7hku;_|Sdhv55>V~v*`bqTu^Tkyf_($5&dLYkLL|(@0w}HoIs{(fHm|_sP%;kAiMpzQ;gW&~(>zip0v<3E{RU=b0-r@zv_QPw4ns+_O!<V@uw@*9ui}On<()0`7X-n!MY$Xk(e+ksX1t~eOdMn$Q)MuY#b^Qv?0q)_3rY15cW#mG5_ndTt&zYJsw1}6VP5G#mcTt6whjWd;=z|f8FV(zedc&^q*N#au0jCp--PHdVf4j?762{$=I$epS^DW-o*#%&77xZvi#h6y(u;8E2uz54&m3CB8W$`$8-iiQyocL`xEH|MHxE0)jv@=A&sz#M54hU8^}#$&m0<&#WdtnooI`|?$vD!2@j_uF6jEcjwK~bT>T2+qWtbti*~**d1G2{Q*<ePVql7KQ>Tz=ww)*lS$j+DW)tArk_-HT`|D3G~g7%7)eg0XT`9_~>Qh>X5$r#t)ai)L-ekS#^EVV4I4D#BVWN@6D{y1=Q6r6Be9gBpRpAHoMhX>yXOn`xv0(Ne3Bm!uQK|FG)HpH5Kjw_sKoN^;3iGDsK5#S^W&or@+^-PFGA_=eXpin0?^L5o0*bP+bq9nq}7m?K1fIckGRP23VKSaFWfwTNf(AIfE+{gPP3IOaNsCF)yn8m=`KyA!eNo&*5#Y0*DM+0w^{oYr>C{uP^Kbl@KC+G1aO9gngSAbWU30zVIo1J}Pa~f!d6A9-v6Da1=>S2E<?7eX#``KA}*E9I>iQ43CYm2w_29aU85=()~3z?zae8%-~M#fiQ{;}l&Jj|?<hp%+&{sQ8wLlseEXmr)rZrG6PM!#3U3)O^QX8uDk>cM#_fLvu4y0gO17%tpCMS6h2f$LsseYPG2U`-6fcn86RMV$89;vMoS0`Um3B!{dT*cub@EAom>nAz=N)-CQ^fI=uU17X^V^^G$Ywy$xdI$a+I!^xL8V@mu5zj-XU?tnYOCQ!VXpFA`?LBA@^QU>`4poQngjU4!M6aIVze?EpkpZq9%?p(tHOdVS@5K~3|Rd+#t9}GhW;(|*xjD>H2KGB4ZYC%Z=ol$U=vhszrEcs!&FF9$y+)VrBqqJW>PW$DPv~;neZt(qw_BS~xZ*Hcc*?pBaRc{P8N^o}xIImP^aoaSoOFk5{%&KAn)DW!N+*4g35X=GnR;xe|4B2=ToobB@Ev<X`^i{EgNnvn!$Pb*F;8Pen!LJ$DQF;<S>mEMqE{Fex6sc<dXcmK6Kd`Wwcv<+zEK^LENznfH$G`cS%Wp>*Fj85LH%(j}#O+5079Nue<%4Hbtikw?taIIJMLwKa{5*mI&(@nC)T@+K8U15QdBptELT20yn0)suwN|%L%dsC>Kcv=Fx$#bnZFImqMjJ=em7qTSL1e7^D58J+?H8G!8bn3KkA@z%7reqVep)KdJ;!I7Tm*XuOfBXMGX8#sBJ`QydRCkSiSbNlM$-y26XDN_qKRiFi5bO}`#4`%Ku1O7%S6xENfCg@R*HB_%;9u<HFfojB?^dZwja2ZaUmmvnS7{m)QP6|i`kkw^JiqRiGiA&`B|W#^CK^zbCFm8!^lK#bK&h@Y-@&D-Zwxif{S>AuZQ?!Xbr)XefHTOp)QX&`{U;?vpr?mo|ieEvK)`TPs2-q3IE5>pLw;7R#4H}eJOZ@iilf-Xo2Z3*<vbck+|k|tMY@nprHsVq&v9l!k2c~8{I}y<0v_GRLh(Z!$LkzVB?q!V<@+Z2e{Ay1{HyF(Zt_zp#HdN=}GmU7$frPeCOEGK5G#-NV#bpP?r|8#bZ>gUUfrSH*d97`9^eQadAR(cK<Gk`%(!$+MtSo!GlaRU3rNBqZOQXAHO$4#zINfyNebIRD?o52_KjU*=g|60h_69YUD!HfP8%iZ}-}SN5D%Z{wVi@^Ikp$lCm9LW4Vp-oc(K>B^rA%^CAsvv~xhXUGO{{DGiB%B2iycgIafPe|i6*<0+5)GEh>uz>ygT6E2GtWWHz~cukRC-+Sa(>+#*+ew%sk;qUG~^yg=(1$nJ|!NS6HwaUa~xTv65443+|^a{TY#_vT6vhQ5pj#iS_`${bG|Jon^)i3|Y<u#46ZiB&x1@&|e6EAnQMHgd9Sly%795j?q`b6SBRo>mZ!ks_hv7WE*2_JcHyw`D_ixywJ!WWoh#Q2cJZN(@bf2JA00s>>P&}%*&!<P0%F;x0pF#Naif@H!g29n-_Fc*b-PlbP4;O8Xru686wU%tQ%h(PE?K_)n>-msW(dxWbTzA=t5bmHW=8|3x{z6CGBI0gfKEKK6!3|~0VXkqj_sP69Q2R+#8h;G*h^Anik=x;$v;{t_<PHp^$S9tCy6U1C7lby_9l9z?<g6IJ?jQ`~dKm0k*5&p=U<L7|X=GbW{kHF4VA>AJalOFXP6+Wt;wX;)33H_`TzWM_zq#c*SH^2lQjU#xg`H{$T-ce}v8-p%5L`cg|J_2;X=)=w2uSAiJx%%mLoE;O>x8*rg_*H`#T&<!ckRw<C{>$$op(GgE*|!8dI?nfI<8F=t%mZkG_xrnKh`qxVD?uYY4vR{nKp=(-s_gY+@KEms#iEEuxC?R_FbD_z9waQ9*SIq*#(6bubqph>ez-E$ylyfU5_C-7W$&0*4@81S_3Rto>`y=W_-59NY=;^uuFSsBdj89t>`yW?;h&q?KRyZnsG616Jl9b8WNERo{1$rcE&k=>x2&`G9~d$WXT`_D9Zj?g4-)||@<kRq-2kRmAy{a*(}ZiZ9|(`Xx%c;(+EK*^0ni)Bv$0#kZSZr7#mpQGZ7^LyU&4ZBwM*v#ZlpxL{{5Q=54GFg*}9IYl6q@C_x!bxE`I`#o8XdqVMu)G(;qb;k(jT<AAiml#`#ZEdJndJ>jP&j78lt^m$pEO2dU%g1*z(9fE{K*Oz=A(zJT7E!?6qe$+$nnf#iDx5H{bl+gXL@FYsS?YE}XE9S^<p5|;SKWGLiAgu{`bhS}ACx@=YBBG$-b)t=)09nRl5yr*vAuP=dsXY}1>RAXXE<c0Vii2kF?<%9xn0r7BZ{ujA|Bi?QcPTUiE7i{!lx6sP`R&7aZ#zrI_FQ)}b*x#8ldw2ww7S_oFe68urUIn8rXP&b}{O}m)T#UjfRCnL-lKkt473MW$2wZ>4Z}ZMH#pk7>?FAPNV;=Gt%%^x`yE&(c-WsoAZ4*eOHkH~^ng%!+OEZoH<>PD5c9V*}+nb2E)Y)hzzH|LF)l|+gM%xmSm&bo%+b&ng1Gb8mZ^t8qtixxQ9_V1^X1kv{HK(OT)WniY6ICvgDDgOn0(N(?ZPt0rOz$>CToi+=zz}h%#|+pr5AIO;37OP*%ik5SNmI&hN?@U_c=xTE!H8cVS}gn()`Ypr%!N|@QB4u5hTih(c=Zbaa3NSBD@(v=PXw?2bX)7Fw;~6_m|zF|ui|Lw_nDs;mqx!Yz{>%6Sn);8WM%`<jEo#$vI1*@r=X@%jLfKoA*M6LfRCLi!sVJ^Ans!e*IImVNs+G@8!qydnkmSm-Y|&(dnd+yQM|_n%zKTt=5GMO{K$z5fEg7`Fj+I!m+GM4QU|~>F4*Srb2qGjwVA6HFTJFAsahO*j~$27F0G-wTOJmDjI%XePlXTlhA`}y(b8olx(fM9WVeK6|K}ikegx0i;_{oFlG#F5T2wp&%fNC`5iM;NF61zq(s#iKXJf>xnE43@=``M{a7y0^j>ZW2$%5yYrPYoLe~&Tl4j6k(WCH2LuADIiYFlBJUI}LDB`{44YeV^pd}~AM%Iq7B;nJ*V*qTh{a>z^uyDqjGaZM?j#H&qsJ1i5Qnu{+C{vXT<0|1!Nd=U;@sjt=b(+CDkCbZSyAUhSzSB?4lo*MAUjx+GxKXtF)lw(I#7a)bf+Y$NJpSs~}tQX^^1ioQF)Mev9u#l}hC0NgxA%H%B33;Q(w-7qp8o?tUaenH8*F&|*+C<RaR6?lQ47#FbS~EzZ4v&@mwy0#6QQu>N<+f()1eZ3XkBUt_7sOebK2ugU^I#(W7uagIQnDFYYUW+%DjrQWo5^Mg1~n?N94)-klWX>Dy%s{`A>y|f_8%h3mPg~t0+y!7g*J-YP62Ns77V%E!Uw|nUeozrgFM&yE)BAu*iiO`L8w0!Ys8P#%*&^vv~>nS{8SwIS75fEn*&k@l8yCc<9LfxLAWKP(!q(z?H2?(P{0K10^nvMUqGIV=ASPVd@j1IC9G4A-B9B8;s+2Jg+VZKT}EU7Xs^mIG<k9@GZ-!e$;nHmn9`O?rc|hGh|shs0vRp7jQRp%fxpc47)*mARc7;7m^MutIv6(FS9blS<^KP#Sw2==zVPBdc%2s5o{!y*$prIxQf?QjU>jtXz6Rgp!iYB-DRtQjAW+j_FvpU?%ebJ5IV~tR03N%aIwD_C9D$i1cd<m?s&NBN;@vKI;|Lh(&%uq+ueJdvhtCaogS-=of!ueguv8srKmh<>v;3ZTUrgR(Jw>{DOH3YzmKXdm`zaLDo}M}-)8!v|woq%Tbq>y>RyQf@V8{oiT~4eh-(|Uw2yY!=UV||$AANb}_LFOP%7cGlo9AGxx>$moFfJ2e;0Zb*EPqs*1S?jjM7z2|18=?+4a0=-jx1I%ES9<umt0m78(~)w8Fve@OxclGt}Ex`k3E9+m)j1|i>LWS&ELbBpuxat;Bx2PeCkd`mQ{6R+Y7qGdUKJ3<qx1OBjUz`FA;eTLej<l!3$-zfxtUQtYu6qDeTrakMM%+M`|^_*5WYRquK><Y16j9aLn1B+O~ED#0z)&wB-N`C(7l(P+q>!C<Nnc?Vrwv*FXDXuteL9zOH3?PX%Xc&j6+&Yz5*@0nfIwe;RNO!Up;U+voWiu(ttnZsw=<!U9Hb0v!a)@EELy9ISG#Q9|=-K+&-m+KQ1@Nu4Abe<mX6K%|Zk<E{6Bp`))SwDm;_P1l`W3L}-)43}`={ovTisS!%`&)BVmCOv5Ca<2Kod`YO8F1ccwLe{S5S|2UYOg+mrKWK&Qa!l{L$J!f@jaLj&2kI$Dy$boIdhIKQ=rX4^_j>&MBq)#TCgqco4@zzMrC!1cgDtkQMz5z<%D$))SIADa!0NcLITdT?ny!~ZMuu?1t6dBe^W*yP*O9~nYz|jJ#Q@7i<g!9`8{1L|?aI=BuMMbo+;J*+k)!{HagUa<6wT4}J5qc;=;?~}?xmLT8XvXd#_fdI%okUx^FUuOgYRFllOV<|!e9Ac;wI|-K#}-WF#Na}D&X%&B3cumHD9tD3-zru>lMS?8Ruoyb76R7i3^8#ga--mI6tt?So}bM9Yq7&3qp9T9DYP$<XIpK`2V;P!^{{>*u;}DepXitMMnty6jvaKJ;sXpWo!`xWIWph$E!R8JSrT4+>RB+ZP#Qmd|hCPyKf2KvcpZxj?xdXY3Sks7j1u<n>Hpm!LS7mP%l^l2CoHB9O-!(B#y5J=XY$|2D${+RsoM$DdyJ%vo&-)4`czPN2}=c@LD^|k{8EyH2p6bn(A8(QMTw}M=WS&9Kn^sM*!{~j6Dk=vlPY64%}p8hl}#dtR~Pc#(1}$IWw&{rIlx1Jwy*U=h?HA0^U0h9zOW`{_RK4o_VcJJt)*TGA{7l9S=X}?F0oO+<*M-vuB=*KKt(f{q4J0$lB7yXz^Gc=Q4MH3xA$HV=Ns{Mx)G6vh{<X6f$4)@=cIMq^SNX3wcjq?2<<iy+vN^0f;OwcNz>lSR?)r%uF6YWN8Rb$N52A0;}*fuSk?j8W`r^8%uO73$alck#Pl#@nMUvwzWEk`^c@={F>FUHa~A+=PV!8377JL&!5!tI`8vOY4|!f?Vy-}>HTG^%c?za>uR~m^}G0tJNHam2Sy1^pycXZwUtl>MXuUohZfXXoonj6{c>~5M9sC*!hZ9(&Jno=0W8B=;(v}8N~?lLy_}Hk;d)dg%|zn-a36&_R~PCv-EpfdXM$6e;ilDIlWBps7V;8!k8r{j!#w~OhnA%^j5#3qFU!8ak1?<>Uf|aeW<u4YidjG$Ke=sQ0vEiw0&j{xWvE51C5yF7-FeYsJp-^qSq&;9x1E}L0grk3)3ufsgx)Pj7%D|gZ5eeg$Ut0D#CO9$(C22Sj<NA<)4poP0KtV8)dRlPq6TMyY+><o$oO^vs5Yf3F(WFPpL%97Wb80E=6y$<2<)%=bq1hEtgCixbVP7XhbxI&c*G7zOFI*-Zfa;e8VG{r8drl)({<ds9KZ->K4xHF7|=#Z;|PAo8fKHt%(->(RgD(*)v7Y-VNOD;%=VI+dfMC|)yye9r}D{ut|o}+$c%KfK)X-Ow}OX@e9UO!cZ6EnFv~VHaP)$~aN2@qROFQjX{AqIS>SlcCnuzSO+!W+tmeqj)$a3?CR0;5BIc`2nXt?<P<3vM9CS<F^Bg4E-#I@Wkyk79v2>V@9;bO_78;J?!ntueRZhodq_Jh_ChcD(?X&c7Lf+Zb1iU<3LDwrSOoK#p#KkRqW>pmS3<4JVXv!q(0)V!gnXgO$$C6he1Sf&KQzM_&85&9%XeaeM79tq3F~f|^=P8;lOA`%gh~Y|G0lib_X54PoY5n)4&u}?uZzZ)Y(!>g|D4!VTdo2d7a5Uc@)svy|^SuL|e@WY_Ty;!VkL0>7X}*P=tUwJ?ckuS8!OB%IhtqJ?=hhjOlDdnm!Hnr4?H%WPO{A@t_H4my1$=4ijC|Sw)73zx{3zwtUW3G+n_6|aT=!CSXu&d_Iy2A)D$?c(vU04v?gvqlT0~<Jqv0$rN}X}LSkXG_`iqcEbH(uGE&VykpthVP6gy;SzS1B$nMZacQ;d%<yj<&ZEom`Y^aeF37JJoWvdgU&t4fP2t#bUls~apTUvvBUo0zB_t}riCupO&^$Au2f^Z(0A$6r=D{<6~Xmz9pctaSWkrQ<Iv9e-Kr_}^@$L$hbm?X@ozq^@MJBiL?-+Bom<x5v?+wgPegjH?J{r_dxnj5`li!54^#{9uvz5`!!q#o*8BNWYIC{b%x@9Y|y`W1UE2)iFQCAx^gpl^)=qoeY0gH_F&5X0k)iAbRVaZ5^J2jJvH?W5@Z6`c~C`4vtN(@CBH7qQ}V+%FX^H1OES2M$V0oKK0mvVs_r#yc^z+J%aYvQ>Uf__CgN!ki}jx<80?QM{dH}eAs-tYXESrwk@CPpli+KrYE@F%FIna0#>^-*D@f^ZL7pLe%i`iAqnccQse*MRN(m!BR-z($<k|+tN3k14NkPz)UUEhSMzJFfQ#|G!TxBr2K3GZSVB&`#aRoVF;8zy6QC!NPHz#Rxs&TWh%QSl^<TGRRR>~VZRf%c{kIj?51ken=dc1IJ0+{$)o4BQTNLw``JB(QUVnJq8ur7zPIiZ~%Qz_P5%LTm7+-YLZ97RE3=6L7kG{NvF&zMt2gVw8Oo&EG!(16?nMd(KO>(bCvYE_l!$;uFS<H8>p4+Hrm{PWiR%70wrq&|ry==3FaR6>AW6_D`1Hqc|N~1(EoRuF^%hh~+=DNDgk@}ISMkoBOBCW?+@N0}P4o&@qF=pn1TD&NWCFB8sdT^`B%EYTYRZ#4%JuHWdOBzuAuGD-Yjvaer>s|b`+mmZ|p<TX$#;p|X#w)|(HXWRPnHoZHLyJbt)EicbT9&qOu~E3#@ldvxkr2-&mU*Dj4|UM+!gwmi<NkShSPGP8UixxRpBjTx5@A6X`JrUYhc`hMywHZ~xflzJAufcoKSo=m2nJaXvG_oc4o+$L939`|FL-%ev5smnE+!BbczP%o6lKvNH*?%(AbF;;7}lAirg((nEWpJA*8!4-A?<6DkC$`|`E5|R`(%MH<N*DdCE@s5#q29&$`?5pQ}9T=8}BPWD4Ul#_goyXN+Ki4YmDE-?9(ZO^}&aY-Y&$2Nfir^@RdhMPbyP)$Z_ZSUO8#*)q7-XAuHGR#*5K+Ov*XF{{@DCGa;9)Pa-D6ftr}QS&ck*o1!j?yui98c)~Qt93BJg!;^ku7Yl<SAP3NeDHn|BwziJJp|OA%`xXZdE8O|bqkH!s`5xW>;^F<f-#SO&K=)Vo&ktJ3@q*m2L55C9RomO6X3}1*ln*KuT~gDM^ty?(>`HU1PKaq-$-|qhx(L^x3yf$L!}HB~YWI>=fO%f87o^!!haPL6-mm8ldJK<}^3hNm_xj^3O)GRM31Z|^w@KK>i<5<gA(6+-YEv1{J@YtFk5J@~@;RMj-A^HFwVbG=zLqp>obMfwkv`nmtf`P!_Q=`+S=~F|YoR?`%E0v7qXwY8GT26k4`@Tz`Tja-Z>3E~v~>o@WTa2&+d4lTk$MK;hcY-|UYaP@Q7Kh_;ppfXUQyU%02S|~T0jj8FSdYD3#dso)VM3H8kztutvm}$a;T)IBUR-zP8>@CV}hDsYmw?KD81AHC8kL27?m=NO4DJ)>zENLwIIa58~CVU)24`=0%V_X!dGSujWh*nY2+}oy;kd`QOiUo{RRq^9(h4qwpmja<WdLdrvB-mcu!JtA4j#s?IxY5Bj+)C=P{xDp$5|QU4L0@kGoV!rl+4~*^RA5OrUjiQ)yLZd@a@|wHjl`XQ)eUOCPQTNQ#iL9%~;~K-$V0z$@BZHUlsgE>buZK@?vI0u4LO#x^3`%qFzUgr>*Rq&t0#n*;Z8ebz-9TI7jlX|P(}-X(KQ!XE@O(oV<g%zTeGEm>-i>1wIH7Ak?Dk+vquQP2DFvI$~+pMk^QTX49T)h*pXbOx)H*Zt&R_MPVdz-j9xb%r?Q{d1lHD(~d!1bTYK31#Ur$Cs=eTkutlyN!N%YfDuDyR+wejq+S4rwgbC!ExTXEvH*7{G<jpMtQ<<=h|*N4s>XX_O9vU0hq9$_VV++MUFZju=C><>39YDs&;Iok9P^!ook;Iw4CNJl(x<VD=m=;KeWW0>-hb_j-Q#MZOFkV&L7UmIf)MoVixm2r>>#?gW;$jOoCvz$VvC-hy3B^K6XW6?hWrv?S=ESb-3ZeY#;!Ka{VtY97qGlO4U5PXDEXOAa&1!sb(N^et1Y4C$zI43{+bs?OA(!G-zAgyCvw`#)5p(MQUa=Q?DKzb2>jkj@m3F(`Gr_Vij-}GV~bmdNNum;UMjk_6`;8kv{gx1fkRd0q2DY0$tl8`<s_QFwylE4|f+hRTw>h7x<Y;0C(dbvmI)(0k|b3l6YYUuSuE@$k-T)lgVMQcX{{GwwkSkukeNqm-@$;c>Hhqw3`f#T8L{<l}k_EUPB4MD4aUnf3ej5VyXR?wA8p+V~mQ<dKoqr!O6g;u)0!bt(ajJ4DQq#xcqj*!w&*$uRYerwQh4j?5ZG!k<)z*)`}jZDTB{=${xJ9D~012d*MBceC#Y&<7qH$5&EDKp_m1=6mJ>7Y208~l=?LZ?33k&1d~Lx54jzCZSYO4T)dBw7J^n+UumfyB8LakVHX`8*7c9(1GkdaF?flJzHUpGy!;Xi2kTkytII4ZwSje*=oYK@KhfOREL2^Afa+jC)7YqXV!yk!n#L%tUxN|ghLyF1c?{me;=EX3Tx1Jg==PZ}InnI3Yo2Rb0?cc-u4(t(#*HXv(FjxymKYB_9f&oA*%lbBL-;-^#uElef2@ezmFpZ8ohF=`hcSO2DAD7h;kWu4lR$)m`P!Fn*uB8KLRl0~<*6U;1q%`$mfcMK%&slgzVe&D^_;db56i*8E?f%X<`=uXK-*=(1d1^uF#%p$4_xz6&Xcev5aEpr|AL_nN4*Yj`4N~BKi+liGk?JC1AcF4GT`?H<32qhmwq*g>AM4f^d)@$P{`Uorq~9-6UU<Aus0xL{?A~T-AN5S;z#@(i?TF}m<~hRW;d<ngSG8~VUF|EL2TEPlhJe~!VABs5f^{z+jF)@UlCqR^&*%sm#|+BVT=@O)@!7qG;RRV2NufP$J(?-%&O+9h?%diRk6)A;ca}@`;}|zp|!S-ey**%refw^=34ul1{jhjH<e%xH<%!)axG7`TPt)5`HIV;j7+vWH`r0K;vlFv4kDb3X;E&N8Vf|!+oN*r2QLZ=i+q^Kg&$qPE3#}KQ5mKEhPxUs{2@GSaf6-6dTJ_xatX7Y`{KpH!bI*(OLeq8v;y(mkL{45zeqCR_}m-bG#}Zv&Xa3Jp-4VnoaeETO-qJp)3~^Y#hFpd2&?U0;4y4t1KA{3TvkwS8+`Rhi?xu&!-=wdT~Ow_yt+$P_nC39ERAo`7FA8jr$gwOt_;E>I&u3u=-bvE@G|^8)Ou51d0lNinLu=r6BxGojN#Xym^$NhBEtM1j(#r3&~tIgB;KupXVX*9?>PMj{QfS*V`5=QWOrn!xtntZHcWbQ-An4T8KCmFkda}VH9A|ERfNk#MGtRHPr>W44zT*!#xv@@_juF-vpD;F>}D084+X8ycm0;(zkyzj@gA|?UM-LDH|&cmU}P}Py{_iEFw(d`X5X&6&A`~@MOt!XlQ6h$q&mKCo{TtI;)$49DDWfkx!2QWLpIkh7DLh8>))uEf`jRDBwJH}Ue~AOmGIU_It6j-;dl-4Pg3yv`r-`JdKkbh;S={eBBwLPRjI0{Z%!qnk<1xriGWQRf3!;PhMl2UuZ|vPv-E>}Q95WnbC(rQoFmLiXXx_G3;0eu{$v;+VBa!GU<TR}Sp3=V31TLuS2Hj3flt9?_`XyO=ise;MIp?6=}#siFdTqk%#5k&bt<y10yiT3k#Hvj78$1I!IP~I{rMcbl!8BZ&pyDC-@;_%iQ|g>m`+n<xA^j~v-~pF;=RjS#Pq+2U*?UEGjncuRmi{5l1nq_rGCtW%rfq1>5dFsJVr`t7PY0OHWGMF!w<}vY!iZ0)+r=lW+5XWW)X^K6^4>8;dLZG3t1@Z;e)&PzkYn*Ywon(O)YJn4Lv>2ql%4J=LNj3ik*}db*Dprl_o`r7z{_Edg+*m+wWn+ABwQ7()^<q)fLv7i~9>jeo!v%wUlJmGS99W=MVA11XnSCYeI-0>czBL_c<$q4)K?0D=R>=NQmXEX3N!@ZR5kRgRmB9q~$~?4Rf>5^s=DX^Z@$NXL$jBYYCH|ct>I$ynbq|T0zS`7@~sb<~CkAHEcebrmlypAJ9^?f6z}Zlm_F)u9vq|in@i&&$Va>izEF~$WYxZn9U`TV8Hd#avb`0A`!lWc)C&|M=gh<)l-u;(!Enim8mkb()6b%VTbdexOg7+ei@4-qj>t0t@#z<Q#|)(4qvX*>gSqZc?2|}`MvLcc5ujg6jX2Hd)~m^Nccrgz-578d9WPpvUlnE&jIAc&jAE%*6)14q3;RH7;*h-VRguhed&3OZ+n<evAT+VCd|IqK7DU}|6F_j-1`27_WlKbZ`%3sm@;*IZ_d)8wWEt2*Y{xLU&|K?gkmTSDCY~=f(w4Tg@Py7+;|Q)5iCkZl5v*}B?H9s_W|+30^-H{fMCRg@iQ*S^YXwZUm%k3!5HJsg2r|H*j2nU)+vs{z@*s6QPJDy=rGK1rL{O0>6(i>3AsSz{M>wZ`be^u7juIybw0GRB@bdU5P#{mNTEk!JS-;OUk-Ux;rwS1O<Fl~1J*LQx^MXk>J~3b=4S=XUu><~Kw3*Wh{WulT_#$Zzysph>E(wRJ*xn@sRfVbVmca3039w$l~CA&oOkJgwZSWQA;;1ME0CYi2N>iOR{2G4>OeXLgM=|W1LHjkE)vk`k;`^M?#4y4)Xe?3z3|fdXt0#aya~iUF~cI|t<E5>VvDq~K+AboXgM}$?@8pBIua&AtA>t)WGXg`o_o68_h+?$J`lnYuw=_hO2CH<7HFf$Z~I2G7s=t_mqP?1;Di7>fWsJiA;uke{5`b1!QeSp<RM+s-!gBI;T8f*uj3JJ7>BpDQLySRDD7f<qye4-4)ejNHy@zlVFBQ|H*6sEPO<?^y))Pty5C0s9R}{_p}9?{yc5k+>1xg$_0k$nlgm6#mDpjh0l;gtT*d!xBvzfOODJrMm^6d5_+OSg9c&uhw5na4`o`{P*%r@j-j?a>E5~NuZSYZKnfGnEsNv13-DCSZn6`mq>UJ)y;;5brGG5q98(^75sXe^A&GLca)dJ@^@HBKeJzJ}te<zmKws)~}X^>}aFe<SWAs#$v%g%D8Bc7##@n9>3J>`AjjEXy`=AmSi=KWPWX5G>c`a<Fho?Vig%}%478s7jwFKSgy9}~W;j0w8}jC3=)nWrN5)Tsh@_*jjLi(?Oc&V5sP(Cu)K7yF`}`Y_w*CO*m4P`4T|xjKSwcN)OFyLvfmDf@8R35MUj8u{734&PPE<3A}^x@qIBnHTo?oGD~GP=tXsmWlwl$$rp!<<Mc%oX~$_sFs7V$T8<M$KSTQ0!o5`Kq&Ptnm{^YY2P`qd}!dm*GO8rr2}Q@o)D(7j&xCR{w?Px;b0~P(8$5dFxEk?=4cpgUfIb=S!yw0U^32EKfgiX=x(PI@cam-S<Hi`fBG{BJYJE{yLXi)y5)~J>vD?oM4k%tsjE1imzn+4VG0FV<_VVr1Y|<I@Z!cT=S|Tx4P!M^yADknr&8@--&=qfARRC=1dB$7OC~?|jF)t9uuPNQ*#k2saQg4+jI|W7%{z+=$QFCv(F;teqgtX?KTZ5emT0O`X<E+zc}&d<OxpBz@PBFhl$9&wjti8@!5eE`=_ZqHDilOYqRE74vXnfAS%KI0>3Y_Wn@Od({TMSl({__fGoaG7|MU_Im+AA%dSL1KD<50O=4P-(XJ`!=zihpK#jf8{dvFEA-VEGORqtrn|Hb&Q5F;5)<5?x67n%EZ-3vcrah_B#of7^tyB$BH7s2u<{^#-}tfOJ4Kr(uf7Xe?d=tDT!9#C855uD@tYSlZs2^o%F0>BQ2G2bq(#U-O|WJpZc4D?0OfXRSxsSU;)PyMhk`v3f<GavS!XfnUL%w$d@JFO-OSj?}kV0`^~46Vzg%+DY~u4>4qo6tJ$$!L00{t_$RT=mK}yuDKT<}RP0jzG03n=%sr5k_y&UxX{ZU;2~rc+eld(n_~>iUn^4R!_%+9Z{SNKlcTKB|L63*E*_~+L*c&<3Ca?P!%{+BkJ$>o}v6+uTZ4ug+HbzTV`NguRM6W4FJ!zw(GD@r&7TtIuijMqWCiEy5ap8<STE=ZI6!OSJwhExS0(VwSzPmCK2Wrqu|$zgQ!ekEXQt)14~yL(<ndHlzPzVokmeTtp_B~+D?fGQUyjXZ_s4w+YTI?^|o5pym)L26)M)sfONNiVg`rtotyHuz<7&D(#l%j+N!)tLGP6lc6JH{M2$k4zxf|dA+8j)&#H?2Pff&_h_@o8$gb2gSmuo@eo_|OFu%1|Fdd0ARWQ9ZOC8ScS6fMYW!(8!<T6hdepDr;m<4t4`CehjvK_@#>Hf-$A7{3!2}Nx96<(H|()fp|?WQzq=53Q*^S;M>OyJD4GEVideB?1uG(L866B-EPB?~3NI9@=nF0O!C_Y`c`FbHC?ARz`!uUYG+WaFyUpw?%00`u|lr2)WhKQIhjg-@=18A%8~m;wAKsK3C}EQ{ZI0ysZOJh=w6d~!{D@O~Ew8a3Q@u)H9zk_rE22*3Q5UjLf=A}mZ8|7hy}rMQ^t>9>A4Q0C7|0lmq0q5IF%V?6;e@|0h6fym+^+{L$8%5!8IlY-Q;u4AME$_ZR3u&dAX;yvf%Pc9JXH$AD_O=?itr>UYnJLgkpp?))sAkWKYhR#Ru|I>@bcusneoo(-c<($9`@6AuLv+X5JLIg@=XMg-FdnDNh!b2cG&+;OgLjyEwTgmLw$!|_`Zj2lrmcv`FO_QA=63_xB<An*+K)cOXE;GI~OAz$aMK7<07s;_$CRaSbUC4rj77zs%Sg<H;_fXRdEOQ&yOF#VxP3s{GRMx{ISS-cAcQ}U@(7_tlj?&-32XM9vgU}Q9pcP|*9=@>uMezS)@9n?iIF3Wne`NxlcQ8W@7Qj-HZ9FYcD1KNxCMil}tmJ_>gTd|q7-4s3xwDH8D?G<}i9eD!H-7oFbA6M0a}&qM=Uv;$%gb|QEB<FhO8zPT!mFzOnEsrZ1we|jp*Y0OOm}s4b#--hRdw}D6n-0DiNf@7hn*v(-#3ftHCWHn;q5yD20JC91G!J{zD$?J2w3Vn0?F`A5=zU9{whY<Wd&Vv9*4b?PYRCuJA%6`4*4ycD9ge&e$G!E+{5vfYiQ|T?MQkb2G@UOwjho?``|e#Zl5dXPos1;85Xl)z5}h|FS>XdBO%l#J6f!ecEPrG&>Zc7aqs{mFXG@Wes@faCz}e1pg69ZP5E(UsGn#D((g?%nL3_{;$v!8>=S|K?AV!Ao1ds+qo9%t`ym5{JG;Z|xC(q0_50Hdr#Z8ojMDNzImFVzfAr?KwCA--b|ed{368;pue(EP1~V}fWllKfgEGmE0aF%p)uOZ#p%p86D)QYEtQ^(Mzq@IEm|ep;Eeg2_f9Li+Ance9p20Bk)5AALJK11>=eqB4bl4jbV%DpKPTHyaQf2Oi=HM*ma68OPJkXCPzsUk&eVLC3=9l4clrV1T;d&73ht54;K>WJDAn`$<vjh_?iKmfdpO;{E#;(yE7Qn@p)rXyRf|r@~KBpbP79)_rl8MK!v*A3MoDlyevvj;KPh#u$7-h&U*70-WjJSTp{L-a^BRnkF!a*pCS_e(!kbtzU9^9p1c&bu!%UE0u6)x2B!GblbN>vYLDLI5#N<Nq^n!||=o9`TVEGdJ`FIrOGIPi6kJICMs(YA6|T!AJr_1SyB6Y+$0!xOzRlk96eg2pWEg*BH$LqH40VI)pi$(}mzm84(%AT1sr#<0<4TJe^h7lg6x<1iiX%_%oO&zXTFV?SAg8){3nOw7th`W2&`5!6PA3)B}z>HQq!p}eV!K~=a{{cgJPhyqya;nJM~_9yw+QGb}802hqNfamb4?`J6`|FA3KSGl0Ea+(#B7pvx}>tumzQb@$SbHmR~yspVqzIL@pIDxR7z-pk}rA5%Pr!BASoxLVvd=uov>MM>&gz6j};U&697HtoqFI}QgFGs5$LMIttoY5$6tk<E*tC0f;^q~t3#GGfhbByi|ok+()8*l)sdr12=1DsMXc2gy)Va1|k9R;|S!V$WLw-#G#BEVAQng~+T2tloj0<D~Dwp5E#r<OHqR|o}G(?X#Ebjm0NBd3hAmm9oeM$rmG7kr!8LNx2kR^+2y^YHq+-*Tjl70@-@e$^ZBYNwYs-{EuGMwG3Btr;*aM`$?8N=obA5lc8ngftQeDggY80?cHD(-FsP*upHPdgqAWq_q|;RBb0uOIFFY8+$hZC8kuvlrzI@yoV+_+F5SWX-)NJJ=MT9!R%cFmV*f>xxowKwMjg4fT!$ukR50B<=H}n*g(14j_5uPP#wxzfMb;417!`zouq)R^$-T!*qcz!Kob|c7zGWWC=<odIBpkviQM^&DOiFB9L0eaGk^n8lx0OKvE}EezEKoIFS?vG!*mDqL86OMyCRUkHl+%qOCGK2t*d&`Z;OCFx{Cm*Z~)F8x_WVIJoDDph6v^@&%&dftQ!!Ck((AF5SbNn>>j8%Vp+wwLKQ(`9<ueY35e{rRX5OLEk7l(1jRAu+nL8rrbb(=_NY89%abQTCH!I+rdpDZ{l!d7;#0@WlxfJl^_A-sr!nLCQdj}X8N<@}1yHOGi0E<)K&&Y4Zamm&lJhEg>98+%5iqO^RqwL2zt2k}IF679s!x2X4Rir}mLXi=9~|vx(=1r)tWiuwS>>xBhoBwfvyzW@dC`1tTueQr*IAuY35h2@lz3qgzupS72~jdPJmt?)?#Xfi%)Mzga6~Ge1SzKfp{W(f+isoMsG^eNQcZM@1pCg#%8aF=>Ug2){WH0OvRjD`h=FA6(GtEf%ROv|fZB4ML*Ba3Hn?w5+9*q!2WKkbopjc+icl!>N`PSui+3IKi1j|+^jf;cURwd!qu>`Z1%LVKjX-4Fa-`bzE(6t+3Nh73#LiomJfSr46mN#r(5dMooMVYtkgpgF4^!Dohdn?<A!W#0u$8^0tXo0`o{pqpt?UGA0VV84XB^&K9rTBpkOc;Gd(CO)2dE<$!neq%qCFG4t;#f^n!lL4phKm&3<-_pMN$5(CIr<ey2jBcCh1Ptg>_^$4~@mnvBitLjchuWtSw*5MxF@w@*#4s7%Q@|o>h(?x;xSI28GIk7b~iItmxyhhdhJqFz?&u78GFomLUbWGa6q<t9s|SAy)HXy@3JJZ}xP{V;sDl+`e(Q$1sS4FMU~ibR2Lc{CjnsgQjl3b?r;Z{agRX4GL`t5@h9k$YQ>|m&Ec7jWS#^XrH<6YWCw+J@@dt7`tf`%?wIrZga8d$Z%0RvxUfBJ;G77Rx1{!x)#NBxM$7Ip02b|J#=OThYNyE-t(y42rPI>q~<=h+m1T<EE|PU!)h(;IJyDN!Dm88!drk7%Gy!(EKSA5O}voXPSW$JC$Z~Xgy>qY`sum0n0@V4KjT`Ctfxw4Ommlud^!uald05|rnAt~@wS+GKD=x<?*wQ{UrT-4^0wicRWE5b58(wlbmM_5txuEnYT~uF*Qg-MtsLs%2KF%)8V@K0U&$w-renpmYLn~KgKUxg*xZPN4LmkAfxk!icMIczv3lt63s3?(D0&3{Mm4~Be0n}c!SZmazdt#8v~p$tX!59!X>(Y)NVbF*XHKwjw>Kj4PD`$S@l#nGY@M>l^d1?KpUy(<yAXNCKCi%QgL`)HyaRSrud7KYBA2CEpt-)w?RCj6xC&AmF938E=XU|NYg-N~z+18EAOoSgvZVmjX#KLk3e+Id)^?dqTdm{L4b?<Fj)+s9na-Zjzbj~<i|X0Zp+t7?MR#6yBG*pZnfnSw+A*N2<RVjkLK(i1i0?nPg&KDgZ#Y}WEb&@vhIY%}S_hYWqS+HJ+0<&Unz1gOJNCA><(|_jSvvE{025IyfQ|1%?1h2p^PMoP;j~wK*P=B{%gQZ=p`z~1R9MSWJ>xm{n$3FZj-pPp?dgwkqphmvcxTkQbV(g9{oICTE#*egQ>@zz2i*Klri!@X{-e;M_NKW{HWPlZ%hyrQr`2P{DI8x4?!NWl#zt_M%zA_4WVSk(^^TMEmE&X$IJ#h$DF!ki5ck{;iMgMa3G2MI3S3d|hN_4ZTKyF_(r<ELD9`EKuF)$2ImwPEun?-+TqWQI3@Pc2xi`pX_LA(Nj|7D1*)3nPhRq65m)B;<5hYIQ^S4whr!7$#I?##*Z5tFVEqR!e3k@q<jTGd4>YJRX+en6Nd+cD$em!3jdN=WoJ}3iSm==?h&}cKZY>ROK@8?7)%*uzoTXH3*q;PehSC4200tPqR+Bo#ICkaD2^W+^Magb0S^R{E8kePx)DSA|}hN^!fRQ)<OV-v1_INGNK^}*|Ori9~Ss@}i>{L5y(<yiJ$=%$n$&i>?vPria$kPKD*-ZD7@DKyg2m;<wH5AUbo#xbacbypa|ZR*o8?a^LBcMVos8RN3S1$HokoOn4S-p`^VxZ%H5hVD_}Q_5tbO)=;ja`ay|AaD608~h~pBk$MWaI?(Gvb-S))KGzl0>v2_n@THFhc`i8mAkHe+Q+S|M}9buYra*4GRMGQXJ?wFMuG~5sm#T>mZULgt_F5s7KwC<Ba62n3AsV(C=L#pWsBveC?+U0SSjt&H0}N_J$tNXhsJ%hk(5L&VFcWmMX3yHhMUpqfK}?Zr_@1q!c1F^qI%{ZU4TJn8Bws;NE2Bvwwl7(%rBQx2+Dse-Idk~L0mWWBlp^8^#oUKoKY;v16F51V3JmVUDrDz7gZ%JZKfJVIr!ovD8oGRR@l^dWOI23c5vcxd20((f{MZ~TZJ>du%nTREu?h=q`&lkM77>`K`w{8q>e>xDyyFzgZy%!BNa8e^J@mO78K`EL5pfgt%~-A^vGyh#kwxISx76Ra#0lr!!{?l!!{f@OU8IUToN%^%%2t--(kN>7d=TMx+&-%E*ZmZK{USW5QoKhuO6hz6Sc6lE!%>P2)jWo?qYM_8nh)jeDJM;Z5ol4O7}==Lat^MA3HQ>lapdwE{j^f$^AJ+vNuf!RzPFY$Fo-nGinea{UMHcX>LV}azi=1)g06)qs8Lp<*zml(U<|}8Z_w}@7hYnK>IrE(+tkV!F>)UJhS3MIsl4n`)~;lL@jBI$Tnsklm&0eo9*0&rAPB@dSaQEbfOs3id7*SlPL*nns8Z=Y?)L8z_7w;^2V&D&5@q*9Jf<pdS1b!*Ed+NXFSJddN-QnCW^+XMB<bZ<N_Wdc73cfuzAS9u6u#v9pi&Tws}gZx;olznI_+$uCqdMbwIalR@ZGKw5>!t+}{odvB&ReY0PoGDnBryFHE$&)lJ;=6_~&DL5t6}b4d8s;p{sIevxgP4TKo(7jpk#s@+`4G?aZZl_bS3<L~9I7Fsd~L=<-yH%dAR3iJ*0PolgxBm^1Um@0!iB00fA^`2yPz8F)W;mC=@zoIrtS1gC?QS5lzjbf*1$~q3^DyE&}0Wm5MLN}cCsSUcQgaTQU*va@%#XNT5*AO5f$Fq79*~wGTlq1_#>XKP!b8V|t2(8*ujkD7di;~GTM51<Kwfa0v*hJ(UX>={B=X?Kzt_LHyUoYjEB4(k}%>E2-*?!^YzGFENB;GVdGy3g~;RmvJJp)$DHdL?uPQOj%rtSXv!u$7UyxTtKjj%OgEx@iozm^t<O4w9Tl^_~bQ{u6MHGte^1_PX7VrT@ai5nkHt%3`TSsUZXDeWS@0W)>KwxiWRg$wJ-*{v(UNzcf*a-PNy^)QZlD6G{n#dhFZ+j<;_Vj8VUt%2(^r3K_)X$#Wn4@QI-x2Y2|dOqhOo^9Ov!ZGWok5@lktoo_q)XyVEy-j?2Q*3(c-Y*c7e#R*D#Us$INK98OBNgt@8ibkI&10hUVdG4(@W_%*u+4LBp=kJDAuF>dh*7aG8;aC|=H*jbmVZ5r%4R0G7sRTZX6Iy8j%5HMtSlStvKPeaNzhtzadX;%rr?lYF^jSsIBraH_5Z@^{MocRpTXvAa)m%ded(x!Bb?8kkDc0Mrnb_}EpDl<O&u3`kDxn^z6Mc@+Hc0eyBHs;^^c6p>l=o386!g9#EawHz0?aILE(4X7Os=5q8}^}GBRg~WhBRwZTsyG>0g+rFHF=gl!=<nLY(mwc8L2-HW}HX29LOES)y@E-^5l>5D4yJv1X*fphXjxze`G+k$UScKYS{~=Gl%|O{Rd6auzNiD=+CtMCRJ8OH(~U+=jQ|%sax++5}j5s?{aEw%qT#lt@bbu(*#v!2q{D+6#w0)i2vv?HaApXZ<#0z1yU26lyZ5%TqU&HFi^c+KTEhYG;qPj=(MILwVudG8f*13-5vLJxH^f^z*4mLcO?0L6JQ*7XP}0OLVt_b6dKoF<!}2)Zlxt^Ch))g*+R#$*1dfXz4*{<Ds~4In=ovknfl&7nAZF&q6u#I&>>eHp`}E7)8wfYSg2ALX8!vlwXa~p3(Bla~M@E2$3amjXZfb`TYZYu=r6H75rH$DpgzLIWhAjxNI`#p5x@1<u`dgxlP*TH8CmMI-5x)*mQ@CFByg@?<22tKW&<hb4sRRb?l|!$)A4b={LU(U|4dWkN@TmKK|LiKKbDLF3%39_h}qD5_28Y8QeF|C%uQ|wXx6gDt;WiWMwIKm8sM9TV}YZRlXZ@jXc)xIDYcIAAItQ@BZ=|-+J<sfBxhb|Mlq){>PKQ{o|+q<p&>s`|BTl@bwEn)S3KHi}{hx>b3gS@<Xw?+r|&Y3@`0ZA)nIoW7`7!+i}T5`7EIfBn$Y0oNbtHr)T70Vl%<(W<nJtL~D1nfL}-q&#$F(s3Pb&8-A6Ma#?@R#f^g_{h4*p7=D+SyeT!Vm$<#Fg&D2M`YAjnlwQL#!cqnZY;8g*sq#V-dhERC=u+J+MNwN?*0^#i28%UjvwD9<w~il>?z5@N)lCGu?DO>Sd|I>pl)fH?gBSGn7&U*jj|kWk+371#zBfkRKMUW(b8#P>x7GYv82Hc2wEfvM@3%3?*X!I#e|N9F-tADd>tdqfW`)$&WiJ%eXSH&CntIU-tRKAa3-w~_EllX-846qkf4)LjL0B`+oP+pPK)(ng*yjM{<#{AW-Jz*x?$=Wkn>IelMcsg>@?$=`!%*#zXyaEtM~|fYIzo^Qinll?vwHUx@BQK%{L&kcOWH-_I3uRnxR{QTQ9iCpt@%okbw4O)Q@SacU5BNayRj@?ya}CAE@5NW(GmEVwL*bJn`?2fzNKBAhX9aGb&$lkWj`HebzNa;Z5+a5vCHI4P^Z5dPl05xjf7J1PhEYBp%%D~(Cm(b)fVX!8=`!)XAJsiuX%tdwdT2uj{@mz!@~fbdUUY*SqDey%NfUphFY7V=w2TVvucv6O6_r?xZ_ld^-IB%pZxRF@BH|qpMLeDpZ@U4cYpZgfBwXlEmd6OFdvryCZl77u`ZrsanBI6W?z}4!7{IEu<vMaU(eujmk!8T7btfrtgLI<vGoY;mtbQvDW<bvGT8aw@t;na%@WM7oP`bOGSKS6&*L;_xYw9H8jQj!LbE~@)2PnUVUIE<%C}MNZC0Y3B)kkUM3oT<EHazaQ=L0KXTcRsU_(91mOKky@97`bU3rsr!JGs=<Bhw96cnO@e9<c1t=%YynHDWsr4AG6(@caH6EA~v*l2-<NJoc3Tpo22KWc79vb3Gv0$yK_meuoFjJyzpy4W)<Z4EVACxJWt<ps7XR_~FumBM;|@#22j#r?kL;mx%*v!4D~U!Jxn#<Qb_OVx5uvYs*L;+hJsk+1vM0+#Nhul-sF%^Z2(hkA^3UhsWr(JPxf8cTQwQB%g860oPs!`NTkH-MgR&5-Qj^L?y=<r+N4`&w-5p>w=%?vH!m>I(R{LCjOf1hu@FOjW~wS8(ORs3~P-hs7WCoy~j;7~(J+7X5s7QhzH-1${m?<FiPQz?(2b!?+J^cOz(;8D}HQ(qVvVGIaQsUBk`+=y;6}V%dFW)`nBXaX(Xw%%>TYy^vjFfs{UrSwGI-YH@yMhjXzNoSOloX=nQE4Hqwz&EY&O1?OeFIMdLagVEwsF*sbDqJaO>vdpG4xpZ~r;~XG{k=(oET|MicdMLUm6q)uSE$|#(Efj991znQ{vq&Vl)Pl}CY8(My1dDrv#V<m@KdT|&t|;%b$ABxgGjZ>u3<7rywpA1U^-o9RHfdcC3$EZ2Z%jH_<V<{eT3%g*U+Z~xK|{&8*hkJ_(l~=f<y?&*3$T7O``1nZM@X$i*fA#zzHFQv4tVP0LQ_WS#@-o&C4L1aOnbxBc^ITlZ$>)33F|x!PoMf+L3}+^rDDj|Go^y2Je}v#!b>TJ1k}#?Rb@OPAM`1kE`SGvZA@Ye3e=mu6oiLl*Heo*u9GATSZ9SnZ92lLEIb&G&mLsi1b^efuNnvvF&z$j-5MIhz88cNvDNA~Z{p=(JzM#F6m|N;bTkPat_(aW(gV6jeiAZ;fkV6$czlbk<esjEs2udNQJaklYkx&8=|I%us?_uyho{9;?fg0C?8HjiQ!6<mxe|4_O*1$%2H{oU#LRd(biWvZ#DHyaahOembT~x*1wo3pxyHe4e_m3=CTx|5CqXfWf6Dm;UHxUSQ=OfG9d~e)&-R0HdYI3=1heFTFeZBX<DIineR;>^bQ02sR-I%zA6e3XZEH9+%rH&yKKQ%E>=p(b<F&cj;Kp=XOvBxWOYa-~ek?8ur_r5Ep)6gzy_nQGv%uoaYl|sIKY@{p;M&bETANJQ*;#<<irr`D6S;k?K61b5kgN)7fmnSDj;*J)yj~Z>X4-`Ae+tq14ib3|km<MEMlH=1z8rT89U55R{FT}f?};9bA6lPou{-B$z%Kczo$>ih#DkyX1*OmAiU!X_Y2$meS)sNTogZem*kkas@?}sY?0m{buu~L6m;&a;WtNR5NYDAy>o&qepkfY-=xW9Bp-LJajkjjaLfsp&xupv}^+q&%Ldv2?`#CID)W+*-r*$s2Thzo!-MwR{^~CrAR~vh&6aeLbaq@$#86R!?UFGXx_g9!ZWqv9WFA%3<DHG}Ty1W+Cz0^j1=S_J$YSJj$5xJ!t6=?Qb<@E$u&X5k!Q>fEob@VP+EEdCTLWE%mG8hD<7*t#_8BwtRqCqRjRK*5^NMMD@+byPAN+kiJ%BEEB8+mMPwQC$JPUlMP{RwAwTr}FZYH#Z`tX$R4g)ChDrLv`c>y&n_#qC*Vuw$Lwe&xt?qnoch1s1VhwNHH0$Utgc$7ch=6eFsNy?j=B7<o8K_Oo;l`mAQwJgZho$0$vU9c)OyOsFh&XP66FMP^l9OT~uIu`SXUJbpLOd^pR;#VAjQA+^X(4KS?A-g?w20sV~f$G3|McQlet_Y9mBEI%v4Se3Buee$MKJT?vnD4#a*0L069-b$=;%104NxW#*5w-lYriclj#pr-leW%H12-4@gT%>qLjmGfrbT}}BDHU(7v=wyV#Wjs(r*A-vVEKJP6)LXZSxiihu1KY!1Wy!R4I%v;IkI8owjNqU`_7B3%9AhIia8bp7HrOq5wSI|8XE+SBNpWpX#I7H=C;k1jW>-R_$-k0mrx&&BV$A&Q@@5P3i&_*!#%MBoK05++g6dgJNfx^km}-88yv)ZiI;de7SLF=y{>&5rNdAL3IK=VNiqp+$(8`9>Jjpk?Wg;#wjeNF0&0ukW66UZ<r5dRwD+gGw70CikR8rAb+fZZsKtAv0!(m;DxoR&PihI#^^F8z)CHyC@9z1Ysn4Z9B*!SPifzvIW5BY64^j^g&9Sk_vIj+XRk>NR`ojb9*^w(kum&rHsnYeh#v#2p!Mo&QUltS1&(THoK!W!vq(%&stiRw<7o_{Jjhq~vR(vF2gWp9&S=fvSA*g7VsZ8jSIxymq_bkg~(=mQI4&f57P#ay-7INcd$1Exl57M~pw80)XOaIyKAhTeNWs;P8?dTMqhve{I3ES-V{nJ_i^ubKCn`Hn;GZL%L*1^~G@^r~edsF5gMHZyW1FUVePS!)rtdKN6T2s@FjaB+SbWSGvn)}*Ylj^=)_7|rxK{925XeV&G=Uvyh>Prm`LRGwb-(@{<;Kt1`(b&|DGf_mz<&S8a7G~S+>K2P(0I5*sufby;u(s(1KLx=HaXwZ0bBVOBbRpgs&wr5*@{de<G0@(lVmE{PvY4tTRGJVjGa&rxv0eU!Iipk_O%*T7HJFr9!SH)v8pXI}{GdWq>Sb8bAR!mN&`QH942>VgcUF&v1>+ENsfCAUw8uvT2o4`$Ux(m8b4&5j_566y*uLf_NfWD5(V{m&mpo*(>6_Hi<{=MtJy>c5kd|YNLw?Gq`VJi6PMsV#~@W$_~+`SgOf`_1A!m<zcVQaIC*5&=Q40f_?96(Pfb$qaMLTz4eY7_Q4gyG8Ct1I2L4u|{}pr4P^A%lJd(5KljOG^Mehb`h%!zSiaa6b!n=fffFy$O51rosAlnC3&AfdMGx`?J}k+*n<Ojr{(62Xw&E>MkehG%K?dX65Sj{K4wZu-I80rTKXE_N{9-?%u!AnH|qMYR6?)3CG+*5%GbUk_mh|$Pzq3z$l&VE03UePEG>o<#;0Bk$`wOrgL}nTO=14kHuS|S!H-NUw@V#(u^oY!K>FEbZ_3pW?l+bPWd0!?OqRH?~$$a3;Yv^U2&isOn51HC*$@{RA*lbfP@>TpfybdoB(ITnR&csNf)-L`55q44mdCu1Z-F1U~ifaC~_kX`a>>rc#Il16SGCBx`UKz;3a(HiSQICK$v8QPbAgl&DK=n=oFP$l~Gpi;}ffN#L?c5aH-t~ng>V0UXkHMoR%}QGTYDNa~!0{d8zEtc_I8GuTR+Q#=*Kq82MAkM)e4ZhsFuAXF@m|)aEcS;zSUfgqTY0;7Xtd5vlD#hhV4HM|q8TFgD?t9mLsG0mOl9bt{T109y(P6tS!vE5ZM(;hma(afJD?9Qif)--0e0c-AVz$h<^xIou73T_wl}Pt~$(K=|;o;sA))4G~&zhY)7l>oTlheIhOi-@Np099&-&yoFGYP7ih6hAQDY28sF*vqr@btVb09eKhDYaCIEk+l~W#7Q^sxiT+O@#Y$|5C0$aT8L2?yl`>7T_(qVD4HrjAe566z$@n-8HUbLz3h%_c2LOHpA7z(RG-^15>W;F#R1*;<>2*2r+AEP-akZyFhT<I^>UsxWuCKv=moBk)NPZZ)SdJZZ63w4Qsa&PPhFR?&!;s&`vOJ0hD_2H?Wukq|S`&4MF=DI(OCjvs2J`+*;S(HFE}`-<S{fszT2-n~SP@5+cBv+=D>2fFgHbq7!0<5M*%r$O{PfxlegsZi8()B>u|EV(w6F}v<IWuzqhaQnLO0k!JPbxlr%L*wd<+xfFzbVQ0#uwJ;^P_9K{_@RI<3z#L?t}ROgSVO2fI^dO+Kl-q4cHKeO22Zm2=ovgi%Mrigd$qXEdBJ*VBChI79c4VGtO1Zi6b5PD5X#25vif%KU?7@DH|<-~O%8@t(J!!D$9+cE*Pw^w9T7#sZZ%82wQ;+b@(*ApLwL)(Tm~CPgJ;lKs3G^hjE$WLJk0|GPHLV4|UBI4_HQ0J^$NEF}R8Ho*Tj$i^MaP_nMCTn|`04$9ZNz(@nokx_)8ehC7Dd>%o@0yaBt10DPC!9H*uAbx_9d3Tx$U!&D9tpKQ#SK=Sq#ut2pXVs{vu@Qi5nN5pfNfk!Gc%%8K8UO*w-Ek&hpx@&lyo+*2>AP54fPBu%Ce=xE=mla~P>dmfOXWAqjwc4oui%!WGc5MPs#}!yOpSV>4Qy`>Ef#|wHl<rDgoX->34T<VR=Gt@$rPp>SDx+>a(F=)`oXpstPn9E%s1(+;x<sii7HnhKTVE{0YZ#~qY~E;da>p3Am^lK;d*Nxvvs#o#bT%z?y}e;`HgNm14YQyV6~v==eCuLDY^X9+%~fWDWRUDwr6RrNYMKXTk>=^n}$LpQa-okqYUi=%ci{v#vOGnaW9`zr^9Nvf|1aeMmSgJuT?k>vzqK8uLoSC)=+cgC2Uz2n1rl`O=T*`7H+n*-MM`aChM41C3{(OEJFF1X@}_XLs_6aad8bog>n-tv<g)l@?;lP9kv_atUzv=lD@yrmMy+;FuWv6*22*5YQ=dmwY-n*#40pH3yhu?Ckj@sXfKp5Yt^LTxW681AEJf|8eJBv3SIVA;OwA%R@ven7qmCzbZWyJ+6^@F{uyl@UraS|!h`;~KNz)@Rxvkv^<<ROw#?(sJ>gsqP0vYnowx@vybVV0t?64qS(Jn3M=pJ4Jn<_tj4Vu|l%d9<`X?-{685Fw7JLRpj&@vVF#u_h0Xm9#VCMvt5wucOqSs?b@Y#k;D#vxB7Wt^&!8BR0NKH<XkT0CLUnL`7C`(0R39Ol@S_X~{i`n&NH`UTyxOlmr#VcOSffry7REO&>=E3>T1KZj1=qMX^6FQ>j^|ZW#`-<RgC<(ecYz2_g_~y;|-X1_gHw^Q@%;4*@^SNA|pgVS@`yzN;DqPO$Z`o%=M0>}o>cepOT+Zxo;jR+oM3RJ=^p40^ya>gq7P!R|Nt|IX1T#%ITTUjpQ{dm|zIK`9Dh?<$z2eYm;k;p?aN`bLrbqr1UgZAudvBm)ay)=(H<+Z!K{mx>n7(>2f3oAm0;9h*YgbXZqC6cn93+EcmT+^OU0ZcHk*iWvzJpWjE$-ixRNn>i?jsZhDe)#bOs7K-FNlb&AZBWZ=4eN{Md?w%;1V1sXZNbWP>5uD3tIr~`Q9{pFa@^C285^f?VUUK?_zM0_O5_k$;k~PCO@4_h9}9Cz6n_gpCTXjET(ME-PBqasFiE<9ld^u^32G`#wi(`jMGse0@YY-2FhdDjE-_N#=g4V^%PF~{rPA<q%ciK#aGt8@EXo6gLmXcxEM`fmWBFfw7$ZA7u(7EJemPJ-CKs0VFd#pX7h2j4CGh{s;Z_dln7O_2&q3+4I(5Ka%Pj!s2wn1vjTQ)#WZsiD0YK1*xY99g5$Qm)d_CR5ZW9?3RGp`mNOKZL8@>E8|MHBTujM&0A)$`OtL-`P_U()qv49u*6+fCj)l-}#3}+Q@)_(;=hF<@l&xT?R7aDwDy1y|OXow7=0`b*-IF{ULR+xdk8-rj>K!DcQuJRasIPqPjg_5j20h2eRqfqYC`7d)pJB*W?V?dfEY;U^qVBv0-9o$Yel`U*qgk=J4byMNrXMT5%@V}-(=poY1i86|O(7Js%fW*6PL>YvJy#EkK2evo>2Cv_J0l!@P<3I3Q*nv-)#;uZcX?R!(;cQ%hTE|9-;RSz1()uZ;)?Kq=lZMbU$|@q?0gvkz;v6QFs~3d2r3NI39chAWQ<)w3EIh~%m=tj-ho-b-JdjUoH){|&enHH1M%NBJ9C1X&NdzThH^*ypcQ~HM^R+3xV^t_(_@qsW`D1nMpOpS;|3sW5ncl@$_6-}+O^y5uB|sW48p#qRV$*mvDM36v&p-1tav9mB)pgb5O`3nirh6gH4u=Zj!xK|4TVaoGFZ}uS$!b%+GAIpzBny*svNEuTf*g-5?`15#T=xTGVgPdmm&D#w1B<iAb1-OUj%Y#2N}6*aJ>WVo`cZau_8@KW}@@`4xSnc-b9%q+X%utZ(fCcUEZdp^=&^`19yP?_SGmTp#1a*6H$Ee?W<yq=>+fYXU5SH5Q2x<lrIHzylDm73}F54A5boCBw0#es&_HtfIXxilAXkBMS6wk0Kc3~LAO(rXwq4dlVFsk;|VCh)cBAI;zH~S*bw4MM0^D!Nv(B)n?lMCU|~RwkkCR`@g|hD&auI!v07-->2ZG8DW-d?lfmw4_ww4So%PGzS6=PFV}!#FTNjy;L#(bbBRS_tB**sU?smr#ZD}7=#ck;AHuNIc77A|-V{boPD)Zs|;nKFv;K10$cx6bY)}S(B1_Epl%3shLv<g&=#2jP+hw8i|AB@<hR*s(*bMJGnecpu~03?@`JRHa(IoVLn`1T3dbwpd#471#bIf#R*$!TF=m?q>E0;Fh%UW3+JXRYhP3RRl{Y~$(3i&VUjB*28^tREzp6D1ucO0(>Ygm@P0$V?&_IX6XLp@Q*rR1%Z~ebUXyu4gN2ueq>=>iG~*B&x907GVbr!~830C0*6#@u)Fr*w8i}4yxlY)R`^r0x^K{0T~B5y^<|_-U?3#ya%;*hBJJXM{i4kS7QE|nM554#sUUl9=$alo|p!!VRi^wC%SO9`Gz4c6V9ik*{d^ZmS@QkD7!kRHuaV4VBXHkUM}N8ieWh@w0MLLHm3MP6#)jQ4Ah}7CRqZYK&5Rl@aWR+tht7MCoReGoEHPZq3;y;Zn%B2)F^J3ku}5Z>v>;q)mB;9RvQ)uRC3d5Lm+_Ysynj33j<17wSi-HQq0+Q6l3EFpTKs{Aw98Rb_}9ZWh)>B5cmM?K$vO-24mYJkDyyesD#4C4OVCd4r;UG0ZgnABN`2hozd{baNXY^RDea14!A*b_z1Vzm_m29x%~E>&>;yk-&r|Kr#Uh+3>=XRF)U$gb&cuJ7*hbBHkUasEXTnz6Xf`d?`HAmNje?VM<5C!u^hYGT^<h8QF?iKiyOiuU}#N@tXwz=BdOzvO^DTL*lfX0b=%3szW)D~klr-iB^Q)1;{9wgtMKYDQgLQuO_3f!EX2337Ps9RyPF*q<xB#j24m1jWDkJU$83pa#~yUX+Xt@1Y$SY{DzP|Us*Zz8?gk|-#UcmF{tDmi@yafeifW0ExgD!*GpSXEk&HrfO|S){wu-bPWSW~92J<Ne2)Npu@GUJ`lpNiwbltP@4r6=X1JNNL$G~(#ra>b|!B!;2M(ApXCDu!Y9v{rdiXsQvohqN1#ww*vwFv9eCkE2-Y36QIvf?gTNXV44D-6W*y*-Sp;QFzPuxQ}`nwGQ0D2l6WrBHFQb^_^mB5PE}<nT^0N4@BT?cOkSc~T6S@DK6qx6~Y&obqM18%?DC*+|^WU2k8@VR7`F`btKCXWrL!S64xYn9h{Bf+l=+ne5i7`n<?oL2_X8Y~4EsEW_75yxS<XK%u)c2MLVeWr#A{b(fW?kU2t_U`<QKLgQ|h>@kD<Fjr>X*lTl83U(coM#_A8gvR2(E_6Pkpr}(V02Y<S;&orAuSVG;CFBt$JE&Hm#Hkvy1I>Ndw>?)pvB3op#r<13%qc43C?5}?muKGGkOmrXr~!vsdF&LAx!LgIrr7(QR)X;UyH_Le8aa;yXOl4no7gr?6%0mO3u@PCp{UrLgCWxdk_pfRTzN0vW#~!^TEu4kPBEJmBT-e<VyW%Oo={BaEjgrJgs#-30SFVcPqi{@xOyT0&~{Dk)+)MVDx!2JAA(3BI|YNVy}dm-ne7Wj>B^NWf#p=y{Da(AY4`?3T_qBPSqqt~p`4uzGnW$y>*(9ep5|<eYJoHTo#89o8Y`cBqrN#-si*WN)JhsnUhTtvpwxY3Ejz*-EPYNSPHo{SWG|=)Pi`DMTq1X@@FN~%CwLew$@j*^G)vN1mCX&)dSHT2JvrCXmKM4uBAvVmY%2FskGYRxrzfU|b1$xOne@md64$sldY8L#t@FVyV4AnJ0+lhfC#<7!qZzYDX3Dt6rr1M^VeB^H_13c$wU5%iX69dWn5C>61}XzlkDGsDw;`uzTuNrI3KneRvoUy<-0XE#l34Pnr{m_$I>(|X^eqvLjf|||P`lbdJvt!rny?5l%>OnYW&ndG>UCxWyp5kU7%b8(0d3x*#HC9IM;L$a%57?f0~!}-S2LsfY1m!V8KMTI8PVw0WU=0uXLPK|W4&?IFz1uwouih^TV=d+_Znwj_0uThn}4477RAh2X}vLOXwy`EFz=C)Q&#>mv$i<quH&8Ael?q`pRO`uQmYRR9E}-lzcT6=<!j5@^WS=7)UZW@^~QS%m;7ava7qi;OX~tJZAQBSS-DJs6_WolzN^9oGa@RF_8B)WS=gyI{-DMMRk^$Tfyy|ic-pQFqXnF5%tzA;-apu2x%eo5OqEbkss7lg&SvRi0Qeca39{gHc2Z$o6|=&7xu*wHJL+ghHSkK<Sq(+zeifR?g)5<8wm~Hs;fCoHs)A|ND)LIn+It9i?KL)Nzo|r5vH^S}Ewj5AbIYOx--XTGJ<O7$L=Fnvof$~QF_oLBv-9(FETOij)cz*Z>VWwqt&$jqEFn<LUMf-TZ^psxIJk~sY*c~~L^M0#iG-@p%Ke~kR?zU^pMK(4J=dl*&3^NPW$RjW+&!X<+#7{@3}j6VKU0lkJ_+-|u|gkMCIw+?7j6qiWV0IgC31maun3bHLTISbmptGcRzY@z36~cRN?m0l2fTyxf#<wgV>djG@LtclZj8qr7O_1$32X5)3i?9bFOk%h^4B$=g;$_v{)%*7R5UbUwwcHpm@2uo4a({WZ?xu)0qtHl@3rhK%M`yXfQ*R9sZJGol$Hm)jCS#H&3)#Tj<La_x3!va;DY10?}Q5}?U>pl44{6`yjnKu9Ol_kh_~5LDbh7b?+Y|;jhy!^=q((yZo$+so7!S!_0|s0ueFx+qZ@!6CEC$OtzCV0wcanPOKMm`3O}3Fq8>Z(aYc;R-fQcF6?+hl(G9{<Tqm?zXxf@DeXKrjVwyKK7vIS4Snuc%G&|4@Kejs4Y?8rhwoW;O985tWW6**1=oF19c}d#OAU|h2-kx)Jnj2iGOP&OQ#sqS_?U02L*e?s$5LuqGP7$w_s^M&+VO-7%z6G65+XRP2+Z~zDPrXJ(q><F;pE~hWJYr+XVMpKy3vsQKBMtl5ImaE;jomacTQyQ0I{YT&DN&9Ng*)s#QQ6z94i|N9+`M_~+N~RRA0$^FJb*v9-nzTx^TOy9tUD?c`jd6&Oj(3R3e1j_W#O9hS`{_r``MxB4Aub055~(8r-T`~68T$fEbit}Usa&p@<<;;HNU>~-JVVkMNw>QkRj$Hbv8!9ic=8T8%e(=pzX|Dz<|=D`7A^63_<~OP04**o7KgJuCUdjuq_d0L)X}CQTY1M>BpxU@0(#wHcbng2rX-99c!w?Lc)^0N2%{Jm7QVHKcKnPGxU_&gy5J@mM)K|RuFA&5a)Hbh%pDod2uhH?U$Qtq#|s{=1@^^D3(odtQF1D1l0594dAJKp(*J~fyj}G)-L7R$GNUFa_{BbrW$KUS6yG^E#kg^L*1;ksc76s=P2CXSX6g~ovfzq7VV1u3-j7SXm@3i(`j_7U8`~Z&okS*Hn(&BJJ@q*^Ru##_0FTu@@{s4osXr0)f3*axoL*bX8+=VqV3Mjto-JcLB@B*KCif1^1*SPeP6;*P;Cj-0m$mL4odNu*uYFP=_7uz3aXLS8i98*I&N$<7XDE--Lr>L>#1sx=Z&?W>+a@!y?<s6Dn=LAO1I@=-j;x@RcXuaQ6mwWuu{vm2Ij}i53bF9U=@$mZgAYJ)rQE;BB2Rs6-D*d*M##^@~FBlUAtos)Z0+I@Km`ol3&sl2ek1}IZXk4&=#=gCRGJ`gWl>+24XBJk9w{}#UNkL8Y-j2Vb3u#)=e}OaeL}RCBE}9roZ&pJ5cK#G}Oaz09F>-!z%_A(MQ*pgf%q|sD>#T-PLMB1fs=vF3)MF0*j(s=dHibre<aA*%P`RRLvSz0n;aSi=9>+SXX^|9lUpn2NyoxbMx^U_~HG)$zC=-Z#>Tpu473$AK!B`LE1s~S1?0`Xv*uwdD>D>syVsC3@!x*;i|ORW{}URS~1gKv~Tb5Ik#6dv{yqj8y{+g;y5pY5zapt;hX~x6gczh;w;_Bb-Nhv`K)5VE-foH8;cBzse2vsld$(U&FQG!w)~V$HcYsv?T-H0HLvFP{P^2n|Ky+k=aWDC*2jPGcTd0h?I+*-`jel3)yuCuX3}`O>Eo3xKOiPiJb*0zF)lh~Sbp-0Z+!IAKY8-)Z+-kXfAHwwu$GbhDAq^OV1=j^WE*E4vXG;pgA%U2i@kC7<9@IJ?lxRM<_D+3+_km#<9@gR?k>oF><>?ceG4<+xfrKy+uym{qFtuJGc;l-<W=DGMx&FmmJJpv-@&|3{r#|M=n~J^Rl?NkF|Hf9#5r;nQ01EdS&wZS`Q)?i`|F;5@Y*A{SOsPH+O$|FRMj0pxR7A37q>|pdF%F-Jd2TNi`XK?{iEQhUbj5B#-Pmt^Gt&ylh{3PPv=l_LOZKV2~?OL{p~Nm`?ZgL`bST{^W!Jq`@tu_`0htP{ozL+{>jH5{^-f?{mn-oeEp*j{^6q!{_I?1Dib%GJgcZmcXJnUm8XoWOrCLIWpWW&c@bH85m|W=S$Uq3l}U@x%H#~Ol}Tf8W%4T+U6~kj<0V|##F|5=2gQQZBJeV~h`jt%BQO1C%DTo9cWc&8)S^v2)aJd9*PYsG^HbBLf1gn}4LOdc8#WqVA8B$Cj`}Mbjw+Jz^d2tZ8ULl&VKvKiuIah)#|&?-B8Hm^8^fEnm!Y}kjW%Zwo<%(DrxO$VyyIe9)3!xq>}jK8E5%A3d=N$TgD(=N_uTItN+&GyPyg;8Kl$<BfAWj}`t-m3WAOAZ|MjDv{t&qL<G=X>ym)6kd-|OpfArI@!plePj=Z?qwY^P7fzx=|8jRrQfWiV^x5mBw^FhZv{r|}?zWd8>eCx^A{_V++{uT!Elkfi?b8`i%pDTF#OJ8>OgoIN&*4lK)vn>yKvISvUvUJPU(W*bb?zjxO8oq9^>7YW?8orHty$8>xjy0-Ylc=*V2dC?c4c*_9gERKT<)m@;<?wWUal!ZZ=J1TYc^1CD=i={sZa%;2ve0V&z8!oic=F+&!AkSwTYnXNIe^Lh%kO;^<ke|uESEZQ;rm%dLX)RVt9CvDqOCW|hS4lE>Q9Q)60No5sds3#AlPD3sGZ}efmj!(kuFXnU7SX0e;SFhRBkSMLTKxZN0QF9)zh3yf+g!wfNs_g|NO}x{s}0Z|M~N81qX+?S^D=sW^XQzE?pd5dO?mZy%bEQ`FIxYK3sZxBY5BIeR&)QUxM!z<>l8eS6?-8%)UQdiY|^z{rVo4YG<ch@HEw;mN16}->IE8x*S@lr;b{F{j{~ip+Q0Y^*mH&rS%R$Jl>&1ywCVJnR$Xmy>+<9TTowG2Z!(<c7c4)N`C4(t7cRm4JY#cf!p_Z4HGH4eUN43bsaOav21g8iy*_cM2X)xuO==I>%7>9b)sP9iuPecpTcE6n5RSgu!XmuLJAs;Y9-{<wU+x(rMgC?5xAUb?oG5P-ZP%f1nCv!dukAwFVT5(w1Iy_dj3zfOSaN;IT$b7gQ3#$=6pCj!OQN)nSOQh8Aj7592~>i012erLY3O}T3>6-5*Lp7U%fn0D7QRm+8aceKCTVJ-XP9uJOzzHO|NSyMJ;8W4yEUYZEU+F$M%#>YYp@odFCxp2Tz1SxL(G!8fnjXj%)Y#J#*{qwLNPL-1B`fFXeEvulw~vmtjQBTxrWildn&?CY)+|G_4O_?_DiaWvy5kz#{a~SC58yX>C^S6b}~IuNbS2;vW-R1oOrt?`}pdO4}=TUx`v!Psg!k?@IvpHrF66zBTe+nPQz>s@+E^{W`Zj6cjSkUVIuGE(5YJ6UPg;rEAs1`Sj*u_jr-&;Ti2yliFMf&Qu9)k{<-0eE;`8{)->T5G5Jv;76o6YPw-By-4h|)v7;m+9RLB)@c_`x5rP;xXDjm_+5V8tm*kaxa(U)@A*EswM!u_o6ZrpW7hVC{bz_<ZT*vHXYn|lTsZyQMjh9Yv#PDZrPbVj@do@Xa!FC&7NOj!<kuMubIFB`>f-RovphU<{#G!>`O495c@SYCGnv!b`)~>7V7vAb89YI9IT_{$Sz}{<XtfUHR()DLIMi(0?@(?~#M;KvM`t)tl3Z9eFD#qqVcC>25&wz3=SG}5t@R=HA7(MCs_&kqowX+VN>dqG1l2iPXwN3PPi?|I7c*}Ah2RA~tJE6XRs?q)d#MeoZtBlM>apV-OhYX8c`Vj#`?tvYt+3ysC%oFP(dwq8?j+IqtiY!iR_EpNxN#Ap^@6(>F5<Z^Eb^Zzi~L38*spoynBOQk&W_I2EXzF6zIe1#`BRv*_vr!)85<qFU&sneKJ^yBoF-OBzaEZHY5jz@E)1X-2GGxz0rVma@WNPj5eE32!vN1_x;kG2(~D}nczzBChXV7QoD83GR3|rTFvd*=c^^+MzRz9OOV-ge;2`_FTC5c}EG4pr@!qWTz^pV$r?WgAhOY0i8YS7YBRuiU=wpWsXOMAt_omqc-27$(df8~iX>lu?WsOK_JwUTcWvRg&o1A?H=5<v?Kix45JJma*@Vv~BU!mmkonjetF!LI092sFOV-77+to@SRPjy2qTcOor9JuTxX4fJ!c9%5`&~47zbl|vUAxU33dd$RzCIv$W*i>OS>{yiBT0SCurs;T3lw^5{ShrYn(UvED2ur_kZgwhR8RFyTx<2k1MaJ?}EWiR1D7`T`peMOmT}E#Kr^94K<~+*sz5Q89nMNqfOR$@RQXNdDML#P`faEgE;ZzAPtZC2HnwAe5w{Q-NnXyZ%tYH4VmcxYB7D~NMZNa-gn#~!DT*l?RI_IUJKg~eP4Fr{gO7n+)1Up$!?x)iX=0+csCwRA?0hf>{WCE#F&p%)qmt2}rsyS*}QDWENYDuLd4J<7orIHz4;V;%+K6ZZfo8F*OC=k|20_xZ_O=zEph2DSMR@}kDoK0t5R#dYS8GDFbFXv$;M>?ZAX03R_LV*l!08M1&Su?dgweKCa|M90JKXvHqS;JqgRSj3ifx1593aPBV_0tOord9U$xTSHqy@7dABlCF$FXMlgE~%qkL*06=cSPU%c3bsc)CEFrPaQ|Qh!B3h!PBRUq&;(3tlz4ev{o-_odGYVQMiRFabXR9k*&ecS#Pmz4<4?1hB}PbLVNz2j7trWF@Lr=t;_XfD@GQN_!T3&vdTF1_1dX2=MlAd`sl@TRA!o;5G{3_Q}`j;`XZi%KjJJtiN*aAi~A<d!#{CWA4R*MURx)OPpuR-dIK6g0<8mATet<z{}93{eF0}XeQ*|sz&Rc}IEz=ndHdLDxxjK1coc_o^S8E_6_I7EzgbML(X6|Dr`BYj%yK}38zm7qitf#aOGB1<zQeveT-s6<?DMkm<1`w|kVkSaxC<_1+~=q>6ZmcDaXNKcejO(JojfgjH&fu6XK%nI%D@yMBC5p01$mr~Xi=S}Be@*oJJ@XFyKDvva~RoD@Zl1lbPt!X9#(M(r(&A#A<hYYM>@eDytj0i;$CSfkuYKasHJYLSQD`5z*>;PzIJlLIH5x+9m!V_Mw?nATU1u-|6Vp@a$i|&En2AiKH?V538R85ec<?2oEm$&!2u<xsEH-NwW}qXoc&DP+(6<0O=Q(e3rmHWwSp#iu5EFk{K?(tN;fk*ssgDUg$qZ-g(G4SM+9Sj8Ja{q^c>=89fa&T-km)P_tVbGz?q(t`N9@<VGDbPwy^e-`MJgzDwEs(auHSdt8@g{FZh<?CE`a=FIIQby==Hz=6l0@0!k<_b!T$2w6XM3a1B(}DQfXS*pGtlTKARU!G0FJIo|_~b?+u<!JzQ>JHadiCHLXjQQ_6#jgwh6NOA9Ydp79cZeSJk_8d0`ySw=?r^^qp?)`h$e|zP2-iHy-R&EV&E!oYp=|*sMf)Z+_+gXERSn|PsUczQ#m<9W38SG@)7|D*M2f@w>wRpX$#h@4yek*IQu5{Nr9P(RS7iZ9q0Qxi=W-!+W0Z5Z<s^Ai{Tj0@ty*nQcgCWB_z|lZZ>0z1=@v>!DCxiXjY*KEluI@pV`3`IaMyvabh`T_<Rl1BAR-h5=R#tb0#m?#oRIAn7x31l|d;dm9?Y8VFVVGM8B0eSVRyaN#<br%ODW)^FK0F-nPKy!y9q<oh+7+|+3M!qnKg$nOlntdQdiC0a?#;WruwbDMV;=6hn@_1q!d{As_vz`&c?qAcQMmxfo7k^Cu*I_ztPIMF`3qwg&tX<~XNc<V+ccP!K-dXv6hjBvLW7^;P7wZ&9%#sN=H<|c6EM)TA}C=bGDm@8H%NocZM7ll%Jx<#xHUtda~ySmkdI+M3B5<ROUl}1wSq&c7QpgB{*|=fFQ&LF!dk>AldR9U36%W#2uO&Rpk%wW?Vu8DBa{NK<g=^~G9-i6WqVkPx2Pj8rv?B^=R=r!M>))qlRO(jTS1wRU~Wu}N`w?l7U(%vHIV0TS<hITn|npEH_W)`TG@eVurf(!{r&aVR=V9+wn8!Ol(_~xx>{c?(z`39=rw(GcHV>UFt+3UY?=cfA$x6Z1KZ6Q+p*%?TV(#|r(^Vdi2;Ld&!)vtAuyNAo!}kZ!@_&6m==8+71pM|4YY)ekc~ilg{e!$CE{0M|EBEy)RVp8Hf*)FVVhTQ#q3gCjTj*Mxz|2#+D@+LfKW+=^Vx|Q6){LWs1o~(uO~$@6mL6<;x?XTAYcZ_vF#AKy!N?nJ4_xfsn8!T1!3OFI&rX@WrG#2<VS(@-(ZyomB<qvQING_I2(~&w@|%1AMG#}W{?D(S|d>Ht=PS_XF);23K%S8o(xts<->yH>F=lGaW-s5>#`p!34^yvCMuB9pqrE`>)?xaJcCo&XbzPZR(G0}#c+=GB5a-dPp~`cJ^*AwySCO}TdXMp-LWPmthdlqIh$?=FeuyN?y8!Gx6!jQoF|hLTF?^^%=-_Qwq4kai}A{kth_<Bt_PxxAe6tLX+QzwWh4eeE+Ba8AI)i41$KF<SReuI8cHjCnDP(!yJWu!j6fVuGQul(d^FGd2g#l&1JbOVPjHdW1|2i{TSOlNwdg=d-Oyw!YtGSu>iH15OHvusM(ikKe1wiR?6jH;Di2nZaWE|=Ai7&%vDR60;UECeszNgnv8X2EWLoT`JD{`9a+(GV!i{6H<jBqD_VzZ6|Gqc^cjd~Jz|!UXYFHdtI>ti<l48@R<=&;p(Q;qEssgi|3=JKpYD8BAdM=eKYK;_iAqWX6puhkq!F>@Crwh>dEZ7R`u+?a&RWLBV+UIT$W!-N{4rYSybM=@HxzID&3o4S#PA2H5fT_lIb4-Uym@RtA>6-GIxfDgA2Z}&kvwrt*eU59^%-*{B*;vu+0%AevU5;JizXzifyJU55{k7O7SbI)M8T%wl59G#cam}Rf!IW0M!^_OHLka77n?&%sY*1;h`r-_0gf$M<8>5CjFxDGK4Rfp<?;N${0ICmW532F3ep;3fIkeT!*hw!O=IW<W#u#7y(QDckqiwx0YB-!O*2x%}oA^iFT^h$@y4OQ_cV%PdpFuu~>3+j~_U9-LP?Gi5*Wkal`F?h%S#=d>%FZvXaoAn>i!!ArZ)AM!m55z!9Y(<_7E;B+VZwo}ch=x*h{ynCs0qy+wKvTN$-bji>P+M)$bR5KP|-|>F!Pt)+=$7pyoDay3=m%DM6xkC@?dflISR5L&6reWH`$#M8xj*fWa}Cx{i4K|utHELh-VVvmH+_2_k%2CeTyZr^;kya00eN<0N_XiU@~g9mzN+%lAZbPE{3En5o+k=rXNvbdFlg}ryo%(_(uZ%Q9JlY0{)TaoU}X4rV4gj>GX!q|0N%*%}%GoxhA+IOY`v)qo}x5>69D4Qx?WGBv>&68L|6hSJrXDkoZ8hTDQGX00q|eGK32F6Z#bLI2G&=2EBqX-%p3T;jzH`cpch3X4q~7$1E+&=F0k3u(}#_IflpGs*qVkc7x-f^kL-~X8rMcg?@#L^S1Rzac~gJSY3hPOVz#U!C;;Vixc_i2V&<gOh+OHw+hk~WU3-??Qf5mn;*k$AHiq=0KA<$R0;Mdf)6bGmK+ULjRQ}O1HA?(4{s_K5Pasr8&%Xo`^;?{f!~_gDp?x?BO`vCr*VPiZ7!5i_Sdp>_0BqaEH}u?$z$=(k}K?!49#G1N6_8F?jST3n@W37j?#LR2w04(;(C`YN7EDvF<rvwf$2Agt?i(x5I>&`n&lzvM097P<8&|dfIwws0s~6INoGH($PTm9Xw?r#hT}OvU%J$FRph=tS5++n1p~OnabM6_Hr)B(jUZt2u(#c8Z1AKeyP>_qYzl)(Jdw<bgxM>MJcpxfmh#{;J!%F&7>F_eiIeqs$eSIs<zP7!3I@xAUeh(Z!z1Hp9{^oC%==ky8PQs{LUwCHb3Q$!y;#006Orgh@wmqhaNTG=BgF~eU&eJ#-g)g{9i{E|csnCH+_VyCVK5zF{%&J=q<|>x?58YHfx->Et8Am#bK~HO74zR|Cc<;_V-RQ}u!F7QH&HzosSK?eTo>`gO;vGAZNin59+V!!g$_g`OVX$Z{~Y2*J5!@I`m#?Hg|*$SpMT!wT&kNAXQ*ogH_^2y9h2SA@Kn~Q1vRmOmM+jDP_Nm<LpO|j<A?fl3_?*Om~M?{Rw>-muwol%n23#yjSk%o`JtUcQz6<Ow1L!E$TZE=viS`3!(s9WgHhrD&n9Se>!5(rPw(@{9r`Th#Ur}BKGdp4dbE{Z;h?^PdO5tPa_Fw&MNEC{9Ol_ksC88?-^|3rGXukKzH&QpuO7|-(deA|cnYZAL=O&8^9c6VkB$u5peX_WrvYYLt=Lu2F1N827^*0=b8w{5tF{%Zn5zxDD+g#waBth_Ypj_ai~#mWM;u2t0QwqhW{1cXq$Y>4+GK7rGH)q+Ofw+NRf`dAOWT=h_jEEpI#hDUb%cKO94W?PA5zU_HHK|0T?yFaQLlRNmpbs5s>CqBsvi8M4*aEfs2i!;zO9W;c4A80kC7ZwzDZ?Jo47E9wmC$pWiP6j@4NlZ`W$!BmCH2QMUw=JgWA+L)KF;4yrH$f-u8fvWxS6pSX~z^+vxAu=hx&!s?h>m>!MS65MvMB^{!aX3L0G(s|w`sR^Ys(eYUyj#`sjJ_RnmvS;!=#X4ZFWmhQ7)m4icYAikjdUgjQ5$?c`PMIvXK{%R^VwcI^R%NL5QPm%R2t&X&#)*PsvR0`KDHXD6ubhW}g?a8~qwOQV3mMt8n9@A?K+XIeGkN$PkIqET0#;$YJV;%>W0m3APl1S4_dH?9N9GmW9V}rR)+g50Y{{qtW#j%99OZn(n8n~FF0G;HNSsq)-F?PP6TN{3DHstlYZ<9ea8tJEjd7N6C^c*Hst*Sm*Vuj#w;1xUd|EWSrlS7*5QKe0^_>5!oMhVz_A?99_w&PNcg!RqiT(k6!e7DA1K||(oY2nV<;kr_*1Lopvv5me`-w_t$Fn#kd5IgKcHiLWlSR!5hvRz6dF0*GyTu-K8JYTQgR-|cW=dYC2n5caR8rvGD&q)|AyuDFQrMpkiDt+xa-o=5ln!DS*+OMAdc#GUpHFow^xtU0aRXMpPc%}m0Y}TT?<;umVhPn_Sdb2nn5}p=DN2kN+$ik=zP)$0XAs#1GSxIsdpQ@(gTUpzB-c_jQUXWCkI55N0+({*)7O3iSBecrd+it6fZ>eHYv(6)1V(nqvny|XA$Y37-3N?9N3!bdw;gVwldV*ljv=BNnSu<<A5`1f5t}fBfD-pIoj|fUZsVI2TS*V>+#9(J}C+25;foP`BDc6Q`TN<9hzB<y63{#c_c1oC^D=12N%ygQ`pYN4Lb^LU1SnP;XrjG_AaUubKi}RTUj7(7s%g$_eJKqr{9z3`SA9$I2w{DB~w@{0V*`Mpv^oajtg@w4eIj3p7pH8oi#~A1(S9ym7aA@=ekUF28^wztdcYeoY7fHVu7SmqXef8BiScCsXF-~Iq^vl2h-Y0+fFTp!se8X3l?Bc!U^3gn<X3NnA)?DuA)BZ494xWDVuLAK&i7Zx*m^zO5DQIsnB)<BmPrvhDo_z5AAY5BvNIv<|x1N0M2kgUt{`_0;lD+xpr+>m;Mka%ZZf+?3f|}5?Ctv&hkAL<D@cYU4|Ia`nLh#W~Klu2AzlU-TSk!F>>&b_I|M4IF8`NL0+sxa*Itl`o|1wRM5^+38As<|&2qjU99#nBQNvCC&Otal2WsHRJOq_3c(jq+jtu7T+K@9eo!2*f~mgUzUKK$LC;ux34=?s2f2_HTjyd1^uKb=UF&&C5Ry{t+t5eGKSI<PDZ(%~?)Th$SCMW`!X!n|f%`o5Z)!TmIE&)~J=^*A_z|Hr`5lP+!1F~%8+VkYxgd0f{YyZYm0Go_t^RR@+n=zkdMjm5$8a^&oh@FrCt_)RcXxEW=t1$G$0(B>t|P81(vNJUz%e6$y<U?TnM0IY({rxd4DU+Vr?6k;a~d6-+c3{&UovQq3QAI$dSV4oIzSzO<cD(OW42xw(cj-(#ukmn#{Jp8190wQBz7XP10t^r6*0RSav2mc8<WG7jX9ddwY4nrRVES(q3U>lrGapz#rOdHt#Y!0-dFlHQo_;HDKKUfX$^g`q;e<I4CV0libx(dgF8{tQ*21NOjuA>28A69y5aGH*<_ZwjJvbUc2=99q*9&St0;bb2J*1}7Mq>DAt7+K-67=R{aA`{rvKQ&d|3ENfK*^^6`Bb^>{TPGWW7F=fGrrUZZsTx;AjO&YbtD;@I<W`KSGKM`T@nS)Zo<^0Ju}ZJ&%4zEAGheEFeV%6hSvuYues+4T`6}kanUpdFfMg~roLQJN9js1da7oMnY0_^ws`Rw0Qd+!|VJ4I8eAP5mE&0L0wfLOXI7+~y))oJ-`cMDr8&Ce%UoWZ%#x3+f9fO@>IN-q=RkKgO^W!Jq{{E9c`tKkA%U?SxVja*!rrCH8Mt+?tvcgY({sAeB>noQdUyD$&ILH#lHfw6DfBL7N{KvmP`O${}qM<UUn_QeWOel<Sel76HH~-s{fBNGm-}<YkKmC`dfA{B4zWalAit#xp#geHu0zmhp5C8nhKm5~^pZw{w)0`_#+tAobg?2u=w#HQ(jm)I+(T2uC&=&B}fByWN>rcP;)lWYB89ZNxU%&jupZw>~|IpE(%2XST{A97x#zM@~Y6~HU83W72?^A6vs`10XHu=zYGh)L+x(iVF+bE+A3jsT(W03hcyd3L;7DX=j>R^a=)OR&?v~H^J+-fC|b(ohk%Brj@9ZQ{6{Ji?5H2>{<e7_j)UG2<fyV`LZPr*0p3qE*r<w3cBGG4j2#tR0GCD+@Re64lSu3d7=dyrTfITD0o?$@XYvzYk56S+}kPx_e8%y1Mak<ZhU4O8Sz*<iuelD33$<Q&QmXiEi|2=_bLF%&67SvA^lt2cIwno%qtQm2GEg1zj_rt=IB;8wt6W8Ox7h%SnXy{2fbruu9q@L?oZI@r*rIz*yD4y~r@WQw~JV3nue{PvSS`a6)bpM3ZsTYENw_XV->+%;Ok6>J_m(^E62Pd}T?f*bS)^G&2>kZrU;L2dlWKm7A2|N4)?`?A5ufKFyNYV9E8=?`NJT&gt7r(!d}u)rv2_BC|$vTlr3puVcEp~CP5SH5o|&cmfo{@3@P{9k_rYti?9`1B9|<H<LF^3l(J_R)ub$%q9D)JH$VHTa_sfAR5O{NR^A`_3mn`Um#OBADZNip~dL{n?YR{WBYik3RT%@cu9xhprAsk3p&Up^AW$BUDdfGhbA^z_>uDVvHT!EO4q}JmnC!89pV`e92^DKBmHav!PVyW)A(-q4Hhc3Wz$aI;<c6%^y7d{jWXw`v2?9!L6UXvn61Eluo3Cp@ue>@!j&4Et|{v&Wy57;$6K^rbADqM>tOhp)}lxnjNg>ng33Nl@GN0I3)``{hwcb`gi{b^!oV!{(i9hKH~Klw9~)-w<q8D;Fo{#T{ko7?JB;fd^{jZVLkvFRpH~a2M}M$U?F2=tWBD(mE6RiTXiNgoqaXNGAw6b$tTV^9NP>XIW9L~M~V$*u4Uso<$!feov8%GuFQ#3R}7HL(O}{gQo%~FUMKGE7Nl@cM#X6+GoB)fX@3S!jBlKxZ8sebchdfWkUTB~HrKWmq>2XBOB7Ejg-I4g?lefbk*Q`)I(Tnh&Jt$KnM^R->P~T-2``7opbLYF<EefPUhi4s=rhRcbRffgI_%GfXq(TAc?cM+<6h<1%4vAKo{$wfAv-kvhJT^^SY>nLl`X4ymx_0(I2o>!E-T)(iyM`bVYWNtTPW5r!ZX<HB$*WxUg$(%XBIk6rrf}f`!I$$;$y+uDQ2@`#LJzC4tASVDgpm>fi%b;3G8)-<A#@0*D%d;4SYOoQf7UgnJg5It?<S$eMTr-TVbu$Vw_I1be%xq;H}W8GAQgB=@{8ny1Yu4Rq2W<T~@`U(tOMiX05%<L9A%~sl$>}N9+D7hYB=jKduC8q_72R&dnR;GE8(=8+)lv9y;sS8>XY3K^h!yKqoiXw}3;?m!@=z%~eWfg1fKKhVJ*@R3XeXaXBPwDZ#SuRb_J^8?az)6=n-MPI4qQFSMC&xeL9g7J6^1P8pT#SIjXC(-aU%9y5qhD<HcB2S&@I<^VtCG<9>DnK{k7oGqKwHEPmZ_dTln9@Tx1>b}S8zQ^mnXS#NDr{2U6HaQ8K_oDhbhE!p=H&xQ)73*|U6;muR^7v}i!xDxiy$>rI9qJ)SZ>fRairU3RP(!^~!|rvXt(v_t)TQx)nSvvLDbl)7AK$&l-n~cfHtIviM>*vv(wO08okkIb3D<*BCLjq*g{w!H4NGO}s3oZP2o$UC;qe+!p55%T?E8D7?{Zd(zPCZxK}OMcr)==qSnB(YY;(fc=0s(i6UH_tr(+wO+AkZ^;*4y=I!qKe4ci=7^nb=|bLPI=Y{M|F`o19B99Q$MmTl|^77oXbMS{1J#(1PyNwTS~*3Mv)#20MxYZ=MJM(0fNOfhR{vwm~&*$jti)1PM-8q<!NsMxH<)vFpp>P1i1Ye<NS51&8sbqxivI3u1Cvr$h#N8p<?M#LEo!tFbeZ&I*>_rATp+BOShJ4Jk5C90T}h8Za+1clP&RK~bimqamhD4_>K545w*qOfXm1+54BE|}b$LGiKHWd7Ix{pnwR6HT)J<Lgg;^1+iI|8#>#zPwL=9t&K0@3X&dbCNKo2z2Xr%@ocroXE&(3J>upJ(yx^9T?K}HD8zkpN%TCCSyQV*FzQ+_<A>}1J~erLn3sT`>pc4ArAX#3DSF7m5PI9EVmr7-LgVL)44yIU_fO}jN_%Qpx>n~#yJ|nY+AIV1nCZY6s^^gbO$C0Ikh<$a(Uy+Fys+KjH$6tJ%>kvyy0TkgY{t1fV~R+GHs7kRxZ#~<gzThzSS`DN=s;$pgUO<fbQLI30v!ojzF=R$dX@p6=Z$|%~uIK#9`~T*h^0}E*^Qiv4GWTA1^ecMCD<eTc2mEaP|5UgBwLoV}JvWXR>%({d^mI+s_}njYTI?qb18WSD(^v^T|(s|LI@;s~Z2Lc>m&{g^$xMNvkCU=UKuT5EhVZcNgzo5;@T6;D<Q=5(iA@Wo`}(fs_%kO4N#cqHn5Jd<=RYCjH~!Tnk^mtHE+iBvdp9$Ccr1#o0Ei{c^A#Ii)gyI5`stjw>MY8+?5PfWP7AYZ<MI6b!>-fIA>^z~7TdC2X&vZcihmE7e*RL}tX{?e3PD6`kUbGB!e)dQLx)0u<w%)}k$60IvnE+Xfs9q+hqeI`^lp>MI%?_IHH<x_Bt&Ou!-e;)@p=v1(Fi<5dB2%_T2@7yL*CR26duf1bzD3Ma@H<A{YBlX)yPjbr8}u?t9woG`~p!&Izb3Z+{C(xi9?!6*OU|MST&zQGjmPyXZIKmN;afesD}#qs*%`27?7=eUc%x^9&lFV4%y^V;R@vwldTpP`E>Ic7E^G8+2~SG*|lDVhM#Q0OZ<O$YhB<jR&D#8??UU(Gg$HYCm~yrH<lO}=WV`m$YByn5jeb?7s7D-Iem!<-xL#(W0dNsAwFM8S{{F^mnWvMG}vIJfO=Z{y^jxN5!@Nq2xJ-&Qs5s5D$15@M0<)xujWrWQM}_Xn+3_{2tAva1SVTXoTkMSXjf<YxJ06uE6re8)Z%hlk`*B|d;oReLV^L0?fkoydy3$rTm5s2v7El6TkEe5WFuBER|^D}W;F!Uw6x`k|<$YJfR{5IuVsw`_sMpV6GRaUv&y#vHMSpKYPy4(N}4z4l%MaQb6mZI6SgSq2^*W#jIuIQwN^N-i%Xe;W!053+PbR*w2B%3EDCF47d7pj}j^A^4qpH<D{_-njOqTX(<czA$?}%ZKu+MLxK1@f#tpXK2UG#{HAK#q6rM9h{G0ay$LX2}{7i-{d8`4<4k$v`kW?JblG$?9>nZrk;egk3&7DJB%^BpKIz%`W4+o;fG5b!REsyBx0ARR9O<qaUL#lBs}FG2ozbB$Jl-)4S9UGrz@elYn`TfnKjS~^?z=fs4vLjPSO%|2W4|+QD)QCzhpNpXV4!yiehy0Ci~>?rZy%oMi1r9>3sCI^fNZRi#LnuHOlvV`;K;ChJR-Ff;v8!H?4~zTZ|boKw6zQ98a1$9S1Xt$)j?kQo|lCtwP0)T;S*{^8)<1$cQN>vsR;4jhw@H;Oi%GrMt7ll#LzsbLOYqSDil%=_&p*c{Fx>CI)nUFnH9=Wv<O2-h6o?GoHdv#1TYuxtUOQsU*QUq{!@0ypJi%O*$;Y2xeZAfTj;`Q8?ODGY%Nkq?}~^UP6D8#JL7KCd%_&4%xX$BBt}|T6q_(y(a3oYb&IAmA$LF<PJ>7rdOvoJU<xSz{Ri8ZCICw1rdXdJU&ncMgF#fX|mW$IjE7EzDyBJp^3kkR8#es>4W{e#DgIer><S>%K**_P{RsXrR@TETYIp%T`~9eRwuYc$4uzfYN(ozOW1|ZF!y`F^)_{s(tf{~(zQeU0zPCi9N|HbPWR@cj7}8{GT15VHxs8IE<lkZMuj$IE9|;u$GydL-7?mf*IpmNE;l&Ivmvw<l=%qc!qn&>A(hUcqYCOPpL=6vC!0af@o`mqw-xTsW|MMbb#<Sy^e(V8u-K|yG?MpaU(<=Y^B$_+K)L;Fn$MUJ+T1qBa(jzr4<0nw$t1ba0yQm$3ZYm$O4$kC!Q7qjo?AnUK2ew7vhHnzf>Lnw^D#0j74I;9a~xnlEc)qA%PZ+$UH`&m>*Dh-BY-&!#0k5d9yn~qB?f1iPH-K$A!7_Xo{x55dO)#3j+bvjj{;hoN;YtuJW~urD$#Buu2MHHg6D}1^Fg^y%K?--+6O7sPlq_U4JLP9LfzQoSJu8@_4m5CYEtx+0jv<_fetm7TQ|1b?XImiH+<*zy_#079MX-gUbc=y+?8X+JINvCV?sd5IViFncWUqCJ`)Pn9rg?srDui}Mfo7JG`GVJ?_Xi}s+<-(RT5W>E#Y!ZiLcB3Vh(aknfb7I3^Sb3!562+d;%}u#_MAOc@G0f$z6l%9ca5Q&xTG^@`R)&G#cE2j^x3cC{<(|L3roQs~BU}rls|5KUf2Ifcy5<C@7%(^avHfFTQ<MtTLV8U6`q6x_sKTv&CS95LYO5F=!0^gGrH-lz>kNrg|4M4%kEbA=yd5R-{*m4)B?plno3M=SeWi(($AyNNj-gpgsy$5g$Tai-@maB&oGda8t<H0W1xu3=mo%yn{_BTbj8SC)u=GYSQU(e%L9dd#jVd?rQh)+N+)Q%iULA?Z9J%!ye?cCevAFvA)KP<eVds9NU+>+a1@{^uWr{+imowY~wXWI3hKSy^R+j59be;wrvIn#xBMy?1I6dx?3d>V1rQp!o#AoDNr#Ib8wKE^A2-C+IQ=<nR}mm?ei||0AN0vkAgCLG>2)P;c1dRQXF8Ya3DClj!2`;fXU<>#6i{Mw8)S;O~^+MNYVXDHE6AM*19gNP`MeT(*a%CDOE2d2{7TfpQ6DHbKjybvTBw&>cq2PN0zP?!#<~=RjV?7Ix1x*?H0(cXDe&3xv+)m`4CVfs<71-VFwIDzG>vbUuhnS^K1{{_72RSIu64Fz7}_Z7(n@ejDwtB$re6u#Z}Vq9uolL3}0obO)H@f9ml!?btD+UC}AGGH6EUrfg!`}FvWA#yTP_d-Cib~PwB=|bw<tdEI9&oSLf8GzS7xYZ)asMmrItNnS>U<@CqAKZ;Px17@#sxhaPt-01A}a76ZQq|8>?}Lys<&0Ay2M3<QV1Q{20u;9xe2aZkva;r8{sueWNeEF>6EH$*izt+oULh_1XN`@1j*?!NWl#s-erNipXsi^tg!;e#gC%#3v%P{HgNM5oGDKnfu6L4eF$jB%H??U6^&ts_)I`^5-WXa){yvkc};tPmrbA-a5TG(73(Uv3a8z@kWp%$<#5n~f=SXPe7!Ul0ySnEB4iVLHu`nPK3FWQbu2o2zR~hc3pXj&Ck=URaKUWhTh+*Wd*9qIn;Go}|+;eFUN)63ZUf>hf@yj?&A^TQS?5Nx;yW7+JY+5=K(T5!(=})3Duwz3R4;i+%n7Eg`*Wx=Su7g&j%a)nTOK%*L7`J%U(>Z(uEMyX84bH!45|31HM<3>t~-0g(EbEz#MS5+-Pjw+~zi@5C}yVsXAy9S4^@C(;Bh#UcmF{tDmk@yafe$}krnb30bsW>TvRBN>I}nqUh?Z53%5_NK}-64itGRLe82hj*||EgDZ8-K%unv+)jNd)@=lAs@%UbV8;<Ge^NzB#DM6451i-Ku?7pAI!&!A_v-?DxaCgDy2=e2<y`)2Ga3q=5AB6;x5@p$e6Rsmz;9Gx0liJUTV!nShR7FQFt+m;woDyRGh4x;9fdSM;T^NKzmfi<nT^0N4@BT?cVSLfk`oB!apnkj?^5Pobu!hNk<c@e>M{La@X6}a#$Qar@oRA;F<Sz-PKjlA*M5BuHY3_o7Cm4Q}ua~xq{@t=GnS;3|NM*dw91|YJoy`XATk=!P5zK6rpU_T~@}z67Oo@f;BA_3yr&3vd0Ya!(5qlW3SCYDd-4n+D6KJdW6Q}zb<q>N_MB&qf;#a7L~=~bzi5iM%kkP+ryRngd{ttR+tD+(U=`*?!&(Ax#EcpE`TWR-^yW*qn;e);{o*Y%$pn1Km!gn;85l8D&om%cyUwgeNQVvc>mq2k$8=qM}o7-n1W4g8>R{dqpbzC>$Ff*Y|g=uX#&XvXacUh7w<B3r3Ecwvz{j(;8jJfqy3yKn0PK3-|o(#J1#<3>e2v&3EHPxnKfKJ5ddhrCU<KU-6zvxC*8@1AX3Oq!60mJZ%<BU`$AE=a^*^3`O`EvG;^uy4*ROEa+M9U7BW{uIXf9<E;ka^(YKjB&E*!=0%!U=!&kU9RzCMeeRJZ1o~k#YR?=wlYM)=bqwE-3c7!=t)cf&(+LiB=?OLD+Z7b;tBObEBz8z#I=*Um<y>T(kk`(XrA)ib?W3eg|ymv{OflD5!B7*Z}5^e8=Je7N?$J|G;(-YIfxfj>COnT%JiEG>&z02LW*7;x;FwNUqfy$WL6V}nV(Tv$6Gi6+3Q|zI|Fm{{pdh6MW+DB<$BfC^wYnHO^RIdy~J#PMq-G-c=ak-|wDp;_M&&J?ca<kV}Nn**Po{k%)nPb<)8@;BxfD?U71Y@&{cULQe-PH!_(xro=bh;;k#kFg<=$56*|JhTx)ykjjwp#amj9}DPosPZ0e>1o3D(u;F!Kh<9`g->4aMUq<bG@MUx=E_%&=`6ShB5COH)|L#Fx!<4n9d(ocT;O)p)bolo8|Po&v2!}^mKT67->boJG4xl=z&cRpP<OUsV|1Ek{<dwDDSRmymfOB`gVGXY_j;rbYb*t5oFZSoCrNlAUQ@~WOtUoO=y)}x~Iq>S1GZd>1pD~C~0diJzIj9R1gvbdoO+)Wn97pBd{E=bzb^13kwp#HB&saF^1(E^}8uu!Rj=6O@xJ@RDr#5jjY<JI%mjgX|Rk7^)fTyU^v%dhI@Qa$(E&cc)4SriPQ+n9f!!bWZinWL^kpznUYj3z2Ui0R(uB~K(@>xJzYikS7U{-BB|KzfZC)m?>;mF@f|Cj<6UgtO{oce9+qbZ!Rc{=pGN%Ky^{wRMyZ9%RO%a7ibpA)k$1$8qk$ooQIm1-U=Dm62fsDuTr2Lw!zT167Qfc1Jm~^HVZ%#~$L{#M$_p&@_N&((bZ_2u2J2y`^(eb*_$m&rGkfwiCPnO;d)YYSiY~ACMfv7Vew>eMgZp?Cl@a{;Mp|a1kiSKr--Vs^Jy3Buqli#Y>tnQF9N{}}-Mf+8x_j$Ea`U(DUVCust-JSYA(%0YTKsd%QvvB<z_KPKgQCy;@L~eEKB3yrwg&uG&FUP0)Tlt2a-9=UzVFfT+-Xuwlk5@Zis>fnLYd?7kNjzxO@=U866Vv@aTMfj9lp)uz`eN2VB{K_n|h;0Q|l`y(!wF0EKL30Tet24FTC|ma{aAqv<=gu3vsq5*WS8&^VS!wLh=9wgGb^C%{`u;l^wDZRo!HdxAEV1?%c<VxtTqQ-mF}qCbURYUaHka?}{eNF8RhCjH+Xsx1*>n;5r3N1EOTef&haDugcrQYc5!6x;Fku98iNENW*-{#2ze(WvF0n-BH3^s=%irCZZRRB4l4et5bk(RMv7B)5W+0zVFtGbq2c6+PoNCFFY$a?qsyNtRC1D$XazK#ZlOeIvChQ`2h~HY%<74w4;hzHXK~yOU1N&G@XUAt(V<|irlx@9*O)dR;LsKX}1cqYo#%;k!cI|^})Ljm)@V@QWq*sbm*9E^qBGC`%WAmpUW&FCGUpNJ(^~dUb<mcdF?1Tc*Z{6XM*Uiy`pZnF~Eaa090_<At)#&M6S+pYjtnkHW}bx!Iqnw3bl>4xawMHBb$owLakA0tM=~eNY#%YymkGpAbhV#i)aHzpVVR0ySMY*?O-Px7DsVVNIN-h^WM1T+G;R$(7wH%A??}ZF@fHJs$)_pfRP-hD<3Xxy9Roj0-49KkKJLm*^_Du0CI<18yL3iviiY`X))hpeil{qDQ&Wn+cM_96I>k*S-52thtF)>6itn!8{h`u@RiOhubDt2-zK0UoC^zOW3W*%NbD-gcnZtuT;?nc+>RyTkMlES^nve`ugF!vXOvmm@XZ<n%wr!txHzvmZkOY?T_>=pu^sZDjhRSR*W7a)P8$G@j4^C~sSQ(bl$J6=19(#G+^iH6JT}g^uk_vYpc2Ue0u`lDLny8(#!5u(?ys?$SHx7?QR1kad9a$`7$S<#dQs0Mz(_dmB5cRjVpH$CmiZ?iACe0)Hq9xO^~Gu>RIrNqS`ovh+7t_>?ljHzU_%PKft@*??0^vzt>ZXXn1IADvEX>dEXyXGDoqcsB9N5eR38KnOG>2QGhtDBTtU;WvFDws(@;4$78S+~j4~XR1@++i!!g+wF&w+a5LP23C^R=`n`5OFymnx3$_ngcC&hTc>_?3A%-H2HMPNcwD&)b8I#h2yrGqnO#ay1vkWpAHCtjEy7Wn{{G+J;b(36~wl92LJjxdZ)My5q|mm3NeXR0Bl##0F#Vd^l$I>l;S@l89r*$7Ev<^env=X&IV$;%vt9{=lN?iLH=Y1v(BHmKO(l5`U6z&`qbjRx{qK1}V%GQZs7FRWx~jP$S9$SO2v`QTph*P}thgGaN4>EfQ@wL+n(^ByDxIewGz>bOYurs+T^nD$ogSQi$?g*337Fv(!rQBmZ&jHrIW<Pu4};|`Ziy1Drpo-iR<v@<OF2ex2Ovg1igF@uJX+$4=V*obMt-eTKF(6m{34GFe)+FaYh%yF1SPC03<6jPg0-*56ZpfTBu+J~D!e~^%D`x*eXvp^h#*#nNB@4);VsmCu@)g9UD1hnVX?iaeBe>F09Fk-D2#OvrfnO;bJBI6M|=?YrC@`W;(L^Lnd^1p%l#ApI4B=bu>T<TBes$^twQN(EJd{*?+a;AdqX-%;BV1Bi-bw>(}9ctX6HNt|n!Ed^UMSPDsL_a8#VU|EEJ=JJO3h+LZ52?kNH5|pX*IY4^Rrd?T397<amDXK)`Ivu%U!lRlR|khk<N*&+r#S5}>+E#`44#a5EOv@rnypNxg@}dF!`))TR3yGSqHr0i`0pjp(xEJ9ggO4}FKgK!05lpf$aI18mP{BiLa|?;$=-neq*<2ir^DTcV@iEpy$Fv5<FG~goq}0y@SA!r6-HBrf04<hqb}-{()@U{)9JwEg=xeD!OHp;B~ar29LiA63z<b=$_x80JF8Q%)e0srJ6=~g>I}3XHc6O=4Yg`is2NdQUqV<>4sdT#(zxi7o!LDwkQU@}_LVF_{ML1HUZtqIGGN>K;GS^lRdyeKdf*YAdqjDG=MRB-%3@l@J{hO29|hqyqtF8?xE*bfH;DmZl@;&afv0%YQb%Wdckw=IeHq#xW`YyAr&4Z8I{K7w!&wB1N^cWf?U5lo&`js!h^F%~9T-)jQ`A9gKC<qLVq4Z=EK75V3=U+`tOI%8I;F_kqy4<UPXUgiIudqM?I<1aw?v~+q%6eQL>NVW+I@nW#B_*yV37`R@n8X+8Xh!_A2C*A$4|54bVMFM*p?fn2zHj<g`WeZ{gm6)SgOL!9lT+JTlJ-ij8O?luq}#99OIE;hK=6vK`k^2ZEyjl1IwsZ&FBoUF*iX+KFCUIMQYVKNm6zur>IcV33{L9lNg><J>#7*Dcek^(%CbkSK?g4J0v+(F|p08+v=o=wrOK9&1r4q^Uo=J_#!c=4vYAOSVt9k?6+J!I}+Pa!<(=Da^-9&_lQ|OP@#DQi!1=fO=767WRLKo0)WJ898@(9JT(sV8hq{HO~nF2+x2Dy`;KWktX3TH;tu<Zi55t|<IgTTKa^?ZG3#q-U0V6G^x!dWTUsoJOq_IW2qoLY_aKYjW6rbNbS@<{k%&F*#KBIUmTr^IeDZAI4vM3(2eJc>-J%~dXr0X9#zGr5b-KgXt#(d9hyO~)JXD$s^<fA#K#|Z<IWdi|XVWyZNtVu%gF~)oibt)DawAYh5-dCDrrW>>#VA<0qJ7-3C5xIYLs8~YmN7pgYm~-qn1S34kD2NSdl>eu>`{w6GM+2A+*`R<ur7xrl({|M5m!f&T3dS}ToiaUR}7z{hTS=(y4|2w*>cv>Vp8S(!ycO1=sDwP_1M>17cOqhtGzjG&-w1Z4qd{)#9EiMdIzQ%b)g0*aIO!kkq1j2f3T&gO71%{LkKF0@&E#Hz;cy@<W@;;vEM1?!QCtyEVDcuod5u{lvNGqRW^d#p!%^1#6xK)o=yZJK@}_z+?S_%X`Q%Fx2dTaXv&96##-C3LTso3FEKMjgFbVtI9Oq#2*>I|>q))4LAHe9E#kL+HM1;n7QgB4!zDcj7<1Vt2{A*GbVMM{W1=ZWehPo-w!~mt_@vacvp7WL_;kBYVaT)T6cr_Q2AP}hrz|))9S%wVL-Qi{d9jo=_dq0-87ZPb;wfp8AcgUq8Xf{yP;xiRl2c*06l)UzQ%^e%U+mcw01E!*fyKOVb7Z$7ovb!tpRWUM71OLVT_4)4m!RxA@=md;FDerC@chC0ZT)+8B(v(IbOki4A;(o(5ohGFE25cKaRfGNGcVMK>XJdD2+fq{=sB};ei?~(180GIW~!q<;%7neOlFzm9%|<#sKzDSgXy=eRM_6&Q(1ZU*fl5c{sx?@+Y*?pD&IVgIyJEhRQ6G8&9XxJlFsy84>Y2qsK~)W(CuZfZ99k0shz^pP7Q4uv}Rig_C_()DU+@PLU^Hpvf!!ZjgTVK<Yd*tW!hg;JLRU-rPyRg1%72{m4s*`z;m-u&^#(7&q;VoumEX+{DO$vQcM9r;ozIP%C6B+LPR?~j}?M-6j!<NAcdkQRm2dB+(I?q1=c-AaUCvOTU&~3+9|&h2i$^^*eWvKXrL{OWcsMoUPTh<TNZ`?ae(y-2lW-$4AZL|x~q5*Qy)8rd3F@4J=LdPtk1j}H;&eo>Y=6q$1bQ{Qw9;#$MbAXTjyM>_t|3f9%$#3)ijneL0pPD9il}|ljjL{6D;JTvZQIbg4p3a5lZFMHhC!Dwkn;rgRpy@+v}=~3b6Ocjx*`3VM$T((yfA<^0H&u7!O1l_`(C-S$}8}5`1PVjMcPUX!zq6R7^q0=e>!zh$wWoE8+}kS$E<f>PrdJX^2G>E}SadgZw<`<579h@ZHzO#VH_b)j)Ob#`>Id`<y)GD?gOQ=-R?l6JW;{2_+hH37}ClB|5k85(bKHq;}v*TT<wQ_U19l`}@q>(MSFwwr2Jr!<w&j!jJ?y?j0W6;HwZDyUSSXlHoFD)}G*C^^oTSA-l-IA=gk|53D_mHaL~_O2x*z*YFFw3Xr2K9#EPdGfr4=dZ|QA%|`Q~msJ$O*o}kBQ53Aw@vjcW;OjzE<0M^a(l`kzIxY~*jOT)4(PEV14ljQ}^G6@g8JawYyB7wpXEeYqOZYz(d+r<VZW;F+8Klr2UsfB^`CvXlzl|cefmL@Wm%CmXgHkJUC8fhoS?ta*LQO3soYC-?{Pg<K3iRJ=E4Farw`z}X+Kl!lJ@>XR4n&eORzN(Nt?|r|1_SABsO)LV7zGH|qi~zZmTgo2P?vGZXQ$dhZY#}hD`_lY@u6Mi`wJlO7<L?U%!ljSMZRzM@o}wLMJKFU+fYn8bV1<WyqIf2FaG+{7HVom>Y*b)D#{)$RKA#FigVlEgVfGBc9Pz{ZM|wF4h+WC1iWRby@@B}q^Y$V>gHRQ(Uu)@rIU5aoMu;6#7YD$D`%GIFb4vA5V`hCrlf7_jc;>XE%P-98gRXe1Ni^hd)MtYj$~c%RjQ>mTLrmTBp`{BxHsG_>tge?ZbxFbyNBck3P6D@6F|WzfD~<OzsCHWHEYe!S@R0B)|nTYN0_+ey|M}ylCs)|-n(t0sxmY378wy4@da#_%OY4z=J^602?cf`#*(1BUN%Fl&Ql7Zzffy;V%Z9H5|Rs!-<}7PG#wO->Q8~;D5*(A>FJ^eY&LlIejh1uM}ZyAk`ar8(?t51pbRMP1`yl6NVOx66y8v5yA-_}$H6lw$_xpwvN0y6%DPPl0J~oyrVw>r+>bNrkh~FARx=xS!2$%f?b=rfuyPibGGtd;%<ogIw~G3%HHN$2+6NAL<?aO^6}OV|Eb0{j(Pbc<<IsM{pHm!TZ!w<obwVpPLHO<lIzaV5;8h@kn$gE3pKguPAsla`>;wTlg#i(d!AJl+U7!~Sp5?Hv;0p;8z_YW(LF!x_9atjv`oIv6aWYVOGl)=4#<y)Gkv9|O{?2;n(QYIF5lE$^8p#|D{S?zth(Gsy?wc=?eCyf%>!i0$O>L=FwvCv}uyBj*d2+ISj*ahFBlKN*LAn>wj;r4uGKqTmbBC|t@bo&9e}@9`Rbo-m4pCp?Em{Mu^n(T7%_PCa3~y^lq6y{rH)0nW!iXt`F~LabX$QZQqL4^{v@8rdyfu#f&t)9DHD}$Lvu@2<x8|%{bJndn>yKj2(n*BLXwgC06{^~accdO_880|G#&7Vj2L*QI`<Jf{P`X<X54a%dZ7#BXU?L}kAtSaXLx)JplP#e`FJZVrzmg)KSio9@s5O*1@K{}P{KZmG35yI}&`3)omuuZt9Q=TyE~!W;eVdg_GRFierwTQ9*_y_{%;Y12>1HnH<Vb>vCkj+aQ;`ZHuEGi_8=$OBW_>KJLW1hOQXDtG25D7BC3QsbQRu@RhW(FFHj)x9j89bQlj(wAyD>m_gCC^cAjELiiZ!?aumg~dVM+Yr=3oL9a0qHgf$E4Ie0+2V1eCNR$}ud^K*WVFr{Ab5uBvC1*m7UBES*X45mDEMj!*{`YqUtyaUs`J<x-8ru}|6{DXpV~;|HqjWn{%bC_zkXLhkx(X-$pXGnkY|Z$m@IP#fgOl#f5!4zv^WUPEqPVXHF-^K-rT(=q`Rp>d4b8dc2RmMZH@)d`5@k|taD0Bnfj0)!1kZgdeXfXvG=A*?25uPrujB>dKku%RcwRcx#}_)qCE2}9=F+fK1N?5f%@1&4B7j+f%M_BQt*fvTnDM(LJdNsW4{dgS2{rFvXbt8L<~f))#_*L_qe#Z<4|sALT`>;}$DR(G6?b!Nbv6|ro8CqaBo0U@dzptk+f`BtO<+J1G;XSnDrvrKyh(~X;@`I@ZxE8P)%?S}pT{^QHHt<JN9gO_g+Q>63!^|Oy3Uw+icKSy`c1Wq%QzHMWBB;R|0nD@L_1yJI1={n&Br(h!hxHTeas{N!fD&cy=Y&_6WF19fx_!R<&4`G(#$bAE~q2y4NJb<wESkR-pGcs99QUp|1_tzRqKX?DVLMdKM)WHVZgCZ>n!6KzBs|&Ospm4jyYZL!P?IKLRf9)OCkaryR-5D?YO^-)Y1;YROy(rNrg-hXPqgqJNY1=4><#oUSbJ8K$KudOO#(;KXmanOaKe9#H=Onn2eOc9jX5`~~OY_oi|CMacK8SnQNAnd5>?KFd$}dWfY)4HO63B^gFUKUw;BTWw3@B$kL5Rw%XrONZiOO9WN~D5nX*kK&+O3moyp7r6=H|UJmJkjJp7cEi5ft3hVYit^S7^5Kk~U?&RFKIP_;AMI>%Gfh$4SRkTDiV_J3hVEO_Z9OsZovUWC$>(R5H`S$gA4mW11QX^onk<m<&=+>M^b!l}J(J(9U)om&LKlOt8KV?GHIkn#>Fv1zjiANS9)o*^nq!5{|DTj80mPo|hMw#WcmR=P9Qh_NEsk@R?Fmjo8iBY}qlx?tu}HZ7i*ZyS4-<(NrX>HPTU|;FofmWHWrv&vVo=*1V=`?vB!mTsqviDq#c%=@e743hTdW&V&cim||?Pmw;mOaa^G$SQ&EHU@l8(XDh#xKUP3m2m`<FN-{F9GhBvJ1!({SSZ<dtGP?4Uu94zY+ekD<1T^I&qsNlYU<reF*c~j(oXecGmw{FeJYSS!LlEcc8AKZM^wq}~?>UJuF^Mtoid?3h&nCF+iU5V{auXmU=h{M7X7Z+1QDlP!#Wy)osMT=7KKZKVs_`yxr_s=79kI%=2G(&@8U!Pon47lk$eCfy8P&EMs5@cWwTkqrO0&&i+V;<A$)V9MJ&7u~aa;Wr)v=ql{WDs+!`t-;sx=7HY8ib~y^QV#uLv&}bh-)Y8LO<q+BH}uiC12gdusu8V&wv>^iyUiXa{z+k!uuKW)P4SebeR$L3?kwwP{hfx|Y&*vTG5-hbUnh5I!C}`*48x^&byD?7#gUN1_A)AZ`OJ2fo9ull7o`90$)aJB3JI3(7ki3kG?Aq4jC$_d{+~S>ND&g4>O<9<flRKC&o-(h)Y!w1mZ?PgyJxlmX5rY5y*gb4*}Bv-ykU{(MRr3m=%zf(|C*5-*+iV>M)EuR)3#6?u>ZGV_NxC6&eFIF!y+(@TL_KRemRbP3xGVY3aNp3cXkyH3FdrUVxu#S@X~7=02GbbOknCyP-sQ>}6lIy7eKkP|mz=j8kHDC39;7jk{UAl%tTu&5H5SP1LvrKd1-1W&~e6%N0-c10k1c-S>;INf9SEPPW=h{vJRIt4gMGpY&0(=#2#c1AxgBAXtflieUs$=>XmV3L95J_-%y{fs)Ud>qD{sgOM<Pn~>#asgwAZz*N-VfVkB6(8e9=*ZmH>7aWoRx@WH71Z!D5;Vm!jSO$WzxeWegwi}m$s|#PX7^eiRo=oR5zdke!K=0e8<TPO=w4dL>vde@<O!QkV+8>3-Ta)`!!YqV9S)t}a^6MOATHRp6MslVdy%>XMr)XTBCuzcU<985et~H`3PSwMkdx_5Hw+z%S_gmIndF^fI?Co9_6#mK^I^#-snQ#DinDCmq11^2)sZrr5QG>~Qp%Yqi%Fu)oFz6X>JvVZ<q-J3P*<rcC716joXM>3d}5eX$rdVBld!9fEyF8~=6vAXj@0B7vynBWbGM2%b~m^1?UvW5$vlbiWKKq%5VKcMZa^fU4gYJDZns5gj7MIAPHue3tpMq{Wdx>6Xg}!+FY89A7j^`RsIj~gMstVdI*xVM^trtM_Wj=uK2{&hjunT6b`!d<LKAh=DqAacM@wp<`LJHqq%@3pTwP*jb(v*@9j%1+gVq)4-YkMk@Ge^bayTPBkKIdUNr^BZ<h?nuA>o*I*nPH`=Lbj=u5c7Z&~9SNDC%X5M(;t|^+5mmzwdM~BB0ZO5_qh#Ii)C3c=#h~Z){P*q1t~@{P{SH<L#4CUKB7icj@_-?HyKO4O@5O>NL|gSTUSP3@8p_8T$2GNUV4|SOE-d6ibVk3o~0pe!{m1Fzekz*eA{7?w&ggTS1d$JYFDkIVkt<<2J!gri1(UBe&2_MIl+BX1HzhpgZX5?H0am{Ohf&{h`OwCY=94keSNJ5iQe<(3sU7@JZNFoS`dl9@!>AYZQ%Zc57ZsTZtgtxfcoJ`ihtvxqhS|8PP`yhaW_c!>&&CHl34u4@iJ%M-pmph3;|LkzkA<P!#b3b<a3<hi7DvB_KGAVeP@%JY?Y#R=O6=li8>2oCNcvS8O+*Hh1FYgNM7#hO(2oSoXL}61h7F)>Cq?Io_C`T*khoFRjTO$V8$S%+=DxUscRl{<S!*n=t!2o?$S4*<PV6X0=CwCzU2nk>(r(LrY5rd5G&6%Lqgr-RKr_nVQ|N+AH&0`O0Cr&I*Syq_~JK>W>n~v%-2@KWMPYx0GMBEtrRvZP_r<j)pq!h5_T*30i#6pFv-?JdoKWR@3JVfgD?5S?6g&*-shSMJN$wrtAx6?g=9KE3!&Vz=<c77X99(q2b_spj8hx|Dr&X`3ZS?%SJIk3J#X8og$kD-SA5?UX>>O;DS=v!>;6;1<JXR4))ybl%TF=WU-8SevbnwDdJ`T;ug*|%Hi2L+u4GX1;`|tc^E+VAWySuZvoz?lkKx}roHj+E;-8Sy<4@cXwQm-AYHWw_y!1894W7~^QZ!3<H%`S#^sUYzT?^cR*;9q3FKWJe%Aq}NuPl%7%TioGJu>-@{5U#&>W>FN&nJgq7)NUo;$4xqg*JwY-kT8<J2lE%-}^Z#7m<>V4&l%Gs!(iAu$2ay6`kCDeX+BUA<1R#mjiL6BnCV$VTw58ErY^{AIT2oWjQG8-i|Lh^V<iKDp<ZFC++aHq*CJ)@9mBcAiey>?5sHjVfl}Y;<Y0lj_e`iv}r<8F9t)!gYTx*p}?SFJ7@$D<Ne)cYQb2F|tP26vUu0(kOZ0<n&K-8u1+D0}<Sdp96yxrxSU^2-B&s0O6W<9o%rbhl>$vfJ*YAt5v+?n&q&Z&tW;7<`*1DJb_>K)C+N%Z*L5<)Om`T5=h{klou~e@jV^mV_aj6^85^#J1pl}_9^ywO{{XQ`(>_lR~0g^bme}-*X^pu4zT#rt9H>D*T3Rw{jpunMOUm2SnaueFv)fooUacu|8LfxUx{=&mia(75;}`xSu(0%I!aIq!GkWY^6<IV+xF|cX~!Edya3KPF5eDai(FLtOs+?1L`sH37LzH}B;&mY`*SnYo1>g`Dl#P~GJJui8@DEW4|<I2EyADU?aDlL^O60F*{9hm?K4&T&@2pe-#{pxQuGxN{r3$BRI!J6U)@*x>m|xz?EQq2Q6R2DOuKXUxFVT|#{qV8h~ZjSS-<C`X}44kv+%G(%NZWK5SM7ybNp-dvy%5Fi;-e$v@J7iSSlR_J}*j_%!Uu^SX9n8Vy|KL98ZD6vU_sb_mr5n0gOChr8KNs0gYqtbvZk+`y7Ycb#BEjpzihhvFZ?2RjHHIm@8?kW#Xn4sJet9&r3TfP~I!(Yd7FLj8QVAe1<whl+WE37p#rDfYCZEJ82U_RbuJVx-%;(swxtZq1F!aEMVnqPdVv)tPs8x_gd1m*KfRUtl<N!#761=s}H%d_v-KM3Tzg}op|W+MJ?B1sSl;*&BEofb$-DcxE=e8IDXj6N*NN;@{c!lRwgguzfIs<_5_ZC?~)=lB&i?aaD5N6WBUG<h3U5Ena^Xzyx1~gga!90eZEBGZM*Sk-mf$r&hr&Wsqijp_;jPT@we$;Q{RyKtn66B!g1UBl@0Bm%c?d$c0>8}<(>ecXaxT$o+>5WsDC;B+a1iWKrrDfg<WD|D$<XUJbBwkEuwU4Hgt5SQOa6cePXEaKN$pGj(4u|ZdyI6MnF!umPJafi-InkxCnwyW;97>V9|+c6U(edEp*m(<2d$3)ilbF%=IPCN%N#Ulc-1EJrBYO2nf>b?77{c0dJI68C$d5MtoyQ1J9^O>U!L1CJLzi(qi#8r9BlY!6T{=QhX-&cJP8ztr7|0YWc$JD`96GlWe!cjjOG%A4xT4{fumQXbO_XM_M{%n;|L)bA@?=e-zrPI<OPx`Bs@oM4=3p3~nL<N*=2e7t-WJw_-eHu+A_|N-{xFY?e=FXmC<Xj#a4Y7FV}?OSAV~9{DZ$vBj3wM#aV{gCm$+%(H~-Z`x&HO>;=MddG0!ub(EaNa6TUvY#`T1gRjr&_PWO8x`x{X7M|<Ap;VvAz{U%V!<0^MP)+2vByThxi*hgl4AQcKdg-NGm($7%Gt0!X*<0#a?9;_a7`-^Z)pP%LdnDxX;(+Mt%uyP3c3-vuCQXCcqWht!N&P6IVI@BX@br!30>TYYh$)FxWVoIgeNY<kmE@2U<{e0RkedY1|?yvQ7LQD!nNc0u?FR(Tq<sejGnTS^wV73@XOQ09e#Fy;BSO!Wn$|4Wud_<r<Zq7x$oc9`)*%@NJHle*%h?AI9mz)6?PDhxZ>2Y21PlHME;NKLLxldpyG>~7>J64xZ<m2F%KoNsgmTHP=}h$6fG6nD&!1z@~lY+Hy?W^EM>i^Xs<ml?G2}?z4pYl*PFujrZd`Jd6w&<Tif;oAMPA`7(ks;;$>-eovj-F*z~oxEMUaLhLMxmt-`V2SQ_26G^8sT44fM#Mg*2u+SWPai+ZVJ%4;)Z8f-bpo{_hd9?$uswQy54aHuLbLO<epZYvZ%><`oS<I|*A><`cBkUJ;$H0Mm$4MyehiJWOa#&*vwZNFp`t*{F@tP3a8YO6M;&S55PgXp?2y>fplLiPAl-A@<!#Rf9ZXU~ayzJBAq{D}gY?lGUd$v$NY+q^T+KxYt!Y6=M8(18*A#t1)ciW|5}K4Aat3%kFx&6_L#hs&;A>>Cj#-O3N<yE^b(z<82^eLY5f4h6b4(|)-+*T65S6kKdYGR5$_wD~K+y?$XlEKzApruCs)z2JeOQt`{*XDz_E9W^VdorUYX0{P1<rPv!~?$p=HV$3MXhuq#a2aBvoqju-{ySK0Qzwf+y_W|3vy=1wzOO{d6gR^u!n*XQ0n2~9#NBK!Mr~Po2iX_{JyBe9eFX)=wDTBXQ*E;^SZ{F(6R)BQjDZDWUas1~sH-UnU4N}cs>p;TfWPxtL((gaO?#IgGMg#|>8TkuQPy)x5#BuDA>q{rtF?kPfHFtIdN{Hccr}zrGu`<6W`@i(Wla6m#nw>jEoRSBDguByV&rj?EieKdCsl2^qRP)?uO3Z-SDPJE-?x*Fo1hWW>Y3Y1CPe%O(2KokETv5fYsO5UQf?jg2?e26D6WLP*NM;_e9n-UwZjllK7ENKm-{II$qZ~~n>cUYLc`>ls6D6ZX!jX{M7OQB?L(F|h<{pe!svF%83v`_wr{jEfNqIQ>rvinkK!^1C{4ywDcG1}gJvYhm1G7pLDhD{Zg3M0LKsLe{vgYOkF_XfIczXFMLB9^`S-0DrUd~TVCy#Hx{kEJ}1tZ+^**vg9jG~f+d*sw(iIK*4ZZvZE{@K9~or8CHO8))jhl7_dIzPVL|Ne)Aj}5!RxojIMwP;zP#)Ef%eff6(e|`C(;gSgg;&{s8wLhdVEjan)mkaE^k0AnohV?q3NAqMv0czv)&n8pDTKEtnQ@J22<ccH7+l4b@`{>Se?%Yjc1$U(?klN2>Ru^H!X7Nxy6j!UUTH;ZFD5cC7+YffC1~6uJ%7RsBHUZofLy*Nj5fUifQ^KAbrOBkTn07=xjCEX0K}wS1IT<PR^~^d0dNkR;Hzs9KeaOkV_G#<LPobD~&k2jT3@LN#P4ERT{<TG4nj%%04jkYiyzC6q1d5`OfDJ)JF6jfJ1N=8C)D0hsi5Brg&xISA9rYIY(il6%5(B0s-`e+FYoqI|Zv^$NleX_+<h@?U$!;5eZ;19&Ywj}*oO)5cGvpRHjZulm(~gg5K8Oc7tyq!Ek&a?a<A@pqE^bUgh9^0(6jSU&(a6vt_|R`#nyqc>!<>{`9?>}p_KvNtk<isHoA#uLRT+aX3Ni^Od&}gHq3keM5xJI^$QIVuv-~LxVcAL}_(n}fbMM$|Ck(MU<4fwF5H@WF!tXusK7P$1EDuSW+A1N~0;z35asAE|eTnOLsBd!=@rvd^`MGxo!gSn;?nt#_`)3q|wyoon9>vqkJ9~F1v)v`9UTf{N9#SkCN>9N-nv7}1<C7oQ)!`K!yP%=;<6sVNK>)lPe0=}nA6u_MZI~44*8Tv=8D{BhFL*X3Z--XALncZ(FAoeV*v}$mDVpT9SRblF`unNK^^d$ze=a5&a({%3IGT{`NJ|$~S0y-(W((<%#=_<06t|*yE7jMC0}zcSXX#1$37u1B^t7Dt&l*XJ*if=6#!OKcqrel>?yJlyj}A9vGsOR6?;E^4;iu$e!4JbuXOfK5PKP$$-`_od_Fd=I{_B??J8z%8u|ckR&mt}mqF0OGF%9b-g>(&+oTP>E6+ejQpXT(HK(~JL>>r&s?_Ruo&E~7lM-Xkfu{*e+XrXN>aSv;H(0aW4gi62v@KQ>{oewX+fB6rM*b)9Y+q+l%c5nXMy;=I(y>s~A__uqb(Qo$#jo%Lc^XO;>V%pKs^w-y)@!xMhAOD$uee)at{w?@3epXoDd~?VJkMF}*#eMk;%M{;)fBw_a(Z>e;ZAEmbMT7R)<w+k;rt_A)9_UH$a!y$79kEQ0CTMnZpN76D453B^BM@a!l3C1$TTiGc8<|1Wo=zEwLT-XUm<TC@g^U}am`lR6D9j?D{s9pvJa3EWJb_Vkn#__uflX1m%xOvpix|K%L~TCS*k9;rbq*hr*P#5cpnNbGr&1z{(ZWE8FG%`E6&Jd}3gunuq9|d|kWJWIi&m>h7i!}wFkHvq-u@N4jiJ-<@*crm1VxTGQ)E!d&frPV-1$zZ!^khtwt+RJ6AHvbddP8MmM`GRqrd9FFOUw&7ZT$alathl__Z`<gmS~k1&-)u>;(eNfnzfPRpdeIuQX1)^@0OQJnHR~kB9oiF^k+V;i46wrOz)?pjxj1zG-lWxV(ybjuwEcDP%{g91XvB{shWk6!72w1%Jc2+HByzOJDwv|HJt7fBawd-~QjmxBu_|qrMfkJh9Y}wjM|~;-E)YFX0M?2SHB<o3!rVe_~p4U<vftZF2$@I8sFq4#VjbCN=E2DazCvtPm)gI3G;#nzd<Aci2Cjg%b{fnNXtILJ%uN36yHWU$SY~!|Y*2LtMgb%rKC(Pm>HCljQva{S0AO4XW7$+3C<;c{%RoqkxK#lNjzs9l8A@{J>?af_XV|T)AFy@gfN29}Eq9p3E{_h+87|UM#nUXT5RDC`rzxyDDNGbZ|?<W28n2+GG&L=RhdE0FiIcH10!HaWWW$p{gRH0u2+T!l)hpW6h2jN#|Nr9mApLX!ecqbC9a$vqc|~8#snfn=<GYnd0(rTfzvPr=#FI{PR6DzV&h<uq5OZe8ez*vU`62{-9;pvpCLC*_cif9DNLAX96jp9lh!Tws<g0n<lfllZY@N5JntNir^P)fO8n2Nl_-gMKAUX5rdkM`kgS)KraO$V<IB`IBXLZ3dTN$GA@xwXo%2NWz=&FdbP$pdm@gEBBq(m{;FhWV=TMv|L#Z|4NqkuT7yQT90z7JfNu_u?kJsI0pJ}zeZ2EH3Yt$IKRx!+1l0tq)YED{ZRv`np-5w9W>&T|e(_I@14L>iXfW0ik@3F~8f_DNE#^nfC|Pk$xq61Ef3bzuIX9}S!YMU?oP?QJCFCCGERFE>1#Db_OBy36=d92`Mhv|vPBCoQKsr@%;3j$yNZZMbp|uGAK~`K5BdW#OAnd~KEMc;}PY}v7cW~DT7}E%tHU!=k>fx{5I#LW4DMWGe@*$E~NaXd}#k8Arse5$C(+WC0p0EXh^2IX_j6w(CLm%G$?cmkc6N+DVqB-e{n2li?lLaq$V&0Yn>-1n-0LN|@bISEqFy^XRy5XH66Fy;b##Y!>I{hEN!HJs>=jo^UZ{K>lj)EjqNU$kXX&gKgQkq0K5l6i<l^}EBi=J}qXD*o_#u0-9sr#!QXZ>~pyaj|C?JA>py}YRGosW2nVcMbZaqt%7)u+=DNQfvV^EHjR56y*C)9^;aijr-aCi93LTyKM1nZj=u5y}z$Ncj$k*EQV$WcKCK(BNb&X%r^3L5=y}hqvI+&rwRtKo}JIj^Ja|DCh=s402PP%^lNZBm|5?-65FOxecU@0~HJwGx5~3_9UAYPC}){A~9^{T1qG~HIK5iMENC;L?+Tq{7wL{_JzuI4wt;rl)?v(8+!O-!$$urx_n?D#z{=PpuAH#AuCmBw`i%A%rAyP*IKI-;zoE~YJ+v@l6n$!^){4$g{76=2-*&-y=T{d90%`kTrRRAji`O!sJ8`p%o+{rG|UKU6m+5CdyXma-f#D!;2svk{|@K$>l6Kl-h6xSxGTm86m5dOR@d?ZOjl;jD&W-V>cyt-#m6}oi+vaj`WP<A5{0sV$qdDxviB1UOY&1&dcw?!POM6Wi(#e1lBOr}9`ccUY`2Y5Ir5xv?b577o|B^O__?TsR$E-L-`cO1-+M;&+r9LWn^$Siy!arCFAbySEJFNKI8I#ezY7j1wm#W&M05|g?>kaUcaqvv8mh0NV1zp6R8StwtY4ybaV^Xi7Q~BIKAMixxDnfFv;b9l;v6)k-rLM{rGBJ0-j8%?KhoNMXaTOSAL^?c>xa=oML*OR-hOBSrmi3As~hWw(LzN()E5=El4KIdr$DX2!dPjC0~8np+5Ye_N(#Z{PAM(*o+p7V>-wlcqkTU{jQ!9o<9brd8}xSY>-k=AMvVU~VmExV!Z@3!<05Q)b^$?H18PfE2-t`~dE@rv^K9bDU_~{c`q9y3mo4PJ@0XHo*6N|XH%&W2rGumB*f$yG(NQ%U;v4UDAW0c4J51#SRHYL*fys5oD&+H0qatnkPfW`+^aN%T>H#K8-ywxZe2hjR>;gy>cgaAXPxgRTX|SQs_9{Y<FyrfF;mN3^2TK}JIbws`#A7FHG)mJ1g=Wd*jN-$TsZ`Yz3H$B{nZJ74u<E%RynT1@a?gPBLVui-8EVMF9MvO(Jq`egOD8Gb-O;@VUoI$5w%LO9G$DPaR?KRc7Sotz9yo+aA5or}0*q1_X6W=)W_dKW!Rm-lFfj%U2QZ6&hf4aV6TW^iTOmPRIN^bolbn&05sqjw!&DrIED@UpY#Y$202U=QQ0NAz3F8P&exr1LuMoO0dcSCaO-+W3-1+r$3CL)guy;<3o(izm!_qCCd$Z3H1lp@%m7&8sXRKhk1VuS!nwzG7{gp#O*@L~)iTR%MvldB&>>704w!7$~z`9sQx<9gW8n;Ovq=Xz-pPRiDrs8mO7906Ljxes}K`m%FrPUW{#TQyA)Q5`Wr=w(3eSA6>9d#mO-AgyWwNIM#J#@-B0;V;@e?gdqbT+@#V!d`J;UEii_0A^qa;ydN3G+Nsm2Q0BV(l)veKBqt<qZrys*3+0wq_EAE;00*q+>Baog1PUxTdBReuz*PD(2jnNpqLM4E&LJ`u!o5F%L$(R%izjpcxx+E^Oefb7|#q38J?ZVSp?n8BLy|Z5B7Nd(W2<lzlrGXDH%9jS<4IwnxdhH%O?BJ?9i6c7-n@cGC(`{T$-aSeEWY^}_UM*W)=ndvM~MIcGnRG)Ha(EEz|QYji^@FFImS4;_D}OkaoEW03{_K|8^`611!(v&)Lp{U>n+7iW_}`Y99yW*)&J{nSSs9u0<6QKO_kh9eVY0=mb`$SuwHxo4DWBFt+V_^GM|9JQa^PysQeT=p2{CF(jGiOtcZ+yo>@(5HB5O`r1|E+bLMMO^ncqEIEB-?W3)W<w#`5#6b4pjo-vcnix1P95SK`L?t-^mxJ8G*{aVm1sp3Ov9n&=nnjU{8OGy+z(3ZN-Xjcr{;&(%6{b;YA|8-NU~yBxQ$YpX;}q&kkD;<?8Yh~vr!S4CT~y)V8}lrjfb$ALCOrJAXjcuUv{*qc^<~{T$FUC!``uW#o)^WeiLd9$dROQXx}8$g20y114dn3InF8(JHMcuW}B7d_2PH$Ut+{e!hqlbV>Xraa6L?tL#QWAy82~sadC7BwS|^eqZ}WutG@14BkrpNo-C-j5>l2HkmqgHvCfO;ITs)lExye`BLw+>K(}R*9(8+GT51+ZsL+=2q?8s7_c=XKvS*`lQ^S*?^kiC1tl=@<is7MCt!{WG{0U+L&QnlCc#Nz;TW73<{ei}W+rH+JuvP-uLwNJ((Wv^uTrfrLicyz?`A}}aP=o~X=-_~i(keDZw_GM$5lxo9Kt7PnX31rEShC<8_L<N0iO<0I#|<&Kn$bj3^D-<><vojq%^L$*OVxdWM07S)6;BA#UVs;75=(GJ<Gd%7wrsA)4?Dt;uxg{O{T}rtm>$cF->t2>Yu)ydhK%Dc=7U(}hzxl0F33VY;cN!_?wvA&!eYlG{^B&9r5rhxsSI}Vf)CoFzGz2p!v)CQVL}Zr&0y8ERZLym^X$gd<(eUH*S=jtfxaf*g2A92?M-0-C)h6pC!wdD9wZ0-0@qtajxgkZ&ITA`!NY4A{!Po1eCz4yUP&9wF+gH|8fOL6AKDF77?5cDhoHIrUAu!|9YOf3UH)^!xROo+^}Hmw@T&9r7qu0I9=XYGE($_tOh6Afy0Zl$26WbaMHd6eIng_|bZdHO`|SI1)yZP|dy{EJ4IHm^!Etowf>^uXvOaXJ>G8!Zo2Mb&Y77?RY2jF2@D!fR+buMS&M4$YMvuZC10K~$ihh=f3mxh+Mie+uy(A)7y0>#5uoCS=sjD{gL(vPvIv?!BElG)ceW1nANb)_R5ScXH-iMBnLtJ!N){ry2AUQv59{UUi1TTtJ{;6%YP@#s}HrxuJH(|#8$SJrCr!6=Fe$|8cd5^?=dY~`|bky7gROrP#J5S>(3|1Ct6u_&<t1diQ4PQrjk3E|$rZ8lGs;aL_nHFop7@hKm&g3e6RZpP3<ou1W)?PXe%}t}n&lGiJo=8|T)>&ofp^-XR%w-@1G$WEKR+pd9kGHR$<1#0uvP4$M)vD;fRl?0W!=#vh)iGzQg^6tz5Vl%58Y5ZQ<C*&kUauZSM&HHdnW*;0{;HzR_$#%WjSsaKm8$X@Le%`x0yyMh#Z)Yd8aTRx^9ek`2_Fc;UeZ6~Oh(2czgGnJwd{RA5H7n7Z?plob{lfAUdRD&pjE;S@XJ4V=)wAt0XQg|MhD0c9S#xrGSLCG`Dizc6j-%Rf7F1@&eo{%P;&<D=6?d)WYNRxdlQR(TYuUJxEV%ULEnf|g54*^8Ib7|CDDclL`Y39MtfdzK&8MZ0&>u~$d<PL;DG{5y|iyb3BKY`f-mpBOzQV2zv$>4&y22J5e2udr?cyPY;Y>~mCbbAw_lOdbFGW=kEwDtPH=uvRHlTr2HVGNZL)0K+oU4m(3_Q^TQj~$9Fr$CA=4FP3h8z5??Fxi4CiyTk{)PF==^k+FHTO0=N6Vb7YBFYa<zo9R|MNh38Z<U2Zn2`;kAt2GF*FaJ*;m%tZzN6SM#u5;@4;JQ*RK7%US(FB_1Yg|6JRb@5}fU-o%9i+j&ZC?rXStAik^;j%Wdj@eQ%X(6yLjYi*}l&;290)ll`0HLS=^M%i>*uD%-k*OjeZ9`n_~Ki|LXJpbY4^S|!D{oZf9vP?%}bI4JBk)p>D2v?UNgg!e@vJr})VzG2yC%;^BmRa#6D>_nxhlSHmJBd>y!*x3ApZZ>$XQ+r+Zv?7Fhl%U3Sp^PS)6yhQUvWQ2qm5>rL-X$YBRv)=g-v-wzB&L=q~`VCi8kkhV`ZY!XzvyQqVRiuo6kTjl!hwpt0ykQs)sJLoKGoNVg2-Az<}`MtN(CMVt1fqWcX>7#`{oN-0V?4W?5Jap*Dvc^St>w>CW=k(YtJ%oZ#jzO{3Mref-Lt&8kj1Yy9izqvS|V-%4wpQZIOSfs;m}awAp0<$YBB{#8DsL4J<XI>9SZHfXiucwNCjTYOD)nyuYFe&BF7GL9M?A#xvUZcut<a)QW$HR9$=Ye=^3@&a+l+D^OcF@trMli}RE(vs54%Hzc8xmWOYwxr=!b+(o%{La0sU=4Op9nP?%yLIg0ckZpjBs?2OZw3EmjeMVS+8N&)H#Lp#8^u$+Zw;PO&WdAnQgo%bRBoN{Cw3UYat`Iy7BWRcuu)fDQE)QKd!Vj_Xn*gJOyB9?iC$*5QNXp5uTQ%}|Dj7Wk6h?uS2rmmvK@>^PGDYGB$NO+5p&ZcSW-#`zd&hs$r3N$tD6)dOWh10ro6~=xq0ZeFL$1mF+yIKP~@T67m*iso`Wue7Zv3X@?nq!hh1mcB2dIW*{R8_3FTS6om0};MBLW#l+dNT42Vto{aoCB<QjEjh&q`BbtuTlN{JQ?7?T7TOoqvLVKrhx3E3Lq915H}i*ULooNG*uf<r1Vvvf2dU(hjdl9^M8h)UGWk43t8#=R0(R3~|UGD^kD-$Jw8))bAn&E2h5>)~-#R^9qy4ZE{)Z$G!^rS}uOFXRoEalvIRhwia-azx{woGj(0u7+2F?*<=Iz624vY~H6i;Wn$=s6bh&xzVCzSLG}hwt@XYYiDQ2Jxl&>XKb+Ng{O>_-{1u<qm-BZWHIjXBx-xQxbvkM7b9fi3HB}>Ysjh0I8tyH3_N)xZVraW;#!njMQ)QaTK6g)f4uv|?e>do%okh9D4So(UdzSFX;W#g@$odzNAhWGI)u$rU`Yeqr7O<B&g0f<l;8z|uKnmv5N2^2N5L>n2U{XNQzKBxUa`o5W|Fcav7KQ@jG5@&GGd!0Qs1gs7YPnUO)C?iZ$S{u{Awhqx&p3??tkDUGhVDQJeZ%tmr-~bg}V=ZcT`YYHeQVNt(XAF6dq`ZOGJEPOoumKpDKQWj5ZDq;4Ec?Ub=te+dLDu7US3Lc7IwFb9HGEV$gXMJZiSkGop#o0sObM^YBTdyoZwhTwKoKM_mc=UVl}0l8zRg=_QM*aC)3pT-QzV$<~O{>cMF>&&~z!6Kw>c`U{#i!VApZh})32ASK%2uOdkVwCjj{1gOU^@b7~ErfMYNlk@@~!Ox!;S^unaBEJEqtI&^PGocsehd{%0C!OM|pfJ(8m78wu?0WA@r$si(#RX>XQsfuHbt=u?iC(+r6_MUMgIPXB*O%o~?8G}h8e|1@_$f3ey>AddAn9emk233)N*RfPVK3jizLv9rWN_l%R!6Mack{lx@>N=W`1GNF+;z+9E`2K>ZrS(NwREKG(($;S4$#9^q=0L2b0w|g<5V(RxSEscT4W@!Bzdq@nx(;du?x28<;W3#brZIn*e(m=jTH$Z;@KMvF+&E>f01&tq4Vxug;JR=%4ssRK6j{n))%$_LnK^rAXwJw3+2a~Nr)W|5TcXftGHJkl+-wpr2cXphXIsd+9V-HULmUuLR3^E`?OKEiE8AHcGG?BNCWl<6$f&A_hIBaz}nsuDXQY&ZQ#$lQ4Mo#1M7^;`*~9OSlAW*Z{6pw_up$s+<S!zXTp<)KEcp{BohnItnFi(2FqNujBUI<U5rNCt%r{vKe(F}#R8MFp(3%>+<nk$J$d}-!4p51ZZ{)8BW)9-jQre<UOryU>!NgfYBg{Eh-?DPACcXBYQ0jf=_Pa;E75OdhF?=tXq!vg`K~O^x68}iDN~lJ+vSz-DdP&lZQq7))6rIlsO_q~;wHznWtg_JCH$k7P<k%?!}+Q#l(frxNg=$Dd4xSQja;*i2W=m-3F+S=IW8G*@>vJtnD#`zPrC5Uix?dsUuS3Ov9VV0hWF@B$((d&&kWHl;tpm5mcbh$S5fF$ZX$0krJ52!4Tvj0BIJ;c@iwfbykarmC1@&TBH`)+nG8+ji$Z@&JW`0USof8t1=%1)U@+<9f)Gpfkzoyn(@B*$n||T-9_1?+gLv#fx7dvY3<#;JLT-4P6tFC2Jh?7SWKywOGv6O$^bwA<epm}0<;RUe5Oo|y4&zRwon{!&Ir^*F{5{>!z2na%Yn)Tm7chi4RW`++b!rd;q)NCjA|iq%2auy8yHF^56I<Fiyty8yN_ZR~0kpShSy~I&T7<SO)V#j>ver<;Ouvpjl&^;VMJ=KZ@wYb7>#>R6+6HfJgV$&qMC$sd$z;L_!Iw;9HC3t((JI5i)BUk&W#J3umDUVyWsi-4%g)jwA1z2d#n!3+RJUc(k@?G)kN56+O;Odi#>QJ?<C@0C>oRs(jyPpkfEDaZW(_xG^0MTu#>zx}VY8Rf=vQa>623tjnZABU`<8jAv>!x1_TPqCihR7f-Q4{>j9}|nfvSx*e-v|!B^9)-`7DDd%R{zp%%a<0eeAc5MX;H{<t7Y2w?>>5jW|N9`x?zSf?3`ga{lm!oOMh%(gdTF?Q1aS_zgM2)6$v-%hzxN`;GQdqA?b`&lEnSWb0UUZor^(Yr46j>Bg`#gUW%cyf!xeMMnA@dava}cR5aZ3u+spjMcR~FeRCW_G59$FF=N=1XzTKLn>Ug<+JpX&{ot|xki1;)}&U|p>9@tT0?L8!Wz>T)RoLif8;DE6fbTpG-o!bzaF;^&L8iIXxrSu#I9x~mkuJsEL}e=9RXA3b0KlQ6i35y-COK@dztYeObp6m5JhMZ64`Av)=_0Ev{wz&CK_aX1)EEWXFeFW)pRpV1Yz)AII^)u7cAWPg7d|A#5d&fAh33gk15k)7c1<>_QaDHaETc+zmV<}=a_5&WhIwjn@{^mw!c|QwDq|~c2+^Q#k^M`oU|-^Bui`e{(a*E7t`MDl4mvFzi)k1ZeKHpSqB6Ukm#VICWnA5QS-bz0M!A5<UJ2Vu!`(?t#rQSJ9Ckx_butMB)f(vhy%5=>^vQ?&?-25;rg^Yj1~D#Vcsel-bm%oV{=SDkf`qF|8QyB3!Y5^iCn<qa^EAjxuD#Bogg$hg7J4e&skGwxGci@WqxKb1b+k)L5WL8v%GUUy!i+ufkckW<n{i0M(~{G*+im`;SS89=?tVIj59#@mt1&D{_Vr-J)e6OAMaZjl7K?=c2Du??fEQCw-K7`s2t%0Lo-cgMXDb3^U)|Jxl6`Ap<Q5<^;8%W{4^?<EXLDI#D1Mj)w3D8eHQaVJs^{vsLs#`IzYUv_H>B9I)nUTf@TEqG1so*q15yD`>*9w3d8dTW7WxccF>|F`9*HHYE%yCTsLwH1GdnT1JmBa*}Uu!Q*Nvk*JYS%V4><&qY(k_3cMe^pk$POh1kc4lG}t-w><%Eu$W=f+b@LrQ*84Z-M&bBqpXuoI=$v@2MTs}A3Sb7eEg&pPo}>ZfR)~M9_?<}&ZFJV?#`o~od>%+E41_A$%g%bf}Pzbj~+jH^623T?L2C>Hfsk8VmnWEo;-bEwR2Y_?W>+RdVjSM9qAE$)O!4=>F=rC&_+W#l0*7n_rb$QPaZj49fSCgAroEzchfyyvVyj<=99^qv|tgvkanBR#}6Jnbf_uvhdVptnvv%Z1St<U8xgL92IbL{$GeXn8td$Hd23td@?Tfj!um3$b_c?LR@p%CpH<fXBX#Ffze+u=dKcNUX3a(5snb7CT&X~~mAUQ2Y0tt!sK;K4W?mIZYf+YKZVTAc&JG)xnL0Er>pB^yiJZ@6D7G=VRr7wRUmQ!A3*(}txyucTq4Brbe5k>U%<xCk#%A`{mc8rmY`i6E?}DUcW(aeYF3t*4<me8r0JnP;5&iT%2ai=3@Q`6&+F?9d?dQI`yN5eZcWVpzF7KAK<G;IWmfNu>VTIl}^So+*>|!hSs9dPJPhtvrBVTC_n~z%S&*6iehwIPb<1a9WzTT`bhra%-Fo%z?IENOn2BUfCjEdpW9X3{47fgl`LF$0YsK#8okC5DHXU0HenBqbhrqrE}eq*GLyBiG0^F3gEL(K|pzf|3>DlbzFhR9!M$++0g#wXh<^kDleolOYBCs22eY_!zT34#@>2Qiytb;h+dn(Jjrv0h_bV&$elz0`%5v{VYjuzDu2fH3_x<>{k5*Q>2OJIQRB;-N*d;nTP+CbBom`)8eBv(MYFvBJ0L0I;)N^pg=~W*)ko5}uI4I}FMht_01}d|fkHa!%Kz%AVM3rK!2$P_=G$sG6G%RqOhP>dk2S9sStZet||F_^)OJD-bsxP2*wzMpl-Gv^7Re5_a<=*4$*oT3=+un(L3)O{_80jDLU=^+i5^Kg$P;{`{FbX>q+>t~Y{c4ac*35}zT6*H1Sxl*?&eV`xhhf9Wx%YmH4u`TX}9Wu|(gaX$Zj#`)((I@@?P|NMK6GgH0MIRE_njB~!2Z#v3Q^!FNPq<W)KRy6BzUQr76kV~S3mWv!ZKo*{}@QR<UFX2}<SRwE`?XBIxH_zVVfWzowRD!QZsLRFT<;vaA)q3S_UlUgtG=FOqBjz-=rdp(%sulfwZC1&8S8KDbmwqAcD(y8_>SOC#``BEekF6{8aUE9aqd=KGs(It}l2bPv9%5XcaMyw*T!~Z6Kh3vaqH+%uI7R`D!eW{P*f)JcO9!^*F+ccQx`Oec){M8h>464Eu<D)F4G(5_uG{Re21&+&x;jPc$}20=)7)r?S~odF%}s`=b=^a>%+OFz*yd)2^~qX!v^F&{tV>dJ{gG;YfstyiJ5txbrq(np)Dw9PW`(QMyYi4;*`#naT0I7ZRp~wdeMYxrOjwiNFFU>-L&B=`{`~ulZ^?+TCcR&Fd`kv|RjIv#`QYjlZ!o%>nGU?9mJA054MTM5>}dI6eSx{6Ijt+ywYG-xgl(awVJSsM-K2%AO+D4<rIqR{t1zQ9nNecqbUaOG38g?nZ)W;(g*LgJQGSxmJ7P*!BDQB1ER)3-%l56cQJ=oL)(^u0#x>@nK@?2V3mHuz)em61K|6p?K+&0o@KwWbgaNYT`eerlr~rXJKr!?^Jo7#L*?!0&tx~g8k4^Y>q6q#S&(h*FnWo`ZGg|7g(D)KW<FZ+s<O+w8hUIWnb#q_W7C8cgdYXn0qM#K8J5kVV7(Ojf4tUR?zyJyu6t!nWXUd8#G$*q9;y{Ql+lsdJiI=JayI;OWtOjmwA*|B+P0gq?_F^M1X+Tu~+wl-;z{Duz!_IM1<dq&mk%qa=lL3b4|J*t2oReEjSj^F9g`=76y?&QI$IfPxL3W-E&=r;;lW@7S|9xOWIsp5iozUUMV<)E64H56y%-&=w0meza$&^#M11mw4q8}#!lMr4P$znq7$rJOsNN;$|Y~lX;N6qNNb&i`U;d@4o1k@WIIa_SM{*g2KaGfJ(%I*yx!?+6!2O}ku0R@wfRj5C<&UhJ=f*yloN0>}uKFdA@-vsBBa#N&cQj{Ln8-9V^7_-Dl?MiIQfaRHDTx}mJpTPgZ!_quQE}MzUyXYAD_9-jcI}K~R*QS}raC3^Wd}Ong$lWg<TJirVl$h`j&eK^oybSWi3>b1jxCj(0$NtU~@4lF0*fB;0AeW!cQdl-=2NsRWT(+2vD9{yyvi7#O1?F8p2iZKCCq+7s^VtdRxz<ke@%9e?^Wf>@-KX(S(-Rr}$1S!woGC_d?NEkM5lRZQLRo;*WK5}qVf7^!4NLsYWF!>RIvmlccP3CrIuPJ5wC1A#J29D~P};_TFsM15!e86^Xv{#~madfAw0;I-jK>z&6Vpdl3O6_`u~u>bmrcO(hU($pXLukR&&f^w%n5uo@*kf^W;j}$^!o9e#T=8my#a9@5$>FzM4~5#4Tz$=um9(fq>#G||7*xzm<hRUFIHW()Dl!B=M8%fs1^EFEVjS!@}A_KlNrba4dKsf#~teE6C)Y{W=UXO4PaYzhM*Fp5~YV1ZikeULZp_SE{ao%U|jS>Zs|Q4BB6s!F{p8Sn!t!`&o9#Cj6JOH6%sH2`4w&l5t^vdy2W-?lGzbhi9!`o0gdIDb`?@l8-JlIM!X?6Hpq&2s|tp*BPY!2D8hiUARB_IFJz;XfeOpe7GELe$Bb^`yw2NJI?Gsrxp@}a`pu<qjAStHwJj-CItQUs1(xa~Nyxguttbu=mZ=`k5`TwJd^-zE8}m0y$N4!c>Z|}$f}em@<FdncwSTwfJ;kh#=s27S$oSHGeP;>Zs-%*`HK9MBo%1Q&PYWAf6_GMA9v%-c;2P;brg9IE3(s*HWLH)Z{hetrLy38!G<bp+X`>ORaE3_a2R>V|rjBmQXx3<s$2OSi44wR(V^x#Md=CcTEY&I8NwF0COndr|lj01;mc9|2oa5fl(_(UugMKpv!kK&GoR&lwi8Uqz-tL?Vfo9LB1&uKm*<gNZKmKgCruwKx(2nyB31QHX9Ce@DZjf)NH)C7>83{f?${KNtqK!F}>1Q@Tkw#QqrbUzCP}_EXN1Cp-`pcM6DYn3Ivlx#r1<WqPHpAgpD7mE5RFVmXi}weUnA3Jp$T+bo=uR@)nxu;v9I@M&rVf}39CM?T|Gkw>^Fc{ap^9rnl}^(mti<Bf1KIn-_unb0A3twhozC7UnVgjgZ7m_RwK}0Rwwx0iacHnrwh<=5G~?Xt$UxL9b^dTBK$MB>6#z>|jAkUMJ!Ys00zCjZjdx)O_8=50Kth|6JyRr@g5Q`PgA=eBK{RTGsAU->CkcvG*48c5IcV<HHb#@0xp86D69YGJNvGK-*aHFr1W7WCT~1w(cGsv219j``d+=l(+A!j_zP?Az);jgU=T?2of}LpPF=MNWc)~VOs6#|b8v`Lk=m*Rhb99GAfbBUf-7OaL@p3~@+m?Ss1QtQiJ5WYwA8Pr^skeHh^e<{q($cs#($p}jr5N)PUWrn@eyhkVi*mv$>J{gZZOrCkKgOD*Ih^#xJS6PNa^vc?Y;HP3N&V=~n=~P0K((o8WCb+N9`@2Z=J6QHDSgAO3{9e?dIZJ(T9oA>pQn2u$lpT*LzoT}UI??tOCQKJh|x_4`UM{2c!-}SAhe~Q5Fpk3mRn#DL0g?M23H{IN!dJ9<JWe_I||H>w7r95^sA=byH{)3Gpx=eOL#!mv^rDJz-W;l?mdWty$8p}=Xp;rQ&{xlJR4LNZ;*4|X<9IFDkWPmvBFIf5U7)CKioNn|KE$7|I23vW@MYe4k9_-`;|@%c>npH+DD%9%_ZfVvb@?@ma1u|bT=Z-4bWB}LV>MIr=LKOM$MrZf!M7EjZZlbH0t5uRx5tG+k#^f{%Gz##$)nuYbS0!Zo)Lcf1fsYj*p`f-l*2+SEZI93$9G<gSgpzxC_)a<EKxaJ`~hGe$d3rp?LSflgHOWEn^&9!`MFAdHRIM_Q}(xp!Q+wAyNCVxzoB9Y8f`-8mN8z<mpqQwza$aL{a<n5j@+8cOE}@v?;kh_=+pj+l(JQX+0*|9zEHSqx-b^h*oa%!R~|WA@|xAL-SGV0g>B+ah4nH>7%De?&Bv<9&Spm7ocMudTB6Q4<9y(y4~G}lDfw`yE{bPgU3MKhWlP(i(GjGVRCnNAJXK)ViY9qw4TxcK6<d@9l(-|!tErZQGbLN%gfH8$e}a>%{?C>3jRZ5Jfy7gI?vCFAV2$)&FZV{jS>;sfgnUAu&}C3BOad(vRRmZ0@10HpRs!7k^*Pb$K)IZ6*heQsway;t$(em{<C#y#MY1kgclR2y^^$QDzwtRdgH3hIt1u3*H+s-w?oD|q)|xP<``8FvSLAkaYHmth8g4S(di&mT&JV90^!4<6+>dh#m@u;?|XV9O;UG*4{QP&@by4CSGU((MGRVNN5c1pCKBM*mL|<g>L3iu?Hbv__?pxS1l>0vEKYN@?6s@R#WC3$8)nLhbc9zSZ;@XJ#2j9mm-*t}AQ)w2Dw7ucSP{@zwX0RD0_A+!Uq(Dt{gFiBj*~Zv%n^e3mj_szpb>T)T_j#A^!1g=vgAB#SQEN`!8t&1^v)3P7OW2hbw8e_Kp1TuVSnJtl<~pBHOvsRSwU=XGLCwY<Ip)&&Vk+FV~4h!t(!n17MDQpIFqn!c?`w){(t}b|M|ZKh^}aiB>y3jD+mhdKOM@;&`@_B6_-2sRczky(1Yr09(rv}j4Y!2?gzHCEQ;`45Oed`mTd9Re3X&((SFZ27O}oUVi3hkyvUQli7&`v{yIZb{X2mR9&t7eFT`V9>ZgUn_#JQy7wPdkWFD_lawLqzb<{6xG_P(zJyKPN_FThU%uafdybvIRwKOry+$qq8NO{*`ta$jMQQfB16VcrMn`hs@eEagC^YP%>+ZWG1yy(1n`Rr{}_44Dv3-Rtblc19B$s*X&K5D_@!TbH!^z1`gAgDU#2a`6;AuP+9p3p}HyI2lbI1x|rcA+yOF<{6q3OzP-<qz9umRv^4=_MQ3-_awy+Q9pFp306+nNTr`*7zNez;*x(m5ftxG<DE7agw7W$;pIPgGjFuVK>B0MRJ~GBfM~q%9x%jqIS}icBwvP!H}q``>)G+E}YAY@@n>Gym+2Z_}aE$=yAM}&Fo!Sm7!n&t2h-I^Llwcs>@u5@?0rVopA(+iYOT%wtlY&!c*Z@Liae0K&_&poDrjorMq$PoD(?Yu*+tge6e@wWT{hBcV)u6u3jhIfPtQk%mbo>$ZI%753uIrZgo$s3n_$JG;gBd_vg|ud^F(pc<7TMEIA$%TrS>ji9WB^i5YKD>_~af*3Fy#D*kT&`QPAwCGzzrVy|GqMUkSsD!ZWjR@E6v;ffvNSw!w}J&TWO(i0#u1nLFGA9Tf+0w}y&)fr74cE>@EUuCk4MnpPF&JnpJ@nE;P-Foz>yugTfQJpn8@M6{3G5hVCBfCl8gvm^c*D+<fu9`_hn>S;b&*9(6YyfSXXDmUk$SPDjE3QjEjHzLnCz56kg_J}3eO%Y=L(ut2@uz<smqvih{(1%_b<P~hyi7?zEg}JUpdyc<2b|N0F9IdwnaJd<P6{nIQ)|sCq0!L;(AA3FI%8MN2RAZwA4_ZCMh4PubW)x2cC~?o(O7#N=~%ykdSN?A0T!g!YuLo1He(CJ!w=to=XD<|c~bD*x_KL8T%xa*^-{?Kdja+ZY9!}4`|#fj?6V)fB89try6$-dV)(lJP@bic??9`f#pb~>Yamb*@nS^6n!$DeCqU%qyHz#oJ-es$s6(!vYGxfv?7?Ht0H`&!VR?IHDKed}+^n0J6wxs<)@69H1iCi=2ev-{SDP%^&vdC2=VkeA^M~>t;9qVFe&x)^4co@w+9d&hEUy!Ex>CjyEw{Jz4sgCP2FZD&M9H>V_ul$R=4c!1x?GPb<+PpkU9M9f`E6@ed$Wg@ro*3O)E?b(n<l?&JIhohh*_c`N<)wI2xtCU&6HS|-79O`evajAl562u<?ZS6rgD82Z{p+9S5cW6d3nk2$c$_+s!}C1s@g!{WoOAe5vC5KxNB6wg6^~U)&7T%2UhlFJn#G!8uXOv-iXN(M}-9uvkM&kd#{{>N8RYC#cTvY%z_@8%uwVJu?;5waV!XWxq~#J*Lz83P{t=R#j0lG;3ZvS86h&|_SqrC@?}A^AC-A_2kV406O-JRTSdrtvN%qvtLvrnixecR9nv+>moc52Iw1SkcO40B9qWnBw!OnsV@q)w_8T;owO-lqQtO3X#IQu5TDz=SWft%?y{_ef#Ceh96VFURbqZ2?;IoUw!Mn-mvfS=zLMuKU4pYu(Y#Egbntvt#dZ&Fw!-aF4t}yYo9iu$<wqaqZJ;8HQSIvZDWANHk^iT=g$kMS|$7WCx6qb#b$A}PnnA7I!yd#j*@U^j_*bnL2iAfX23S5W&WoH^F>PPcx>a{I3)%2vO)U&VLuX2;#j@kz}^cJ`r_f;R;wMlI8AH3wEtdz(kL`}4;v2Lxm6BqFm-vadvh~5+V@Zf-whJ)^88sh-Xl1rCxE@)60AH00~@!f}4uiyR19k@3i-s>8bs<Vzr?aqfW7hYcT`+pw1`|$h+QSEeITP=B)tXlKizrA_?Pf_n=T37EnBk|v!{X^9GQ}U^<Qg|KHHBnXFhA;(d!<C%45nM4fo53%(b*t8qu>I&8j6gJxJHoh~cWgq{YpZ*7N7|D*!)#W7mdbQj5GF!kP>Kl=(NGq~3vcU=h3SfywK+;^ntc7tY0+vU3QPiOTNFF!Z-q#)UNwqR_?YN1bU2?YM6W#W8_%do3<#FN0{arx_XBN>cyBsO2jx~(sz<Bl+ohsf1l2pOZ0PN~w=b(3s;GKxd#A^0uuKG5VgmK4U0(AI9-GDw*Dz!&5Nq1Pl>A>RXZb^f9Ct@X<&FyH_$4(wmC*5__<51cgih4(+)!$>k`Uv?-(IT97N`p2lO7-bRWm@gt!jyxO5daDCsmYsty?vQEkU#KyNYU$Eo?)pq$9Cj%^YdVjFlG&u3=5|+j>&fbB{gFFe<ct-XtQL&P}0>^_$Kl@-t`umczD6+xzIw{)B3hI~wJe;LQT$zh0^Y701pJ1(wi)_GVwrrYHu-iIn${>yhBYrDL~n!$L5s_Kpdwl-0KWGpbW6+xE|>TA^#3KcXtNtF8ZxD%6>_`YWoiu(Z`*QP~vI_B@wtr7iz3-Wx-CH+VK0X*VL$X$s+5EmIxibQG~?Wggn@OrePMAxLU@O!3FZ&PsI*d0ulG)NebRtx><@mnAmB;m$EbkQvrsZvy#?WHv$Hx*S0{yt@Na9-~V+dB<Mn3yx}<U6Q#*n+}o@#>!vH-ven-M0rX{LjG+chk+v0HZj&A9m#o~4QwxoF?*-__Ln_`C5aO_m~&UfE9l1*efVMbagbFq8d<cHLjyV&S4oXaMcwuiXFOwi%j-<sHP~%fvXhV=4)uN6aU`FJ3$x>@6<VG-ZjVXgea)4%9f2+KY|(Px+odNBk6>Ni5cWsM-eK{q|GJKuGJw-PT6GTF{DOVa7cy)1MisRC<8tl5X3<}!XBD*9c-Mx4e1i;0t5L*5YdvWeSpqZVSJ!JnC?>^c_Y%r-KPJ$WHvTIT7AcO1w#{9B2XFcNi#c}7Km&G76%i`hp+oqwjxmry{t)LzA-r?3qvYP2Dz{wq$m<RMY;tM3fgdc0DJXAn@s&81L7tqdXCzeFzt|r*HbI5BN$QoXYhpf)CllpK{`}>S+5Gf1IpM#~3(%gQ&)`2VC-WH$IX`*x@g@G@R1u@&f~=<lJfXiBp|mzkCWsmXhZg;r!^!7$amT=hb0LYyfq8z=diB=#v04_0X-t?cwjX&%BA7iKbZl&w+S+7<wc%G!*fj6xdE`F8p?c5oIT0I?pW`Ic>f$eITeB#BKT8idp6U(5<Sg;@76OV%o(S6*5LOPzPlLQ)%xAbVYhIz0#U4t~46NjV_HiFVXHI#9TZW6c@BImZ@60-|wbP#w5~|f{?uaji@3;yYn2unJcEsi{(a13(=p3Z|-D3-BORxet5N7i7NTS=MpXT<HzmsG7b0gm_%ZvU#D!28T*;6IO;<DGD`&JC@nr`)XgYVNxI!i!$lH+tV={$e;_SOFPomcNZbY8rBP9PrUARwdaJpbY4^S{1-xBvFQ`tBspPe$o>Wt5qQ)x*f5NnaeYop3G`H+3>b0dVu41}J#Gm<RtB5ab$<nq;K;aF`LC)EM8*f&~l+h>G)j<WZOeL&QLq-^|icnjquc3O@WOtA0cNd;wBBKr|qUpEFH!;FNLEkB7w~P6vza|N5!7n4Q5m&$odA6f)|n*rv`T7&gisil>92c5cCEo&I1LelnG2JY~kPuYekLHN>TZ{$h;GEjveGF`Sp)g6@lk(253wGmgWs8Sez&(6JGxGiWH-<{!n+vv~*yP@~a64>&%FB(OADI{pL3Xe($1_i@x4s4w&LZ-XAxD4!d2%j1aj8Ap#fx+AS8<D38);71E9;5j?QIXxzHV&><uwF=^rZhv$Kk6vpG@}bmN#>pkx4YEl;pJBXSY(>)vxV8S-C_PU{?Ut>n%DMkLSA}}k32g-oVC!853vzBq0yVL)t}@)CHZd@5Do4~QC{J9BbuiE|i!^xt;+3*}i`g}6h-LBbLEBnPr?7xg{Tk&L@E0r@3poV6Qq)=saoTXSabj_b1Pqk*O|SHgZ0~Q}ozW$F2#kc>#6BEiczMP^U|1YS+&f}ac<8Dt$GgE{YlIN9t;JUJIF<lUSXV3ttP|oI8UZ2r8T`4xKTS#sJzCgf#SV>oQG3)u1w|x6H8-**XB%16z^;HsI-$|9N*&0~GdT{v;y+el*o~bKSMhy!S6QFo`A3__Xar8|@e~_?PZoshc2BuFi#938mQ_(73qe)cqzyY}oP28c42*1R&&-F?jhD;r1-2!O{l%o)*-pGez0*&c_$1N4QCfvfsvi)~AO|S3Vha=V6|A7bxPv(;FQ&mPxtQ>hO5BO_Z3_~)&-LcyJ8)o#uwae4lwY}Tne5?Q?RAgfAN&hM#$Vrl4v&ryqUBc{m(Nike!-A@{?AsUYasN9#zAB&vOy{<V-D5@zROhbX#-?j5}8<1#<k#h^zy}IP;hV9Z5%KB2eh&UzmW_BR>1&zB$~a*XJ=y9_YkY`0{5SwAqlWcqx9#%?2CHX>nhK(x9<*K!h%5W9=srj%4b<JL2E18P>O&+5$L*wrwr!&0yxEgd-e~CE5=qCR!2}my^&lT`->257*ImwDW9VXUdQ7~YA~<Db`#eYNsf~N4+6mllVii8S0KpTWDYYf%v=<)a+(bVkEWP%9mz7_nBIz*Jh?mI^U0Rdae+en65qfMppB#2kkKIJZkdqYgFOjT?YerbQa%yw!O1vYj1e>$1Q%Wr`lFusDEU<Jk$Ek)1Z%272*bn+#1>EjbB9nd!-m7}m^twh0|8tZ2X(D%&!LJ@$Za3pVbzVU!Hn@hFf``h3PqVei~^zH0avP307a8Ja{{)<une+EGg>p9z`kpXMhu8MyA&#xRmDi@tSFb9pLolK+#bkSq6QMs<m#2fh*m_h`EGMTLwvKg7B%Nep;o(h@k$jJCU39#0M;h1N>}#O5)ZJ2aH|D}wz&q}vq#M?Y-fEoO7gK!JMojpPj_37c6XaQ4<5E2JbnDc%<56@hKKki>)2SGaxGWTQ{S=>S*)gs>CuImtn2}Z0jk1HXjGEX7F8Z`8;#Y_5Qhkp%C~4%2YxkwX<!oElcTi1m^iy!=gX<uugf-Rd9@i$%6o0lQ$KOG5(+%Iw#J<%|6*9c$a;r|QiXC|fOwTnwW3YGp1<CI&pNze@!%=NS6kn*25bkGmkNZ&-#*xIwC@;D4jj#{u*0QwNZfZ6<ODHC%nxBk)mdESaU#K$#ZUAp2{^F>Lrqs6hirGI3egdem+dPQ`NTm;tO3(91l^Dv$pm^Z6Nx6@h8gQE6j3P(Fa!ws?PT+ejNt<w%`X1Cqtmmry|Hiy%Im}Ib8k)tEIj*B@VJz+hMVj~hQZ_LaQq37Y}^OoFv;Y#4Bx%Z2(VZJi<ZM1qrl9hQI4(@Ut?|j;TF$KkjzaNH**HL6DDeoLA*&fO&ywZ!xH%o#~;U@pRSljt1_MnoHcarM_b5Dd@Ycss|%Ip4~ZhDPS3c8=&ay)_(_j*HQNy$aT+y9y28r$JVsIQjoq%l<}~S_a)HKXlWK)(;6FcDGil))m8Mm<9!y6I)|2GK$o*k-OhS|yVxhG1U3t27O{>neTVk_`USR?#pLCiawx~N7j!nmY>zSNqdc0&h29CVIk=MZVMLwK^6dkcfMlNS$j-$`GAnF|Fprr8?7D@K$Xt<^7aIb%Xd~b-Z$!(1|rP7ee;3Q%0aoLpA?Tr#v0c&F#?a(BZTZ@7?j*sm&^?a{i{@M;=G_S953_G8%c?eT(_Z5!dWb?z8@~eH-XDuhVzddSikJ{U#_V%d#S3YVDUVD4eE<I^iJ78BkSDiy|lo!iSyJoaxH;HR}tc<phA3EhZ9MUtGL%cAhc9rYV7M*NGHqJ)LESq1NN0WmAyp&j$;>?{yi%mahc+Xa0ydLC~l@wDU!lQg<xLkHk`LIjo?5?qnyCOgr-ID3jxzJG&)IZmES9Vsk<L@&K%(5;!9hCW1uA$3j|G3B2Ao#nCg+z9YmAia(;p+w^rh##I9d;dYwtF03tHI_btt`Hd80ImnD>_-ZkQbpp)t*mYs%=rGe&ozFn%CT*uZQ+=gxqM2bI4&y%Z?#uJ1_g(IveXhwKdiisx^mquf~zhD(RV;mC9<@nt9$~F%6{!x~1kL^x5R{c(3Fp!=JiB&S-dec8oIcnap}~$ceNq!Z=5jhhj{>N})F=DTo8}8IckNXN_19=lJbX3Q%l^+=UrB+wo5E8Jb_JO&U%nF$~dSq>@uwuwG06OP6~d2en2WFjT_<`%x(;`@*@+ddsGLU@kg4ogvQqv4yN$c5e1M_3>2kF#WN9R>E#_63(L+%p`{Hp0@^}8wsF7C)=W4E1uI)bF`mx(YCwne92juI~+P=_n_pKG|L9dKGMm_=(5N1ZEBRdk3FEO$h$jqhkJY59n?l}|2|$fmTM6?<|)~;Q?)zhrF8T>>}sKg<ys?W-84$*S0@?z@pXEO;V{MIU#b<#3mD0i4&*iCX`m6OpQZ_Bm&F7Z%|;^%V1*Z0>13{=EA_6)CeYe6r>kVBqyq|1u4s>plhO68;lPW%fPm&G`hxzha+UE#tdfi){6dn^D8Ha^S%V^R@Pe4fzPo|!W+2Hda6r%v4*g0cpoCVD(VY><WZ0Q}<TxD2xFM4cM0G^NS29SZ;ucusTNAzdDMQz*q)!fV6TCMT)-*j-Hnfqb2z3)<EVh)NC7fXh5RW+uhe<!}<@p&3eF-$S4b|n_y;0uV9;8px^ub`b+aEr73g10_)Jl4f9zA*Z@NqJ14wL4C-A500TgkRy(S{ZjPcQGjZa#jv^}6}=VWge*)+6J*#e}5I)?;Y0)k~f}e7ZaAC4=GPB<=5}z2wnu`eg9rNo&wdA3TzUSOcpj>*?0(R<l`NBt|x9YjYdJO!D4g^Vs>+-NEI38ne<ZD%i1tG48_rUnG}>wGJ<Y|6gB(q6u#{G9-QCWPD=9k>?T}af8cAGS2$KDU6ybN|B{%O~qN!5e<YKPiJ+z?Q{)4;2ntXc>DR#PO3&Aw4VcSv6w0|O5z7tV);leCdJQ-H2ozFcdUx|Xe`ayR?`}DOoT0aBfC}R+4lD`%8dW(``TcwKhNeJcWo$F_|STD|Nc(=s8Kf*&JzsZ!cHN>EjQ$UW^?}KO#llYFXk|~QScxlPtqmR1;>!|mQF^ov6Msin6*m!;$TBPJCPpwhV!ZLSm<T_1^ln%iVSD?38w4?8DUoNF;sP+uQeJi9*{<So~0Lj<hi3OHiA=$c?OE;0deG+F<2hiKdL-3#pzme0Cpn$e>YCj%Ew4c$sG)#_qM&VUT7sABu8M<gWJL3eTA5D0wNdx+u)Xl+S|5TibZbjumw^)a_lK)k18#@tXh0<6JVJw89%-el5v#~Vp{3#NKZ38SCOvQ+=}ebSwOlT)=;i=zo(U%Rj&80r26P+omCZ|Wm=6Rvr=;DY9Ux=?WXM4Y?P@Kf0v8YqBhJ}T>iMQt2PgHb8Z7%%A<5*h?C=914Tc)?#xaW`GTC($LTnqUD8>rPI-3qO=se!ahdq;gmcsi^YvX@L9q8QRFl0`Bg@LfrX^F&>wI!zTWXAAW@pwb$r)4ExTi3INGo`2k^U?$YACc_X5Cfth7P~S6;J{Wu}G4oT5eplwTk?q`_?C}+yu`K_o84=6MO7K2#|DCG+}?!_U=NRzScfm%L}VoxC<*7qE*%=<{MuNN0X(@)Z}>TR;|qJSUGrusob~l5)7}+)HNIN;d`oduwM`k$3OE$@HR~c_uz<w9b@Yk8#8g$jN+h6*zxaWbyC-z`eS^gV<+jQQwX1l2Ame9pslL4R<B+7zDvQM-0Ic`(HS1y(Zq}ecFrI}^%5-ry-R(+8-ypY#k=bB?q2Y#-TY_1dQm01dp_6@Xi^kZAAWAsLFq6smcW{_hUETzOyKI`iO=#WyY-9t(Q_kSm5$sqa%<tU@pU!<Ej@IMb4%<@=U1)Z=c`tLmc{j}SmRW4E~zc+mB;0Q!q-b@QRNX<@>(#8A|4zTG1VNyIB(n0{0$zL%x3%AjIhV}55cG4ARDLfAAxIEHl3I^VRYhv0)i^GIWz>}7FwqSsPeN*>G;z&Cp@%@8gqJ<x&_Ya3y{9z6gl@7sU*z&$r_-i!}iXN><#JBZ@fLyKd!wUXCTa<*VdNNtIl~xNFR2$43Wp!8<UQ7Y>`Qe$H{!W7=`B0M`k^`ZPd*h4S5O({5<QpFdGod5YGt5U&`*P7y+Xnjg3wbEhoqx7N_}QG!R&9bZ_wMx|Dp5f*zu97M16Sr^}z0p+?y}sv?iF_uHyLZLTtoD~@GpJc-C<gW0SfKN_fVM?U+;)wy1DR;~5(o4|!DF?Gbnw2w}aTB4li!EpMZRk<T<E1Pih3eAtm7}X}D6oK;a9-`pl2WVW$t&J4ZM~xNR6T*Sh-n4BB6;VJ}Cup5wEkXQp+YlH0&L+2OHaSznx08kY>v|j6W?dU?<*w0schhZb)3M@4wt8H@kL_VLOFP^;)<%f1u}XPqf37+w{3a2*K1(DEjdu-?@@Bk_tR-63<z69Bi((C}(9!UGj&#l^0b#uun{rGVp}e)DAE1PurZY?uO=?`%siKRxPTJSfbfd+>u%a)88nc+p^F{wO9dOKgjxh8ynU5Ur!B3zPwY=C-QwXBu-YP=Ivw1K{Q5_~<PeSK_iuwrRM>m>_-YDw_&))B=yn1dY2~aUjMk1{UUxo*y<CAHspNM--zFpF8`jbo1yRknMqv-ULMUjl8c)GeGJQ^kAL~FwQ^ELk4ndIdf@qNyatCDclX6kajwORJTyBF{Ff{$v;&(d*zPB=FGG$6VHV<qLY;6wg=jOR;lF`n}EZYyraW|bPV=s)0<!bQQ!oNkTM;hZlsa6#}$;z1I@%x5FKC}d1Woc<KA=w<}Z+so(ke7wb&YFr#~RU?kHxJVGqWsu6SQ6f~6@ogJP<n@joHW&s<wMutlSeFAzsR7Rh85$nZZ8;7;q~gy#-!1QpB;R_r|2pYyQ&Za4cw52*ZO@aF?Q?8=#~PvU(u-s!fGKHx+Cvrz9GIW+SVuV%PJ%P}cbI<ar&A4xP77Im*=`5ThH#q1z=hyqmP1V>QG{~*8?mEGnlMu^CJ4GU?ckRzFSXgPX*r^niH>4k+h1`zC~&GTtGd=&U_I)q$oq5!r8V+<a!Zxg)M-oYyZq>nsJ-4597(fzakiM~F|?m0!mE85c-I&a(E1IF_oqM^-9E|jfgfg3M3zs72upR520x+U40#kbcb2A8juip^lwquh)9$U-eO0Yn9#kMy;JGb7BoG-`0m#lLDWSBbh))B{z&*RHl`={$(O`I38&_lyDrr295m%T67&-f}C`585sQ9mc2Zr1~wbq>NU&Y|ZV=ZTx{n-|x=r)dEyxIa<dnIf%r8RRSG^Gw6@^ULnldLppOSOsbigq-P91qd{BzXVw)d3!SaF&3mcR>f+TuxTXKwc!d3EhtmKv^wN*~>=gzz^qA0qoGfb2XQ+0x!|^wg6RU#Bf2_GNNLuw&LIiTp1CPh0|`Kq6jj_z+3ufGWbXct*o>%x8P+!Zcphs%pW2>od|cQM53Fg14~d~5uJpdVw7JD@{5V?^S?0wHH9n|Im*lw@c^O;h(-fmoOp1nfqabv`4-2~Yy-OqI=ba^C(w9_cxSoMLmwtLe=HSXF>+=53cm(nB3aCHB(0N9aBq~+PRKVLMAlSO!iDjPDt$6t@N3sg`fl)p6zhbPq~NG@%qxg4z#e{a6T=X`%V|<5e{6A~gf85O-P#f581R#a<jxB88&$<s^*Dr-d^4)5<r7-^lZm=E0g@jwYNBtS6lppx<VvpGE)h%8ylzRS7zse2Dq{etfxtxxw4P3HHYGtJBiwRkumb+Gur?{OW&|Uo1y8%t$m=tyG(?z#`MDl}ekm``L$kJq>)~C&kZViTa^+IVD*Hqg;%ivlxu}ZzsG7j2WJ)w7Z098l1x{)SJ~!Itnd0WBkQz7}<kU~+GtgsfHKx6Ss#E1scOAB)mCRDBnURF=Ygq@WHRPJHFPj;oJ`Sddb}m@i6jic4Nt{y&&28F%IYGuCry;=5Ki$6?lNC3&iQUR;vSLdWLr40oRCD3qu7FHGzIgu(?NRI6j4UnLp<e2}jBdD=25n8$@1;L~4OhRgQ<m)#saouhVZogXE08!M**cTPCvnhabO-HWFzF!1C7GcZq7f0`K(1dPu2`21XjE55u-p8lCEu-++lAN08!5N&2J-AmHoJG%-CEsXkf(etCZjc~kO*R&8w#i6BQ_U!lGt%$r`v<$PrGyvX)QqR&4wyWR8-`QEuHHmN(fsrZI}aNQv`?rjGvxAib21C@y8`DgHz;uiX_F-%Bk^|Wu~T}>zUg2(Va9oLD#1)`!9mTF;E24y0};A%won82-~dx>@97)w)HD2e{FBTcI|2F-FWTNlc+}expjuPHOshW89Mj`FfTg$ZW&e>U=N^(l3VZs)L~K<7!RdQ7Wq~Km=&-;0XH<sCtH^PFhMENt3D{u2>mRxX+$Oo^;Y`nCAFyjn-#?(Ew-D@)}zOdW9y`__1yG4>!-2QYLD*p7lQ=U1w4#b_9v)YVSaJ2Zmm?8sGD`WlsRxe5;Q)=Eg&1msk~Q|3}>Kry;O6`L8`!PvxE<~t`cs#WbDB<iH~LzP~VaPid=~3KY<ei%_5UQCh&AH6UoU0zjdq;w4{u%!CtO_Q4#kA)J~vM28VH27|qIX0cXU?ESa7P&>mp^&%8~8=6z0}5Kb+0sBjw?nq@2wLxRIYP}#Ng3#5R(g1!x!Z00K2<9(h2tDn&~uBxqaj7Y`I_L8W!^HnNuKE3S=-S^k>!JED8*Z4ah_rHI=|Gx9=;Na!kgZ+1JJI`M~`}pzYN6KR*ln5bo?75mjBtLMK`j-CKxkOV%3Xm4{SGMeUH2HkR(404~-!F^Dflt;5!nsDTZ=J!ZGel{f(>ER34&d<7u~Tl?Z(2UKWsmVlp%Gvl);Jwx7>Gq-@};_3zPd`~B9C8sqdMfSMqjH7`LF@%+N!&iVwSgnHLmLwh2f|yAhwV7x@?koiIHyUj7-}b#CB}}Xt7rC0RD@NP4IEmj+ZO2<Iy~h#R?Hf6D}=e1SR}zxkROjzvKy?P8D2*F4cjjZk^-N6p2-Pby>@;tF$cbAp0j1yr1dk)5{w-Q&fC*Wvhf4c2xt~;NYBjfg(%jOuoB?eR<ujFUedCr%2v$n{j>JxXwtsa`TSAYPo5@hiX?`!$_B`lvg|3+A@&Q`Z9+SUXnq-W+!;*cEahbM38R@W5w}>Se{azv(eTik*DNAbH3qa^dA$Gun=JUWK%@HG;R6dn~6C1n5JY+5CyuyVgj7Xmzz~U*2x<2W|Q(?4bpj%jcQWXV_Mh2g2d7ipR8rP2IE+bV@Iwv*=BEH&4R8qW2|GsSkHV>bM~lV`=)x>s)u&Pfo-jK6aw2_xxgy)ODN)YU{@QtHwnw^qq3rJpeD<o*(dCZkIUz@lRq+aetQ8SL3ZvHi0LsG2r=4KPp(6SS)3@Z&jjK8rgT97LKQ->Fq%kyF_C6FPLalcgDxn*aky0>-}>zMe28_JmbN89hy+JTkqT@tsCJlrBAYO0GnBm&T5UlYK2WFb=v=5Au8j(&Y!X&dr<jg1!V&|)#>u@a?ZS49y9FL0KQRR*?*}eO*k-j*Xeru`)bSv|*M5L!s)P`+J8Ce@JeO(+>`zc}aYl#4I}+2xdJr<+*b)NG5HN{wZQcsGOqDWkW|ZrupN3plRDq=C9v~8uBCN>mEMxf@%%@I#^iyN`B3L49xg@FJW~-I$IE7_$MVawMIE+_<*N9|X!FDj2a*7*<n6NM={G)0@6S!kpl><hQ!)>q7M#Vj)+uKjcc?PUoCD#$nSZ)=KwThl?Ui5tPqS7u=U-*}FmKSa(ds$Mf^!?eI-=DAfz3PKgqJD}&TUm2*LD$`Aj6rX$FXwC>YzMm{c1pUnyK1>sb-5*jQOAha9445GUK%4{^m@ajZ|<`R&1Xk?CskKvFNAUpI2>;IB_m-97#5N-<&8>-t0;Rcybze}ERJJp+(x5>)r7<ubr3}{rb6w6Zq;>#Rd1-@Hi%4Zr(WeVE2DQmspD;OS(Oh7K}}yJ2A4eDbAZ7TT<H+wAblCPs!Uw+?45PkVK%BfgsQq8D#KZ%;W-<nskPFTRh<?(wI9C5d8oO^sa~KfGAPqP;?ir>RIwC{uZ#^OmK&~5%cT7A;Ms=*OsW0x;KTmg?@?bcMI7lvgN_7N)YoCx5z4#Aaqt|4Q}(4t!x+oHgS@}cck?n5g#vxDz5s$Obl%TV)|(}u%PZ3stGK8&W8+LqSS<Rd=&xf6>hM(otOfJeqt5e-&W?<x4->X%I$O}ag8`|L&ik>_QOsUTgi4hI{7_xRu=(&fR5`r))-cXzC)=3gd>b8&N9i_vdO9DE?m7h<n0J)MNGK4LoOq7iN}vZPi%~LDt#T1kMrL@0DNQ-@eR-5|#Ky=HXNFIu2(TIji$VdLW1W<wHOptpvG;v4OJHu&nW^CklX)m1sacxr9yj&^b~dqc+Y3F6Ec_}MPgPr$2ZfDQV*4wvnnboe#JDfzeu7hP`D{Uhc|W7ha*9ORLd3pW^HWEfwJ4S=7%!<9di%eeRUYHw?`So!dn}<$VQdvO0-RFBogYoKW+-;u!OMRfboSrA|J%XG>b>V?(Nl0r(&(mRj#f=-Q%2C=M<W{NIuX#BGO>*gf5a{kpct;m%NW)MTjQjkO-A-uGQ-K3IuRC(9=hl#l?|L>&*r7{9+DE_b)%r++3n`A`)o1K50EB26_DGY$;@O!>Saubb}7t_g}#da_`mO9hSyQ210_iNCresUqVVuX%G5NIEM@;i@#o_(j<-(`c^!u4E<N9}@tZ0vrt40eX`XZ^@eNk2!bO1dqF_a~3w<#^vEy3VdWv|wR9Iw!hBKJr3UJoDhnSl8xVz^L!&cB_HqHxVE(hiQecY1R$#ii4e&iO~sVF21)C_lGS&!X8S8uoQZNm?lcBM^IH}=K`(s-vbJy7kIGV$;vU@I(YJp&?B_en}~;p1CX@O3yx$K9F{Rc)8ouzck#<CW~@?%>?8d0bh;eVK3TOyj<cS6A+j9sMA>Vs*f3&+UUrwkyr~`XI8^!0MY{iSz`Ixy2kAo{<RwSfS~Z6HG@*CUP414z$1Tf5c15HIFnCOLeV0BNm3fgH>8O;@)E`@-k?%!wb(Vs~-O;<(@ToWRc?aewG7|z#ge{3w<8!ch$$x?H>F}_h1#2?#6eDvU6)A^M9?3;a14K6>@Kd+*={{Uqi^f6-d?-NQ{h$S3bJ?PF}@Zx#w2y`4^OXR`G{UMswRGT0VF-n&UPUjuULc6wmzr(c;T0>~3Ms)+F%2!GB@5A)38NG2S(>s>MR##nYv08PBG8sk0xWiJTjfc45=;gK`ymWj&vBt<RI@Cj4^ioxcTK0J*C2u)c<|3J*PA#KPf~_In4<Gb_75X*?UpE+v(oJx=iNW=^jbUi=q|#`PDe%3@qm%Gk{HACqV8-zM;$VXz`Mr=yjcaRvWc8)E<FGRWgUvSXvm$mA{<Hjay+tJI+`9sYR6_fK=gG6Hrr$OP0OMVYHC>ghmdvgKQia=Qm2)Ob;(!^H@%HnIAPWHM)lPiJvH3i5N<->3NnLn}_;*W4mX$k)iTbMx-yJjFZP!h=NuGLpU`xhL{|O{`4mqdY%bO!4k&mVH{y{rEa%-7l|eY^-l=sB>J4-m$S<_O-iBUqwdyQgWIHoTj!vYl=qKE*@36r2Bvg%g=m%$K{+drvFIk^CjYU$p_Q>m8Qdaz5*!~Zs85T_}aF9LA%z^y5<nc#`xIW`a<+G`rN4O64k47Cv#G{kv&mXapR3qw%RM*I%V}|xQr#8Lb{xoGFcfS%qw1a{9E15I}g6`c)mL=)ro89Lt(|VdPrbzf9-0|5JUb7I~e9$0sn4|kJ8a2|HsQvA(jF!Rc_U2DK>ls9+^81R)4}T-HrB&9jgn%YTJEsa-6OhI^o@8&I4Gbc1z9Ust}F34YA2)C;>{D61o?_gwv6q)1E%3Oy^FJklkwJbe3|6otxaQ9PjCf`p)7RA>^OgrW0J`L>`sM(QCw}tTuf*MpqaL_ZLYaSg+Eq6@zMJ8}sHw^oMv#yF%MBO!&VZBJgcThd;Sdp=$+%T6wgrfN8Uus?10T>{i0NYw4ltc+s2M{&;#x(qBL`z;!g6chC_K6d4$~&O8_KK2mM3^9B=i{snZoQPS6kx~eV|8=W#;I2+-|Y<?;{xFahPsa;xP*OIxg(q6EsouM|{;UCF5Va!ymq2dFzSXjE_tu*~RO4DwFOOGRXIiJd%e@e(wgsfpKGhNNmokkf~SEd&>l57gltn!_?`4!=oU@9D=8qCbuSQ_rDj{u$4vaSA#ti9Q;Olp1?hc%@$4S3u=)a5_Tg=eht-(4Uox1HHxF&SlNsg011_V1FT@ZEq@w3d<#t(41&lfEN@UG*VQ!v1q1z|o#`j63|$NwzdUnRiq$eKG6HAIG)UtagB&#nWV#b{-Bo_RG+bZXF=CGNP-LW*mJ(JPvDYp1)HFb`_&&`yNJ>_KmjjcNJPtNiZ}-U{i*#``#<ZgrzT?J@Kq47Gy{~x>L-v{@LZxo#5rWk8HgYfyoiIC#{MYe{g}&qwra59*{X=KPi-<Qp1?}7PtT2adjajHHFrjjimi@*CV^V2B;P8AUF4pJ#pdHN90lgzf6RiFBmA2?YAxZqB>Lphwyl%EmAI4g*|}(=am^l^Fj;}XALeYVPKIt;!Pn&Enubq5`#hK!U(vuFOAjnd>V5#qG^8s5WH-Z6&eo1_Xg)n9mN~(uI0>?FY{K^e*Ag`jt<M%773<4Rw{TmFe1HG90{H))9L6kpe>GLi9QjQONGqiJm*Mb3vmfiw)x03PE#yE0-@3g_eyKu>4tF_yE&)L=8CH3L>@V7C|7Plf&<=aHMd&Lm(yxqeo9c;wjXx8-RUK;IOEZ`-+o*6u)wfEgn_jDC?l7X9XZvMIfggb3*`5cJU<zw+l<n&g%BQF1c}+)-D<TSk^pa3JeMROf}BaxW>wpj-xZxrCNN#&<V@rXLCqqa%gf0&!c2b>7oUho%2$LSkJGiT4pze#CbWrcMIfuTbzj26qdQ43J<Xvdy6fpDb2wE3=jk9nIl6NkRS*)%)h2GV;PrIw>7*84iO%r;eJ02mXrF{858cu`yQnJ^$zqnMfX4)X7=%h*YXq(W3d^*Nx^%;esQ=*b3HXj8x4B?nsR4fN3V<!yFh0xGjI~_Hmo(QdH`uzkGuL9WB|(QYohuk@tr~Gn3zMyc$;AyIGwJ9gIs!XDeh*xNH;4@_pk7bhJ3INtQCv)$@d_{)dR#V<hAIN2Zd4w!*5(=#k?95fRm&lDuBfX3GTs_y|A2<sWj0M~byqH*cGZh7VPyN`8_w+0WQp0#mc&f6Rk>V=>HAz+u58nqedRW-*;iuIvY@<c57!C@c2&qgkQK$^HNC{wk<?t-2O+Ik<WaaN0{01o$H4@l<(pm<Zdc~f9*hR0e|co;V8hX;TubL73kTmn46|7=r^Gv61ZJP$=4<Hc=+CP_kDGH~WZJb6RAaukGI@%e3_UY3NKa`p51mVxX603v0wY)BP4(FHIn+8%Jr#(r_3def=kU6ULI!lJ7xvW$s_NbWFxjh`msytnvIb_}n46lI&223}S)rt`hE8Cw60E9u=-&j-a8|zBAwoB?4NPUd(^LE_G)tSEG0nzgrYakzZxy7kTtTYJLI2fQcHi&6mIhslOm!@i{06hUrNz8h!!oQQT%$4zs<#arA70%$Je6b!>#z>Iy$0*>Vm2bqhv0O3+uIqAALe(0$xr4`d`>WRu!dVZ&Bxn2{Lh1@k9VKOKTS`}spPSS;!qz=L!3|3Nhrr1Gc1SD8p4jw(qzo?2M9(NC7P5iS@k?*y4v)ge_fmK5X6tg>}#?Fo4nf5Fih^%u3%+fn-*Hf$SPI!RqxpSf{G)hY$~6udHubatRWOy14~q_7P=6yA;A3??Al-L01i^&uI5QCTCNwDtIS6)vV9}8Bc{T08>DojSs~w7C)vgf?d6slOA>BZj)Da~8!eIT)*K0|?xka4HoD1ZSgkD^4|9^#U4rq@NCathq8<2JvxU=FsA@(!r2>2Gk#`CTsiqzwAV7iV!$Pi1`s$EfbtT)Oqmd~J&9zbLO!dKS)nSjQldm$xM1T>B&ED=6i=NE15Mq=ACb6U>%$ODsdezHD+5EEBNyTu)(0|q|5tbNk{wQf52Do&s^`c~^(0J2J%iW;Ha((_LQum=DmWTDNOUQM*gp@+%SJMz|{1jt`){4Ft9UNriZeuS{zH-L>sKUnST{_lHT%Zyz+OTSH1foD%7NVYz{eTFGcA=P)0d;Xxjx6}|jm;tqRfq<wN`X;DZ{1pc54RSxafYU5^KVWOprI*&#BuD|3qsq8%Y4kE|9I4oyM!+;%y|-iL859Sc6vTefk~$$M7_XU6bets(m}(F?1rUVkWeFzz<=Xih6f%ngUP%ew(0iT^Tw1vS`yAZ<gGi<YS)^sx^Ir<Vbgb=XO*}KgQv3AqG4nOk;T;GOLBynZ1P`0nAps0c@5HsJ=#|kKiFn=^YVwG+FVrv`2q&luS0^lHN9?TdR<X+arTiZx;PANP1!{n;%-HkKal7mY#5I4a%;gZTd=DH7JCG5Ok}y*<t%SszDD$3$6jabjz&QC8C^CPmTtt(Q%kIYvAeQh_Q$${HW$tx;?nt62KrrOAUy6lC$eeMX`!K~C|P=1Xz78+$2Hcei<Gf9yV4k>O+nqD$OsMHXNr=!$+BXpkcKq3iSQ_}1-7fjRjnw<)m*hZm+g3CP;{p0tRp8&-L*TfN~^F}C~rtuVB<U+gl?Gz3_J`a!bQv^EOg#W{{QyAwY!buO7O4r(mMx$1hxo}k|i9&v$8D5nvra+rDP^ca=-@AAX@|)FuFk!k+sM6B!0hUolWA{iS0?8WM=)y<m6#Te*Bk-K*^u{7xq4?x~r<I0g#e2&Wv*$3f*0GtLoORTeoiA2VJlPeq@3KhEwdUEnFqwt4r(fU(K!j)in-}$sJrmd8Kfba0}nB6|@DiTyanRRknDBZVF%P+`blEzZP8Q6<oj8QXXDQ$tPzKm)a`xYgkSSPSIv!kRHr@8+^xV#ALLnJs#5@DQR$VP_LGqcF?P1;E`?LAK-7p9Np5l1fudZ8=yLh7EC9s-6-e<U2hn~W4sXJJw$OvN%s`upBWa#TTsu7d!ueq=}TE3YFsS2abGRqw*8%;<(En_jj@{S+20$O>3NTux9nE-W!X}VA-J*btU!4<>RD$O^$20cG&<e(2C+<5AT!xQD_f%`aKPR%mi!hmp123-Io5<j8<XC#X1Cuwwp8)k&65>x<?hOf&Z%W@<z&-eS#H7)uTxp^tEXCRc(}WxR%9_~7#%}hC(DL39wA@4gYjMGt0(U^f>w_GL`;C+MH3ek7Hg@5=_YV8>gIPbM@D*!3M@HqvjUJ8Me^Rzj~W<)%4`6AeX20Q&~_#}IFwf%Ixu)EpNMddrA(8wb;O)UGrM`WoAxIf^@`krTnQz!d75;n<o0|oEFLU+PohG|h+Ji}mt`k+O_VhzQ#N>0$?jZL#GaQCt<sqadP}3p?8my&LII*44U1X4)Ts$eGIwR1^~}I?%abr}V#Q*=AYA-ZblL71L@^DK7HF)B5$;_Yp7p}2+#!qW6LFZJL&BGXMOYz#`f7d>FBB4TOUo`VyhO+^#3_1CmqK!iD-H2lq=1%2v}7P;Mo8!OId?Q1I7OWMlH<FZF3HezCVa?`UA5N-bU}HbKO`_`v>{Bl_Pm2`CPMLriVq`<;M|TT=oQxWnrLlTS3Sk}oxWS1fpfA*uA}+;Sk3g*ifGIgqeTYg($>t<5>vQ2E#VGkq-V8$yaQHpfD19>nCC(Z9>7O{^lL_1NW;w9dY9|E#}h)lF7wv87mk_4@BsL^Hcmi1R)vSJlD*ySDTB4;<`1n};l`PVA9~>2HyRH-{0KI4SPz}~m9#tyf5~Qex1(*1S=(xznW<`5LA>)PADo_m<vfE!HiFz^6w8}&+ctA-=x+{@!11~qv&JR43PsX9%%~@Fueh#jPcW#gK+Y-XoYQk+NzPh#DjSaH7;U1w!ac?CKo>v=TfFq>2o+WM6(rJg9lhBN+sY@%#vYo5er5=-7SCdOp24`J-S&T}!L;<ufaR(C<}|JL?W)^NLqtBYg(I)qh<v|YTrMGF1f~P)3f9WebKiLA;YZFk9yxpIo98co^;Og|YVIy8I=g+30`YmyhI0A6A*KrR$&8w?Zw=Mu&tM`p8X+M}W|l;&?S8L2R;>2hn5<IjsRu(JISeTS^*S6?0n->oB4}>xc+LKfn%gBoLc_dRFtSSC_Qi!RG}vkIPTXN6>Ud*Vb6QRu#$}Dp78%{SIk}7gF<l@ehrSf0<J~^<kgQIe3j1x|TN^=pFVQEs-f*+$?G@o@=?1aiE0%2f^tFYH5(YO>!`lXyuLM5}+Erp_tCrT(#CIkj=ltpW&Yo{nrs~qa)@=V7HQR?%Y5!NzVki0JOb1aTNCcdicKjjk3wXrM9Ac*d6gwL#bc~Oz2C8TgY+2$@Pb;PpQO;vp!h|}P9#P0D3Q=X$iM>9i@8xooG;oMX3WZJ%-}s@|?8-aQfj9KJUB4SdJu?c9ILh)5)VB&=WmV(Jpg)4J@!GHJ_voaN6PsVHWk`56d*-0r22h4bB|@7U9}F<t6i9q@S}Tenwi%e5`feXnvko2mSD@I^Vfu*&!LH9;Xz0SGz2nvlPBWY7VMLqUR_33(x`|s`Iw>I5Mu!;Tg-d78!;`6QKlogW1t397+KXVx_?Ud5ET<)TQ#ugx?C>SjOd743D4II0({V-z3UQvuLtYL#k`xmLW>fTt5~YKWXc`4gAJBbg&SH>?^VG+KjYye*+fbn^D(z^sT3i3}wBOdF`hnEoLYqA`PS^qCgUyNywoqP>d9Q^mek@f|HIf^kh1voo(jE^{vE@{ksw-qw|KMnI6AC`y!FF;kY`IPhCn^`SZeO~1|KBg3hs_}b$)2@mh?Saidf>IT{KcxfL`rWVR}%7$q4S<%3B^ibR6Vf9TTAWDr>^k;Ph{d?ig}^SiAJW(nv}Gxn)_Q^uQs5;p+~2z24&HdpzcG--~ldVtIvlJtqKj{Tu8)`t)O64tCgI@E(CjlwX1423a;D0<5B{EEsriio!H+Tmr<UVE`V+7PT4`c%Adng$V*q&u(hdmk<~}&f1iybmvq0V4@B;VntJLCkY79;;VH=z^_AJ)-3mY$+Z}oX<uf1T(?X<Cl^;{jM^1U{Fy~E+CmI>)oso2MV;v4@9aO7e&R3KL(`!O9Vp4rN^dmD=#a<*1r?L|p>zp(efXPPxx6x{nA2NZFDbe=vHV3{ZIb?$nXGJEDwkB^kPx{5Yo||Gy>KT((#v~`C;|Ye8DjCQ|Q7oi5FIq5QgRjg|E@VA=AVwSxnj=2nt*$K?!(;2f1Z-jzXcS9Af2=LokRgdOfs-kJIc;Ie<&n{9IR!X1Zpf5sdSgs5`O0;?gRDst-E^H>xB#-pw8*-6ktw)t(_)j3rKa#Wi%~T*BH0&IZ<CI#Y_cdb`Y(+lVM~_!S)OI-Vrq9;Lw1m>wilWqMRU*XaZ3rSBP^tk?!|9ke)y3y52}iKb&sg!#~Z$YxtDu>yw#^DBqFI4rwLE&QAukwgt^5dYBxE8M#ZQ^oPs6*9$d~AFTjbLmxusuASG-l#_>>X1MmGdqYVfCjxf3hodXT>E<`CKN}b0QRt2&U1DDd!2BLO@50J>Nlz9}M{?N~KsL6Ff5v9>McK}RfX=kB<6tZm%t35aLnnw2}Xl($lbFu5=Ryy=MscRpqtD~WIM<?`oq+}(n9(O;Q&ZI4jS}s_pxB>LGQFb<H4Z`pB<RN*pzF|96B1@Iz;n3wS>M-zXN79Ulb;DEBcAc{zaHUmRUWi{khc5yV1lj{c<i#j12eYjfQF$xbgeX>`o0(YYPegihU}`%7v`l7SI$#|R5W~1Xa(bhPl-Su;H-e}U4PZlV;0E8=?E|ThzJ9|g!XV+`!l7WiUNhJnp`{rcL0FJdaxF$5cujWk+<BL;3Ft3QLpX55wl^GlV|-Gt9g|5iYNfN6)=l4O7niw{%KNZ7qoR*IonkS``4rc0ax(eihBV+0jd~Zy4s`*cT^wK5FMCot#QwO>eSJV#Ys(8mINvN%)LP;l$ty~L#^lWt44WAWhR_#7`^bDgO=?d>j-5!7)EisD#=3g<h}L2q56+r2<_+2-tmwMrcTtZ;G)!=av3hMH2Ilifr3qCQy;ciohpsOBDc1$|A-fy_!P2P33yaPoZ2Nn_jC*}+KJi6og9#HxyHP-Wksrl{jT&x#vQhCiU5l_W;Cr6%BGe4~NsmxxjfNdks<r7ZQb3(#)tQ7YI#)>dN2OODZks`G)N}Y#YV$6=BK(vaZ>SCx{e)i%2)6zK+WkRHDn^p_WGnjK7j)RPKWRuXX*nYMfS5WM-R*t@Dl`Cf1>g*Vh)Ok(08*L^T;DCzf)OeL$t7r@DXqY|YX^8VfS%aK#6!sNlN2{V{=luKUaGTLpdnstF$wV!<3xg#K!6nFSY$hqFZLv}GpRI{uEpuSISpr@9l8@y>Y~aRqoUE&r5GYI#t8&mYBYh1R(W=<*8)QR5<8&A98z&h=L@zDpQAi$Wxg7fInD)W3Jn9yG7eg!t~Yc#7%vMpk8KymnNTF(k|0B?nOab}X5tR66~4+B!r%kl{x0kZq%6a@b|4h>n;0S>%2{`VSb-s!v9YPlfDjH82+ZBpEow&XE8d<`{cleHy|Zz}UVBzNhSd#K;@w!UIj1x4Qnu0QcL7yMuIy5iLGq#_?p@LB|Hu(%agh!rv=TVmW?Sxk9ba-Qy5JQ3LngHa<!rL4n{7+?N6;G#q?rn50mfdlI)d+p6!R5=(iQ;)#|jrKVqeTD*bqc-xJD#91+cy&YYRrs+=U}J7POP$IOAmRmD)C<fQ%;sLnzNQO?W6-5TRZYG8!)88@V9nLb+X@&K`m^%`Hj3;sz)^)!b&nW*-O+xe>xfjI!Z~kZM4Jf!@JzLV)j@DV@_E<=mjjtH;DSCJ9ev6Q|H9+b}=SA5%9$@o!n-w)OO9S=mh#FUIM?Hd9>Y3?>r>f`(+brkF91S!gsScLsJ>O;g?V#L7eV*3}BwRQM8@k`AUjdxED_j&jPX{H}<$Np-x1KpPUR$9T{sWU+;14*pQZ!x}%KO*(?db@=B9AKXh*^a@r*0BSP>64Pg$_ftda^vo}nkC}t9e!x$xNLgypx?wvG=Z!#1mS=RLXktP|8Cx%Yg%GE9C`F$-QeDCDC8AHmO&${e2OE~hBJ>g6?F7>ANV^H(ps{X??t#)f40^^)@TtZnvrn-sfQ1(CWlWZx%g5#KbT%^;ISE3|qIA~4n@l;|gIo={4USP{8Z?1<l+la0`pLd++aE_odryx}YrkZ_hSRQn{#+Mvmz72{*Ka};NVj%zKh%$spLkg&elm$Ck3fGRtConyqAe8p<qcB{2Gj|<zV%k`^0zOZZJc@V?3uqsZxQEoI4(Q)qdigPzK<eQC>@Jnuy|wv(gw76DBkt#*MT6+@4#|sL?E&@lnV=n!~fvy#r@`}Gv6g$dcl`r7l*jzJKb&z0DK1rTbqU#u~-faH@Oa&c?;O>6NQ!K1rEY#0ql0fJ;LwgnIQ^Nd?|ieY!Ib48pMCT_3)X~_cb0kcmC|9#zUtsoV^5fuC6WY^jluDK~%6%Ln{iJHC}~l3)i@jOBc_bJ#)74&4=zk`^foo51pmknt%Vn6HJdrs`NxyvrK`W*vhz`wA?VdlzM|)s8%ltvbu;JTrcLjX&%$)FEtOR{YSmK1o;)W9-4sF4NGtFTt&wbpi_)m2RlYzoRmlblUy)>;QP%8Z#Bh%Mc=<^uc4c$bZMLW4Vu$?LRChQ4kX~5tH*TO#EITKtREI#4#V8^L%b2`jxiM}Ow17%7V!}W$QBqH^tVL1ql_Z~V~L!qu!E=u7(1bl+!uerPRwEH;{ctnJY}L48L%HQFG#fYxvI*wg?o-g_pU8$*e*c*u&t`1xh<ugQMi48cA#`#L+~l=?}nlQIGr<8j+?<Uu+TUvNJ-B~f(eR)U+A2>pXK<7EX?^>sE0{O_GTF%!_}?>?+Er7#t0W5(ra>Z!UvxnD)>xFf)D9(t-3s^xKp}_ptwg3I!(gA;=Etf?q%{dQKu6Kj({LF2Gn7pAg~cvG!TwlESW)zC`e3g?C*pZ07kj-6Q%Now!YL0FquJbL#$z!@VR=CyiLf^0(}bsk9sqQTd&9Bq3<8-c|ZokW9-mUlGhW6i!y7@oj#w|aiTK*wG>p^q9@v5Gu1|fvketDYR*|UT#ZIeyc9yFh14wexs1+ZBW(InOzuul_mTiZ2Co!IpIPB33^w5fIxWFLGhlkV@8a3T`$1UlldXbz?AJi!KDJb@T-zYUv+OLcAHXy1saaLF4gmd*YFyKgQoj<V1nj%GY-Yf@$5{e6KqjLeGjd=6%4gCpb&x#xq@HOI4?|ny0nSwfg^OJ;j<a^Y#JeKW@bM{X5m+{06W%NW*hT4fx|pEC>u&a8tJ~_imPz)ygDnrIr`PWjeWdfFPEUf^GWEU9bE8Bc8soAD`2{c20P!As9c0Mdp{Z>rQ=2IAX;n^D=j-h2#EOk=8TTqhV7%d|*F<;gPU1Gr8!+90kOPX+jdmY|r(Lv4L21pR#W#2;sT4CDW6P)IMmI~u!U-mUq)U0y>=XACJc?3zT1;}%Z9Ym)O&LK>xMF9{fo#MhRlOp(m!b1qx0=}GTD_$GDO)z|y4+F9mR+`<Y@%h$Ze^>QyHC^&6BC1}D>f#0U||yzIKS8|Y)rDNpI9m9Py?xtm5=gtuiB;<x~`1OF0>|V3(28I*NvvTcgDjeJ&F{QQgAB3^WK2F=5*DEG^kCTt$yTIX$GT_ip(@)kOChdX6%+DKkcX0Gbfd-ri~&g^(Bo$fuA#qq=a=8sT?|O6iKNsX%q_loKYktavgQL9oLk5V9&fp{s6{%c}>86Qu@aXP@P%U_2ptlCP)t~b%l-Oc2uf|@Hh)c4@Fg8Is|0X4A4Mh!oX5&>m^bymQq#52t$-eSC3&Hp3Ba1zZejV`Ka4w)~J%<Z{`c5yw35oC!?gn{xq_af&2A|%T+JpTn~D#U7{`?Ibs{9LItGnBxhrT9+1%vo4{gG3Dv3DkqM1=+##mlF4<5wXjbW3L{mjsi%P#%tm2Z7@<lA}J^<p~CDO_jWt(QKBgr0yH2)f)hmvoX;8SsWO&5b#v`4+(7`=Ww!Jahvv~<@s>s~<F4Le&jzLati$AB(@>`BEh_r*9%ma{4y#LoA<HsM27X%miYh>^wxsrK%WI`sIsu8o7|%g56@1SS_0&{Nt?-(`qa)&9#~6O*Fj7}rVG!EJXA0aIu6bw#f-U8A?i*`-$z=~hAX!|)}{P>MmR<^&`F2dI>Q9&P&SvY~|S*9@5MXwddzQO?7?Pk?KPAV}G16P><;XwW$$i?0T=$rLaJS*9jJ$x@B&u;GxguD-7*(qmE}5gg$tg{)%S*zzKyK|QjFA`>Q1rreD^;={u6cN7y-w89{C2_+d@Cee`+HF9C!)O^sq=yu8&Gy`2rh4p}GVwX{!$QUCjW8M(vj~MrcX+6<u%rdu4F)h_Phy$x#p)?0McfWbUR^~;6GRsU!P9fkEvlk}zu+Rd>oRw%Ec)`$YG=GlgI1uJrPnxH~M9D^-LnjSxSJ-^U(iqT~Ce4U%4)Y-eYiK#tO-yfMf0N>yuxgSxfrAlZD(f|;Ld|%3+hu1-zRuIBnsf~k=9Cp<z37*++UzB5A+@^lTU*u2hsv~7Oal27iw_wr)Io`;#<7d_3DPYz6#C6Ew-K(8t4r$`6A^?MWe6p7jntx=&Q%=#S_@QCcz1{~H&Ea}p*iTVDJX`>lm?unQm_{&8thnM#xr>4{e4<J02ak|$J@kP7{K%>!hmgJ1raZFg4ngtqlS_aMH{1#Zt~UYA|S9XP9Mp60KOO#CvR3={86-~ttDQZtQFZgWZzeH=+{vB>2yYD!s5WL9g*&g?v0xMmbVk2!pa*%-`Re+JsNb;6WDLt9M;zs2sMIVG<{M5((Yi%lxv~I1Qqkx<s^lXvNAZkbClMiczpse+V&7dQ-GV~;vprwc}m}D;oyK<z6d~wrr^<bsslzhm-Hqjy>k%yd}P-t|6RoG%Z!&513Zuej>B2b=i4VtAgO3)IF7owq@w5$6ucBeD0T2!GWAOV#P>-jAnl|7lkNHk+B?syNSU(d;o7TDTMGx>8w@TF3k|?xm}QEzGEZUntG1>CxI+uHISRUM);~uL6kLp*RWT!*QSv`{Vcv(`)9wePAg=m<c|=6o1+zwCmPHxi#Uv0WGVai$h<>p1==yHFN7cj$yiZPmZh4Vu<V~`oc6qlb{jp`WnzQlu4r<po)Iw(WVGrIoO~su1Wa%$sts^rNM&4Vd?36@Y6Wi5a5LZ~`OC12ZUUc1Kn;31v8;<Xy*NY@-q^>;!)L{(C2Ev%_e6m#0Eu@n?;Yt+=565y82vdas72w>Rs+4B6sq?ElC#^Q%P(h(P8V&W<D}9^MsnGjmXcYOR(QwglPSZnUsZ}(d65(|h)-$1_<nacJ-oX(?))7Veh~SK>4N4sh>h1LtPJZm-Oh6c5NEkIBxz;jo3>M4<nR^*A^&;Rll*78B4)u?wmRQqT4}A>Y)fiw@iV+9E7JCr+`{NZ1?j!JTRwu~W2yha+q%DhwE=peXrbq#~9sbKUny%7FNl_U_{m~Ga_95~REH_X3$GzqWuf5Vab;4h+HkYd>t0y|A{7Um=r(M1K?o$=`(o!q>Xx;C{eM=?+c<v42h97PQq3?5_$9J7SwW0{xUf2&|lS5rDRZ`FejF}TwsOW6?dmfM-qkt}Qvo_;`?P|c=((R^*z(&$&4ri-fI)LxD)vuM!!dc4J8a1|{se`c?T;+JsNe@dBx<fwa5hUpx4$s^EAa<Px6l*K4g`W#ouN8Q!#14TFF?RZ`)@V3P%&)M7#hbi|W(Qa=irIgUBGnbH7U&3E0I2K{Iv8M>`Uq8jK*`;4X}XGMDq3t)QJ-GLz0wbCM`~KQmKBOVhb3XAHe_QvN``?h?8T`qm2%I+w#|;ap4ST_iG?UV^xbIGEMhQ|wdg4G00;lE7w3-S)v=z#LSZo~yONq(jY+L7q~^uY5Ct^Xm0aQyI`L0PL}RqPn|5a9JB26>V?e%YUDbbZrVK-t|3b-^^gm-^e3Vfk0;y-d3pAE2P8Oz~hm<^(RLO$@DpA3cX;LXsSw|}3`;KmRy52Gwl&o7@x8Gt#U)B{?awn~MtRa1vU-GWAy7KZg*(v#E2RAYPkU%Px^hh}t5}eN;NOJkq3Rp(C(AfFk`OwayCmf|Y*1(2&&$%J|wUMKaA`u)z^+KTNrofWL5yR{5dgF+iBmSwxJ29w2>x;sUYr;*`L>Ok=GhtkN&>!PU?2EwvxBx(q*x<fw2Ec<iRtpan%1*(@KjXsshMM~hKK}a;o_H|%ruFGE4j4ma^N_e3&Ia})nX_xeHx}4~l?P2|ObTqc&V&AtOccE^weQ6P%Z6&rlNv&EoEwwApva2qkGf&F3Ypjv@EkjWR#$548`Ic{!jQ9q2vg!5@`sH-*`}5O%0AZF97h$LE3R{v(J#2;QAb?E`+GG<8|%Dn>T6GM#Y`vCE>zKFn09reCVM#)?kSWD@Dl#si_}D91tgWjuvT>-a*}K783tQ->_y3rNaudHf9R1{+d{qDhhw6`kSt%->xL{#Qg$_AIZt9fYFg17Ubz_M+wf@)fyw-_IRLlL?cVwE2kqo$-!gvXtBjdOu~epvKjb24HcPjIj06oUvydS<g^rDL?aO^SxfQt&c6w9y6>S#Df*}$%H0e}EQ|B_()=9~PhJaF^(4q@9nb>oTfv`LcoMO~$Bx`haT?-E`CI7mk*M(KT(+HBg#jF($UE&H44#z50+(lY@5R@p87YOC%b^p+(6F7UEN+Q)ay`G+4SyckfFWxApePdWTaeWoM9IdmMmMy3k(uqYl(Pk&0B=tgBcv*~RFLLLa)hR)XvvplODTV0*pWQ2i4i1Ng%<4(Om}iIEW;T%r$DGt}DP5q(2eB^cZ9Bkt=l$Wh#(@M;%R-)ryc;UIdOU9kO&J|R$d%MQg$U~FNAAFjx7;8C_(i=T^PtTRkgiVwEI<_cR_b_#Fr4C5z5cb58f(N!-fKB5QiSUhs2QNjURsW@dXr_RXy65?du?G677Ps7{6zaGG8tS084rL}-H&rUFL`Pw>7Lh&@ZXFs%=vHiOc{NQ92ha341AXXQ!_mS+0i&_plQH1!bs@TK2J-jEvOC|&x^Z>DQI9@-F;-8Xmdh4w`PQ#tjQ>`TNL_6vIw;W*X{)F?l6e`q7AO?Xm56y-P45!0fk0z<GpyHwt&&Sy69lz*XtFxvV@N0cy-i@>-DO;LXSb8d`95KidzwHwt!iU{juK!mXBZD6K;h+#POhBUv?|Y?lQfIf=yVQ2%Vv!ry`!c7E$Dcs!Q(O{C*tt>-81chiDwO7UQ7j!$|7&<L(l{kK-Y>cAQ}~gE#{EX^vwCv_hR|0uq<0J*abnpIiN*ue`bs5Zw)E#0aMan|#oL=jsxGK8YP~ZF$4Uj~7$xRUFg`ed-3S?Rx#BTgBFJM*5@SV2sd~-735px4KvynxYaS_fdp{qoE=%9=5T-33`Nc8}%L^{2=TPXfP+SuD^$&u7JI54*R14Hvy$OUKD$SfZC|gpgWx&c43L<0q3jR$6nGz;G3=&Z7l-+LYyoZB)o#2$NhenI(GaN!5#L$<A<n$!KO~qs|fcSLd}XxaRxSnxZT7$IKZ{=8$rz8h}4+7avVwwy*5tf3B<sXe8zYaKzU486@1(8?@*sBh_<atg?<xGwmORv$P$-^H+z1}WQ3)G%;cmlxu^KW9+s{W;<h-JLI?BaZH=1vVfh4B4BDYbJzWw3%g;TGJzBmSr}l4x@QPREfxzRHo=;4R8v&?S`ps_8#K+_KeLL6<qJDP=XB+zkNbP#x882dzDW(8GM$a2=xBI){Vh}Il6u|`I2rt0v(cqHT@v->|my8EpB+Mq3B2wwKN5gJ{qF#@+=kx;B4ZIojME{5`D8@f8-V*?zBHjqAd0J~w6S^7tF~(Bc!-j}RLmW9R2wW2@@M6?snuck|N7NR3i>*-;2nqhF0G28^kwi+#*EaqEf>ldYs(0%p!kuB<np$Ew3UTRADHDd`<0)LiFf8R`rxVMX6&xK-Rq_$jZ^^Gf6VU(4iVB(MGp!FajO3MmA0T5$dxoS^QJU{{X_n#3@w<<^$1BN)u+{2{YE`#N6E45(TnS*-!|lBBhCA_|?OWU*Ff9UY;5OCTSsaL^NUX-a8$S%ML}GDPaHa=dYum%ML40%xUjU0D9!{KM5TH2#VTTs)pzFmb<$<h0oMIU#93#NSV~W8?kZIAPBC>_?NRmY>T^x`cm$~%_15(uMck?P3j6IO5ptb5Uudu<mjSl@SS`-x|arvgf%N=@2TozvQvW7uWJs(+iOygL>E?yjO(nd#+PH~lRr`IJMipFs+2@|*qmhlZ@ZxQX&xtmoN_#$QYTuQ$3l7hCdb!eZzd`IjaRc!g&r!SnxF)!ic5Dve4;wx~O6@0^K>}WL9@GxlMDqe=>@H>#O*gs$((K1WgVo#w-ihyq<9FNe1Z)Bh^LO;+~)C#tN>u^$LMpZR=+x5L+h~#s(c*)DXLYyt~dxZ2wVLwPnB&2DFLT>%kDSka1FGlTc8WhcbwB>iv$#=`^F`2^9q5zeFk|L`#=TWdnEfdF!dOdHz8$Na`3_2a$1u9IOm`k+_q{k%2J>T_!5z}N;sRelMdQIL75q)F9$QX06KkT*o-TsidJhT{Zp(!QilWN6Xd`rX(0zn72Q897HX~Rf#kA&kYG0M2#Ls6p2d=#HVZ&(7NAd$T1R>YGxqKyq-Egi*Wy&^vFB`Sd?#Ke!bd${CI5Gz;DTljW`wrn_|#d7b(M-Zxp=|0L)cou^YH}O~ir^(?jjU)yKfvoVS?@&mzgq}y?cY*=$fdCRVB-o*rMgQS>CxDenfS2jTE{bFVKLBmFwHvf?kHYz>07iEGW(2YcNkkPP*kK@pSP%ge+3QAoB=8P-X#y?p?QMYslT6Rp8_*J2S>~r9Ntji70!f!+ky5CosR52~mq{Nu#<F;#4G;zVa3JF9WD{%BYm^WGBWW%VyfdLMcv6wC+rB@*XBj&wU9K**n}+D|+6^U+5>ADSfAok%z@-~OZ?ihwY>I+1_LS?5;=ar$t^)OZv*e?@epr=d{wF?|;f<*Egz)q<DVODI6KoScb{Tjh^kAEk)gJPK9sdmO5K~gWa9AiK0SviZ&y5k=HKp0fAnc_0d0>P~;0$aar}=o&Cns<E2_GK8XraRdDe0Lh2`+`e8i@D?XH#nc6Du>?VYIF70UY4t1}qFXvPYD$u1q;Y8-8yPj~gP_v=x!E4RQXz58rWm^D;k_dFtdldU&8eJmZZbuY3MNSxW^jOJ&l7QwH~Moxjky{NTBV{<d-H+m|k%z0kP$$io*dUWSuvp-{lUokFj{0md=F!!`yl9>pT=u?Kvef<CaoDcwPO0cLy&+iamA3>N4M8QQ0&Y{0xROw(F;*$$3wq`Lh`c(VZbd#DrlI`<qCKVlDq<zb-c<XaXHS|_JfxLJA$!^G3{H_kqE8+ZVTDV&W44vG{jnHA)~v-J-*?hG7#*c^Rc8MO?(bq+U45D#|zb0Njuf%9=KH+KqAYuVZHggQdWg2HbT_F+6WHqi?R*n?Uk^miMnLxpK(rn!omp4(okXGwk=WmG3&#TPUKC+U!Y6{oCDFROMQ%c{@Q;NsX#*!a$<qhSNeYtKg64)L(Yka!g;hqFAbXxFh@_3m&?2;y<Qa$>1Ge=YPrXs~outSq7X*Kzn$J$}4&P#~1!yk3YX>vv(!+NPx*>Y7r7Wchhfj<Q-k$S6^9LcXo_mFk}xIe23$fk18i@6UtVr156I+cu+@%=#bJ!8-+x8c;cClTZ_gzQJ<j(jjXbNShysQZ!|P!~$W#Hb^5YEEP*NCoQXdq~M5Jdxh;Cywr{gHFTRWMpQt9o}5nuFNz!6{#b-jhC-=ORZMxLOas~N_8Q_RW|I^N2fN-F&JZfyT+3*XmF_2&MjY@S))eaIq@J2AgUF>@md~1QD!@uLJw%N*96*#(bTu<t<t{=X(5xZxsv036HqCO3;&pAhM4TyhCMnAn$_W4kz0#AJjanmvvEnUTV|xc{*CiB734~iG;AO{%|G{AEtJvA~a!vq*y1u8GK(x)Gp50H%)<8Ac8BvBUVuUqF%hM3EPP1)PL!&Jdqa}S68EZtDI<)BI-#I14<e@<%{>mwpAs#4iyfat!9io4cCo4@Yr>y`IgG4MJ5EwXWkG3k%PY)!1D2C2KQ06~o!OU1O+EpO*)wDUdOP~e6>7Nd_)=*gDc=Sd4b^7!rH^;I|b4t}$>ECq=jPx{I&&qW+hzD4Q7B||1uSe)mQJ09z&OFOAZ)Srye`euqe@CV^h`*9ktDjB20#h5z$7yX-`Jb8Bq9pb;zx^-BZ&`=%HP8K;=YGv|zeJvkg;}UIz3I*D9Hd()+fkstZgn;rd^&G=T_24}%KFEkw<*3=+~Z}HTPIb4k_BXr;p$`|%qoJKPC!+?xa6*sor<d|Q?SI)M+=}vF9`9n*Zz@g=hqfCyZxru)q>8Zzwq>pmrYUZ5289MeiSQb@g9&DbmBVQVG-%p{t^z#`I`qiu7>fQJ{t=<gA=c0D@=6jw{!k=Y;$%O`-H$u+lVhSAr`uvXJQ&qFRymqBu%R`bHW<3+N_zwg0p82OHH3aDsh-uG(2KpL1!m}R=IU9^!0zhsW2<gF%xE$`6dDf^%pe{BeEd!+oE%N=jEkVX1B2Lu)4Xc8lBHsY4yAhmiiXrsOP*lYU})3d&=u4?^gO_PM^}E$&;<5+?~`1OF3na(jW87(w1dBn`%aVSJEFd;d?VOXzpYZ?F{-+z=eS|=Q60(0{*g8DXh<c|LDjcjvM42<j>oHRwuuOlw@ex%>r%|pV{-I`~iEPsb_7834hKhfWilx0R{RT6JUa$jemqc7y1*>^I%`;^_<8zdp;fay4M6<x(lAkPC0^C8hjP+epHz{>!%a24U(CJQqw*&EFF*{RX$jD{w~SWNng`%TV-b#zCGw1JEn4Yu_d8JGp8Iu5HpIX2uOAf{^1UNYPVP^Q;TH_AhMUCJ?h|JTU%ZjYN4Q$4)AHm55j2FE3PesTf2itTZ{LGt*ycCqv$})B};SlJ-1~TmUCoAw|8fh&?pGyg0}E$4&_B#s&{bHggC`g$)%D;WwKD3uQnCUM?l;!a*=upxa?4y^`xJr9O_RS70zi1zOxlLr7fUo$vy5I#lI!D)dk|h``k(i7K=OZhrLlulTpPBUMO5Wemo0`Y}Y8+BPGu+GH8q*QDll3!B%?`zz;^nl1o;A#gfg(Mq7W=&(6w)sCFJsBj?vJr~vftI2s1D`<y1pPr^X6_A>T+lniGygfE?L-@_VsXszX4J4>s>qIv`SRXSB3^jwJsz|xPmsH?fk7Mq#ULFHDOsoN1-ty!P5c@cPi>&Y)u_)2HGuKX^W$(P&7P%Q`C)q)x^GPCzh71~OgSE4c<nH<07zEvwPt=nqz4-rSFaL;2wuL1pk?A`*cUiCH2F@3NED`qgNx!O0w;K6f`q^4>`oBbLq{FQvQn4h50Zw+b2Wh$NiFxU(*a|SQZR)08Tv+0I3UC4_bvHyR6J9*`q8z26B|NY-ReCCn;A3VMP#=Bp9{KEcQAML;R{*BMx-+%9C{Br;G-|qk5*OQmtoc!{6eq+-;SmI7;ovS%2gfBjR=H|Qa-uUowX#I<ipEdm@wNca@wKt;XA73zQ&Wc;e1D?F{(#;=#7sitriGd&3Q#`WS?cuH_LK#uqvEs0B>$BG;AHOwu`R6zP?ZwIKpHBYq<;l~3y!G_I{^{}OimU(dr^laPKU#vnfX>M?*Kb~bdGgV3Z(RS`jSv4wU!SC}PZqNCrRmi59vGvMA2EN&+QO9nT-Cw2<d;{hl&)F(XD?mF(obqpUb8;pN{@6*YJ)25Kl4xfzj=HAjb|q>|I^Kve*O6ykKg*m6F9-&{|F{#@&-)IM+G!MHL1WFH8E-Qs7`Gv{@8n1`)uo6v%iND3<k=<*)H6(w)Pmlqs0xs?=7yawU3s{_J3CY;U1V(SqJz8)=+|wCC$TUU&j2ie{A%JZ2z}vy-TC?ZA6op!2$#W5aH4Bpr{PMU<{_4@E(X9WBC6b$Q@(&KRP0zk!m{Sk?GXi>O-$^6Cv3qjEj4f5?X}E#j*UcSCwz7rBV(jpaHHX!0-kD8=qF1(II0EBvq}~7aTeROSA)#L5dzLWqw`FVxmE~Xb?8&ca?&9a<xq1Diy9$VM1P`DhpQ)grVEfu+qRJG(~I%NEBNrImeuwqJ-Nq7adP4T1}u<8ESP(Q3<umP;Et}4|Xx_^kUi+2&N#JVpRRKqv_`{f2P0WK5Czp2ong=d=`0D(dpBi4dL^i7O*&~i@uscEA%H^3fw_+sj)p;48%ponecT<|A@<r<`b7!+7p)&t=l3R>WfKIgXQj#`XgtK=FKt{N=Y$sG;KL0m2vW-!Ua{7EtL~UC0p8v6bVPMOI4WEJu^71ks4%Y6X!5NoeJX^MMP5O7H2bkWG1}4WZ^1NgCgkFIkSBR<rzlQ9rZYFi5#*vWeUi3Tg?V~&qZ7PZW}=812PCWJARVE+SceQyIU9lg9})QB&li1M<uU*l##95a-LHC$d7ID7Az4MFv$_Jy>123;={R+4h%)nOerUR4d&m&3fNvP-{u$(MnO38*Ft+4#OxkLMIQBv8kw$R079BFo~|zOIV2wXZ3oBrLlj9mnNk>mU!kE_twUc8j#g^x7>E{qV&S1QWuSnMD|6sO$4Op^{4a#B>RkA$D)_3FruN)xMny3bi=v5=b5CX^%otaSe3!g0+p}7h^orTCidRbWK+g?Xl#e!L%}D7W?P*S<Nw0#r7E!t+mBvhxXZC}i!~|iXS{^VLcfumDwtCm=HQS!ksyW7dSJv%3FI%<WtvQRcDMlff(o_bI4G{j2hgulXyhGLCC`1Frr?xL+$*I*fON!)7VD9Udqs7g)e!KtTo0Io`c=O#~eE!eh-T&S5pMU?gTOWK-kU8m%69z{CzH$AX8y`LoN0I#MGY==q*NFc$&W|%2agY4<+FDqIe@ex_uE9_E@91Oyy_7vOmJ#s%4_`ig{(Lr*YPKUa`%W?YK#>KN93}JEr+LhvH8lIEac7Pk7Lw7=9kMs7#HyITz*pN*Etk%2eDdPWPu}O_EUHWX`KSADKXo8d%YiVH+OVs0k(srDJ49(pCk{<w3XB8O)}NiO5(2TEX4SJ#*C*p&!BGwQEmtEldFHLpU;8<#s^+F5D5=(+gK|XYY{X+Zc;&XViprt2X#Ge^NOeg_@Mm>t@zfU||I^|ZAHTN##mCPwZP~U*gW7CoKS6&}EhaOxsSndaP@+I`1)&yCC^3yKIG^x6o8FaX=Oy#TpOwZn;w7<Zy80!#Z=b$TGaUgD0Vy$0LgIho<(t2}ZX;SWaWC9_^Z)I?_cPYs-n{<m{!ibS{NhP~G6$6>y_%QKfe%jU;$wMTDXo6-@sDfpe_ROBkvy*YB~m^h*Bq>cu96~x-pwi&ia#ucYfa_RIC4^?bNqKK{@oMe!U0)3OX;Wy5M<A=1)*~T%0sdQxT#Vp?=;)=qd2(;&=cEeN0}f@6pyHw#|zJ5AqYE#(m@V3h+FhOrjeBSC*lAwt*b^|z<AHDl~RLV%KCNQo;ZS~gpVOF6SU1Z2r6SUw5&>Z6JRsWeaM(4yn8g%)n>ao8<>_qXIIo_GU@d}W(U<;DG<b^)|q>p%F@!55!iU-Y6>usO!aPMjzDlcvWchqH9<Pb8PZ{`x_NHlm1bz`6DaF5_F1ht+c;}^iZ5QVR@CIRZo$QB0?dA|kQdn4fReVDfuf^VC3vwum7eD;&Y+{}Dz%p9=v&e++vz2zlFVP8NUWtPRHAM9wXfV1yMz9qm;fw0Mh1iVXPE#gT+0BYc#Rk;5#=Jo0I8xiy@;$Q_mCFip*ZAf15+>R>~{-d(b_4+P~W88I!gD(_oO$RkKKE9uX3$?b&P-Zs`v~4AoOeZ$V<(~?zLa+3cXE7xnP}TWfUYf4~;+?J($iqir7e|jHvjP$&x9`)+z<qW&<huyRO=dP}m@m?MF4M0mg^S8k0#i*gC^jsL>C*W2vz(>-I`hK5>Ml>^HQnp$-ZAmaq={gzBG@+Sr&?MTm;jz-Gx^F6VkCEzk5!vYKqKN{1p1gBeGk@)QrDJL!m|Sa?PleWr??gTJb~)6nm9h1}fgkHR>K(~@R%%A*p^o0e%n-a@Ue=h>!_IZ-B#O35-4bKA4Y>6uv}(_6Y!GgAk0U@}^A7~8GoMX|7Vl&HpNwP-5phZqV~#KjQ?kX*4zVr%7uW#c0HwI@iht*RU;C>jIB^h6{Q7(!kQw`}bk)@}ou6A&$1*)qk};4mtDJwA+OugT2nyl{=0Q#BiO!$*7(Lxkd1qH;;=ECmtfRU(N!tw|Xf^Ue?lC+R~VSMAGepa;ON3OI{e8-e8`N#P%P$Vvof_K{KoJSvcKMy1ST-4cHdO}8nxlYCLkfj$*H_bV-Z*jZzX{hVpz@CB%{v@wO;Gq%d{wNAP?@YPyvv9f-<G<qPGGW3GTcfN%-vu9-{jc<n0XfPm&jeAHb<ZI4V{70RJYnj{`rW;%0MQ8I>wjsr)7s_!~p5}RMVk@tn;f-u8pYoK(tbS!vTCq8o<rTMjm{|%>cLuY0>Cf#ycTg|3gL|@>)^n+RA}_iDQT9S87OWQ9L$4E)e?1g5$^P~@4K!TJM02=(c0n?4W`#;u1a+wpW7q22tkVsHbX|Zhn?~=NU|up)QCaXWJ5n+n0={z6;$VpS&BBcj-~0TBzukZDoi9HAaltv7AXb>MV|vyQIM(W7C2I;VXi9`8^am9pED2#+X)?#iDJ?n%?H2OHTY%CV;D8B)k$fyu2+=&Fpd3M!$`uN>qwJR{LVvmI&`ZLFCQ(AP5UVaFf`ErkLn*3~vIUyoRvo@|c&(w&tGox0gQzdJF;xsItKFK$vXZHmOHAuP>9(2JbZ{g~(^ZBS!U)Whr5=4LIdiaU4{Q@?3km<Y_{VlPaYtArg@Lm5<M3AN?i7plRB;ivwn!us*F0XgTTE_vtc*}Upi2&3D9pG<5-lC##y>}ZQ<_J8SK;95u7VZc@{q|sBey>I!{mdHCZD`IdG^K6-~L@<Zn6Kf$M=8tJl_!B`t0>vZ@+N!H=o}4?0vi~4<~>4&CTntP2T_UjnCfRf8qmR%WGl9b<~=Z*FS~VlQ({ajx(=+dgH_ACNIA=dG0NAlc2`-fA#j{r_Zz55LdkQ>QDCHeu^5XN?_Q=CofKZ^2+{OPfz~O3!*r{AfqE_VgG}dZ+!T#lgD5C{PAb`#?#Rd1&9Z{1KI)WNp00|Nx>I2y%yg11bcbo`t`|kZ`05goyk)#?!Wu?jqBgPdHt!ig`02S@YWW-`1l1t5c=iNh>YR?>G3D#8^a4Q0a<iUzW1xofAY@>(qW!HfBMCnPkuCc|AqZ$Ug0a=$sgXi@yQ#E{x3d$_SP?-oP6-6(G4g$dHK0)Psn}dsr{e6vj3wGCa?VB<{NMBzxOjP2A^&{|IX(>`KgL0D*u|Wzj^LChR^n1Y*fMK6g+*0=?stxIsd?JKY8y*@VNh@m-c`3skHuiijh3|==Z|j<a@uj!eQGWn3T^Se{%BkUrv7gGItM~_~`ey-hOBDz3=f9-1^|1$<M!QkOTF9Ij{dPeUoSY<>vQ)_3)WTZvEpYljmNaeEP%5GrtD%#vAt^JUw~pdz1G+Lc;j&?<cRkG<o?09P;;`5iNf98W8KvXP@4G|99-;bmNaNO<w)i&!2uFslx7JVnHfKb+&%490Jc+=FN9B;{YOn$<t5WeEWUG(l4F_BD7i$z*i;!wjxl=h_$m8&be--$679fI$Bp*;FJT~fpt4`>UHk8wroB%*;nP{V825~3z(bNU%vI((@aN5P;>aip~E0<l5!yV>1T6=@TtT=m*XJNZQBV7h4oJ1=DY7+4USf>At$FrCdBu=@>gYVujLfXm(OX#wzhXbDA}XOT-6F}v{SIlHPNV>I%FH3xeA&!`6w3({{WG7(Dhn=vEcma@#ir;IsP*j&eSPJP`P!~BP<s!rC!E|Sm1JWvz9onRTS_y)8yFGv|jUQzUI(=-HrzA&|0uVX~GVn4Lh(#>`+><gKNePqa8cAhRmWRo30m76lanWO9$d~AhGQK@b|FoUR6$di|G;YT1Jy$*^2U(=OK^o$r8?icc%~U^C5iRBm9?ej6Z}H?)HVDLulo)k^z=X&Xol`Qoj7;t;hcn1kox;^B@Jl(eBo>zrl1UAeMsUjxs)+I{)~63TN;%NQH^ujPGCnw1V|ml6>KhlQ-XF(ecJ7ukzstsCx4A7eT7vA3)n6c7XVL<FlV|vCn_@(*9c?-}vw)0Snu_{sTBWP2Ty0OF+^67oNEB;p4ZS{w+59$_oJH#)mJWK>x|VO<sL5TLn8o-2{nt|E)hvK7JYHnwfemB`=Ox3dnxU<&e8EmbIHXS41c>jcB|U`pPN`Ms4+WbzO&`%EM9QA*pf3Ld1S%7o&!?!`%u{AoQ7PFC`ATH5%jrE)5Td)dR>wvkyH7IPj<dXI)?``waS&On&_o$8sUvkz@t0e+p-3%h?unO~^RZV(EFA7>H1+Xv_hZJ|19+hu}#Dc*yl<ZhiVHoXw3gf(;Pl!lAKilQs`^v--7+`L&EWmyDTnnbZ+n-xDV<<Dyjy{zO&yo0^pn(B5q{W<@BkC8<SMRTDPhh@ge#reH})=_nrBN*y_}?F~00#ZnFvn6i!B>2SPW%QZDC@MvoSByF+cNwY#j8ybpT7ew@tQ%F)&9)4ju=<rqG#jfu~ev)m|iyeFsI|zXod(H8Kc*{oz2GKx&NN`>Am~6wo&6w1<EmS*>_A8dYrghU31zH4YK;&5GhGA?Hc{3oy28-cr3bHIGxHO9#*Fq#iO8H7L<%;wdop-ia&DBH0G}zoX2c!Dh!pWs?1oy2iq*r0$sI<0ldFY`vxQ9UjuckoQ1cuAr<(xl%!QsSi4u{4ILJ=4{P5A~xR5HU-$ioI^-lp`z#eU=>Socal2(`YhEht&zusk$eGGV)qZZ|?sdYJ|qP(H_j-`GW;7Qff@+i0*9>ESw*vXqVvkv8i3bPYtkzR<sXKSh`up2zR0YW6PYTfya=!DIG_WboO%w+FyM3dz8M@&UsR>Z4LVX7cTc{Ghp?7oCbS#M5VqmeP@>iAC^dGDwlyx!COk|D)d5r;N(;T_y8$Iw-+2uvN~XieUYAL7_OxHJpR?q&G@-Vuca!1kngO2YbJnI8uBWg_)F+DSgz_pDN=`aam7)mi17l@_X{Ltk=h7i4(!0R&jAgz$@!_YxdRsF!Wn-f7k%oyk3CujT(b7Stc}MFWgcSXl5d})~klix+p>4D%F$H!djaJ1&NerGVr=kT$v0qs>r5~)|_LyUbs0zc}(WSHS<CjQ?!;cb|mLTZ9xr2FEkVi@R@|t#dKD(@NorC1uel6E{ghK%4x%8C$9x2*V%2j2`u<U8XbN}`4;BV-0_Drn4Ky{+=QJW^u3`n!ZNv~s7ET%4MGK41dm#H;UN}|zOs^}>&``eS*huY_4Mq_$N*T3n}y$2mU+pjYvfdBr{`>b^kPy$=^4pJPLk-|?4Ow`Sx*Zu+zgIas%58gqU<QhvE@QGDp!|QPUciLk}ew+Po2c9y5!)nQZhEz!Ub;+W~$d8jvZL-&NnW86ZcqsbH0|&O`6P|PQ#WY??r>0jOEU4)vpFMQv@(=WZ2XRjiE%;Wj2*5mP*%DCy{=GOw7b?*`Nrhn7UzYfe0QAsq*&EbsMwk6z=D6&@dK2MLIAA3z66JlOa>P6wD^=RY6~$uCjszb*W6UfM*dQ7x+W%i{zjEh`(zefoQ70v-AqxUKQSJ?-GWs*A7&;{2_@yj6!A}{?X63Qd6DLhEbdb(M~Z2=i|6&!4*YvXFIprl4i$@B(P0RYuQ>^cad#&D3r}v124#rtJ`>1QM)>qSDy(W?KPJ*svQOu|6#-D@^W7xv8lD?x3&i$CB+RmQ=!u_?c7t=bXsGYl*RO~3tFmcTGG#Y`cwUGv3)FSXoD(S{r+~O3QZ+#h(?t6S;5^C=~5)6=kjsCr~*@Bq$ehsj*eEb@^`#$(Y6n=R7nTe$d6&oIaqW=DqgB9w`cw<qtfzDuk2`sFs$T@vt$P}GRkSYp**$)yu%B#LDeZGh#G?cho<_maE?jUg(cx5T)t+y-chW1oQ;9YVdpdFT<QTuZmS+Y4u=7yUgqFw#;})_OV(v7Dk)&)up>P7^0k+y?=SNgrsqT_&&dinT?WV*q6JY+hS$&6>Ij8*(SXNeIe+@0Gym)1OHz12)tlEvv<t%lybk20wmOGFb$Jex<?<Y+)?%9|Sx2hTjQTAl2Gm$cKH=}7BUv#)tSYQSncj9g7%E@0O}h9Itw54O*N$D4vyoVy9}{-x*No!=996@)D<{SiYYXm`!6r);ZoC)6_iJcNv0ErPUgUJt&A{Cq2C-k1b5~lpcJ2QIpx9xn'
_vendor_raw = _vendor_zlib.decompress(_vendor_base64.b85decode(_VENDOR_SOURCES_B85))
if _vendor_hashlib.sha256(_vendor_raw).hexdigest() != _VENDOR_SOURCES_SHA256:
    raise ValueError('Embedded upstream source checksum mismatch')
VENDOR_SOURCES = _vendor_json.loads(_vendor_raw)
del _VENDOR_SOURCES_B85, _vendor_raw
WORKER_SOURCE = '"""Generated verbatim by the notebook. No project imports and no network."""\nfrom __future__ import annotations\nimport os, sys, json, time, traceback, pathlib, socket, hashlib\nfor _key in (\'HF_HUB_OFFLINE\', \'TRANSFORMERS_OFFLINE\', \'HF_DATASETS_OFFLINE\', \'DIFFUSERS_OFFLINE\'):\n    os.environ[_key] = \'1\'\nos.environ[\'HF_HUB_DISABLE_TELEMETRY\'] = \'1\'\nos.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\n\ndef deny_network(event, args):\n    if event == \'socket.connect\' and isinstance(args[1], str):\n        return\n    if event in (\'socket.connect\', \'socket.getaddrinfo\'):\n        raise RuntimeError(\'OFFLINE: network access prohibited in image worker\')\nsys.addaudithook(deny_network)\n\ndef atomic_json(path, obj):\n    path = pathlib.Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False), encoding=\'utf8\')\n    tmp.replace(path)\n\ndef starts(length, tile, overlap):\n    if not 0 <= overlap < tile:\n        raise ValueError(\'Require 0 <= overlap < tile\')\n    if length <= tile:\n        return [0]\n    values = list(range(0, length - tile + 1, tile - overlap))\n    if values[-1] != length - tile:\n        values.append(length - tile)\n    return values\n\ndef tiled_tadisr(image, predict, tile, overlap):\n    """CPU accumulation, independent axes; no uncovered pixels on thin crops."""\n    import torch\n    from torch.nn import functional as F\n    if tile % 32 or overlap % 32:\n        raise ValueError(\'TADiSR tile/overlap must be multiples of 32\')\n    h, w = image.shape[-2:]\n    image = F.pad(image, (0, max(0, tile - w), 0, max(0, tile - h)), mode=\'replicate\')\n    ph, pw = image.shape[-2:]\n    sums = torch.zeros(1, 3, ph, pw)\n    masks = torch.zeros(1, 1, ph, pw)\n    weights = torch.zeros(1, 1, ph, pw)\n    blend = torch.hann_window(tile, periodic=False).clamp_min(0.001)\n    blend = (blend[:, None] * blend[None, :])[None, None]\n    ys, xs = (starts(ph, tile, overlap), starts(pw, tile, overlap))\n    seam_diffs = []\n    for top in ys:\n        for left in xs:\n            sr, mask = predict(image[:, :, top:top + tile, left:left + tile])\n            sr, mask = (sr.detach().float().cpu(), mask.detach().float().cpu())\n            if tuple(sr.shape) != (1, 3, tile, tile) or tuple(mask.shape) != (1, 1, tile, tile):\n                raise ValueError(\'TADiSR output violates tile contract\')\n            old_weights = weights[:, :, top:top + tile, left:left + tile]\n            old = sums[:, :, top:top + tile, left:left + tile]\n            occupied = (old_weights > 0).expand_as(old)\n            if occupied.any():\n                previous = old / old_weights.clamp_min(1e-12)\n                seam_diffs.append(float((previous - sr).abs()[occupied].mean()))\n            old += sr * blend\n            masks[:, :, top:top + tile, left:left + tile] += mask * blend\n            old_weights += blend\n    if not (weights > 0).all():\n        raise ValueError(\'Tiled output has uncovered pixels\')\n    return ((sums / weights)[:, :, :h, :w], (masks / weights)[:, :, :h, :w], {\'tiles\': len(ys) * len(xs), \'overlap_patch_mae\': max(seam_diffs, default=0.0), \'covered_all_pixels\': True, \'visual_seam_review\': \'required\'})\n\ndef load_backend(model, assets, options):\n    import torch\n    device = options.get(\'device\', \'cuda\')\n    if model == \'tadisr_x4\':\n        from tadisr.inference import CogView4Config, load_cogview4\n        if options.get(\'placement\') == \'sharded\':\n            return load_tadisr_multi_gpu(assets, options)\n        return load_cogview4(CogView4Config(pathlib.Path(assets[\'base_model\']), pathlib.Path(assets[\'checkpoint\']), pathlib.Path(assets[\'prompt_embeddings\']), device))\n    if model == \'difftsr\':\n        from omegaconf import OmegaConf\n        from model.IDM.utils.util import instantiate_from_config\n        cfg = OmegaConf.load(assets[\'config\'])\n        cfg.model.params.Text_Prediction_config.params = {\'ckpt_path\': assets[\'transocr\'], \'alphabet_path\': str(pathlib.Path(assets[\'vendor\']) / \'model/TDM/utils/benchmark.txt\')}\n        net = instantiate_from_config(cfg.model)\n        net.load_model(assets[\'checkpoint\'])\n        net.IDM_ddim_schedule_init(ddim_num_steps=options[\'steps\'], ddim_eta=0.2, verbose=False)\n        return net\n    if model == \'unihir_restoration\':\n        from utils import load_model, UniHIRInferencer\n        m, t, v, vt, vg, it, nt = load_unihir_multi_gpu(assets, options) if options.get(\'placement\') == \'sharded\' else load_model(assets[\'base_model\'], max_mem_per_gpu=options.get(\'max_memory\', \'80GiB\'))\n        sizes = {}\n\n        def capture(name, transform):\n\n            class ObservedTransform:\n\n                def __getattr__(self, key):\n                    return getattr(transform, key)\n\n                def __call__(self, image, *a, **kw):\n                    result = transform(image, *a, **kw)\n                    sizes[name] = {\'input_size\': list(image.size), \'output_shape\': list(result.shape) if hasattr(result, \'shape\') else None}\n                    return result\n            return ObservedTransform()\n        net = UniHIRInferencer(model=m, tokenizer=t, vae_model=v, vae_transform=capture(\'vae\', vt), vae_transform_gen=capture(\'vae_gen\', vg), vit_transform=capture(\'vit\', it), new_token_ids=nt)\n        net._observed_processor_sizes = sizes\n        net._vtm_device_map = getattr(m, \'_vtm_device_map\', getattr(m, \'hf_device_map\', {}))\n        net._vtm_vae_smoke = getattr(m, \'_vtm_vae_smoke\', None)\n        return net\n    raise ValueError(model)\n\ndef run_backend(model, net, image, options):\n    import torch, numpy as np\n    from PIL import Image\n    import torch.nn.functional as F\n    device = options.get(\'device\', \'cuda\')\n    diagnostics = {}\n    extras = {}\n    if model == \'tadisr_x4\':\n        a = torch.from_numpy(np.asarray(image, dtype=np.float32).copy() / 255).permute(2, 0, 1)[None]\n        a = F.interpolate(a, size=(image.height * 4, image.width * 4), mode=\'bicubic\', align_corners=False)\n\n        def predict(patch):\n            return net(patch.to(device))\n        sr, mask, diagnostics = tiled_tadisr(a, predict, options[\'tile\'], options[\'overlap\'])\n        diagnostics.update(processor_tile_size=[options[\'tile\'], options[\'tile\']], pre_upsample_size=[image.width * 4, image.height * 4])\n        out = (sr[0].clamp(0, 1).permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)\n        extras[\'tadisr_text_mask\'] = Image.fromarray((mask.sigmoid()[0, 0].numpy() * 255).round().astype(np.uint8))\n        return (Image.fromarray(out), extras, diagnostics)\n    if model == \'difftsr\':\n        canvas = Image.new(\'RGB\', (512, 128), \'white\')\n        copy = image.copy()\n        copy.thumbnail((512, 128), Image.Resampling.BICUBIC)\n        offset = ((512 - copy.width) // 2, (128 - copy.height) // 2)\n        canvas.paste(copy, offset)\n        result = Image.fromarray(net.DiffTSR_sample(canvas), \'RGB\')\n        result = result.crop((offset[0], offset[1], offset[0] + copy.width, offset[1] + copy.height))\n        return (result, extras, {\'processor_size\': [512, 128], \'content_size\': list(copy.size), \'processor_offset\': list(offset), \'steps\': options[\'steps\'], \'warning\': \'upstream_text_prior_uses_zhconv_zh_cn; inspect_traditional_forms\'})\n    if model == \'unihir_restoration\':\n        from utils import invert_image\n        result = net(image=invert_image(image), text=\'请对图中破损位置进行定位，并输出破损定位图。\', und_only=False, gen_only=False, reflection_count=options[\'ref_count\'], think=False, max_think_token_n=10000, do_sample=False, cfg_text_scale=1.0, cfg_img_scale=1.0, cfg_interval=[0.0, 1.0], timestep_shift=3.0, num_timesteps=options[\'steps\'], cfg_renorm_min=0.0, cfg_renorm_type=\'global\')\n        final = invert_image(result[\'final_image\'].resize(image.size, Image.Resampling.LANCZOS))\n        extras[\'predictions\'] = {k: result.get(k) for k in (\'text_step1\', \'text_step2\', \'text_merge\')}\n        extras[\'predictions\'].update(refinement_count=options[\'ref_count\'], confidence=None, confidence_status=\'upstream_does_not_supply_calibrated_confidence\')\n        import re\n        extras[\'predictions\'][\'damage_predictions\'] = [{\'box_normalized_1000\': [int(x) for x in match[:4]], \'predicted_character\': match[4], \'evidence\': \'model_generated; not_source_observation\'} for match in re.findall(\'<\\\\|box_start\\\\|>(\\\\d+),(\\\\d+),(\\\\d+),(\\\\d+),(.?)<\\\\|box_end\\\\|>\', result.get(\'text_merge\', \'\'))]\n        for key in (\'draft\',):\n            if isinstance(result.get(key), Image.Image):\n                extras[key] = result[key]\n        for key in (\'tmp_img\', \'tmp_img_gen\'):\n            for i, v in enumerate(result.get(key, [])):\n                if isinstance(v, Image.Image):\n                    extras[f\'{key}_{i:02d}\'] = v\n        return (final, extras, {\'generative_reconstruction\': True, \'alignment\': \'nominal_resize_only; generated_glyphs_unverified\', \'observed_processor_sizes\': net._observed_processor_sizes})\n    raise ValueError(model)\n\ndef verify_active_records(module_name, distribution_name, *, kernel_torch_source_policy=\'strict\', kernel_torch_source_path=None):\n    """Audit active imports; only inherited Torch Python source may report drift."""\n    import base64, csv, importlib, importlib.metadata as md, io\n    from packaging.version import Version\n    from packaging.tags import sys_tags, parse_tag\n    if kernel_torch_source_policy not in (\'strict\', \'warn\'):\n        raise ValueError(\'kernel_torch_source_policy must be strict or warn\')\n    module = importlib.import_module(module_name)\n    dist = md.distribution(distribution_name)\n    actual = getattr(module, \'__version__\', None)\n    if actual and Version(actual).public != Version(dist.version).public:\n        raise RuntimeError(f\'Active module/metadata mismatch: {module_name} {actual} vs {dist.version}\')\n    wheel_tags = set()\n    for line in (dist.read_text(\'WHEEL\') or \'\').splitlines():\n        if line.startswith(\'Tag: \'):\n            wheel_tags.update(parse_tag(line[5:]))\n    if wheel_tags and (not wheel_tags.intersection(set(sys_tags()))):\n        raise RuntimeError(f\'Active distribution has incompatible wheel ABI: {distribution_name}\')\n    record = dist.read_text(\'RECORD\')\n    if not record:\n        raise RuntimeError(f\'Cannot verify active distribution RECORD: {distribution_name}\')\n    rows = list(csv.reader(io.StringIO(record)))\n    if any((len(row) != 3 for row in rows)):\n        raise RuntimeError(f\'Malformed active distribution RECORD: {distribution_name}\')\n    module_file = pathlib.Path(module.__file__).resolve()\n    verify_paths = [module_file]\n    if module_name == \'torch\':\n        verify_paths.append(pathlib.Path(module._C.__file__).resolve())\n    if module_name == \'numpy\':\n        try:\n            native = importlib.import_module(\'numpy._core._multiarray_umath\')\n        except ImportError:\n            native = importlib.import_module(\'numpy.core._multiarray_umath\')\n        verify_paths.append(pathlib.Path(native.__file__).resolve())\n    verified = []\n    notices = []\n    for path in verify_paths:\n        matching = [r for r in rows if pathlib.Path(dist.locate_file(r[0])).resolve() == path]\n        if len(matching) != 1 or not matching[0][1]:\n            raise RuntimeError(f\'Active file not owned by metadata RECORD: {path}\')\n        algorithm, expected = matching[0][1].split(\'=\', 1)\n        hasher = hashlib.new(algorithm)\n        with path.open(\'rb\') as stream:\n            for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                hasher.update(block)\n        observed = base64.urlsafe_b64encode(hasher.digest()).decode().rstrip(\'=\')\n        if observed != expected:\n            inherited_torch_source = module_name == \'torch\' and path == module_file and (path == pathlib.Path(dist.locate_file(\'torch/__init__.py\')).resolve()) and (kernel_torch_source_path is not None) and (path == pathlib.Path(kernel_torch_source_path).resolve())\n            if kernel_torch_source_policy != \'warn\' or not inherited_torch_source:\n                raise RuntimeError(f\'Active package RECORD checksum differs: {path}; expected {algorithm}={expected}, observed {algorithm}={observed}; policy={kernel_torch_source_policy}, kernel Torch source={kernel_torch_source_path}\')\n            notices.append({\'distribution\': distribution_name, \'version\': dist.version, \'path\': str(path), \'status\': \'kernel_source_record_mismatch\', \'expected_hash\': f\'{algorithm}={expected}\', \'observed_hash\': f\'{algorithm}={observed}\', \'note\': \'Source drift recorded; compatibility is checked separately; RECORD integrity unverified\'})\n        else:\n            verified.append(str(path))\n    return {\'verified\': verified, \'warnings\': notices}\n\ndef probe(model, options):\n    import importlib, importlib.metadata as md, platform\n    imports = {\'tadisr_x4\': [\'torch\', \'numpy\', \'diffusers\', \'peft\', \'transformers\', \'sentencepiece\', \'tadisr.inference\', \'tadisr.pipelines\'], \'difftsr\': [\'torch\', \'torchvision\', \'omegaconf\', \'pytorch_lightning\', \'zhconv\', \'einops\', \'model.DiffTSR\', \'model.IDM.models.autoencoder\', \'model.TDM.models.text_prediction\', \'model.TDM.models.modules\'], \'unihir_restoration\': [\'torch\', \'flash_attn\', \'transformers\', \'accelerate\', \'fontTools\', \'utils\']}[model]\n    for name in imports:\n        importlib.import_module(name)\n    import torch, numpy as np\n    if torch.from_numpy(np.ones((2, 2), dtype=np.float32)).numpy().shape != (2, 2):\n        raise RuntimeError(\'NumPy bridge failed\')\n    records_verified = []\n    record_warnings = []\n    source_policy = options.get(\'kernel_torch_source_policy\', \'strict\')\n    for module_name, distribution_name in [(\'torch\', \'torch\'), (\'numpy\', \'numpy\'), (\'PIL\', \'pillow\'), *([(n, n) for n in (\'diffusers\', \'peft\', \'transformers\')] if model == \'tadisr_x4\' else [])]:\n        audit = verify_active_records(module_name, distribution_name, kernel_torch_source_policy=source_policy, kernel_torch_source_path=options.get(\'kernel_torch_source_path\'))\n        records_verified.extend(audit[\'verified\'])\n        record_warnings.extend(audit[\'warnings\'])\n    report = {\'python\': sys.version, \'abi\': sys.implementation.cache_tag, \'platform\': platform.platform(), \'torch\': torch.__version__, \'cuda\': torch.version.cuda, \'numpy\': np.__version__, \'cuda_available\': torch.cuda.is_available(), \'imports\': imports, \'pid\': os.getpid(), \'active_import_records_verified\': records_verified, \'active_import_record_warnings\': record_warnings, \'kernel_torch_source_policy\': source_policy, \'kernel_torch_source_path\': options.get(\'kernel_torch_source_path\'), \'python_executable\': sys.executable, \'prefix\': sys.prefix, \'base_prefix\': sys.base_prefix}\n    if options.get(\'device\', \'cuda\').startswith(\'cuda\'):\n        report.update(restoration_gpu_probe(torch, model, options))\n        report.update(gpu=torch.cuda.get_device_name(0), capability=list(torch.cuda.get_device_capability(0)), compiled_arches=torch.cuda.get_arch_list(), total_vram=torch.cuda.get_device_properties(0).total_memory)\n    if model == \'tadisr_x4\':\n        from diffusers import CogView4Transformer2DModel\n        from transformers import GlmModel\n    if model == \'unihir_restoration\':\n        from flash_attn import flash_attn_func\n        sample = torch.randn(1, 16, 2, 64, device=\'cuda\', dtype=torch.bfloat16)\n        flash_attn_func(sample, sample, sample)\n        torch.cuda.synchronize()\n    from packaging.requirements import Requirement\n    pending = [{\'PIL\': \'pillow\', \'fontTools\': \'fonttools\', \'skimage\': \'scikit-image\', \'cv2\': \'opencv-python-headless\', \'yaml\': \'pyyaml\'}.get(n, n) for n in imports if \'.\' not in n]\n    checked = set()\n    while pending:\n        name = pending.pop()\n        if name in checked:\n            continue\n        checked.add(name)\n        try:\n            dist = md.distribution(name)\n        except md.PackageNotFoundError:\n            continue\n        for spec in dist.requires or []:\n            req = Requirement(spec)\n            if req.marker and (not req.marker.evaluate({\'extra\': \'\'})):\n                continue\n            version = md.version(req.name)\n            if req.specifier and (not req.specifier.contains(version, prereleases=True)):\n                raise RuntimeError(f"Dependency conflict: {dist.metadata[\'Name\']} requires {req}, got {version}")\n            pending.append(req.name)\n    report[\'dependency_closure_checked\'] = sorted(checked)\n    return report\n\ndef main():\n    stack_dump_enabled = False\n    try:\n        import faulthandler, signal\n        faulthandler.enable()\n        if hasattr(signal, \'SIGUSR1\'):\n            faulthandler.register(signal.SIGUSR1, all_threads=True)\n            stack_dump_enabled = True\n    except (OSError, RuntimeError):\n        pass\n    cfg = json.loads(pathlib.Path(sys.argv[1]).read_text(encoding=\'utf8\'))\n    if cfg.get(\'vendor\'):\n        sys.path.insert(0, cfg[\'vendor\'])\n    result_path = cfg[\'ready\']\n    try:\n        if cfg.get(\'action\') == \'isolation_probe\':\n            try:\n                socket.create_connection((\'127.0.0.1\', 9), timeout=0.1)\n            except RuntimeError as error:\n                blocked = \'OFFLINE\' in str(error)\n            else:\n                blocked = False\n            if not blocked:\n                raise RuntimeError(\'Worker offline guard did not reject a connection\')\n            atomic_json(result_path, {\'state\': \'ready\', \'pid\': os.getpid(), \'offline\': True, \'network_block_verified\': blocked})\n            return\n        report = probe(cfg[\'model\'], cfg[\'options\'])\n        if cfg.get(\'action\') == \'probe\':\n            atomic_json(result_path, {\'state\': \'ready\', \'report\': report})\n            return\n        if cfg.get(\'action\') == \'prepare_prompt\':\n            import runpy, torch\n            target = pathlib.Path(cfg[\'assets\'][\'prompt_output\'])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = target.with_suffix(\'.part\')\n            sys.argv = [\'prepare_prompt_embeddings.py\', \'--base-model\', cfg[\'assets\'][\'base_model\'], \'--output\', str(temporary), \'--device\', cfg[\'options\'].get(\'prompt_device\', cfg[\'options\'].get(\'device\', \'cuda\'))]\n            torch.manual_seed(cfg[\'options\'].get(\'seed\', 42))\n            torch.cuda.reset_peak_memory_stats()\n            started = time.monotonic()\n            runpy.run_path(str(pathlib.Path(cfg[\'vendor\']) / \'scripts/prepare_prompt_embeddings.py\'), run_name=\'__main__\')\n            temporary.replace(target)\n            torch.cuda.synchronize()\n            atomic_json(result_path, {\'state\': \'ready\', \'report\': report, \'prompt_seconds\': time.monotonic() - started, \'peak_vram_bytes\': torch.cuda.max_memory_allocated(), \'derived_offline\': True})\n            return\n        import torch, numpy as np\n        torch.manual_seed(cfg[\'options\'].get(\'seed\', 42))\n        np.random.seed(cfg[\'options\'].get(\'seed\', 42))\n        started = time.monotonic()\n        net = load_backend(cfg[\'model\'], cfg[\'assets\'], cfg[\'options\'])\n        restoration_cuda_metrics(torch)\n        atomic_json(result_path, {\'state\': \'ready\', \'report\': report, \'load_seconds\': time.monotonic() - started, \'stack_dump_enabled\': stack_dump_enabled, \'device_map\': getattr(net, \'_vtm_device_map\', {}), \'devices\': restoration_cuda_metrics(torch), \'vae_codec_check\': getattr(net, \'_vtm_vae_smoke\', None)})\n    except Exception:\n        atomic_json(result_path, {\'state\': \'failed\', \'error\': traceback.format_exc(), \'runtime_context\': {\'python_executable\': sys.executable, \'prefix\': sys.prefix, \'base_prefix\': sys.base_prefix, \'kernel_torch_source_policy\': cfg.get(\'options\', {}).get(\'kernel_torch_source_policy\'), \'kernel_torch_source_path\': cfg.get(\'options\', {}).get(\'kernel_torch_source_path\')}})\n        return\n    for line in sys.stdin:\n        job = json.loads(line)\n        output = pathlib.Path(job[\'output\'])\n        output.mkdir(parents=True, exist_ok=True)\n        started = time.monotonic()\n        try:\n            from PIL import Image\n            restoration_cuda_metrics(torch, reset=True)\n            with Image.open(job[\'input\']) as src:\n                image = src.convert(\'RGB\')\n            torch.manual_seed(job[\'seed\'])\n            np.random.seed(job[\'seed\'] % (2 ** 32 - 1))\n            from contextlib import nullcontext\n            progress_dir = output.parent / (\'progress_\' + output.name)\n            observer = UniHIRProgress(net, progress_dir, input_size=list(image.size)) if cfg[\'model\'] == \'unihir_restoration\' else nullcontext()\n            with observer, torch.inference_mode():\n                restored, extras, diagnostics = run_backend(cfg[\'model\'], net, image, cfg[\'options\'])\n            if cfg[\'model\'] == \'unihir_restoration\':\n                diagnostics[\'runtime_progress\'] = observer.snapshot()\n            restored.save(output / \'result.png\', compress_level=1)\n            extra_refs = {}\n            for key, value in extras.items():\n                if isinstance(value, Image.Image):\n                    value.save(output / (key + \'.png\'), compress_level=1)\n                    extra_refs[key] = key + \'.png\'\n                else:\n                    atomic_json(output / (key + \'.json\'), value)\n                    extra_refs[key] = key + \'.json\'\n            restoration_cuda_metrics(torch)\n            peak = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0\n            reserved = torch.cuda.max_memory_reserved() if torch.cuda.is_available() else 0\n            atomic_json(job[\'response\'], {\'state\': \'ok\', \'seconds\': time.monotonic() - started, \'peak_vram_bytes\': peak, \'peak_reserved_vram_bytes\': reserved, \'per_gpu_memory\': restoration_cuda_metrics(torch), \'device_map\': getattr(net, \'_vtm_device_map\', {}), \'per_gpu_memory\': restoration_cuda_metrics(torch), \'device_map\': getattr(net, \'_vtm_device_map\', {}), \'diagnostics\': diagnostics, \'extras\': extra_refs})\n        except Exception:\n            atomic_json(job[\'response\'], {\'state\': \'failed\', \'seconds\': time.monotonic() - started, \'error\': traceback.format_exc()})\nimport ast\nimport hashlib\nimport importlib.util\nimport os\nfrom pathlib import Path\nimport tempfile\nfrom types import SimpleNamespace\nimport zipfile\nLEARNED_DOCUMENT_TASKS = {}\nDOCUMENT_VARIANTS = {}\n\ndef restoration_memory_limits(torch, options):\n    cap = options.get(\'max_memory\', \'18GiB\')\n    if not isinstance(cap, str) or not cap.endswith(\'GiB\'):\n        raise ValueError(\'max_memory must be a per-GPU limit such as 18GiB\')\n    requested = int(float(cap[:-3]) * 1024 ** 3)\n    reserve = int(float(options.get(\'activation_reserve_gib\', 4)) * 1024 ** 3)\n    if requested <= 0 or reserve < 1024 ** 3:\n        raise ValueError(\'Positive weight budget and >=1 GiB activation reserve required\')\n    limits = {}\n    for index in range(torch.cuda.device_count()):\n        free, total = torch.cuda.mem_get_info(index)\n        limit = min(requested, free - reserve)\n        if index == 0:\n            limit -= int(options.get(\'entry_reserve_gib\', 2) * 1024 ** 3)\n        if limit < 4 * 1024 ** 3:\n            raise MemoryError(f\'GPU {index}: insufficient free VRAM after activation reserve\')\n        limits[index] = limit\n    return limits\n\ndef verify_restoration_device_map(mapping, count, *, require_multiple=True):\n    devices = set()\n    for value in mapping.values():\n        if isinstance(value, int):\n            index = value\n        else:\n            label = str(value)\n            if not label.startswith(\'cuda:\'):\n                raise MemoryError(f\'Model would spill to {label}; reduce scope or change explicit placement budgets\')\n            index = int(label.split(\':\')[1])\n        if not 0 <= index < count:\n            raise ValueError(f\'Device map refers to unavailable GPU {index}\')\n        devices.add(index)\n    if require_multiple and len(devices) < 2:\n        raise ValueError(\'Sharded placement did not distribute the model across multiple GPUs\')\n    return mapping\n\ndef unihir_weight_limits(torch, options):\n    """Reserve a whole visible GPU for the full-resolution VAE encode/decode."""\n    limits = restoration_memory_limits(torch, options)\n    vae = options.get(\'vae_gpu_id\')\n    if vae is None:\n        return (limits, None)\n    if vae == \'auto\':\n        vae = torch.cuda.device_count() - 1\n    if isinstance(vae, bool) or not isinstance(vae, int) or vae not in limits:\n        raise ValueError(\'UniHIR vae_gpu_id must be auto or a visible logical GPU index\')\n    if len(limits) < 3:\n        raise MemoryError(\'Dedicated UniHIR VAE requires >=3 visible GPUs so BAGEL remains sharded across >=2 GPUs\')\n    if torch.cuda.mem_get_info(vae)[0] < 16 * 1024 ** 3:\n        raise MemoryError(f\'UniHIR VAE GPU {vae} needs >=16 GiB free before loading; no automatic resize\')\n    del limits[vae]\n    return (limits, vae)\n\ndef restoration_gpu_probe(torch, model, options):\n    """Validate every visible GPU; aggregate capacity is only a placement precheck."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required but unavailable\')\n    devices = []\n    for index in range(torch.cuda.device_count()):\n        (torch.ones(8, device=f\'cuda:{index}\') * 2).sum().item()\n        torch.cuda.synchronize(index)\n        devices.append(dict(index=index, name=torch.cuda.get_device_name(index), total_bytes=torch.cuda.get_device_properties(index).total_memory, free_bytes=torch.cuda.mem_get_info(index)[0], capability=list(torch.cuda.get_device_capability(index))))\n    sharded = options.get(\'placement\') == \'sharded\'\n    if sharded and len(devices) < 2:\n        raise RuntimeError(f\'{model}: sharded placement requires >=2 visible GPUs\')\n    if model in (\'unihir_restoration\', \'tadisr_x4\') and sharded:\n        limits = unihir_weight_limits(torch, options)[0] if model == \'unihir_restoration\' else restoration_memory_limits(torch, options)\n        minimum = 32 if model == \'unihir_restoration\' else 28\n        if sum(limits.values()) < minimum * 1024 ** 3:\n            raise MemoryError(f\'{model}: insufficient aggregate weight budgets; inference workspace is reserved separately\')\n    elif model == \'unihir_restoration\' and devices[0][\'total_bytes\'] < 75 * 1024 ** 3:\n        raise RuntimeError(\'Single-GPU UniHIR needs >=75 GiB; use placement=sharded for multiple L4 GPUs\')\n    elif model == \'tadisr_x4\' and devices[0][\'total_bytes\'] < 24 * 1024 ** 3:\n        raise RuntimeError(\'Single-GPU float32 TADiSR needs >=24 GiB; use placement=sharded on L4\')\n    if model == \'unihir_restoration\' and any((d[\'capability\'][0] < 8 for d in devices)):\n        raise RuntimeError(\'UniHIR BF16/FlashAttention requires Ampere-or-newer GPUs\')\n    report = dict(devices=devices, placement=options.get(\'placement\', \'single\'), cuda_visible_devices=__import__(\'os\').environ.get(\'CUDA_VISIBLE_DEVICES\'), weight_capacity_check_only=True)\n    if model == \'unihir_restoration\' and sharded:\n        weight_limits, vae_gpu = unihir_weight_limits(torch, options)\n        report.update(weight_gpu_ids=sorted(weight_limits), vae_gpu_id=vae_gpu)\n    return report\n\ndef cpu_staged_checkpoint_dispatch(dispatch, model, **kwargs):\n    """Stage checkpoint tensors on CPU before Accelerate casts/places them.\n\n    The pinned Accelerate safetensors loader otherwise holds the checkpoint\'s\n    original dtype on GPU alongside the converted model. Clone function globals\n    for this call only; neither Accelerate nor the upstream module is patched.\n    Final model placement still follows the validated GPU-only device map.\n    """\n    import types\n\n    def clone(function, replacements):\n        result = types.FunctionType(function.__code__, {**function.__globals__, **replacements}, function.__name__, function.__defaults__, function.__closure__)\n        result.__kwdefaults__ = function.__kwdefaults__\n        return result\n    load = dispatch.__globals__[\'load_checkpoint_in_model\']\n    read = load.__globals__[\'load_state_dict\']\n\n    def cpu_read(checkpoint_file, device_map=None):\n        return read(checkpoint_file, device_map=None)\n    staged_load = clone(load, {\'load_state_dict\': cpu_read})\n    return clone(dispatch, {\'load_checkpoint_in_model\': staged_load})(model, **kwargs)\n\ndef verify_unihir_vae(vae):\n    """Check the real codec on a tiny tensor, restoring all touched RNG states."""\n    import time\n    import torch\n    parameter = next(vae.parameters())\n    device = parameter.device\n    devices = [device.index] if device.type == \'cuda\' else []\n    started = time.monotonic()\n    with torch.random.fork_rng(devices=devices), torch.inference_mode():\n        image = torch.zeros((1, 3, 64, 64), device=device, dtype=parameter.dtype)\n        latent = vae.encode(image)\n        decoded = vae.decode(latent)\n        if decoded.shape != image.shape:\n            raise ValueError(\'UniHIR VAE roundtrip dimensions differ\')\n        if not torch.isfinite(latent).all().item() or not torch.isfinite(decoded).all().item():\n            raise ValueError(\'UniHIR VAE produced non-finite values in codec precheck\')\n    return dict(device=str(device), dtype=str(parameter.dtype), input_size=[64, 64], seconds=round(time.monotonic() - started, 3), status=\'passed\', scope=\'codec_only; not full model inference\')\n\ndef load_unihir_multi_gpu(assets, options):\n    import torch\n    import types\n    import utils as upstream\n    from accelerate import infer_auto_device_map, load_checkpoint_and_dispatch\n    limits, vae_gpu = unihir_weight_limits(torch, options)\n\n    def validate(mapping):\n        verify_restoration_device_map(mapping, torch.cuda.device_count())\n        for device in mapping.values():\n            index = device if isinstance(device, int) else int(str(device).split(\':\')[1])\n            if index not in limits:\n                raise MemoryError(\'UniHIR weight map uses the GPU reserved for VAE\')\n        return mapping\n\n    def infer(model, **kwargs):\n        kwargs.update(max_memory=limits, dtype=torch.bfloat16, no_split_module_classes=[\'Qwen2MoTDecoderLayer\'])\n        return validate(infer_auto_device_map(model, **kwargs))\n\n    def dispatch(model, **kwargs):\n        validate(kwargs[\'device_map\'])\n        return cpu_staged_checkpoint_dispatch(load_checkpoint_and_dispatch, model, **kwargs)\n    loader = types.FunctionType(upstream.load_model.__code__, {**upstream.load_model.__globals__, \'infer_auto_device_map\': infer, \'load_checkpoint_and_dispatch\': dispatch}, argdefs=upstream.load_model.__defaults__)\n    result = loader(assets[\'base_model\'], max_mem_per_gpu=options.get(\'max_memory\', \'18GiB\'))\n    result[0]._vtm_device_map = {k: str(v) for k, v in result[0].hf_device_map.items()}\n    if vae_gpu is not None:\n        result[2].to(f\'cuda:{vae_gpu}\')\n        result[0]._vtm_device_map[\'vae_model\'] = f\'cuda:{vae_gpu}\'\n    if options.get(\'verify_vae\', False):\n        result[0]._vtm_vae_smoke = verify_unihir_vae(result[2])\n    return result\n\ndef load_tadisr_multi_gpu(assets, options):\n    import pathlib\n    import torch\n    from accelerate import dispatch_model, infer_auto_device_map\n    from tadisr.inference import CogView4Config, load_cogview4\n    net = load_cogview4(CogView4Config(pathlib.Path(assets[\'base_model\']), pathlib.Path(assets[\'checkpoint\']), pathlib.Path(assets[\'prompt_embeddings\']), \'cpu\'))\n    limits = restoration_memory_limits(torch, options)\n    no_split = getattr(net.transformer, \'_no_split_modules\', None) or [\'CogView4Block\']\n    mapping = infer_auto_device_map(net.transformer, max_memory=limits, no_split_module_classes=no_split, dtype=torch.float32)\n    verify_restoration_device_map(mapping, len(limits))\n    net.transformer = dispatch_model(net.transformer, device_map=mapping, main_device=0, force_hooks=True)\n    net.execution_device = torch.device(\'cuda:0\')\n    net.vae.to(net.execution_device)\n    net.js_decoder.to(net.execution_device)\n    net.prompt_embeds = net.prompt_embeds.to(net.execution_device)\n    net._vtm_device_map = {k: str(v) for k, v in mapping.items()}\n    return net\n\ndef restoration_cuda_metrics(torch, *, reset=False):\n    records = []\n    if torch.cuda.is_available():\n        for index in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(index)\n            if reset:\n                torch.cuda.reset_peak_memory_stats(index)\n            records.append(dict(index=index, peak_allocated_bytes=torch.cuda.max_memory_allocated(index), peak_reserved_bytes=torch.cuda.max_memory_reserved(index)))\n    return records\nimport functools\nimport json\nimport threading\nimport time\n\nclass UniHIRProgress:\n\n    def __init__(self, net, directory, interval=30, emit=print, input_size=None):\n        self.net = net\n        self.directory = Path(directory)\n        self.interval = interval\n        self.emit = emit\n        self.lock = threading.RLock()\n        self.stop = threading.Event()\n        self.stack = []\n        self.replacements = []\n        self.hooks = []\n        self.sequence = 0\n        self.forwards = 0\n        self.tokens = 0\n        self.artifacts = []\n        self.started = time.monotonic()\n        self.status = \'running\'\n        self.error_stage = None\n        self.input_size = input_size\n        self.generation_sizes = []\n\n    def snapshot(self):\n        with self.lock:\n            return dict(format=\'vtm-unihir-progress-v1\', status=self.status, stage=\' > \'.join(self.stack) or \'between_stages\', error_stage=self.error_stage, activity=self.sequence, model_forwards=self.forwards, text_tokens=self.tokens, seconds=round(time.monotonic() - self.started, 3), artifacts=list(self.artifacts), input_size=self.input_size, generation_sizes=list(self.generation_sizes), observed_processor_sizes=dict(getattr(self.net, \'_observed_processor_sizes\', {})), intermediate_images_are_final=False)\n\n    def publish(self, announce=False):\n        with self.lock:\n            data = self.snapshot()\n            self.directory.mkdir(parents=True, exist_ok=True)\n            target = self.directory / \'progress.json\'\n            temporary = target.with_suffix(\'.tmp\')\n            temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf8\')\n            for attempt in range(20):\n                try:\n                    temporary.replace(target)\n                    break\n                except PermissionError:\n                    if attempt == 19:\n                        raise\n                    time.sleep(0.005)\n        if announce:\n            self.emit(\'[UniHIR progress] \' + json.dumps(data, ensure_ascii=False), flush=True)\n\n    def pulse(self, kind):\n        with self.lock:\n            self.sequence += 1\n            if kind == \'forward\':\n                self.forwards += 1\n            else:\n                self.tokens += 1\n\n    def wrap(self, owner, name, label):\n        original = getattr(owner, name, None)\n        if not callable(original):\n            return\n        owned = name in getattr(owner, \'__dict__\', {})\n\n        @functools.wraps(original)\n        def observed(*args, **kwargs):\n            with self.lock:\n                self.stack.append(label)\n                self.sequence += 1\n                if label == \'gen_image\' and args and isinstance(args[0], (list, tuple)) and (len(args[0]) == 2):\n                    self.generation_sizes.append(list(args[0]))\n            self.publish(True)\n            try:\n                result = original(*args, **kwargs)\n                if label in (\'gen_image\', \'gen_text\'):\n                    self.capture(label, result)\n                return result\n            except BaseException:\n                with self.lock:\n                    if self.error_stage is None:\n                        self.error_stage = \' > \'.join(self.stack)\n                raise\n            finally:\n                with self.lock:\n                    self.stack.pop()\n                    self.sequence += 1\n                self.publish(True)\n        setattr(owner, name, observed)\n        self.replacements.append((owner, name, original, owned))\n\n    def capture(self, label, result):\n        folder = self.directory / \'intermediate_not_final\'\n        folder.mkdir(exist_ok=True)\n        number = len(self.artifacts) + 1\n        if label == \'gen_text\' and isinstance(result, str):\n            target = folder / f\'{number:03d}_text.txt\'\n            target.write_text(result, encoding=\'utf8\')\n        elif label == \'gen_image\' and hasattr(result, \'save\'):\n            target = folder / f\'{number:03d}_image.png\'\n            result.save(target, format=\'PNG\')\n        else:\n            return\n        with self.lock:\n            self.artifacts.append(str(target.relative_to(self.directory)))\n\n    def heartbeat(self):\n        while not self.stop.wait(self.interval):\n            self.publish(True)\n\n    def __enter__(self):\n        try:\n            for name in (\'update_context_text\', \'update_context_image\', \'update_context_image_gen\', \'gen_image\', \'gen_text\', \'decode_image\'):\n                self.wrap(self.net, name, name)\n            vae = getattr(self.net, \'vae_model\', None)\n            if vae is not None:\n                for name in (\'encode\', \'decode\'):\n                    self.wrap(vae, name, \'vae.\' + name)\n            model = getattr(self.net, \'model\', None)\n            language = getattr(model, \'language_model\', None)\n            decoder = getattr(language, \'model\', None)\n            layers = getattr(decoder, \'layers\', None)\n            if layers is not None and len(layers):\n                self.hooks.append(layers[-1].register_forward_hook(lambda *a: self.pulse(\'forward\')))\n            head = getattr(language, \'lm_head\', None)\n            if head is not None:\n                self.hooks.append(head.register_forward_hook(lambda *a: self.pulse(\'token\')))\n            self.publish(True)\n            self.thread = threading.Thread(target=self.heartbeat, daemon=True)\n            self.thread.start()\n            return self\n        except BaseException:\n            self.__exit__(*__import__(\'sys\').exc_info())\n            raise\n\n    def __exit__(self, kind, value, traceback):\n        self.stop.set()\n        if hasattr(self, \'thread\'):\n            self.thread.join(timeout=2)\n        for hook in self.hooks:\n            hook.remove()\n        for owner, name, original, owned in reversed(self.replacements):\n            if owned:\n                setattr(owner, name, original)\n            else:\n                delattr(owner, name)\n        if kind is None:\n            for relative in self.artifacts:\n                target = (self.directory / relative).resolve()\n                if target.is_relative_to(self.directory.resolve()) and target.is_file():\n                    target.unlink()\n            folder = self.directory / \'intermediate_not_final\'\n            if folder.is_dir():\n                folder.rmdir()\n            self.artifacts.clear()\n        with self.lock:\n            self.status = \'failed\' if kind else \'complete\'\n        self.publish(True)\n        return False\nif __name__ == \'__main__\':\n    main()\n'
RESTORATION_CODE_SIGNATURE = '250c0b7888415fb76f288d2f9f67718a967c6baa361fae085360806ed22439ef'

FLASH_RUNTIME_SOURCE = '"""Build isolated restoration overlays from staged files, entirely offline.\n\nRun on Kaggle Linux with the working Python 3.13 / Torch CUDA 13 kernel.\nFlashAttention is compiled for the detected GPU against that same Torch ABI.\nNo weights are loaded. Notebook 1 separately runs its worker/GPU probes.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib.metadata as metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport time\nimport zipfile\nFLASH_CUDA_PROBE = \'import torch; from flash_attn import flash_attn_varlen_func, flash_attn_func; q=torch.randn(16,2,64,device="cuda",dtype=torch.bfloat16); cu=torch.tensor([0,16],device="cuda",dtype=torch.int32); out=flash_attn_varlen_func(q,q,q,cu,cu,16,16); fixed=flash_attn_func(q[None],q[None],q[None]); torch.cuda.synchronize(); assert out.shape==q.shape and torch.isfinite(out).all(); assert fixed.shape==q[None].shape and torch.isfinite(fixed).all()\'\n\ndef flash_cache_identity(torch, source):\n    return dict(version=1, source_sha256=sha256(source), python=list(sys.version_info[:2]), platform=sys.platform, machine=platform.machine(), libc=list(platform.libc_ver()), torch=torch.__version__, cuda=torch.version.cuda, cxx11_abi=bool(torch._C._GLIBCXX_USE_CXX11_ABI), capability=list(torch.cuda.get_device_capability()))\n\ndef flash_cache_output():\n    return Path(os.environ.get(\'VTM_FLASH_CACHE_OUTPUT\', \'/kaggle/working/flash_attention_cache\'))\n\ndef flash_cache_roots(packages_dir=None):\n    roots = []\n    if os.environ.get(\'VTM_FLASH_CACHE_INPUT\'):\n        roots.append(Path(os.environ[\'VTM_FLASH_CACHE_INPUT\']))\n    if packages_dir is not None:\n        roots.append(Path(packages_dir) / \'image_restoration/flash_attention_cache\')\n    roots.append(flash_cache_output())\n    return roots\n\ndef find_flash_cache(identity, roots):\n    """Require both runtime identity and wheel integrity; never guess by filename."""\n    from packaging.tags import sys_tags\n    from packaging.utils import parse_wheel_filename\n    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n    for root in roots:\n        for record in (Path(root) / key / \'flash_wheel.json\', Path(root) / \'flash_wheel.json\'):\n            if not record.is_file():\n                continue\n            data = json.loads(record.read_text(encoding=\'utf8\'))\n            if data.get(\'identity\') != identity:\n                continue\n            if not data.get(\'cuda_probe_verified\'):\n                continue\n            wheel = contained(record.parent, data[\'wheel\'])\n            if wheel.suffix != \'.whl\' or not wheel.is_file() or wheel.stat().st_size != data[\'bytes\'] or (sha256(wheel) != data[\'sha256\']):\n                raise ValueError(f\'Corrupt FlashAttention cache: {record}\')\n            if not set(sys_tags()).intersection(parse_wheel_filename(wheel.name)[3]):\n                raise ValueError(f\'Incompatible FlashAttention wheel tags: {wheel.name}\')\n            return (wheel, data)\n    return None\n\ndef save_flash_cache(identity, wheel, root):\n    """Call only after the extension has passed an actual CUDA probe."""\n    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n    destination = Path(root) / key\n    destination.mkdir(parents=True, exist_ok=True)\n    target = destination / wheel.name\n    digest = sha256(wheel)\n    if wheel.resolve() != target.resolve():\n        temporary = target.with_suffix(\'.whl.tmp\')\n        shutil.copyfile(wheel, temporary)\n        if sha256(temporary) != digest:\n            raise ValueError(\'FlashAttention cache copy checksum mismatch\')\n        temporary.replace(target)\n    data = dict(identity=identity, wheel=wheel.name, sha256=digest, bytes=target.stat().st_size, cuda_probe_verified=True, model_inference_verified=False)\n    record = destination / \'flash_wheel.json\'\n    temporary = record.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(data, indent=2) + \'\\n\', encoding=\'utf8\')\n    temporary.replace(record)\n    print(f\'FlashAttention cache saved: {destination}; retain this folder for the next session\', flush=True)\n    return data\n\ndef export_flash_cache(packages_dir, runtime_dir, output_dir=None):\n    """Rescue a wheel from an existing session, including older runtime builders."""\n    import torch\n    root = Path(packages_dir) / \'image_restoration\'\n    plan = json.loads((root / \'manifests/unihir_runtime.json\').read_text(encoding=\'utf8\'))\n    validate_plan(root, plan)\n    for name, expected in plan[\'kernel_reuse\'].items():\n        if metadata.version(name) != expected:\n            raise RuntimeError(f\'Kernel changed: {name}\')\n    source = contained(root, plan[\'native_sources\'][0][\'path\'])\n    identity = flash_cache_identity(torch, source)\n    for folder in sorted((Path(runtime_dir) / \'environments\').glob(\'unihir_restoration_*\')):\n        receipt = folder / \'install.json\'\n        if not receipt.is_file():\n            continue\n        installed = json.loads(receipt.read_text(encoding=\'utf8\'))\n        for native in installed.get(\'native\', []):\n            if native.get(\'sm\') != \'\'.join(map(str, identity[\'capability\'])):\n                continue\n            wheel = contained(folder / \'native_build/wheels\', native[\'wheel\'])\n            if not wheel.is_file() or sha256(wheel) != native[\'sha256\']:\n                continue\n            subprocess.run([str(folder / \'bin/python\'), \'-c\', FLASH_CUDA_PROBE], env=offline_environment(), check=True, timeout=120)\n            save_flash_cache(identity, wheel, output_dir or flash_cache_output())\n            return\n    raise FileNotFoundError(\'No completed FlashAttention runtime/wheel found; wait for the current build to finish\')\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 ** 2), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef contained(root, relative):\n    root = Path(root).resolve()\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root):\n        raise ValueError(f\'Path outside bundle: {relative}\')\n    return path\n\ndef native_build_resources(*, cpus=None, available_bytes=None, environ=None):\n    """Bound compiler parallelism using CPU affinity and effective available RAM."""\n    environ = os.environ if environ is None else environ\n    if cpus is None:\n        cpus = len(os.sched_getaffinity(0)) if hasattr(os, \'sched_getaffinity\') else os.cpu_count() or 1\n    if available_bytes is None:\n        candidates = []\n        try:\n            values = dict((line.split(\':\', 1) for line in Path(\'/proc/meminfo\').read_text().splitlines()))\n            candidates.append(int(values[\'MemAvailable\'].split()[0]) * 1024)\n        except (OSError, KeyError, ValueError):\n            pass\n        try:\n            limit = Path(\'/sys/fs/cgroup/memory.max\').read_text().strip()\n            used = int(Path(\'/sys/fs/cgroup/memory.current\').read_text())\n            if limit.isdigit():\n                candidates.append(max(0, int(limit) - used))\n        except (OSError, ValueError):\n            pass\n        available_bytes = min(candidates) if candidates else None\n    requested = int(environ.get(\'VTM_FLASH_MAX_JOBS\', \'8\'))\n    threads = int(environ.get(\'VTM_FLASH_NVCC_THREADS\', \'2\'))\n    if not 1 <= requested <= 16 or not 1 <= threads <= 4:\n        raise ValueError(\'VTM_FLASH_MAX_JOBS must be 1..16; VTM_FLASH_NVCC_THREADS must be 1..4\')\n    memory_jobs = max(1, int(available_bytes * 0.75) // (12 * 1024 ** 3)) if available_bytes is not None else 2\n    jobs = max(1, min(requested, max(1, cpus // threads), memory_jobs))\n    return dict(cpu_affinity=cpus, available_ram_bytes=available_bytes, max_jobs=jobs, nvcc_threads=threads, requested_max_jobs=requested, memory_budget_per_job_gib=12, available_ram_fraction=0.75)\n\ndef offline_environment():\n    env = dict(os.environ)\n    for key in (\'PYTHONPATH\', \'PYTHONHOME\', \'PYTHONUSERBASE\'):\n        env.pop(key, None)\n    env.update(PYTHONNOUSERSITE=\'1\', PYTHONUTF8=\'1\', PIP_NO_INDEX=\'1\', PIP_DISABLE_PIP_VERSION_CHECK=\'1\', HF_HUB_OFFLINE=\'1\', TRANSFORMERS_OFFLINE=\'1\', HF_DATASETS_OFFLINE=\'1\', HF_HUB_DISABLE_TELEMETRY=\'1\', MPLBACKEND=\'Agg\', FLASH_ATTENTION_FORCE_BUILD=\'TRUE\')\n    resources = native_build_resources(environ=env)\n    env.update(MAX_JOBS=str(resources[\'max_jobs\']), NVCC_THREADS=str(resources[\'nvcc_threads\']))\n    return env\n\ndef validate_plan(root, plan):\n    from packaging.tags import sys_tags\n    from packaging.utils import parse_wheel_filename\n    tags = set(sys_tags())\n    for record in plan[\'wheels\'] + plan.get(\'native_sources\', []):\n        path = contained(root, record[\'path\'])\n        if not path.is_file() or path.stat().st_size != record[\'bytes\'] or sha256(path) != record[\'sha256\']:\n            raise ValueError(f\'Missing or corrupt offline payload: {path}\')\n        if path.suffix == \'.whl\' and (not tags.intersection(parse_wheel_filename(path.name)[3])):\n            raise RuntimeError(f\'Incompatible wheel: {path.name}\')\n    if plan.get(\'dependency_problems\') or plan.get(\'state\') != \'prepared_unverified\':\n        raise RuntimeError(f"Bundle dependency plan is incomplete: {plan.get(\'dependency_problems\')}")\n\ndef build_native(python, source, folder, env, packages_dir=None):\n    """Only a verified PyPI sdist, including CUTLASS; never a git/network build."""\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'FlashAttention build/probe needs an available CUDA GPU\')\n    major, minor = torch.cuda.get_device_capability()\n    identity = flash_cache_identity(torch, source)\n    cached = find_flash_cache(identity, flash_cache_roots(packages_dir))\n    if cached:\n        wheel, record = cached\n        print(f\'Reusing FlashAttention wheel; compilation skipped: {wheel}\', flush=True)\n        local_wheels = folder / \'native_build/wheels\'\n        local_wheels.mkdir(parents=True, exist_ok=True)\n        local_wheel = local_wheels / wheel.name\n        if local_wheel.resolve() != wheel.resolve():\n            shutil.copyfile(wheel, local_wheel)\n        if sha256(local_wheel) != record[\'sha256\']:\n            raise ValueError(\'Cached wheel copy changed\')\n        wheel = local_wheel\n        subprocess.run([str(python), \'-m\', \'pip\', \'--isolated\', \'install\', \'--no-index\', \'--no-deps\', \'--ignore-installed\', str(wheel)], env=env, check=True)\n        return dict(wheel=wheel.name, sha256=record[\'sha256\'], sm=f\'{major}{minor}\', cached=True, wheel_path=str(wheel), build_seconds=0)\n    if os.environ.get(\'VTM_FLASH_REQUIRE_CACHE\') == \'1\':\n        raise FileNotFoundError(\'No compatible verified FlashAttention wheel cache. Attach flash_attention_cache and set VTM_FLASH_CACHE_INPUT; compilation is disabled.\')\n    env = dict(env, FLASH_ATTN_CUDA_ARCHS=f\'{major}{minor}\', TORCH_CUDA_ARCH_LIST=f\'{major}.{minor}\')\n    scratch = folder / \'native_build\'\n    unpacked = scratch / \'source\'\n    unpacked.mkdir(parents=True, exist_ok=True)\n    with tarfile.open(source) as archive:\n        archive.extractall(unpacked, filter=\'data\')\n    roots = [p for p in unpacked.iterdir() if p.is_dir()]\n    if len(roots) != 1 or not (roots[0] / \'csrc/cutlass/include/cutlass/cutlass.h\').is_file():\n        raise ValueError(\'Incomplete FlashAttention source/CUTLASS bundle\')\n    if (roots[0] / \'.git\').exists():\n        raise ValueError(\'A git checkout could run network submodule updates; refusing\')\n    wheels = scratch / \'wheels\'\n    wheels.mkdir(exist_ok=True)\n    resources = native_build_resources(environ=env)\n    env.update(MAX_JOBS=str(resources[\'max_jobs\']), NVCC_THREADS=str(resources[\'nvcc_threads\']))\n    config = dict(resources, source_sha256=sha256(source), torch=torch.__version__, cuda=torch.version.cuda, gpu_capability=[major, minor], torch_cuda_arch_list=env[\'TORCH_CUDA_ARCH_LIST\'], flash_attn_cuda_archs=env[\'FLASH_ATTN_CUDA_ARCHS\'], model_weights_loaded=False)\n    (scratch / \'build_config.json\').write_text(json.dumps(config, indent=2) + \'\\n\', encoding=\'utf8\')\n    started = time.monotonic()\n    with (scratch / \'build.log\').open(\'w\', encoding=\'utf8\') as log:\n        command = [str(python), \'-m\', \'pip\', \'--isolated\', \'wheel\', \'--verbose\', \'--no-index\', \'--no-deps\', \'--no-build-isolation\', \'--wheel-dir\', str(wheels), str(roots[0])]\n        log.write(json.dumps(dict(config, command=command)) + \'\\n\')\n        log.flush()\n        print(f"Building FlashAttention for sm_{major}{minor}; MAX_JOBS={resources[\'max_jobs\']}, NVCC_THREADS={resources[\'nvcc_threads\']}; log: {scratch / \'build.log\'}", flush=True)\n        process = subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT)\n        last_notice = started\n\n        def progress(state):\n            with (scratch / \'build.log\').open(\'rb\') as stream:\n                stream.seek(0, 2)\n                stream.seek(max(0, stream.tell() - 32768))\n                tail = stream.read().decode(\'utf8\', errors=\'replace\')\n            counts = re.findall(\'\\\\[(\\\\d+)/(\\\\d+)\\\\]\', tail)\n            record = dict(status=state, pid=process.pid, seconds=round(time.monotonic() - started, 2), max_jobs=resources[\'max_jobs\'], nvcc_threads=resources[\'nvcc_threads\'], ninja_finished=int(counts[-1][0]) if counts else None, ninja_total=int(counts[-1][1]) if counts else None)\n            target = scratch / \'build_progress.json\'\n            temporary = target.with_suffix(\'.tmp\')\n            temporary.write_text(json.dumps(record, indent=2) + \'\\n\', encoding=\'utf8\')\n            temporary.replace(target)\n            return record\n        try:\n            progress(\'running\')\n            while process.poll() is None:\n                if time.monotonic() - last_notice >= 30:\n                    record = progress(\'running\')\n                    count = f"{record[\'ninja_finished\']}/{record[\'ninja_total\']}" if record[\'ninja_total\'] else \'compiler initialization\'\n                    print(f"FlashAttention build: {count}, {record[\'seconds\']:.0f}s, {resources[\'max_jobs\']} jobs; see build.log", flush=True)\n                    last_notice = time.monotonic()\n                time.sleep(0.5)\n        except BaseException:\n            if process.poll() is None:\n                process.kill()\n                process.wait()\n            progress(\'interrupted\')\n            raise\n        progress(\'compiled\' if process.returncode == 0 else \'failed\')\n    if process.returncode:\n        raise RuntimeError(f"Offline FlashAttention compilation failed. See {scratch / \'build.log\'}")\n    built = list(wheels.glob(\'flash_attn-*.whl\'))\n    if len(built) != 1:\n        raise RuntimeError(f\'Expected one native wheel: {built}\')\n    subprocess.run([str(python), \'-m\', \'pip\', \'--isolated\', \'install\', \'--no-index\', \'--no-deps\', \'--ignore-installed\', str(built[0])], env=env, check=True)\n    return {\'wheel\': built[0].name, \'sha256\': sha256(built[0]), \'sm\': f\'{major}{minor}\', \'cached\': False, \'wheel_path\': str(built[0]), \'max_jobs\': resources[\'max_jobs\'], \'nvcc_threads\': resources[\'nvcc_threads\'], \'build_seconds\': round(time.monotonic() - started, 2)}\n\ndef prepare_runtime(packages_dir, runtime_dir, model):\n    if model not in (\'difftsr\', \'unihir_restoration\'):\n        raise ValueError(model)\n    if sys.platform != \'linux\' or platform.machine() != \'x86_64\' or sys.version_info[:2] != (3, 13):\n        raise RuntimeError(\'This bundle targets the reported Kaggle Linux x86_64 / Python 3.13 kernel\')\n    learned = model.startswith((\'docentr_\', \'degan_\'))\n    document = model.startswith((\'docres_\', \'docdiff_\', \'restormer_\')) or learned\n    if model not in (\'difftsr\', \'unihir_restoration\') and (not document):\n        raise ValueError(model)\n    root = Path(packages_dir).resolve() / \'image_restoration\'\n    filename = \'learned_documents_runtime.json\' if learned else \'document_runtime.json\' if document else \'unihir_runtime.json\' if model == \'unihir_restoration\' else \'difftsr_runtime.json\'\n    plan = json.loads((root / \'manifests\' / filename).read_text(encoding=\'utf8\'))\n    validate_plan(root, plan)\n    for name, expected in plan[\'kernel_reuse\'].items():\n        actual = metadata.version(name)\n        if actual != expected:\n            raise RuntimeError(f\'Kernel changed: {name}={actual}; prepared against {expected}. Refresh the report/bundle instead of replacing kernel packages.\')\n    import torch\n    if torch.__version__ != plan[\'kernel_reuse\'][\'torch\']:\n        raise RuntimeError(\'Imported Torch differs from its metadata\')\n    if model == \'unihir_restoration\' and plan.get(\'native_sources\'):\n        identity = flash_cache_identity(torch, contained(root, plan[\'native_sources\'][0][\'path\']))\n        if not find_flash_cache(identity, flash_cache_roots(packages_dir)):\n            try:\n                export_flash_cache(packages_dir, runtime_dir)\n            except FileNotFoundError:\n                pass\n    key = hashlib.sha256(json.dumps({\'plan\': plan, \'python\': sys.version, \'executable\': sys.executable, \'torch\': torch.__version__, \'cuda\': torch.version.cuda, \'cxx11_abi\': torch._C._GLIBCXX_USE_CXX11_ABI, \'capability\': torch.cuda.get_device_capability() if torch.cuda.is_available() else None, \'bootstrap\': sha256(__file__)}, sort_keys=True).encode()).hexdigest()\n    folder = Path(runtime_dir).resolve() / \'environments\' / (model + \'_\' + key[:16])\n    if folder.is_relative_to(Path(\'/kaggle/input\')):\n        raise ValueError(\'Runtime must be writable, outside /kaggle/input\')\n    folder.mkdir(parents=True, exist_ok=True)\n    python = folder / \'bin/python\'\n    done = folder / \'install.json\'\n    env = offline_environment()\n    if learned:\n        env[\'KERAS_BACKEND\'] = \'torch\'\n    if not done.is_file():\n        subprocess.run([sys.executable, \'-m\', \'venv\', \'--without-pip\', \'--system-site-packages\', str(folder)], env=env, check=True)\n        wheels = [str(contained(root, record[\'path\'])) for record in plan[\'wheels\']]\n        if wheels:\n            subprocess.run([str(python), \'-m\', \'pip\', \'--isolated\', \'install\', \'--no-index\', \'--no-deps\', \'--ignore-installed\', *wheels], env=env, check=True)\n        native = [build_native(python, contained(root, record[\'path\']), folder, env, packages_dir=packages_dir) for record in plan.get(\'native_sources\', [])]\n        code = FLASH_CUDA_PROBE if native else \'import torch,numpy,cv2,einops,yaml; from skimage.filters import threshold_sauvola; assert torch.from_numpy(numpy.ones((2,2),dtype=numpy.float32)).shape==(2,2)\' if document and (not learned) else \'import torch,numpy,cv2,vit_pytorch,h5py,keras; assert keras.backend.backend()=="torch"\' if learned else \'import zhconv,linear_attention_transformer,axial_positional_embedding; import pytorch_lightning,omegaconf; assert zhconv.convert("abc","zh-cn")=="abc"\'\n        subprocess.run([str(python), \'-c\', code], env=env, check=True)\n        for source_record, native_record in zip(plan.get(\'native_sources\', []), native):\n            save_flash_cache(flash_cache_identity(torch, contained(root, source_record[\'path\'])), Path(native_record[\'wheel_path\']), flash_cache_output())\n        data = {\'key\': key, \'native\': native, \'kernel_modified\': False, \'weights_loaded\': False, \'model_inference_verified\': False}\n        temporary = done.with_suffix(\'.tmp\')\n        temporary.write_text(json.dumps(data, indent=2) + \'\\n\', encoding=\'utf8\')\n        temporary.replace(done)\n    return str(python)\n\ndef build_flash_package(packages_dir, runtime_dir, output_dir, *, allow_build=True):\n    """Standalone wheel preparation. No model Dataset, PDF, or weights are used."""\n    if sys.platform != \'linux\' or platform.machine() != \'x86_64\' or sys.version_info[:2] != (3, 13):\n        raise RuntimeError(\'Run this preparation on Kaggle Linux x86_64 / Python 3.13 with an L4 GPU\')\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a CUDA GPU before building FlashAttention\')\n    output = Path(output_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    root = Path(packages_dir) / \'image_restoration\'\n    plan = json.loads((root / \'manifests/unihir_runtime.json\').read_text(encoding=\'utf8\'))\n    validate_plan(root, plan)\n    if torch.__version__ != plan[\'kernel_reuse\'][\'torch\']:\n        raise RuntimeError(\'Torch differs from the prepared bundle; do not downgrade the kernel\')\n    if len(plan.get(\'native_sources\', [])) != 1:\n        raise ValueError(\'Expected one pinned FlashAttention source\')\n    source = contained(root, plan[\'native_sources\'][0][\'path\'])\n    identity = flash_cache_identity(torch, source)\n    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n    folder = Path(runtime_dir) / \'environments\' / (\'flash_attention_\' + key[:16])\n    cache = output / \'flash_attention_cache\'\n    previous = {name: os.environ.get(name) for name in (\'VTM_FLASH_CACHE_OUTPUT\', \'VTM_FLASH_REQUIRE_CACHE\')}\n    os.environ[\'VTM_FLASH_CACHE_OUTPUT\'] = str(cache)\n    os.environ[\'VTM_FLASH_REQUIRE_CACHE\'] = \'0\' if allow_build else \'1\'\n    started = time.monotonic()\n    report = dict(format=\'vtm-flash-attention-package-v1\', identity=identity, status=\'running\', model_weights_loaded=False, model_inference_verified=False, kernel_modified=False)\n    report_path = output / \'flash_attention_build_report.json\'\n    try:\n        if not find_flash_cache(identity, flash_cache_roots(packages_dir)):\n            try:\n                export_flash_cache(packages_dir, runtime_dir, cache)\n            except FileNotFoundError:\n                pass\n        python = folder / \'bin/python\'\n        env = offline_environment()\n        if not python.is_file():\n            folder.mkdir(parents=True, exist_ok=True)\n            subprocess.run([sys.executable, \'-m\', \'venv\', \'--without-pip\', \'--system-site-packages\', str(folder)], env=env, check=True)\n        native = build_native(python, source, folder, env, packages_dir=packages_dir)\n        subprocess.run([str(python), \'-c\', FLASH_CUDA_PROBE], env=env, check=True, timeout=120)\n        save_flash_cache(identity, Path(native[\'wheel_path\']), cache)\n        (folder / \'install.json\').write_text(json.dumps({\'native\': [native]}, indent=2) + \'\\n\', encoding=\'utf8\')\n        report.update(status=\'native_and_cuda_verified\', native=native, cuda_probe_verified=True, seconds=round(time.monotonic() - started, 3))\n        report_path.write_text(json.dumps(report, indent=2) + \'\\n\', encoding=\'utf8\')\n        wheel, receipt = find_flash_cache(identity, [cache])\n        archive = output / \'flash_attention_package.zip\'\n        temporary = archive.with_suffix(\'.zip.tmp\')\n        prefix = \'image_restoration/flash_attention_cache/\' + key + \'/\'\n        with zipfile.ZipFile(temporary, \'w\', compression=zipfile.ZIP_STORED) as package:\n            package.write(wheel, prefix + wheel.name)\n            package.write(wheel.parent / \'flash_wheel.json\', prefix + \'flash_wheel.json\')\n            package.write(report_path, prefix + \'build_report.json\')\n        with zipfile.ZipFile(temporary) as package:\n            if package.testzip() is not None:\n                raise ValueError(\'Exported package ZIP integrity failed\')\n        temporary.replace(archive)\n        return dict(report, archive=str(archive), archive_sha256=sha256(archive), install_under=\'kaggle_upload/packages\', cache_subdirectory=\'image_restoration/flash_attention_cache\')\n    except Exception as exc:\n        report.update(status=\'failed\', error=str(exc), seconds=round(time.monotonic() - started, 3))\n        report_path.write_text(json.dumps(report, indent=2) + \'\\n\', encoding=\'utf8\')\n        for name in (\'build.log\', \'build_config.json\', \'build_progress.json\'):\n            log = folder / \'native_build\' / name\n            if log.is_file():\n                shutil.copyfile(log, output / name)\n        raise\n    finally:\n        for name, value in previous.items():\n            if value is None:\n                os.environ.pop(name, None)\n            else:\n                os.environ[name] = value\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--packages-dir\', type=Path, required=True)\n    parser.add_argument(\'--runtime-dir\', type=Path, default=Path(\'/tmp/vtm_image_runtime\'))\n    parser.add_argument(\'--export-flash-cache\', action=\'store_true\', help=\'Export an existing completed runtime wheel, verify CUDA, then exit\')\n    parser.add_argument(\'--cache-output\', type=Path)\n    parser.add_argument(\'--models\', nargs=\'+\', choices=[\'difftsr\', \'unihir_restoration\', \'docentr_binarization\', \'degan_binarization\', \'degan_deblurring\', \'degan_watermark_removal\'], default=[\'difftsr\', \'unihir_restoration\'])\n    args = parser.parse_args(argv)\n    if args.export_flash_cache:\n        export_flash_cache(args.packages_dir, args.runtime_dir, args.cache_output)\n        return 0\n    result = {model: prepare_runtime(args.packages_dir, args.runtime_dir, model) for model in args.models}\n    print(\'RUNTIME_PYTHONS = \' + repr(result))\n    return 0\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'

# Pixel/inference-identical predecessor; only its validated completed cache can be reused.
STORAGE_COMPATIBLE_CODE_SIGNATURES = {'004106dde8519341f2ec8f8f024743e081c52c10fae16e79516a4410ea7c5f0e': ('41e8e659ad6567bb34735c0b0c702836866ef56b6ed652e2f444e462ed666bbe',)}


Dùng worker riêng cho TADiSR, DiffTSR và UniHIR; ảnh gốc không qua restoration.
Giữ provenance và OCR gốc. Output UniHIR là generative_reconstruction, cần kiểm tra với ảnh nguồn.


In [ ]:
from __future__ import annotations
import copy
import os, sys, json, time, hashlib, io, shutil, subprocess, tempfile, uuid
from pathlib import Path
from dataclasses import asdict
CLASSICAL = ('original',)
NEURAL = ('tadisr_x4', 'difftsr', 'unihir_restoration')
DOCUMENT_VARIANTS = {}
AUTO_EXCLUDED_VARIANTS = ('difftsr',)
DOCUMENT_SOURCE_RECORDS = {}
DOCUMENT_WEIGHT_RECORDS = {}
DOCUMENT_REVISIONS = {}
SUPPORTED = {'difftsr', 'original', 'tadisr_text_mask', 'tadisr_x4', 'unihir_restoration'}
PROVENANCE = {**dict.fromkeys(CLASSICAL, 'deterministic_transformation'), 'original': 'source_observation', **dict.fromkeys(NEURAL, 'neural_restoration'), 'tadisr_text_mask': 'neural_restoration', 'unihir_restoration': 'generative_reconstruction'}
_canonical_dataset_write = dataset_write
from threading import RLock
_prep_io_lock = RLock()
_storage_guard = None

def dataset_write(root, relative, data):
    with _prep_io_lock:
        global _storage_guard
        path = dataset_path(root, relative)
        guarded = _storage_guard is not None and Path(root).resolve() == _storage_guard['root']
        growth = preparation_storage_check(_storage_guard, path, len(data)) if guarded else 0
        try:
            ref = _canonical_dataset_write(root, relative, data)
        except OSError:
            temporary = path.with_name(path.name + '.tmp')
            if temporary.is_file():
                temporary.unlink()
            raise
        if guarded:
            _storage_guard['used'] += growth
        return ref
PREPARATION_FAILURE_STATES = {'failed', 'missing_assets', 'runtime_incompatible', 'inference_failed', 'timeout', 'resource_exceeded'}

def restoration_failure_state(error, *, phase='inference'):
    message = str(error).lower()
    if isinstance(error, (TimeoutError,)) or 'timeout' in message or 'timed out' in message:
        return 'timeout'
    if isinstance(error, MemoryError) or any((s in message for s in ('out of memory', 'exceeds max_', 'max_storage_bytes', 'storage reserve', 'storage budget', 'no space left', 'disk full'))):
        return 'resource_exceeded'
    if isinstance(error, FileNotFoundError):
        return 'missing_assets'
    return 'runtime_incompatible' if phase == 'preflight' else 'inference_failed'

def prep_normalize(config):
    validate_preparation_selection(config['variants'])
    cfg = json.loads(json.dumps(config))
    variants = cfg['variants']
    validate_applicability(cfg)
    for name, opts in cfg['variant_options'].items():
        if name in EXTRA_CLASSICAL:
            classical_parameters(name, opts)
    if 'unihir' in variants:
        variants[variants.index('unihir')] = 'unihir_restoration'
    if set(variants) - set(SUPPORTED) or len(set(variants)) != len(variants):
        raise ValueError('Unknown/duplicate variants')
    if 'original' not in variants:
        variants.insert(0, 'original')
    if 'tadisr_x4' in variants and 'tadisr_text_mask' not in variants:
        variants.append('tadisr_text_mask')
    if 'tadisr_text_mask' in variants and 'tadisr_x4' not in variants:
        raise ValueError('Mask requires tadisr_x4')
    if not cfg['crop_types'] or set(cfg['crop_types']) - {'page', 'group', 'column'}:
        raise ValueError('Invalid crop_types')
    for key in ('max_source_pixels', 'max_output_pixels', 'timeout_seconds', 'load_timeout_seconds'):
        if cfg[key] <= 0:
            raise ValueError(f'{key} must be positive')
    if cfg.get('max_storage_bytes') is not None and (type(cfg['max_storage_bytes']) is not int or cfg['max_storage_bytes'] <= 0):
        raise ValueError('MAX_STORAGE_BYTES must be None or a positive integer')
    cfg.setdefault('max_storage_bytes', None)
    cfg.setdefault('storage_reserve_bytes', 2 * 1024 ** 3)
    if type(cfg['storage_reserve_bytes']) is not int or cfg['storage_reserve_bytes'] < 0:
        raise ValueError('STORAGE_RESERVE_BYTES must be a nonnegative integer')
    source_policy = cfg.get('kernel_torch_source_policy', 'warn')
    if source_policy not in ('strict', 'warn'):
        raise ValueError('KERNEL_TORCH_SOURCE_POLICY must be strict or warn')
    cfg['kernel_torch_source_policy'] = source_policy
    for opts in cfg['model_options'].values():
        opts['kernel_torch_source_policy'] = source_policy
    for model in NEURAL:
        opts = cfg['model_options'].setdefault(model, {'tile': 512, 'overlap': 32, 'device': 'cuda', 'seed': 42})
        opts['kernel_torch_source_policy'] = source_policy
        if opts.get('tile', 1) <= 0 or not 0 <= opts.get('overlap', 0) < opts.get('tile', 1):
            raise ValueError(f'{model}: invalid tiling')
    tad = cfg['model_options']['tadisr_x4']
    if tad['tile'] % 32 or tad['overlap'] % 32:
        raise ValueError('TADiSR tile/overlap require multiples of 32')
    if cfg['batch_size'] != 1:
        raise ValueError('Only batch_size=1 has a validated memory contract')
    if not 1 <= cfg['model_options']['difftsr']['steps'] <= 200:
        raise ValueError('DiffTSR steps must be 1..200')
    if cfg['deskew'] or cfg['dewarp']:
        raise ValueError('Deskew/dewarp need a separately validated geometry profile; disabled')
    cfg.setdefault('gpu_ids', [0])
    cfg.setdefault('cpu_variant_workers', 2)
    if type(cfg['cpu_variant_workers']) is not int or cfg['cpu_variant_workers'] < 1:
        raise ValueError('CPU_VARIANT_WORKERS must be a positive integer')
    cfg.setdefault('png_compression_level', 1)
    if type(cfg['png_compression_level']) is not int or not 0 <= cfg['png_compression_level'] <= 9:
        raise ValueError('PNG_COMPRESSION_LEVEL must be an integer from 0 to 9')
    cfg.setdefault('adaptive_gpu_scheduling', True)
    cfg.setdefault('execution_mode', 'sequential')
    if cfg['execution_mode'] not in ('sequential', 'parallel_models'):
        raise ValueError('EXECUTION_MODE must be sequential or parallel_models')
    gpu_worker_environment(cfg['gpu_ids'])
    small = 0
    for model in NEURAL:
        opts = cfg['model_options'].setdefault(model, {'tile': 512, 'overlap': 32, 'device': 'cuda', 'seed': 42})
        opts.setdefault('gpu_ids', [cfg['gpu_ids'][small % len(cfg['gpu_ids'])]])
        if not set(opts['gpu_ids']) <= set(cfg['gpu_ids']):
            raise ValueError(f'{model}: GPU IDs outside GPU_IDS')
        gpu_worker_environment(opts['gpu_ids'])
        if model in cfg['variants'] and len(opts['gpu_ids']) == 1:
            small += 1
    return cfg

def prep_worker_environment():
    env = dict(os.environ)
    for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONUSERBASE'):
        env.pop(key, None)
    env.update(PYTHONNOUSERSITE='1', PYTHONUTF8='1', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DIFFUSERS_OFFLINE='1', HF_DATASETS_OFFLINE='1', HF_HUB_DISABLE_TELEMETRY='1', PIP_NO_INDEX='1', PIP_DISABLE_PIP_VERSION_CHECK='1', TOKENIZERS_PARALLELISM='false', KERAS_BACKEND='torch')
    return env

def prep_bootstrap(packages_dir, runtime_dir):
    """Install missing PDFium into a private import directory; never into kernel."""
    if Path('/kaggle/input').exists() and Path(runtime_dir).resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('RUNTIME_DIR must be writable and outside /kaggle/input')
    from importlib.util import find_spec
    import PIL, numpy, cv2
    cv2.cvtColor(numpy.ones((4, 4, 3), dtype=numpy.uint8), cv2.COLOR_RGB2GRAY)
    if find_spec('pypdfium2') is None:
        candidates = sorted(Path(packages_dir).rglob('pypdfium2-*.whl'))
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
        tags = set(sys_tags())
        candidates = [p for p in candidates if tags & parse_wheel_filename(p.name)[3]]
        if not candidates:
            raise FileNotFoundError(f'Missing compatible PDFium wheel below {packages_dir}; attach the offline packages Dataset')
        target = Path(runtime_dir) / 'base_lib'
        target.mkdir(parents=True, exist_ok=True)
        subprocess.run([sys.executable, '-m', 'pip', '--isolated', 'install', '--no-index', '--no-deps', '--target', str(target), str(candidates[-1])], check=True, env=prep_worker_environment())
        sys.path.insert(0, str(target))
    import pypdfium2
    return {'pillow': PIL.__version__, 'numpy': numpy.__version__, 'opencv': cv2.__version__, 'pdfium': str(pypdfium2.PDFIUM_INFO), 'python': sys.version}

def prep_vendor(runtime_dir, model):
    name = {'tadisr_x4': 'TADiSR', 'difftsr': 'DiffTSR', 'unihir_restoration': 'UniHIR'}.get(model)
    if not name:
        return None
    folder = Path(runtime_dir) / 'vendor' / name
    for relative, source in VENDOR_SOURCES[name].items():
        p = dataset_path(folder, relative)
        p.parent.mkdir(parents=True, exist_ok=True)
        if not p.exists() or p.read_text(encoding='utf8') != source:
            p.write_text(source, encoding='utf8')
    return str(folder.resolve())

def prep_kernel_torch_source():
    """Capture the notebook kernel's import origin without importing Torch or loading CUDA."""
    from importlib.util import find_spec
    spec = find_spec('torch')
    return str(Path(spec.origin).resolve()) if spec and spec.origin else None

def prep_log_tail(log, maximum=8000):
    try:
        with Path(log).open('rb') as stream:
            stream.seek(0, 2)
            stream.seek(max(0, stream.tell() - maximum))
            return stream.read().decode('utf8', errors='replace')
    except OSError:
        return '(worker log unavailable)'

def prep_check_manifest(model_dir, required):
    """Verify complete file list and hashes before GPU loading (not only checkpoint)."""
    root = Path(model_dir)
    manifest = root / 'assets.json'
    if not manifest.is_file():
        raise FileNotFoundError(f'Missing {manifest}; prepare/inventory assets first')
    data = json.loads(manifest.read_text(encoding='utf8'))
    indexed = {e['path']: e for e in data['files']}
    missing = []
    for relative in required:
        p = dataset_path(root, relative)
        if not p.is_file():
            missing.append(str(p))
            continue
        if relative not in indexed:
            raise ValueError(f'Asset not inventoried: {p}')
    if missing:
        raise FileNotFoundError('Missing assets:\n' + '\n'.join(missing))
    for relative, record in indexed.items():
        p = dataset_path(root, relative)
        if not p.is_file():
            raise FileNotFoundError(p)
        if p.stat().st_size != record['bytes'] or dataset_file_hash(p) != record['sha256']:
            raise ValueError(f'Asset checksum mismatch: {p}')
    return dataset_hash({'files': data['files']})

def prep_offline_prompt(config, runtime_dir):
    """Derive upstream's fixed prompt offline, before loading the restoration model."""
    folder = Path(config['models_dir']) / 'image_restoration/tadisr'
    if (folder / 'CogView4/saved_prompt_tokens_nocfg.pt').is_file():
        return None
    base_required = ['CogView4/' + name for name in ('model_index.json', 'scheduler/scheduler_config.json', 'tokenizer/tokenizer.json', 'tokenizer/tokenizer_config.json', 'tokenizer/special_tokens_map.json', 'text_encoder/config.json', 'text_encoder/model.safetensors.index.json', 'transformer/config.json', 'transformer/diffusion_pytorch_model.safetensors.index.json', 'vae/config.json', 'vae/diffusion_pytorch_model.safetensors')]
    for prefix, index in (('text_encoder', 'model.safetensors.index.json'), ('transformer', 'diffusion_pytorch_model.safetensors.index.json')):
        p = folder / 'CogView4' / prefix / index
        if not p.is_file():
            raise FileNotFoundError(p)
        base_required.extend((f'CogView4/{prefix}/{part}' for part in set(json.loads(p.read_text())['weight_map'].values())))
    identity = prep_check_manifest(folder, ['tadisr_cogview4_realce_17500.pkl', *base_required])
    key = dataset_hash({'assets': identity, 'upstream': UPSTREAM_REVISIONS['TADiSR'], 'prompt_contract': 'upstream_fixed_quality_nocfg_v1'})
    output = Path(runtime_dir) / 'derived_prompts' / key / 'saved_prompt_tokens_nocfg.pt'
    receipt = output.with_suffix('.json')
    if output.is_file() and receipt.is_file():
        cached = json.loads(receipt.read_text())
        if cached.get('sha256') == dataset_file_hash(output):
            return str(output)
        raise ValueError('Derived prompt cache corrupted; preserve it for review and choose a fresh runtime directory')
    python = prep_overlay(config['packages_dir'], runtime_dir, 'tadisr_x4', config)
    worker = PrepWorker(runtime_dir, python, 'tadisr_x4', {'base_model': str((folder / 'CogView4').resolve()), 'prompt_output': str(output.resolve())}, config['model_options']['tadisr_x4'], config['load_timeout_seconds'], action='prepare_prompt')
    try:
        report = worker.ready
    finally:
        worker.close()
    dataset_write(output.parent, receipt.name, dataset_json_bytes({'key': key, 'sha256': dataset_file_hash(output), 'worker': report}))
    return str(output)

def prep_classical(image, name, options):
    if name in EXTRA_CLASSICAL:
        return extra_classical_variant(image, name, options)
    from PIL import Image
    if name == 'clahe' and (not 0 < options.get('clip_limit', 1.5) <= 4 or not 2 <= options.get('tile_grid', 8) <= 32):
        raise ValueError('Conservative CLAHE requires clip_limit (0,4] and tile_grid [2,32]')
    if name in ('original', 'clahe', 'bicubic_x2'):
        return dataset_variant(image, name, options)
    if name == 'bicubic_x4':
        return (image.resize((image.width * 4, image.height * 4), Image.Resampling.BICUBIC), {'method': 'bicubic', 'parameters': {'scale': 4}, 'warnings': ['interpolation_does_not_recover_missing_strokes']})
    if name == 'denoised':
        import cv2, numpy as np
        params = {'h': 2.0, 'template_window': 7, 'search_window': 21, **options}
        if not 0 < params['h'] <= 5 or params['template_window'] % 2 != 1 or params['search_window'] % 2 != 1 or (not 3 <= params['template_window'] <= params['search_window'] <= 63):
            raise ValueError('Conservative NLM requires 0<h<=5 and odd windows')
        lab = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = cv2.fastNlMeansDenoising(lab[:, :, 0], None, float(params['h']), params['template_window'], params['search_window'])
        return (Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)), {'method': 'nlm_lab_l', 'parameters': params, 'warnings': ['may_remove_weak_strokes; opt_in_visual_review_required']})
    raise ValueError(name)

def prep_image_qc(source, output, name):
    import cv2, numpy as np
    from PIL import Image
    expected_scale = 2 if name == 'bicubic_x2' else 4 if name in ('bicubic_x4', 'realesrgan_x4', 'tadisr_x4', 'tadisr_text_mask') else 1
    if name != 'difftsr' and output.size != (source.width * expected_scale, source.height * expected_scale):
        raise ValueError(f'{name}: invalid output dimensions {output.size}')
    if name == 'difftsr' and abs(output.width / output.height - source.width / source.height) > 0.06:
        raise ValueError('DiffTSR geometry distortion exceeds rounding tolerance')
    reduced = output.convert('RGB').resize(source.size, Image.Resampling.BOX)
    a = cv2.cvtColor(np.asarray(source), cv2.COLOR_RGB2GRAY)
    b = cv2.cvtColor(np.asarray(reduced), cv2.COLOR_RGB2GRAY)

    def components(gray):
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        return int(cv2.connectedComponents(binary)[0] - 1)
    before, after = (components(a), components(b))
    return {'state': 'needs_visual_review' if name != 'original' else 'verified_pixel_identity', 'expected_scale': expected_scale if name != 'difftsr' else None, 'luminance_mae_at_source': float(np.abs(a.astype(float) - b.astype(float)).mean()), 'components_before': before, 'components_after': after, 'component_loss_warning': after < before * 0.9, 'glyph_fidelity': 'unverified', 'ocr_accuracy': 'not_measured'}

def prep_usage(root):
    return preparation_storage_usage(root)

def prep_store(root, image, budget):
    image = image.convert('RGB')
    stream = io.BytesIO()
    image.save(stream, format='PNG', compress_level=budget.get('png_compression_level', 1))
    data = stream.getvalue()
    sha = hashlib.sha256(data).hexdigest()
    relative = f'objects/{sha[:2]}/{sha}.png'
    pixel_hash = hashlib.sha256(image.tobytes()).hexdigest()
    with _prep_io_lock:
        p = dataset_path(root, relative)
        if not p.exists() or dataset_file_hash(p) != sha:
            dataset_write(root, relative, data)
    return {'path': relative, 'sha256': sha, 'size': list(image.size), 'pixel_sha256': pixel_hash}

def prep_receipt_valid(root, receipt, key):
    data = json.loads(dataset_path(root, receipt).read_bytes())
    if data['key'] != key or data['state'] != 'ok':
        raise ValueError('Incomplete/mismatched cache')
    dataset_load_image(root, data['image'])
    for extra in data.get('extras', {}).values():
        if 'pixel_sha256' in extra:
            dataset_load_image(root, extra)
        else:
            dataset_read(root, extra)
    return data

def prep_run(config, runtime_dir):
    global _storage_guard
    cfg = prep_normalize(config)
    root = Path(cfg['output_dir']).resolve()
    root.mkdir(parents=True, exist_ok=True)
    if Path('/kaggle/input').exists() and root.is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('Output must not be in /kaggle/input')
    for key in ('data_dir', 'models_dir', 'packages_dir'):
        if not cfg[key] or not Path(cfg[key]).is_dir():
            raise FileNotFoundError(f'Configure valid {key}: {cfg[key]}')
        if root == Path(cfg[key]).resolve() or root.is_relative_to(Path(cfg[key]).resolve()):
            raise ValueError('Output must be outside input assets')
    try:
        started = time.monotonic()
        _storage_guard = preparation_storage_budget(root, cfg['max_storage_bytes'], cfg['storage_reserve_bytes'])
        preparation_storage_check(_storage_guard, root / 'last_plan.json', 0)
        canonical_config = ImageDatasetConfig(variants=['original'], max_source_pixels=cfg['max_source_pixels'], **cfg['layout'])
        profile_dir = Path(runtime_dir) / 'profiles'
        profile_dir.mkdir(parents=True, exist_ok=True)
        for entry in list_pdfs(cfg['data_dir']):
            profile = cfg.get('book_layouts', {}).get(entry['stem'], {})
            (profile_dir / (entry['stem'] + '.json')).write_text(json.dumps(profile), encoding='utf8')
        if cfg.get('canonical_snapshot'):
            stats = {'dataset': cfg['canonical_snapshot']}
        else:
            stats = prepare_image_dataset(cfg['data_dir'], root, config=canonical_config, books=cfg['books'], pages=cfg['pages'], limit=cfg['limit'], max_books=cfg['max_books'], profile_dir=profile_dir, publish=False)
        _, base = dataset_open(stats['dataset'])
        cfg = plan_selective_restoration(root, base, cfg)
        if cfg.get('_selective_restoration_report'):
            dataset_write(root, 'selective_restoration_plan.json', dataset_json_bytes(cfg['_selective_restoration_report']))
            print('Selective restoration:', len(cfg['_selective_restoration_report']['selected']), 'unique source crops; see selective_restoration_plan.json')
        budget = _storage_guard
        budget['png_compression_level'] = cfg.get('png_compression_level', 1)
        preparation_storage_check(budget, root / 'last_plan.json', 0)
        plan = plan_restoration_variants(root, base, cfg, applicability=prep_applicability)
        live_plan = {**plan, 'storage_already_used_bytes': budget['used'], 'storage_limit_bytes': budget['limit'], 'storage_budget_risk_before_cache_deduplication': budget['used'] + plan['additional_storage_estimate_bytes'] > budget['limit']}
        dataset_write(root, 'last_plan.json', dataset_json_bytes(live_plan))
        print('Restoration plan:', json.dumps(live_plan, ensure_ascii=False))
        if plan['resource_warnings'] and (not cfg['allow_partial']):
            raise MemoryError('Restoration preflight exceeds max_pixel limits; explicitly select smaller regions or review limits; no downscale')
        if live_plan['storage_budget_risk_before_cache_deduplication']:
            print('Storage estimate exceeds budget before cache/deduplication; available space is checked at every write; a storage stop keeps completed cache. Review last_plan.json before a large run.')
        preflight = prep_preflight(cfg, runtime_dir)
        stage = Path(runtime_dir) / ('manifest_' + uuid.uuid4().hex + '.jsonl')
        stage.parent.mkdir(parents=True, exist_ok=True)
        counts = {}
        measurements = []
        reused = 0
        failures = []
        rows = 0

        def write_row(stream, row):
            nonlocal rows
            with _prep_io_lock:
                row['runtime_status'] = {'state': row['state'], 'inference_verified': row['state'] == 'ok', 'device': 'cpu' if row['variant'] in CLASSICAL else 'worker_report', 'verification_scope': 'this output only; not OCR accuracy or other devices'}
                row.setdefault('parameters', row.get('operation', {}).get('parameters', cfg['variant_options'].get(row['variant'], cfg['model_options'].get(row['variant'], {}))))
                stream.write(dataset_json_bytes(row))
                stream.flush()
                rows += 1
                counts[row['state']] = counts.get(row['state'], 0) + 1
        with dataset_lock(root), stage.open('wb') as manifest:
            shards = []
            receipt_locks = {}

            def run_variant(name, cancel_event, page_entries=None, assigned_gpu_ids=None, slot='main'):
                nonlocal reused
                shard = stage.with_name(stage.stem + '_' + name + '_' + slot + '.jsonl')
                with _prep_io_lock:
                    shards.append((name, shard))
                with shard.open('wb') as manifest:
                    worker = None
                    try:
                        for entry in base['pages'] if page_entries is None else page_entries:
                            record = json.loads(dataset_read(root, entry['record']))
                            for region in record['regions']:
                                if cancel_event.is_set():
                                    raise InterruptedError('Restoration cancelled')
                                original = region['images']['original']
                                x1, y1, x2, y2 = region['source_box']
                                common = {'book': entry['book'], 'page': entry['page'] + 1, 'page_index': entry['page'], 'source_sha256': entry['source_sha256'], 'source_pdf': entry['source_pdf'], 'source_image': record['source'], 'source_image_hash': record['source']['sha256'], 'source_box': region['source_box'], 'crop_type': region['crop_type'], 'region_id': region['id'], 'reading_order': region['reading_order'], 'canonical_image': original, 'variant': name, 'provenance': PROVENANCE[name], 'method_identity': variant_spec(name), 'input_pixel_sha256': original['pixel_sha256']}
                                state, reason = ('ready', None)
                                if region['crop_type'] not in cfg['crop_types']:
                                    state, reason = ('skipped_by_configuration', 'crop_type deselected')
                                elif cfg.get('region_ids') and region['id'] not in cfg['region_ids']:
                                    state, reason = ('skipped_by_configuration', 'region_id deselected')
                                else:
                                    state, reason = prep_applicability(name, entry['book'], entry['page'], region, cfg)
                                common['applicability'] = {'state': state, 'reason': reason, 'evidence': 'six_original_ocr_pairwise_rates_and_budget; triage_only' if '_restoration_selected_keys' in cfg else 'source_proxy_and_budget; not_verified_degradation' if name == 'unihir_restoration' and '_selective_unihir_keys' in cfg else 'explicit_configuration; no_automatic_degradation_classification'}
                                model_info = preflight.get(name, {})
                                if state == 'ready' and name in NEURAL and (model_info.get('state') != 'ready'):
                                    state, reason = (model_info.get('state', 'failed'), model_info.get('reason', 'Model not ready'))
                                if state != 'ready':
                                    row = {**common, 'state': state, 'reason': reason}
                                    write_row(manifest, row)
                                    if name == 'tadisr_x4':
                                        write_row(manifest, {**row, 'variant': 'tadisr_text_mask'})
                                    if state in PREPARATION_FAILURE_STATES:
                                        failures.append(row)
                                    continue
                                options = classical_parameters(name, cfg['variant_options'].get(name, {})) if name in EXTRA_CLASSICAL else cfg['model_options'].get(name, cfg['variant_options'].get(name, {}))
                                identity = {'model': model_info.get('checkpoint_identity'), 'checkpoint': model_info.get('checkpoint'), 'architecture': {'tadisr_x4': 'CogView4-RealCE with upstream joint segmentation decoder', 'difftsr': 'DiffTSR released IDM/TDM/MoM + TransOCR prior', 'unihir_restoration': 'UniHIR BAGEL/MLLM'}.get(name), 'vendor': model_info.get('vendor_identity'), 'runtime': model_info.get('probe'), 'document_task': globals().get('DOCUMENT_VARIANTS', {}).get(name)}
                                if identity['runtime']:
                                    identity['runtime'] = json.loads(json.dumps({k: v for k, v in identity['runtime'].items() if k != 'pid'}))
                                    for device in identity['runtime'].get('devices', []):
                                        device.pop('free_bytes', None)
                                key = dataset_hash({'source': original['sha256'], 'box': region['source_box'], 'name': name, 'options': options, 'model': identity, 'code': RESTORATION_CODE_SIGNATURE, 'opencv': __import__('cv2').__version__})
                                with _prep_io_lock:
                                    receipt_lock = receipt_locks.setdefault(key, __import__('threading').Lock())
                                with receipt_lock:
                                    receipt = f'restoration_cache/{key}.json'
                                    result = None
                                    if cfg['resume']:
                                        try:
                                            result = prep_receipt_valid(root, receipt, key)
                                            with _prep_io_lock:
                                                reused += 1
                                        except (OSError, ValueError, KeyError):
                                            pass
                                        if result is None:
                                            for previous_code in globals().get('STORAGE_COMPATIBLE_CODE_SIGNATURES', {}).get(RESTORATION_CODE_SIGNATURE, ()):
                                                previous_key = dataset_hash({'source': original['sha256'], 'box': region['source_box'], 'name': name, 'options': options, 'model': identity, 'code': previous_code, 'opencv': __import__('cv2').__version__})
                                                try:
                                                    result = prep_receipt_valid(root, f'restoration_cache/{previous_key}.json', previous_key)
                                                    result = {**result, 'key': key, 'reused_from_storage_compatible_code': previous_code}
                                                    with _prep_io_lock:
                                                        reused += 1
                                                    break
                                                except (OSError, ValueError, KeyError):
                                                    pass
                                    if result is None:
                                        tick = time.monotonic()
                                        try:
                                            source = dataset_load_image(root, original)
                                            scale = 4 if name in ('bicubic_x4', 'realesrgan_x4', 'tadisr_x4') else 2 if name == 'bicubic_x2' else 1
                                            if source.width * source.height * scale ** 2 > cfg['max_output_pixels']:
                                                raise ValueError(f'{name} output exceeds max_output_pixels; split crop or explicitly raise limit')
                                            extras = {}
                                            info = {}
                                            operation = {'method': name, 'parameters': options}
                                            if name in CLASSICAL:
                                                output, operation = prep_classical(source, name, options)
                                            else:
                                                if worker is None:
                                                    worker = PrepWorker(runtime_dir, model_info['python'], name, model_info['assets'], options if assigned_gpu_ids is None else {**options, 'gpu_ids': assigned_gpu_ids}, cfg['load_timeout_seconds'], cancel_event=cancel_event, diagnostics_dir=root / 'worker_diagnostics')
                                                if page_entries is not None and hasattr(page_entries, 'model_loaded'):
                                                    page_entries.model_loaded(worker.ready.get('load_seconds', 0))
                                                folder, info = worker.infer(dataset_path(root, original['path']), int(key[:8], 16), prep_inference_timeout(cfg, name))
                                                info['model_load_seconds'] = worker.ready.get('load_seconds')
                                                from PIL import Image
                                                with Image.open(folder / 'result.png') as loaded:
                                                    output = loaded.convert('RGB')
                                                for extra, relative in info['extras'].items():
                                                    p = dataset_path(folder, relative)
                                                    if p.suffix == '.png':
                                                        with Image.open(p) as im:
                                                            if im.width * im.height > cfg['max_output_pixels']:
                                                                raise ValueError('Extra output exceeds pixel budget')
                                                            extras[extra] = prep_store(root, im, budget)
                                                    else:
                                                        data = p.read_bytes()
                                                        h = hashlib.sha256(data).hexdigest()
                                                        extras[extra] = dataset_write(root, f'diagnostics/{h}.json', data)
                                            qc = prep_image_qc(source, output, name)
                                            if name == 'tadisr_x4' and 'tadisr_text_mask' not in extras:
                                                raise ValueError('TADiSR must emit both SR image and text mask')
                                            if 'tadisr_text_mask' in extras:
                                                mask = dataset_load_image(root, extras['tadisr_text_mask'])
                                                if mask.size != output.size:
                                                    raise ValueError('TADiSR mask and SR dimensions differ')
                                            image = original if name == 'original' else prep_store(root, output, budget)
                                            mapping = [(x2 - x1) / output.width, 0, 0, (y2 - y1) / output.height, x1, y1]
                                            result = {'key': key, 'state': 'ok', 'image': {**image, 'image_to_source': mapping}, 'extras': extras, 'operation': operation, 'model_identity': identity, 'qc': qc, 'seconds': round(time.monotonic() - tick, 6), 'coordinate_mapping_evidence': 'nominal_canvas_only; generated_content_unverified' if name == 'unihir_restoration' else 'crop_and_scale_geometry', 'worker': info, 'warnings': ['possible_added_removed_merged_strokes; inspect_against_original'] if name in NEURAL else operation.get('warnings', [])}
                                            dataset_write(root, receipt, dataset_json_bytes(result))
                                            if name in NEURAL:
                                                for p in folder.iterdir():
                                                    if p.is_file():
                                                        p.unlink()
                                                folder.rmdir()
                                        except Exception as exc:
                                            if is_preparation_storage_error(exc):
                                                _storage_guard['stop_reason'] = str(exc)
                                                cancel_event.set()
                                                raise PreparationStorageLimit(str(exc)) from exc
                                            if cancel_event.is_set():
                                                raise
                                            result = {'key': key, 'state': restoration_failure_state(exc), 'reason': str(exc), 'seconds': round(time.monotonic() - tick, 6)}
                                            dataset_write(root, receipt, dataset_json_bytes(result))
                                            failures.append({**common, **result})
                                            if worker is not None:
                                                worker.close()
                                                worker = None
                                row = {**common, **result}
                                write_row(manifest, row)
                                if result['state'] == 'ok':
                                    measurements.append({k: result[k] for k in ('seconds', 'worker')} | {'variant': name})
                                if name == 'tadisr_x4':
                                    mask = result.get('extras', {}).get('tadisr_text_mask')
                                    maskrow = {**row, 'variant': 'tadisr_text_mask'}
                                    if mask:
                                        maskrow['image'] = {**mask, 'image_to_source': row['image']['image_to_source']}
                                        maskrow['provenance'] = 'neural_restoration'
                                        maskrow['warnings'] = ['model_predicted_mask_is_not_source_character_evidence']
                                    elif result['state'] == 'ok':
                                        maskrow.update(state='inference_failed', reason='Missing paired TADiSR text mask')
                                    write_row(manifest, maskrow)
                    finally:
                        if worker is not None:
                            worker.close()
            names = [n for n in cfg['variants'] if n != 'tadisr_text_mask']
            if cfg['execution_mode'] == 'parallel_models':
                from concurrent.futures import ThreadPoolExecutor
                from threading import Event
                cancel = Event()
                classical = [n for n in names if n in CLASSICAL]
                small = [n for n in names if n not in CLASSICAL and len(cfg['model_options'][n]['gpu_ids']) == 1]
                heavy = [n for n in names if n not in CLASSICAL and len(cfg['model_options'][n]['gpu_ids']) > 1]

                def cpu_work():
                    from concurrent.futures import ThreadPoolExecutor, as_completed
                    with ThreadPoolExecutor(max_workers=cfg.get('cpu_variant_workers', 2)) as pool:
                        pending = [pool.submit(run_variant, name, cancel) for name in classical]
                        try:
                            for future in as_completed(pending):
                                future.result()
                        except BaseException:
                            cancel.set()
                            for future in pending:
                                future.cancel()
                            raise

                def gpu_work(name, pages, ids, slot, event):
                    print(f'[scheduler] GPU {ids} -> {name}', flush=True)
                    return run_variant(name, event, pages, ids, slot)
                with ThreadPoolExecutor(max_workers=1) as cpu_pool:
                    cpu_future = cpu_pool.submit(cpu_work)
                    try:
                        if cfg.get('adaptive_gpu_scheduling', True):
                            reserved = {n: (base['pages'], cfg['model_options'][n]['gpu_ids']) for n in sorted(heavy, key=lambda n: len(cfg['model_options'][n]['gpu_ids']))}
                            run_gpu_page_jobs({n: base['pages'] for n in small}, gpu_work, cfg['gpu_ids'], cancel_event=cancel, reserved_jobs=reserved)
                        else:
                            run_gpu_task_waves([(n, cfg['model_options'][n]['gpu_ids']) for n in small], run_variant, cfg['gpu_ids'])
                            run_gpu_task_waves([(n, cfg['model_options'][n]['gpu_ids']) for n in heavy], run_variant, cfg['gpu_ids'])
                        cpu_future.result()
                    except BaseException:
                        cancel.set()
                        raise
            else:
                from threading import Event
                cancel_event = Event()
                for name in names:
                    run_variant(name, cancel_event)
            from contextlib import ExitStack
            from heapq import merge
            page_order = {(e['book'], e['page']): i for i, e in enumerate(base['pages'])}

            def shard_rows(stream):
                for line in stream:
                    if line.strip():
                        yield json.loads(line)
            for name in names:
                paths = [p for n, p in shards if n == name]
                with ExitStack() as stack:
                    streams = [shard_rows(stack.enter_context(p.open('rb'))) for p in paths]
                    for row in merge(*streams, key=lambda r: page_order[r['book'], r['page_index']]):
                        manifest.write(dataset_json_bytes(row))
                for path in paths:
                    path.unlink()
            manifest.flush()
            os.fsync(manifest.fileno())
        data = stage.read_bytes()
        sha = hashlib.sha256(data).hexdigest()
        manifest_ref = dataset_write(root, f'restoration_manifests/{sha}.jsonl', data)
        snapshot_preflight = json.loads(json.dumps(preflight))
        for info in snapshot_preflight.values():
            if 'probe' in info:
                info['probe'].pop('pid', None)
        report = {'format': 'vtm-image-restoration', 'version': 1, 'canonical_snapshot': base['snapshot_id'], 'manifest': manifest_ref, 'selected_variants': cfg['variants'], 'crop_types': cfg['crop_types'], 'states': counts, 'partial': bool(failures), 'preflight': snapshot_preflight, 'provenance_categories': sorted(set(PROVENANCE.values())), 'configuration': cfg, 'code_sha256': RESTORATION_CODE_SIGNATURE, 'cost_plan': plan, 'not_integrated_methods': UNAVAILABLE_RESTORATION_METHODS}
        report_ref = dataset_write(root, f'restoration_manifests/{dataset_hash(report)}.json', dataset_json_bytes(report))
        prep_validate_extension(root, base, report)
        summary = {'canonical': stats, 'states': counts, 'manifest_rows': rows, 'variants_reused': reused, 'seconds': round(time.monotonic() - started, 3), 'storage_bytes': prep_usage(root), 'measurements': measurements, 'report': report_ref, 'failures': len(failures), 'cost_plan': plan, 'not_integrated_methods': UNAVAILABLE_RESTORATION_METHODS}
        failure_details = {}
        for row in failures:
            label = row['variant'] + ' / ' + row['state']
            detail = failure_details.setdefault(label, {'count': 0, 'first_reason': row.get('reason', '')})
            detail['count'] += 1
        summary['failure_details'] = failure_details
        if failure_details:
            print('Restoration failures:', json.dumps(failure_details, ensure_ascii=False, indent=2), flush=True)
        if failures and (not cfg['allow_partial']):
            summary['failure_diagnostics_archive'] = str(failure_bundle_path(root))
            dataset_write(root, 'last_attempt.json', dataset_json_bytes(summary))
            try:
                archive = package_failure_diagnostics(root, summary)
                print('Failure diagnostics ZIP:', archive, flush=True)
            except Exception as exc:
                print('Could not create failure diagnostics ZIP:', exc, flush=True)
            raise RuntimeError(f'{len(failures)} variants failed; completed cache retained; dataset.json not replaced. See last_attempt.json and failure_diagnostics_archive')
        final = {k: v for k, v in base.items() if k != 'snapshot_id'}
        final['restoration'] = {'format': 'vtm-image-restoration', 'version': 1, 'report': report_ref, 'partial': bool(failures)}
        final['snapshot_id'] = dataset_hash(final)
        with dataset_lock(root):
            dataset_write(root, f"snapshots/{final['snapshot_id']}.json", dataset_json_bytes(final))
            dataset_write(root, 'dataset.json', dataset_json_bytes(final))
        summary['snapshot_id'] = final['snapshot_id']
        summary['dataset'] = str(root / 'dataset.json')
        summary['completed'] = True
        summary['status'] = 'complete'
        dataset_write(root, 'last_run.json', dataset_json_bytes(summary))
        dataset_write(root, 'last_attempt.json', dataset_json_bytes(summary))
        return summary
    except Exception as exc:
        reason = (_storage_guard or {}).get('stop_reason')
        if not reason and (not is_preparation_storage_error(exc)):
            raise
        summary = {'status': 'incomplete_storage', 'completed': False, 'reason': reason or str(exc), 'dataset_directory': str(root), 'resume': True, 'storage_bytes': prep_usage(root), 'max_storage_bytes': cfg['max_storage_bytes'], 'storage_reserve_bytes': cfg.get('storage_reserve_bytes', 2 * 1024 ** 3), 'seconds': round(time.monotonic() - started, 3), 'next_step': 'Keep the complete _dataset directory. Free disk space or increase the explicit limit, then rerun with RESUME=True and the same selection.'}
        if 'base' in locals():
            summary['canonical_snapshot'] = base['snapshot_id']
        _storage_guard = None
        try:
            _canonical_dataset_write(root, 'last_attempt.json', dataset_json_bytes(summary))
        except OSError as checkpoint_error:
            summary['checkpoint_error'] = str(checkpoint_error)
        print('CHƯA HOÀN TẤT: chạm giới hạn dung lượng. Cache đã xong được giữ tại', root, flush=True)
        print('Giải phóng/thêm dung lượng hoặc chia lượt theo BOOKS/PAGES; giữ RESUME=True để chạy tiếp.', flush=True)
        return summary
    finally:
        _storage_guard = None

def prep_validate_extension(root, base, report):
    expected = {}
    for entry in base['pages']:
        record = json.loads(dataset_read(root, entry['record']))
        dataset_validate_page(root, record)
        for r in record['regions']:
            for name in report['selected_variants']:
                expected[entry['book'], entry['page'] + 1, r['id'], name] = r
    seen = set()
    states = {}
    for line in dataset_read(root, report['manifest']).splitlines():
        row = json.loads(line)
        key = (row['book'], row['page'], row['region_id'], row['variant'])
        if key not in expected or key in seen:
            raise ValueError('Unexpected/duplicate restoration manifest entry')
        seen.add(key)
        region = expected[key]
        states[row['state']] = states.get(row['state'], 0) + 1
        if row['provenance'] != PROVENANCE[row['variant']]:
            raise ValueError('Invalid evidence provenance')
        if row['source_box'] != region['source_box'] or row['reading_order'] != region['reading_order']:
            raise ValueError('Restoration crop geometry changed')
        if row['state'] == 'ok':
            image = dataset_load_image(root, row['image'])
            x1, y1, x2, y2 = region['source_box']
            scale = 2 if row['variant'] == 'bicubic_x2' else 4 if row['variant'] in ('bicubic_x4', 'realesrgan_x4', 'tadisr_x4', 'tadisr_text_mask') else 1
            if row['variant'] != 'difftsr' and image.size != ((x2 - x1) * scale, (y2 - y1) * scale):
                raise ValueError('Restoration variant dimensions violate scale contract')
            if row['variant'] == 'original' and row['image']['pixel_sha256'] != row['canonical_image']['pixel_sha256']:
                raise ValueError('Original variant differs from canonical crop pixels')
            if row['image']['image_to_source'] != [(x2 - x1) / image.width, 0, 0, (y2 - y1) / image.height, x1, y1]:
                raise ValueError('Invalid source mapping')
            for ref in row.get('extras', {}).values():
                dataset_read(root, ref)
        elif row['state'] not in {'not_applicable', 'skipped_by_configuration'} | PREPARATION_FAILURE_STATES:
            raise ValueError('Invalid variant state')
    if seen != set(expected):
        raise ValueError('Selected variants silently omitted')
    if states != report['states']:
        raise ValueError('Manifest state counts differ from report')
    if any((states.get(s, 0) for s in PREPARATION_FAILURE_STATES)) and (not report['partial']):
        raise ValueError('Incomplete selected methods falsely labelled complete')
    return {'rows': len(seen), 'validated': True, 'glyph_fidelity': 'unverified'}

def prep_open(directory):
    root, base = dataset_open(directory)
    extension = base.get('restoration')
    if not extension or extension['version'] != 1:
        raise ValueError('Missing/unsupported restoration extension')
    report = json.loads(dataset_read(root, extension['report']))
    return (root, base, report)

def prep_validate(directory):
    root, base, report = prep_open(directory)
    core = validate_image_dataset(directory)
    return {'canonical': core, 'restoration': prep_validate_extension(root, base, report), 'partial': report['partial']}

def prep_image_export_plan(directory, *, export_context=None):
    """Name successful snapshot images by method/book/page/crop, with portable paths."""
    import re, unicodedata
    root, descriptor, report = prep_open(directory) if export_context is None else export_context

    def book_slug(book):
        text = unicodedata.normalize('NFC', str(book))
        text = re.sub('[<>:"/\\\\|?*\\x00-\\x1f]', '_', text).strip(' .')[:64].rstrip(' .') or 'book'
        if re.fullmatch('(?i)(CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])', text.split('.')[0]):
            text = '_' + text
        return text
    books = {(p['book'], p['source_sha256']) for p in descriptor['pages']}
    groups = {}
    for identity in sorted(books):
        groups.setdefault(book_slug(identity[0]).casefold(), []).append(identity)
    names = {}
    for identities in groups.values():
        for identity in identities:
            suffix = '--' + dataset_hash(list(identity))[:8] if len(identities) > 1 else ''
            names[identity] = book_slug(identity[0]) + suffix
    entries = []
    seen = set()
    methods = dict.fromkeys(report.get('selected_variants', []), 0)
    skipped = {}
    export_rows = report.get('export_rows')
    rows = export_rows if export_rows is not None else (json.loads(line) for line in dataset_read(root, report['manifest']).splitlines() if line.strip())
    for row in rows:
        if row['state'] != 'ok':
            skipped[row['state']] = skipped.get(row['state'], 0) + 1
            continue
        method = row['variant']
        if method not in SUPPORTED:
            raise ValueError(f'Unknown image export method: {method}')
        page = row['page']
        order = row['reading_order']
        kind = row['crop_type']
        if not isinstance(page, int) or page < 1 or (not isinstance(order, int)) or (order < 0):
            raise ValueError('Image export requires 1-based pages and nonnegative crop order')
        if kind not in ('page', 'group', 'column'):
            raise ValueError(f'Unknown export crop type: {kind}')
        suffix = '' if kind == 'page' else f'-{kind}{order + 1:03d}'
        book = names[row['book'], row['source_sha256']]
        relative = f'{method}/{book}-p{page:05d}{suffix}.png'
        if relative.casefold() in seen:
            raise ValueError(f'Duplicate readable image name: {relative}')
        seen.add(relative.casefold())
        source = dataset_path(root, row['image']['path'])
        if source.suffix.lower() != '.png' or not source.is_file():
            raise ValueError(f'Missing PNG: {source}')
        entries.append({'name': relative, 'source': source, 'sha256': row['image']['sha256']})
        methods[method] = methods.get(method, 0) + 1
    if not entries:
        raise ValueError('No successful selected images to export')
    return (root, sorted(entries, key=lambda entry: entry['name']), {'files': len(entries), 'methods': dict(sorted(methods.items())), 'skipped': skipped, 'partial': report['partial'], 'snapshot_id': descriptor['snapshot_id']})

def prep_export_images(directory, destination, *, export_context=None):
    """Refresh only managed images; keep snapshot/cache files under the dataset root."""
    root, entries, summary = prep_image_export_plan(directory, export_context=export_context)
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    receipt_root = destination / '_dataset'
    receipt = f'exports/{dataset_hash(str(destination))}.json'
    try:
        previous = json.loads(dataset_path(receipt_root, receipt).read_text(encoding='utf8'))
    except FileNotFoundError:
        previous = {}
    if not isinstance(previous, dict):
        raise ValueError('Invalid image export receipt')
    owned = previous.get('files', {})
    if not isinstance(owned, dict):
        raise ValueError('Invalid image export file list')
    for relative in owned:
        parts = Path(relative).parts
        if len(parts) != 2 or parts[0] not in SUPPORTED or Path(relative).suffix.lower() != '.png':
            raise ValueError(f'Invalid managed image path: {relative}')
        dataset_path(destination, relative)
    for method in summary.get('methods', {}):
        if method not in SUPPORTED:
            raise ValueError('Unsupported image export method: ' + method)
        (destination / method).mkdir(parents=True, exist_ok=True)
    current = {entry['name']: entry['sha256'] for entry in entries}
    for entry in entries:
        if dataset_file_hash(entry['source']) != entry['sha256']:
            raise ValueError(f"Image checksum mismatch: {entry['source']}")
        target = dataset_path(destination, entry['name'])
        if target.exists():
            digest = dataset_file_hash(target)
            if digest not in (entry['sha256'], owned.get(entry['name'])):
                raise FileExistsError(f'Unmanaged or edited image already exists: {target}')
    for entry in entries:
        target = dataset_path(destination, entry['name'])
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and dataset_file_hash(target) == entry['sha256']:
            continue
        temporary = target.with_name('.' + target.name + '.' + uuid.uuid4().hex + '.tmp')
        try:
            try:
                os.link(entry['source'], temporary)
            except OSError:
                shutil.copyfile(entry['source'], temporary)
            temporary.replace(target)
        finally:
            if temporary.exists():
                temporary.unlink()
    for relative, digest in owned.items():
        if relative in current or summary.get('partial'):
            continue
        target = dataset_path(destination, relative)
        if target.is_file() and dataset_file_hash(target) == digest:
            target.unlink()
            try:
                target.parent.rmdir()
            except OSError:
                pass
    dataset_write(receipt_root, receipt, dataset_json_bytes({'destination': str(destination), 'files': {**owned, **current} if summary.get('partial') else current, 'partial': summary.get('partial', False)}))
    return {**summary, 'directory': str(destination)}

def prep_package_images(directory, archive, *, export_context=None):
    """Stream only final PNGs into method folders; replace ZIP only after success."""
    import zipfile
    root, entries, summary = prep_image_export_plan(directory, export_context=export_context)
    archive = Path(archive).absolute()
    if archive.suffix.lower() != '.zip':
        raise ValueError('Image archive must have a .zip extension')
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive.with_name('.' + archive.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        with zipfile.ZipFile(temporary, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as output:
            for entry in entries:
                info = zipfile.ZipInfo(entry['name'], date_time=(1980, 1, 1, 0, 0, 0))
                info.external_attr = 33188 << 16
                hasher = hashlib.sha256()
                with entry['source'].open('rb') as source, output.open(info, 'w', force_zip64=True) as target:
                    for block in iter(lambda: source.read(1024 * 1024), b''):
                        hasher.update(block)
                        target.write(block)
                if hasher.hexdigest() != entry['sha256']:
                    raise ValueError(f"Image checksum mismatch: {entry['source']}")
        temporary.replace(archive)
    finally:
        if temporary.exists():
            temporary.unlink()
    return {**summary, 'archive': str(archive), 'bytes': archive.stat().st_size}

def prep_previews(directory, maximum=3, detail_box=(0.1, 0.15, 0.4, 0.45)):
    from PIL import Image, ImageDraw
    root, base, report = prep_open(directory)
    chosen = {}
    for line in dataset_read(root, report['manifest']).splitlines():
        row = json.loads(line)
        key = (row['book'], row['page'], row['region_id'])
        if key not in chosen and len(chosen) < maximum and (row['crop_type'] in ('group', 'page')):
            chosen[key] = []
        if key in chosen:
            chosen[key].append(row)
    outputs = []
    if detail_box and (len(detail_box) != 4 or not 0 <= detail_box[0] < detail_box[2] <= 1 or (not 0 <= detail_box[1] < detail_box[3] <= 1)):
        raise ValueError('Preview detail_box requires fractional x1,y1,x2,y2 in [0,1]')
    for key, rows in chosen.items():
        cell_w, cell_h = (300, 350)
        columns = min(4, len(rows))
        height = (len(rows) + columns - 1) // columns * cell_h
        sheet = Image.new('RGB', (columns * cell_w, height), '#eeeeee')
        draw = ImageDraw.Draw(sheet)
        for i, row in enumerate(rows):
            left, top = (i % columns * cell_w, i // columns * cell_h)
            draw.text((left + 8, top + 5), row['variant'] + ' / ' + row['state'], fill='black')
            if row['state'] == 'ok':
                image = dataset_load_image(root, row['image'])
                image.thumbnail((cell_w - 16, cell_h - 50))
                sheet.paste(image, (left + 8, top + 30))
            else:
                reason = row.get('reason', '')
                draw.multiline_text((left + 8, top + 35), '\n'.join((reason[j:j + 38] for j in range(0, min(240, len(reason)), 38))), fill='black')
        path = root / 'previews' / f'{key[0]}_p{key[1]}_{key[2]}.png'
        path.parent.mkdir(exist_ok=True)
        tmp = path.with_suffix('.tmp')
        sheet.save(tmp, format='PNG')
        tmp.replace(path)
        outputs.append(str(path))
        if detail_box:
            detail = Image.new('RGB', (columns * 400, (len(rows) + columns - 1) // columns * 420), '#eeeeee')
            draw_detail = ImageDraw.Draw(detail)
            for i, row in enumerate(rows):
                left, top = (i % columns * 400, i // columns * 420)
                draw_detail.text((left + 8, top + 5), row['variant'] + ' / source-aligned detail', fill='black')
                if row['state'] == 'ok':
                    image = dataset_load_image(root, row['image'])
                    box = [round(detail_box[j] * image.size[j % 2]) for j in range(4)]
                    box[0] = min(box[0], image.width - 1)
                    box[1] = min(box[1], image.height - 1)
                    box[2] = min(image.width, max(box[0] + 1, box[2]))
                    box[3] = min(image.height, max(box[1] + 1, box[3]))
                    patch = image.crop(box)
                    factor = min(384 / patch.width, 380 / patch.height)
                    patch = patch.resize((max(1, round(patch.width * factor)), max(1, round(patch.height * factor))), Image.Resampling.NEAREST if row['variant'] == 'original' else Image.Resampling.LANCZOS)
                    detail.paste(patch, (left + 8, top + 30))
                else:
                    draw_detail.text((left + 8, top + 30), row['state'], fill='black')
            path = path.with_name(path.stem + '_detail.png')
            tmp = path.with_suffix('.tmp')
            detail.save(tmp, format='PNG')
            tmp.replace(path)
            outputs.append(str(path))
    return outputs

def prep_inference_timeout(config, model):
    import math
    seconds = config['model_options'].get(model, {}).get('inference_timeout_seconds', config['timeout_seconds'])
    if isinstance(seconds, bool) or not isinstance(seconds, (int, float)) or (not math.isfinite(seconds)) or (seconds <= 0):
        raise ValueError('inference_timeout_seconds must be a finite positive number')
    options = config['model_options'].get(model, {})
    hard = options.get('inference_hard_timeout_seconds', seconds)
    stall = options.get('progress_stall_seconds', 300)
    if any((isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (v <= 0) for v in (hard, stall))) or hard < seconds:
        raise ValueError('Invalid inference hard timeout or progress stall limit')
    return seconds
import hashlib
import json
import shutil
import zipfile

def preserve_worker_diagnostics(worker, progress, destination, max_bytes=64 * 1024 ** 2):
    worker = Path(worker).resolve()
    progress = Path(progress).resolve()
    destination = Path(destination) / worker.name
    destination.mkdir(parents=True, exist_ok=True)
    candidates = [worker / name for name in ('worker.log', 'worker_config.json', 'ready.json')]
    if progress.is_relative_to(worker) and progress.is_dir():
        candidates.extend(sorted(progress.rglob('*')))
    copied = []
    skipped = []
    used = 0
    for source in candidates:
        if not source.is_file() or not source.resolve().is_relative_to(worker):
            continue
        relative = source.relative_to(worker)
        if used + source.stat().st_size > max_bytes:
            skipped.append(dict(path=str(relative), reason='diagnostic byte budget exceeded'))
            continue
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
        used += target.stat().st_size
        copied.append(str(relative))
    record = dict(format='vtm-worker-diagnostics-v1', files=copied, skipped=skipped, bytes=used, captured_after_worker_failure=True, intermediate_images_are_final=False)
    (destination / 'diagnostic_manifest.json').write_text(json.dumps(record, ensure_ascii=False, indent=2) + '\n', encoding='utf8')
    return str(destination)

def failure_bundle_path(root):
    root = Path(root).resolve()
    output = root.parent if root.name == '_dataset' else root
    return output.with_name(output.name + '_failure_diagnostics.zip')

def package_failure_diagnostics(root, summary):
    root = Path(root).resolve()
    archive = failure_bundle_path(root)

    def read(reference):
        path = (root / reference['path']).resolve()
        if not path.is_relative_to(root):
            raise ValueError('Diagnostic reference escapes dataset')
        data = path.read_bytes()
        if hashlib.sha256(data).hexdigest() != reference['sha256']:
            raise ValueError('Diagnostic reference checksum mismatch')
        return data
    report_bytes = read(summary['report'])
    report = json.loads(report_bytes)
    temporary = archive.with_suffix('.zip.tmp')
    try:
        with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED) as output:
            output.writestr('last_attempt.json', json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
            output.writestr('restoration_report.json', report_bytes)
            output.writestr('restoration_manifest.jsonl', read(report['manifest']))
            for name in ('preflight.json', 'last_plan.json'):
                source = root / name
                if source.is_file():
                    output.write(source, name)
            folder = root / 'worker_diagnostics'
            if folder.is_dir():
                for source in sorted(folder.rglob('*')):
                    if source.is_file() and source.resolve().is_relative_to(root):
                        output.write(source, source.relative_to(root).as_posix())
        temporary.replace(archive)
    finally:
        if temporary.exists():
            temporary.unlink()
    return str(archive)
import errno

class PreparationStorageLimit(RuntimeError):
    """The requested run needs more space; completed objects remain reusable."""

def preparation_storage_usage(root):
    seen = set()
    total = 0
    for path in Path(root).rglob('*'):
        if path.is_file():
            stat = path.stat()
            identity = (stat.st_dev, stat.st_ino) if stat.st_ino else str(path.resolve())
            if identity not in seen:
                seen.add(identity)
                total += stat.st_size
    return total

def preparation_storage_budget(root, max_bytes=None, reserve_bytes=2 * 1024 ** 3):
    root = Path(root).resolve()
    root.mkdir(parents=True, exist_ok=True)
    if max_bytes is not None and (type(max_bytes) is not int or max_bytes <= 0):
        raise ValueError('MAX_STORAGE_BYTES must be None or a positive integer')
    if type(reserve_bytes) is not int or reserve_bytes < 0:
        raise ValueError('STORAGE_RESERVE_BYTES must be a nonnegative integer')
    used = preparation_storage_usage(root)
    free = shutil.disk_usage(root).free
    return dict(root=root, used=used, limit=used + max(0, free - reserve_bytes) if max_bytes is None else max_bytes, reserve_bytes=reserve_bytes, automatic=max_bytes is None)

def preparation_storage_check(budget, path, byte_count):
    """Check both net dataset growth and the space needed by the atomic temporary."""
    if budget.get('stop_reason'):
        raise PreparationStorageLimit(budget['stop_reason'])
    path = Path(path)
    previous = path.stat().st_size if path.is_file() else 0
    growth = byte_count - previous
    free = shutil.disk_usage(budget['root']).free
    reason = None
    if budget['used'] + growth > budget['limit']:
        reason = f"max_storage_bytes reached: used={budget['used']}, additional={max(0, growth)}, limit={budget['limit']}"
    elif free < byte_count + budget['reserve_bytes']:
        reason = f"Storage reserve reached: free={free}, atomic_write={byte_count}, reserve={budget['reserve_bytes']}"
    if reason:
        budget['stop_reason'] = reason
        raise PreparationStorageLimit(reason)
    return growth

def is_preparation_storage_error(error):
    if isinstance(error, PreparationStorageLimit):
        return True
    if isinstance(error, OSError) and error.errno in (errno.ENOSPC, errno.EDQUOT):
        return True
    message = str(error).lower()
    return any((text in message for text in ('max_storage_bytes', 'no space left on device', 'disk full', 'disk quota exceeded', 'storage reserve reached')))

def prep_wait(process, response, timeout, log, cancel_event=None, progress=None, hard_timeout=None, stall_seconds=300, dump_stack=False):
    import math, signal
    hard_timeout = timeout if hard_timeout is None else hard_timeout
    if any((isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (v <= 0) for v in (timeout, hard_timeout, stall_seconds))) or hard_timeout < timeout:
        raise ValueError('Worker timeout limits must be positive, finite, and hard_timeout >= timeout')
    started = time.monotonic()
    response = Path(response)
    activity = None
    last_activity = started
    last_notice = started
    state = {}
    while not response.exists():
        if cancel_event is not None and cancel_event.is_set():
            raise InterruptedError('Restoration cancelled')
        if process.poll() is not None:
            if response.exists():
                break
            raise RuntimeError(f'Worker exit {process.returncode}; see {log}\n' + prep_log_tail(log))
        now = time.monotonic()
        elapsed = now - started
        if progress is not None:
            try:
                current = json.loads(Path(progress).read_text(encoding='utf8'))
                counter = current.get('activity') if isinstance(current, dict) else None
                if isinstance(counter, int) and (not isinstance(counter, bool)) and (counter >= 0):
                    state = current
                    if counter != activity:
                        activity = counter
                        last_activity = now
            except (OSError, ValueError):
                pass
            if now - last_notice >= 30:
                print(f"UniHIR: {elapsed:.0f}s; stage={state.get('stage', 'starting')}; forwards={state.get('model_forwards', 0)}; text_tokens={state.get('text_tokens', 0)}", flush=True)
                last_notice = now
        reason = None
        if elapsed > hard_timeout:
            reason = f'hard limit {hard_timeout}s'
        elif elapsed > timeout and (activity is None or now - last_activity > stall_seconds):
            reason = f'{timeout}s and no recent compute progress'
        if reason:
            if response.exists():
                break
            if dump_stack and hasattr(signal, 'SIGUSR1'):
                try:
                    process.send_signal(signal.SIGUSR1)
                    time.sleep(0.2)
                except (OSError, ProcessLookupError):
                    pass
            if response.exists():
                break
            if process.poll() is None:
                process.kill()
            process.wait()
            raise TimeoutError(f'Worker exceeded {reason}; see {log}\nLast progress: ' + json.dumps(state, ensure_ascii=False) + '\nWorker log tail:\n' + prep_log_tail(log))
        time.sleep(0.1)
    data = json.loads(response.read_text(encoding='utf8'))
    if data['state'] == 'failed':
        context = json.dumps(data.get('runtime_context', {}), ensure_ascii=False)
        raise RuntimeError(f"{data['error']}\nRuntime context: {context}\nWorker log: {log}")
    return data

class PrepWorker:

    def __init__(self, runtime_dir, python, model, assets, options, timeout, action='run', cancel_event=None, diagnostics_dir=None):
        self.diagnostics_dir = diagnostics_dir
        self.cancel_event = cancel_event
        options = {**options, 'kernel_torch_source_path': prep_kernel_torch_source()}
        self.folder = Path(runtime_dir) / 'jobs' / uuid.uuid4().hex
        self.folder.mkdir(parents=True)
        self.log = self.folder / 'worker.log'
        self.stream = self.log.open('w', encoding='utf8')
        self.counter = 0
        vendor = assets.get('vendor') or prep_vendor(runtime_dir, model)
        if vendor:
            assets = {**assets, 'vendor': vendor}
        if model == 'unihir_restoration' and 'font_dir' in assets:
            font = Path(vendor) / 'font'
            font.mkdir(parents=True, exist_ok=True)
            for source in Path(assets['font_dir']).iterdir():
                if source.is_file():
                    target = font / source.name
                    if not target.is_file() or dataset_file_hash(target) != dataset_file_hash(source):
                        temporary = target.with_suffix(target.suffix + '.tmp')
                        shutil.copyfile(source, temporary)
                        temporary.replace(target)
        self.config = {'model': model, 'assets': assets, 'options': options, 'action': action, 'vendor': vendor, 'ready': str((self.folder / 'ready.json').resolve())}
        spec = self.folder / 'worker_config.json'
        spec.write_text(json.dumps(self.config), encoding='utf8')
        worker = Path(runtime_dir) / ('image_worker_' + self.folder.name + '.py')
        worker.write_text(WORKER_SOURCE, encoding='utf8')
        try:
            self.process = subprocess.Popen([str(python), '-u', str(worker.resolve()), str(spec.resolve())], stdin=subprocess.PIPE, stdout=self.stream, stderr=subprocess.STDOUT, text=True, encoding='utf8', env=gpu_worker_environment(options.get('gpu_ids', [0]), prep_worker_environment()), cwd=vendor or self.folder, creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0))
            self.ready = prep_wait(self.process, self.config['ready'], timeout, self.log, self.cancel_event)
        except BaseException:
            self.close()
            if self.diagnostics_dir is not None:
                try:
                    saved = preserve_worker_diagnostics(self.folder, self.folder / 'load_progress', self.diagnostics_dir)
                    print('Worker load diagnostics saved:', saved, flush=True)
                except Exception as exc:
                    print('Could not export load diagnostics:', exc, flush=True)
            raise

    def infer(self, input_path, seed, timeout):
        self.counter += 1
        folder = self.folder / f'region_{self.counter:07d}'
        folder.mkdir()
        response = (folder / 'response.json').resolve()
        job = {'input': str(Path(input_path).resolve()), 'output': str(folder.resolve()), 'seed': seed, 'response': str(response)}
        self.process.stdin.write(json.dumps(job) + '\n')
        self.process.stdin.flush()
        progress = self.folder / ('progress_' + folder.name)
        options = self.config['options']
        unihir = self.config['model'] == 'unihir_restoration'
        try:
            data = prep_wait(self.process, response, timeout, self.log, self.cancel_event, progress=progress / 'progress.json' if unihir else None, hard_timeout=options.get('inference_hard_timeout_seconds', timeout) if unihir else timeout, stall_seconds=options.get('progress_stall_seconds', 300), dump_stack=self.ready.get('stack_dump_enabled', False))
            return (folder, data)
        except BaseException:
            if self.diagnostics_dir is not None:
                try:
                    saved = preserve_worker_diagnostics(self.folder, progress, self.diagnostics_dir)
                    print('Worker diagnostics saved:', saved, flush=True)
                except Exception as exc:
                    print('Could not export worker diagnostics:', exc, flush=True)
            raise

    def close(self):
        if hasattr(self, 'process'):
            if self.process.stdin:
                try:
                    self.process.stdin.close()
                except OSError:
                    pass
            try:
                self.process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait()
        self.stream.close()

def prep_overlay(packages_dir, runtime_dir, model, config):
    """A separate venv per model, local wheels only, kernel files untouched."""
    override = config['runtime_pythons'].get(model)
    if override:
        if not Path(override).is_file():
            raise FileNotFoundError(override)
        return str(Path(override).absolute())
    if model in ('difftsr', 'unihir_restoration'):
        import importlib.util
        filename = 'unihir_runtime.json' if model == 'unihir_restoration' else 'difftsr_runtime.json'
        root = Path(packages_dir) / 'image_restoration'
        plan = json.loads((root / 'manifests' / filename).read_text(encoding='utf8'))
        if 'bootstrap' not in plan:
            raise RuntimeError('Missing prepared runtime bundle; run tools/prepare_kaggle_offline.py --download-small locally')
        record = plan['bootstrap']
        helper = dataset_path(root, record['path'])
        if helper.stat().st_size != record['bytes'] or dataset_file_hash(helper) != record['sha256']:
            raise ValueError(f'Runtime bootstrap checksum mismatch: {helper}')
        if model == 'unihir_restoration':
            module = prep_flash_runtime(runtime_dir)
            previous = os.environ.get('VTM_FLASH_REQUIRE_CACHE')
            if config.get('require_flash_cache', False):
                os.environ['VTM_FLASH_REQUIRE_CACHE'] = '1'
            try:
                return module.prepare_runtime(packages_dir, runtime_dir, model)
            finally:
                if previous is None:
                    os.environ.pop('VTM_FLASH_REQUIRE_CACHE', None)
                else:
                    os.environ['VTM_FLASH_REQUIRE_CACHE'] = previous
        spec = importlib.util.spec_from_file_location('vtm_restoration_runtime', helper)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module.prepare_runtime(packages_dir, runtime_dir, model)
    manifest = Path(packages_dir) / 'image_restoration/manifests/tadisr_runtime.json'
    if not manifest.is_file():
        raise FileNotFoundError(f'Missing {manifest}; attach the offline models Dataset')
    plan = json.loads(manifest.read_text(encoding='utf8'))
    wheel_root = manifest.parent.parent
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    tags = set(sys_tags())
    wheels = []
    for record in plan['wheels']:
        wheel = dataset_path(wheel_root, record['path'])
        if not wheel.is_file():
            raise FileNotFoundError(wheel)
        if dataset_file_hash(wheel) != record['sha256']:
            raise ValueError(f'Wheel checksum mismatch: {wheel}')
        if not tags & parse_wheel_filename(wheel.name)[3]:
            raise RuntimeError(f'Incompatible wheel ABI: {wheel.name}')
        wheels.append(str(wheel.resolve()))
    key = dataset_hash({'plan': plan, 'python': sys.version, 'prefix': sys.prefix, 'executable': sys.executable})
    folder = Path(runtime_dir) / 'environments' / ('tadisr_' + key[:16])
    python = folder / 'bin/python'
    done = folder / 'install.json'
    try:
        installed = json.loads(done.read_text(encoding='utf8'))
    except (OSError, ValueError):
        installed = {}
    if not isinstance(installed, dict):
        installed = {}
    if installed.get('key') != key or not python.is_file() or (not (folder / 'pyvenv.cfg').is_file()):
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(folder)], check=True, env=prep_worker_environment())
        if wheels:
            subprocess.run([str(python), '-m', 'pip', '--isolated', 'install', '--ignore-installed', '--no-index', '--no-deps', *wheels], check=True, env=prep_worker_environment(), stdout=subprocess.DEVNULL)
        dataset_write(folder, done.name, dataset_json_bytes({'key': key}))
    return str(python.absolute())

def prep_assets(model, models_dir, prompt_override=None):
    if model not in ('tadisr_x4', 'difftsr', 'unihir_restoration'):
        raise ValueError(model)
    root = Path(models_dir) / 'image_restoration'
    if model == 'tadisr_x4':
        folder = root / 'tadisr'
        base = folder / 'CogView4'
        required = ['tadisr_cogview4_realce_17500.pkl', 'CogView4/model_index.json', 'CogView4/vae/config.json', 'CogView4/vae/diffusion_pytorch_model.safetensors', 'CogView4/transformer/config.json', 'CogView4/transformer/diffusion_pytorch_model.safetensors.index.json', 'CogView4/text_encoder/config.json', 'CogView4/text_encoder/model.safetensors.index.json', 'CogView4/tokenizer/tokenizer.json', 'CogView4/tokenizer/tokenizer_config.json', 'CogView4/tokenizer/special_tokens_map.json', 'CogView4/scheduler/scheduler_config.json']
        if prompt_override is None:
            required.append('CogView4/saved_prompt_tokens_nocfg.pt')
        for name in ('transformer/diffusion_pytorch_model.safetensors.index.json', 'text_encoder/model.safetensors.index.json'):
            p = base / name
            if p.is_file():
                required.extend(('CogView4/' + str(Path(name).parent / part).replace('\\', '/') for part in sorted(set(json.loads(p.read_text())['weight_map'].values()))))
        paths = {'checkpoint': str((folder / required[0]).resolve()), 'base_model': str(base.resolve()), 'prompt_embeddings': str(Path(prompt_override).resolve()) if prompt_override else str((base / 'saved_prompt_tokens_nocfg.pt').resolve())}
    elif model == 'difftsr':
        folder = root / 'difftsr'
        required = ['DiffTSR.ckpt', 'transocr.pth']
        paths = {'checkpoint': str((folder / required[0]).resolve()), 'transocr': str((folder / required[1]).resolve())}
    else:
        folder = root / 'unihir'
        required = ['model.safetensors', 'ae.safetensors', 'llm_config.json', 'vit_config.json', 'tokenizer_config.json', 'tokenizer.json', 'vocab.json', 'merges.txt']
        paths = {'base_model': str(folder.resolve()), 'font_dir': str((folder / 'font').resolve())}
        required.extend(('font/' + name for name in ('KaiXinSongA.ttf', 'KaiXinSongB.ttf', 'TH-Tshyn-P0.ttf', 'TH-Tshyn-P1.ttf', 'TH-Tshyn-P16.ttf', 'TH-Tshyn-P2.ttf')))
    identity = prep_check_manifest(folder, required)
    if model == 'tadisr_x4' and dataset_file_hash(paths['checkpoint']) != '7cec8d0917305037cbafcb1b1bf81062add1613b561f5d7e2d1e599fe0baf5e0':
        raise ValueError('TADiSR adapter differs from official release checksum')
    if prompt_override:
        identity = dataset_hash({'base_assets': identity, 'derived_prompt_sha256': dataset_file_hash(prompt_override)})
    return (paths, identity)

def prep_applicability(model, book, page, region, config, size=None):
    state, reason = selective_region_applicability(model, book, page, region, config)
    if state != 'ready':
        return (state, reason)
    state, reason = preparation_applicability(model, book, page, region, config)
    if state != 'ready':
        return (state, reason)
    key = f"{book}:{page + 1}:{region['id']}"
    review = preparation_region_review(book, page, region['id'], config)
    if model in review.get('excluded_variants', []):
        return ('not_applicable', 'variant explicitly excluded for this reviewed region')
    if model == 'difftsr':
        if not review.get('upright_horizontal_single_line') or not 1 <= review.get('character_count', 0) <= 24:
            return ('not_applicable', 'Requires reviewed upright single horizontal line with 1..24 characters; no vertical rotation/reflow')
        if not review.get('allow_simplified_text_prior', False):
            return ('not_applicable', 'Upstream Text_Prediction uses zhconv zh-cn; traditional-form fidelity not established')
        box = region['source_box']
        ratio = (box[2] - box[0]) / (box[3] - box[1])
        if not 1 <= ratio <= 8:
            return ('not_applicable', 'DiffTSR requires 1 <= width/height <= 8')
        if abs(ratio / 4 - 1) > 0.05:
            return ('not_applicable', 'No validated aspect-preserving 512x128 contract for this crop (require near 4:1)')
    return ('ready', None)

def prep_preflight(config, runtime_dir):
    cfg = prep_normalize(config)
    report = {}
    failures = []
    print('Methods without runnable released resources:', json.dumps(UNAVAILABLE_RESTORATION_METHODS, ensure_ascii=False))
    if Path('/kaggle/input').exists() and Path(runtime_dir).resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('Runtime artifacts must be outside /kaggle/input')
    for model in NEURAL:
        if model not in cfg['variants']:
            continue
        if model == 'difftsr':
            if not any((r.get('upright_horizontal_single_line') for r in cfg['reviewed_regions'].values())):
                report[model] = {'state': 'not_applicable', 'reason': 'No reviewed input matches released model contract'}
                continue
        try:
            prompt_override = None
            if model == 'tadisr_x4' and cfg.get('generate_tadisr_prompt_offline', False):
                prompt_override = prep_offline_prompt(cfg, runtime_dir)
            assets, identity = prep_assets(model, cfg['models_dir'], prompt_override=prompt_override)
            python = prep_overlay(cfg['packages_dir'], runtime_dir, model, cfg)
            if model == 'difftsr':
                assets['config'] = str(Path(prep_vendor(runtime_dir, model)) / 'model/DiffTSR_config.yaml')
            checkpoint = Path(assets.get('checkpoint', str(Path(assets.get('base_model', '')) / 'model.safetensors')))
            inventory = json.loads((checkpoint.parent / 'assets.json').read_text(encoding='utf8'))
            checkpoint_record = next((item for item in inventory['files'] if item['path'] == checkpoint.name))
            worker = PrepWorker(runtime_dir, python, model, assets, cfg['model_options'][model], cfg['load_timeout_seconds'], action='probe')
            try:
                probe = worker.ready['report']
            finally:
                worker.close()
            report[model] = {'state': 'ready', 'assets': assets, 'checkpoint_identity': identity, 'python': python, 'probe': probe, 'checkpoint': {'filename': checkpoint.name, 'sha256': checkpoint_record['sha256'], 'bytes': checkpoint_record['bytes']}, 'vendor_identity': dataset_hash(VENDOR_SOURCES.get({'tadisr_x4': 'TADiSR', 'difftsr': 'DiffTSR', 'unihir_restoration': 'UniHIR'}.get(model, ''), {}))}
            for notice in probe.get('active_import_record_warnings', []):
                print(f"{model}: cảnh báo source Torch kernel khác RECORD: {notice['path']}; hash thực tế lưu trong {Path(runtime_dir) / 'preflight.json'}")
        except Exception as exc:
            report[model] = {'state': restoration_failure_state(exc, phase='preflight'), 'reason': str(exc)}
            failures.append(f'{model}: {exc}')
    Path(runtime_dir).mkdir(parents=True, exist_ok=True)
    (Path(runtime_dir) / 'preflight.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf8')
    if failures and (not cfg['allow_partial']):
        raise RuntimeError('Preflight stopped before PDF processing:\n' + '\n'.join(failures))
    return report

def prep_flash_runtime(runtime_dir):
    """Embedded bootstrap: no need to re-upload the packages Dataset."""
    import importlib.util
    data = FLASH_RUNTIME_SOURCE.encode('utf8')
    digest = hashlib.sha256(data).hexdigest()
    helper = dataset_path(runtime_dir, 'bootstrap/flash_runtime_' + digest[:16] + '.py')
    if not helper.is_file() or dataset_file_hash(helper) != digest:
        dataset_write(runtime_dir, 'bootstrap/' + helper.name, data)
    spec = importlib.util.spec_from_file_location('vtm_flash_runtime_' + digest[:16], helper)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def plan_restoration_variants(root, base, config, *, applicability=None):
    applicability = applicability or preparation_applicability
    result = dict(schema='vtm-restoration-plan-v1', selected_regions=0, manifest_rows=0, planned_output_images=0, planned_output_pixels=0, by_variant={}, resource_warnings=[], storage_estimate_basis='PNG upper estimate from raw RGB plus row/deflate overhead; before deduplication', runtime_estimate='unmeasured; CPU crop timings are not full-page/GPU estimates')
    storage = 0
    for page in base['pages']:
        record = json.loads(dataset_read(root, page['record']))
        for region in record['regions']:
            selected = region['crop_type'] in config['crop_types'] and (not config.get('region_ids') or region['id'] in config['region_ids'])
            result['selected_regions'] += int(selected)
            w, h = region['images']['original']['size']
            for name in config['variants']:
                spec = variant_spec(name)
                item = result['by_variant'].setdefault(name, dict(planned_images=0, pixels=0, not_applicable=0, deselected=0, method=spec, parameters=config.get('model_options', {}).get(name, config.get('variant_options', {}).get(name, {}))))
                result['manifest_rows'] += 1
                if not selected:
                    item['deselected'] += 1
                    continue
                state, reason = applicability('tadisr_x4' if name == 'tadisr_text_mask' else name, page['book'], page['page'], region, config)
                if state != 'ready':
                    item['not_applicable'] += 1
                    continue
                scale = spec['scale']
                ow, oh = (w * scale, h * scale) if scale else (512, 128)
                pixels = ow * oh
                item['planned_images'] += 1
                item['pixels'] += pixels
                result['planned_output_images'] += 1
                result['planned_output_pixels'] += pixels
                if name != 'original':
                    storage += (3 * ow + 1) * oh * 1.01 + 1024
                limits = {}
                if pixels > config['max_output_pixels'] or w * h > limits.get('method_max_pixels', float('inf')):
                    result['resource_warnings'].append(dict(book=page['book'], page=page['page'] + 1, region=region['id'], variant=name, status='resource_exceeded', pixels=pixels, method_max_pixels=limits.get('method_max_pixels'), reason='explicit pixel limit; no automatic downscale'))
    result['additional_storage_estimate_bytes'] = int(storage)
    result['deduplication'] = 'content-addressed PNG storage; identical outputs can share objects, method rows stay distinct'
    return result

def preparation_policy_variants():
    """Active methods; historical registry identities remain readable."""
    return ('original', 'tadisr_x4', 'difftsr', 'unihir_restoration')

def validate_preparation_selection(selected):
    allowed = set(preparation_policy_variants()) | {'tadisr_text_mask'}
    rejected = sorted(set(selected) - allowed)
    if rejected:
        raise ValueError('Methods removed from active restoration policy: ' + ', '.join(rejected))
    return list(selected)

def preparation_run_groups():
    """Source baseline only; text SR must be selected explicitly."""
    return {'baseline': ['original'], 'light': ['original']}

def select_preparation_variants(mode, selected):
    groups = preparation_run_groups()
    if mode in groups:
        return list(groups[mode])
    if mode == 'full':
        return list(preparation_policy_variants())
    if mode == 'selective':
        return ['original', 'unihir_restoration']
    if mode in VARIANT_SPECS:
        return validate_preparation_selection(list(dict.fromkeys(['original', mode])))
    if mode == 'selected':
        unavailable = {n: UNAVAILABLE_RESTORATION_METHODS[n] for n in selected if n in UNAVAILABLE_RESTORATION_METHODS}
        if unavailable:
            raise ValueError('No runnable upstream/checkpoint for requested methods: ' + str(unavailable))
        return validate_preparation_selection(selected)
    if mode == 'classical':
        return ['original']
    if mode == 'all_eligible':
        return [name for name in preparation_policy_variants() if name != 'difftsr']
    raise ValueError('VARIANT_MODE must be full, selective, a run group, method name, classical, selected or all_eligible')

def plan_disagreement_restoration(root, base, config):
    """All six pairs must pass; rank and budget unique canonical source crops."""
    cfg = copy.deepcopy(config)
    options = cfg['selective_restoration']
    evidence = cfg.get('_ocr_disagreement_evidence', {})
    methods = [m for m in cfg['variants'] if m in ('tadisr_x4', 'difftsr', 'unihir_restoration')]
    if methods and options.get('thresholds') is None:
        cfg['_restoration_selected_keys'] = {m: [] for m in methods}
        cfg['_selective_unihir_keys'] = []
        cfg['_selective_restoration_report'] = dict(schema='vtm-selective-restoration-v2', selected=[], skipped=[], by_method={}, options=options, interpretation='thresholds_not_configured; analyze original OCR first')
        cfg['variants'] = [v for v in cfg['variants'] if v not in methods and v != 'tadisr_text_mask']
        return cfg
    units = {}
    for page in base['pages']:
        record = json.loads(dataset_read(root, page['record']))
        available = {r['crop_type'] for r in record['regions']} & set(cfg['crop_types'])
        for method in methods:
            priority = ['page', 'group', 'column'] if method == 'unihir_restoration' else ['column', 'group', 'page']
            kind = next((k for k in priority if k in available), None)
            for region in record['regions']:
                if region['crop_type'] != kind or (cfg.get('region_ids') and region['id'] not in cfg['region_ids']):
                    continue
                identity = (method, page['source_sha256'], page['page'], region['id'])
                alias = dict(key=f"{page['book']}:{page['page'] + 1}:{region['id']}", book=page['book'], page=page['page'] + 1, region_id=region['id'])
                item = units.setdefault(identity, dict(**alias, aliases=[], variant=method, source_sha256=page['source_sha256'], source_box=region['source_box'], source_pixels=region['images']['original']['size'][0] * region['images']['original']['size'][1], crop_type=kind, contract_reasons=[]))
                item['aliases'].append(alias)
                state, reason = preparation_applicability(method, page['book'], page['page'], region, cfg)
                review = preparation_region_review(page['book'], page['page'], region['id'], cfg)
                if method in review.get('excluded_variants', []):
                    state, reason = ('not_applicable', 'explicitly_excluded')
                if method == 'difftsr':
                    box = region['source_box']
                    ratio = (box[2] - box[0]) / (box[3] - box[1])
                    if not (review.get('upright_horizontal_single_line') and review.get('allow_simplified_text_prior') and (1 <= review.get('character_count', 0) <= 24) and (abs(ratio / 4 - 1) <= 0.05)):
                        state, reason = ('not_applicable', 'requires_reviewed_horizontal_line_and_DiffTSR_contract')
                if state != 'ready':
                    item['contract_reasons'].append(reason)
    selected, skipped, budgets, keys = ([], [], {}, {m: [] for m in methods})
    for method in methods:
        if method not in options['thresholds']:
            raise ValueError('Configure a disagreement threshold for ' + method)
        threshold = options['thresholds'][method]
        if isinstance(threshold, bool) or not isinstance(threshold, (int, float)) or (not 0 <= threshold <= 1):
            raise ValueError('Restoration disagreement thresholds must be in [0, 1]')
        budget = dict(max_regions=options.get('max_regions', 12), max_regions_per_book=options.get('max_regions_per_book', 2), max_total_source_pixels=options.get('max_total_source_pixels', 12000000)) if method == 'unihir_restoration' else options.get('method_budgets', {}).get(method, dict(max_regions=256, max_total_source_pixels=32000000))
        for name, value in budget.items():
            if name not in {'max_regions', 'max_regions_per_book', 'max_total_source_pixels'} or (value is not None and (type(value) is not int or value < 0)):
                raise ValueError('Invalid restoration budget: ' + name)
        candidates = []
        for item in units.values():
            if item['variant'] != method:
                continue
            score = evidence.get('scores', {}).get(f"{item['source_sha256']}:{item['page'] - 1}:{item['region_id']}", {})
            item['ocr_disagreement'] = score
            item['threshold'] = threshold
            reason = '; '.join(sorted(set(item.pop('contract_reasons')))) or (score.get('reason', score.get('state', 'original_ocr_not_available')) if score.get('state') != 'ok' else 'pairwise_disagreement_below_threshold' if score['min_pairwise_rate'] < threshold else None)
            if reason:
                skipped.append({**item, 'reason': reason})
            else:
                candidates.append(item)
        count, used, by_book = (0, 0, {})
        for item in sorted(candidates, key=lambda x: (-x['ocr_disagreement']['min_pairwise_rate'], -x['ocr_disagreement']['mean_pairwise_rate'], x['source_sha256'], x['page'], x['region_id'])):
            source = item['source_sha256']
            values = dict(max_regions=count + 1, max_regions_per_book=by_book.get(source, 0) + 1, max_total_source_pixels=used + item['source_pixels'])
            if any((limit is not None and values[name] > limit for name, limit in budget.items())):
                skipped.append({**item, 'reason': 'selective_restoration_budget'})
                continue
            selected.append({**item, 'reason': 'all_six_original_ocr_pairs_above_threshold'})
            count += 1
            used += item['source_pixels']
            by_book[source] = by_book.get(source, 0) + 1
            keys[method].extend((alias['key'] for alias in item['aliases']))
        budgets[method] = dict(selected_unique_regions=count, source_pixels=used, limits=budget, threshold=threshold)
    cfg['_restoration_selected_keys'] = keys
    cfg['_selective_unihir_keys'] = keys.get('unihir_restoration', [])
    cfg['_selective_restoration_report'] = dict(schema='vtm-selective-restoration-v2', selected=selected, skipped=skipped, by_method=budgets, options=options, original_manifest=evidence.get('original_manifest'), interpretation='all six normalized Levenshtein rates from original OCR; triage only, not verified damage or accuracy')
    cfg['variants'] = [v for v in cfg['variants'] if v not in keys or keys[v]]
    if 'tadisr_x4' not in cfg['variants']:
        cfg['variants'] = [v for v in cfg['variants'] if v != 'tadisr_text_mask']
    cfg.pop('_ocr_disagreement_evidence', None)
    return cfg

def restoration_source_metrics(image):
    import cv2
    import numpy as np
    original_size = image.size
    preview = image.convert('L')
    preview.thumbnail((1024, 1024))
    gray = np.asarray(preview)
    _, ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    count, _, stats, _ = cv2.connectedComponentsWithStats(ink, 8)
    components = stats[1:]
    components = components[(components[:, cv2.CC_STAT_AREA] >= 3) & (components[:, cv2.CC_STAT_AREA] <= gray.size * 0.02)]
    foreground = gray[ink > 0]
    contrast = float(np.percentile(gray, 90) - np.median(foreground)) if foreground.size else 0.0
    return dict(size=list(original_size), analysis_size=list(preview.size), ink_fraction=float(np.mean(ink > 0)), contrast=max(0.0, contrast), component_count=int(len(components)), median_component_height=float(np.median(components[:, cv2.CC_STAT_HEIGHT])) * original_size[1] / preview.height if len(components) else 0.0)

def plan_selective_restoration(root, base, config):
    """Rank disjoint source crops before any model asset/runtime preflight."""
    cfg = copy.deepcopy(config)
    options = cfg.get('selective_restoration', {})
    if options.get('enabled') and options.get('strategy') == 'ocr_disagreement':
        return plan_disagreement_restoration(root, base, cfg)
    if not options.get('enabled', False) or 'unihir_restoration' not in cfg['variants']:
        return cfg
    maximum = options.get('max_regions', 2)
    per_page = options.get('max_regions_per_page', 1)
    pixel_budget = options.get('max_total_source_pixels', 4000000)
    for name, value in [('max_regions', maximum), ('max_regions_per_page', per_page), ('max_total_source_pixels', pixel_budget)]:
        if type(value) is not int or value < 0:
            raise ValueError(f'selective_restoration.{name} must be a nonnegative integer')
    candidates, rejected, seen = ([], [], set())
    for page in base['pages']:
        record = json.loads(dataset_read(root, page['record']))
        for region in record['regions']:
            if region['crop_type'] != options.get('crop_type', 'group') or region['crop_type'] not in cfg['crop_types']:
                continue
            if cfg.get('region_ids') and region['id'] not in cfg['region_ids']:
                continue
            key = f"{page['book']}:{page['page'] + 1}:{region['id']}"
            identity = (page['source_sha256'], page['page'], tuple(region['source_box']))
            item = dict(key=key, book=page['book'], page=page['page'] + 1, region_id=region['id'], source_box=region['source_box'])
            if identity in seen:
                rejected.append({**item, 'reason': 'duplicate_source_crop'})
                continue
            seen.add(identity)
            size = region['images']['original']['size']
            if not maximum or not per_page or size[0] * size[1] > pixel_budget:
                rejected.append({**item, 'reason': 'selective_restoration_budget'})
                continue
            with dataset_load_image(root, region['images']['original']) as image:
                metrics = restoration_source_metrics(image)
            item['metrics'] = metrics
            reviews = cfg.get('reviewed_regions', {})
            review = reviews.get(key, reviews.get(f"{str(page['book']).removesuffix('.pdf')}:{page['page'] + 1}:{region['id']}", {}))
            approved = 'unihir_restoration' in review.get('approved_variants', []) and bool(review.get('restoration_reason'))
            excluded = 'unihir_restoration' in review.get('excluded_variants', [])
            nonblank = 0.003 <= metrics['ink_fraction'] <= 0.4 and metrics['component_count'] >= 12
            faint = metrics['contrast'] < options.get('contrast_threshold', 64)
            small = min(metrics['size']) < options.get('short_side_threshold', 512) and metrics['median_component_height'] < options.get('component_height_threshold', 6)
            if excluded or not (approved or (nonblank and (faint or small))):
                rejected.append({**item, 'reason': 'explicitly_excluded' if excluded else 'no_conservative_source_trigger'})
                continue
            item['reason'] = 'reviewed: ' + review['restoration_reason'] if approved else 'faint_ink_proxy' if faint else 'small_ink_components_proxy'
            item['score'] = (1000 if approved else 0) + max(0, 64 - metrics['contrast']) + max(0, 6 - metrics['median_component_height'])
            candidates.append(item)
    selected, pages, used = ([], {}, 0)
    for item in sorted(candidates, key=lambda x: (-x['score'], x['key'])):
        size = item['metrics']['size']
        pixels = size[0] * size[1]
        page_key = (item['book'], item['page'])
        if len(selected) >= maximum or pages.get(page_key, 0) >= per_page or used + pixels > pixel_budget:
            rejected.append({**item, 'reason': 'selective_restoration_budget'})
            continue
        selected.append(item)
        used += pixels
        pages[page_key] = pages.get(page_key, 0) + 1
    cfg['_selective_unihir_keys'] = [item['key'] for item in selected]
    cfg['_selective_restoration_report'] = dict(schema='vtm-selective-restoration-v1', selected=selected, skipped=rejected, source_pixels=used, options=options, interpretation='source proxies only; not verified degradation, necessity, or accuracy')
    if not selected:
        cfg['variants'] = [name for name in cfg['variants'] if name != 'unihir_restoration']
    return cfg

def selective_region_applicability(model, book, page, region, config):
    if model in config.get('_restoration_selected_keys', {}):
        key = f"{book}:{page + 1}:{region['id']}"
        if key not in config['_restoration_selected_keys'][model]:
            return ('not_applicable', 'selective_restoration: original OCR disagreement below threshold, input contract or budget; original retained')
        return ('ready', None)
    if model != 'unihir_restoration' or '_selective_unihir_keys' not in config:
        return ('ready', None)
    key = f"{book}:{page + 1}:{region['id']}"
    if key not in config['_selective_unihir_keys']:
        return ('not_applicable', 'selective_unihir: source trigger absent or budget exhausted; original retained')
    return ('ready', None)


In [ ]:
# Chỉ dùng để kiểm thử checksum bằng fixture tổng hợp.
def assets_inventory(folder,license_info,source_info):
    folder=Path(folder);files=[]
    for path in sorted(folder.rglob('*')):
        if path.is_file() and path.name!='assets.json' and not path.name.endswith('.part'):
            files.append({'path':path.relative_to(folder).as_posix(),'sha256':dataset_file_hash(path),'bytes':path.stat().st_size})
    obj={'format':'vtm-local-model-assets','version':1,'license':license_info,'sources':source_info,'files':files,
        'inference_verified':False}
    folder.mkdir(parents=True,exist_ok=True);target=folder/'assets.json'
    data=dataset_json_bytes(obj)
    # Inventories are regenerated from actual files; weights themselves are never overwritten.
    temporary=target.with_suffix('.tmp');temporary.write_bytes(data);temporary.replace(target)
    return obj


### Tự kiểm thử ảnh (tuỳ chọn)

`RUN_SELF_TESTS=True` chỉ kiểm fixture tổng hợp, không đánh giá độ chính xác OCR.


In [ ]:
"""Embedded source/text-SR contract tests, with no GPU or model loading."""
import copy
import tempfile
import unittest
from pathlib import Path
from PIL import Image

class OfflineImageTests(unittest.TestCase):
    def test_original_is_pixel_identical(self):
        image=Image.new('RGB',(19,23),(128,27,86))
        output,_=dataset_variant(image,'original',{})
        self.assertEqual(image.tobytes(),output.tobytes())

    def test_retired_methods_are_rejected(self):
        for name in ('clahe','denoised','restormer_real_denoising','docres_appearance'):
            with self.assertRaises(ValueError):select_preparation_variants('selected',['original',name])

    def test_baseline_and_explicit_text_sr(self):
        self.assertEqual(select_preparation_variants('baseline',[]),['original'])
        self.assertEqual(select_preparation_variants('difftsr',[]),['original','difftsr'])
        self.assertEqual(select_preparation_variants('unihir_restoration',[]),['original','unihir_restoration'])
        self.assertNotIn('difftsr',select_preparation_variants('all_eligible',[]))

    def test_difftsr_requires_review_and_upright_horizontal_geometry(self):
        region={'id':'line','source_box':[0,0,400,100]}
        cfg={'reviewed_regions':{},'applicability':{}}
        self.assertEqual(prep_applicability('difftsr','book',0,region,cfg)[0],'not_applicable')
        cfg['reviewed_regions']['book:1:line']={'upright_horizontal_single_line':True,
            'character_count':12,'allow_simplified_text_prior':True}
        self.assertEqual(prep_applicability('difftsr','book',0,region,cfg)[0],'ready')
        region['source_box']=[0,0,100,400]
        self.assertEqual(prep_applicability('difftsr','book',0,region,cfg)[0],'not_applicable')

def run_image_selftests():
    result=unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(OfflineImageTests))
    if not result.wasSuccessful():raise RuntimeError('Embedded source/text-SR selftests failed')
    return {'tests':result.testsRun,'real_neural_inference':False}


### Lấy ảnh gốc và crop

Ảnh giữ nguyên pixel nguồn; không chạy TADiSR/DiffTSR/UniHIR.


In [ ]:
_DEFER_PREPARED_EXPORTS = True
CONFIG['variants'] = ['original']
if RUN_PREPARATION:
    for key in ('packages_dir','models_dir','data_dir'):
        if not CONFIG[key] or not Path(CONFIG[key]).is_dir():
            raise FileNotFoundError(f'Điền đường dẫn {key}: {CONFIG[key]!r}')
    BASE_ENVIRONMENT = prep_bootstrap(PACKAGES_DIR,RUNTIME_DIR)
    # Canonical dataset only: no restoration manifest/receipts or per-crop
    # restoration QC. Canonical acquisition/layout validation remains required.
    _profiles = Path(RUNTIME_DIR) / 'profiles'
    _profiles.mkdir(parents=True,exist_ok=True)
    for _pdf in Path(DATA_DIR).rglob('*'):
        if _pdf.is_file() and _pdf.suffix.lower() == '.pdf':
            _layout = CONFIG.get('book_layouts',{}).get(_pdf.stem,{})
            (_profiles / (_pdf.stem + '.json')).write_text(json.dumps(_layout),encoding='utf8')
    _canonical = ImageDatasetConfig(variants=['original'],max_source_pixels=CONFIG['max_source_pixels'],**CONFIG['layout'])
    _images = prepare_image_dataset(DATA_DIR,DATASET_DIR,config=_canonical,
        books=BOOKS,pages=PAGES,limit=LIMIT,max_books=MAX_BOOKS,profile_dir=_profiles)
    SUMMARY = {'completed':True,'canonical':_images}
else:
    SUMMARY = {}


### Kiểm tra bắt buộc

Chuẩn bị canonical tự kiểm ảnh/layout; gói handoff giữ hash. Không chạy QC restoration hay preview riêng.


In [ ]:
# Notebook 1a chỉ xuất gói handoff; không cần helper báo cáo ở đây.


### Một gói chuyển tiếp

Không xuất bản sao ảnh, ảnh ghép hoặc ZIP ảnh/báo cáo riêng. PNG và raw cần thiết được gom một lần vào handoff.


In [ ]:
# Notebook 1a chỉ xuất gói handoff; không cần helper báo cáo ở đây.


In [ ]:
# Chỉ chuyển sang OCR khi bước ảnh của lượt hiện tại đã công bố dataset.
if RUN_OCR:
    if RUN_PREPARATION and SUMMARY.get('completed') is not True:
        raise RuntimeError('Bước ảnh chưa hoàn tất; giữ cache và chạy lại trước khi OCR.')
    if not Path(PREPARED_IMAGES).is_file():
        raise FileNotFoundError(f'Chưa có ảnh của lượt chạy: {PREPARED_IMAGES}')
    print('Ảnh đầu vào OCR trong cùng phiên:', PREPARED_IMAGES)
else:
    print('Đọc checkpoint tại:', WORK_DIR)


### GPU

Bốn OCR chạy trên bốn GPU như cấu hình; không có giai đoạn restoration.


### Các hàm OCR và đối chiếu

Mã OCR và setup offline được nhúng; worker OCR chạy trong tiến trình riêng.


In [ ]:
# Mã nội bộ nén lossless; kiểm SHA-256 trước khi nạp. Chỉnh cấu hình ở ô đầu.
import base64 as _vtm_b64, hashlib as _vtm_hash, zlib as _vtm_zlib
_VTM_CODE_SHA256 = 'f82ec9fdecac4e3836d5c053841f48e2ffd045146f1724f50ffd45ecfb907914'
_VTM_PACKED_CODE = (
    'c-ri}Ym+0{bs+Y=enoOV7@0jNRH3^1g&vTq>8=@;*xgmCt{##CvAGE#QAiem1d)k)uv8I}SF)B7a%eZgUOQH6hZwYC73)>3Bi8G1aAfHNH~c>)cKOgJ{{^4dePt#d)zjSY2RAw0$jtjV_uPBWJ?Gr>7|zp)7ly+{'
    'zL>{h=p~a`I?ugmI!*H^Pts|&HN+zQbehN4`8esRBJ+5V%;SC@X89mp<aOsoJfCyDS(J~gN+<BhAM%@7oT=Au;ooumCYr?AEb7Nx;^^ciAEndAAjxLq=th;5$GZ1fGLBy*(~GUGH^*m(pB^87Cwy`I@-RF-K0fo>'
    ')RliePJ7Wf^Ml5DocpzM!SMKt!_(8Fmxr~wSKG-avz^O)62jo)UYcHnSLysBp4Wn)F^{u!d>Q*eMQ@GC1q|JvMe}%?XYI53BCdP!b&};_dO?p?t?wS6e&_Hs{Ost}AwXFH`i}Qe+Mi3;joHmdrtZn<G1d!T9-S_&'
    'JDaCNz)beh)|SFplt=w>lx1<IUX7v*r$s&fFiWS5bgQgatX}tqNjx5`pV(x2uHGL^Z??Yu?#=h#9lSy4U!UOA)PD8n|Lm84{x5#@FaGpbKmFM+|Kjg{{gXfa&;RG&{OWIh25)}-FaP*gKl#I7{i~n+=fD5yum0%Y'
    '{rbQA>q#`f7^GLzU;X9Z{_=1C{5L=S>tFvbKmULJ_RoI#C;taj`iGzXhhP7zpZupk{3H1Bn?L@S|L{-#`j`Lozxn0g|6jlU@Ba$QKw+o_ZH)5CSl9o}-~AnY<KCf(U;fFzh4)Zs9KjMV^zy4e`^m5W<X`{8|M(}r'
    '`FCIb!%u(y%b)-GufP1`Uw`?hzxn&W_{~rM>^J}VXTSQN|Ji@~!~eRr_3GgJ$M4SUF_vm_X&5hb3d8*MPyQ+N#KWW}wg&Og3vn(&_!TY2xu3QWgF$P{gFng8OR{8|<<Ybs`)S?tvl#YWzJR?GNE!Syk6|-Uy)mpm'
    'KMl65S6M#CH`EdA=wWgm;!5!Q!*l8gS8ad17{o0Pn(jdHZs2Wy6YFvFxE5e=w-+t%(i^68@1hPrlBvCJ8c{Y#`Z+clG?F}?!1hDfFFe>*SXTPDW5}W_Xb=b87%V2U%*U1hj%kJjDa!gu(*6v#Fzl!d7F2i<->4lX'
    'NNPJMbH*#nypfHfrVl+e;%Pq}#C~m&54WG<wjRaTgXBC0OaWScM~`BQ{`QVvoW3A5@#sJL=%W{3{rs;M-sl(qb~^HU|M&m))t7(K$5IXN^;cj1w~3d}@!kIf>;FIg^RK@AqlMT1#lQFR(N|yo6EFMf%fE@d{Hrhj'
    'S5J|b1~f<w$J6s<ib&7rxO7H~>CGaV)_B&Zvqc^PebJv0O8$n8WG$L$nk@KAThte1gu|$xr}LXOmSD^vS+N_#XKrl1n1)jz!TJ-wo<)G3!n?sNX#rq4yxH5`rB{>aI?NJWIsCr4_mn?m@%aSEA{{2WnQ1zWrPnwb'
    '0KEwL?6AvR#`8SsN8@mwk87o-!f1XDVBtzzI%@o=*26j0K_F+zbv(`#n7hT#_86Rxi&+N1z_;Nzjc9}o^&L)z1;GT6LzgX_mob!}b<5a+RW}YN$<*oo54<;k2c+cik6o?+X1QFH=YR<8vKRF);^{zREn4KMESw3<'
    'T!@PW_HtnYoo~3OD!*Sw^CX&@y6HTDbsCNBGW~fv3-K`5rOxN+Vpi47FdB~$+zR|iTgYTgtKfl}=~Z|hPvbcd>4AbQt3ov(ZyH~Pc?x*Z^Sr<N>@hE02iD=N`M72<uipb0eSl==+q@HDhg+q>d3*_k*jlRm5l`)C'
    'd=6O3N0ShiuTy3=kC8zbY(uf_%cg8DI5B6%$$^4#p3v;hVnBC(<1D(TJPv@NA6HEKEE){PajE;uX5+C_;9mPDPp9YS3mEt;EdVQD_%tbe8z++_U$%siu7r6qiGg`kv-D7i$uvBlE!<tm>`*u(p0Lo5^9>o3`;o-n'
    'VgTG>II^(3+jztjCx?}X`&N^+M|zRLzR#n1u9w&&0UelAYe)<|zY8mRYB&Yl_2BRnd{A5?SEW}yo`dv$2*8XgHUqp~OyUq3v(0fL>kD|1aFzlOn@p2D41w7g4~ZMVDKy*$ob33eQ6fp|xWDDa3H)_7m__*S5DwQ-'
    'Xe#XC%NW+fymn_AX6_)#V(*J+yoe9y^K|al-T`AULnOq5G8L=^C9wEtRj~N_Y=s_qBharIY)bsWs~LX*GcviU1#7?=)};N5(J%gYIQ8MA|JlqNfA!^`w!B-b|GNsvh!PpVhO>u+2O>%EZ*K_yuJ4~)S=v2)uIzo5'
    'c%Q)X^ni(@7wYgUzctj9zO9=fXt}Q0fqH=+d$ZA3KmS?cebd{m0Lnh#&GI2;q3-Er)Q97CGR@Yn6{oq$BFnuVB76!^VOuyz)m9mn5@TWjaySzJXapqx$8dJRn{mFu8ZI}jfU{dh@Cq$gL0MpIyicszEATbyn~x)D'
    ';$+&Tgu%hr`@#AlU>&a?%`y%AH(_x+FR)AcBOtERtb6-icM#F^#`i&j$OyZ1Mnph5E_oHAghAqk&x^6ZMb$m~wMIb@tOcK{cvM`-zZnM5JUDzWW5g5!RDty(ASL)K;VPA?9u@43>S1j(D7zsV5%vfYR}A#B*rGc!'
    'THZ#!ENw{Q80y3^1ozo_R8M0!q6Xk~G{339Y=NmXdk)zJ+-uBTYAS%grV&op`fXQiTTeK=u6xvQT?hisy)?TFMqxn}guo((N3Rb)KMcP(I6XRebN2Q=Cz;da+AawpgJd|&Gdi^v(`1y)i6Keng!9@RFE9j-Pv<2J'
    '^xB#HTvgc??%Y#VVW<CS0TgU0b53G<d5u{E63M{Y4DLX`VGA9as+EM7wSzdq8XEAOYzky+g!48){vp57Lvfm}S*%b$>W^aV%dkOODZ-2y=`>I`QqvMU&W*_1_)`hk_?uW`W<m@bOp6&=UsX{vwWSxx<r0@593`<='
    'idn@6n?vPxnZ=~eU2c@PFJdf-|3Frfz%{&M?QR2hudT!oJyF6NeZx{|xHBbKYXBqW*Vb@zg{5Q5ip$$9m9^Rg0vK0UUCdww#Qtr~T@HYg($cHll{agp97yOX%77~%uWV!qMwV^B!j36WthFR$)m$y<ag*Kt^>MCW'
    'DuStBXyRV3sI(9EjES1iR)r$GS`BIufIJOJtEsm?K9zK?6>3nbz$#ae`350QEO+R$sEV@QACLmQ?meh`J%F}lG>;-VxA8CPx2%(V1<k^_`nzwB-{4mnj+ersf!F<XcH^ZmfKjX!k2^q$kmg9y1<!bcNBbFyhOl;k'
    'K;hBEN`0tI?fbl^>qaeES1QH2D~H#O_7L!y_ZxwC5yvxX>Jn*l#!cgR8{XLn`7NgO^hz;Hz_TPEtPUD#w9;%oQg=M~JFvCWq)bL!XY`FY`4Dffd{X}c=iO!n@OGA**MeZHngL=(bb;Nekic%Sle?CUdc*7x($d}!'
    'fzOw{aWPG=rg%;a8+aHHk2@@?&`CVcd_C6z6X3;E)rJ*K0K$^s7S$l6$VL#W<}Ap$W0W3=G4Z8HphQUMp+;KgngZD{8U@^Sq+qvBnuh?X5wZJ)vRJB)yMRjIM|?wgp&!zZZq8xvSB#E|TT_8R;6^|Ja595KAGNj9'
    'b49acO}0IsmLK%4vF`^#L+yXNup(MC5%#@<srldm7X=ESF5A*<=?z-lJ6X)*?<UhhdX>?Yn!aq^ViF=7H^j<5HEk0RA`tkaWkYkK%UV#-m7jq~er1c^Y@YVx%)GfVH^BYH9B()qd|Rw92+u~i!%ot4e0UxAfdc?i'
    '5M@B*@KP5R5fFwb$?M){N2hPk!WYM{PhK6K9ln(7!jVYBs<p#Q=%VhuO8Xb8<MEiqWOF46>{_fKM0_xEfOC>-(=eW1Ci8TPLWmFOO!`EH%uA-HnuNZMu?ry3=O}%Pay7#{?QDj919*`)i85S`qZrO)n=sDCQ4Xth'
    'VqNE4!aAbml4cF*6}OHB0Ma)S1d$D~Kgd2~n4uZE#*;^vR7nxFt?=0MlRyzpR6oL}goOSf{eqpO8lt^6j;DBq=$n_os^LrZV#PkIeSY%JJ9=q0*Z|;6rhQ~DrqlSGgeQ$qX;2Xnjl(vMobTpeynA^NesT2n=u^Dl'
    'KK$b7#o=4EHGx|6#>iajnvHBWPM}u3Mt6&*MO(%u)M|w@6jKdZh0G95FP+N<(uSEI7!8E33a#j`q^i(Q$mec1Yu%E5BSF;&;#C?6D&}LYqEF%bR;t`7sospe3We=k{rwwrMa2Q>ooa-mZWxf-_+gq%eFbkP=`O(@'
    'qZqH1ld3`4R0#UyT?&*aL9$7l5N_E?G!wQWLv`T{?FY=2H!e{15>brLQhMF`3|}9d@cM*PH95zd=cu2LfgJ+K5ubVL#lR)q&<V(mswFzmdrvxVFd_F1zi^LTHz@A7fxX)Xt4DGY&4G7~#{{KFsu_);a?kNX@n*pN'
    'IYUw`Dv~9IP@WMW`MsE?BAFufP3Jehp^Yc_6-NQQ<R}>ryet(ujAZdFo%cr|9;3k!UPHt`j+rQuF-$F^MM<PnlgBJw%=@^{KC;fDI!p$%^n<C;-gc+^gQ-T~Jer=zK9y+n7lWw5rQnQ#^AM(_^@Az4eBI_EfSb(U'
    't8=0Du!dDyw+Sr&=X`Xx>Hfv{uOD0o4G?pEwp4(YxFG?e0cZk2)F+(5s*bPQNgcViwgn2x&;kyAg2hnhM)-ORb+^5n97a@K2arEQtuH{FAe({ZC-DU4z-k5&TW$r4h1gO8h=R(U-EO@f&7xicq>(Rk{OUeUDX8Xv'
    '8&84p3jrf-W*7h;GvJbP-?q?T<lwiD^exjEZ?g>h&;Ulo?*Q}Ah>0hNfN+3FwoR&y?S8E;s{oio0EUdu(nSs;h1hl=yRHnWY^MDw)D&$=``Ip39>slSVr4CIO9Wux8f>M;$Ho|mhaYsdo83UORkZ#=RJ;hUqD%OU'
    'KhflAF(uxkPTYmrs<4<(7gO4WSdz9IBZhur8_$9EjPqohC=IrTh>GWzyg^XcF^`d)iQL6SfR04)68Ib7jgejTaZ%M()j_DhV410plws}xi=KJ2MQ;qJEsJv4A{K~oLj#$~Gf$=lW(gn!VxG?7cb<|+mdtr`!>0B4'
    'sv{QWJCEn{#VmK_FOvW1{6=b~K;p|-7<t`WWXSh1?un`Hbn8?$sC%~}f%lR9$NNfR`Fu;*9sm)X`a!FxOu)7a2PMt7esz_ATv+Buebk0AXPC8-<YGfWfFnQ(6my$Y+%1S`_z}oM4@6<#ECikqhCXO{T8iWMA<`>s'
    '|ITSOpBMC0)D_Bp)E&`Ri)Z!)ilAJ{?6|BWrR3i(-R#oO2AA-IqOv8P<=&?t;2hFVv=AtDz$0tWY^?!V`mY93AmQdJnxd1=OdX)C(I3ar+$xI8$a!h-j3+ceK)c4)!|*ndm8n;ywTN_)PuhAN%h9YpEjgN(37iPf'
    '9bWhoYeNSxx5PJ(x6n%A3KqxZ{Z|GQyp%%(7-3q2Qs`f{#<D>bcLy&M1d!JTuVLR=!cG;`l_t|gVP51M^fl^(!W@YttfAg2!YYJWAQBVAQJQAVc>FN!`Pc=}c@Q4^$jA_(M**GGVzYTX1WFVSNOA+Z93^9T@B>ST'
    'LuI!I3_-epV?nOFGH;m7Gc=PN0cjh>W7a^8(<xg0dGKE{K^3Nlx<L?Egcu)CQJ}0qkAZOHqbs0}K#!-uJzil;36QmH1WXRn=y^PWk&+NQ1t^JhGk2abX@?#safBv~0v7240ohsJ@V*N~3(h800);?xxsqR{9vlV>'
    's7Sgrro(XbP<QmoLnx-m$U@t=ntE}56$6Yy@m$gklRhB+8!238$~y}0y%v14B(q>!Zt^!=n)(u)(cd-3Cst$l-RebFC{HTMdeGf&&{b?S+pZKfsd~P#i1jd)?nv+evhLGk-Mff^<B2A{LF7$aY7m`i*E&_qJkmP0'
    'HWn{57=o-ghLYC$?OkcFaUqsk>G~#%Vb=_@lC_46M8QIv5G299t(7GnNCzFa@vgVy`8E;>yxqY2*lPyfHyx-*(X1+Bq2)7Qo-7+pGD`v{qT7^Rj&nVPOiednu>xOpguE~+JtSk_OR5@|PorjlW@JA8Rl%iI(hR0|'
    'A+bdAO0!kiPC+Sjf|o8C-Ffm(BF+w|`wng4=RgO?$broV$VbJ|7-&y_907N#_bTrXm+BJiu)F3FY05Of**MKLO=gY`xl2PXcpoF6C?!LmfZ=gW?4^)8kY~ALcU)2&sMWxb#XP0ve9_R0dI1C^?36AIMk(F{BzfkX'
    'W=se(l<#EhgValb!p*fUUc-Bn!ZwA`;uDfk7W9f*e4=Z_XtCWfsr4A3z&h!f$iX_ru_(H9w`CvqJj{0C8CtmIa<VqVQ5;>~D8b$-Tx2g4Uhu3V3DhpEfN&X9<O89kj7oIb$fhWh8^w{Z<&EK+Nj2av)eYRA1EGUy'
    'u+2-ynmmgMVJJ0cV56Qn{NfpG@$$EKxT6hi9H;$@x?$&i)ZHSZZ|^rcN1W%P7D=Q4A=qqye*yr#&4Hbtw6-+pY?^3UB{EB1Koea&(wwyEbE!V;KxNff5CD*2u&paKD@02lp%xZ7$wK0i%lt11B8Zo2)>uvwnFI}L'
    'x4LwNpkx2RSp#rOjWuQ|2ou99E}rZHh)YIcx-m%snsl1<D~jIWjS%&5-$4TTu#Qwfl=5oPAiBq^J9~6QQ~>C0E9jSDmga9cCQZ|Lp6xbvcbj$navPyua)<`YMp-(k+p4{4xk%O*o9Q~sk}b5bXAl8-X-i9mHMRov'
    'YZ2aYhh~9`bZA%cba*-no7pB;BTcW&v~4PH8%n1|ZfD%%2+qinC0@A0Vq&C;)e<hq-NZ0}1{ydpJ06OF;s}Ja8sOALo%goryipmQ<@2JDBMvf%jm|2qfFW4SW7g&VhOp+G@Hh}NVE$<OE2#@U9PLqOoW<<(x!7r~'
    'pg*h-cTPc`k503@ywcD)^Ked<_rPZGg>|s9v;%zuBqxx580||34hs>-Ew|eUP_P^#IXwzu>MiiPxTe(!+6@dRjcB>65MT`_>Kj>x*c0Z9q3s}8g+`vRs%xvYXzW1a-E!kfalnhZy1gDO<cqQj2#(%tg!;V@u!DEI'
    'zO+)8V=6=uxm~#LU8_ryW~)Id>=ay!+$WxLqAd$n>dLqUth`Q5R!MfY+XV{aP?{?8vR>P^RMTcuV@175*=4uN6IWf{|7Z+s&ckIc-124;P0{yaOPn4-5<n{Ud$_z?wlj#sago)H84aB<VwHo}fozmRpMR&&jW(&o'
    'J1b=%Wm+2aun7jdVV9xijrRZHZeZ1sYlhgbotz$g{`$b{EwY=?l!ZUFRn!$p=AhSF!u<?-Th`)_#^XYD=)XS(8e!|O5nBmx<3ElCV=SrC1&@8cp;IX;@qr#p<?=vl-_ay;GRsk&j>j-*pI92Y5}gs<f6U<E$}@ex'
    'hJVQBLt`yy@O9iKj_M9ds0B3%ba)q$RDt>*rm$l177Lh}_ID3n)q<rpUYs5toE>^+2cNz=WXTq`kazUv?C|r$Q}5*T==H(r_r32Ne!nhvncmsq_s;5U7{ot``;{8_bVHgo{sxR<uBw1Q4XBm(!tOxtuj<~rH%Gts'
    '?oe5_kvS4J8U*VBKYH`>@OvH!{dhKb$8Ts9PFn;-JsAX>;z0y--x{%t3R**E%w(H9FwZu3M?>^p9)5Q4?$sIaNd*{?0xOF?pPE(8k<iPFwJhyc4dLj`+r!f{gyL96ps>L0s%F19c=hh^t>=HM4*v^)vDPMqZvV@W'
    'HNk6lr?(Qu$)I0W0;9jst?pX)%|>W=nZV3ETCG`Ep7<d^Yi;H@YJr0}Q(5r<fo(dDt3%n9IyGoXT4;TbE%)i+=SOe6qt~wwUmn2<DopP@Mdx~}{6}vOUmd<UgL72j$NTK`_%(06?|%F6^iV)-f6F^~^OE3{pFSb7'
    'tx@0|pXzstEv!*V8dm#T!0aENzJzW3>G!=C2X7B$H$pFOJgdeTKJR*mub@QJJACsJwu<-a==IT=*Zjy@9X85>#xTx-Et|%La}$^v81u@7`q8_Sm$=tx$ZrqNJT(SpCu(&8U(k^Z+TTJj=xrTa@9cKVi?g3jP#g%V'
    '6u|anG*oYP1A)^rP|~YK5K~w$#gzi2{yS()i^>xsfJ3qZgXIT+;87vB7c4a-xt#df4uQe<5x7U4GvmMH@>{*^%1cewQRUM3?JNhiK;L6MD4q?5Z?WuVj_C;Bj3Utd;lb<<f@qDn?=gL;YNv<`rcl`u0ir^~#M}>7'
    'Gr%*OUI{%rQVC`rz%je%qrrhbZ`-?Hi`Xrhs&Ca*^WR!S!Vo_BsV3E0ZHfTk?A8Y8_($?9kAb&pjjz3GdPFNQU>^ZdFPcJ&g0!zs_LnVM0Z-6S8Y3K+Ff>i*%o-2OinhUa^iXP`Wqs6p&<%lKI2{~;&9syzvPQ9P'
    '9E{s=Du1q`D6t`60tH?}a6kmsgSUkr!KKh<rljsok_=gK{^o6a>b;Pv_O4c%;ZM1w-gSAhHt{8%tuut3p)O(_E{H3V3A}|pp`s6%h%)%K;BF~r#~dhM;~M6-x>2daRbCvwdvoSL2<iu4e6A)Sut@jkr^oM3kS^P`'
    'Y88wkZX{ZP=mkZ4t6IwEv2?h(ZikC@y^q@r{6=+TifSaXZ(ePg3fJgF9ZQ;$^O*W`5#t4u-evC44cjAzv+!XP=%CRhSWs89RNklussenKto`TOhCUlD@?^Z)3&1wFU*i#Q!nJLc@IAwe4X2X`j&xN>9(5xzc`un3'
    '=$Ul`KgOeWO^C7|4JHkW0HDSMFZhC){UFga8FgAF?2>ZOl88%4C0OO#V$T)FAxvs22PJt7RJ<$pn#!+pDWnQ`J07|HL6KVBvtB|`;jE<6rgA)6lcvZjFu{_4L)jxbJ_$M70Fe#JrJiIz8fjVdo5y{!j7EAUN-1BD'
    'sV6saTlUoySsT=TFck$m?K8H(#I`y!;GU@e87XGtF&TXDm1{pUx@BJ>Y{whcZhLnP!b;-`P21VK3;NK#_%qX{+#f@$0p1fJC<Uuhj0L7ULfm+5G1VU9eG~q)u0|cqe%xOAS$)fF8!H$8Py&A%e37j!j<5B}quo32'
    '*0g=MbE_J=^KvwZuEAiStmkcSgA{f2)zAO#op+1P>F+RR8VxfqHP;}Yed@DXOC32vnDyAui&$uB%Z2;y|0gtGm(YY+gbKnR^M!?>BHF85D}M2+$**<((GPxb^@AU5cRxl8>aZp|O`K2Q1sM_Gv7-VB3IXw*h;NFH'
    'T_h4S%%WnzAq>w8E@T`W4=+K6=MCcoA9PwzyCokp=?QIm5Rk0Kdlmb-3vb;K2k?d)zY~Cgnq_e)ve7K=D}WPZS~vg`0Y_ns;GwGxb!yG+=5D?DsD__AyN$-Py4T$Adj5a>=U@H&?<X8|uNgdtEr2B^5~>fji1A5)'
    '4n=j1KOILae4(6ts-rBjd%{%q7e#;4P$Ko~Y9c#B6GfzR)TZ`fGR;PH4~Q`8z_F<67WKCr63nk{TeJ#Qz(7N|e<U5`5ZXjObxO}Yat{jtf$rv)3oyKSut_Dh2czD_+q#+e3Dy$Jt8g|L%H*)yvi3KGZp2xK*ZAxP'
    'iYAN69^LiQPhLtXVwhhvP6or5X@5cfdHTLU4lP?budoc159*7NPEiF9UyS~pfQ4q~Hz9517<Gf#rX1QcRI1OXX2<ok1-Nu3G=QshY80}et1HqzD-_A#lw4MXc7~Lrs14nqq3oKtG{c;c3nD?D2+T>Wf;%GUrhPl)'
    'quCNv#ssPy!77J&{>4v$xaFfb_3~f*o9PY)#BIQWL{8wVpZ`_v^}qW0&l)(H*szKtbpe(CVn)dM#ZOZVDD^)idaKB;(6AKeN+4$|o)OH-)E<P{4q1#}RGd*U*yb6QW5x|x=IkOiRV{%qO2|T~!wNCFRupg$?BcQ@'
    'ga+NBeJ3_jkPsOjWd@D4wj4nVJ1SR0k54t}%5auOT{gKVJgN38E;6wig<jANY7(W8wPjzGX5`1`DQ+lM%eaIWOLbLYSjIlo&Z=lJkAYl3!+Y_IfA3{q{ro?`M3N1fwsTjJ6=+-%{Q%ih*#4Pc2x)5hE>|q%SIJjj'
    '{%P)A{NmquCoew(>X4F|wQhlK#(~oR;%DCY7k@cHaA3)!wFPWSzNZuuLcN0LDP17b&eSbABp*vOh@p0wU&0Uuo_>?&pMfkkP>2!~<s;ZsH#T^;*p$Kw1!l(eX^~u^#MI*)x2Iow&;T?6w?^LfVfgys#qrzk)$|HN'
    'a%s$`$?p!^y+4?U&*?AzG8LpcpjOZyvj@@Sf63O><0`-;wiJd5m^E;Mf`Y138KzJVTP0%*IO~Abaa|8^SGZD#z^=B_EwBBS*bp$aZg~s<h2Vkx)DpbjnPCvM;1z@*{WzH^s$A#MSt~=G6?{aJ{-AJT7Mx+3Ms6{m'
    'IH4|t10APm7Mx_LXhpS6_M>^eoX@1K#gR@po)5)cZYkYZBsQD3CIXMQ-+Z?JaChHCKx}!<-TkMJo;)@Y5bakS9nr4G!4dTvv9aZ#W6Ua+<I<&b4NnJ4h-yPEV|-N#n1(I~rYvbt%7KJ*zsU2bz;IQ~IWyptEX2t('
    'Zm~b)zyl%rulj`Lx|fn6hvf%&0PM(3mSo#>Kc7r5$ldk1xMaW@#q$^b7C>=n^`j|TGe+YbioG#UXNEe_RG%EZQqz3I-O@iVMo?0I!~V`u(M}J?Gd^(7=Mk!(2vpR1Vn9K^rmo=eYa<^mCcSBtjD5fPcy||AW%xUg'
    'b{eN~7EORJh12fU!J8MqfBaVIEe6R#8BOjngGtS0u$htLe=7DE6TX@-;G+Bj%yvKL@<RgKw}#O8JX*kk5=}oHFXldX6a;mG)|oRkQWB}qs$?FKe|B~O3^Lk4_~MOl+4A-tJtCiV@0)7E*s2qzC&ZW&8Cti)3Aw);'
    'uz0*%p7rx+mZ@2f;4r9|`_DkOVJArpTr*CuwsHQ?7s!}o&(YO#HplA&^f&V6i!mFrN3vqbqC-!-U~bK_W4p_`H=?=7E@%Sm@Hg}_Psv+Yh%#CM*S&diKEiVktuy^5hVaS#EAN4~+t_`G_{J%{V)4OzdiEqxe0e%U'
    '{a2pD@^l*+6&e|tMn;83U`h}ebc11VU^Ac#Dm-)3C3aGdoOyahCnq{<`#7ssbVv#ha|1nHc^^~rE%iWsC<W7v?Mbf3A}1*s&WEa(8?IovMIrSs*QE=<%yN?Y1V;M=I`?^W+XRk#Seu@fjkTYS%~+kmU3c}!X*8xS'
    '8RNAXqc!Qy7**@nT%mG|c3p~zST#n9Xkh@T7$jC)Hb@Mh0DFpv`n5&QbsV1i4u7WzP!S<l7ONj$=M-iJs+^P28gH}4RWd++@k*E=N6Wt<WY-2VLYxa3<Fc?EM;kykig7kz-xtCabztWarNb8#gB2!+C>p<^T|&PY'
    'EU48Lr$nKG(1=4Kf$W7yCWlD06AC_vR`QRPY&K0Mi-|c2=Ez|B0`3F1DGWx6&6S!Gau5&5bpczFc1!I+`NvWk>2aN{|CLKQ`-Ulw&59L)Eo&rhb4Tb_W~Q?@j;2U&F;<6Lj>5VSrZDo!CvAf&TU6xw6z!EDs=QcN'
    'z$Qf@v!ZsjZ7)W*Q9chiy@IC-8FE2Zv4lOJi+}7Txcc@0jl8UQDV{5C)PZysP2H)0S`VwH<Kd$UTjgF16M%Cmh{JYx4j7be>L6&*Q`urX8p45TY9}o`N1R!K^9062jg+fPBvpaG5^BAQiuaQ0Kr#E_0booYE<7;U'
    '#ouh}ZfM%p)T0qO)vE6caZxKt1*R~lTs}I8k<d`Ar1Wx1GZ^;LYajlr<4VXB$D(6Q%gVoogZT#jyM~>6v*+Qldo2RrfXPYSd%z7`?*Zzsn*~bEwB-iB-88#y{Ev<g#Xm+JCUt?od)N&Vya@H0T9D$a{6W@fwz@&L'
    'vJ-3$o8l%3Vrp`Ib7j-%POG)uEOmnuX`*nqi&;k)3~oTUL)f+<iwl}H1POxOP0_q$*NP9q{bR)A4An5``yufiJUweFG~*nAkLTh!%C}z;x(L8a%^;$v;>P3`(zh6s-yKXH{NFjq?6X4!-x{VQ5KiKJlny|Ot6jnB'
    'Wz9RqnT_UCii_9skiypb7)TPowJ=)&yujyiG=pdIV7jwKI49(KTV;AF@c@1YVeZxm8qu2bxgqjnDS&h#?8vRee-3e}>NvWEcM{w^oW*mQGZPt?W{{$U76gK-<F9;<-Pz%(6l)G!LLgijpzxi9U373IZJ5Z5)eY-F'
    'j5x;P!Cip^<zT`(b~yv<ZZ*4x&`6GbT`?C{l;47IE>XHwN(Ip)NE~W{Rt`v`XroS+uZ`OW4{&)P`t?%fI#w)WXSLo1R(6haSxzX1xJ-$&03Pov?g9gVSCy|^Mh}fZV|Vw7*F=$q+k%HxalPz%gv?Suc4+;kOgzwQ'
    '=KBasdhZE>%REFmuqjjbl`%Ol>z+>pi{fMjIx?ryO8J+8_G;dRabDtBt!R1J1=k|LFl?Jy)%<Z?%x-H{YBVgS$&ZMU?E`hB3v{(1NvuW!*tAXDN#hdAR_QQLa1F<_MItRJ0n0g;0m>tr&P3CeLt?weSDVHK8*J}B'
    'L&NBnaF_$rnZr3iuJX7|O;RaqIBMz(XFe{3x@W@&N1IY?6*5g&!7N~Fb=1zHuz!<ix`2aErNj^kXc=Kw#b_NYQPOD|QO<Z#I%};3F~o^-H6gYthDPV8L#zX6gPux$;!uzd6^ROf!M0(kx=#dC5OjEFZ^O=R(NEj<'
    'w`>$<HBJqbs~6#4L2Pl2F&MDhkf%3|E$g)7vdTQ?nq+nr-wD(@4TtenD2l5P_CzRdTOsw5J@+oqnT=Ql;!!qTmkOHASEX{M<F(la_5>&nB?i@N6N7`~)T{!z&6wF*JJeCUm}A()z8oUY(K+)niLaPc0oHNUJ0=6F'
    '*bj+-gQ>tu5n-tB)@G=lrX2noQxB-an#ZsV#X8wZLJ0;+%6>aYc^dYTbCd#XGgwv@;@i$%7uE~@*zMNo--qf&vs;RbjvBxbNH|7=?-LBD>A(evvKBUu;b&NHZh7U_^|J8kM;#WUC@<-zM~b!VRfKz9!VjTW=dOe3'
    '>cJ5&G*4$n=?xE5#`Pc<P&%{-Yd19#1+c6z$9Tl<!+7ZxQFf6*paMH=Y)D;HH40OIU2`{}j<IKnboNCu$tEZ~URMQGa{~Aji6a7DR8yDb<mAW1)tNz7$z2v&R>RQg^KC{sh|q4M`3SH{{e1!*Qa+!?Q}wP}C5f9B'
    'swHJsaG+vfNf4szmKXIhIN`Us;~j5riC48X4^;wm2fFGiWUirDtNo>MSgJ2Y!)ZGNq^h2(pl8H4Ad)Pmu#5#2VrjRv+a<?zZn->3#UO4gDbiv}hDa2nQ~C}idYO=!BCS_d^)?e`Nk;a~>QR&#p;P8^QK(k&((J0w'
    '6{i<h@WwSG&YI|RRIP15L5Z8C*EY$nBD!)$S@%9rh!hK?auY<^I;20veXxc_M1B@Wq`B7e>$|NrWJ;$#kiDtzbk@@THL?^_1(238F$T6;Ye`~Y=`kP?7|j)yeFat&z6_MOj~D2e*(<!p8{iJ175?g)WHebnoyC^0'
    '*27p7naaDtX~AS4o4Z%Z7sl7!ayg5!!7Nhsz$Q%Ag%YJ~HJ<2J`>IJq-VTQ^Q`j}t8Hhp-S61<HKD3om+_3h7V;|2{a90uCZeeooK%Cb--svSyquU{<oc)zYb9k^W*T@#YMFGJ?7j2~?a**jVdW6z0h$(7b&46T4'
    'QUg5GJ{MnmA^i1L7<b~XuQHrulzcTM2&^K&y}-1nLimK&i2SMPd}?+J>(w-{V>eK@s_)p<yP`3YjA<>sBWuc908K|I(_v}-OCF<;Mvv=XX>4H+pT{ZixAPkxn9X@4WW}HfNY!vi>AhutQ1oe|TYMeOyz_XvK#7Z{'
    'kojSfgt1!;1;=5y&*SH2ZE~bvU>z>wDGK_o(E&w$(<v6>89qzeQGk)6gz-xT+R6yRJDihy!xuX*7Cd%T0X(!wr3rjHVS(TKg;GK&03jI~Gx$Z$74I|*s%ZB7A}{L}Y1(lh(egLLgbh9iP6$>hOSlMhfUCA5*}5IZ'
    'k%|@-2+{{KSo=g(qVd@GSLs@`7DcL7_txRBxRPPVU!z`J*n0WUqz^app=mzsEvxacP@`MAy6s#StKJl<s7vJ=Gk~sZQKeUjXxBy&>Tuv(rxUnFaze?lYL<>~&f(A}!&N<h^ig!5QXmM5Re2+^7a9L9maw-N^<ckP'
    '2|NkN<jnHn;ob}?a%JEQva)@U5D6x0AT?tU60PoZlq{7)qV@`7v4Db>myZQM78vf0$D$155D;yMgT5W<JXC4=d?BdX2UL^8;~vwuEFI=o7?~0`nr=WPH&Q`UboK&rhetIcyU)4QkV{$ic0jKqy!4k0RQ(HaZju<2'
    '%Iww`Ex4o<<_G?JTXOsH(>3QVsn(yf(cK*jWT>E-_bDr%MtnS{8J<k~m)odm;fDd|Oq)#RDf?ntJ)npuYWda8y0I`3G8$WJdHfuO4IQFQ;i@5->xved92!ZNI^p_`xoT^9Txy(I=mZvBl;?A^ENb|nW?Kd5+b)$;'
    '5LgJz*ddQWBg8N@3t$gIbPa2uPgSwNj~3Aw!4E^4Ivy!<0`e2D#Cj&ireS=o6)YBFS-6;TS@Z!-c#*rQfNJ`?zGG5_DwO*q)9HSAYa;mgYWd*QS2HrxB1sr#>liHKJDpX}bZKL|a&6e2VP$!=*j(35;AX{ODU>uB'
    'xlE*Bbm=H9GXDUePkM`-Irh4D1}Ko9hLI1d)OYr@l8W5SsN@W^rqd>4SlvV{R%suJ=P|2Q1WFkf*{Qo%$#u<$0s-#><5UPxj0q@!oAA^n5^Z9>4?zc+=X}c?sRE?f6sy!8w1?y>qOOc@hq$T5lNnk#ufN%~vcHkE'
    'K;k$bY--Rtn`+s4N>D+`LMVxlMnz!7msHR+&SC^Fn%}%6H4LVy!DxEPb-Q+%PqqaO+coy`S7l1QhR&R*%lzWB&{WVoa8L)`UYq;|^kYz-rNVPyIpl5D?E@7_Lie(ADX2!Za$(gh7^1>oS|v4phnQS{-l=hpmXHp8'
    '=1b?q)=5=$t^><%SVh*VBh7vAz!a1V+pc?KOOjx@kgcT(oa<jb93*9?qDlMNy{j<X6V;A}Y^f24o^Fk$B0kwn!@5{^<Wp;fb7z(MHj;d}q0p)w@(n|1Z?xy4aL}@5#eG*nbFuF7@r17zv3KtZ_lbc(-EnLH0WZRg'
    '6ZK%)t{9}}fs$I4bsqkQfueT{Rt1$a$3%sbZf|N_V69_5Jr&x`y=J>;G~g)QWJ3y7^ZddtvtxAXqe@q(nLxN!J!N-rI~M^*+4?KpYJv*|c9PytRlum_rpy$8A-~hS4<A9|$L6(HG*D)uYb1)UVy+F|Ez3&MP_W<F'
    'VnSJeMaAj}&t;;bg#c&THQneWUEu-n!4y!LeY@^Wfl6WA_EcJ53PeB+ScC)}nUK~{$3f|?s7{Qh0Z3_hrS-b9q>^W4OQjw~0Vy;?RKWa?+ZXntbkkBfRJy1(e(V_4R9Cle(*!D2rZaU`#Re3W9hGO4rCX&#mZp_n'
    't-Bm3t+Q)))?E%-@KFUK-3lY5P_27<Y#mEwxl_7$M9f+G%AK<ErdzkkQrl|n6R*%1*_c?ZeNqHm<EX7Qv8Z8Oe;cX?+^$%b7B)uF-lNBD<q1qr_#Lyl(MD9C^otrLJu4A~_R{S!Am3uGv}@by1O1l7eLk;kw$<L1'
    '7W_mP71<DRu!cIsIU4`c^|%%ZC3OI%apkkj20)*0M55{v!CVm)yQKT2Mt3tnVgZ0W4kaOB7++!mcZ@5Fk(D1VVmL2kYbnJlcEB1y5EK|O?M{^@(X1}o^t;u{L~D)Joh1WXW0eKSDtc48QCdPvB(APz1}Ut9tee}U'
    'wfk^z*O*PuYmQRCkwurW&te77V!L+o=5w2au;Og!fCBw_xIuCNGMPvh&e<c2mgdZmt;*9cP*SaXkh%F<nax1=xi<OeYx>T%2f7EduPm;NtxoXN*YN2C@PTNDWC$d|s3)%N?rlCXO^3y=K~)p1-bRHORwinRw|$jm'
    't=o}8SN4}+m5e=PR8>UYQlXwjmG49m1m3N#mHQf~4yiSXN<poLR{ERo7@??Jakb6|55+J-XyjPZWTsn{RKu#LW{MV}6^YBmmi@J$Y^)5*<&Lr_a$3UX((uldMSd|IC({dGeBaCQlFis{(Anm&B8P>w(dyn~E22kF'
    'C$O{h%tAZq9?8z8#9IgujY}j8uSPXvW@Ft!s77gZHA}e_(CY81u6`k@Q5As7ZIvg=>|9!z<QQIVW=&1a_}H}z<XA)G(BIV!+kBEFi{Pd|vCUr4)<!k{$x?$CH11v1e0i}g7yB%Veb@3T8^n8{{&+C}QaD4BxP3lP'
    '2E2sOM5I>fzp{Z4xZ?8qL@WAiIMfw(Qn<k0{-$?pEseXfIBKbyS%Yn7G+boGYzuN3Smz(1Xlwa_>MM&s|CWhTOSbca2edoxnBi#qVkWuTh?2P+*Xs1%mR&V{(^{Ah(yr<IO)_9(n2R|$nGoGRc%VnGmWBd?KEPNF'
    'R_^QoY$$n~;0AoD(n1l<oH{LY<RG$e6{Fm2qmZb0i{Y`d@F0st<Y&b_wO$hDnmTbKqFC2b<qJOd+U%b9RyxU#binYoD$W$tNZ46$%+*$dyx|5u*X_^;i)xAzL&CSA&kmgU8O#rbnZv*Avlta1YBuPbBQ)dKT(MaI'
    ';~O(IbO6e6O@!BYAmTFUYWG1ndf4YvjOlOb1?v&wzJ$!f#lgMPM}8?sHg}3zf8fAFZccOy4D~3<!{InOXM6cVJ^VO{t|j=n+hbQ(hZ1tk>m2EqczHo@^wqZRC<qMO=T@LM%PVm9Bs&#5fKs87s{l6GOi$;SZ`gDi'
    '?b0xSSFSbhbeD?ub6$xQ*Me#wE2RAo0c=IH23&0!uJkAoMK6HixrRuV07;i;92?cL-eI+<i2X__+FGeCt$S6<sMe;MD5x&_yOy9qlTFklYkj1!O6ahqO?IqcAo{f_X&d)cg$;24FI@Cxv9|Yujnc5#wC76Ose?ms'
    'Zgn(pykTFK=rT&ixM=K~zM}rj>abH&N_-><w-+rV*k7>S#RXGb6%QF@Eu$3>aYettUrHaK#AwpQ!b8;m(IUwbKIQHldoL5olPF(6ep*WXt7$|_(E@l^pf@d{W8|QxC3{L2i74gWMk}diT{jw~Ys<z7ErP<yE%R7<'
    'Z_ZghS7Gan6S68p+H_~EKG7S%?$XcZeRU(F?q?)hy?Y_|8s~o-69Q?c0?yph-0nq~@);v8ksBVI20e-p!q;&m31sM8usRIPOIT%F`kI#PC~$|7Bm+POk`9H!Zy*i^vqWOy=+HLrcOimJ23uP6kFTsCGawr3e9u=Z'
    '^b11Tz`uN{5IA8xy7$s7hxgT>Y;X-=We-A7ag9WnPnE6BjCQAPG>%P6Nl0FliyWC4OcsC}jU<b<ii=`g#Jrsy4@2w}dL(8-I`ck8S9|Ude^dCfxu1x;<}ewLxfjbv12AcrK(UE#k&}FL4JUY~k0f5jEjMitzkHvZ'
    'K{C~>dXi?`pi5i;?L~zMaojClyPSEYB|GgyojQDg2vJZW(N`ZPfLbI6DiSS*=h}j;8y>wr`1~;Z;_&qC(eWFy2cT${+m}EqfsSuqHdQQB4E;q8d3nmW{>&}+Xr<1@v&c$ao?Ls;a7fmiX$(L531YkDUDw*tgX@p`'
    '4{kn&>NmA^51t0_G?e7RM|66Nz4VAu$YVgAZHltP8kOfV&B#<Orfe9*q^KWFFC%=J<dP*E-ok)D7>wE61mj@?v$gXMDDFEb*1~MR_yG>Cc&q|jjcziOlDwBEM-suoP4p`sNZP7>OrfGN2sUbDR0IOb)?tLUXS?Ol'
    'F{qrySQ0%{$wI6C97F8j11oQ_;ciu9&8HQuZJX*>TKB7)9}$Gr&Fwv@25#GdZo3E`j@XG!(5Zl6+W}!rRrI2gl_uB6KJOGUbGIU$wQRJLg$EfXqjk3!*Rmr}2bXrEA17l9_r!+db<eytCg)ufZ*uB$&@XXz79(77'
    '?Li9mO$(Nlw)6I7^sQ?oPbU_0F5FQ2)--ze+>>L-;#ANVAFcW_&*zF{GsWj?0+L)lzJXeUML*WLxOzzrGN8hdI!tlAZ9yuM>X0n_Gdr+)g;@!=g=R@JKwMy=ubqDWX>Cd5TB+j%ucZ;e5r@+)7n?N8z;wWI-rmEI'
    'ZDr&%cjXVA9I(UBZ`!Q(C<UazL>~hWpz^FRd+J7o&SY{#CSsg92p+`!1g-gm_WWaj=N}`IO#ln>$reRBnTl`^n2YGiUYX{2pP?d70?cK=D02)wv}fEVwgYB?f<aOma^a`}PJrhIuXj*ehFxO5N#wv0Lg2v|*}z?A'
    '9Egc8#J$%5%wXq|E>Xm*bHSe1;~09EVf+ZlS58VWCCRzOY)zQsO1_N(;c0&rnI9F<^Oony9BFqHEOC~Q(;WUTO*eg6<ymzknZOa__6&+=U`?4<k(yPmPOB+}dt~7~0bA{;qWIkpg!7~~s;sZ#VfWBcF;foKx}3aM'
    '#f-;oX5vI6qx2aJ0trR<d7K|$p%)m2%(v`xCTxZ&3qx6->^63dE(SUyQw#3XF<O9k4HSIvTDZ_{cmfCh@u$DbC;tkIR3{K6Hi!e9UE7Fc<yt;ZuB{^c!YYBSM#+?dAz>l?TnLnE$}No?r90yvjmcu1C$sU5uj@J>'
    '1f|g_I86C6G~!sx2g&GWmSR9g>Mp9Qj(h4zalZHU_kk7B3B-74UNwndL0;qFj_6eoIQ|7b8EQc5$Z@NGkzwXY5LUB2RHAdM7!SFR=aq3?<vvyTLipNT@)dad6?OS*WnBY_1nZ&P<J@151&q1?2|w|I2n+O#C_-N_'
    '(-<7H#kvnoiHC08ruowBaxkjh##4+>*fTfb>MC5leW>5sKH=JU3mfO9&R$fw0@#a$UpDni$#jQsM&cN{E5SCc?B8n7V!N8O7q7s1i`x->NtUyv17Uqgjf^A40~EgT1Nm#=a+0^5@*>s0tc9t6#roF}Z5YSERg6g!'
    '@xuoFhIcrztTV?$3qErEgBrs{o~uvxLO%XNI{u){6O7})gSk_pQj{?XRB_T1)Z$4-5}{vwkw*u&ynz{DBGz6*#1u(p`XW`?jgtCtFX2@R1o~#PJOrc_)Bwo&GzMvB77{*~b1|N0A!^Z6rBN!f)XN)YTS+Nqt5VHb'
    '@eTJVtZKk29;S0ZJ_TT^SstYB@FuG$bhYqeoh5_-x?${P{sgEWtovrXz5Cd_>pSn@MMNihAWsAHL`q|<6v3vvq%G9g-EFD}XqPGingW-XI~v9J$ll;yjdUio&>KVi3%5pNfL5F`Vw6;H)Ic17mXy5G3V0t!{dht#'
    'E>N+krV!CTkB8&Z4O)Mk%&L$5<K7SP<bMGLSUu7Ep?mF!HOZpGt*a`8#4g~k%*V1c{MEp-(h&=jF?!8nGe{FPN(Xg|vqXjicTAlW0-@GFIMJ>&*XIUxk*iVa-$LmWWVh*PB`<+n6NR?vNaQVtLtJMD{6@6Xq(@vT'
    'b6FFl{i^XGpT^fY1vtc5k^U6#?S+?|;$^+IkaQz>;0iQlVIA3pWl_Sa<O6P5qpN2wnkHGA!-`A#p_~!MAZ&%1mKFr{KB5;s&!rtNV5tp`6CDFd!+LperPYZbZ4_nf<V4Mh{X7uckc7snWP6ri>68VS%%xdHVI_q+'
    '2Dhwl;GMZv`{1md!nHTv0w6D`FvOOauo<LLe#R|fg$^Gi!(lv+DQcOjSXVX32!_QEbzfL87PdC19q*7&YEsX|>T&|6$sHZaWAqj8$njq-pE+EI4Sk?|%~7j~|K63-fnY;bMp_nSL26<RR<SxV8b=4b+~}!vK#NBj'
    '^CFB^F&dUpTLFiZ-$Oa1Eeq)(3F%^$$DCPUT$@woLr}0?$%xdK@+qGkpT53hsm+qvl<k}&C$?=8qeztEDUrjl0H$D{@`<<{;y6Lf-T^jDOOFMf^Zi%8J2472vna**TOtI>{l;xg94282l{C(_D~vjYYReVmT7Ry1'
    'UPs*T>PrIHL5f!<SsywXBvEKyyJ9<iRYE^s8!jt~A9YF1#kDlwN1l^7xpeVJF&t&}6ek~{wI&`)T2HuBC%t$uh{Y8kQ?~*cZCcUjgfQ3ef$qfgR<^fmX|pl3<^j2UQEr8@!|3jrewmwAH)Ms5n!+FPJRnq+?+DEF'
    'm2L_+PKL59K<R<pp&Zcy8??oX=UOozD)oJ6-#QnfXV%8)l{zy`U(fjhdzOxwf9n(8$vx&0{g+_bCS#7K#TeI-!6=AH(s4vJ@gk!*-9k75<aM7s62Y+3b#I#uWsLAb=8J>Vqk}hR;oFnL7jJ314AMTDio<z5jpoUZ'
    ')tsRspf-&cb21{z@>H7$FM0bi?H^9hPN8Ge1)?t${Jrzk`}X$BqfcKP@9j39dS4`GWN0u2UT&dd!$Nrw@(1B;o}gVLQIX_QOx$aO_&l0^edrFiKR<Y5fP1(7CeFR)?%w`$?>jLNIqwWTxs`S6b1Nuy<JqH4p&Z1$'
    '@nSybgpVH(Mvoh=1a(u`t|IhJh~~KQCg~*|%O42DN6kG?H}6UF&++POob;1i8+g!#!+LnNxw8OR;gsRnU}y_@OC`_BU}1fz)&4jd#j44bjvA%!{d6qh9Pu>7?=M~*eEZOQ74^JV94C1yu(8v6kGt$8{RR9NUhiSQ'
    'dmHpCtTudO^InH^V1bha!x($gvIi}1(EUDkzyI6let$*xzz(D!Gz~V#IhIF*lWY{tf$O3i>t5~6tJen8=i{5%sI+1*(G5d9UQD77HOaV|puL2=gRF3Yo4_%@&$!ovaa!o!cxT|Pz2#wKfp9*L<LQS4;`8?a0?1bU'
    'Az}D*V;JCw4#yvCzaJga65MDNV9(J9?m6=MBM>$)6n=C*O|kI$;rBi;5WV{XfsW?M^x{J<0CZm(q-s-Qs;1Q-;3;`d9suDKKa3?*+6Mth<=yg3FMyj%y5e0&v$AM$36ynHP#gvbp5JnHPjrERm2rwk;3P$zVYa;A'
    'DTra{#Py#2zTxV?P)5^jsJF<rFSfHBI3}eDK%v**V>uu_;G&`LX#8~#2;I;+lrN@K@$2@dSwld&G`X&4;Kq|_d>u}*`G;D22t~ZjQt>!F_fEMBk~T;Ve~v~2$lK#>r}%wt1Yv*n+ko-)+tU{(&oP0=1U3h5<?{%H'
    '7z$95MIfxOqN%rf(OjkD;rlM#??T<}-%ZEZ4dw}r#@mJ@$o6p)SuEK=Ut{+iz4#7oosTabP@K-g&e1>G>we5;#Qiu}-YsKy*8iaF4KU!-eli10(Dl+3GaQ*C?zzSOWXbuO1DAU^pG-a!fOyJaARgLWpTB<X9ln0<'
    '^(dzrrc^f&lda8+HIB>SL$t$APKNo{B>>0p^TijQ_}FX{<TR@uHDC9psW-csN0S7k;2RZOY5COJI(O78E;38XeP19?;yEC1`!y`@3lzoB2w{Li_a)JWeQ%0dtX@hs)amwcytrOR#jLY*lFnzaP119eyFS!}Yqrrt'
    'O*)z-WsCVR>c`LNR2{@7%<txqo-d%{2Rrc#fT2UkaGSgEI5Fqffi=yu#ru*LG7TBUS(3Gi=tH+d&6dFLW9(+dNdPmD%~5AyB**(G2~MGb?ZdaHs3|x-efcS>2(I^M<3+Y!_Rph1lFi?<<FkX8$=g%!MSA{45?}3m'
    '2*`^=k5Wgu$%5F*ZN7o-1h!+&hqhc$-b-Bqwa);eqnEFD&fve->1)#No#AX^;#zWfq@LG<vX~~LWbSk!OXWPCk_SKO9gEtvl^@?tlW!lLdY>MAe)!6J{R$-Ql?{}zllB1|^aBCNDTc=TDGpEL3?>WTPU5-u^6c#i'
    'h#9lovl?DMQw+=3RKYEW#cFtSSb!uC(yQMFFs9{Aq4@v`-euqvY0+AEa9e%-slx3A$67Sc$Xy778Ir9AFmu4mM)U3F?k)wd8Q|5aIa$|2I9FqYbB>huUk_A+vez%q4t73&`Pmzf8wP<o?q85~-hH3~jw0<Zvflz&'
    'cx;U0ZN9maZr&3}JV!#!oqQ+&-yXeu`JMosi?sgXf$EJhi?!AUYz*Gr*4uZdUmSive0%)v^u=NL`RVby6TX&t_-xSZJ%0Ekjt9@4MUUg=v!~7e!@bAx(<hG~J$%yd51&2mMbXn|PvYj|VRO)r_J@z7xWC(sAMQQf'
    '-NoCj&zso)!}wXW*NdM<&E5TIu)DwCAM8HbAH?yK{%-W_>Hgkc6g_$}*y}%w_WN-Z!@Eblr-NtFqsP(iV0UO*>P1hRd+@$F*nbE=qrIs2=<(y`qlb^4J&t?v{?onwlRao_H`?t#8$N6T+|LG2p7ozS8Vuqmy+@{{'
    '$4`5YqdoxA-+lPx>7)I4zZpM%6c6?v4dZ6q>ksyxJbn5M{zOk6Me)PIaIZJ)KYI3L@5!U)?vwb5)zahP;Ni38)4jd@N4rm>Xn!yqMvwY&zyB!e#ZUH|yU+Ifj~+jK^cbLN!r<a(PaY4SK707I+3Y=hN*<G+lNNoy'
    'xAzzk)g1I50?5POU?0Z$^vQ7WWUs&9eDY}T5e)j_5Ww8)KaL(hdHN*Y-HUdgHXlE1?lmK7>0#||>)o4!FAk1g9en!gFg!hcdv<(!aCUV3CVYK(_U+@BJiX(qoPz~Jt=n2gp@uQ;E&7~-0DnFwLj~r>GIo_cCa)$6'
    'MoNZ&7{_Q7WnMJzV}7STh|uwlNJf+xAxAmnGM*>Hgsk!9&ITwYaFYqTFt`hKF@tkIj!=H9jq-e!wRU#SK|Ejd8X#-!9KHGOtMBi;`tF$ken%W+G+0dIeDhJfNj;eOIAg1rRE-IQHNLCy$db@-Y7%Lv46z>BGSIKC'
    't{PKREW`vpMsn=~W4n{>@9sa_-`?GOw7uJW_W05EQyxl`r&)-+&iT!~hBHg1Q^s#X4(vKbsRF7nq~~6N<45^50{)b|Wi13iqlT#`7@0bUlEh>jQyA32LOJbj4zper13Wd!ovi}zuzX-S!Jv|CgcrKl;dVX($j?XY'
    'zec&1(CY&vjm!83CkO;XVD#_q*cV5KdYZ+34q@Fg8CNC}Qk!{xifKU#-I8Yzg(+r(jc^EJ_4uY;O`W6hIfz~PXwp`~fbazXnIiy*;&>L%p@W{>%#WiiYuAi&o2*`HPUN`iZrR~!-^$NoSE_TKISabF*aXaD?G)l%'
    '%ySg*t<@MPHrlqDuNarxqcbQQRLghE$>Jl4WYOchAN3Mc^wNByd3<`IoV)p^w=VJGjpNjzx(bd!Aig0IH60{uf`mv`ZrsWA5*0cTtV>wA10}5G<Z+}uyXOnC1maHEWz_-9itxlBsvx6T9;0P4I?N)}L4)q9<7@$B'
    '=-RI}(5}STRzoi;EPr~D)dJg5#F(Dm>elMi{#}DGov{kd$5uMhLboD<Gn;-Z@Y$#YOQXvvCAiVI=wjUFj<f<=9fzO_WS*cxK`1AlR1B@vLU$v8DS--^9o+|N?FVnKFdfJTzxn00rJ^~?3M&sEt(6jTa3^a6ws|93'
    'qsGWC)@s?{+SgG(N?9mY3|czMN_Gz1C|W?+t9UWh2bSy7P4O3Qw6$|-&#l8qm<$RVj*Di6Ey8b2<ZLVWu1@q#^*D`1?H0SbYu&1>$M92kcvq{tr9Zk|y{VS9^AQ0NsKbISGz(<L8JkL~P-iMYE<&rx0PseW%rE4q'
    'vDSlY5~cI>>hv<$Pn`%bxdLE5TYQCXLbtN9?dTtwwKmiuyjy2Xt}<HlZrO2$AufkCH$K!wd?QIqDj+CFdy?XWGZS&%9b^Xzb=T)L#vB1AtdOo)a%iX?>47se(bnTJpi&fL_ZcR2$LK$3S?5^5E9{p@9Y=toIZ=7o'
    '>3}q8%-7F?^7(YpZ{yVlr8vnBekn@?Q`17Rt|rQ8{ksk%x3WeOR*fTE6I)x7d5Ut)TFF-A%>+)|^LWmsF#HX11DIn{(p0Hebx*>3MH#Mj6}kZ=j1jb|9+Py-PDf$Kcc327+rt9lgrcXO01tiwBFi;Csyf+Cwq*xU'
    'J^bF;=|TA7)xq1hM=uUup%Hhm#n!to56%wW9-fg|^T8R0jlgUL#077wh(3j)Cd5Ru@a4tv%fs;P(dTat&fc9KVq>+ft>4jT*rLBB1PizQf(Q)H)L@phka>f8d%Ns}XL$!xEDle(r|4Abu3w2%*Z$_>yNi5Ll7%YH'
    'j4r0SDMjPaojEb~I8maN?2&eXYVDzhNs6cskN2&Llmweikvo)AtLyy%vk+F+RWf~c%Hy)ZJk7X_e7W=jW|eb35dD;lG8dt~W>x*~^B~Wo85)d)Dl2)5b9#Aga&39c&?l6zb!8cKoXa2pQQf7Gaqd&J-!*Yo1G|+_'
    'e7-jtp-z$(h9IS*5SHX&=;J++dj#Q|hV7$HFDe|>IYtB#;)j&slnRy>41Q^1lpIbojAcs8fRoJY@76k?FPqXwJLLZW_U;t5s@V>AF-|7vnHiMEYJ1Tyb=+SEg58KBhS-KjVc)2Ya?j<^l)8Us@8B&Otbre+kM=km'
    '=kv{vjB*UEQjY6gz%ha2Yyiwe{^*fnJ9hQ_7I%WP4s=t+y#Vib8;|ZWk?N}77(QfW$gT6AdR4p2;uWk==z?3PBo;yMTRgs<4{&6Eqb;B8*NW3Byv5^P_q29$bnkMCnk!jKPP&-QC=8@WB%ch3DNR?!3Mx{9H66d@'
    '=qYFck2evdrkkdodx={JB|&sUax!Q}7zwMrm+6Je2uf5nOueqKUL?(B7TESo&e<CW7-cy4&y(CI<R>aXgqyx~`Iw8HVq}m^$896#AK1iV#Odl0^C(FsPz8|P2zxhqocR<WNsBrxL|JLB=rg9S$bjt#qpY7KZE+#N'
    'u%979AyK$Q=a$a1wqL8`{%_%(K{RGhj;XjUV(zaWz>aRjQ&g?`wM9PMep)L)h@4UtY{Ee`IDDYLbU;C46ko#(#98han#SZ!+&)Fy7FL9c!f{mSz)Ji??!Hu|k<MU`)#g3U?Zk>nH<zJ~w3?t>syY?9B#}^`ToE3S'
    'M*_M>*NSomL!@$JP<w%^#G>y@hl$7y(a?yrw!TpnuHxWIvihpR5<)LUv=MSb#%-vAwK?G06MKuKp;a*>)Y=YFj=)Lzr@Wx0N@I3Y8IDoj>tIEhys$KkB(Z0h&S8d^Mg^?J)5~O@PEoh1Q6}8?(W^}Y?m$n#$8eld'
    'vhPRNxuaK&Fceoh&fBMIOT;<xJv{mT$;;1<-n|Zw-h6f()Kx^cLZL#D8<28Q6%8_g%5T`96zYC}QL$Z}1On%d7*{mm9hDGc4s@Zm+*n1~!0z=|ooM$5KVUgcmBXW}<-A%MZc8KrPwRdVu@bgcT14Q0RlHR%RE+N#'
    '1*mMh!cZAPDh8xDjC$lj7Be}nH5vpZf9AFnH(<RBRzk0ieZeDvS^=rOKk#?J;n6RZYqMn2z)uv%l&0f>4TT-~f@1&c25@Amzc5d^{f$sgy~bHT0zsVmu7X_h2&0(kteC5Lg6Zv@B~z!bDg&Hk2wAbhbh&%Rb<9=L'
    'J&F$n)sZxZNF|6gH;7(fcFBA)s}-s=2yi(j2;9PVpd%P3f{DD115|_LBiV*}40wmcQEXdP=aO>--MJ{F6F{LvF6-1u6p`y~CGw<Xhr#Ivbdtz`^g3Aw3i60nQz}Q$s&(^`S~qlKm@OtCH)fP4h1N|0th<kbBIS2S'
    'kF;1W-ehR>Thm*1bXEafOK}P$ek7Dg8=vCXkB)sSCWamF(@i#0r|U%6J{gCFX-lRHK{;1JWf<`gDmLLLDzgP8q!tP0P+W<#om;5VY3+6I=$DlmXVYC`Z++Sk8B?(({0vSnbkr*G!7H|H<ytpo%rb``tDU^bgLv7l'
    ';Iert3wG58u@b3OZ<ETvc7|D_N9NP4RkCvX1;or@$9BYPwHo<-)pEqFCIw6sw;1OofjPSeLJDm-nB8o%irtaXDTorJKZ>U3Fvj&;ms;4``kjLM)19dPg?;c~`s=qIalc|Dk$DuGg>Px+iy2?rg9HasFr4)#M=y7<'
    '_>khrp(1uM^QLi(*e(P)OgU(wb(U^;YS5DKO+8>SRr1aT_FYpfef04Xo}NSIAoHHb<FQW>uL@y%<U4P(UTwD^oXO|mcmX@riW0%EvpBx+cT3r@hW%+iHi88{K?{g-u2+|LuZ~}QCp`R~_Xqa#n@?BdXu3hQ+v-F^'
    'OF-KIl6P+kY;}c1s^t-uwySP~Ema8?D&D5JHz}ya4=Qe);FF(V8bwlUas(=Bok9mC*$|bAP|W&*8lg*8@9C=<f|-gCyx-7iFHo_<A6KznHCvchy0*nM`4J_>=%YOe1uGTR6R98k-Fh=93dWQ~ndF!s^?kW~ta}`#'
    'LF>zGTLEa$)2_&#uFgj}+D;5&49bF*0_hNAzVDD7wKBY!!fDx97CAY-p0T?eYF%L}-E=ht^&f{+xH6&EF;k4XcXcDK%eX9rUN`IB4g7b#2mkI-sCmA?wvz%ERc#ai#$5`q60Aw}-DV{|nbsX%H(?Q6lbZO776%rE'
    'VwACQ7jVH#MuFny%jse%8so+qMB{r;HUJzvZHJq)DF)0Ts$?iO&ox0GY>0gy2YTIfb#mAq0;^_o9k?$vm}WJR?Ht}X)IE&EF^*=e_d@%3*tEF11U+7HJk}zDG1)ASozFxSLeyVvINIHnJF;pRDXQC9zeZ=?G%xKx'
    'OisX2+I4OMy3(>TrJL<{Vsyeve@PBal$wLjW65>$oN}j|6cnt0WYs61+pBXi)C(yY)Rjh6W(4)Ek&hOW-ZV;Jo5;m~=_W)CulyyDE*qyYg_nW3ZM-^o^Wyi9-wK~TQ{i)OFXEdPP@$Y+hber)%fP*Uca-^b>s(OW'
    '<GZ4oUZn&<D>SlyQ79+-H($YBSMUTjDKktPV+J$E;d_lD{Rx;Ki5bTDDCvdq5?$$kB~yYFEvcI^3@JBP87`L&2~2rJ>H03c2i}!Mw76#1PW8uex@!DI-9ptQ?h_ud495mp#Y#X}r<`xWgF#hope(Gz++-HkYu7b{'
    'dOK~Tf%^(gZnIg`r-&CSBtS*jD&_)AV6qNnaEHah0mDK=1uWiNZb~p9{sV5{s%vgaXi{Lh_EwS#!YHEzunW{Ji>>`hpsOI(62l1RuQrMob9Omnit*f>#34m2LZD$wp*<8?Y^s;Ft_{9|vFUwNO{IP7Y!mZU%c^>8'
    'uepX~U$J8^s&!<hObt#4a?8^ccG>hMJYVEFWspIeNHi`E&9vBYHDzUYZ8A?6-pl)#riSTx?pd>~x>9-`ed*>d+m7Tw5zM(4hTGoLp!yR>G~Ro(zoh2JP*Y_c9a69Y7}uI@fLZotDoB`1>{?!a$NMQ~Ghj8CuDa)^'
    'XqVZA5f^#Xn9#SE=6Q<7n8@k+yN%t4ut%ejM1gJ++XJtkJ>l1Z%AKPRCjdQLOn_OMUUY7-&=rjDx=SIF5hk{GLb1`eIz)bn@?}%++X#z9#5To&O7IADNPj2aPJfXr?%HDZaHtrj`HIurNQ7x7KUjkR+ySa+k&|%>'
    '6g?r92w0q;5TJaV(pcC4UJbhgZ=r)OGI+}Ht!Ojh>>ezSMlf#M(Ca52!&>ZML<w*<?*Pl&9ptds$z@L>Qtv4&L>Oz8Of&+AGZZr`*Z?@4I?`?*fV-yHW~N0G790f|dLpKS7BkH((jI=YXyYKEM9Y=XJbYwNNQ~aw'
    'a|!0}TUe3#yau4Sy`Xi?ImK0pYeZX%*8mp$Sapo&sucztsOyDAP;}lh(@>Wqx$YInAlB0q29^{C2SiJ*P_sEONzcgyje<18!Jb~g&ndeJur+Lfix%8};_U(M9<J(eF(prcIE<*<&>;mCHUpAAhnZqRDEi0)b`X>J'
    'QFc0g;}~{JmK)5|N>p6M(FGqxa-)y&S}~S3a(p(^F1y*pKo>y_Q#|!4gOSE0zQRC`d`5&6_fg}Sa<gG9=TuTfIqY7`ol84jNT@MRdU^TnMiHd3iCL~hy7xJ)4YGfMZQjKxMwM)hVInzTI&%y}y_a!O+zuP;v|XOh'
    'G8Zz13Mt1}{?Y^!^9E6#PpTh?zk?;o^hXOi45UPd>bJO+kGZtj4cwucC5n&i_f*P&9_@Z?4ntz@O3r{D2I4P&R(YwFQ09!`BEbZSlqiieAt<}i4q~y3IS{l3DT73>#Fn(K#$u~bEEU1pjBFNMRwi0JREM=&gzYZ8'
    'rM4(Ly_;XKCBWB;>29TSTf^XFFRO{5tAc9Yl}%3^Wld+!`84@{l#z(W_LhqPr-nm#x^>TvzqxwpnIw2xY`I?^JkIl}^e`I(s^-s9N8?njQO>Z&0=FU`9+GoTpXusQk(3bTtY4ZU4_lPgD~)qTylAv4HD0qa1KxEL'
    'g@L+GdbdouQ=JNr7A^Y4;Bplb>G0?H(axA!u=zkZ5K`d8sGvndyEMkC*Ls)?O-gh`4|C6~0vX~%RRs-CYNuzfyic|{S@p(Zq9mVwb$04)f3sG`gU;o6TNNW%GJx_eQU^(K*FG=Q2NuSiPk9>ZOnn^|LRF$N49XsH'
    'VlzztQJsi$$p8z<TScjgd;z{t6N!O+uPGzCar*!;x)e^L59k4D#Qrd6_nP}|CMR^H76@Bjv$3m^Ra<XsmMe^V_^F@vp$QnI;KL|x=yx=F$})-`kBf|49#IO!IMWOA&tu06$4E?NDkPc2y%wu`nz|Hj0m$%A>CM>l'
    'M5zW`S4NP|t<a^bOd3wfL1rMgU9MGN%OgvpyXTZyOSczP1@*N<vg?Gv6qOot!J=@BPN8>&FvZ=wWj7r&`|c9c0h5A|wC*ZP8eLHF;6#(qUUdPAjVPL0<775=bA?<fwsCKp91I%RW>$R(%5B?;c9m?axAJC9NG3~F'
    '8<%<kaD5+&8W_opYLf)ID77G2TYi=1hK)z-4mK#J1&@lN{3~0Anz05b3{G`x=|Hm!W!S!={7k#4IGhqIJC}LZI1Y1~*RsxJ)w&`z-4$%aYL-Nkx=Os(t)7asw5zALip=qZYxrpD%H>MqSUHnNe48zI+pMTJ7bdEW'
    'B0p1C8n+vjRi*@8y$p!!Tp*d!i4VA-v8XnSnXV1HQ%qs)vJd5aHKpO6GCQWB=OhUkEC0>7OX-8rF0W)huDv}u9DR4g&b5WNO3<EUxLGLJNAR9kbH?Qm1Et?oL04^IZCw*}d}8sgo0KJN&+&h^!bCGcqy%g?SpH_i'
    '^4(IN420AsLmC#mnc|OZifZ^1Uh@Xf9bJjClSz744VU#9v!pRjyPKGbe<5?DR?3b(0*i#r&QfHZ8%q*^tU8Nm%t2Lt7>!r629ewr3|;v89P!k^UE)`*n7|aTPAXGI#j#qibbTs1a2n(`pXd~nGL^1fHh?#hdzO<;'
    '2-Cb2#_Je2&KFbkv}3a^Q${?Nvi>}oVf-YvQbc<bWx0&UXpN0i$^HO|Dqw`x9@$!2R->jNWhTl0(EVrWo5p=Mjb^~w0rSeumWtX|Z}zcn%8_9qi3?<=4?DiC4D~hT#|GGSlo_rX&1<EXp_r?2X_Zr?qPTb(t~v+a'
    'O^vTf;+zQ##g1-XlFJWn=mxUoT{5kf3}Xb3P#SXTC7x5EE`+1%&K{_?UB?Cqa(vryDy%%elvS3h;3+1vxJ?yb%3F=?^9m*)dLx<{t9$C~Miaf6F{i}6ns8Rk8T+`<=yk;oF(Sf6%nS=4mS(u1%q!M!a~1xEznt3~'
    '>96ETpczv>fC-@0EG4=7Vp-%S3G|rNK-7VALw1YI1?C<{hm<lelHJjaHCN5g!K9a*FVaOO9(l215;I9c9D5mVFh<6Xb+N^L(7{KY0fDdPeGO8-;UJZ`6)q@<{RSg;ll4T`&1=>Yx6Z4m2W(;JI8*rj;dy!aWF)V!'
    '_l}jPMZ9>lg~O-F&qin}I9agqzFy$H7!Ze7FX;u^eJo~ZU1S9vG8PT$B5-)#a3Cu#B263XE{$~vcBvJy<5e5gvbYDjJp4#g%bi<$S2!E{j>xtHtkWLSaWr7`gH5Ka0#{`HSYoe|!dc|>8DMVHZ`g;#YW7+6IVch$'
    'hqW}V2@EB4Dt8e30CWK}9tumEz+LJbZv}Mg!J7j~qHdT#L5`SYyn9G%TcVB(2P-w;P-BBw)S+w06pO2!Sv3Qr)1A0UNba_5r$KfnId|z^-!bprjA-lXBq+8yD;8_#G^|>(pwpf+8p%;#QOFhz5GG2Eh9b?<!OsmZ'
    '?oiT=M9!jUQ${+fJ@ig;zgDENpAPK!P+EsXG;2LRvnmo+dZPjwk=+$A3!BU>ll=FdUVS)})7mMqvj}CG4Q7ZtbOw{MRfjASI>PoBcje5G#cbh9E-39(m0_&4SsN7!x)XJrD*7R?DV^7?LDC&#$ur??EYU?PqSmY0'
    '95k;`Vj%g%A(z8JtPzkECC9ylhnli&)D2`pRiDFomtwZd$fI)^3)vq3vw4c7zxLbQ_wMaZN&it=cuE?p*xWSW<#c1Enqi5z!fRO#s;%2n^PGI~)(%;v{1jGig7m`GS^t+ICS)VNbRQLX`LsXh<y*;v88+(`Fv*57'
    'm7POw4JO#5j{oCDc^(0uHBgb@IG6*h!IPJtnZR%F$46CC2|Uj^q2-bjUG20|wN*TerX0lD8-_(3STMdW3>)_!TICkG@%JLlE6DOa$fHwKE$k&%8Y?XGWM}AUS1obYLl66=@RuHi>=4}tOtDu=ms}->-IrWfQ|5b6'
    '<-+ZUmBrYSpono)I9oP%y1n6fu(grKs&sp@_2Fr4wYI)QZ`Wz-UE}xGMssCi2Fh%;JhpqNRbB9xnqGl6m!fw{%erfOUSgJzj`FM;s7+F1vWzV%7g}_}s=L~j)^g%u-5F~~L~Hlf^p$XNmML`?Dac!%y05If>6O9Y'
    '6>)KZ1!Iwl$;S(7>*B2SewS^PnMNy}LyOmG;tcFWJB$k=J}XCnJ*kVEEB}6=OiS9Fkb&ag$!|LG+O_6326U!CMD0CJFU+Nyb;Z44TdLB60^HhHo8$&TFu?&tJ}neKcC6Ckz_&Qcje-S}8d21@l;Rak;s(coI?z5z'
    '>Hyr~Io+|YCoFfS@eR~pWOyVuykknaOn!*J_ky|oMW0QbDPZt4zVd7htr|!?Xn2R{P)H1`(iC7?PRzF?t*c>*JsILt8-gJ>P>z4*2UgjJ8{NwCDg~UEJmdJS3SgwCqg2j(aidX{&lzfQ$COv@X%rY+;yPfnLQ9N7'
    'ktZ>f7e3lXVcOr!thPXw6gJW6TFws@P#X|U5Av(i(lwYya?kOgE;E6%BmMw18bkH;EmKJ^c@<6hI%$SE59p}Ld|~tCQfC^9ZnPk`sV<kA+3U9XN2jIhbPH#)YHFL^5xBh59lt8LX3F@rVIKipm$HH5IIhaBS^^k('
    'o6;q#!WGdg4BaTkut0`8<B%tk?t0gdY}|unaI&kZvHMO1ma^X7ZD~h0oMS5}OP%Af!(5~CsWC}mgm@L|2leV0F7>4WaRL*0Pu2!fw`np20U?~HSDB)Xl1(jBD-8cr!Cr|4U*-$VZbS2ncy9EK?mdv}x`8FNi+`&!'
    'v=edW)P*D7Inytpt15i?*2J#SlKt>p8w@&B>%jw#XQ`sdmhKg;1EI{}%18IkiM_I|dgM}|Zd~(tLbi4_MpqClznI#FsjUU&OG(GU40Enm+J-H;V>H2D1;XlLyd}lEZOE=|`gG)gu0;e_5`9axVy(N5qDF8!qA^9='
    'Q?sDhUk!t~Q)pMjhmdxJUvSq~ipokAVQ4UXnf4d7fmYNh`IJjT4H^^P#$|r>d?>A26=^GsMKr}u^r+R`;EWC9IFws~Geg8%Z!mujtO$M&ztFp|7&B77mfn-W&VX3LbU?O&hU}oxpn19rRt*qVU<^zz#SGzB7&Al<'
    '{;ohM)=q$4L!<$yw%fAUNf!z?qD-)~aTJxP{Z5T9+{}fRBhznoZ!Z!4s72auOAHt8Wuu_rb0xm2KlibeQ>|86%2iG>9*v#CS=e=*d|WG+ehiDjKYmNNsatred*6v~=vS~D!5H~$CYVY49dkQ-@AadRTrVJqo<P9R'
    'LDqcLX&oc_w)I{rZJG9xjRYcLl*i{c?ZWn4j@HU}70YnZzzlw|zhaUY91P`bYciIFDcWQq!Bh{ertH`|OSZXHnYVnsDqN$DonZamzoqTjvi7DeF5BZSk;=kg?v{2|HAa`JRE~sUTrvpZ6KxOjqWy+-2D)Y(`UX!9'
    'dwW=w1m5C7t>3D1%)YW7#7-$>Cu=<%hj2bgHgX&vx?bZeL0EA_;^|bNgC^n^y0FL?TZQ4chuhuCNphlIiOIyyt%2>mPcUg>c|f!R02SfTmN)BCdaJ9Ihn~7i(ubZI=6w&nR}996$K8F6lJD6@!BBa{N#;s$zr~Zx'
    '_AjVgE>5cgaUknWAvk6$=``0;p;ci9+GZN-o`cW!^1}c>8&`)F1(h>IaL<))5lb!*+=~Vmc1PV?yU2Bop$PL72d>o4(x``a2X1V)$}LfJB&?DYZ0qfknf40i7xm#f@(G%lYLji($aHB7I^4B32UrH83Q>zRTZhVD'
    'Nes-fX4O%ow<6Qh8W?Sb2QD8x7OhvS+fX*UbYf~d7f$L(E=3Otxg_pZkIhmu^e^|;7OfaUr-n+7{v{ot(1sm4cV&BD;~Zq?&gwPdTo3TXSm$QtfmxCUD~7pdZf&tpt*V{dHOJ}3jJ<{WwG!ezMMG<0Ne|X>x%Qk~'
    'Z|8n5DRJ!umw>pw1v&nT%E}c7;D<U0jrxqQ)zD=wo~u-MMc+Ckc172^wM~~?II4HTJ(kq%g9j@QL>dWR0kV9;f{7j%?JN1M!+blwpSEL#YOX;JuT{Pky2L2P2npK1;%?)G;|5P%F4d^@v=QOXI5i=lOJ1z3<mjf2'
    'n>*f$kiN;%5gP3~<_l}F{@&H(HSECA^xa(A1dS*|?PPN82kY)$Ep3|wa3J}H%rGmpaG`dybbS2eZMWnfTtT<RJ53d&<xx?x!l$^zGwZf0U5rb`E&m?VKfGi#yH1PF$F9D3=DTjQ!Q_J5zGF5YEhB+qB!Z=fRWXi%'
    'DOm7fU*;q|toD#D9)b@Zm>cpR^+U@m*1?1Eb%*t#5bC__T$xxjs(?t)`2LERE4xm`y*@InWo>VjvQAEFvz+C?p3PIrqzj`~9we=5@01lMcQFXXg9i%xii~B38_1S4q^)6YEuV(gQpU#7?G(|%Z?$@(<#4GEBU8bO'
    '-rZ;S+lr;Cow(7=RL^S3VxyXF-^VVrjBh63)#BF%wVC-W2t;<v<#6FkT3V$!r2EyKThr&A(iJ*gRW@(mbpC(w_@Id<cb}Gpie;B4%#JD7r4kdiR6&gQQ$>BSK3PQo@6~FQi@~qTF9WA9J0rml3CC!T<)EU`$^i0C'
    'm(@$dG`wO#s6wzP><)QZpzng)k+Jx)%|36H17EAsd#~=*)=)&5$V_I^+^5%+S8J%gcRG^67nfOF@{FQ)>;%;{E?;(|UDDz5pfIY4g(V{G_RXmd`JrB{x~5xpFenO+7R5On#~5$R)gV8I*&Pq*Za$>YL?}X|W$&tB'
    '9d{LTQ8bz@NV7$MmWp%{)<Uf+txag#@8S5gRM{^T>|_GU9)PW=s%;tUHe9m^yCvG`N>^vW)?Q&QD2=t1ts#g2ssx7hi7{bz*5)unMLQA@8nofb!=PXi(ncGdsvR$OdjxDD|E_C@S9B*{XN6sGEw(1`LpiV+-!(g{'
    'gyV1a!CcLhEir|!W9ePn8da#L?cwM>FG{<ppL0Ck_o65j{;%5lR$Bgs>!4PkWL-N8-z1v(`C>MvyHczim;kf#ViUhW)y1|1pk}&J)N|MoX1irOw~CS7-)~9T-;_|Pn1FNVM*Y29b9A6r^8On5nD<)R=h?j5ncX@a'
    'a>8rx_Ui5<BN>!Mir{Pav2%Uf@0f^&UAxoG#T3g^eltHtjWydxMG=l_mqsgT3!ksnlDn6Ib#>q<iWY<J3Y1>KUo{TS4^FRDEespDDwfsyBhsa$8=*xT9lt<l0}uUQZam|DD3{&I6a#r7DMBLd*h$iC9=|v>-pMLa'
    '0co|{8@rp@mQEjJSK{l6P06o|d-7ghUlhk%Sy4}AsL++5B-8?Al}}fS*K&l2&IS-#P6f2`lpjG=G!baJBko*X_Mf?^RVzv>oe^C%G&yqQ(IClW)zN)vYwKS6VuFYjP9Ov=Gc0kVmRH?(h%Sgo&k6?@6~stuS_sW;'
    'XGOO8ALW~N9#1Jq=D^B-Q=DoT{XC+HPOo^&(wj^+F{Qs+8h52CQUlg5L{+mvRIUk8|ALf*)f^FwVBQ~@{DZLne*fqse0z3$dic^idGk3i+PrULFGYL)e3WFGIt(<IQOiu?qyc%>XT?lHKTc-&{(U&GwR|0OCp;2m'
    'JFNy*Fq}qagTGXm{W72M8GuO+2$^#FIgqBKG0wmZ=R3kAiPDtG-zPYrJIrcUO6PsLHy@$eZbXBD(haC+9c6P-P2tu={BYX|f~pQh6i|V@E_8S0P+S$}KPq5WP{F;tSA@GI&<Tf74V`PULCQx7DZRSUOQ*3V?qxbB'
    'gXPArxAeVvlH%d_9z4pD!PpDT?!1eS*qd2Mb$ue0mu^rv@>nZFU@swLe6aASnUq&hy21*#>3+2}bma;ix)TO8a4pdd90GFB^qAz1AjGZl`^gNchax^YiZ!P|1XQ<mNlj+u6oIOb_qkp45n$Jk)>f6-L#HUZ-%B8^'
    'dk-X4gTY@=bV;8MQ&@S{u#Vu3*htV?L>a#AH=jKPuChu0=R3gNqK5+1BT--u8!$w3eQ5XL!{*bc-Y1`U&BrA*#ahjZ6Z6as@g+EFkTxO1nek0zuAQ2M{HnNcr|ieeZP_GEC23Dk!GVLS%;PBYE<Ts)2!drmN216I'
    '7TP)RagF-KTO3(|$bb1k0~L!Di+WR`Lb!L>J?>E%ii00ITg|graqKGbnfPPPaodI2qQ_5!^t>%s!8#d`qz<-u9l7fzLuEFo!G^dMU-_m&)lod2wbe856aCn@A}Y^A4|*L&i*era(5>#|<!3wjmAFa;HK#x7C7PeJ'
    '*F<f5oAlw^uxnA9#jUhyg{-N^(<>F1kFQjZ6`guuzx($1O@#{&C0@iNZ<K)womO&^`c}71o_94|ne+3kUHjM?lJe?XR+U)?3kB>?%0Q}w_LetjJelP;xM0cQlHO;O8jBlR-5L4*LF;&9+NPv`s@px)WuE2{H^!cV'
    '&%Ir55=XG`7E^km^-;?@nMBtbA?1#msw34b!mdS2SV8N?AYMbV0JtqOHesH3FiUpI0lF!469z&+z?9>W>Pa2hOSXjita~H*`-W~TP=~Lq^vz8sy9%O{si7W)z6!%qp5V&oj^u!4Ta>?=Eh?pP2bP(AI*Zcb5TmAh'
    '--*u8F^(2)J-4q-;3<?*<Wq!~`C?j4$9&urCVVwMHEc3PM}wj}RvHK8CtT@gXR)FpK%8SxfAYFp>jM{PIHVIgX_k5+J8A2y-N$)Q#)q7qZAWtPB3G&1*h8F3(NE<~PH)7=mR-upUs-|6zRj3gl|eQyZO3z`ig6__'
    'RMQ_^H+6=<ayK+t(yC`pZ;w_PSxaO=YRCzi;wl6TvMU8*ySkuS9o|AYFr5>vkilXy!|Q2m55=c!F-L!nev)u_4OSRTf&K-BfPh$IYimp7VQg%RLkn}N-%HqxN?3luufB-pTx7VIf~1>{b%Con>_z^DNmbSBBqRL@'
    'z#vIhxd?JP<D{ob;qu-B(iQT};ui;}M+a}t-lEKkj8Mz-vsv7ypt<y;*$u29!uSfgP^lKaJvn^wR{4Y6ZG|VNhbITX*M>)LULJlgd~xvN+lSD!8Io_pRglaOl4*i<VBu;L5<s6qx4ci-aB{AGq3CU7s|xi+67d?|'
    'ES^);n!1GO;Y>iNh>VGb_u<@NwFNc%JA`A?nZ%h@Zg~C!TqbCfb8%&nBOytZSRfH3Sdl4?SBmPf)Pt3y>LNx<#OhvfLECoBCDYW6`>DLrukIAk4&4%9&a!7@@nlNy0zI3kj8<w+U++G3a<^;Hcg+fkY1C|mfS9L1'
    'oK=cKT%!_K@r|0;W?gdRQA|uLY?RdDa8`C`OwB!@1{A?)74t(`Ic`m41f49|QEpqeR_dL(tI}`Q#6SB8_p9EiDqaM^M#1}`%PBK*2n6O89#9C4elDZTtoU7wq&BD4(GdK<+`Vs)8%cU6_`W_xPVb$QSz?u{s@3ft'
    'swtnQrS8#APfOda9^WoS6G<|wm@O8`VUm(+YWU)?FBS*fVs~&j91LS|_8JRg54*7U#(?3L0l(<Md=GsBe}#+ZU&IrUkt9p4vG>mKsEe7A@jsq;{{EgA+1#b5(-c?N+;g}~J`g!ykDj)RMDxIpV4GePTM+}3q}@WC'
    'V;5A`(`t(Sl`ph)S@W`8JGF;4B@l+aes9tEH>?zd863mbeqLNvt=5c)GR!3vgRO}>wx)7dN~k(iJ$(#mu{aE(0*ie8emKTvx0+SGP0Er~<$$t)e|1pOz&uk-G4Ig37~o^rTq7My+85>*?b$~R6x+?;GcxyjR!++0'
    'z&DJgfCad!mJU(c)nJpI2V^DN3jB1<N)uOnBO^@ifByO7KOB7a@Qa6E-hXNcqf7y7jWByUn3|{M$S_;pB;jeL!2)$!O^Qn&NEk<Y@a02L7TtgP=<yeWPwzka{Od0t4!(M78vBZzi3~-fY&3#GiAFc{iV@UU$P9gH'
    'IDyhoz7(^*u^=&UUqeCAw4!4e+mt9jDcJ@`Q&VrkaRFaDO4@Q$WoFUdlmo$9Fnj#G&2&~xGdw=F7@+ird$~3jb}Ku!l{oAzF)wMP&LmB4C5l6%+F~4<z&7n+RID=?&a7zxpb>2!zDL96U6$T(7mHmrc{wo`He}{D'
    'lFr!PG!$6T-Nr2(xG)=xSbZW&D-HSEZdM671VL;oge+pH4SZQcg2ktZrNOD^1AzQG9raMjU)8{*lNMz2A@)uZJ-k}g0qxTltIH!K4M)0q%1XY9K-!<~3tz9fL${Ps82deM0aHtj-XmfWJgE;2i}Qm%!H~Ox`;X~M'
    'H4g4*6Ck<EToT3Pcgq<M=?>_sht?m5>$eT!F!cEBn}IbPX8VrP4MV#&dC2PwAhbq5Yja>tmk>b+HXZ$s75zZ>&4ufVT(=g^p=l8_r(0|PRzQDe)I;S6=RBxcwL-&6c3hde0Tbr=7cAQOb<lJ<(|FZH7RY2D_86b4'
    'zz})lLL%geL>UB-TVgUH*nM2euu4pj4VY;7M&@tN{$#Spg5}r=1iXfO)X?JF<qWj}-k}!%3rQyyj!C*n4P%2iO!V3Z&SH0-Alm`I;rUBHYXOQL*o5B+Cb{SYV}ZeCwx7#HS4el|nKrYoNO8!fF>s-^r^-I^!gTyF'
    'mhEYhypS+I6&nR?yeMWM^z4&Q2{c}2q7cW=Un0<iP>u(eB!&TYc}?!J_=GA*e1_sX7kcvGQ#GfujAr#%gP>Q_*%E2tGkv=1AAzsAk8_NEpL`Z*^202|ri{a|N8>VNm(LftO_<xNo|lV70p_KFC+MV`<d;b^ceiVf'
    'A4QIUa7SThi8A)^F$VhLZ^ptf&%k&GC;A}B{Zv^#<R<P_ziZ#B--%-tQ`EQjsfsuz@kHW2H1$VwI6=FD=xui8zS)<)jOn^6!^|TTCnapNY6$M(ZR&R<f|qU^OM7Ddq#0leYRNYqKl~;kcK|*tiU=WRz_&;-iiLzX'
    'ej52o?wpXT@X7};x=JCDkwjJ}yEft6JIOG&-?}lbD&@~=VBuo>Y2b)4;3>}vLr|uk6ui?Bk5u;&K_8~X`oFaZW)``2@Jes+8f0cc-4Hr$o9-=g@%6{iqzwDbp{ixe%mvfor%Utv9=qiC=v~j&7p1;oqMIz-Mzqc0'
    '5(V*a;C^E~>|1Ugt671w+mNr&LncSvCU_0Ktg<kz9^D6yXXp8-wkO$&(jdc`nVmde)^@VwcGPzd-aD9<9j2=JF3d8gy3taF^L^gd&reaw;a!yHpbapDa+(i|DZCW4W7k>q!L~A|YYAov-`%_;Uq(ex{x+d}z>#UI'
    '$?wXpY2|bl2P)?U>Zdo;Z`jfM`#HxRihnE-33iXAAKSF{>MZg#QYS-O)p1&=2Sp;+)=91<g~eTSAo7dZyfXZyQY=DqR`aNig<=`5t@aS}OxM(lo3~OSrk_cvBeeJdum&jPmSu%ZF)K^0nsB~Y9hI|a2b;5gkS}S3'
    'GSz&(Z<=vZHWB0A`Z+&**`O0yOxpo-gKX@WNNrZn`|y_T|H>~OPh_hh8C*L^{?^3IIc*NOIEKBH6|N~md)LgH+VE+MIp8AtT_k@kgOC*3^mPZexaDu@$@5O4m>hK>IX!^tmhBC3HwXZ%dZDk+adDB)1}{zFCBP_~'
    '19CZkA-5@>4V839=nMJWM$`p66%~9V`La`=qhD&h0O1Sv<9G)xe(><<v-@8p_WL9LUPKM5#{4e9S;<Dp#3zJlhL(aDK&_$oqCkaGD#5)A>p6e#T20lab8Y9!{ZiAkdCv#1GCb?9+`!lsxHZh<-dR66f{%J9eT0OD'
    'Ki`*Ur@nd!7Pv)Sr(vuYO8|NqObk!*3q#*P$!9k6v>1TWD$Uq=$$dbivntJ%i?d3Msi_6>;L$ERO+M~=-;5?^Lm7T$IY0&?Mb^QTIlIGO<in~(W4fK(F+D%U*C)p(AMBymF$?(gt1lA~+{-Dl;)UUx4UN1_8F7Gg'
    'Ll+|3VP`@V9msO9#{yM`eS;deQB#-rl1kTe7gP+^cMObXMmdset9?kHx9f{8ul8mFvGnb6c}bj2jF*WW8<n&eQ^cEowmvvLYbU2?A8kLwyljjn0*q`vdy%`O3Z|A<rV?)c(kw6KF6hNJJ@vqcHL6XC6ccPC`n=8W'
    'X+`q$b5H?ylbY;M16OM6L=6~Ch~MH`eQ0IX!0RAyL&Ogt5}1u<t}1am5YL}hj*s{=3Y#+D3=Lm41XP`A_p~@V`5~uyg*TF}omy|pOlAACv!l7WRE<2cD(o}|i@&!|emzr;r$ImWIos>QLp{v_u$65|t2VDU4hsvv'
    'MZ~m_ks*TXo*)zg`0);`hFR4gyvY#XK&E8CQ!+60Y+Tst#rdJRSmwuy+HdSL+&>%EU5aV*hDn{avGTka?5wu95<jwWQ7qthkOYjMdJrz}Htmx&8TWK<5`ab#ff(f+gh7L5TNs#daHAN_hOB%)puAC?gRP;EHJ2;q'
    'ng<Se?4}}DBummwA`=nk+ka*?p<z_b&IDAPCgv*mj_2D$hG8haI&TU=F6QMPS=>+}i+g#nO6%1f&*cNJ5>vOh7N8nuI&HNoPm53c&DYeEi^^&#MVSdUwL>Nh6<5+q-KL|yb&z|u4^>Y)X{Hn$u47p=2x_0%2YR<f'
    '_Y_hwVJ+svs#w0Tv!`j~jvAZjXZ_^a{CuWTa(^+aw7hV5iLs1d8=u7W8s<^yyU?Ex(k)^{Dxc_=dBus`Xe*`>xWNdmhyzMS<D1!4kX^XcQXXh?P_>6>7Je7$meO|7c(VP3Pb*N?%~0F3uJLZ~H0@)EWU=-Ho^$j6'
    'xCB7i?<Qd8O2*beYB*Ob#YPgNg99kJxTCw0y*WwA;Jj~sVy+Z4tQL|N43(O<#hFml<|-8#ctApiG1ehq`bP3BblEt?C%o<zf$jSa5lDr0XmOc2NFxB6{*kP65uxFA8$;myF4Q-4oxzAgulJrR)sGH0Qm(-pggo`k'
    '8NV#2*y((FVK(e)a)~1>mS&BIB=V!LSbf|m9%u(MEVG)JpOg92NP%TBC>B*Y1x4E?=0OiT=N;sgJ&dK64Lpk<%yH--_|47!Th^>;K+#uo35%z*Rf(>a*Y<m(T8Jx~-RJDnB2q(D6r48g?X0!duotMwREin%Xa~z<'
    'D`8L<%!noZ+VQO`h*r|L1^8{V%)TS+oPX-H*6CJ<eq{IGE}iQ(KGNq5-+i8J6qM0Z6_|OgHnkd4f?i$LFjoa@br}O$6@1#|Rz_;0WBO?ZZ;&)LP=?Vk_DM1vNg)d-!tgV|y$IlDV8BEn9kb<>v&OdYxs`&=lr~<l'
    'e#szYTj-WWtW=PZPG!C|2Q0fEcetwoq@>Cp?tsGLdBkrnL+q|x5&YD{I(7Wq#>;sLpD#MX`~fSx@GKyrxX0Eu%0&=q+D3cD)^nFzBXd>L@H?TwW}kGJ1^e)rUTB9ph-80?r?G+4geOb8ejb^fboAmh4RK06WYO)+'
    'CGJC*W!YaFAIFdTdT?;y173de_`$=$SC2mX;{MaGzl2D#^!@zK2X}_!5AytnAAI!T>B)!ZqxbKeoPAiFkI(Mp#qi|({qYZfzc?G6k3JZm<#+Df8UFAzACAsH`e<0(`Qce=gYb-Cf-$gOP#*t1ujH?#eUG^;r}E2i'
    'NQ7x4zb8zI?1S>M@M8Q_5FS6d%x7krmd$Z`Pwqc>@cF}m8UJG({dGE_1u|1ge>pe5b3^Cp4gKZb%+{Af_Xa}Vka?@jTA<$DcPN|~F_)xSBl|I^z-U_Jv%$C!^+BsbkOPVmH4^Zu1FTT@&WU+a8t%{3QPHCh<<XU?'
    'gP+~WlB0X@y~7TB^X*@*lHdGl{oP-e$@I;)f3rwleD~MM<juE#HcMJR`~0WL^EcoA?aDm-(|K~5j3#UI;PtPQ*@gMg5b!sz|9AOr_2%`Tl<?)>uP9V5vn*Hz*M4=pglPAL;*-{R2|$kf)~Op3rI51c3?D(9LhoqZ'
    '>3f_3L~^aN<YPB>(Z|9^$K2cwHfUz9V%Ao1g$h=00(m^k<X+eGm6Q~{GC4}tof7XQTme7Wh-;e>NIVwyO*qM+smoO8`V1A%>?2yQSm>Z>+_dXYqr9T+aybU!hdi1u$GBB+$-y4cGe7!v;eVPhpO5D+XSaX+^rxRE'
    'qc^YrB1cZh6drx|XQO2N-TwoDF%W&QnylY^`xj<A4Bx!|cga74ki~ZL=G#A?P1@$&n{QtyL-=J_n$_Iw8?&3<y#C{4^4))#P23i^BsS@|S`S<4$G2C#@uYtbc6$QYy-X{^Ld$!<{CK#0_N+R*{Rde1*{rd2?~m^F'
    'Z(*UPB0qTctk?UaXV3cmL-WtGXRY2JW$<Sfd_-KTf}EV9*79=P;4eO%sYXow)%8jH{mnnSK5c)vfxdI~Q<dhg_Q_%I=<t^5Q+2EL2i<4SzRGre)cS)?FM0NC)&7US``=&?t!K~1ro}s(w)|lR)^2#7-GV0nho&~B'
    'Gs96YrunF7rH9o0%sm+l_=(<zaw2CQ3d}Frq-Y)+mc~8wcQ>|~CIUr0l8O}y<&2-f$FvGmIdH>LJPUfF6*<S}>**9NBCWJ{^vjMJe*YE?J?ISN%VG{}>c`ZYH-heVQSccvJXi<w5H*~#QzJDP@)kK+qsg1se+k+Z'
    '5M$rG{?9-@{~I2ie>>7KvFQ+!Emt;d5LyhqL-eUfk4?N=d<9T%N`Z@LZ$g%x5ncz>0DZ`falD1>dsq9=*4<}aQluvOj}zy=Hf)mf&xe<W?fCf`lqw9;#~wvUZIH?q7XVFR9tjqQof2*kE(D4UYz^JGsAcpZ>#?ZP'
    '*S6UV6Bc);`Q>n&C&P{^#>^VD3h}nCNDBsy;!p1cLNDyEGc-w_fu&L&oq7e{rvmP}!#?iQUuFJLOe_yQ*st_Z_Utg*e%XzTtT_VqlSZn`KTq%iYSDcRS0UG(`O{XKIpMFlK(Fj)FgZu2`pn!8$(fmkSea8Sb46ub'
    '%Q6NdiWSRM>p6aXmE22$DxPB%rZ+?XVAAV@!Gp)2d_`%@*c@kM>t(B2j+keHO^S>V=5HmI=<MW#6Sx|tj2@kScyi))?^8U$Pab><2N-X?<jJGYO$UKRnwZo4rzM=-UzO(Y=C-CR1Y&GTMGi{n<w#C94#6!j{_L6e'
    '{dmr5^M;W!TsB<yjY^6;EDg&GiQ-^q9n`cL_O;C<D{}{G!>*Yuvyp#atyp;prhNMB7&5<<(^du`EH7XI(=R{!kyxHXECbl<)LJ7T16Lbjjz+WGTKfljFF+!^3v^^Su({@6wd7f5c9dOdrv-y;&ri)Ly$@U*1jq@n'
    '?7W%?F%!%Nd}U-e^IfY2wE*`wTVZAB*Z?ja8RD#a@5q~hUYfU9^YNF3A(rNvn_V1#e*cS4{?+5J#M{M4c(e04C`@KVF1nlmwv$j^@AYhyW6X<W0s>bQZ<;buoOi@eg@+5)Ji`)V(*;1D$s3T2y(%QVY>AoR<WEm;'
    'Y`(Qt@%5Clv$xD0N?A4tIfu_B!JputMx<Si@fhNb3Pn4?C2ST6gS>7h=P4qmYzFw%HU29JA*gNi2I9!4U5#UdaF_RpLYTkYMShL<c!xOM<YH28m}fpsDs!ouDlZJvyGSn0{aw;1k`~?c=63&g7-Rn3e@TFY&Xg%+'
    'q|z?Jt$;2Xa<RFwX2%!P`LN}W_mD@bIT(aRM2v`Ya3%FHlTt)DGy^U2%X;jNL{7%uZ_Tu~+!iYtWPPc6ORXGQ9yFzp!wLb}?Lh&_DfY`Dx5%q1uz4A=r~%4nx?B?ij%W=yZ%G{}D5f3|#l&R2Dy?E-y)MTG0q!md'
    '@c_`hFw8B1E4CK62T`ZQLxCp(p$mU!_UR3AF+n*pU*_nIjGx1f7{IQ<dXEvp@s{DWA#NW!d<_5GX<=p-e6hM*q(NN-!AL+{D%Kwdumg-K*o7a}sur<=Mvc>T4pt@vik+`3vq+bVyu6sz5=xj;^aNqqkhwI6r2ro^'
    'Dty;5l}IaW_84?V&sn(-21^PGw@1=8BC8R=9cw_)R(fgZN=XWKX+gk2`JEU9>aV>AYDe={a2>gg4f+-zvywKMMJER4sZJ!8-i4$V>eite39l%Z70Yq%DcVu1IG!vXK4?_};m2s^3Gz6MCgpUj$ekT9*g%)C*)yL7'
    '#t&fF_zwF+k8nL)<otT(WjB}KxJvSaZcht=l{zDDoihGzmpfQ?nw4P5_gwmSxdq<IR&bFrvIRs;aWpRWe?!yN=h`#VRRBEPR)~ftUq1ZVqsL!=H30YB2ampVMxF}b$MUJA5iZ&p=tl{<`7xM>__aHMX#lMyeF2dN'
    'Lxk)GN6Q<s*ht9G#d5Sl#4QH4Ymw?6&4Md%H1Gh`9s}^DyDgj-^fG_R(lNx+ATZ2bGc);%f*qi7C!@^*2J^2TJ#iGxHDqa;YC8aGSFgSo6f+GIISwAVhd(3sot_U<Wii<Wk{5Vg<WVC~RN@YI3=Jayu~kt$dj@Cj'
    'c3LBAX(tuzCD9b?+2DhxCI!x#&H$&UE|DGwNXz%k7X2{!z2w8wJ9kb%OMK_#haaBoW25}q@ZRSI(E_kz@g0cjBeSOU@!+rl#{nRFtctCbTDlYA5yNGWGZ{o4IlO+{#Nf%N25-ce0&E*q*IBzrIVSfu+`EJ6{GzH|'
    '+|U~Pf(KWu+z#v6*uN|W&QS2a8VXX&C4%w=JIQPtL*Q`|sWpH6He|9ZC2xl`I-F|z)@6}G1*7}k9MUwi7gQu8UyUx?I&VyyJ-Td>Zn_NO17<^*f3O{P8dA=iIhK{@F)+4vwtwCcnSYygXyJ^oRHwVl@%Hks??rfZ'
    'n+dcT_V~)Kyz4n~b>o%3QN4ECOS-4YHx67QQ5BBWG}C73^UL}2IU&BHhc@HCYR7CtVDFDmT*+++t0vxdTi-TE7vadrmrgK|3`s}c6wZjVlo8Yh6KbZS0gaZfNh$dqNxFfEh*-(iyc=K4SD(VXNSJ5s5*^Qris7Y9'
    'EuG|=D&45~X6|^`bq{hy@!ugCmFr<H;38oGJEj}m^%8qhQ3zZ?dK;K_kYLd6|A&vi{0Wq`=tY=fjypkjof}SY*7;$zK0hyC3E9O%$4K~i|5GWfeNtkM@CJcKCc$ftPDfZ#wK5-pSrKcrvz9Sty-uWnY<7!|R@)r0'
    '5Y=8kxkQM%S*3g@m>cH~q1qumJT}+6p+fz}t_rC(=dM^HY;QdisFECe=<7{GXw%?a2klSkolTch_2H5Og#aMTs8pi8&gqB!2)ARm;%&}6!lyh1<&o<vM<^|yqw3%12U76@R<{D)22eW2iK_qzkJ4lL20H&y>Km^J'
    'ZjLMRA*&jAf2MBDANuEX|BVeR?<14er)WI`eCJ}ED&L`I&KF_y92hkk=;oB2F}OHyaN_eQ<#kCdarKZ0J3h*C`f~oi=fQ6{=5D+wEO48(xMtycA`FzPL6LPVUzb)TACo)EIJy2ouJZ)4g!RS6ZpZ<IgA*KU;S-S+'
    '-Elg7RbPFJ*aR-pomf^cU|g8rh|HD!QrWcUm_sr)E_jvt4zY}}t~`2up0Z78TFO~$1@J=eCX%hBj(+6;E}mOoU2Jh5@!%_#U@Z@U5DbJ~NWflLFR1qLk6TIVA$kEnbz62|AWXQPpffOPoxK7SP6DP;n9kZZfDiUv'
    'eyVC;`O@|fm957R<aX9-;@aEN&)S>D%2vyw2)KA%3;i}dtp*L9Lk8{!gRv11q$|b~fO{AI@iYUP^HfO*Ty}skTY#h=K(`K?b|8i^&G3JmUN_(ziQ!+A<b$~ho}m_v?W``f^En17I<ki<9)DNYwA_(qm7WVBQnAHG'
    'EhD||LFuS_u&B#4Ysn+e_H-I13Ea|cm23`EkAUJCO^4yt6l^#a{*VDs_5gpn;s?$rT&>{X0j;6QY(9NJ2?jtGtPLP^6L8aL8l;wMSIc=>kjB;`ep@}K)3CZHkqOpPE7rC}gW1CZXy2SsVy9q#ade+KBnyH3<ZoPB'
    'gm4I;!IZ*Ur-@M=op=h#*@XfX?TNaxAMd##BW{P6%N~8*Z`$dhhX*ORW4qO=tLCU52C+BLGH1K=!aHO>t!y&21Bb#M)^~v^ffz>e<FUdn(g7f!Mj$lZ+N6zxo72Wo^rN|pdfxahRp0eYRsCkwabtQ^)EXizNXXxV'
    '{-A!JL9?c*##426(r?g{_+2#(nr552bJIJ}KaQmKWMG-ve!vjW%G~Fy0&Jdfvp~G9q|UweQteXq>uBSAt2z(1w~bkncj?fY?(d)qX<I$mc7yNOex|N~UIqqTBA7ZiU2snY6je3}NmD*ulRqN-La=ii^p)O{sFvDX'
    '5eq-T5J*79_zDRy@q~0IOVSIjYojyu1T;hh;nlp8O5*^Wv92a+{sYMsJ9zLvS@S%&(tuu|d-`D(8q;CVJL`+prVhxyj(P;%Lcra&6XshmCrCztd=@nS1*A?5nU;MR7R9VUD>1R$6EHZ|PT%%7;|+qVIJk9{<+JfB'
    '_RO#Tg}f^-GEEbH?DyyfH|x!A)tW`#yH}gBWKAIpc15MzxN)7eVHYNBeK~9E7)6#13=B#O)Y4Y{=oP+?g2>xo>zOsG?lowq65$$YMZL4*9n96*T=p$!$D*99t3YW^NhIxHIKWY<VW3aZaA1lhhma@-QOL3Py2hHz'
    '^GVHZCio(&J0CPt4&kA_<PP{CqANX621pD(HFb$W5q{IWlcqkOIWT8RrkNc1PHEnbaJLmr7+uAb0Or6G%1*3l45GKGv@AG0iKxIR<AjX4=3PvE4b&Od5bxP0?54$YE-4}fK8-q~5z&YN#9lIj=kvtT0FL41Ujm-w'
    '_6h+ii9#BYL?}`^^CP9TH35rFi;H}8#ZUkPjU0wGx~*q-zhK3I13Yxgl`uWbb_Ei_7<pgRD`a33-YaS5CLjp4#=Jv%hv4q^)-;Rl6=^+-+1XhY$ouj4q$%E7KC(K0)oBn$WBb*)WVXb?jC>R=l>p&0;IO?R?GRpt'
    'hmEO`Zss88c}j8(=rEb)Y28_^$zw7M^p8)Pi;aaE$cxTTXNapne%jHTIWYETq3()S@mPG=U~f|%Za?reP~_YSGM`zEbf94&R_;9Q5kc)I*RZr3v$ognnA)E~h<Wf_aArDmX^hP~Y;BW>jkkdtq+;T{xnCG3Oaahn'
    'i<8B_+ZYh~3-HR>;_B(bPj@<fdeNwmxS1gE?31g?i$vAlp=^&7xB-6k`Fvy!@IwG^wgVomm$<FZ*JeTr>$dt7zkM>FPanQ2Mr%YB$g6>f<kG^=$Ct;sc`Kj61G>Fu^Or3@3Y4c@rt26=$c!DYS0fO|=Hz04&mj|K'
    'H?7RxosBC=8cQCc*1J{wtdWsa5mL_N@y>i($tctMNAqaqBN=h8-tE~6^wLB7&;m!?2RKk<)~$`Wcs8tqC}-XHs$8|ODk6EyVVuOFDFFku3~d7sjuyrKbcB>4D&XnK*`4Gtf&V~EP8y0C)R$u>PFSCJ$oajZ_v|h`'
    'UoOm*K~Oz-`BFMy>vr*#WYIP(9ry%Q$#~98yG;R|Poa-jR#`>H5<)w%iNGW?v3X?^5_4k1)qDWEwnaIH6#}GW{<Ys^7SKPaBU6n?c3}P`uraM(A7d$f+J{9b2P+I8<l%te!8Bbg)&rPR$3oJn^W39>rZ{&9n{?HP'
    '`ZLIuhSXwRQ4YM)skzfC`RvKpNlrOj>)z2f@(uKPT<OEW^#2(8Z=@c+fNwQyM+Er7eYYy6Q?k6CgzyvUiXEmWp}B`%<89N@nP%{@5$B$_PPQPuFcm!?%~n%il!YgzT380Etj*6pfBeZ$1`q$|<d^#Q7eC5Ec#X$j'
    'p#nArSo0;N5M)!tPv5-$;}tJl@_Yi3q`&!9PLZVF{@E37@6lwQtloV4Ul++3qD!aco7ewloy@-b)2rj;lgXRc{~?+F_OIT&{&R@<|L)fi+7Ge%lkdJAnK8qgKN*>-OQ~{d{w<Hqns8#**%qk9E6^TnPhs0I+k4F8'
    '*IzjALQLAtp4dnzTBJLmTc(FGJ@zd2y`0!czRZfiDKY<n0kPHWP#MxN`pADxXOTS5WVhEKMJ%wRTp58x%5{+)4>BT?ZO#vKqEicF)Jd9z-jR{lw44a0wV`XokyT1bj}sz*AElpT7kEILFu+YXvpBb)_wXe#gersN'
    'Oy(KJ0pnL<TiFgz`w0RD-0PYKs537R90D3yEx-F)vuj0Iy?pcf|13omyu>{|dh_jnS`&W2F`wgMUKO0d$>wF{{#a-h_CQ+V^`}2jXw5MCYn#v!8C5Z>>dcO$A){gInl`&r1bzYslg7L-r|8eiWQ^k-;lHcNZ~qEL'
    '`JYA*QF@$a%^d^ZrGKn$aa&=3pc|B_{&&?d)~=ex<>NL(Ol|${<L!-7A#Ts?!qa3jfAj6%jIc7*oIUfk#@w47O}T<QnnRw-%w0H#erT+DaeNM-d<hpnW-0<Cco(YKZ~pz8*Z)T;tBGI&|5|7{fDHWCWOgxuJK+-c'
    ';GdLt;U4*GfIY*-z;<t|_V6yK3~L1+{V!`;#$!Jb8}@Mqm$|4-0Z8R^C+z!^a#3K+A>2;MMG5y69e>&^hD`k5Gk#)Nk-G`C!IM1w?oX2wndMXI5_qteHgPjxB~>ljHSUzx5T-VJu8z4|EA%wyeCpg-i~rlEnZT{!'
    'O;B`?1)3zOn*Qi@>^J?IGN*^lIQ!If(MwUI)bAuM^#52^gs4}wuqZ29q1b>i2$_=LV5!NC5-2V?Wjhg3I&kXVFE1}cw-E6>YTG;|tm{;o5wG8JiPT_-UF(gLMWPb1&-mY7$9~n1_=!k`N2LFto*hIotW~j~R~&5*'
    '28UUGyy5L!8~*&jc6k5--u1APke~d}OxVEG=1&yu*kGnMzA#92$=%&hBQ3410Z6rcNfsd7b^F4cZ6oQxdm4`NTT9n5CK^Da<laN3l(0~k<}A&{Kf92lD>RGcUz&sZ6?h;P+6wJ^%hh@zQj{p1QRJ|)kyPR7x8}<M'
    '3)|9cvo#fq2hnu4hUU19l;lQL)OnPmxTubg6f&bi$U9}l>5yGb8`!XP&BAuW!x-ITp|ch)$2Ik&gLV22H3CVe^cL76gm|weq1U76JyyAMxF0ARi*vq>&X*^BsQTF6mjXJ3#RYbj>p{(|(^4H#PUrqZ=NRvLg*~@@'
    'sgV%J*VmW{)m6}^*n*LmEUO&Cj7n0l=W6gtobS^Yt);GW+OkMn#}Zd=$9tJ*%S_GsyZxrM*W&qw*g{|sRE`*ACDeg2<ovyB9#mzva@r>li1+AD?!rHO8tqr!8rAeSNXMN9DAu!M5n$kNQSO{8&5pVY+8pTSPdxhJ'
    '^_g#oTuU-p3THdvXj_M54fd6a!K<xt31g_nV-QI>wYhteFE0rCc|N<iSPuYjR{dma72v|p!5?I>xLQr<d+z}bA!{;@uOJMPFrlUM?c{<V!*Jd6hRIdVu1Xd{-wlzGaN45M8=Zfl{P6~mtD(wg140<4U&sWv8CBSE'
    '!m^<JondiV$nQl?5Wr*uFe%d}XAfb_&uChSh&81h2RGJb1w~Zg|LI**$_d{DPnLb!dE4|EHy{y6?)30?&^t#!1Z{iBVGxR}sQMaw?zT;^%#f%uqF}7pYCS|DN?pS~BM{PzlTJx<wEH6t4|VYKn@s#$(C2_g>&b-8'
    '>Cb-adroEKw4u2SY*=y>kZ{rWJ`JO1W4H6i=EbNGx~d6UJ5g2}^%|&K8?D{Y8&cI}RDmLR-UF?xk0eELj!MFB;*D+x1uh8mqUVbD_suGctTZvp5$y<0&uyHL+cXzWrhomv0#j{1AV!=7<teD$jw5ckkuJgwEu*NF'
    '-_O0J$xRF0YrsChG`NkP0x9{k;N(ZAy9x9@)`YV}IDi}X%sL@}P0&@_>ztlN$TjibO8QU?x$n_}zx3t5&6$j9QI24)ov8Cy6z;>TBZx|~^)<s4)Wshkmt!;u0~Lc7w+T4$+n;lYLyT0HTiDl6$O8pM=bg|tcYd*r'
    'Epy*RiF}&Idu%q#KzxN}`J$SjS1wRh>VD|NQ>&XHQMQ#<q?6+Gqg0iyPe^baea(mib^B=0Ff|aCq9Hcpyu59_aGcTgDl(5cd=mErJ8&&WGz*rS3s_^N^Tr|9Yp|l8^8f03?qJQ=p(Z5u5H(Y@w{g-Ze;4Z|iQc{M'
    ')NRsVgn9?i^`Fu^C;=Xj|IFNX_Lz7EVt@n$n*drrnvA}gjp;OoFhC~}vj#fbyk77c@&Qr;2THAg$cLJF!F|CcIafk+u7FxozKz<8o6x=@selwK_G7ipbBl&)$CzbJej_pG;XwooY@dsRf{-FOBl{_>{L+FOQBzwv'
    'LpEoJ>fY3|HIn>mjmQDysAl@dRcFcXDHq!Mcgp)H2m-i1Je-49dkPsuFvSHjB0|zSiFc)75L*EvlX(NdSd=A_AGOJO3QCbr3zWh~oWOYvc^1;KVZ|v=3yntWooNzNhA2tf3Ezqx^0Mo|mc{dlyQO#1@8R#hMMdcd'
    '1rQ1tB!g)5dF}AM0)5IHG<FN0Y9b1+?9dzU?7$J%qr9&6ed}rLHH+DlXSGICyEMgFw%JIqlVN&R`4o^%?VQ_JqWuitVkNi+c9OmKxS$<st8Gb%R|;T_;q@j$T2P-GFy5X^eogJFVA8~56~q+m)>}u81Q0~{@Z1n;'
    '&QnL8IE2Q%D)Z@p<1oB~AX}W@b+q{%XBVb?ARCz`#SFr|l36vko$1U1u%+8#icmHO+~lCq?ud#%^&Btk9@Y=D5E7`PwLb*lq26YG`V*NN#%sd`G8=4uVTf2jHnsDJ_}VjsEaWs>f#V;f`f*+zKvOy)(uAVQc>nr='
    '7rXgwxM^)$3RBH7uCZzv*ajhU*@M>(GuSva&mO!({hmu}`-=yXqmh%4(+G%gxZ!Yy4O@(QN~Ks1CvMOcp{+%XDNl0G+jV^=b|6IS5AjgLeOD3KFcvz2wB=Z+M+N%ofnT4hN(cgTw7RgN3tn~2c$wFRi{o5JJ=D#x'
    '`weM*EhH9Jqli#e%h%LmkFvH>G&DlzTFBg}FX<dJX`dW~F<x$_>AWg+$DYwzmOBlRgFP(bZA{sgo#&<LKxY<c{2(+QU$1P|JfBHZcn4^KcozX&cp{u6<>m|K_?BW345zaX7v+a~!R^u*Uy^xI9<tr+Ro8HA+`TtO'
    'w>5R`HGE<AIuvE_eKQBkOUk54mV*|%$#;~?{P7d;{0uz9$xwr@7jty?jOsX5Xyv>*eqK(ei;I^1urxd$IqDyO_2{#w55N4W1gcVjmw8Fv?DO@id-{Qdwn11JiB>dsH@N~X7HjJ+I2#$c44rQkEXTmyvTJcE#J>mg'
    'm$PEYPqtqn@4fflkHDf0jJ9>-9T}nPIC%;&`54a0qe}q99m&+vusnHXb``|!gGVp9@F;&krmGf%Z?c|Ut#f5D!oL>GHj~a&qvY0gnp~su)Zsc|7NAD+U#L&qceqYY*b|68@jFs`1H|vv{UsjfHaK9D_Y4YZD8m}P'
    'vyK>?pjn{tb%dd5(om|a6!@B21%|Wt{S`HhaUKu@k9xiXk~3V7DU8eTfcc8uEQmwLH-m{;<+D5HUOY_R|M295*W?u@eTBR}D0F)t>bK_O4|qm#f1A=~(>*bJ|6jn*mOBNRc-_#?z{al*hp9sUg4pNu;@X_G&EPtW'
    '_CWpaCUrsGI5U==^Wmha-5Ly+qLEwKK{@~0nVk3}toT@1q*f;DcTl_W?S0R;{eE}7^hYAb-S`yw#3w!bX+Ln8=OC~5gc<CB`_7*yqwju=7KXn_u3a6(xpdka_TM4fgmTio%SH=y7iXvPpRdj7KYlL@2QAee5OC5+'
    'fGqv<>Tnpr=a1Kj_DHm#IT+&cgIFFQ0hk9dhahQF=kLmg#Wb-jX1F&P+|4o0rY_uosYZZm8UwQ2*urKYblV*Ci`3sYLhl;tXt92Q;pSo!<C@V$z3sUqmq(29x}2Nd%xC3Dg+mmx7v*w3gVZwMI8q(+1H_Vumy_t8'
    'C;#&4j~{>W_0vy3>ZXvdl72sW{P2rU9zS^W#b<S8mm5Ree(84x){i|9)QeWnJjcjM`zYGWUi!1AKSf}s`%j-f{OOaYs6AAI2jn6Qh)I-I?Ch${hqU(ua%%pIuOEK>uuDPamH-<5oXg<r!Q(&t;^CKFMe7<yLNoj1'
    '{?i{*u35wEn5|M33(`y?wB{F&pFaH2<HtW40NUxpF9)AK`urhpCe!YrqTY3{4s>borj$ymb}?kh%MooahZ8>)3B(<?kW1`3f?_$$a#n)Up&QZ~N}Aj~Pr<KHj$PWuY>V9#sSpJU4f7yyhVnInN<Iocr5UrD+k$=x'
    'b~Ho@*=ERn1j(tE>xDU0;?$fk4M%2(2?pa}eUL~eqOhcC$26H{)*APoQ;c33%3yd}!`Pw;FtLUhu8K?0&D>6+cU9YKH*0D|&=q^>(UC~%?H$J{Vpg4>pH~7H7@*LiP-dR`y5*DlG{GK%f(;SclGMg9@r(7UAu9~?'
    '#NC?Q(Xz8zt;Qe+P&N3|ENoYm|LW<3$6r6iN$9GHU?<eUtaxd*O9h?@4fdeM^f|O+>jKXXhmLo*m*3f*wnK&MrJ5GSqIKF_%3s=WRn1I8>s2@9DBA6z_)Sq}v<0)R<+qWH+FqJhB|{5)0f{?5M2$vQN6*5OJw9YX'
    'W6IDN^0!K-`D~VEyL%qajEp4P>q{R8WJ_PMZz=<5EA64g*@ru$M-;0c1)z;Kn=d({6YG_{x7+({$<BT&q`2u;z)nYRej-(Si3XlXzny%Nfx}~tn<iw2Mr;4V-1@<?K%d4fbD(Smju{p6DH*kl+&N>5RX#%R5?0Q3'
    '>|o=zg$0uju`ERRJ~oPNH%FLnOugn`P)C$#pF@+c`5TQB!pPt&PYqNz)*T+o1a7izBsOG}Nt=sY^gR}(_o1mzNzEici%#Zx!NIt*i|_v9jNQPlSwz2$@^|NAgRxQn;jWgeNLYI{H!|06euZ_6ID$$db{n7x#|3R3'
    'rvi(Y{GAZzn-J!GGSdCdNYCD6hc)g$%B$ib{%Htmy!u9MXv6&PK9HrmsAGqcREpu&Z`kg*O{4rW?wD?JdUl7^iOlGK>?wshO)f<{<R<-pq&3NhrWrHps2Jp{Xt*c0Oz#f1ucAJ3&nItQ|6e7gD7Pw$Dqkn83r<?s'
    'QuXHcwI1XqYd%*En4>a-Kd8YKI6FuQng_VVNqmDt*R0QroIKCi(K|Z*u+x9*J%yFOo89Dt7q}H*3B*i1gnfg9$Ete`lVqkh`FwuSNv_QlY@W@oVQONCvOCmYZ*^U6zb1X|XODR_f1!gQA&;b1TO3i#u|J5}VCK5p'
    'GW8b5By#WWzTfKVzL<XR_H|80*IK_{vzT>P)(~%@)nxwNpRF*|&s;&E@n1~7`?WYPiv}>%{>7~=4SZCEww1wUz93WkIA6dVv7?nWk9-N3#TGMxD%Q{m4peD+1>g)T4h&rs%Oin2FnbQrmaKt<+ckSgfMqZgi66JZ'
    '{y7sp_pt`l17nt#Hyk|2I3odY@wfOJRh#A+07<p78k$f5?&#FxrZZ;oTQ-ylr8{*))}&T7iY<0B#{h#(jJq#t@W4FAZH88Cd!GVlVV>0@h3UI3vTglRL$1cnSN#*3TLu1$USasOi~a+qBiZo)Ev$n9q(G}Gb52mR'
    '2d~-*AH^;?if!{#3)I&{GN=Or-%P)vc*56*hx=J4JkK<u@0*BLT=hn06aHS^-TJjvP*bAi7Cq`JIYXq0ZnG?5k4D$h2Y&&wZ95t5>u`&#5n5mXw*SG){DnD*y%t3(^kna1s9rzw@`3XC?6>6{waGDxzfq1&2m3RO'
    'Z#8K?FZgaJu6~r>s*7^Np~;y@m_)DBO8Blc?c{_eo3XDU`yVZsnpVC%?LGT0htB7WJ6E~gZEvo(rQ#~tX90u4D@(%ZF@^DRd{@Di!S5;o<}k^V3Ud!p>9k^|SuU&ZX)@ntDYu4l)@U^8tQ)Mp-Y#)A5VX~`q)LG5'
    'tA1{n+2Va^?VI_~%pBSe%zq(%t#aZ1F80y~l;#;%$Q=^qauP+X-r~)4L;UIAHR$v2%xh}5)07Gddn;^P<xh5V_ohg~5?WGroND&uN`4Ppt>N6t=CX+^TT!%ZFC1c(1+_akj&iw=BM6GnuB+^FGZvkFi)mmG=b+uK'
    'F`5CQ>7H}4PxvDk7M>Cd2`<BeKDJ|t(b_Z?o@8}l;-OQ#P^-~Vj246G1Bx#~E}_K=VnN1$QVRZ9p#Q`(4CfdAr?B|NB|tbtC`KGDgQLO<TIMx}fwWK|xWJ@5OE0Gnsr4y{YYV5ynywf(s9b78myX7G$;o_P706AP'
    '-POXxH9hO?-<Nzfn5WtL&<Z|F+fJUmQX}iyzuG9Jm?tu|(O61iQns0qg46_7ea=odyS;20OcgX&3`NI$6e)5#H`AWbY!K@O)_VMj2Z1U*ff$>*?nmu2IR(gxuqYxoa+mXC(%N)o%BeZEX4^TV?fekz1Kk~FjILAl'
    '(D3EzSD;c6xxH7x30qH#WgorxaAm+h@EW=c;(TNOH?$yQa(jKRj<~$lLRlB|h^wK=vr`(8=scH4+7tAtc6`KtJu?aa>33}}nQ&jZuvy``g%&93Q1;`EWkWk#JA_1HI_PX;l8NTzIp1HNi~@Dm^|1|_*c~(Go!}!;'
    ')b1mBQtRv=QvRZ7X-+Wv?^ZyyGJ*K;)ntwOd1xXpI8Al9Z^(zkHiW7R%#X+wXoFdpl0mW+#t%1KBfLDW*2Bwk)xz7eohX8sDK@%ICc2sQU+$>>Zi<1~X|PTBUPF_&xGN23lHMK@W7vukfE|VwKKg#*&fl$IU0d*7'
    'mxWJx^o;Mvp<xUzaCn_Mgx!8zXk^}EH}E@T)@>G)Jt8hp={lDfP@Au!k%_mWl%zo5m9s&ZSQ*tu5x6y#ySHR*isP%gxDP0qru(*y9qC>GYsRVy^)@+z)`AYIeYZmnRGE3gwQUl)C=skk7V#izDueOnYI6lBbFEsu'
    'sh2S%1MbvcSf@V&E;m$0GMB&#GbNS#ICa7ci*Bb07LsZ}@*UyRd!pimYnhdt`zh7V)9ab<FN+bw)s6_?2K&wy!_FrAovF{fR$)@B@1ZT*eSzIfS*G#)Z{(;*9*S01_IbXZjV1|DFbrh@EdFsm1scW1D8n%sx_DJ!'
    'PBX-^x8Bwcz1}cma|Z+>-R?E{!;&VWx&k!;+`hn)nT>F?W{AO<n=R~=qa&Lvj|%$QI4r?tJQ=Y$afB*p(}m1RFpC2E$HBp9YGj1u{*=Xp&jEytEUPZ?#!VK)0~N2#4WH!l1>T^=xZ?bs7CZctCtu&bEG~g2k>3eU'
    '7O$ES_m`#FjHEDve>m6))Q|XQHwEM-Gb^c|QUpjA7`|Op5k(Y9>8<fY_@(4^Hy^?p*k-)yK?5RhYL+$&)i5HoQ`{G3402jWMn<R>kf0;j#@jxeFUPG>h8WN9zkhm$bXTnu0uQxD=pXI-qI~ktf0BIt#iLIjJ$#Tn'
    'dhqb^XJ6ib^5Y|z6L|B<<DWjc|Mbz19)14k>A%$PGw2&G2QKBnW*;7qrq2FuyVPE7RNYk;)E*o*tL{f9_<wGfd@`Sz^~xF8N^KQnJk6jClS<<@Y}EuPPC3Iw$F@ys=a^J7PvOX!|Ig{0NwHo6cEhNRlYLRlFN@W('
    '904&fWMVOgbSk6bIQdCYERchE3HzWJGuAMi)f}@dOY({i?A;1s=|{!bc4}s}H8lZQ@ACu_-RDztJI~6?_2n^EcK>o{_B%iZ@K<2|HA`8*5#Y%(EB&&_pC`qO5_^`cE6R3DCrR-2j%E5ddTYAEN(^@$o|n2}7F)7V'
    '^s>#Q4un#-b6eZaEpi0VZU)jP#C|MtxHimFXa##PFxSFmz8X#B_OXMvE|RE8KXFQQ>8bRoqp$}|Mdk|+h7~?}S&k|BC3p!GEi><}kJ`v?;RG0B_kKG$I*p`&#Pi|4w4WF@iYmQTgApgiB>Buk*&f!>p_*>dXZ<F{'
    'l$kAarp>9(r?d5CD^sT%FPod{r>4a@<Wwy$CM$9t<)u3|GWEDcb*f{jA4gSoP#=&qUY%P!LF9K;ue(hjz$7IfCnuyrrrsFxbRt+$k>;h-@@d1sWyxD=plzDtxHh8hC5)`Asim{SrzT#G_p4~}u!L(_rm5>T8z;6k'
    'G!5n5zTMYln=2zubbX1qm@RI=1BbSFO+Bip_YHAX-?*JxqJ4xvHt&0<kfTohK10|5dSG7m6^1-i=?3(wB@MPA*1wCu7>3&ChK=g>`k>(KQ8oP<a6NoY^~Jx{nH+4=H%a*sK0G=8{ST5ud|aQDn^GXtCCJG&Dghpi'
    '-Dl~-8gS+4=)`|^{LToig6jyB-N7kCg5}YySMB7g?Z{9)6CYP4rXIRT&CD$gT`!jE4(&l(omYKzXnWXj=c=!-Qu~)W$3Y`4U;;*p#JEOYbzWIgr78%<Y0H?=$)DcL>^q#uXtoYU7TYd>UT<16_Bf5N9|%gtg{Zdu'
    'eP#&G#$#txx3ROtUQ#b@T)pifCg!n>3^1JFZ_z^-Tgv-m{^|EEhx7HUTpf+(hLQXnG}F~&Re(xL7@fQ3geIVDzJl+-6U<!va{zSBRjK46UkHn{EH9Jd=WD}WK==h5JHxw>O!9J`U`xySONzV)(ZsM<qXN!HGQ3(9'
    'WZgFdAs*r(RAfG#8-eT6REbf1Z16GcPku3<<x@D(Mp{F}Q^Sma99$Jk_<fvwSu7BtN2ngpBrJE5k0AjHW=S%WW81K!f^0z?0b;s~x;i6oB5b73;rO}F>{1|%*EExmlz0)uRWT*(qkVFzP(fk;bMEE~=;5lkD3+;s'
    'G#2OOD{Ir2C@OsTF>rsxPnb=W5jTRHis1y|agPdPU^Kpy2i|1>)maZ?v*?3q8~?MUdok96>JNU9+>?r+Y5LfEgmv_&>``*6dPIfcnWDyAP`f>IIeC<u)scIfN7ccv`fYEh_)u;Zv%xN9P&toi=rD(7=61L={dR{j'
    'SjU)f3hEX34!lU+fua>Afm9KqYgcu9lX}SybJ=G@#gLW&B?0#|%Uh1t=5QVNT50N&;33_^KBzMuQK$t@Xue$LSFKB8jNrS0yYy%Hv>)RX(CdrYUo7QkMlXQ^2aHEOLk_#9LRU)c|4ARFuEsTgV`>}vwGG=*Mp<YG'
    'IzUrrd*9Fwy9JCPru!P!Svf~>EV03i>b4H}{<JMUc6COg{f6WX1LrYD(rg2fSX8MUlzLE2B&5<TotZ2ua)QZ-u(_@qI_kVmy;s~gy*dlbt#(bh`N(#2Tfi*rH!w4Bbh5zlpoA^^P0W*src-wwXe?5KZ&ZKFSO?8>'
    'RY^3efI4dZKk{CQ%VS6aFH}tU;DKOK`1>ea%VHWfeR_kY(->HA>C>=gy!D`3IM!Rn@{M0De+-E*mzDUxF^|&=nKUGr9li%(NAs%RYAA|CwPW~u&Qm-zgmAQyY?1V9fE1VBUxI@kzb&hCFmD$023Raqm)JzVmxbBH'
    'Gh&vIb%wb&GzJj+=hG{*y|a#{k?IQO_iaPWOEez=gOqJa8p~3G)C<c?@jfUrtD0Z50Wf~mCioamBlQApQv9uI<(1h=SLT1O&dh($fC^EW2o(-H@UdxM^xNU!!eNOxvp@yPnS!dVcPTb_@z{^FRU58lcWCWeFS1&5'
    'YD)HLnI;JiAJE9w0?2-S1nq?1p2ps~Ltv*GWy>wK(c3MxX`M+pTpV4?U*3#n=hZ1FA6}u1d4+#poi*<}ku`#<I9D?0=LU!rH5kpO>&uyuZSt2S63oX%H-!Y(sfz!Q^tqyk6RqRbT~y?WSu}BGQBDt5CF|L0zDB5W'
    'V)gGs?(C)E^-Hjw=c@!@v{uOGuOR*AxL8S837nJPKlvY%`8huYpIyt1(`3%Vua1+iXU}Kzm$Tb!yRm&mp`vm{X&#P~KP*?1lFm+^R1<IzOlVrn_FU#9aUm-Yrru;VUzq<HE`7C{Umk<|oT=J&i<KwTIm2zwUzplZ'
    '?^~`aH&Mg65w1aWGM8F0Bf|^Wpf9FZi%BvcnM<-V&5;-~y{btLz6}7E3yc78<&W~|V7bDGCp*;bW&ufqg7e`71XF1{(N)6Mb5-(gCN%nMv@IY;uPp$vRhUE?Onn$pL?%j9)<SeZ{*o+1s8YqJsdHD<mIzm;j?}N}'
    'dKBEn&?U(j{%Tb<X^>spgu1eiMj0zaY8ck&4TdXh=->6>A#`z1)w0<s<YN+CRc;7OU$&0`iLj~5A{$XMvscKzLlZaZZaEWUCn=xgh)Jhq*C5X=ta?a{0#+M40bMm@_J~Q`=`rZeTCd>AE0Gb<RS7{y6y`0hOnvL%'
    '(GX#9){>4{bscObsu1+%rbaA=wHY3gX0(K!+AmqDXZnTTWxIFbd}@X`Lt*IT_zsMSSP)zgP-z~)63C6>FABxVE$8b+sCb=?3oOX)1iq>t;m}cciP`0ovXdOq6a!~`0vwqAb%ywNXKEm$0=#oDGZPhpw*1fN34iJ#'
    'Pf=fy#~D^F`q?`=#m|qp>I}5Fc3%9uTzzrc7v2=xQ_(&(c9y=ftSmKSs0nqH=aC3pm!Fax+NMv@C5yTqn_YYr&ZwP1!!FapkYRF*8HlW%Wc5_^i@uZIhavpFJ{~@0#R6dC)LI1BZ`Gn52@c*h=|1K;O1k>)inl9e'
    'vrseUZB0ykK=M5kMO$)!b4Fy|^@ULN@iX?b!j^qVvw(v{?0M0HK?Pu`;Xm(n>1+mifY97TQ`7atiI$zv=@{%~rwWOV2pTlNXcaOYx|@2MdiYI;zP!Z>C^byHD?)9BomANl7?FtSCS_hak=46J9b^J*d7duNZzM?6'
    'T1Q316QzI+Oz--R+TmtF+;DX{cjyv=FM^VNy6e3)?#S73Bk~Rqj;WJymJA-5q}voyD&uG?4kP4p9Fsxk`v;!hNKr%>4qjJSPRfJcoJ2*#^x2SRmK8q%m$hOkt|QC6gi7G6K?Pl|t_Fq&1ur3(#WtJ=5iRkt({S-Q'
    '@xP*DV(&CtYEl|DA%CTJ4t*dyCPic905j_GA!28aq#N}<acvaGRgo`8lUCX~zCLZ=IobR{_Uzf|_33FFaN@X{tZu$W!RHdtlMv12lAJWy7JNyD8nl!eac8-s_RN{vPO_RQ6OQ*sfqo_!KXA?~RBjI_vffX>F71Rh'
    '31_;8#{oR{0=rZ`Y03SCRl$s|hpn`Ca`eN0_`839^bddc$NgKdZBJ5iCy8)v(ocE~e9)OkV^j;AQN6_*wdG!A&O;5V`0%YJ499bIIh;=`=D1qo;Zw8q+tu^B${*uQ6AC>)xg9Kp;f+4n+DH0uFrE|K%h>NJQw*j#'
    ';HgZXi<b|_On9f#aMOkaRN<M#&=1bVKt;BkP8+xko-a_AwmMcUUYB&T=%qt!VQHwCU4VJYHcJJ2*8Qsg_AR({A=G7x7hwN|dOKzbGMBTRTBPogUJg52_at2u%mLtj!*AzCpf%hVJ*a<+{$5Z7Xf^h!rmuoJEt8Js'
    'e0p-v$OIn(2VF{`9hdy-YQd5SGu`x+cV%-Cw?}!@!05jZ{5qG2ZBLq0w@KI40yPx_B_q|26e@kW5w!fY-gm9EZHI0pXD6W#Nwig}3fh^L){rNF2DGiz!5}%V3id8O!>poo?7W?YFE@K!l=vP<{Avz|oZp(67{Z~2'
    'FX{^0>HOl1q3yw44e>wX?sv)r3UDQ2_##wyKE-1Az%7}kVCz99z}A>KgNo0=QenUBfo2q4z6LY|YLhqR>9RjU2)1Tlg82B9^MI$z&6}7-$HWi)?SftRpq%Q$lbbWkO-en&h`Azd)hg3gWJRqi5jo&?qDB}F=Dqjc'
    'L!)L=<mNgjP&y+XujWafoR_bx58_>OW|tQQs?MCnteVbO4oAjp7DYVW1|mM(cA`%z0^Jw2`#rJDhK<q|p;7q~4s^lyuM3;Y0v|i_qG2`(FFv4+5EcM+v7*7!OBhrP<-!x_sB!XN35g)~*$NOeH-3w>(jDF($*I0('
    '{C>!E`1d-@z{g2+==*o(lsmI$+e^_yV~w=;&=bR*EJz+%LDIO&Ip|d#bosZAEXgnNWcHDW*nW3yE(X$p%e@$J^@SXTqv?8Vs61g`aGr_4hqxix1p4KQa+-rT&@0#67^?)SV-6&D$}Ti?d$83=`Obv?!M?L`o9+AL'
    'wX3{g#C7G!o;&*ushJ-+3JOYZc2^?t5*%{opsp?r%Q`pvuUa|!l?)5Bjs+fCGdA`RMi(G6u*>|L`4W<?Tjl?~WRg$MLA`(f^Ut;KJWc9d(|egFe8Nf(BmX5Yz?<ke`58GCnKs3_xXpp-({hLjx&at#m=8<CxRnKl'
    'cC41`)ug6ShmGtu400aZROa`!-J;fWHWuJ81<zM>(Y*g-QW)*LOtc8~Tf{1EeX8a9?UOFqq&B41-?T6pk|P#Qbm2;1)-_ivg@<jMkB8C}S1m>aH8rk2o)MeQ(#;BvXa`x2P7fRo4B}&(hzvxu2?%u~W(t>SO|3Yq'
    'I^Y#wV5vRMX-D5?sXZ?ov4p^W&XrFRVlvl<5OYYdK!MWd42e^-Ant*5V^A5M9Cat4#Y@F`e_(y~<&Csqhl32^O{R`1U>a-+K)>TBqAk(GrN)bVHUPcNd~r1xmN*Uu(}U>hi)B$j@xfxW;x`-TU#Xd(6b&VDw#T!T'
    '+MQy?XE&*E>Mi>kkFC!zN#Phol^@+E(j9oJ6fe^nyc~Yw^du6g!5hx&qSt42Gip<Td`~QaE$ykuSPs=gWDzbq(8F*WwV$X+2X6{zSFRGrT67M!P07#*X#>23QlXJ*uDjq9K%sBuV&HOHrp*PO*z?}&d$?b=ayhng'
    'ojUJI2hZolNNb*43mbZ3&jQ9I@CzOqfdWs+5Zwgr0^tb3uPkurl6$dR-`8%npbc}%)MkL@`dhYqBy1rPx>DOmYKpiG=@`4QIv=zdDNyE1-h2N3%KPl1=pGG2Zy;GoQ3<^QIsVG}j9bA7!dsKuUX$SLq<;FziNd`Y'
    'p@03}d3v31etA9Iq=XegZ|crm+{LXBxwc+2YzJvV&$2D0hg(W(>M#iG)LOUVZTL<j(#-bt-wyM_KILr}dev=+c8gIfy={@|)EL}7wy|E$;5O-7zz^wOH2F0TVmH=t^ekA68?L4a@)$d)`Ud0=rCNW;Hq1ilze)ad'
    'i<Gpq+95WxBpfoRVYGlsK8bP=$Kcnb-1xc$lCIX2G*41Y9(Ff|;vjHa>WI2cEm&hvlbV|)=jI8DDJcl;ASTf6lpzmlJL{??mBwyFXZRyIhp44|(Qr}AYww<pTpKb&5^UnDP+TM2xYk?L)4w`dJUW@3ot=0GvN;2Z'
    'M4Z)Ym_iIo8Q@CT{m6DJwQC7&MgjiRFNW^*CfmY$$O*4;+I_d3;3}hSkcn}Ef0;R^Kciw8zXHHH>5aeV$hKLr7<LR2K1x)IHwg{)x>n`f?k}5KSd@oN17ZfEuidUb&~D>xGt>oYUb`$vU(Tx)!UxP}t<%gLDMfu)'
    'YA085+;05wy_(Q`TNpy`MrTRE)Nn)wB%WQxN*<yPIPylpTu@IVN->%Zl<siu(vHj_0RF+011AW=i`w3mg*)@1XWei%*50nZ<p=6xyEoLc`Yx$gjl)SvGu6T$Ac6)`75@a8i`T(Cls`e}q@}<F;P@9ymn3pKkuF#$'
    'kYQzt{X%Z$17VKeQ%AuH1JWUAgvwtar|k)P-ic$`*K$nsz~h(bQO|F%9~ar2@Z4BqgkbR!MNv?AO_^0-Zx{SPOIabg(zqegw?HcIBU-0Os}3Y2r#Qcw7~RF=G3xodpszz6gE`HFQiYU(;32gn-aS1zvDS^+R5sP)'
    '>0OzgS|E){UQMRuP<{fsN(^HIHJT8EXUdAn2jE^A@E34XEoMsx-vl~SZhlWd-J_3=s`4TgyRV1T#8)Cm;wpYt`#K{oX_A{=xhPR@#gM)$Gn?qj3&||d=>#;BV4J8`$e}Kl^B2VoE&ywt;Mi!!x20bSGngT2NU=Fz'
    'ULqpfm@+IWBh)eHq-dqjo`K5bc3MZGcb|Kz6P&5C8stOM<l5|Cvi_!>R5TavLyhYzDqSNO^B8U?F~DQhlnaPA?JbKb#yk#Ia~XE0j(qLu5WnP4d@ph<hF7C}QB-a@mh-tbsfuktI>8e}xf9(O$!DWwrx4?**~4~0'
    '{#2d5c&u9T;jUmT<ETFQUysab9W2)~jJ|%6FUuTu8~>L7O2?~o)AyPdGDgiGopu!VGL=C`L@wQ_Ii%6_qDvR7Q0tJ2dh3H_ZBoE!@B|j|QcCz|R2?K43Z6r*P47iYh7jOk`v=`_QYk?$7BYDP>r-hbCkK;TFNS>F'
    '3k+>;MUOmzAerA1TZ2D*l<G*eJ||b}FRuKK&IV=PNl(K%{cgrExY<`6akvBbOCRLhH6quTzg;v)H&d!D!|))zeJdWf6I=sRbq}zWe>#7uy(u7lO$Vc-IZug=SBEQudyCQHR{9jju6^ze?bQVlBM^|TKwXi!SYHmo'
    'zh!E1(S<y)TIAGMLfhPeobPIv%StVyf*FiZi4dT?mI46_{PV9KfAOGzD>yEKi3l8<TQa8L5Cz|l94MVTQ6y1fwhM_QO7zduM>EX-l0c6kd5&43P3M}L2l!pn?c>YpVgq5wz(|<i9#3MXMo_hk=o0{H;npA&1B^B^'
    '|MbJCrq+*h%x*KkSmuiffI-e@M=vMlKp}>e_I7C7>t8Muj4y(i9p{kXJS{zUIR?b?fv<S=0E&x?|Kr18MSCP}AEsx{Et4JT2t*yT^^gx4yOKGGIW()Qn4(%h*%xvjBb#NT%0?*bz@dYAbr4aCAz-g^n?=a_2a9ii'
    'G`NlXhlYKxXLyq{?SpU;ashA~UAzA+iwh&@(02oK8mA=~cUjuAZ8$$=N+l7Af^r67NWaY^us+zXL&#f*Q$o6$&n_<3>4pUf>&1<Zje=PQeQ=wq?YaGW$HR9KLln6Ved2K6<ZI_UPKw18z)8hX5fVsB^iM~vhaXiY'
    '&erWD^rGx+ahImym~X`6x#KN}^@B!alu_^&o|yZP!h8{Hb*+Fx<$|H9W=NEm`4SL_E3yuQzevA%Gy5`%N>Je#on7fEh($Wr8Sa^_AS}|hw$J+|o|iMT`9KYu;*Nzge15^$r38S8Z+UzNTPX0uSLCq>;4C^VI6#ye'
    '0&E8%&=G>jFU>V(r(hpdXR;%{gwce^A2E)6n;nDON-I6QeJBYw>NhH0=FqSvgaC{I!0iJF<Q%8Fu3Acj<D6<eCZvJ)==v+c22;u!AsE&{yU&X&1bT3&q{pP701$d{#3lURx_Z}q4DrMt$1n*Z%lSodL1GlthcpE5'
    ')HGJF@+7X;EvG>^51s6BZ0=M*wJDNoQ_bdXG6iQCH<Prw3FcEL%1l||i~yE~X}7TXo~fo!cpAHI7ymT+<orM<b7{=hB1ejz)ukeZ1$Ny626rWRz8Njq%B_~eTUNV=P>VN;HCrun%Bx6j+vHILaYWpd@3R1<gKQdG'
    '7ULRNH2NspNCFpxnFZf5M3z*ew;Cx~3Ev-7jh%dj$wmHPpN{BO@?nKYP(~n!!%88%+$)TSp3TLx4LIl$-N7ldp$}10+n?KdU@8R?hFf3A&$+qz#;Q{P_<~aFvKS9wt5!t99A(3I1K>OLiQd)$u8xOR-E|iO(RB0s'
    'DxnFfM$2*to@*FDPRb^}zyc5M)Pml26n1$kO9kp*{uA$A9Jaj;tBq2G4XXKiIVuDc4DK(B3PwMg?!a*gcBldFF6Y=7id}{;`*B`Plsn9m2cK4k95A{UoFcqHD%lv7nRmz=NdoJ*JV*I)g`~qV)$m<4m>_uhIcQpC'
    'LKv<a->dtQ%0qTZf@RwI0gel4h8rgN5$NBcL!P<UY}Onq*`f*VC_hsH@a3PidyK!>3231;Ec8tly%Ng`H*;$bj10P$fWon4EIFADf_s8>g*Xj<wV(t>#+$Xl$L8Wa`{2VMPXL~HYy<ONr_15?%+@HAOxCmK7?1!K'
    'EwZ$BB-b?%+t%sH*`4Gtf&Uo7JWSIDyiMw(Xz;P#NYX30g1Gdgc*RQ^Q8G$#jv_E8EASA<jqcc->mfaD6Yv#nVtao3kWDKn!uwX0QG=V<o!a7)fx8<1)=90FBMlF<2tQiv@sU~o35r7t5{f3kR4#Vua?jHT_8`+%'
    'B|2bd<{JL}!O}1e<2fmh$u7c3#Go_PjDjJN<#~yh66k$e+@;Vl9#9I5ya<E_a8%-R(@p><JU#{OI<!oAE>%L4&<4*FZgWl$(1PbZdDe$HqK~q@Shlb16eTPx8nOsVZ@9~9b5`f%T(72v`5Rp^l^?8UGI6n!5xK^o'
    'g$-ExpcbE#nB*@Cyt4(ugx3{L%UwDG@<`c1RA1?oSXTpLs8Emh9PTpevF&fit~tD<c{KtmD&A!?STpOUMhK<MP9};+Y)6ILbUm~8*IgMD<v(6sC2mp^0?q|p1ypkzxefrW%z_J?hAz=lI`nB&lv4yj0&A27uEP7;'
    '*(?TLXmyJNr}?^u@DGIVH2b=si@dZ6f6VpyGtRCI&h)@t9hviAUgjQ_PGVt2Zd-&apu3PIFIkkY%tRk6UwIY5ezDAF>uHXbX_7{5rK!M2g#qu3wqgpOCKl%U<1U5ow`dUfJKKqGq~?e_u3NLfC~X0JrOLC~1k1;$'
    'W2*L;Ri(h134K7ulcB0aPiRhNu|xz^H`znrN6nNK-lY1|pk`q!xL6IJjQW3Lcry5G3p&}W7&_T20=L3Xw}X>G5qAyYPq;06RMha7JuK}0I>WzC@vnZYr|$^HEbMz^e!dranW!vz%nH*DcGH%IUztU`qKvE|)ELwR'
    '(-VS>;d3YCl(W{WDic%;_<5y%=CAPWJ>GV3@Zi}MKI2nDPr06ZUHQm8zjDv7woap!V}^!)L9l|<dRq>rRCk*jZ`&@qBHTYI0>KM%!aJ~;T+u66$a6~ONwP}85%?ev0^o|4jUcn~5lf)_lfRRAv8<z%T}s3U62wI%'
    '0q6Yrdf~KTrR@TFt#hlAf5;EZK=!0I!&IdUO294<ti4l&TJ>z(#+HMQt)Gdm(LBP4dL!tP8V@vCNc;!prnvAjfaEXj5O>rbm=|vd38V05wj;wdpu=D>1rS43MRjOV|7q6F+g0gtbWq=0SExEo!@fYt9<|u9DF;sb'
    'q>kMR{?gbT0EO!AU<ul8a$jPaZ}nh;4{Eb!zKwdGbFWL;VR5mF-tkELVG4`w(4*1suMZCq7>`XS9XTHub7U^mF}uX^4DvJdn-iwSlmJ_;#>$~TJX{$*H-wkIS{Xi9O(@xUcGH2wIw=52-^us@lwK8xS1t!#(u=!8'
    'SKZ~1#`Ao*B$X;bzM)8CE)6fzvtUn8WF+TKXGS51({lFQ@O48AyB<V?f?FgozT$GEuomZL+fMQo88eTQ`w3&knR%T~g?CJ3nkBe&vc5F@k71OHsUJ~kY28jDYp2!DrpiQIVPYt8@Kf%t<Qxl_6Yf$KFGB6~*|!d('
    'C^={iPQOM>mB;TrU=$?24D_J<0%_x^<&pTQ>o6tv;V1!iBCIKV@NrZ*gZ4p>aN@kHP|rF&YRc@U1N{=I>E<`hR<eZYy#R9=&~Pvfit?LOeW7y9d<2Et{1&0X??sW9qqD;g;j{;V`f8G_r)20}%}&Ff^G4mR+TK92'
    'x5Od1U%%ePC5vIG=>%Z05YJ66e$2N!8}H@*N}<*5BtK_lK8l^p>Y+uz_Y8@rU~Fdbkga;aL!Go4Lf|O*DC;nuTc;_t51&A(wL!sF_w;amhI@;@4$fwe=MP~8vFz5-$RI3afdw!4VgGm1f12xCyZi5b$)4D#tZFfG'
    '2upQ7b)X?<R)fSP(N~*yA^_0UKg51zQ5MKH9t&f!d(*q8W%qgB+<hV4<$Sia<Uq%ZA_W=PyLg~<0ea7q1H*d8>gI7u7jd=7XM;&z4G4o*Lap>wfreiA&wSyoi<qhdau%*(4VG)nX`Lr8C-W&9T!Gr*-8lXx2bf6D'
    'Q`(8-JIKlZ*RQDm|KClLg=LTH$rf$8p=^Qgdr562<7o7-_fiw_rrc#nyz$R|eG!}XWS!Jm65Wb#&APCY`NZ1qvvd^>Xb1i=g+J<lkN#N_yl?kB?=_k#pZW%57dshJT~}C0?E#;h`HaPbmQb@w3jz?-0Wt}I%38%J'
    '5Af9~r|Ynv_DCPn7X^<ckmbz&Eg=|J&0nAH{`%FKg&qpOKHH=1=C9x1{dG<G<0}YjYMi}vg4<<8Sp-i#-O+0j5g9kWILNUv?^GK643UQI-6U5Gl4@)gVuC6{OL2hm1P*xwL=MO%%1;8OPcF#d4G!c~ihvD5Uv(B^'
    'Uc}J(Ww}C+FV_xZ|EyRVT`$aRjSEU^3#gqh$}uRpsn03cNaXrfeN|$uFAwri+YZXQ6}wg){CkwH)CkEwoO2!a*8{0maMSQV4k5&(J3tqt1%BKFRoxVdwhO4bM<4XFcOW(I)It*A&%vI0Ja&P>JoM()&9uH!+yVBu'
    't*xVk^}}y~$96!&`{V|2dc@u8ONw~1xrc9@3&8b(n5!L1K|ipfUP*{Tf+&!31MShbY`bPqY*~>60dJup<nF-?MwNw>sMsM@ll~4bUaNY;2lIvf4OrD!?Iw%qdnp3byyhK!Az+UES^$QHZg)aFoi9HGLx~Nr>97dk'
    '2FR)AifRNPgkWcE;BBgafN){GUTU+7Q;|8aR2xVpjkw^lxF~QzT<vvA(*SVbJRf%AdBigBBrs_ls*xM0_5Y{NHib<iuO8vuZFuwE#K0ZJP@tv|Tx8^eZe=leie`yp6Aw=go(>#nmy$*|9?s$S5=ui<TU!217*1aB'
    '%-LL()`&zEe}_l~S+jUgxk5;@j^Fj8RrI&%2QVhxR%lL&$Io_^jX$$c)t2ic@e>!iiD4E-Dd3u!aq@4y0alVW^<aRzBh4Cz(-YScdtS-QxY9Z3{IeyFVtT7|r9SF?u!FkIs*h*~<2I4<TsjulQYqJ51mbT14u|aB'
    'Ob9SrGABc(orekoE@4OmfIcIXGQaOf8`frU{G-+eZLg30s&)5ls@nZ`bZprD8<>|QkG-YYl<Hs$-Nu9lEfKI!=}hAmKY>A(46-6}Vt-e&5gIRVx6}BWue#LhD9PSS6!tC4qKV$?>7VyJIUW^oj><cT0HQ4k=MtQL'
    'e8<8La5a0KNdKeYI<+he{bZkDEur}0AOr2;-{M|v5onDxR<)4zY+Ev<#6<1`l8CDNzSCcR0%Dx?L|f(+h*ac$)lL@Eb%oJ#B;o-^VKON%CR1>8kZ?JkFRR2vGvSj*pNqTMBk0rv!}A?jOk#0xGULH6A6JHpJcv5-'
    'gwnrA9OODCx$#cepRM)kG&$t-+MF2AzPf@}>}J9#__#CR5C{35^q1zXc!l8Wt26brN?pMn;Uw|y_X)BT?)Uq_`_uk5Xj2JLRZ^~5C*DDnuT7>7UbJkzRKo0j$<s=<RhZvQ>%(K7bOb99+#v$UDVvx>0byoU)f|on'
    '5WJ>cZ2r*SbF#Vd33OL2oi}SoT#s8mJ!#%+w}Rd}-13XaGJ0!a%VfG^{-o;OGWG>~+CiG)SF&9IwpNuy&gIH(mhkN<W1fk7klWuvci8!COyaUVJ;(5m^eMANj$w5q)h47i<v&d~`e5apPP-{VzhGwpGOuOM*An~z'
    'asJ%_1UIKs0RIo-adDk32~iWdM-m&GR0Sx}K0y6X?tk`h@X6yZzIyn@S6_cMc<}Jk`(J<l^eYrS=&qpsWccShE^Yl_VSOmfi_;TCs}=fhKR7)@P$OjTR`%&fC;ZfHR`c|XFt_oA&|mZPgAe%`_XV>iBK81Q#V-l}'
    '+T!+gk`M1}Xk?#0`r^T(FFqT5e*Z@gKc`W-><t3tV+WFcaWN?kkuBeR`}gbQ#dm)@ySP1i^ZJibDtF)6z_{?s>6>r=X3_jz7%fX5RXH^I=JkJ0R&QSax8yg!dh`0<j{FZX?n^cHv)}ydSu%X{?Vrz*@o)bMB5z*2'
    'dHpA)`8H3gH?Mz%kR-Gge!fBT?I+*;I(a@>o67|Inan1?{VQ}`ee?P+&5Q5;+WdHKe*6j2qV)_v#LnCLu$p>#@DTs)U**YS^4)(l-7DX`{x>UdIJtu1ef#$b{AId(o98#(*nAKW+|1{)g~|m?X4R}_S7J;=DIzuO'
    'p#%}V_2Hpo{lBk@xo!qUQ>+>*9EYkF@(9@N^O;JY@C4ZA<%@#DVXW!^*uUZxY(YczRcG#k>;A>Ms4Bxq(xpXuWZ@L7zgv*jl3n^JHa&Y=-Qf-Leu95jiO{0;B^HgL|0VqbL0x5e!nU^R{Th^hj;$tTA7V+weHmlg'
    'Jh%ITy&LySr_5}^@78Yk?8d6jIG`?1s!NX;&PCKk-*K->FW9gwb_`X05C=BpjUCF+)coAT%b_O%-~dM!rjMRNg0!A|qdZHUjXfNYgGaa=4_0%=M~LWB)uL1VUMeElJYsXN?FTiv$ceDMxOwe?arV|7TZP`>AMax2'
    'm4w}PoAnpvf``%Okp^PkwuR-XulG{NA@DqO4q_7%&8XH^{S}GUr#KBM)|&ddlP*|l5Qyd+bjLlW=oEjY^<aqi!oK${lfNzPC%2)^q_x>XxGz*aSa{r%w!$K`w?HrCn?m<3)^h{Htv{#AZ*8f@s|Cm8!nJ&rzNscq'
    'F{|gOYnHK!h4#kx=cXwEst3)8cQkfY-=fSBqK2DS%S92Cm+MObEn~k9NNNO3G8!fomyCfVX6gi1HFHbqT%`RZK;W}QvWO31j42t}7p3MLjvw?x>n48UGVSd!YtjIE_)*`__+*1;c(hyy!a;aGI#JgdDHuykYt8z$'
    'uvtDepDFbx?Qq97_4N^)7f*qXYlinfc^84dGh`0+Ss+QLWcRWskI+ocU&4HIeLbv><8>_(s&7>amsFdL-pQhx9`$5iNd1TvP%PYz<#mv~h!cfjurf9@n?!5&!EI?SHFU1a@gl5&nC$R-ixkY?7}*ekHNq28PyMTs'
    'FS#qe1TQ8-m4a3CM8-a8Ug^#xB?PVmfCO*<?)L43C2C$V`YyBE7pwGkZ|^z`6Hr)3FaaC2G3=b+Re=J#T&Oy&ZO8Pz2hJ)-phMvm7-&FejtPh95PKEqT-EtNs}o6A6~kv%v2zlU3_h8zQE#Yx{ijz%f^A+k=*4PI'
    'm=oywpNsxh#DRUzmska`C&BK^<pPImFuCX7IT3lCG|{=Ig|0<AU8^KJX0pqR^?Xebn>I0ht!fJ<p7L~J&K)xGO-dyAf<$ELquDw<=`Js^-A-+5Y?b1fXOd>LR3EpShr~gF+zLk6;?y}3%CpCf%M?PMQIFMvo^PKX'
    'L|>uYL$Ju^*-0=5R%0`m5ZLs|rk1WAGgG$NOHLicljGA5np4x8`e1N=C%NZbo=Va2_EN;dqkG#^!&~1OQ$v5ZtBrQgF<*#&6eq&(caR8P<x0o3ReR-{s1tkMS)zyE`N8OL@ABtCQn5IdCtD?)e~avD=sQ&aBk8Lx'
    'dCMw2!-6NeoM+Bf^?!^UA}Hb_>-*sc=q}`!hsTiD!YjUGL_wAvbYU#S>#$JvebO<<-^ozx$)H)v$5c5V%Oi5+K+eDKY4@W&<c}1qyFfI&p6a?gErOITi|Vioghb9?n7w;}NyAXHtLN}`FnR6dPLSMCpJFzSddC>r'
    '<{oF>iRQm*#~MC>KeLUVH1f*YO;F7p1!%&f5@5tf^9f>lSZ(SqlH%I-bzGxqUS2{rT79GOL*7tBb0Hy?nvPe9<iFHRn$Mh_170^XfM#!C2@CAVjd8wHZ(ree=<sB_!0s+#NRI0vR^zAlg;wHK#naeamBrL_&X(jW'
    'K5J|<R2>Q!h0{B*SPDCj=S3Tn*F7hs(UDujwBkoF4{L@LkusT0HomroWvU$b9404gF)j0((|*?1H&@+?90rS%LtBptMc^VG_!5v+NfkS7Dn_IwaC+WV?EPr5e$=g*16M`9B`8C-d9b#mChey60u5ftmMw&w1A`HX'
    'qW~D*a>6w(oMQwZ&!$%vR;MytkZJa)LVS6O`$P}q87Dg=uU`(6IYJ~8RP|_x3Pk=Ddv@dc<xQ4i(~lhS<YD^ForSyR5PrW;zfYC#FWN0-H>!ZV6@g8`<8XV1tOl0dLHmcm<3c&Fr?#bm^m#~q9!j5w)aRk?Gi4|O'
    '@rsOZ%sidY4$V;@fJ!p7n!ak~Xqz08gmCDWeIMVvugad{lheKr%ye@60T{AXPjAyCe=MOvUXRS(gKux?FOIx26x(wAK{Nmezd8t)Ty0N6#;pJt9z!|}2`J2r3q@yc2w!mkH{Tt^;aWr(3TT2a8;fn(er;S-YNltu'
    'Mc$HG0_j)$t#=!rqLhR1n_XNjx;T&U`H-LI_IWrX=hJ-l97ys7XQU2<xcN(SZq0)WlrHhV1M+=kg_S$bPs!(zQ}f;)8*)TeBuVXN)9O;_N`h@8r;Vq%7+)xJPr#L(AaCfRW$a*rUQsa@F+&P3r+KF1OAY%8vrQ-4'
    'TJeJTTq$$|j7IJz7ex!v;-ga)?{p&X452<c1#;a=aRjOcm{iew*0gb!OZW)v>VDW1qGW<j9lL~w$B!Or*%H(y7YXKKAG}$oW{N3Ed;|s#Nb_|{oOuZw+!xN`28ya($|^8|;HGGNeOm98j>hAUmDJ06MxA)7acvr~'
    'BrsQ|4%EX@-N0Q?4ue!t$0Wpkt;B>9alUD5ds-u%J9rRLFwv2+FQmmywgo~zCggKY!_+1V4O1f1n8K$ml9m`7Gponr0;7^I1`To$yb{-pCHvJ->)*c@?4<2PEt+(X3u2EBx@`>GVs>DQxfrWF7OnBm*YoB2676jR'
    '`?97pdqCi{kZbP<DT%2Ff-ljD23nv9Dne=<<|DJj<D|F*J{d-1IIJ9#2UzkNruUWZL{kQ0-@SZ+`)J={O6&#XE;b?@q*`;^9|4MKTiPRMl}|=kc9{ZwO;jy-SDj&JAvp2$3$Z$dNQh}du6Y%8Ydg-zc6S69D#3T_'
    'zsVUrrUM5~hjvHMSY0J99c2Y@5G11}-6I{5r1F6tG;mE)dI}tg-T<-U5iLHN!tJjf=^si-MXI}U1cA&}VNUJAv#jqtq3Ke^cP#YKY1?W60IWFVz2JVnvMgXNF|+GTJIvXAWqJY<yBmGzKekJy13YM@x4YqSAa%J;'
    '66wN8tVV(NQ#@>47|RBoEpxm*f(VZmp{XD)whrN0SEAE2!WNFbm5Ghtw>Sef`z^DvCx9Zax!A^Q+I`w{l1e|YHL84*E!|%IxY@L^AV@<=H32Wn*$id68TbWYMa(i@m=nk0pU|9_gVr5i@gmSWi0h21ZJI1=q~OIN'
    'G>iayuF^c3nWu*Okxhl;kl{80Az-N=NIm4eKdf58GLW7;d<gEV_qzL^5_f=m)Wy^}Ww>OD*Qs-{M2()C9+zekFEe#Z_dI!zLb@YX;q9kS-O-Kn^{$Ue^P#38^>On@-ttI6evh%NEjn><+~llfk#z?7dwBCzBxQrO'
    '?_ut`0wBcTRxQx@r*}2P6~R-}KJJ{R7$3@CY^!a&BAYi9EFs}548-}8^^5y$FV*@D_Sf~{p>jMR#;OM^+usDbb;Ey2{#TqV^V`d`i-Xa0rV()Enj|Vz55iY2$*AMEY?anv4Jh7F&mCrH$xY0Z0i7NPo3w0X_hCB;'
    'qA0t>9|vH`lm^Z6wS>+u`rZ`;hg1Zv*Wl1?S~?V)JjIB+baxdVA*uv-Fg$3eC)*ZA@%8Na3_LPW8DR%2eRdvNWK#C9R>9XnVXW#`g75gkoU~Jivm-rL>4LZQ?TOzfG#OXP<+@rW!-C1zivl%e7qC$?H9Oc&gGa*0'
    'X2ZZuW=qu%H<mf9Ra8IwJ9};QKe>@)3P{JYdI{L-Gq6WZ&GAFfBewr95cK)N=}o=CgpOfgJ=rpbZJqS1vU7S5-P8PfGR-zi-VxtsTCe}U9;{1!tWgnpV#cXWR89$byW#0l_y{PIxzykhw`l7V&9O)I<Pbrca4cif'
    '?f3~Bna2p(4;Sf-{n3O8Utpp>_<->kWIROSiF7lvfSu3Vkfi$-JBLw5UkZH~O^VAL)Cw<FmuLVuvU}_3#c4{x<*a$|tb(F}OqrvAL`e>YlX_u(izwchpjx`f$+3~}i{Ko=her>n;BjVy8|?i8Fix;>?R~==8Iz{a'
    'p8_VMg59;6MM6O+rAu0ifhYAzak*Gs=`!fS2jxV)l2;Y_-@`nePjlBLwYx;G;`EoQwfgo7)2s!zZb#D6RW7pLCVKEAPB{I63)_eIAUU6st6W%ODYV(OQ!xx5sA(>DbuR@)_7Qu@IO@g20}S;XoOdAq0J~(EO}<*L'
    'SCeCyIQ$y$ZP2_}N&1KaA*D*lJj1u+5xPYS6azXJP4ek<Jz@zIYr8+43Z;O&JOGP>ag-Gd6pdq$!iYlu;cZ1KA)qQQsS&2R0mJxl+TVEBSZ(2id!VJg7m!>P^h83GKc~^VESj4<pXJl5s;sQ@lN?Lo6!7_gto+@u'
    '?OM7M&ik#XGb>RHaSr*z`X^*hwxl!&b7_mXWO<Axa@y<fNlaOW2%D_=1QX6gyYcRA2g?(4Tg!Hy08-YGw}I=@@M=}$%VmCLEftKewr7Ql<=n6(W-*;lQlpNsb>uuoJ)3_P82v>D*-q#SYs)uMiZl7c-My8gX{r>3'
    'W~V4}I#PrI;PLqW@SYV|DHS_4piu1wf}uIpsug;hKrPC1L4K0Af;fb@MzT06s`lU6n=#sHpz)Cd>R!@%^i3YqwuIAv+hcXmAn#e%#F}r~&h9!NI`5qA@4M=qd8Gx9<k@PzjziDRJ@k5;*kp+7nATidY*R>VBF<Ph'
    'oF&W80zvFq?u2wVK7q0CnjjY55v`YYvKwIGs=zinH;#-npr{~MS&l7pR~tWCWR;;z0R7Z9R&T`^r8ZLEPfY8r9Tw|IDc|je2B4qy8v|Av;DTy!FYLjy9p~25Y1pITm`~n=n}|&})VIoZ#RDTjV(226HE1v8Jv{Zk'
    '&g%Ps)Kn6=wMP7SfwrPwJJqsH+XA-wH*=Wv0e1+r%-kS509y@i)ez69t5XW3{tgB1wn^&-ov8OB0zTtt4OXqDHm<?c-(gxx`-SpFX;!x3C!W_{cf0Q&jkGNvY6q7lBf9en@jhz&M($10Nn=_`vJ^wEw)EMnlhYLJ'
    'Pufsi??C<X)d$$G$@8B*h!aUgXJWuEfuopwV};lLu3Yween=l#<Yg8`TO8R0DV*E9jRz$t{Iquc@KEIH8oS?fLaoxP3i;&PDMNIJTS+lTgZUXjKIlP_z6KR=(!1*&5@y%cEoK4fHz_uOTB2~Fe1`WH=G`Q0G~;-|'
    ';S;sRR^ghG)VkYDcl?~YTz)^BQRLv;W{8?MfM0XWO_#p&X;8Q~xvaw4ldrqBzkB-sT;vN&D{4+@aB))K!^;?OOAhNV#f{Ec`<p-aZ1nBW;en-|W+ALLDQ;n-DBNJ^fPo=f3O_pkq@#u&c%#M#yGQMra$+~#ChndD'
    '19XvszE!C0d<8+`l8Msyw-o3pC!c;zgAp&qNONax=J%-2q9lcwlCvP<JnoBSJJya33C;Mu9ovGlYT#M97uzL3@gQMA1B{AwRSnu#7BS1*#V}BUQR?p9^LrQH#LB<hp4yo2w@k8Q2-=>|ts;KvNi%D$T9?u;kc;hG'
    'pjKd?)o)yeC>{+_KsOOI;f)AAn5<bwz_ubyX!c$&E7~IK6V5sET<UjysqMaNCeiinSAz}O<Dk@Q38o}S)kDk&-CD#ISQ2K$G@x2BLesY6;Cr(xzuU=th#)lf4gIIhpfp|WU!V@N**0DCzc!t8{kZDd!%v$<`}j3_'
    '9qEKCJtB9@D0hIW#i;zzO-}oQZ8O+Me;73gVcxq4ipslEFvgXhF@-6o0BltB+Z23|YdFm;CitMH)45*K?ucr4mddaVS7zgFsdYVeeeAkz_VfnXSQgN82dKH0*@}J7GW5g=S2c`TaQvHaEH+ladmfXrH#+p}=W8<!'
    '#OirYQr5T0IS1+dFZ9MO<;@(WRJ;EK-ck)tPj*;a{H4^s4zNt{!|%ITQMaq`!1%scb$Y9b(cT_Z+)p?{u6Imsn`Ed>T}qWHaz`|lSp4OnX}Z*hEUBxG_1j7Pz1P>M`q;Ktqdf9r%Vhp{<iZqN*Bd%A?Z3*Zb4sK|'
    'KP0~9%)sA~pSz{Gby3tCvuVA5Y$`^K41A>AdHG=Djh^XFve`-3WYH{?6~|*c8UYX6HnM){LN2*2dtnsnE(c?^d~zB6<dQ$}q^8U=C-k2-dHSyqZ_F%Wm#3uj{#ieGM_gbRGn&&Rfv$bIq~Y*pQ7FJ-+UAxhbf{@t'
    'xe13U)ZB|x^{AU{eR-KLuTZx^r7XRJ&-rJn9CrDO9y=cNo!Jolxq64;`*uPVp%SUzS7tlj58MOQ(Pq7eZ`}##yELrx$bm<F@iJvUf4}3z$6H?tYio}(b~VZr-VA{fSeI|+BeSgT(@~#kiO#`20kCZ&tFvWX0Be=y'
    'bm5dDPU}-M@X-%rL;y1&w3zu)o%Gn59z5LY9`kf>!zwFf6i)l4Ub(J@5pai4&AM`8U})X)9681n`zc+c(Yx9CA?6PQK9%gyguHOGQdZvS&1R10dbC<E4XIW8u73P#=A40iS&;B)M}pB!Th~Blk5s|52^;ncx;LI+'
    '=Jw-{T-v?p$#at{J9IW5sPXAENdpN2-*JR&0L_lG5O)bXsQZLC$5%@t2k20QzBR3JnP1H2)v6o?q3r9KrM94TrG6e^G!_J6${^*&sldT{l_OI;zofLuVKBREt7*HC+wdlI9MpoSMMNXP1RIB`(v`=Ecxep+LWFkT'
    'C26)hGKwn|+D%a6alJ<q6!2z8D>4&3+m>87$N6~NN^y15<W|@h-x-_4FusP$szN7gB?;ma?QZ7ez}OyQ!du@!++KP&5%9<J6dT`k66hpM1HWjrju4ktvcV1pYi}gaWyhXP*9(qeS6OhhU=-G+hGxmj06chsGD%W_'
    '*ckyxa!mZ@a>zLk8^L*yT;n6dk+?+Ta6XWxqD$SL$paiF(l!0L-~LxvStzi|22qBl{w7TPq?9_k*6_9}dAm#3_{MkIQ4fiPzivhnkXd&~^z+44E8BtyFJHuMRY7yL^a^!tZVi-zu~CSB!2Al-GYN~Q7gz2DXGxGZ'
    'm{=kD$yxY`YZRgFEzmqcXwTOO${NM9SM`Ip@$aALv|?Bps?4AwF!tX{&cMb06@V$2zllR{Y0YG&!Sg*DA`>p)_uv_`DnVAJEi1p<3H<vDSXnH$$XAnTIh0~g%umkZL3k;HOGDjDLtK`Nb=A7eXXSZOtp@NJhGn3('
    'ZfZnayh%sJbXs*kIyoVndg3AmtBJYrCiCgIdvbj80X$X$PdClitGSsupNMtI>>5xu9oJ!lKgQ*f0zDwhBwR!ty}3^)!-d}wcy_wYnZEXcxa$v#)ytxovF+kCRY^D0pu`RRS_YSb_*9O7vsz8zH}M>>w>R7Sp=*^+'
    'K&3|FRaF9O<f`QK5xH4G(t<!W<`ykR&liTen|0z>{(>?f9C-nE?e~tY(vGk6ER)C%NxjUEF+fHY+cw5eq@OrERPcC#8-`aim+olAJ(GPue`y5!m-%u=j_c%0kFq|*caO^p7}h8^qZSxq_#YU}pr4rwfkj1&5o-yD'
    'LdJ|ju*U+ewa#=pH#2G;wcy0Gps<xm=>Bs(^S8VA%h#RK_%;!gI<v*=j-HUv8_4NeDNZ|4$DDF@KIfvq*wT`Vg5Jno&*eI9u7)%iW<{Z|T_j9UD^*b~N37yE)NwJ*M@1`r_6#1~PR%1ku5OvXAr~Pa(#8D-b?i2J'
    't>V<W889`0BB&+ZphTRuQ%#;pjS=Xl&5E!{bd5-JW;q2yl)9EYdGIN*Pqt%CeP}{C@l}E6^B&G<pXXVHv7hvhS%zw<n*?=kaLU&+M)$3}7-gP9JpU9J4}T5laV7EE_%$n!-R9NjsH=a(wr=mVqnfpbR&!{u#UAc8'
    'u%KXGE|j!J@(|OB>QZDJ?$umm`tf{-7P8VC?ber65&Y>OD{b$e7gy*~4z%JJ6L+-^TX=v3|4h^dkz<(o;k_^*y33Th5E`h5^|oBujN<o-Sxmtl?9cP`f!Iy4wcAiZC%J~dA=)3?zLlJA0^b^_9wimsZWdr(p)tad'
    'KAZ%8%0$HbIyQ`O;-X<;Y9gcl$sUD}&WLFp@8I2`XN|EE@_#>H8+Hak^pW@Ap97M?Y-E;?djSO<J1(C#Mze;YtlUODnw3;y{0k=<rZsZ__}w^vgiGMRYVdZDa<2ECEv}tT5N+041l48R-9LHg7GpuFV)>Ot89-+j'
    'vxiwe&g$7l=F1@6v!ovn_OlpfB@IiD5&`l)@{$TtwLO>Dr5{h&($_hmF}6v*^tJnaVy0L{=rM()EaNT<1Id$+H7uW7-fqH-dR<lG0gM2@PF#9~bKkNxgIVhazx^qe%<;A_wbb!8Yj^y^#+367-|<b{O)S=3Gmj(3'
    'NSXiPwhcE41PN62M4R;kgX<PNi|e_9%Ef+`209Keg?qM7>xko8>w?qcWjF3a5KEE0)f{;0@Tj)<#{E(u1Q91uo1{|Qe?)NE-A?dDt#UpHyDO3lBh31oHGawVMzP@`eIvHrl3SsLR`+jTV^9y4+hkFf?)vqydklOJ'
    't!I|$s#Q%&`=u90J1JKKLsimDYsrZk$09a$b|FNG;LRqqk9*Y=0_1mDSFmko*gZ+oxut&=9N(~}1B_tC96$5b#QmzrTghUxxn#3*Y9l?_)>|Qjdu?Hl^z&!&bsP2kl;F;oWm)TZC$c>@k2>hCM-gmV;+)FmpfX#8'
    '?%^@EKMvZ5LN5)!3xpBjF0_FM?M=<8gW-~s&lrJDwaI*v>q8@x<kb_}QoWb=>dGd=cNa&eXS=5^AI@JCgYf$r%Wh6(ftdJUsltf7*wX7mh|Pq2{rshZa73A_!;%V8U4ScMslf&+fO8K1(Hy!EWmi*Yx9KG6GEaRz'
    'lABbQ2-s!JaVrQS8s`Q3MkpOP5oC1Zmv$m5wq4RS{{J%f{y&an$DQEs{a2()SS8uRVpa8g(TLS-Y)?;1YN^%TnC=<N=I{s>$tva)i=>!jeVGyg9|Z61!VqkqZ($h5;WQQmo$MQEWnp3L41@)?1^*wTzvSZUd+{<~'
    'vZ{J!*B0#QV&=>DHD0`kcoCnFt8#bahqnVB-JK&k=r6OUnX3|9gUCQ>F*T*vK?+jd6&goLS6SOEK@NCfOmTit`<NXF)6*=^4kjahs6V>vFQNKCrOLJKaP(RC%T<jVqqKV|y3cPI<KiVIybf8@#0t<Y=$&4&yN>4Q'
    'phA6MA^kyQbw=w>X2ett9%;*}Ao`R^xIbSKR7W4fEY+UgoX*Z_YEo~mmZ<RLvqN-gS<-X#@@c2+36-b37tog8b0cYym4}R7p*hyrdT28FoopdCc3Kb*9wp(H2x(|6=Zx~GvS!gpO2%y>|L~FKlMVF)S-O(eP}&&5'
    '@7OjcRuMj}m&&O+t8Ah4f~60$XXi8EeS{~E*N7m+!fNr|cPdyay~T<)5IQ@chT*RH@!S_GS<AI5YgB}4k&73D0(kRn$s8K!=L!7Ze0WPO0b+|LChRTG<|1r%7WfY!9a_Lk@`^d@7?$aeG^*b-?dLB)p>(xWp7#p^'
    'gCJU^J|%96Pk;Eu=bt8%m%q~&%vRA^`L^yaZdD*k1xb`x4{?F=+Xyd+Ib+}gz4OIqzgSY%-oK{4ku^3HEuhrz|MKO}snDNK$C#}5&&e6|q*gnR2`qts5_w7&YQ!oZPM3?>{359xWXn449O&1mKkbe9dG7$zbKv|u'
    'e)+p(I77;Cm*6~4azu3NhEoN`9w;W(hLFnZ21Q9PiL@xD^6o7!=A)t?1FbM;ZFxNV;<HaO&avKgRN+1>Pz8VU0Oom>EEg|78^C{&4)Dw36}*U^4fKjx_MGQdJyzqpRO1-)6t9v2{Fs}HuXa3$Rr!6+hkWqG=YNsR'
    'CCxE8AQ``eE%i&d^y?v(pY>MMBqd`j>Ro}KL6|kbt%i?ok%TqHG?M&Y*Pv9qTeXA_TnV&o@n>H0ulnkX&;MUcJe|G#hiNj!<ky-p`iswhX={?s^Q7BwhWQNPda#S=DuhzfFpZD=w7<yuz{n9aC_@&yfgT@F-tGfj'
    'uPNo?{t4~X|D17JY)_-*N)bUqx{2t-9Bt8)E4IK?uKTyVolBlL{NmNP;RDJ$-g4D+Qt}mMeh78hz1J%3ymi%H7L1N{-LTd4xJ>Rzjde)eP`{ie$FRG8a)SG+o?0jOO(ZQcEtAIJ<!=)MxWy@515_@4gwd9N_>X^<'
    'nhhjnr)DbVKrh*#0DiqD{w$f(?ZoK`%FcXTJ!`S$0wqB*9l!ji{tgcUiRqV)AwYA_<j+<D?7-J2Zp^R@JLYK_ORc-!qCN-5iT@ymXVDE&GHdf;^WFZk|82}POyUYkDZxWWhw88d;?r3#qK}CH(J+@m&Vj;fE$utI'
    'jbu2_+IQY0|A{GPfL@5d1Bz&E=j-Dgo}NRKcuNk;Prgn^DjWHFwJ!8XB3LiYencuBIZMqz7!N(39zDBQj%U**Rz2xoI9C$>dCREAz9B@Pyb1-;lrADmdVdP?H*#(iI_QScb@R=x6I6lLADtBMN#H_>VnK5xpowFZ'
    'zpKQ*)URqTES?X<Ah3uTMLD9V+~0Tf+D=F9)<~Taq`%h_KD71N&CVR?06xL3@z+EUt1mwLbZSrcR@z8nrrhve^9DYizFFAIMQ2Go=G`;KXXvi?lhc>~4JyBkd&cb(zxXRYe0~*_3*sF1HO#@hrw8z^xfPs>*B=8)'
    's(*h0;`Gmwp_Px8Uwr<Lz>Gf`Bqz|fL2~;0pG<-AfA;gKjt2i8Q=k764B%h>%NL*je2|>u7t=8<!ZDurz|NtnpE6JUTUJ_|vZg)EM1w$k?NC9vynS0)j4HPs_*Yq)?AjWYZjM1gtA%?p%&S>cK`M@7nT`j<z)@6@'
    'Q4H{W#gTidf8QK)@c(eW66w8!W0rkIm(1=PN$kKJ!-4$I1CfLVIIT2?Fqb-u%`#k{+}-$u<;78{zj1iogjsYp4ZrUAQ@ia^Hu*r?_@U^IxW`Q@KRhv}VUi##1xfEat#G*lgb(MA9Uov+fbe-Kal@hbo%}c^f7iZ*'
    '0ONZ0WrUSb;J>cCVnqFAwxsQHxb*>Z6#pqZ$f877W0JLUhM@eR+r%IK^sj?AsFJPgTXc6Jb=W@fFb=mK6zdczlj__m?4rZ;l6tyM{_v;&S8_?qV*ND?@K{`m32|Ip@n|$_uyKk1sNU_kl4V%L*epvr1=bgAL*jr_'
    'T-;{CU>$SKcPyst-6Mzv<}E*@OHQ(q1+UQF)<e&$TJdtCJj9c1I^J0>Ru@RNpU(YIpS}C-<On(TzA_&i&QDm)`EzC3$dyU$K_>Uh6VT08$l0zlUf%T4l~8_AQ)w?DZ)yCy9u8Gph$t&K$cT+(kNj+X>qnD|`M9Az'
    '?sQkquF6O<UL*y5C07S}rILD#beMZsc(^)epUS2q2lN)5^e;a9dzi5)%;~Rju5tQ?FaL>!&l$||3jTcdZ2Reh_x6RS2Q<h9NUaPH5j)pqEQp|SX!h63<oM<9kZymo1Kax-C+=st9|@Ae-wl!pkwn;%?$2l0Ii6D}'
    'AThZhDuB-Mx|zI-HUIvvW-mVxk2s_z8kW%6(fH-B<~X9?;PCAB8GsD-`QK!=RXwq3Rg0ZTpV+xQ$8r2<)t`>{Wl%U@e*yo6u8c)S;r$GH&_DKha$1qbWL5nOI-R_dO&khg2Jv6A{L1jj>MMVja0aUh$_n(Un%H!{'
    '$sQQ&#7pus#xB)C`_ue(azxgP+EjgQnb>}E1XcZAmW)yMDAXH%2+A%{$$!SP!q&e=vF;axI7y1kL7lsG>y{`>M;~W%{d?Tc(bryoew5Fq)lT^G_iB|5twTNmZ6Ler<>P*ht_fNq-=S|NY;<X*3GeT``BqKyXf4Ny'
    'xGLk#!VH$PmM#u3m2EB(atJs}IZw&E00A2qMz<v;n`qy!y`9_!4s_=Y`3GkIAT64YxL1nQNp%LZCGW?hbN;JVFAY|cMuP&>#|s#BB}2Q`g7V=d93KvcKGWBOr$2ah|Gh`gpFZh5xc}h0n4sa^hrQ?Dy$|!@6(jw='
    '6!xMoIJgId4l?5h*yQK{+nlpgfW${ow1hlklsCO4DHIghOrCObT)sJ*>z$;(J29oy2V@>@8WmdQd8h{B!(hG(#F47J7b7krmmdG_)tg8{J)Dl(R0F*z#0V#-UT==31K@aY2uBvPPw{rMMRt@;$!@BR6$47BB>H-z'
    '{j8Ln?!D0pq$)SBWVYCs^V0!(k=HPx#w-)9JvkXI`#^zx#7ewCY>?b%sKb_Ffg|^C*N&p_rE#(_-PZ6Bu?Y~{AX~EPAz8P=yP#`?9Z*33^hg{M%P|?n{f10<MiKEoGJ(G@c$&C5X%i=EVnjn;tN9vsfEHI%cTd^G'
    'Qs~Z)vp{i6wB$gl?R2#~d^;`rAGy)%=|MrFXg6uw8{UQtug(8xBu88Or=~qDNq&6h4g2+V8O6yxXD&d2mcW$ylgSLBAk-ZW^yO?)C@ba-mTq9sc<Wmn4*|@upCLND7yGa&1S5=_;7Y!t$cc)m37164hfkoW91Eb3'
    'qS7B<oRgei8D4Rd(2B1*tOB-~(s>Xh5yo0@*W<HZs?w|z)E<krD1y<5wGOv@eAM{HIXFj8@!=tRB0+;rnW^wavT5sst-9>z$2slMN#SM*H?a(>yr%>hON#KFC2z!Nv+ArEvI#oG`e#6~L|^Ic9b7lp?rTYk<6I1H'
    'OaCG}OTj+0q1L9A!NNCK8e+=TAx)bX2iv06p+>8gN0{9OUAAH<JmrvS?;-m+!rPDWNqeJABLUNSyUgJ^G^g$PX@k4nHiu1{PaF*k(4+H~M6#5{LSp&D)d^~X(szcD>0w-oL|;m9KjOAv)Es7ag(c71wX}izr<GQo'
    '`($>Z<i37`%F&KM9vJ+BWI^=MJo;FKl;y%W*pkSK;xh}Ynj1V_pyy@r=!r?OM7RA@+AHpas|kKj5M+miIw|yt0q!w3+O5re>w#NKmRURjIOu?S0c!(kmmA{WW|xjLe2a6g@Cu{(GO0a%POUXe%Z=pwqYL@#;nQ!^'
    'UnNvzN6CfgT?yK~9Z-jFpHT9>(TwK1iM4<TdvXjbSK}W_9iwwNZhCN<<4<F48=>aHpgdBE7aiq+vV|t{2~=^K_@6gwL`3vrc``RM7!XE`IJ8Z4<%-Y;=K_G)$tftOy;n&v_&>KdSYp96A_Pmtv)dL=AjMC@#&YtR'
    '&j(qixd!69CCo&wopwh*&yEf3195qBQ3?|CU(y;xg$`e4HJxPB<61oc#PQx_<Y8w8vQ~1*CqOF@49nu=D@ce}inHs@a`8GCMA4H@BPNm1i|2y{!?9|dILN>{!!BsU|8w3L`2TtgvV|3P#6NJ?p`+0pe;2iUDmY%A'
    'Otj!AAN1#=>uLWFKX~-+!zT|O_P+NVE%P5vhqJ+ol*u2hMvIH*bV|+^_a~FuhiM0o^ljCEw&7R+y|DWstOU&ELl|T&o6c9u_C5Ls!~Gs2Y}aETozWDO`wvrk0b*yZ-oAJFA$wmh6Rs$4cbd)SE4AO<hq`X?C0<wA'
    'Zo_Zx4_$MPvmq2~ORt*w4EV`m``#f--rxAf*Pbu&ejC4PV!ioj**Vm|n%c;>xpzPQ@rNlkwVXYkosAX`pbT&y&=T;zBCkoM;p=)G$Fs)&uIcZqhX!N=PssMQ%|4dF0CNu5x09YrEj;Z~U{eEF>Gl9R=w>|nQSl2r'
    'i>NO%7;`o_ZtL1hKj``)Z@ca^dERSz`<{ez)SD;$IsUF)V)MOh*vj2>b1OJ+4XUXnA2qNA6wF4gYjqP#wzob^UyD`!+K1^Dje)A>HPQah`xnT#+LtVfwLYY5e&zMpKHUSt=h1TKa$q{ZF^^i`d;atZ9)Pg++E?v^'
    'mZ>*J=Zvu0!#T)Xe!f6yIa-d_rc0~Er2YPTkDH7+^Yq{&C<(u7)6rS-4hqg4YB47=&JHhXGKR)<HJLQ-)VqyKbTfjXfI>~dzR;x|WF#M^;o$piN1EK&!wEUQ!|e89zV`dg@nUq?h7Q4>!`a!Crs~7gOiz!eq&LU8'
    'FN_|IOfdINZW^i}3aZujada@Pvtu(lXoUC8wMi2@qohf$V(or@F&!|Zp%5uZ1PT37SZ=ds{j)yEHS|$a<PH}1ce{+cB2xw@s?KziiREkU_C|8~@sGcDA5)frIza!r&w0-3_2n3yK3G=#Fn!+7+|9<2?!9phOy@BQ'
    'yNy)v9HY(o4X}tBzs0I%he@Fg9=ui%>{ART!(Ce5Fzt35xua}XX$i;wwfx6FrdV(|y=t}y47EdaF<6a;uu<^uqXqek(yx^)k^41Tpu6>lcoD126zij27boV)_~p;1dbRH%`BjvbtGMIw7oY!~_@%gs^O5W20j+N`'
    'fv+@i%y8OnV6IO$;b^A9kdaZnSd0%TJE&{5a01H%JCrI!LVv*N8-b>sgY{3*-vlPVnSA%<&o7eY7oYt+OK|Vd+n?Yq*b3Ah`1Si=^_$X%^#B!{+K7~n%P-Xxs-r1y_t(N6=x9v^_zSoyx~ez?J<dLjt{JnX`p+Gq'
    '{DMX!5rEk_pzUNb{zl!rW=<K!zfE`rf1BkQ8&(MO!6&Ps7i<RzFzZbIWqjF4NL^@<6|Nph*gjNS!c^OeMTDWW89bJ<MC)2dePchc8|C&dQxr5Q=?s|w=&$VKQMyJnB(iRx>L1f3TzEB29^;`>Wf3@JF)53=jZ^_l'
    '@tD`5W}1Rk0zVE638;>9d(!FdbuEhz9XsLdr_7}im@B#@;PB>XRPLltIj+%PL}_nlP6!&U-8YBpoy)L8>*joVv{lx>)cQemx8v%K8mllv1D{ZiCehTS-}II<CfOSgUc7Wz^~l+2(V6Zdqkm<s|0?aDATiHtMa90X'
    't)4AldWB?&95$4#(wm(>gt7-dC}cJxglH(5?s+0h70*23z-np#Y=%!_K5}Xu7-GAZ*hd-d$+c3hJg8WH7{12}EInb?>e5opGm5H*nK{I}H!=$qpd#<GQ}w;O*~t-sUZFO>HR*qRv6bYDL3^v&Bx5gA7}lf{Nl@Hy'
    ')DfMaO=m0VPnPX1Q(BrVm%N9~%G-P852(=H9aDyWU_a&bKtk^o<21^Em?iYn(q(YUz^$amgCc%cPCek}?JcQ)TkD5zkMG<=uiC%mU=5XEj<Wug{iX)V1U--L?m$6P^J`f3{GJW`IPash_%HwEUqD=JL3J<xewr*_'
    'emWS7H<Gw3q2~$}MPHkr4qD>=QE+BqXW74K*>DncXHriTaiO)LSNmvuX=&N<7oYul!2e8}cX#HI&Oe1onDo)9YKEgnx4@GxKL5=i;kqT91sX&@!5PVQ*i#rlGm(fO{0!uf<rkm-GtATH|A%-0iNNyvzk2x@dS>-6'
    '{1cdD^>`uwq8$_##VPndk$<lBL8|DvA+RWX1rQtGL>1$xI5teKl9(nV^&?gK7L4J&6qaQpdE#cpjPUZ?cU*Hs^2Gt&?I)z)ZGAn=UcYs3gUk2bo&LS#fBbXeKXyI_wuT^*&-qi~$2j=A!|W7+yYsxgMF%f@krois'
    'q4F3mY=5!L4*TdQ8-#569jPOXvoC)`=x1UFdJ}XhA1r2*3G7o6E#V*ZgNA<=_~$+L4i&GyyF2g%pFX3lH=%$a_fUx4{pRRF{<(wI?P%2pv*CrAf<bKP&e@#21Ii;#v4;fEtNt4BWN4`ht)8b!6!glS85n0tc(Cw}'
    'sF*P26i^kR`xTU!{&e@Iy=;0+c)Kvp9_hdoIpt#X1|6^NVQ>#&ZFBpJ5>4*#Jixy^@w76pI4gL$%O!>*yvrwt#-ihqZ<-6`_wLH}3eFQVH9-3K#?6Kd#(fp7zC?@gx$PCs7<&y0Sq1Kn;w%r0O@ay3<MH!9XK(BL'
    '<!_g%bVSDyAc=E7sGoc=1qWa$5bvu#0i}U$*ML0&(Sgn#DL-2E!_Yxu$Q9?{^BIgB9T)kaK<8ulhp!Fx@l^SnEUXOAGG$51+F%0zYN~e)3p{;}n#nu%D><E)zyAsR7vtKTce_2So5=%Ol_QSmfK`GH{4|5Id^jiQ'
    '?dd%{Gb{^>eE|(NU-djN;gUB!2ZA^j^|zd@<^f{5o$MFxv|VBVC40sW{^b)o?f*LsI9PyY@?`w-w<if62pF@4mKyIGEfJyQv+A6wk+j)uoINLH?JzKAmMmv4e};m==YN|eLpaFIq`k_-$zX3hZ&xsl7bsWIf%gwE'
    'T5_bN{{p+5O%cP*bBY_B_!7}1eZJwjUt|eajNrTd48v~Ff3>((aE<M5RVGF2v=qT>C+}ykG)NO4u56;$KUd_u|7qiWf)2j_`${<gFPa7{6Q#2h`ATE@h#dgQsPLr+Vg_+tQN{8xk~&g}bUG4DLvjV9m`3v7wL9db'
    'zg%SJVUmonA4bX;+X)%z0Y4T2gM)myQFHoNN<VU5I26BP<e=+XrSO;P447_7t)_FK_hePRz@qEq38#5q6hh<CMuzb4{Ftsvix20E*<h6CvxU3}5zq`if44$N0g-K^phC5{Ak0uQDUvOf6);OquWLweUX&Vf*j{-l'
    'zjw(KW#H;5NkJ-uju|@Vz?4WNQ>CMJmP{i{dTogjy7zt3ftO>}a6vO(pr^%XnEc^C{GZ9$Y;inV(4Vef`w3p3q2~MhU-d=GLgt3y5-4UYeFP!up(vIep~RmPkC>D2hF@KfWUxdr;RxlmDe{*;TcK+Ht9g>oDwNB|'
    'q^SxMx=WLo#O}-6TU=)AUMSjZ{6uIJzTwkoV__yjD9gIc=E5igZ{71lvR)ZQkV$r@D0wZ@mLo8yMsk}SO#D7*6>&c)fkx^A5L4<F(&8}^QeZY>Wv6U4VxDx|5WT0C@{Ce$0c>4<BCB@W+r8A^{8sYDl1Ajhj^!Px'
    '2MyoXj@g<p(hi>rF6#_~OyeWPw2)Jy+a;Z;zFBWcN-h$lrtplxsH5!SaO*N0@*35f%fd+cVK_qf%poFdT}u%nNpjpjfuWKJ%?1<cjxN4J#`?K-Ov0ReCY*Dt#M}n9o=PTH`bL>~)fYw|y@}v?dE63R^q5&bMa<Mv'
    'O=*Yew1?T!W1mstzXRAh%hgf=4Kfgq7Vej3ZdR-V5ikN>8^JhkF5QN@Gg)U$C`KAX<|GRax)XNF<;$+%mKK8WG*pGXr0mp!e070$jA(5{$f8eKYv5SZ+2LV6TDq{f&9<d$Y3BS%C)F?%(#<tak#uvSh|2FA(Uv)e'
    'DCiqVs9NGH9{LaP1R54VmwpLy1RO#6434WC%B!$nvi$f)3ehXs8m#Nc2qjDUvu2?j2^`~wW(Re&iJ@4@D-+IsFyTz##iHnoc`_nI2m_Nur4_ri!TzuHYGa3W2Js`@N-7+^_&JW54=;91Jkq-mdaFYSjyJQ6-p#0d'
    'MN*`s@tu3``A#JF4<`HvDt8unf#MyHG9WXuzFckpuC{Cgv62pq+gt1eE<jv^qZ8g@&Oj1`U9wU(5KRq~)4wI0h(N>>5ez^0O%O_(IjhbCW-|vbKba-dm%mz4XB^t6vKwo1(~AXFY{{tcb97dOj$;e|cjo;h&<f;G'
    'DJXiF+#?XF18Vp!K12ih4Xr)8wqJpPNA4ji3_KOfD<2h?SGZd4!T2vvuSz=RY|Wl43S`bH!P7dHyJqKFjwVe`TgChT(|9;RJqX3TgzwU4*7~N~D??lb0ml|8a5P<=&>gU9i5SHT9b-<xa9vQ}sUL%uTl+@sp@D26'
    'JT-JxsLJ>f{QfT~WISVZKD-EX)<aMqKADT5lcS5jySED@Eg&-R+gVL^FxA@W7oYtj!Yr_7R6pC7yRP%miIktd{Mm&PCmK<4uO}xj|C^yt@>Qg~ENJj27bNui9#k9S6<-%WWP69tL0?T7vDv_To&jof<JSYNH4sCs'
    'MV_=!7s<xA7DBX8#b&7>Fgmz2NR~?D#sIvScj=9&+L(M)Eo)E`AO5b}5F(N`OJfTPci1PVYt(|meb!epI4MH$j*hm=11v#VT>omNdd!mHC~nNC)-FulerXP}!3zGr=MgPaYs{z?+&vgHqbd8JUDxS`g^XrVobr}y'
    '<E58%#%NykC%r?UhzWxq_z#Nj>J;wn<t9Z5w%^3w*CiyJ6nHJ}K&TmV^~+dQt+KzU)bX!R3!yG*X@RMvQa|LfXwE4XE)<dK_}#TgOZ?POgTt>PN?bwD#I?0rWT>9qM60u^9Al#&V(Llx%v#yjH`YjTSg)!aB6Fv_'
    'f578uKHS8(L!z8x0^dX!M>dq2%mK4OVI(hFI41ppHTdH5U-nDF*j;wlO^UG(EI?zc>^{UHZtV2b-rBnNKoBNf{dad};$2kTD{lr5tHY)kG>Zjs`_tFUiG=96kQ4b(OR8aZp%!Xj-^744Qg@rqBZhs-bCg~=i{Bv^'
    'pO>E{cacu}z!j3R|8Y27J6f`$nMy^r@KVqAvZf%TZ{a=br+so;%29Z6Q_(^r>j}LON1x^R<<DTXsF<6imZ%p}Muc*{)}dlNed@Txn4+PZQ!{x%=uaraFUc$EAoOW*b)uDY$hcPh!C<xM4=&`ok|WQe692(!xdd9l'
    'NR9Ru|JX{3kvI2Vkf#&*V6)#f)4O=$@MrfDw=anJC)6SShCztLgv)JVaKjdMbTnVKw<tIxY#815+ydX7ClN<&WAWXj^?$kGHu-zuisxhCo0Br+4cnaXq@&<bc4}m1Ls^>9RkI14x0ZJPw+}x&Yi@U6!%fsnIdd+T'
    'ZzPmbx(!9&fAR>SVCocNGq0u1URv+8zIlfYf)xFD%Y|5;BW%rlw5UOGhGA$z>0QYj^Wf<df~b9u{<;7D5AmNT?>|n_0(^)NG?&?$c|b<TC{i!x1m4@Nce%{tXD>FCG1#Uu)|3*7;}5$&^t&)y<oeEj@(5jY(eOUM'
    '4d>lzyyqJ(OOVX2bNH4>2*N|LHHQA*Ci=bR5iaVkth!1&>>c-kMCfbtfs>E6O`-UxC~Cc^Qx<BzB3}D2eWuq$>JE+xAXg_YT)UNAa_tsz&*d^<-+S(JA-Fx5p~y7b;y3iG4n4y?^Z@3_Db0=eKyRFz|0SW>yqPIQ'
    'IQ3T(#<<46k|^j>c(Y5+WM{C-0o?2tQ95DUV0O<Cp4lm+4Mffu&xVrH+Z!<{^GFHXSF?_Y7}sK)eJHEoQ1FP-zTEN&HaUOmhQykP`&T#Vrs<>Wv|UbH73qRznCE(-8~rl{2jO|*dse>PH8xZanX0%(w4E5BL~_1I'
    ')4%f|jvOeQyB_FyGXUZ7xSwJAX>Nt+eQ)FNSJM;39~w%%sctLnr{o4wAex@L`m@u-sS+r0$TLFB=D3OsacfZC7!8?$pY0@GDbIKIp(PzU_5uyrXMJ>gz^FtZ%hb+4C8(lK!7*d!vZol4cEX&00&jDgD=O!AKI2oD'
    'zv046(zhz6jaOZnUeCc=6sAwQ#*+x=XCWZTHcr#YunCG@ucWC8mCShTE42<Wfx1GJs0Mn{JLqFP52m>hPR~7)>7i6GF%oN(KH!0It{XGNSY{~h3-`0|Qe1*SpHn5$IF5uP3_VJUz9eDUqNB;Wc=V%8Mu8-`va%op'
    'JI6kv-kVasFOCRA_q}6s_4?5Y<WfSMQ?NFrZ!G*B>)dGL5L_w|j!WhoRhs7m?VH0dKK}*X)TH@wM!WIgqNkAk%vAgb&c=@d`%kBl)O%3&g}7>h!l11qgM~K_T*$&F7y6nfXf^Ghg7TvMd*I?AXehp1nd)0=^bKQD'
    'Js6xe6i~G3rbhzTl10AcI>aEMZW2%<IwjmngJnsX3X3cYSBSn9*TfR8s&E^0tfAYjuMKAd)E!8Yzjv49`AHwIqfRr>3K1=eaAQHzXm72Shud%KQ<M_0)aVRF_AQ2-okFRzY`7e^hoDppM%(lQEe{#R$$*M58MXJ&'
    'rw!3V@9~V{T@8*=OgLiFpv-aN&_#vs;wg_(ReLL6UQ9;$cr=2BIaGn3GzU3446xhK2@vhQyRhCq7=o~un>fg=dp{sc`2_!4ihFRIiP5T{up;F##Y=}d<bhg1XSdM;{@uIon4l!a`^eQ8Zb5X#{MD3wzwvWJFyekI'
    't4ewzJ2x{fyhhLPZUMEqe}Nx<Lh(0@!p>C3x9D@VmY-m{h^>3io_r_tT23(dZeIXwy=U)oyxN(M>AH*O0k0SwdxKttw|1yL{}MrGe>S7>;N3+q5E}}F8t~(0@`HYUOukkQB_F4R0d*T*FGsSbdfk|IWCgshlH!RG'
    'EA;XKk-oi#(8(C|7ejVl58P-ddrWev0fI-)m4=`25gI<T&NS*f<GkF%KFo`aSx|hMgO=ccvptR*kaG=AYPwvAJgifH;H&^C1CQYr_stz4@|eB&3@SUb7T0U*hOGc-759yWaTQ3+n(k$eqIZv_hN3qw0@xW;=&oAK'
    'HF>mihd!cF_xmMQxZbJ`ag+iazF4tm(TLW!VTjT3_)9WQL0LV6!W)7u;2dWP=QXjK<pQ_v3GASP1ARGg)i+b+b(NOFPnjEJ!gC^Mdj9)FL$)#9V7Egblx@Gh6Y678>d02{j*zM{SAW6&G5f?<VLkI=PwNp1v}vy0'
    'CnoJXYPD%ajIZG@LS6P6#B{*d6bY2Q_u_{yket#;>@%5yV*QFlRU3K=FAS&}v2sO{GI_xWDNAG>mO+R`><9;c`Ll~==tk#lgz<h9ZZOS@&~z*c3BSkJe4g)!PcX?V;(f#bJUw@67ZNR5lMLnhL@-5^K2@VPRxpk|'
    'XYBmj4d+_i6Ke~lVe)Xq4-f>5%r>ajv1FQy%GDnl*)SKR;wdJq6=;9*pI-5?tVqP7SrwaY6%(YAB8X$;#rIHdkp%Y!Ha@J?r92`lX%P^VXH$wwLhm5+26Q9H!F-)siiwdn5B4xy<gS+##Nhu5*U$tqm_Ij-oa(m8'
    'WUuJ?^?F=k_c2P5Y8d!#*X19^f=_1Nlbuy+?QMo49qdt0PtIyU_Shlp=|wgN5;xXJLPae0REB|KFxs}r*iNM6E|}?EtynK7*>*om_tKj8P9Uzu3`9%gLQ#-iHStD(Fh-0+4eT`Tia7NN{%g-UF^DF8J6!XwH-#Zw'
    'i?mixwiC+S3)`yq=m+<|^RV~9{r4W-fAZpaJ^3-L3<&aBzMw7M!?QA7*Ei+ldVi|9q;yr7+sNpIF!)H+krvaRDlqt0K+P`G;Ppw^Cnuz&46~KmZf#wu%c*Ui!s98KOC__BdnFj9b6%%>+>iA}+%pIpJHnP^vBh-d'
    'bLXOE!enX;66ON;Uf(KVUU$eg${q{sV(%%Hs}WbUIhn?!-q(^n0&S)?k!7oQrh;H1I?xFS$_}?=K0IWz8I~kXE?xpFqf|A)3`zH#7n8HlL4~$p-5EJAOZ~ND^i!}#3n-0a10k30Ka{x)XCprS3B!U?11g}MTmtQI'
    '-N_}6SuaP#z@+yh4H6`JxI&uom@t%JscU-7vh3D^&CK#7Jr*Q&ViXZ6Q`3Q<Y|x(|i0hnML+r1HPtD*USE)&Dd2M0#dDg_C(NA9Jj)?Qs(vyrQKET!$Grhm@IU1FCWh1`Mjez_tG_V@ptBmDhj(5J6&TJlJBt<%M'
    'Kf9$&OZ@d!mt<RxNXVm?*nrrxbo~Vg$<ife;nbGbTLJ1Z%m*eNb0@*2ta_XPptJqyMXknm8oK?s##}bsbYui!z_ryamm#H?vHj6P9legI8(*7ffifabX#|k~3Ue{~>SKv%Caz|}Odn0->PXaoFF?-=Z#84k*?u`Z'
    'P`DK=ot}xej_!`4qk;~f1XUxUwMp9U>VPX@z<zAsuAs5<Mp6Mds|MC4vuq<0DOD-2Sm5+nYRuY4VlyvYmOguSk3Fx@3Uz7(|67a|G4MCON@^S0Ds3Y^t+WpX0n?pjW<UW-O_Q?P<#_E|Q)vqk##Ym8oGn^pp?43B'
    'jMqxzIQ!b?%>ysegmtKne(33<bPo~Fv2fM$mvv*{;gV9|aO<+*X+r1wdb`Z*2mx-}FuY40T9CF!3%@>GmyzLSXP|JRD2V4*6HXF=*{`{?Qu|hSIb<%Plg~9FM4ptoCO+I@`tFJWWCb2?2lafH>I0eEFgqm6u0K5B'
    '1dwj6uY%3vW!|~0nQdlhg}0or1UNoiy@aG!U<0KL7%G1#Y&Rv8@z><-;*vFzfakU=_%L^;txMomxb6Nj{s8TePkYyN_riLsi`M{KXWR5&dDQmKIQF)(UGU(DOo{s-`*`GvljnG}u-_kT|EEpyOZLdNVUQ(F35So2'
    'q+Mn$K&P;E+Obo?F--EBI_v|D4edQyKZgxKp0B{ym)XfkGWEc$sKV@SQ2p0faK^q22$)Dyjad^$LwLWgNW;QXq)oI?=qS$`8*qf0#XpG!&v?CGiQ84h={?V`#x7(Lk1ei6`*H<Z=CusEB^k1LxSCApI<%H{?r;A|'
    'fBWOz?Qiwi=}yG^%33P*>;B<L`l3l=kr%NNlGZEhL)%%qV9PU-h8O?1W0m!UX6p?eLG}tLE~m+=*9#l4>{MZ|p<~`S6|3OMGiJ%=K&#_)^AcRK38JQN(}tCLiq>r8Jl>MLxf{e)<R0e`?+JE9HNd2fxM0Dlu!PDt'
    'R_E=u)&tKq(unlcVh3-YvAEG5J?OOl>S-)pg9NtwTN|ebwdqcvJ$GFi&fB*!Zs9b*JK+389Pfh*c+3<5jju83i*I%v2S9ac+6ZCT>COcOK7ej<{Vc~BITy78XNfOW1r2CwL(Ur2@E-Xl-~pPC#XHcg#D~)~upn`V'
    '&<<T#{CKa@f+s*l4;Z%fiY^5A(XY}2yz)pJ*YWfz)g7@*3m-_?FvdJVa&$|YyuemX8Q$nL@}5}woJWLu!?Zuo$Dog7=qzNdaQ01?j&x8dX#k!nnlYxDrYwBx)<85N^w8qJzN{12+UIKxvbRLjyi~ZpV?rS|WcR+5'
    'FFcAi`uE4#9J$S{cb-1|e(&k~FP^>sqWA5m@BQHZ3)FHjK>^ImHZK15>0Zjk9z4g4F<T!_TPTTK4rY_tVw=($j9QcIXuRCt-p;1St*`HW^UlHRZ^19Rq;Gxw%{y<s@y$K>joF=BUmw0XJRH3aKQW+Sx@>)Y_nU8y'
    '-ul-1!|ClykZQL<*hIU~0qHx?hx0Yw&0U`K7a;JrcJ~kZgJZNePoe6=*AL%1d~=`r(mL%gYSe}L{^1NN-g|3)zO&bS>&+y;$bpu&S6L(PPxI{@g(GMWul_JUFS`@9_V(V~ouBVZL;clqwm%0E2|IJ=jrsYST|pXh'
    '6WM5@XUC=K&Dj_zcAxv*+MAz)^ghXk33m&}UYD2K3lM=<d28<uH0PS|ZNs3K%h^e5?`>%H!|8ahF}~9nzkZ2zY>!7c*R8$g>+Ald9XL;CeGM|er6gT$9Zp8)`$zpb*N(ro&ld0#|5I%2Eo!VuixyN&PwHz-9FPr0'
    'mokRQ=y2J3!;SLm2XDPK*xhx*p|J%^b$IyJ5NDlWmo8<>cX$7tP6{+Rf!#c6=^tx2V3)&7t?^q>A+oV;%G27y_V*o;@DQSNeu~j@ur1}g<!p|_WE8YZG)^B*W@ohAYuMbEw2IsLygwMV;5W_Jz82%z>rXGv#-KU~'
    'nY~S9_AR_>Th5Z_NL<N-?|nac{vd6nKggh=e0I1@{<#0$QI<B8_?j>ed3mAUF0Ip=V|E(k&3Q>pW>aWBTke~wN%r2r3BNw^y%nwx4I8?}ja?2_iyR1mZUsjR=|Tr@&bIX;t-S<6uWL6zK!>s^x!&fyKOIdvoM@@*'
    'Mm<Hz6sE$sh`4u0i)^@$|Jg>5d{9ylZg{dfndU9vhNJ$nhD)}6m@OM88EobA+TPnh<c+<<MZHc_xd%VjrtZst9Pa?<_@*aU+#m5(#t!m%Rk5vzuD019jFyfpVZ~LHIuiEd@$Ag-AJ`1*?BwV&e6%k+o*2@m1zvmD'
    '^2q)rVnyaUY9XoMs1EwWqmdg_HYNT-skA*;f`;C9gr&6eMqzJ{-Z*??G+bkamxVpQJN(wV$>IUg?)Djv=B?db=t5;KtE4#PHzx<o;eO{0A_<tqIupiq+Dvc_%#v*Sgf~N@x`4(%#y9TU#Pb6J*wh+%BDpEyl@daN'
    'ni8DRe}4kx-mm#eVWQ{f^-DVIEVajvGgkshB;>z%K0pIoEDd~5gn{ZzWs<Lj6%3P_jCo1>9{*cT8kbVeC!aFz6<UDG(foOTxm;w>%m^{S4d+T5lF;p--EMO?Bn8`swwQ&PR_5({b4?vLZ)U@<X<aVM4=Xk<shiDx'
    '*N(STOBq<53sFj<?6Aha>hfu1#^`9^8QU-d!aYoui;lw&>c&#ynT&W}!@o^-oyw$k+_6|@J2(0*1L?#rO}%`wI%12s1P8+Jt3ofS_?3~Du#+O8mstP5mC#EWD~Dd<535koZ$DiW?Z)OmXRd2qEA^0ALSmU>)afY5'
    'Xn93$n25dqDaR{OLqRf_!`TE+s^n7ZQ{%pjk?b3Oj7CijN6O}waz-W@TIB?}LN|<!w4sdjxy>>|5BTpw#9}ee42^_)9^@+B#+<R!(HWk#ZB*impKGupHg%0yLmEG6>=ez=AkRj2GQ?{Dwig7MR}TNx!!s@eb(T9`'
    'UtcR819uB-T1*Wio;PA$AaBvbv?5;P68~PSvnSgR*adrYg>?BgLJI-9Ve@x!XHi-QFKY%N<qACK{oy6r$oV&FxFxsEkC1)+;XnRaP(LFj3hH$kbAM}V>zQOXqg}Hzn1S4uP4SvUp@jC;^8)B<25Sn04&vEGo~)+A'
    '28Kny(VJuVJFx!Blpr1;E+mGV;lvff0p7Lfh8UVe30pW$iXu_^Y|7gA*=Hz)42X(jjDPvtC5EJY_UV$NWtRMD1Mw05d9LXMP5cZ-lwiqzGx_~r_8IBl$F(p99b&-wzDF?DQ0pL|z$nVvHM1mDPok^7GH+6dsEJWs'
    '5)cF<3GgCAGz+twVOpi{Q`Q{H3q&&{KHavJ;Y_Y(O6i1X4mt16a={abPq>Z)j6@jHH(4|JK!VW_qzwCdLU|MzfbX~v@OS#fXTRfWn^ozZBqE%Ejsm2|j!zikoappPY{{;Twf+N~0+;E<60dtmV|2f8M@bH%Uzx{`'
    'SG*#gGOx%NQ;cOQkA__(Fbg?uYdPd8BC`>OsEl9!7Qsbi9rKw>+>?d@DudZ#!On;WBWQZmb2)%KLsNI(^aH=eOi)(1p{sCYDihj-TZH*@F|Ca{qD$}((WnoFBK3>x%ngNTrbNZdg_dxZUy1;+1tqngIQ?SAahK=4'
    'oZjZzDQ?0LoB%+R7#v~%E}5pLk?p&}J@(l^A}|U$yW}g_SwVP^z?}=6&d)g<??Ui!gheujj$wPMd7-P`baiqtT14rOb2;&QxzY&d)Ib9|H@YkE0gYQD;s43SyY?s~`I1|kPU#^Wi4HhBjO<esFlI-WwlRv9W41)t'
    '!v(>|d-$j$f)E@ELt3q##c`ajc!hiu8HM9A(=nM11XI5XdaZcAGH1UQCr@e?vo+uryMOr8zuv1ej(6C9P76@%oy#ai1H!1(l?PJ_uva;KibH*tIC4Jt>0sly)~mwUswd!q!+z3ib`l;(P3@b6OMGBa${QvC&zvjs'
    '(5xl}{>*93=myF=jQRk8I0$1t@3Y@8@*DVDoZV@oQB|E(kq9cYMoh8@bscV9(zQA0bW)Fla+h;^LV(g7*45@h%KhirNPx$21{W{?Yg&h|7>qk_Us9)Cd?+>_V}z?oIm-AcS3!K}zn}0v+m+~F0r}>XSI5Yleok@)'
    'CL26*T+Js3boA<z`n?ilH99g;fJ8}`6~u>s6UuCgUU5c+rM@+>?=|;b18D~qvYkGA@*T`tFguwetXgj}It2}F?@Ma0A7^vAa>jw%>*@XiPI;I^s>bOyfAZ*AGJ(f=axlt4o1v%z{B%Eo-sSMsC}E%0h6_NXo5{(A'
    '>$uTwD%wMr>SqDwlCi2Z*wzyC&%bq%L-u{qrsAvJ6d;6SJz3?Ti4Z1CFx*n2-JrQwr;wnUkWyfw45i>Zy+dyPNjCpBlB}Vb^vsY7Y>u`o`{-Hk-G|?PeE-G6cYO(=C0K^ZWOfGiz4eB47>#;^We=~ef!CbOb9I2|'
    '=i1v=0bF!BHUSxKqMD1N$?Tw(-cBP4v~`v8ixWb8qUiOjjbpKe++Znt(4S(R(rs7^5Cf<E$&R#2jkOXE2~X=)ePpDr7@)43ZCFV=-_Kt>eedDBt`1((<^*lDwK^L<krA+{I@f<k{KwhoV4kj7ERQW>sUI>twu05z'
    'NeZ5k-RLmA^b0~q$<C!racowb{k%7y<v@S+wPii*JnP3>BBbjOX`F-(`a{1DnoA%$sGt@kMTa6+V#49(eem9cg#Aj%!Fjl@wDbkc#2zyCMG%tgSY~BJ@@=%fk#(NVA!Psq&^MN&9t=tz4Ch(<&YQap2_7CT+TsH&'
    'AM=fv6uu&C`5{LzI$59MyS%wc<CqsCw8G&~CfOiclFJ_lFdvLyAxFvS3?|-jX}U}$%>lVU^JE5_2>oniRos}>3?Ad!Vae;V#yMWQAqr0EbheIE8ktC5L}~aXd_?YCXt>F1L4BOSo~*=0`*6K3Zjz?7$LM3%&-E?u'
    'P03A(=Oq1*G^ccOr-WoycJ$e3)&Djb?@1kkwZh70aQG|~os4S+x^@%cjQbp^C+rt5l|nx`j$p-75^;thnH(62KA>{DV`Ar^Kr1nUn?b}h*0XUO@8D@U?xXi;fi4SihXXiV`q|M`*P3rsq`QpC+Ve4};Wu<;1&it9'
    'sC)XV7SyX^T()pA^7O-^V(SMu-fZC_Oc8y$FinLJ;bL?!^T7UMii$OZ69ARtyyQmy)Tl}fuy)avb=`PJP9h4%AtFYylcZdTtK}VV%RA>Eje1qbmU9%BKH<>KRxuxM+9z=ZNP43#tzyM#X?r}6E#a9=Rt}5Lj0YYv'
    'Yq))zeiyu3b!xR8(2m%-6ex?zcT%pVYn})gE&=%5y3%1D*UY>3;>=)-8VjdH68x^-(4m>-wn0N}3+ezXyp-g}25ZrV7$E}KYB{rUS%bqPaf+8V<MH9Qf+Yy{lI<@N4jy3$1qpwXNV|i<QqYWaXf>sHDFzbad9_qF'
    '*$NXe`sTds=O3y_Od{Ylo4GCR6vai?9P=|+5-Mj}6ERl8P0-`DhYBPgEM>v%M${^#=*d>-$^^~&MVm;*&Q3QBK=o1Y1wqEZh5B`l!iL&NdXFBe{}aSQx_CaT05J@rC_Kie{?FY_wF4|feVjHzJkh#AF0}__Fcmt-'
    'LAC~&M>8Pi0^-iB{gzx#*o=-YK&G^9YY0*rLw1x04AEgWU5NP@zAKGRn3V&y<eAV|iD+4zrdBv<6(xOAhMt>l;9zhjx7@OH;1`>f;15Qj_xAjQ8Ry{Ts0NyP&PJejk-bAqnw5&4n2vdXQj<2Zj;5M0lvZ>b2)zCg'
    'Fje@mckljiD#m!~lX}rd12!hJqv}%+^cdk42;GYPfMe?@u(87LDs63Jzu~*wbsQ^gP*v53pq}aNRMd&I9vu+n41hsdu!8d7ghjgX9nDv_!$H<>J{lYVM|XY|gW1qvYEOuYm<r*aR|Hi!4$n2#Z`KXxgaCa|*ZCV&'
    ';JgmBg6};?H-(2EJbL%xlLrs+C+%H-s=Gu%<dENYH)&}sXv5t$NKj@HBE}eSn5$u6e6~{`lIxV9B05s(5!zWu4km=^YojLdIksXH1>0jm&b~oaA#@an>8+*=7gLz17PYl5=bTo+WL^-j=!9^}{7lcja$J*Z+vhfF'
    '+jH#7*vivINlFwYLxo6EG(y3}t14kXV_FZXB4s_KEPocPSLKm;srGacUS9Z%!j3<jqeruxyeg3d@(4lyFO2X~fQ9ym<qq{ZC?3c<p`^1m8Mr}&>Rz84I`HmG`D<N1qGEAqS4W}Q3(W|x;sp?ZsrHR=?fm)$A;yKV'
    'M$AJcm@v+MW-UwQNY-JN<2Cm%gftU;CZ8{g(+-^n&_F6gvMsAOn5{4o8Asgfk9@gUVJfx$q^F)Sx#djaV#S4bLq_J3S2*YZHo$~D4Gl4Y?X6V?j4P$VsDM>t!`0|efdFVi(5hk>#}r6sHt0`+VK-tUx)ny@+O8Bd'
    'finJEp?tV$9pu6F&k_dk>UQwot_LKDD<oF{Bl-5rn#rvatVle)LG?|H!!D9VBOgIEY7STENY1cE(a`c3VZGK)_6h?fd#?_Yc(^V#-vhGP&OHcWo(!o6^-7&JHS7{DB>i2l(IjbQr8y(3Fgh&b{|wgEEYB!XKs898'
    '0TVZ&p-9eg;oN#~dj|Ry{qSBt!EiVP7!jS2Gq$dwi~1B6QKT=wPx_~Ls(1j^W^-o)a(5blUFqOMf`KZ%`Dbg()jYqX7;F6qd)L+B6VQt^`G^1Tmk9$ws|OXf3^mnbZ4*}8T?&mYO!!Uu@>)e-xF30cg<0`R-4JB='
    '!!1HSztnRIexR>P5vyFlt+5dXunrvw7zBr~%C~M^4f5=~-q?|Iz<I1c|BZkbqbm}6Ui8&>&|d0eVzxP@awTUaNms1-x*`zY$|jIR8HA*inlRE+4voY~G(g)i4;Swc7Pb(B3&sFJB=0F3c!ML(dGImjrXza4Cxh$^'
    '9(wCgoQ%k@VtXbcTouf<HxBV0jG-^P)LEPC5e5TTZW9LMnPeR!+~4Gfqm#sHa_Uuv>MJ|&guZ_`Pmc5}(xfeB6Zi!sC2<+m0~Ch;Y+w3mK``zh$Z_kEGpWKfqr|&g*PGMz);;^3ORbhx*}@#d6OHYhM1)~a4E9%='
    '#F)mCDBTHAKl!=Ld7tb{dUundo9BBoSxSEmJ(g`=X#q7I3Dhq@T7tl*5?hq&JTl>zwKU;)BX$|3saEZEO-SA0MB@V<_?rCSis1x6cF0{AH`<{;r)m9GB}6#+tA7w=(qI>L<I9U=W(Wx(S+@dLQ!F6UMjo2x70*$<'
    '=rQC{hB_BU6m|p;pyrIHq1(T;%WL|c0Lv=yF)xdwj6>ulfSzInX<8pE9MBU9FEWqoRmLrL-lPM38n}4I-Kp8jZIiF385ChSF2s!!KGC+Q)7D||irmrlh!tKZvLt7hLDocTNEjTorwYZ$=@T_^@-pEWrCJ;$8p71j'
    'vIqbs9mzXk&E%dD!C((O&8(^^l{Z{xH-ge=q2^#R%SW|xfp|Cpf%9@h5%k4~mKBJnVX6RR4HWq3sGPUf6;Ur{v)u|U2n-nKc2E>~Fb$+E|6L{6kTFiU(jmBXT4vi6)P!97*_Z@Olw8y^H1x>E^gTgOq;9q7wFR2P'
    'y$@o9PwxogWcvOz`|hLnG;7NQGkjK|=3DkhIUuRCnps2wbnpN~q0?I!C^Is9xE4|>(P2IhM@kN|_JDt2V@&>OD(Oc#RgB5~w2v7{F|BS`pAWF%Ocn6w&*%0#$_m53gD;yRYa3_wQx`(=)2Vyz@;gb~&ljKnKhd5@'
    ')|-1TvX}p`!o;|rYE7pmG6kK1moGn^Y7or=e&-;!E*qvnAB6;e|AjpJbps{Z-^e-6!AMILh6e1jDo*<9Wex3&$tAj(Jbw8*bk61;?T}>P)@ni-YS~5BnN(QV5}CMsM&*c{M*v@Q2{G%b(b=C=Qm99u&wWl@ZWz<N'
    ';QV^apmH8$A}4LA-esoK;@7R;Y?RF{Sj_qn+Af4sT(243Sj?IWb5WBAb>(ZN(bU{u6;;N2S2K^fa~mWbVzutB&hlFlfjG>{vZX?fA%iR3t-v)d2)bx~pP(0KkkPSt^(<)$33!u#)gyw1L44&0X}2mFPemH0*ELRq'
    'zh`msrs--LK2k`#V?L{L%rBt=rpINmJ*lZyRFK~53^c5Y-gUGIJ`duNsg)^R4l9!PV5AGNjH0~FC<hp^!msZhEQ&>ZEm$r*uQ#S~;j7qzZd@B1GRZy`ajz&Rx)1)>ioS}UO_F4#eeq_(Cic3)LDuJN2VBAlM)HJ-'
    'pTnW4(5Zuu*VWq~bBgRO8>6-R#&8`MJf%A~tg9!V-Bg%$-EPQiY>;F()MEN<CUiqxk{ok2M?C);*4MUOk^1;~P`Rw?JUbsD6r8MBTT*z)&j{wVgrp*02qJQaAR-;@o3!&T^eB+MejT#yW@@*Bc+Y_NvzoxEg9j&x'
    'ljsPc3!~yAhgVURI|C4RPNaGeCZ0XpMzN?AwCiK5P^O`@kkT_wIjNC_s9=*+L<z_!Hm8|xN-h;i_omY%<_XKJ__}B-lHCe9x7Uj18L3qb#?vA9m0F3+BrPW_!mlY!Y^{QX!o~U@|181F6#ksnugK<gc&z~MOYQo4'
    'D`A|4txFk0lYYZ<28l8+_<<&u<79v9Fi=|_ZVHILHkWoBrs9f2-&ZP`${u3x%OQF$<t%QOAFhb<c?HPMyTny%x|G7<Az;mr6h4T8_YP>R7r2U_nhdGW-u*V_uEY)HE2}s^BxEE!LJ;N@pKd%x>Oqe<kk$sHsl;!0'
    '55D;c6Q@fa>NBTpI^2tU$`Qf5`cMv2s1-y(wfq&OHkmFbsC1kpxXGffrp%*p=E6EP*{IQ#MR~#16M1Pl?w^1T*~b|49F?_Z@_ldmTm~61nN>M>hnN_qlg`HKaO#CR=Adr%S-owRQ%$NE5H}@otX+}T&6JlFu#LV2'
    '<kN1CV1+#O_%MfQ=~{+8;1x550AtAVA3Ye&<@+X2N>+~ig}zyTpE0qte_9pW%B3>PwTQbiF<9>+^ahO;-f>Oh&=Z#QkFiiul5Sl~)Q<UcdYv-#uIAzeqi)Ms+0f4yCNyreYI*z6y3%f?POx>Us^}ipa^V8SH?%lK'
    'Exx7NKfEmK;4QDdoA-8TB?aCXvl(eWrgXQg+P=P~#~V*BK{6r?T6y{WD=bI?(1y~5_kQuR#aWh*dc%zK(HDp>^1e$hu*v{i6?`!OL6*O+a)cd>CUimQoQLTCcR2)wZQgg;PZ{j7mSO;#Kg)v4yu@CgUtht*YZ^F5'
    'YH8#a>Q$B2^y&AFU$yHXzmwGGDJh*W#Crtc0fUY?0MxCONv+aqymuYy3EV>Ur3+4u%|qeckAkbO<sxzLdX(T7crD_;41HSQIIrFcZp}I8FMM@LtGmVuE*UI4MmB_=HKb-POOOgOW-UW7BYGRQ$qTh;8Vp>h&1q2S'
    'NKK-kb>@Icu2qE)s4r*P*`pY>u*uOwLoP6ib20(3o;|tR-fM!s<;O&k*hS-|#6|<-Fjq)~p!nvu%;j!RGYr!BV0e>S(ZwZB@G_EMY_|e;3)slUucF$I!i#m>TiTE4IPpt&%Z{=VTj2%32MxoVn^7lC6hM;zn(Xr;'
    'C@E?gCjq~7GrCqxuek#&u>#!#$sR9Fquh%B{pmn*&0ALvwA`PMPDG>)724!Ex_59jbbA8Qc>~M*dZ{_#3H)Kt2xHZ%BvlrZEYV(XQ((uG)<(h0gn_?EOGf=T?`MjQBch@~DG9;A0tzZ;lRu2y^<M_x$m9q96vcag'
    'oK@+tlafPM0i1K3Jd&*aIvwzp1Mf7fWkQz<*P7t(&~*-IPZxCk19CJ1q5NV%E=I>=y6?fco4)+jw3++>FIn7l%?=Oo;Pp2oG$PEoHCn`+c2wS$Eh~=RJ{Lf}b;Z@c$-NG$<0ms3tWSO}Q$tj#H)D>Shgk-;?)}iw'
    '?*Mg<G1UW<_RGEy{0YN;_3z;}id4gaXZL;}xeBN+q5i^F70#GVt_pZq&|TECC*Mh?UwrnP6VZ?)Vb(V$%l$bl!M}V$MEmlS`8ST1`#bx|iyywA`oH)5>66F2t?Xcfx-gSQMMe0ShDE=c3D2j$jbr6hE+&rx$}py)'
    '3~^zi7f`^<d+J>7!;3Lk)ry2#f^Qh$RC5049+<;aw1Fs!dk$&jF<zP>{bTPZ0@H@HfUrFw;#d?u$0CkVfdFyLc1)db5bmZh5pnW;;CMUth*bJ}kBQ0he%>ii&|QKI_V@0L;JYPWQ~3y?2euN5!5LF5czf%Em!IO;'
    'f6<a_)V({ZB#S(T5SiR^v9H%g#CySGfWLwn1vUtes_q@AbX4-AX@uABgcn*7z`YQhawB-28<rxwIYCfi+#+yo*IUPTJ!#RlnKBsV@yp-EDeaI*&9RVD&>I{U_NVfkOO|>>(pv9Cj&et!acSCj-?m>A7`MWSXGZ1?'
    'Tg<kw0$WKC<%=}@=YugGu%DtQ!{21_j{*<|kfi0XLnk3N-%;BPc{hIf{~;IGYT~=BS}OJ~!+~{$g<71Kvh^f{K#C$p=ra_S6}X`lL2W37uQQ^@ut(*mv$I;=Syx!`&O5tZ>+Bsdy^y3_Vb(0kr)s(o?<@Gcvmvi`'
    '7_#G107axu6|BShG4T!#WRF8w*%yd+oV?k!>VA)N0+dTQV~(mKm+5deWkZnJ3oc11&5PR>qPk~CY3u4B_+b9yg{yOzOL$RHx4}?ZJ=|KsSwSQ)oA<ocz+2&EDiO7aVgQCcsGE)8WL>b~B54q)fS;`L!UvI123@SO'
    '6I@Pap|TfPqZm`o?fJ-P{vF04JIsPsQVnHblUhMRD-gb#6-)?0{WQw07vI;7?M&<m#cv94>%2YKcvkgI>3!Xp*_5b=wofTm&Oegy8kRH~60~;R(8Tr4Om0MW!+D05Obj1M9oM644x$GZ+dOw9b1R+cz<+1cA=?U9'
    '(_yWM^^<HX5Ktp|y>4MZC7&imz%6Sb0#30DBo|^`?nTSGAkn>g6y>L!C6f_OCZj5E0DifeGN72r#D+{igaT?f?jNAJF^pFf9?wG=*_iYKwwA6cVz+UGy+?m4WWaTK+`_ZP!_3iMHqc7NRBORog!eF~Bo@Yj;83Ng'
    'jIeH?@&X8gxycTv4A`9_I$6Pe!oM;P43fg-1@7asQ(G3*a|4Q7n_hiy-1_%}K5WpQ?-eeF#SfFvULg<{|0!uXU?Kk%0K^KsMi!L$QoV#b-?_Yf8wG~L{$Q!gkGcy?(lHE|Pf`4<fpJ5Vp+%nxs&>9w^o3TLFhE&k'
    'uBbLaf%Nl*kT%>}IBCJJVD|i<Jcq<H-5ly4rFjg0UMJ?Z>m*v#oXHCXeX*LDIPiFMk&EOfLEV*LU3HQCDYrO!|HTuzPz40&12s^i4l*4#GXcmp2hcH04<eJU>3ON@HNll(S^8<?T(xCjb<E%{C8uJOqerO+O~RDG'
    'TlMC6bUw^rmX@`8r}cJMva+!IhISF*{itGol*@D}m2Jk`ix2_V{%?*5+_E&>Y~#pj$ovRLPAnn@4PancWo9Y?&#0&aFDs|NyjJc?>Igrexknt48vKV@)7flsY?zeHzVICylKg$rKkxDHJ-*$}+bLp@WG7h>yYQVA'
    'GKoIO5HN9=F2j$Fl6CG2MB!pYr#L~eBAELGw4{V!PM*E{Z39}!dX!DiBi%2e63RuK`2Zyf#3wAl!y<s?liBjycnB!K-S@pOyD_msTObXZIu+&)C!6+=?ch!k#E@e4YK0;Q8B}+D<KtTGaHjJDv$|7Ia_dDH_>Kb9'
    'O$8R7Cdp-Z!hBAnlQK8KTeyyafmh5g8knCjqfR?|HkiaR@ve_0xWrNl)GJ`(sgDFMQ1jvmv(ykgRK{|dHYwz;nXW0%3ud3;5z@~OKXkvZz#2^-O;7un2avwiz0GV3=GCd3CAS=zcr(BLi3J4GD*0_Z9!4)fYXxc6'
    'QMlWPafg_o#NA}?sxkY*ijd6F&8L(!2kSQDEd}k2hMktHp&M+%Qci$tZ9I%i&0l^p7~6|Av6;s<A)bZEf&zyPymsWL!32~C&xNiSxMe)bE)C})3H^j=gqW+lM&S?J0RtnEfccUI>NQ2^a=gI8ZjweUeI%W+Y*&Zq'
    'STY?K0>VI@_IR4P=BREY%h?kBRPxbiszK9y8tew>mdZRlxyG|X+;nGSyI0}^zq>;0Tgu9$JC*i&h)K%SJE9bc%oR(r5Q%fXU+U`=aydo_EfO2f1t0v#@N$hiE=G)(*Xp6|vTQ*CisfSwIfh<vdycF)rPQVeij>OS'
    '6}lfxU}f}9{sgWl+j)r~zWInhu8H9YbAN`_)~&P5c$O1obR?o=JJ~Cw9Vzq)nS(FNS2fs!03V{Xp$i9ww#t7H1FG$cC<65o(bl^q-$^Qc#n~fxvVrox4!s5uTCbhlKw92=t6NG$;Y9QjemcFCyegOJe2xY<g#QBu'
    'c(h`J?r=CEz`yZoda>$H)Ah9=)8C&SPiJQoj-9C8mXYKNxGvMfSwK6w5jt2p3=mP2G7UDuZ^h8YR@;)N4?E{ihZ#l!C|7lIXPu2NH0#az@KA!-<!!;xVkXVkh`Z7K+7w*^is_0vURX*0F$p^d7fbSGu}d0@RMDA`'
    '(K7-MJHY5G56pv1g4>UKw8ywyc1VtuD&3MhHKFTS4pQcMS9USp1JmclfWniJY1_F8vNV)GW2;lw?b46{2m$x<_H9ouV(?ev^Joh7paFt)Ko2g<Pr`6^3tKJ{w%e(!MHaa58BtR^39M`d5Z%}(*9AK-An+*WkmJj>'
    '0y=bnhpX!uLgE&nqNSs%y^Bv-EJlu_qd*35eB7abBM*)d3VkCiNSLkZnE2tM2D3RbtHfMO@F%`UgrOSiiv@C}Zqo@%JtU#yNCF4_{9b!YiL}2nEDI3iO(>gb=h9_YI2sWua9Xbzl5yY1pH;=UpCpNd0jBuBnv$nq'
    'T?FG=40m;rZBo=I#vL_sHPD@acj@fWbz}(^JnUPrVV3>;IKOVEF5^hy>fjDcrd4E6@T|TLx~$91M0s_8%_Tw5bL1wWP7Qn~MVljGA@YeExoNdmm%zj1{+MoySme|ZtQ$rrKv7$4^=22>3x^_MdeaFj%+_BJ1k#D4'
    'G5q0Qk<f!)1mxGs^An~fhEF$X4c7QUE-9#qO)*s<r=Uun-T%(R=R1!d{ov6HnPzd3b4<5j`6%AgaB-YzcY&;WhM2fos(++3oSx1uj)~sX-7CG-5Dg_d+g7Ps2gyLVE0qSd6SSK>e|G=Hqx+9BJInp|AHR4`VqqT^'
    '_ae^_$@9F2d4%vf{jFWP70;#={kQ1n=svJVcj-fn1OvV83}|bHpwX}IxuVe46tCGaqdmTn7e|ZPYVKd>@`@iH_2)gbsS~39Uh{4FW(a$oqX%}IyYvvcXCHo3Th$Z1mrZduzOhS0jKLHLBt4K=0jTB9n?&uWbj`CI'
    '!-0>!OZ4vTHyh5ypzvx>+IH<=_d(9k$HAx&5)U^T#+YJ5lm12S5(V&M-`^ADHt?1hnHwR3!MppUy*u*UMvc;oUHRXdt)cGbSnj-QGm6*CtsPSy!K&+DB+0sr48Wq~1)iftVrkNCz(zG)<#EM=&;Su_FI6)i_mKsr'
    'Vmo*lFXg=*&%ub7skKO_5gLutXBBZq>8eZw<@Jv)RXwm{;T=>>$4@Bfb=3;drh#S>`|=vrE_dj=S?9@PNOogW!Zu#NliY3lCL*)h#R6~ae(Nm_SlkQ02&NkQcjwJFH(G8AozoVRJo|XG<7%oKBGkSW77-hJJ(!_L'
    'k$;!mL1Aj6g{<(EPop**l5%se{dzcSyWtd+*#vgYzb|tbn$*4POcV2rdE&m!OI;}}hO)%450+VWuQC~G<MP3##_c>5c`UZG+q`p~miW`eeHt`#!Z~3MF+IH5Eal^(>k_d+vti%LW%2#BD3v~?{OA3-?^w8T%dEP3'
    'A4kgKL|vyV1RF~>pfot^wQgqp>a5Z`&GHq7<vAXupbxhji9HT{`WF_+Fauu-gtr8hVbWVHoz}Irwe<kqa`AaG9<3G`LB@e}e6gR5PC<#vCD7J2Nl6wZuJ<rooPg#*VcRZcJc}cP(>wO(6a?xE%(yn4Gs{O)?z2aa'
    '_5C9p2U~Mdl&m=G>fTYHFg$O43WkXWCy;MHM(gCHH(4&M-6?jwX@2IRHj%g{JgIQ(E(P8vx8)Z{+b8!BAH94=bT_s7TsWr~Nk->`(P)?_k&}kKgOtIxHHdgN#X;B7$AuEj<#=^+Fa^3q2>94p&`9FXL1v{DX}&kg'
    '`=Cg`L^L1YfAZi@o<0|k)<qvJ7t?vOpZ6E=qo%d+`*9;dfIKXQMyk2G8A0`I!GbTLx$I<h(gaqz8jNZkAerWO`YWKu{`8&6YEi?l8wr|8cQIFXc68FmoPe}b;d|JJdvB357j_1P2r5Ei1e#YtvI72DJ8L9kw9$b;'
    'RcGVy8{vyLcI$PqnCr=bCLrjI{(PbQQo003VuK3YfA{{g7mq%8*n9Ea_Z~j~Ztpwy-+%u6(fudA2mD?hzw_wH{rCR3_vpz7JmJMtjh+?`-ks!*dIsls0a2^D*}K;+m#-(^NNTS8_h82CRIOmH-WCqCnvgxmFF*}@'
    'dNNzgVT7}zi-+emY?24?gQxF3`)=>)vxiR__yzEmyvCCYt1?b@c9Pd&QupfgmRqUwmp7?!kJkn3JmWfp3T5L>kIBRfv+)fOTDEy4^a?e1x)CTf3?JEcA3T2g{Gn+WNaRk?6uY<xFi4L1NBc{3*#PlxG0O;;8f$}j'
    'pVB?i11B8*QMMdE$I><K4IzX~DKZ<5-1gh`Vt68<LgS$&9K*<&-HOZd#pomUIGywtWFnL2gUP5r<u(0yw3y|+Np?KKvHPp8jEp9rBadjd5xTh%YZ;A27PRENkpRITz+O3^M4qEQEy5gXU&{1!vhco%3f=W28p!0`'
    'D=stmD!9fC){&5#uLG0CXS837uwiyW+%<H$(*>htb+c$m9piWS#~F5Dk0j+9jcO;sBDY=FzV$W<lsKK*V-T3@SB$ZEZJrt7xOn|&mNv^VbCD9JVll>d`3giUj31rM`b&HT$24%&S_cOyiw30Mbt~9NUhmdDp=j-0'
    'PDSY`kuEb{Z*p1$5TA$^&TFqTMX`4xHGLbGKO)aHY<Xe8z?$#qCO;l6riAHIgU`6|ULy%A22rqHug9eTLddiN0Qb07Sx*B;p+_3@)^U+xHX+z2oPN(RxO*jrA#R{lLoWaJgmv_;!!o4p3ZCI*FSp~J?>XnyLCxM#'
    'gc1|SI3tsBQi;)S%|<j#i7XWxn1<$-C|6fx97nJTk6;%bA@=OpG7U35NO-L`4Es14(RJ@Qo0If_^VvDNXdPWZ^+&KTkIq|1=lJ&-<)6vihZiu_=W?!{jX^!DnX2Gu1ZG%S5uKLBt5>w&o#$6=<#TBEg04rP@z-e3'
    '7xag*fS$wLUBLgJ!)bGI2g~Bk#(Cbv2}KgbpBLdzd<!&Nr&;wS+@lbch50KVPo0lw={};{S|HPD_M_EE*hbY1bo^X#lAn((6F?gC_Z+H2wsnS(Db~*K0HK}l)nQ-NF7WS*ppVc!28GtVu*{g;vD*2lfmiD{*wVYg'
    'u0G-*elrIa?_u5_h#d;1VwfEs)}$+?GQGj6<(K(#Sktd@gu+0GlL8-gub2@LIIb&QzVQ{vnh-Nnz6<;A&&`t6s#`qz`WA^NUCw5`!_k@jqF!w4+%^!Jzpkh~n^M~FK}IoF!Ix(9!XcayzG}pFVEb~8X)Mm@Sf?*w'
    '9WU_FICum!b&pEbP}<|?WDr0KyGMoXoBHMvkT=O_h`#se%Zt6+2inQ!Kxdd`JkJmimg_>>4+7$qn8Lhdl4-jQ;UziudUKEZaBdS-!z{XQ!T=Lbn<eiZ|LR_H=ZeK!EJjC&`)7nP`QBpUlK?}kj5jlJE859{pVJvh'
    'K5x7U(|<-?fsfpiF|JR9tnVdn?(S~xC5dn}7dZD6O19`_lgTPy#^Acy1i=T_=8An+tfhs0W6p6;0y*9el=_|I3|V$jx!*yx7A7ZF_)es-r+I%+`aG`(&^p~Mo})oqb#Q*z3C|3S1rghizv1hj9Arl;j3FwTg#<W<'
    'd5#2E&XzC%nG!Y{vMHq<RwPD-lhtII<v@=FH$|Fe&uOZIfoI9^Tuo@b(><=MkJq**a!GQT;wrkE3_QGq``}%ezv7X<B-GE{v6RV&!|H_I8H`GP;HBkiI)L+`56y7o9FP08q>~QRwxv_#OkDfuE3~4$M;ri=rxv3h'
    'tzeA9w7d3tQ$+uv+h?nHbXpO4$jVpMtKT_~Qd5-jYIs_sP7?W|hNn8J9}y~ykr$1M>%->o+nW+ujE2y<jduT`;WP@Hd0kRLN$sM=eS$u=xR>1~V(SzLh>z={W>zl})LxN(kc*Znrz0%LN$SXq_LJpoH5lWkGz#EJ'
    'QyI`|Djr3iV$4dTq*F6_><6u+wms<Z!go8nFx~#wUiXFyjs4VC1Y%L$#@qM~)n7>qDCr;mN1W>2#u;!jdwpfvkpv!XM{WpdQ1ECY&#g{Kx7~WBx8j{{7{(Y}<@z5KwRlX|vP5jhc55KrA{C%kYQp5@F*(}u*1{)s'
    '`{i=ha{=F-($<XaN9Uc<a?{S&(|y=d98^Cyh=Se*-73=vl{<l*hiN}2e1ZlWitJ>5``DeiBFH<kM9DStU^YRi-+|YAgnac2<Dt@>1AEb*kMj1Llq$|$D%>9oM)M_JT!5O4PU@43BV@|xJew%EKtfrd8xmMUS0M+e'
    'pyxZ2{>5yy+{SB&gz9g4T&eSs^YvD*v`fRU*$J;v_7sXFT-}@uklW~Rg_MfbxYv~Ph;q_vm#MN+V_fvluF7fdz18%I(P=QW3oGcK*>tuX@Wo<MBXC{~lkf63n*25_$)m%4sXYP~*FWuN6T~qIz)husLi$;{(sIcf'
    'txs*wIhVT7f==NT!>vcYH2?`|xf(kOfd^}?<EH2gLy+CCaa!*N-M|grE#-+s`I1Id_RZT!88Rw2g+M7M{pDc1fkvWqGMWUDVS+|EI_eKDIDNVY>1Z&+uHS%uU!j*Vo2i@BQf_ySqGC5_TAlRzz2iS@PY9?JWOu~{'
    'O`ps>8zCVTFPkaDxlatR&-v&?bLs=@ImnTfC8r<^r&kY-(@lvR=K>h)6oEViI9={!8Jrby{s`&<dq?3ib_+Y|jZ6|&wVLWsdQ*v*S@fqlhNDqscor^7HJpGbi5y57G$jwFu(|$XnV)6LaV?$BPDq)B6Y#X3YuaoI'
    'mvg)zNXk*a2@OTL#Ku+oj-oCk+#_9^*#SYj4U?nE#eAGNhIqvq(IMLq<a0_g>UTWf8JbPs2#kE_#8k>)0!x6Q-3BGU;rx<d?p?kG9v~j2`iNp2_>d#0cuIZPLkK%6hkulc9008n{0sjjGso}r^!1BVC{$P{KKWqE'
    '*kH?pmf@6gtr!MJ-|!5v!Gp{zW&(&n{V<ucny<o)0$&RYiz$GLYc@zKv{wa-U+I1=GlzyejMAKtjv)_Ce_*v>{sOv)h81Xa8-2n3d05FUZ#D=ZI<Ek_XaSX2g1KykRG5OG3tW^@iNx1Xi3wy*(jvq2j1-KE^5ea3'
    '*Xgyy2)LGxtFFuPspB)Dj>K-TJ?+kZg167GCD?8N)X}3m7PKr`{m@X1JZtc7D|!?g)dgm&j#u4I$gIeu00QyxtylX#{B|R=lY?c%(8<U?H~_kMRB6kkYLNWk$i|&)w5c|oz(zfABf0AbVuQO-IEIadG=;^i8dM8}'
    '<vPpvq_liIDkrBBPokKZ|4Ob!*Yg9)4g>kd9J5{n<I_P*fW%`WFE)Ey4M(`^Mik309;;&Q$-NvQ{x_yp)lD2)5V;@ZJLMdLtjXe?)s?<lI-BxuY(h&<>QUwAy1JU6523?GTYB0l=)H2Q2xaoIb1cF)Y}QboZhQ=C'
    'HT3bZ&J%JKiUFA=Uj$c!gzN&#;C+VMY}BeO%_kd-*<+zOycWE(@5sh8hyZBRrP%pk%Tq|PTm-ShQm4b$!RKVi-hsi_^@jL@EZOjATo1V=R2p(hc0+poj=AePz0*9*CX;7S$`6DigwHzxlkXN!S7IqVgK%zhm%9y{'
    '#6GCq&tiw_mLmoGRr+`}<<vsHM?aR93Wlpj5?t@7j$O4tUi0?xj4)A3WpPte=|<EQE1G-^+WN5}UT4MWb}sf>7f{XlUhDi0UFDYTRy{H)2IYQ?;bq5h-wC<pB-t6ZxkRF2akiltP6K<>-5O#EylJMHE_Vw>bNn30'
    '%2aOSlKU9T7pu|D_RVWOxCr-ISn<iVD-8Cbr#lA~=l7XeC1cMDuK&zxhX!XsSP$<i!ebi)o=_y)V3;ne(7wa*$|}!drv{!F6!?v&H={QiZe@_=`6_0!afSM9Xn}qGA|B`Qk!V3!cdxHF6)|k(OE*C5gYbD+PuWm^'
    '#HBu7zZ9HY^t7dN-Fd|_dcpyrRR@HMSg{JpKiwm9RmAdKt1wYDoHWhRsErLqsD>qTG<er5&DVF{*mzTT`?h-sLXqz~UVykC@J3<*8HTLT^;S|jq||01ES8UtOyeT8PKLwq5t7AON&{9baZ_`qa5QNXp`cpk#R7{4'
    '=qWk&wb?AyRp>{KAL7lc{@pO*y*$d*<nQaN`)cMvxz0$rqk21#sBlr>k=Q#|061;eg^cGtG}p{l%U+Pz(%0<S30C6boPFPD?j7}G=q>bcMn-C?h<C^}wg|(}VCbSBx67U(A-C9X1p*CqR`Kq+JSfUJB$qj<Ofu*c'
    'E;}^x(d1|<)1ZG_FWujn{bpaFXIc$89C0h^KyNtf!5pF*3*0x$$A(#|?gh(|GZ2>WF-vGfTAi+HuVrZ=Ab2Y`fC7a2fh!o6ghQFchr0d5T+BJKN)<%5niGEWiJAyq;4cODZ#a-x=4VV<Qg?%Ev%>XZut|b~oYZB|'
    '*|^Iox6m$m&Bm?=8CJSXoLW89PVB9FJu0qjL5`tv{|gONokEp0i#q{B8kZ~m+LdQ>WzFHNx|=;ib0K4@DDMex1Ebx@rhN^8Y)JP(<|h(ltp0Uio48CJ1~cchXB~E1_493Z7cia(Y`7T4pu?TckWIt5_t-6~yQ)<S'
    'haRb=yJD1V-1@Xs{f<V08(tQZek1qH6?!;!(4j!fU*0jJ=sj`l>JgCVV-BJ%-tlg@BaUBW*VpW5p{t53z$)bIFwpdl6RszjP5JiL8B(@gpb~0o`+{dAbT4Lcy&RxpH2tm%KofI{T@Csi*v|nQV(4mT2-C~wDGVJk'
    'g8GBO3L3Z|aJ?5$gzu2$n8gbqLaCzYv+$}&>=qiH3B4A-6$*xHrd^njvi4cOb&EkP#S}v^*JA2=fHOZ@+<N}v{yUEzKYH=UVN||5$rL`EU>pRCA$9QQqd;U4)33hj4?rW&jVNNn(1(J)GZ?vGb9s%1M1z=*`NiPn'
    '+Z=8#?>V_KEEfgCn-`I4T?{*vpB@#$=RGDfJkwZqk7&LjnmvqI7f>dQTQ5!M4I<ZNvV9i{q3d&rSueFg5b|MPBGK!ysV&cFVY&)46$vPxM<dDSuMk4+hJCOag1k@)BA4dkOC9M!MZxO^!1^yMV`1k-nqkjkp1j&*'
    'bclJ;@ur7t)tnyr4fajxPY2`KB5%L-hRmrv@K=8nHHbH+KGI>A!%h&K4J06}ABm0~de|GE{x++$oE;Ok8<y=*Z7r_zH7?pUR!Br^<W%<9<t83IHTu5btbMp_66=<iD3$E-(W+H|V!g}vamA_{z7Td8*P?LWCiCC|'
    'rl`CAvwF#MTt09YD)PIK#n)Y@$kG+Re4V9LG^q4DduVSuz<&sMyTJ+qq7us|4ee%UuSM|F;-JL5Pg55VdXHnKM(ZwJUC10zfN>=QQqn`1=K`N*YgA&|N^K{s^Z>8)p3+@@^8wsD8-qTG-iV_fo-Cn|Q+6vFox^$A'
    '>mMyfBS}yyvYNRE9SyUkOEl_B<SRJtvgJw-Sb3aHbG)tzo`D=QDm?PM9L>K)SIO?$hhpmnmdn&6v;`M$SRH;e&M<FET>5rmGD;Q}6>TYncdmiDU9jeAj?Zd^No)6U`y4cUUzrcLi9VrrmzCB%JqWOCYg$lVc~?rq'
    ')VrlI!a1UN;YC5VEUXrNQH!<ngfABtErWz92Mr~=gulIBk=JlmnG4b>)Rqnx4<~Va!l3i{GP$pBA0r<oUj5aNs?-G@ni8<4(L6_`=_0s(abE_j-l^UTT>{LLdR-JxVKt>HI#w80G=9|JjB>^b?AZb99^O^f!R6a*'
    '&)5~edCBQDUy;YTXfnQCR%OsE!D1~G(91VyeaLixEa{x$W#SV<(KS4AbTr^ueXbxH7j@Oa=9DuE$%c5b95n+9cO^*Z@QGJVZC_;OCNp%arWba$?<E`?dyO@sAm2AVL(v%_z&GhC`F6R6EK%Bx&-B<fw*xkHvmwTI'
    'hvNhURSy58;LBmA_PZLoMsedtUB_TnJtcE;&*UQPwWBNBD`xz5W1@0qAVa*oMVC$UjAP1NXN+rNnRmvAPi8F+17X%D#3gM)bHUlr-o*$*Jr(emI}h5s!Ur44N64cIiy;V(2*@vxT$v8zrh0P-sV~yGbkd(i`sPC3'
    'ga=w7=7*>ZtM4MB++NHh#O42xqphpaa7GsIMsR71pLr*q(B$rqm)YSV`q#EFg7#2HCI_n?Hm=M^Smq)qV~wB`9)!sm**OrNT!bDvo(Xe=0@cgHg-us2E0}>yNYOZTe87KS6!@u(v}eqWW@j770-cdtW$G*l?V{W$'
    '%A`s868FH?!$vq($!R+9j+W++1pMqO-{^?rV-P;DAX6sh)^RR)c-_(k+0T1WYI8USvAjmUsh2cf!~}H7#sxa?HhL;9M~ueGb0C%;0)H5I2239id(ec^p?Fv?1qy0G*rFLbXi_WU@PWY1ZD*`3*4Fh6Yk%s&R1tu('
    'zM`SA_~c3>)ND|v6w1l-zxa+Of%>$Q1l5Qco=4=S#_Sm7rc!(5cwlzu#EuON4+xFu@#luG&0PQ!&QSb_iONCi!unxTD(QzE4(^x>%0Q%&Y^uv@>mf^7eAOd$7$YB?n|dLg=$Bq*-p1IjRlcnj6sloA@vc?IaZ2oU'
    '=xB|GmBJP{8R2qe`3c7n<~EmnLydK^54Z%1J>J?v&nQA&?werd*sZX4*gwf87nX2M-D{%-^F|6cmnlkHtXykbKnc7?uNVdNR3XJRAz<5dP_2C3!W!UZm^1OUFU3uviebcgfp`7E9?F$q-NeHXsy)x^nwrbdGzpB6'
    '&9CiCJi%JYSJyhTo=!T4Q+~BLpjLVmj>)Cypswnn%iUF2<7j2Y*<2G^f*o9%Q@|i$YBUtO3@?zjp$w8N!hE5#CN9u`#U@8$m+5pw2`sJ8Ut^R1KVFZ+ep;yNFNXAt_A*}0!hS*j*tJMC5`mhOM<)<$SUYd5(@`>C'
    '%nn9qQU?Jag@VLgPy~#b$+EcdKH^kOd}`ZW=W;ud*jm_blwTm+9<5SfJd}yEoGSn{2xbH`{m2S-+__9Qh8=v?EhQYlFDlqO_J&!!)NHt_1Bvs3aFmZnk`qLox;d>vjr;93cte^sl0(^m!9@c5QMX=1V&~Opf;b23'
    '*axW!9o$J?v4F~c{{<w_v8<?#**~>E?L{Lo5@j1(cYL>Pa;x(Vq-qsTeM%!+j9d=n7;eZ!Td!Nf3he7$4M=V1*-Cfp%mr%L!3JIHpwN&jlsrv<@t`264e1){rrjcq({7pOxSr=Dl{a&|MUknnYvGR@7tbi<*S_=S'
    'TRxugJ$kr3f&Dm1&c?IJXj}0C9r-m1iE6y3rR$OlV{}VlFi|cS2I4$7k&CA9?($xX+5pG!9r4bX_^nP04hh?s!d<_-go~Zzd`VIV?@PwhV7hGuyZNg)or4p4@hT^@p9DCy*p0)Djc@q1`{H$L7!Qk^!3M@}+|Lm7'
    'e{?=qm^O}fVeuuo%?S_VNB{K)QV5W6a#`CTjMKTR-5^Mg*=(ZkoP$V}OejZnDBg+*q2Y}NWhu6p?^bS>V0OpZ5j#)1?)qMAR5PSzIT|Okv#=I^>={vbsi776<I9JpvOtF9WG`87Zx+yQvP#`jK$Z&1nY(q2grFb^'
    '7Jz}mpss>&RFNqHI6*tcFL1>j-&56B{JJV~PMlFB9aPAdoP|inisp~_QY+4Is0z1x;#JU9I}ITSM`H_t7!o(_U#UOnbOFPsoTbCtNL{;{gdB9Q73wZ^Q^o}oW|(&STwh`{@nvxftjBdx3ZUbz9uYQHOswKVK;`ms'
    'IY;pwSz|IV`|Y6zCV!sX4H*<~Cbw15IL-~blLWOxk~`YuxfOi9liV@8G|Z?&CyS1m#1JEU=wEQ$+RfCaeT0{*%W*apeW>RMs_p&)*~Pj&*lQkh6h~whBXqP)&}~YiL@a8OWN<$G#8OfQ7y!)9M>7hDoE@?>bHpYH'
    '@hll*Fe{2Wgruwz0#T$03i}Dw?1s@ocIY`9pLWn);kNZ??gtyerLv=|F)Hg=@g|Jp;@E7QsF6O>fw2z4<5^~-ypSvP5q{Zn{B8T5vMKmWGtb&Yev+E7K^zV3Qn-1YEOfp5SiNK<+hf^xrdz&_6k)$SDVzKf-gBSp'
    '=oCC2&eusc$d;1}2aJb4O@SpnRm5JPne&323I9?8=Gx`DL0773bu>ps;|l4o1YyA~+1+mrzH6Kx!MRa+Hf%f-{L?UUJnUITROReARaVTrq<jJHZrukOLL7V{26JV}tq7Nf*9ETLDi5Ymf|iF2@#?+QL1#PGaV5*O'
    '`r1XzzTyf`F-*qfPcD89g+<U`pwbV3C(RbIClptEei1$$&d#QU%grjRtDYV7xu}Vw<d|LA&Q#TGKJumv?v3~T3vU-nI0N;yF@oRR(%ub55`=L2>ZHe0rU6%xE#0_g<MFoWYTAe2)a&aQ{)dVPmQ_aJK;op{N&RPR'
    '(+%pmS~NpH-A=>}u%K^Uxy28oqG88z%5L?>A077mwMtC5JFHiUy*K7%X6}{ulF}u;ym`L1Yj8G`KC{ZNDC9bamg{WhNqXAO<4rnW6*l~O+@iwK);9h|&Y0t;Sj8$M5bNW--RW*_+c6GM9rXuVUqmR0WCz6m39D4^'
    'frU7s#pUN}G!)05(Q4;!Cs=2PLDT%!Bhs<o(SmpjU5M%`H{uSCJx06H-Ovk-{Ua|NP`hpSSVIUNb#^HnAbhshjaeK)7_+rkiD#udznaU>5h*R&m?NqJ>tXZUt9|;l_U2Ek*j*F@Ldnea$4><srAyNvXAr?wQ@WOX'
    'T&g4#uF=xc#iWSGf&Yl7boX^?yRF83=W!NHOb}sO9G*aSx)ClwDa%eqjzs;}-B`NDP5gRL^+9NhG>l(-tCJrm;e&eAUQua46psOCW9!D3OML-t2SV$C<8FoLaXo*S^^c}AiYcgI@qNp}`<8(p*f$5)WbI+2N_kQB'
    'R|4KZMr_ym<aJf=c(Ji-&jJ!0+I?YYUV5yTI;~^lFAm~g9E7Ksw&EyGTPjGV)vft2530X3tyUUk`fR?eHmA%c9#BO+7s#J4#`}*M)GCnH%P3AER+EQiDuOW8D`<?u`bKBbZXNvVb_=0u?1Z_)!S=UaJ=??|P;o7}'
    'JlTj@4F*MDM<_<Yr~N}5i~}-rQg<2l$cMkU<H1<oTZ!2*DSPT+^Y4<6<ZQ8{=x=N|;V>hhzS$vKYDAqRBV4+^it2cJ`gRkNr>A|eVY15G-udHIWBtfwY#S{C@lz)}wW`;;`lPEIx8B@#{G~wmCgUp}sIdXmE01M!'
    '<Kk_F+C*G%b)ST;Y3$smhfUphsJRv@s>7p~Fh{ALPPYmutt{BJHjK;E>hb5RkMq@30r2)op;=%s8AB+GvDlk2>$f(pV{my+;yLND)@0^7!iD60iIdAc$RctvU%&C-5Bh^+_@fsd3%MD2d|<3#RP~;21>#+2m89xJ'
    '*wpdw>R;iQHMD8T&Nhg{NTHHFj=akE6XR0S8$n^%JLvumD?TBpx7fvivna+>DhC<SX=7Kqz*XEN#kswAZ4Oy_uMZP9$okglnOWMaG5bJ`O<jt?-XZq5hsg+!N19N#nxA`b6ys-HPsEo+FmRol@dlP&F&k)BIGWPo'
    'N2jzq^=1ou2wrmv^gdp$C425lPz+1nP}4!HlE|bE7(%Q*&H_%itjj<r#Ij@z58IIYE54>K-b~f2zFKoIT4;nacb(kplFN>k_?ofl_;9dOFKZY5R|_3G4Ui;@Z^N#3G)v5>@0hdG86bp+&v$&`UG>GgC+E&zCHGXV'
    '`V|(uA}yR6^P(XHGd=?)Q@lm2ZCYe*2_r!55|kqWxmJz~p+qE@Krp;E<XFGfrVH^Hk(g`XN|P#&ImYfR)~_KR>{usy?b<Dn@wN6nb2v__geG%n$lXERXt$dgmD^dJ&`tZ-Zrj?G`_^vPVs@MgHqXd0ew+@OSngca'
    'ks&@<FP=Yl-XMeilthfxWQ5n4iFk^Q@V<=#sTHOsR{`B4l2o{%hWBddGj7CODFjD1#YPNfbCwa)=`5MgW)n>9N}l92pv3ti!+D&KM(S_uanndMC_2oh#*2$!A3GQohauv4OEf5;F~~$2YH--7>Pdo^B4zK^`g$FA'
    'MsZS#bz3UrKnU?Qq~&clrQ4@>(jaEVQ>u+{<(-IN7jmhYntlkdhFgo2$O>z~e=DFYO29un<!S529^zhhxK0(iTEX|6WeYO`ayAlI5eXv9ROo6xjWm`z+-i3RsID;P!?oKgJ>!UsQs-edq02}F=DR#<C7tVfAqD={'
    '>c16pO^QZ<2>aL=A0fDxN3Z0^w=M}MI0rg-7Vk_HU5j>)x{i9bVg?#8(m~*B8`=sAUbp6|*{M(h3(sLvKAkl~kNb<yJY82WL1{*#8<M~0$VfDYnh-MVTi}fy&~CM#HVeh#;Oal<pBHL)WiMFk0$bFfv2_gTK;Z3c'
    'Kyc8Zw+<0}kF$q|PCBO3;w0Ncj3C1gi(oMR1VL)WkWn9jrfa5fFOB5E!}stk`J;#L5%kNu&hyxL{taj15i-D23byumgos85qXim@th2Jr1!BG^yh{e`Y6*vj19ySRLnnXg?k#!t5lp52jVbQ=(>eS#$qw|xXZYg1'
    'htFR;eeeE@M^B&h9)58D@%#Me2M^zS4*ybS9AMc-J=`9lY{h?`?xj${2ag{9QSb5ncOE`gIGKE~VwD5d3x4JN!+R9jXzX-d)rSD1ri9<(kV+8csyBdn#20V8Z3CZN7V%cHw|nOe+|ihI&m%uVP59J+hLql&^%oc;'
    ';nN4UlHKO+-kW3y%@Ac=akf|!WY4HTpLP?r5o1OHUD`<i?GnMCK@gBD=IjPAG^VSG@2Yengrd2kQ4~6)ixqe@CW%04@uCY?Ub3p?`kw(+Lx42S?W!-JB&Q>aIXD_EJXQ_Xzsn)KC4!;0uw@0b3FyZ64?U%?15{IN'
    'U?-?A23TXM(Maih*;nI#Joq=ohqu%IYB@{2r`=OTucF%<Z!cl*Go%(`jOU{P`JNG;75#y+LA+X=Z;5Ok>TNeVA52z5@>ahj+|FE`m2&S3G^9(=^lY{`mOmG(X>W=tOO#XVdtl=83lIvXr3c9P@F56H^Of_6Ld)k&'
    'GA5B7tTz@2l*y!bLVlV;rJTrwYv6=X)Hwasv)_)+k+b4ohe(&q0zuxyEGx4U8Wp(!_^e0agB}o^e<-nG;X`vyK|g1BcOr=JUL#tE;`eN@P=q$#k`30<dEsqH%dxsD>e{RWDZ2nB&<aT=!up&%=W*bxyg)srJ!EZ%'
    '-B9s-#q-g`hVJWK%8sNp@WEnovZ7?*$V_uVp=i5Mc+^EC9Tx+ddP$Lux{~dQg*VQUGdoU0oDdw=j~H~H5XucMfr1IyQ@Tc7GOo=wP5bc5+hKDZT!QfT8a;RDwKfM}3?mYDp#2Xw!eST;K*%)R47h8PP4x)ws9clK'
    'EKr(e^J^PXqIPzYJ0S@>|2Ts%C&~bS*f0*83*!h;E}uJR4?0MBEyR()>qn&Jhp2;7H3O5sbs%U8MyUz7sRh5?sLC)(t^!)H$a~z|`a0Qx*~8-|@Ho{8j&&Nl;TEaRSucTqRwFi{5zT)f(eLH~(u)4Fh?=zb|Nr8?'
    'tw)aJO7FdYMGl3Xs^k=_yD4%mJ<NeAk~1PS8WN;N!m74h$gWFwEvvgql~p8LtroCP`()UH9}L4U78Z+zVHj8g1`N+U_+f<oAN(hdbBj1}$*k<2nY93Msxu?w9w$zmIOqEq8**m0Zgc||Bc+Fh<(v(ek<dAOa_F1y'
    '=2>(Aj|~}J$Ol!i)H(0c!uN;pMGArmWk#GX%_ihyLuQN8=w#-<$5XRjryPV2H?>a9sQP=D)31ymI-ZZW=Kpd#21~<na$t-q+^nb@$_il{g}-A)@Q{7X?<*q3vMf`e{<~=2%#Y?3^OZ#kc33BUX=f9^2_6BSKmAhP'
    'KP3(j{Lkleh{3d~UQ6v&=p0<AetTv&R$?ru7o;$KC5))n>(q{K3#zkl;t<qr&D=?WNvWpzh*B0N%8S)dMn)dVeKmSDykc~5l&J{cy(ABXTP4yLvY3zz1=<#~V|##S%>2p9r+dsKWu5}eeV{sEF9;|cNNFGwf--tQ'
    'uy)WxHgu}TepFtMn>f^*u)*@jdz25o`(lVyCuswqSKT?NoyZ)K=W=BX2i*R=!nhfeiipG1i}7MvO&Nr&JO*glMR7R2bMOiKy=c6@NFGTKcvmwuy)`Sxgr;rg0)Tt128k2Rkioya(G&n|7}z4)Y*&kEx62S=!~^r%'
    '>?xFguI`b-Oc(-umJzSa{4B0Ckj_-{x^LDHFeY_7Uv~07aBcs7+J%2uF7#ha(YfEOs{WS`zk2lBXD=ZV3l1`c$p{Am7-wv`a9A@#pwyYSOmuTx0k;XrR2Yi#?Yt_Wi`HoCD9XW6FFQKaap#kZ`Y#W>FHJs2`b*bR'
    '(}nXh*wD;YGr7o_0U|h24)?@;8{~Q6I+gpw&E%S%`PtJ)Uz`8u7DCW$cc2PVFH3fLk4MD;2HGc7wgp+ozxp(985GZCJ2m;DD|wfDxkuf7EI-DaK{+wo8=U75W*I*?&fd@dh?$i5>)4_!3A;Lf!hgr#j<Im{t-?#S'
    '<8*A6|5!~2^vuF@43M*V6Jvg6KBKOLoryZ46dIfklHhdIpdK%_%=x6I?+o{?uG+);UzA&;2A1bPGwHi`^W=05CJHgpz;rRXQJ0%inx;UR7M%WUx!jy4>TJ@F7e;ir{y<rH_x}AnYR%3HALf>o%1Dh3MG!Mnj%qqp'
    'M&@(U9x}|SIh>*Fx6ZW)!vys3b9zFQjI9`M=p--aP_#5H0A-stH+DimmGpReu{4`6Yfa8z63(<Au5j!1&{)3e;2RWaKPH+CjXHcjoldHeWsrz%G9Gs`BR(mLIdJVe@^s_~uN~PynyJ673={TmNRC8|v{T#nmJA#d'
    '0E0Vk)4q--KRrzi?x;$m)|F7CT|^=WN;?4&h6ek8RQJI+`oJ4E?-SSADm}_{HSWq>OJC_JZa?tnO7x9CHxt#`S!!~H)n&A^HpwNE(}k@}NHrBz4lY4YxsLea8D>@}uT@CjuEfU()E`_<n>qd)kvw!6LkYmG3|S2w'
    '3K5!H0Gwz_JC`{Zm5XXAw^q<YRo|QrHUi1-5cU(6C(F^&T^mu^Bic`kEyN2E?SdFr-4z2IJkp;GDA5qR*)kIGE6rR)OH*DTfdW^HPf$~h+cW|TWb!WTtURC3Ov3uuX)u-cFmTzPrKuvxKx#kITH(ninKZTS7fMBw'
    '3-z~#1T;s@uPX>d295+XbT-slOlsK^qgGxpUQ<O=B{QdQa3SGyw*yk};;*lD_B4XrVQAbm%-UUlnAXw{%g8CA)>dF*p!KX=EWnSn6G3!aPlx7vHOE^;gDGrPG^jO%S$33TzXcT=O!v^<r5HX7FEt;Efob2trG%H6'
    'c!BwAxweG2X^1^c_W}#e%y9@fI3V%q)=Xye$#`Bd5FeUAFc1kQ3tA*Lr&BW`%3RX#I7DwpI?s)72?5RD_@{Nn!NON{s4Q1-_udnZjY^;g%#WEl+?n0k{1WLPi6nU5|9>Kg-mYfyG>eyZ9DM5B@Z9I#*XelG@$3Y>'
    ')hN+`EtP?T8aHha$dzHR?h<!Iv+#mQS&u>JAW$1{6H2){;!&<U0Bo``1zvU3r$So}Xw;otlS$c*5sUg&=+{P7B*O2&8IZ@=gDqjxF5f_L)s|~2Kh3xuMgPLKVdU+3xib2=>1l==K4-+WwbUt(@#{9HE$S#D*Vd0^'
    ')?`P$cnXg!U|d{HuSz;S3)pZq1ox=8c}}Jk993=Stq>u~oLNF=Tu5j(Os)VST$y2#2MK@JMiH)VuqBJkj#wB{j(ESWlC&q=>PNdP`eEJC9Jo7>gBu4>t}D0@+yUBQ{n5pvsyN<Zpd*WGfyN_tD|Iq`i%8*M0QT2B'
    'zEiFDzYK?bpr*wTqa;<gQPDlFXU{KIoxa<CqFS0PUo%!R{)OZ4JfT=vOFPKW)pT_MPs5erjnYlp<$_P0Hlkdyj^fz^Zw4JF-Eal?jcEx3>5(!xQwypUMq?;}xR3RD_D(~MHQ7u`&~uv^wzu$FUC@63#L>HpE{F#('
    '6yvow){WwvQ0SG*_9dD;Hr1LuiRP#CwFioNr3eucP*u3&4+7)@Vn(Gg=Z33(3y|-KqJlM}uhj)e2`c0G=;mdTI!-o%DLD%|Y!Jk6-I%}dzVPk1Pnv!ZwV{Hi0i;uCZP*cqYJy9`ygML#XRAYt2@*8N20TVvRAV%?'
    'v`K|3Y{2h!1n>mZAsPWM0cr>Y>V@qrWg$@{&VblTZfEI0P0y}E1{Jr`j#8_ymvv%C2TJ8XSI(el54nnbl2P^6lsTA&oGSewrAl$xB%&0W4mEay*?z^<){U__YG7Ssl){#Pe|8H^fV$Y=mL|j#do5Pf1N860?88Nn'
    'B~Z#8bPiOKpbR)4740X<Y*i?sIi1x;U}9hM?GyfC)>t}v!`4ltE+$X&ZR<#BsSCPC%U>MNE&;%(&cX`OHNnd>tkS{IUok3m*QE$1G{r<E=Xzb(p>1OAb??p=QqAozkXVU^2iQ{Bd`a(YF}rH%z~DFrGKR9)t4E@Q'
    'iPR2^QZyEG+FiCLf^dFesVt+3q03-DCC_qzUF1E2%Ew=cPZ_2c@SSV)qrIfpC+?lq7kx|PvE=a^wK;LLK&Yn;?%Ju&s*|K}-v)`2-5`XcUKYthuF*LjrFr9$;ORy2!irxciDMK&%US569aT$U#U#W!?F(tBJj}|$'
    ';rWZyfu;KLlwFk)AV){nMo)A2NIi89W(#bM%&^cO$!f8;Xflfsv1CWPl6j;sNRs&g&u;~uEt4wBM89jO33Q>u`f-`LUe+(M50%*%0134fSxdV*%%lLYjs%{XFeIdL{a_J<Z-X))Yc9uQzYx2fGv9UY)UwLCo>mcB'
    'Uc8wk&Q5Z+Vk^Hr9N{PN;sg=FKodAei}Y1PEHvQ8jjkSg<QT?sxRA>Q^$gTWA0M%RT6r*pYUV4~H%K<nNQ;Qr<7&v0KxJp=k)LogKQnz=bjGETU=!<$22o`5Ge$HSZ`W&#O>vf%H-H}tDbmmh-ZcV_jEcZk_Ir}<'
    'GBknqQlR$IZVM<{YymZL5~7Ig*05NX6B^!bGCvqy|8SGT39#%n=8%{i@KpoS5X`?pCvdLBP}0!hlDtpwl*2PD@MywG7CKmBQf7{7fZOcVd(-Wnk2OG59{GehK5F9&G&_-4xiIV0b$;h0Ce#(0)FiqK%lDGtktop2'
    'DJZOJ+79nyM!MP3(tyJ*U|pw`-~iIDZ}&plMSR9vGv5%R36AZror_r4-o3xQ+fg_1Z|~72@>=P9w78;nqF4@L43jm2CSwaQ9N+vH?#1&MHK3{U$$-JLwFip?0Cd^b*u5>0le9(QnmSMMP9h&?bS=;DK6bglh!tCx'
    '=nyO!Kv(*B{jSPKhbauXYhdOp=4c%Ia7{>2?z{$1Svl)*XT)#n@s4nRP6f@!5!h1pHQ})mR+nA3nn}Dy&<{nJo|e@qM=OOE4nc3s>vT*n5A5kdDj^eY)W1)2BaXa%{3xp*LDmO#a_wXh=3}LR|HV}u+>F;dZs??Y'
    '94uyC2OhNo^>T2PJ1`|Rs#tnDni>+xxT5`dkAiIQphH6uzPfHTLsU(Thg=N&;(KAC!)vu7Icw0_p<z}ribn1C?-fm-Uvt#jz<IW5avg9}SZ*?$i5e?mOx(CIswv*0n?dJVps3l!!cE3tq`H1@LNmn6P{~Bl%j(Cd'
    '8QHKNyK6B$DaRM1R+w#_`rg3_Dxbs<<~c1a$>)ZZmHY=@-ot!XH--a1?o+j$O0RF)p?*Hs=)Ia+rk3XStD%Ey3qa~RjXhbT6TRmtMK>MUWZpX$quGF%v&!FqFK^$56yTO*Iu*mzv^3~ohaz~35(`(2t+$HW4z1fa'
    '15{H5JtWmV3aj5jd1-Rzs?(9$V$wA#WrC@2q=^qxRJ;s~*+B;_iL<J~twZ%i-((HzG5#oSu9qR)m{VW%DNUW3dku%UfW?c@ncQ9G)M2?m3@4R4NQ8zOCGLGpf>o+X5h_bsw+BtBsMVAh^eWQvv}M54#)mbVdQl`)'
    '_nO4P#;343Ic~A;0Nl2kJ?`*U=i{IXGW0by2-<k<l2k-1LC<^T4r)VoQ-{wGCa*J8XCbt1bKM#Ofz@9&UKep>Cey{HL?o)Him9O9Aw<iB16i9EeWhGS1OdP<xs3*{uT66da80##eZ;<E*DEPT$-92JDhm~l>`hgY'
    'k)ROb=~q+9CS;w1EOiiAg>NJgNY;q_$=`lRuWuum16qfWYg!u>ED0{PD&FGgW6jkXLi2g86Ux_}Y_7TzMAT{tHo3?qCKd3*#AFz@boZ!X=SllV4Bbwh+9%x-X8JY}Ipzt~HW9ra2g@r&HcbtSo;dby^Ge5%f%kC&'
    'KpL8&ylMKdje$Q-u_(`uCuKH1Vseau-NWIasnm{KyO)QDI2$^oLxaW4b-o;hTu@bqaJIk$x`2k-Z;~L@4xe3aXL30;+g#Ac>vcdC59fA^9l1`twD@GN^`@Ymy+Q}Npo5)lQ#=)vt<q&9)s^}e&YfFW(=5E0-yc08'
    'w(}E26~W6B=V@Qyzse&@IpKp1LE=S=O9rolNhM&dv-!F*EAe!CKH~8oF`${PEJEa<i)W>Ql&on3zl$d{Rv@?FG=S#KaRCsftWzW_j&UytT?ts4To)m}pc|nQoB@Hq{CvZq@5}XSTiGxIfGn7w!7g^?MZ!^sO6*QK'
    '@d9Khj)47>sfLdAj}QzfzJLP`Ai9VYM3AxOH`*~EIK{#NC<+)GaZ+Bu6^#+J0GZZ@-&GfY(a2f3kJUnvX>wRj@N#UsnccCWsf3|r<g{M63at++P9mJ@36pPnE<iD$q~<Orkt)Lmb&4dRgL#O-MneSnYt0B>)pkW='
    '4qla8#Dv5k4Z;KNxPXQA@(m%4V+{s$NbsCh!7RFbqZ`TI;M4<@hklJ}`qaIlbBF!R!;Ra4#3fi|`;#q%<A5haIH+iX5j-GnES2a)HB_fxP9|NyrY-@sS_HUB<xN+5%Ou~3Aqw_`#GI9pzK!q!YRx$rz;oH+B46S+'
    'guVh&Q##8CP5kMkCS}f2lHNIBOs`Y(00xQ*Lxm82$WahN2qK}urG!!=NEx<Hp#Ml4F3*-`_vSKeu>*?_IAm}cE(>)8TFbtTogoBwWhzL6?Xs1D0F9r2YC-pN6fLw`0JH-c4mpC67_B@B2N8&Zw+IG~#r)WVVyuN<'
    '-1|3p)jqObEa+X0Q;Z*3t0@)PMMr?jdS%lzV+2<~OXGk(5ahUF0LlWDv4+U4b%k+!=9&1rx)4AFDiil>kD7J0D9<IHQ0}|Eggl?o3MC)J@et~r)ogL&^8#b2g=#0?3wX0>M|6D#jxqAOi~04vs3P52p`SV>uk%Z{'
    'qS^IU8VgJ8!49p&ebK{!Z3|bTvVs+CW-9Cg?kU~U;J)mhLDC?ZIi+2ap-I4lDadWHJwxa^4s^KMC0n1rc=G(=izkmqk6t``KKj*PKY#Ml#Xn9kz;nDoQ`SC~vQI_qQ~ntZQzr&zS2g`C!dhG|Va#!(r4e9&KLjCJ'
    'E?x`ng<_jHHflLW>i~%}?T!U*jAurPnBRf&?u^YkE$1hQOv=tQMt6g7dHS8ZsZh8%p_YOkHH!9<XoLbhw-Kug334{e4X|i{G6rZQ^d~L2gK)ulf5Ae7A0vEPK<qL0#y*HT_o9=>XD*Ck0j0uL@~DQYuVTErXOCVy'
    'q5!_zw*lC=zgdp3|8fU1sC?m}00p*4k<n5+Ot{_}rPKU{VPaRg!0Dl`!|RsMfbkWjl^jMbTTB$titXg2%V?M=eQ<P{^AtdHlu_Q4E2g~aMtRoKcSPB;my9<xnrJ&D^oX^dD8E6N*N$6*c>O-(EMc(C=5sz-&Su$U'
    'ITg@T1aibVlXbEWf&jnvV^m+^0c1H$aM5z?YEie9*+5&j6!k`?e(U|r{@S_B{QWHW+uPdQ&(sFzcS?c^$4xnPijdFi&JZ7W@+7Mc%DK8;#yE*}w|?Hb3<y_n*f#}(Hj8&ILF@};NzmdeI;_YL^S6x^R1(nWTptwb'
    'bMMv8EC~8_h|$Y$yESinQTF`8uOjGJu<}`E;npzny0Q;L<oo%&SsR%(g6DGF<T?a-uZ<8f`=wwVnDusKb{PV%WPQQ1DOA~a|EZ(57V`$5Kyteoky|Gm<7bO)b(&zg6(C<w7VXNyk8UR7U4;e*!<GVkw-o*%rSJEV'
    'x}K&6u4La=?0#=?>s2SO-xHOzY$I~LY=?Vk71W7UPXV=>v{FCq1h&ejeIlx-dvFKZK6?7ghhIDy{rcgHrw_lffa4fl6hm^@Z=1{&kf~(nG*HP4jf%u}lR2{n3<6H_>Qf^<s#gR&A?U>p?#bc_j{TznA|v{bMuP6)'
    'tT@q$q^TLdM0Hdy3!s9wEmqaWzI%yHvd@0`^ySN^Uwtw9{Na~h{_NqSUl5awK#VAbx#6T#r$#IFIb>rw)71?WXc12gIpxHb3VL*t%-$!siOTVg0&$Dcka}}C0xT*|p3@U=D+j6NZ|%OO?5nB15*^b-%$D1=+yiUk'
    'hfi%wMxd%#pp(wkULqV*(jM)l6mQHu7R|Ghgm6w>Wh4~vI=5mnYDnX4Y&Wav5>+LFp8X5}E2LT8r^|x*x??9$attTy4q@MBzo^|N{M{j*eTfYstez9G@#o%Ed85|%eXG8EV~fSiqtoTmJhI|Zwj$hl*e3uKT7=ee'
    'arR+F(5PTLf!8|`+QBPc@@*<)H%8YHG{YbxMvK%Hi2Vl>Ck9sFg7d-;cuFA4kBY~&wjZ{W-Ru}qJIil+KyYetYNDzzi1hKG`B*@%SP4mV`*!!TqmDmzhC`LX8fp?lH(gy*W{}jn6n-Xqqup=-`5p(4ALGE0JZOYN'
    'qs|&T&KO$FXV5rivexmVa3!HIrQC(sulZtHv41(}b-1lRon`_OGl8HXxG=o){PE{mIX1cp(O+Td_GZkX%n=OAn*sT_fwM0BftJUigTvvlKc$=`a6AY}m0Ls~I2U)J0Fjsys(}PTD=2D}0pFFeTfF7U2@|jrZ43)$'
    'm13K{iVQ}%LIDHNVjDRI!co#UX@V~a=<emuAKo$5+gDiTsD|4*SOp^u_a)t@bd6nDU$Da%Z6gK2RR`9zp0CKi%7bh&xeS6FhDit~`aA*T6z6~-U&B^=oSn@U%Sl`%t_S5rwR33RG_kLZ<J4F~8f;uJ8r~uQfuNht'
    'XeA~B3-w28mZr=IU52!|toK(nXV$}nzpLvx9!&ctEI0*=iN^>0t}nQ?HfokUsOn)j<NP%ur<!g@^%Iezf+QNz0R(C{b#z8XF);&l#gSc~N7i*#Jd%vAF%@K08qq=Goj5G>ec$S0HJQz~XLljklYZ{G#Z2Cb-a?)%'
    '0YQyY_McB?kC)>u#WZ@-XfMKjG9UH<41WaI{FS;%F?o$xqMI}W_~~Ru1Rce5Meh&={bdw0$cm<)WNKF0YF=2eWBxE*diUT!?cp=nkQPBP^1X+59;Jjfd4TAq8fNVfK(?wPK<*pq%0QQRfqt;}Bd-9y@$J(Cp68q8'
    '@x=yrz5eO+?PP9L5<2$nS`v`k$kP0t98*jfabvZZ!>TVJ^xpl|Fuz@A)=Y9cs_{!u9D&yyJ<T?YjyY?F8cg|G^cVVU2?6&&;l&VJyeHY0{WCcIcHscv^l-$Q9W7re8!78x_B&w5F}c!$n8K<f?q1Yuz?*7RZD+Ij'
    '+is`7IyYzWvz4d}Gz_MmUHe^+on(s6^RK?(boQlD*^QPzo+?Pg6sRy3Y%MV!f^e*3U{>|>gTaI=lWY!n(x%IS52BI@?GRLZXbJ1Ne&POYB7k|kF5eIws^sf#=vUaslKViT&sxjl$_OMFW-~w6Mk0FlR5YD{*AYe|'
    'TlNE-(QtLI=zNR-G#OxYabRGkd_hoRmyDOIi{OfQLMT19Ieqx2%OCnl2^5{w|3jXHf|S0WeRyyXX$Dw-q?2*gPlx;h#1dd+st$uj+93KGL-T66xHv%xJ0SXxRqifkOrECODTR<LM|eZfWQYmrN>(3a1O5p%Vl$uT'
    'i$&4-$-%+L#~&Y?&c+58_-}oCy#Y$McpSFZ2ygc4Ot*EsT%e}e`N`qE5AV(Hb!>@+-^1+9`jhGGrw2bh=rH9l%0}`hT`d999M;439H-8d4$^`dA{k6R{P_4F%m9G|OLGc4*)F#g!&8iG@#t0|SXKcF-wKuqY)~U9'
    'l?%L*61ipdLj`NosIq3HO^uEb<Dv(w<S>K<_E5tJ*;P7#ZiAET4rz<M`$2|+&om&pD)^zd&8nqsJ7$E2c0IVRI<hG#-ka)Gr$q%jkdSK^GHk%B>^3rLu;5?M&G8mvHwWWgb1*UJyQWRk2*Vb8qd8Lxk9Ori7O@%9'
    'I_pVHn2&gk=L0MfJffXHid)4o|Jp?dc0nsX%^yH8WVWZ-VrdRCAYl%`*Ee`}XrGOz^94S-8$bH#FxQrvK}XfN&ESs-2vbc^_%iqE?oCVHZ0)>b_Bl4cC<=sVo3x=ubx8YYVhOj-TeohxlOFb@rr~C{Kn6E|4LWH='
    ';cbH^gzeiJS_I}?D&}vny!cd;cp8LA=;UJzAnzJWG0Y67Q<N8(WlF+fymS{HVwY3dd4V!VHJG;&9kUDZHc9#poq-3g%;p(H5!9}vmKZA*`q~Te=_%kt%iA7pjEtg+ot47FWLQ_`Z<W5nMS$N7Jiy|{YO*68eugnK'
    'IJ)f%^<Yu(ih<~I+m%koQhkM$S}%j36NdvoaA@exv{6^$b*wp9c*dcYL)gM@<>wpOi*_4}c9{Wd451Omm{>)-Yv;vnDxf1;U`?<LEQ;hjBxK-pA1a$~RtKN;UK&QXWq11^X*&?4rT^Bz8K#idqV*Q~;=-Ucd*V7*'
    '9E^~iPMR!Y1g(|yQ}H*#nm+SCg_EWEEI8}KDbK9vypK0G)8Mo}I6C~$U+hi>;8py|x;)=5$~9rsbX?39E>}hj;~*O#j5HmBJ4+Q2M%A>Tr(@yU9LZr19$9uM))pOuow_7yo0$+@P+oTA^3a}<0u2i>UnAz7tG(|l'
    'er<(31tXj=_9ZGB0%&5pKz#FALXPN)=eA|g@vesoHYf5sKGDllIpDYB?cqGUz-uP435p4j;^eLV($6|1n2-{s%`VmeOlTvD0Oco|Km_kwUE<rf)k_KYCNbo7pWM)U4<(+m{mr}j>>AhZ=;*n|IIT-W->)5V8tLRP'
    'ryfS5?*ADNfADu|CPYt~r$xiN+R2iBoSr?2XNk!aJ_sfj#o*~jM9;1s&`H}H=RTPX;X>BhTXObR_ou1)T{Bx#4{K(xb<XdRNz%K?4qf+RcN@nGm~hO(um%d0tU{-(*l$Dpd-(%M!!ucX{#u7Fd5LC62{To`U6FST'
    '6=BpK^yUKYB{(hB_!Lu*fZrPit5Z?Zd>wTN%qW!UFrXbor)Stjj?kI%P0M^;t~M{snbxcw4YtzyQ^+^hGV2(U9X9mem9vz()o3ZpYTzwoX}fg(TDQ^PjeKZ{<z~fQw_&Yk=ns}!*13!QPKmI*Et{GGh_xP&<+&q`'
    '^E?mjTywYhslvU8D_@#N0kid<jk`jkd>2l);$1tM{5^7y0}2f3jb3KDMXNNX1bfk9cDw5SLcX{DIE;Bo_J2C%2c7M^Vo9wpwdxox6dPTNk_M8$W<Slz$axIeu5(^5t)(i0lg7?OsVrzOkUiRS8e>(gR&Kv`LK_?I'
    '%V&~`#5d1WL>7y5K8v=sqrJ8V)bL9$e$IU+w{DKxCb$^E@pP3_?q!Y_teixrvw@c<0s%n|QOHN4{klZK3|82W8nS()KX)RC%L*BjDu?!va(W4!WGXA<_3oPf?z#>(DX&Cl$$-i3%pvZ%y}}h&8P79$c1v6ms{PAa'
    'PlYR#A-D1{FLat|@NkgT=qR~sWJ!%+71k43YCf?gkIKMiNKy+X8eE7oRJ<icR78k1t1D2on>2Z!HI9lLydFxa=4V)N43uMvus((UDBJwla78|DI(Pp?sw|84s|pj1ES^&}B@U2g0gV{z?YUr3m;_TW{!Dx5Jnz%A'
    '#v;wXq$lDQxUc{^*F-Sa1|uf;pe2QY_KYuZCN4T@zMZP30hb%Gh#h4cAwRIO?Lf;zVi%WSSBWdQ_hL{3hD@eP)H}!atCdk#5Q4c&Vg8PcK(Jbzk%F`Xw1t9lATK~2Xs2*Umd^S`I`)wFzO_vFwU%R|mI~6gT|rk-'
    '#<2#DGXfM%TssoVfjTS$P(&<UR--1U8dR$Fh0*n#I<1p<unTxG@BvoaEJJW@G^!#7Qp#dY*EbN90}`@a%*#ru0<kN=Z|!_FCNoAB->lIov?c|i4q;B*j-s#*J=i<=7LmN6Vn!=~{|GbP9Pw@t1S{MH$?lul%VaYM'
    'dq^Fq;247>Wl#hBfygL+dN=wcqJu6(SGrg8)vA4VoVqSIc2&1wF;YC<ZdH8hm`*;;mYdV*+CJTsDw>WMEh+*;b9OsDvyl~y7YIZv8qo&Z5<X^R`L3Ikxs69O>B#Y6`KlVgzwX>D%M9NkA_@xB_*^Y_e<K|{^+T8x'
    'oKu?hJSu|H_CrH4)V9`0d>1fFW}Vfg%+{lW`;)8AUd*9UBb)3bwf0gua^eJKIIB_gRv5^8R$@3yf+cJSXvJkH(aRo35Qk1i4j;E3JId3(7xLu*1b(iH^?}?QNsG_4YIfbt(l+QUrDxW(D^OzPb)Zvw{_oCV-VQ-j'
    'j0*5;;VHjsLTI9t8~8!idf&UW%C%5knYqWI*}k@-&P<1u?JBdKW}m=K0Uo^k`o*uG{PpPNv#(z~dNTUr#j~%Uzr++xmEN9@7o|BOD`$NFNG|8f(n!Ioav-vajZL@2%)M%QUTvL<khli4mM^{oq&<+>ZkDeIH$JtW'
    '<$3d*Yf?CzIJur-=;;5;G<Nf^DfA;?i*FVBk){?&Qt0BqrcH6|;DUepJxce=B4JTSOp1{m(jz6D;k&|ZoP}Gj-f}J6U;nC7yzwp|zQI_1&y!7XZ&7nyk>v30K6{CVvz{`d^|D_~FZdTh4dNepExYfo*%6xdHI$!A'
    '%I1Abo7S;Tbn|n;-0U#JEdKTa6DoAn1{K*<F}JAWHtgC%b@ZubdFl99QO*e5P1*}k>8tfl3W-D_uY$U00}i7pngZZzB~0Fe3wsu{Gi~$<y{VNFC02TIRTj@OUg*k3BCSzC*Wqnm<WJ8lHW^Qn@2KjaW8K~8#``9|'
    ';ycSQr|ee%plcnPae}Yt#$9$iGkbpmkM6ifgjK&b^?>kvFQt91lBX$dNSz2qfcOP$6>EHs{bRF$JnELw%ZpS;x6;7ne{K8_>vSI8Cz^JNnmxkzTl9@nskj>#_z*fi{7|B?#agQ$&roJ^qC1l4dq_*`8*D{mEInl|'
    '0D{HU-i@Y0y;@6_T_@p`pPq6WraMom)839A^+ECxN3<F^XL{2suOyO)rV?GBhSnso77RM%3JwN@QB4lY0;G&3=i4*z+w!WgJFg$BHrB8v7p=0st;DKEg7hq4$w3WrJ`z<UWzO-ypiy~8v~U#}68C>&JBhX|c1l+n'
    'rQYt!kzjBc&spW=Ki`#$-{m|OBh+3@o^HhVy(!qlR^VaCy8JU)!KNy&#I}TC>7a`uG3T~|$)AbmFJiV_z}=?(6nQCWvq?-9#MD;tEV_<YZH=g+AVKs$Q|d+%%s<B1#jn4KC4>Lb+y;9&Rg53VwHr$|f@spOWTvaT'
    'IDgQ4*+HHs>_;RDH5pOZG|9oQsBIutRr!nCHD^)oM|=;1l};uUlE$40x99AwF&UwIBsh^*5gl4h!=nCAw#@%HmGN@Bp&d(9Y&@)MA!I!w)acio7W}f+r*;achpB0hOycAAQ$jSpw_&Ep_$G~K7VT9z1=7UCdTl5H'
    'TN2QzI2${<_&BlAU&W*W(gh)NfZPg-v#1x*kh{Kl&C4`k-BnA<Iu{Y_S<MG@G4VRV-y6^xT1JPwePedhAhcTyHC{&1ifA?McWOx9<<+{*NNIO#Z7x$1c^Gf>&i6`h^{?#K_eyPcYL_FoGKx=?QDEs~34R>Gj(y<^'
    'G_)k9<dLE>r<CfBqOzCk$=bVgwX{HmZ@~h4BJ6pDWRBy5_EO-FFso|{bv8>3Ih>gT1#x`#Jn|U4)MuUoxb0Rv1uZv8R7pwTCC4|U`hpXjhaurvodYH(N8Q*(%a*#kL_;9thZ;xi6=!7TwRtjaL0h_Jp%jqs^O(kG'
    'I?+@-6W^-sk?ttW54aA!7zOt4%abF5za|_+F^v!kHE5+1P*~Z;)lDZi>h!A1L6Zn3J*iXxWgrcQvFS0FFsV_CV!%NZe6L^yj%D%jI^zf-6;fcaQ%AG%Y`(yF@YQs^*{+ZIh6l{_X7yaPtG$yOI@TuDCK3QAJ;+YT'
    'j8Zp>W>BbUWM0|b!CrPElW*Sl-t4ebLvC@2{u(#O4F>bB&7_2#NeC{*xc2hr56z<bG~+vOqXb)CjY)QV!O8YvT78;riFyO?Gfi6WvZRj8r8-^GeqhhId|v7)4+M;lfOteXju(e_UAW~lw2+lRZgI!xG*i3{S<zP0'
    'Gv@$M`jPUbT!5wQBC{bwJ*w)2aY1dICAvR8eSTqP*m-|4uU3F3z>%{2XFW^;gK)k{$8u>Ac#V6U-17r&JWn`itMJ^<sKhThlSljoAQN+1&1_tMO3=sVnKK@;cZ9VoD)lF*oK;$9krgNrNjIHY93Q@=azQ<<fS0&%'
    'nG-nZnbjUcya#i3q^NRqzVu25kC3ytNS+ei`5A*v_}d>(THG#lPg&a%3iiQ~p5Q^lXyo`MMgR3O`zOV#s|SJ+U>}&JXryClGO;GhmIqBYo8(b6pMK3|{+BjIr<Xj1rCsjoB8FRZh#x?j2pgmaZcfwR^$dZMBIvKZ'
    'mJBV#PNmdiojcN4GkvuaUQa+(1F+)Fo$=BL<1eSLr{~q_W(vr$^D;Z01N<8pYLP)J``xyjK<?vgvo6O|03X)j;mBrV$^jw`v^}ZP(4)}R0o}UbM-QLodjZRG7&Wf);}9(J2vkAfbEE>tQJu22hnlTwfS(N@8KEkI'
    ';~vaCcmk$|><`&t_TWJ_r!a{nAQHVcJHYw$q@-U)Uquqr?jH|&00C^<oHKbA!h&KZt~>5zhYmV{>xhkUHJko!i=s8IlT9M<MR4hassCW>Wz(E4*Yn>S1rSqpDcSqBA$j(J{hj?=_wx^wq4P+ZOmG9C%+2dRc2&lI'
    'aQUfG0!KIS=9N6(S9Njv0QTMoxY;baP|W+-Wza?Vi1Xp6L51L7)31jxN=^<)frV6pll-x3JM0hesj~h`!zj!5ZO>{w0jnCXDp;a#jF2DU`>Lx@t%&02Y<@nn_e53kv11&9kesgnIv0yDfY5^9xA!)Cp!sOxy(LI}'
    'wl#C_c$$^2PfWJclj&N*C&(cLW=3bVT8_~UvRR&=oNNh7WHKQLkkjq?#kM@RXi{%C!CIr7XsKbIY9d46E1<)xj=}j%Df9R?7t&*%<_8;{$~SrgKb=D$%3HGa0Ib$g3Nw)ScHoA;=s1hZh?+^3ghCFuz0BHd5O#EJ'
    'WNt~QnhrHzZO#aYQDh-KDF^v7FKNgu1dnqwhW;?Ms5|<Lj^|UMJayHdmg7?fD?Xo2japuUxCEK=R~9Zcb7gh1zY0qCn^X;GP(T>M-+Mxd9*b5GWWHz>0@46n1Ess07=t<b35F+b-j)I`W`E0~8bZOqV}yTT?h%!S'
    'sqbLcxrCHgN0*r9itk!Yw@QA6{OD<GIsh<tWwf}oyqCErTqK+)VR&rK_p`QNdQdQWq>bh+R89CV2@fk_usq<t?5@aIB8EYME~z{gQRE-?KSpa(i;qoXKE#@zY)-2J+-<NG*2@M56dR>n1v$cefgERg6gA6p6l)&G'
    ')7HyOqguS6KsHB``TX(c3ao+8_&r1z(g@Dk^RK=jRom3OTmUcOb`9RTRiFI8AM%-U37iXTRe=lvYZ$4uGvH~SOvh$XnR8eM=pyFmi2;G+o~;EOL()x6eg2k-M@#fa-k9@tMi~8c5(IR!X=P5n;1Io;uEmvYGF|tj'
    '(R()!%+g>U9zf0sN^2dK!|2mTdl5Kob8r@XC%(~M99C!yCOFg@bRELBnccdi>;f~IS;q974x+HC1u_A=GH>mLT+A4}$W7N)EK^L6)||$|EIa^)h-}{B!a9hGB85t#cysJ~7h??H8$EwlVF05z4E^Dfy`p^R>=7M('
    '=_~+Cp}VOkhLd6t1_}^;wZsw(os-F!yc@X;F-TskST4#}jO=#X!)*%9A95|>8E@@>25|I|NDcgkBCP!QPZtzKEVv;jAF|a_To;#{+JBvjO=-|l>c?}n!E*4wXeKSVY^Q-;?GB@LcW<E88X;qYe}!QfWgif{a1HXk'
    '8!|l53ZLx&4yX@Pq+v8U<V}NceZ~yR_l6u9LFtV+|8aLXO*kItgnejxUK2b=dKh|$68DL$j--zlhNoC~00my9q143`L6PPzM!F?|l%81}w_|?cA!DluJ?@}&(^r>$muOfrVSSmSAc)NCB(G7Vt91a5^+?36#t+ts'
    'LN|lXLqq3#MW)Nng8U_v9kX^JPxYLK$y*oKxo=o*pg|0dH)(KG+0-F*`9=>Wy-|_N>DIX$C&`LYt~t`T;Zt|F#8y{t^C6}c8c#S8cuB7{({}b&fYIl*!|}-vh7D|%R=c1YvpQkb2$ppg#mWJg<$-SDdtzH=2d8U5'
    'P5`XcM64O|Ht0$Oz)>D&!@vjd9){oI%>^hT!B@?|X|2Zr12MPbSJO>+oDUVOx;dq^GgK|$2w`;vIyJ+w>P>NJ&-h1P3+oI4SB?E`m(C8uL8_{}Iw@LGJ%BygRuQRWYO_L1KUfwD?!Kh?BT;Yrp>h)>hZ=b!y$0h='
    'mE*r*)3Rx>!eI~CFH&#w=8m!28!ASP)16wZ><Uomp4+4ocvoae^A9WxUuoCbk#9n*1e#7Cr#&~EvI;Ce?uf@y%-$;Uag}0oxL{`<BHpr=7C;dPIA^1CZJjAbHA&7(I0GG4o<)Q6tO$x*+U>^U^&NcxC--A2p`iab'
    '-J*~mDK=Sf=uIG-dB8(*nL&tM9X7zeG4pC#t;(@5(VMA>H@gfsi*yeZn6W({Z?>|lfwgkIg5X`s4pdg%7FZW$wlM<JoMtPdY5@?Lrl>I|WTYPf*C=7U&zl+V`S-G_y;<pH&$sL8^JO)Eix(~Ysyf|n<_r02yPZ$u'
    '_wVN`IOLOo#MB?}A05^VAeA!-8Xm7P!q*x+J_jHvQQTqupOx!Zre(i<`W%g<Z#Ou7OxKm;SJ2jSVU7<K+@J$_((}ikyPISkPi#x#CAK}E_!0S74&6+c&-&|=#q!uOWa?TodXNWk<k5veMona4d;^KJ1r#wK&tWdB'
    '6KdS1-f~*yf++AgSGEp1@9AdT?wdi;RmPiE9;p6r=c~`>3noc`d)9J2KApdwM)$)F-GM@bKI;RdB*G!2m|rOjQT3O6g(jA!4o{v0Tma!s`ZL+D9ugrK0`V>GAWhd1v(Dp(zj`=&{PYFeru8gNHp}#Xa!!BB^YhXb'
    'e%0};sQ7SjFr+ocpQV@BPWvPGzrw<V`Vc~-TbIvyxh<AewU}0wvb2>I;AEpbgC!_858hpZvmczS%hf3xA@Qo;evjruc1oRKc18qG&0I1{Bbu#Y(gG_J9Cgk>rjlv$1^K4-_AKXe_GSvUXR4F+naKnc1MN^A0d3#t'
    '8kzNo&6XfMs~_AR`RXWpO}l%K96}X-lnMuaC5Nv|5Bw=$QQ~M9jVJ3-dKG>IzD*~*Jqy?fSLFpRLS>bby=270VsUij|H>?dNgOv+)kV{XFBv_a<HdGL=EyvfElkMbxL2;S2LNGgK4H${6h6s9E;2EpIEzFH85EdJ'
    'koyp?Bj2Wk3MS`4z@Dn%0wDvO07prVDLY)^M1-gn^J*lYjW)|J$!x1&)N7?p3}-khnVDMU3e-Cxl&c9CJRu^_00twS8g~t>H%|G^`nZGp1+CNi+9Kb|VmU^aLJU4%7zJ}u9`TnX`#;WZXNL!O@9}?(dUf3C?9j}q'
    '{{8I=0!X@8Ms26lT%w3}pnU4&(Ln>-2I_P-H~$%dZ5qTrPcHfj4;x&rf(KFQ>iwhI51;2IMG<Cpex<iCn5P4nxkIK>O2<6<%iDfHrf+Cqk?4e8#&-F*-L>6K*&y@}ZppNZ*Y`;zOq7nmh@Z6@RbB&K6qzdph==Av'
    'y`?s)3<B~fPTZtqL;Cik=0dMkR`e<m<78M9K@S}^A~v%ioyd8)gxTj$XiSD=Zu3*lZgEnDhF7=QFc_ZXWA_K|W}ugCmMem@{!uP5<lO+$x-CFNzmNTi{wT{8qv=UKN3hHi$njLM8dRXW2SkYCIAqmL3RMQ2iG=VB'
    '`=&&E{_9<&z(+sQCCbm4llpo$Cr}~Jh*xn84b@bkqnWDe`-isGL0dbR=rovB%HB?np3}vIsmuHunIz+Vo^*c2=3q_<nSjm0mlCaF2kdq3>vb+=?3}JISX>uMY)WL-#mO&GUCIUDcIQr`bKJpZzq{jU=ks&nCGX@7'
    'S3Ld{O!?wEoUQn7r$H^Ed!Y{aC+yd(K^HRLun=P&`RS(Q)uy(SuBIY{QbT0mR#e|=grt}s<s!-(Ee2dmkCs~t=Hv)+50^vSzMhX9S)*f-Mll0wx&2F%$<_X)WGwDQt{3G^vOXfc{eFj2Di~vPXQgOUpIfdzoWuSl'
    '>jzGet|_RjM&KLx)~v@{wQ7={v~+i;b}QLOp%O4tt^O46!SE;#{djF3r5O-tlXTRX(0jzdY5S0l=K`x0UG`Bg+2BjBl5Uq}DsbSyqUEY=&whEBUtElwl3-8vRmwnoNf}ntOlqLbAT+yADc3of{II8y#uG@~`tPY{'
    '+YF_^OUU?0XN&P=F!gL^L9gLUFKadIC@j%Mj%Ip*jmPFYEKYb=Yi(_4OaL2Aap+jw3S2s&JQBKzq<-4Vl(t#aDWN@@s)g;Mp4i>lobLI?VP9!aIsqFn*=!dJ)U07=1bv2VN`mB%kpOI9+UK}vmc!bVCd*Afi&L@n'
    'wk?ycX3O=mLA(SQL9e{Nko)vO3V@Bi5C7cj;_%}ON3QF^FeUB7igULE_mavASJ<>qpO0St>e-7YkC`{heEi`(YC^4PIhH!ra&5Hdt|xp_$27Jbzge3!e9EAVq;BM#7S}E~p&G^DjS&Q7JVpv6IvVH?A3yp0%ZI;$'
    'hDqj&>FepDILvDrBWsBp6E3M;?E$UFWfT&N1_>Js=e^YC1PeLW-q0aqay)M%bRsxI+3#?+MrUdFetQmpRNdGam<z)DHm^`w&>+z4q6RsPV$PfAkrZtqFLE8+P-8q7T(HR$s4!5DIFq>1(FhL(2qiC!4tRDvUH8BS'
    'Zp7#IcrmX|voeDbLNFAFP80S?aP-dPy1?F7B}xPHB8m}ur*!BZ-)pIwDa)Y^9Agip!pjs4Snkiz2+2huoCp346U4pIe2$X~XCq2&Z|Les;6_C*K7lkfT@VT)siAEk)DTj1L$F9ssFAg#^?eg$FLj_Ob#BOh(?uAc'
    '9hg*pum0Z4)W$nbCvLLdYr4roHzfNF@3fupnkqdqU!Pao)e6Oue@zpgE6l1tfm>;a(1{c<XoJ=yoIEMlC+Ih$$9Z38=AYM`(6o+1pq!cJ0Y&g)TI<)g+G04GcSVg-GX{t?NnoCo$7YdiH!xRBp`Cso?=JBzkO`w^'
    '-T9lp!Kb)?f@@so$o`$TIboX)E@xBePL>U1#*Hf{_Xyl+`;?^SxZH4iK;O5jpDax*+Flj7u}a|ULV;-^f4z%@n_MFNI2Q=5E)T4p;NJ=q+Tlzt2GIBw-wb$Zn=c1y9n#_<{EdiGnPjlRWQ=w4Df+0E+00C$)9g3T'
    'Ui<=fnx+YOO-(e~;Q2JIG*Mpzb&BTorp$P`MagrJS_LAW9R{iu(ZL$3+2pRZSUB(M;tW(7yKMOSJ!l<GOts(OY{yzPY%El3w=i}CrQ&j;!g@;GqAcF<+FEWmtL^4rb8S(Hte7|4%U$ad&&`Jq3iQmVU$d0#z0@kL'
    'y}oO^NE<BHDCbj`ck!Z2tKGiozZn#CxB9(94c9v_+Lma;b&l#i-h$;4IbodEcjexGeVvtsfuwi|;<KPxwO_3S&+Ec&nPEL`a+jo0X~T)iTAQbt>7$CZD&k><w_9b}a8GTGTwmtS%8^LtYVMz2=xt=^uxeDHe%Bw<'
    'lBaGFOde&fnX=RM903}ni^QX)D`X?y2}P(Nbj%{`J<hU1kdB{n5Cx%;?-N)g4l+0m$sXohe9kAP9_J^i!%>X~j#Mw22L5@f;n)<WZo3(38#f<F#H1bg32N1i4=nsG?UZ1C@N8eZ;JMR!c4QCd?an*8<EJ%(qo$x6'
    'C<)P~wXFImg4o<mizvi2W=P|J$pBqW5$5G?;&XzzFS5&(jcTzGzlp%lt33(VXMj6g;xGyYP7z`-s*r#P{*9kP6$;|Cz)ck5YMi+TtCJ6dQfLv3_(7NdD~rhITd-k^xSzTA0L?;aa-ZQzV-Qer{A+MT-w)KcL-D9S'
    'E9c+@jq&}ikj4CLE?z^Hbjb6VCFBAOb;m*!SDZ!DMJfpFj_#^)G?>D}!7(wm0Hgo(z%jib#UxuDLDCQ)4lb|FBEVW!$dj<H7;~+w8%728vQ7s@_gjn*=dey&L)U^lCmN4*;|8nLH{2M`bbjvYHuNEwr0bKT7!~IJ'
    '7#!Jehk+}EWO;`)B-M~TpDc1ZF>8MV4zFPvq3GabO(veNma5;iWJhU$Ve_ZcCKH1v&Trr$sdq+%2zL&X>>-(&Y=v!Hl%&y%(k0~D)75h7@-LARl8{2|?Vr~ooYcl(2`d=+*tO_lW2uGxjlthsa<>h+K_2z8a?D<p'
    '-u|lhOvhAb-X}+n^0={IYeV}8JBPB5X;+;KiXAnmW6n0vbm!`6pDOhz(moU8*a<GJ+&~GW4!AM+_ONSljgp3jEeGh(q{Z6lXm6J5S47(@u@C6oEjp<0-u=iN@;1%_iw0^mE`Yy*csG1%gz$8QE80e!P|OGL(Sh~B'
    'D50i$R%f#`u7x$TOOzH=nk?H&Z4>pnL+qC1+T_|n@W?>=?niTQlXR#D{7GcBX_au7H@D}+JwH3!Qg0L)`}*1esyaA~I|om*6yq$uE;@z%R!1N5g}%counG$T!EzkG!!3~Q91&wN&2=#bQcig>4r}0jY0+2xEUxDU'
    '_x+u`RQSH{>_s7#*NWRGZy!NE_Z5MpSC_Bg%vDVJ@TRU|KcsV3(sL^C@kw|R)j31e!}B&;IqgJ&JnG!UbgY$0b}e5mjAyq#T5NIh2%Pb7Zdbff5hHr%qIRvPy|E~)9xs3>qCRF-YYf?6BJZ-IQd^HneNCoje^(q~'
    '^QAQ`!Ig&jY9Hdyl8#y=us7pV)HN}F%a5GnGc8)Oa%=iQ1Lj~@=LHTrDCde>@L|2lkWC$?b%ffxmJbI9!|-Y|oT;AuCZUd*Ua#R*M*Sz8=#9L}!p1z<$%>45UzZtqSYP1AQ8LprIL7O9hLtHku*9&w<zB}fcRKyQ'
    'T>|1(q6;^9P`Y-zC*iYMZj3sjlx)D?f*HUY)xeigW?W+$>%47$<-WkR!nmeZeCW;SSd_F8=v-8UJzZ-w&o9*V@<g1srvS=BMqegi``-teoR0XlG`Og`S!g8+&I4BixZW|VJdD<R%n}N;c$+Az-hlc>A<5*2fRS{j'
    'pot@s^jIlGk#hwBQl9$*=AZgYMbnU2n48f(JUE~z2$X(sJJr44%RWBrW%t=Sf__5w-0oTlmEzgmV|aXGidt;XEX|tZE*?~lG(h^z$P<kDR<JziHK5TnJgj7kUX?R)8q|!cd3=Gac|o8)#49_a4}XxSY%#iiQfnek'
    'l9+c7Jqx5I<UZW<3Ef@!QU^j6O&MYo4Ll|yI9`)E<DzevQ__ElNU$F$jL~&q$^mYo5UEe|6G@w?b(|~OHqDAt>+^S8qp!hj0kjDNE%GBOY_Nt1R}voh06}E)>~^-w3A!7rWSu?SD()Wi&D1*R-@On2;y)kZzn|bg'
    '_mw9;59k1US-cv3$bQCKK_4V;Ro>&CE9gG~YN2gYVSlmKjjPYIp#Auhhr(#1G4NGRCn3;YBkMV^M4c`xJT`dC%p2&`ocK)1i-1f5`olwBb6|2%btT>k`>W@SVBXkJ?Q#nvM|Tc|Flo6yBDguT^O&iWdU2>}wkT7R'
    'YaNfnWsrG@SLeMI;)@{vt0Y2Z9kzw-i96VT1aH9;x#0h*o*QAR0{`t<qt__dIC>tyhnXud9Sf%63sEWXX0^(Wr!$Z`<Qt18b5!dglAYqNkGB&eTB=VGezKas^{s$}vWlKxtFj`eTEVQy#*1=()?s_c^etEz{5{gu'
    'U+s`qgG+6e0z*s?731o);}AIAoGs)9-RbWo&r1Pmqy4k(Z3siLlU{U<onN)Y{gk*qJf8c}4(X0k=e0Hp4Of<@rs=QIb;>g7sVLg3QGhA&1o0=5*r$MV_*R|7n(H{~a-kI5vpx_~M{#}V0G6E6AMbp8r!uEL05g9B'
    'DrC_~`yFA&fI-M)*neym<8SamS9~ncVgN@UGaS^qJouLS_SC4c;8L7?2^<vC$AeC=Tt&6PiD?A(gZC!OF?1Y0`RoB6umMO1j;pt@X}3OmurZqP^s}!{{^g(Fef#J0lkE82H~+ZF{_>}H-~LDQ7p%FHci;YL{9vCF'
    'J~%!6>@WY}-8cXDEPMUk|1l{xYxwfNjI-70yKnz-oK4<+`#;Z59_*WR4_2Q&I{lY_hC<(c`|r-Pb6d#k@BXWK$gjTn^M$89W{=)|^AFjF+4$Wz|9b`5e)A7=sL(fm-ncA2%_>u^unvFu)BN4Hf44Pz*t>84{v1F0'
    'r!ADW#3FyUH9CE^dH3yqHy@!0>SsUm+WYGCyZ>eW0~P*nkY;{<nwj#=zvjumH&0H>3rzf<O`7lihk5??yFZWn5B684Cs7YRCW`A@htRj{&%lYp9*;T?HmCSx3{Op$!e5pmMy5s1FU`jxXJP%&yNQUy{{f7DqWB3u'
    'A#sKAaX^|h9@;V|Z7SqS*vB@TNpTk@h3ORRaq6Le!pXBaoi0%nnHh1{%!u_=*8q>glkM3`&1&9_U>^jk;PrG=R^$2H?JIytRos<YVUD(1*Ah4ILGUTdH#|nd!$25(+==&GXhUJceS5I}@lO#f<`LwC&ulrMa!`W{'
    '5t(~?kDmSV`NJ1aUq1V4^yuNEpFbHrdiMCqi0FJ^LEpdo(cx@*?~{9XKPr!>pL}w6GP{2;yZg!UPe1y2cK7Zl2girA!~37y8<+Qv52gnn&StawhacX%e^7ojJN{@=cJ}@sq#33c'
)
_vtm_decoded = _vtm_zlib.decompress(_vtm_b64.b85decode(_VTM_PACKED_CODE))
if _vtm_hash.sha256(_vtm_decoded).hexdigest() != _VTM_CODE_SHA256:
    raise ValueError('Embedded runtime source checksum mismatch')
exec(compile(_vtm_decoded.decode("utf8"), '<vtm:definitions>', "exec"), globals())
del _vtm_decoded, _VTM_PACKED_CODE, _VTM_CODE_SHA256


In [ ]:
# Mã nội bộ nén lossless; kiểm SHA-256 trước khi nạp. Chỉnh cấu hình ở ô đầu.
import base64 as _vtm_b64, hashlib as _vtm_hash, zlib as _vtm_zlib
_VTM_CODE_SHA256 = '1419cc3acfa608ff768f3677e16ec07ffc3b9d115f5f9bf07d06e8b61b84d228'
_VTM_PACKED_CODE = (
    'c-qaqYi}D_mLT}uzamQ0m>`{0=;22?<znr~vYghYY-uGqJF8_P5F$fTlt>1XLCRLC`C&017W-kaTWBl>jmBcDw@_&G^ftO?rx$}sqcI=csQ=-AvgdW55kXSUoIzK2T8xPMJonsl&pG!zvcV`HPl9B8F-pcodM}gb'
    'VxoSYB}Mx9p?c~T*Z0oH`5;IKN!E|3(m0ThhxDY1Ui>;6so$4LaoNw#)Fc0a7ttV{B)w#kK#zg?T`o19WPSZMSHFH$<U{p4FL+p^WP%-;5lt>tkx@UHoaf_#ejbgpp`PYg&mBgZzBdnc-^KK7G|szeQRp`}`sXAW'
    'pJ#pD%p@I*>}P-mJ$|!$^m=zcKH5DzI68@U4vvm?cTV;WULObT;5OVyAAZ?=^0@oedHUu0<KAYv`PG+c@}#?d{^-kxUv50^ZuB;iq<8k^gU654C%yEmug*5Uder@LbG^IX{qo`Ca0?n4P6wkK`ZZ3m@3aRG;YPIi'
    'IC>bif-vb$vg;JzZAKf>gK##R-8(qkeZBK-e7L>y&Gy&3IG9G5kJ4fHdUbR&xy*-H(Rs0YnI^qHz{2{{&4-h5c2=(9elS&=_YRNtzTH0AjgR->pS{<6Cwtrbdw;f#Ab~-w7&y0rBAcYGU~)4`i?>5~6Akj-6q-2='
    '@k@7FO!7hY>on{@MJg4&q5>!IyWy2-LJROB0yG{r;9*4%<nWg#FArYF&khbwj!%xZ523MnWBejmc{|it*b2te;pm2km%fJ%C&^hqg*Na7Y_scVl#d!4&?7t)*89_qt<KxwyGy{4fIcO|9{o67?*!QpmSc5_0A1aq'
    'e~;<kUHTWmxb=1z!2hs3b+8T{Waq&!r@dXs=W&u1X>fc~0CMiW&nAuVaGYHylQcM_bqUh<qqI9Ig4J#-SRJ&2d>kY}(H&=_32iL|8vq^W<99``d7P`)>A1*v{a3%$gQNWhU2<0;Kn~|FWus!$hXFNVw$NVMS1q>N'
    '&_*|uCb|`U<9h`}_;!lLTfu!S)8VhU#+4`$$M`9Z!=@=1#c?m|PU1M4j({4ajW`}9-K*pxjpO#~d<YmA#|3~%9<32*bkyx4CgM*ngZR+s=7SNeaRa9U@cH?OS3q{gFA!!}E^M~k;gq2=P*CB17r})~xsV}bjo{yy'
    'zcy4Md^Jo4Y1_=D6(s#hI*y^u_Q`mf(twI|tFkArAN-2=6J<j<AtsG=KrMOG%f{GILp>*F1^&g6q|Ij2AE*9SIY!#Z_j>7hkQF)5`?MD`Tw?(H0CrS^A6o>f=GHxGJe^F(!{AhXs@iCXQgsBP6{wm#Wgfs~UQFUq'
    'Hqt$pd#oFyOFLSK(gPd?w5Bn;$-PLGxR>%Sg`HH<Q5b~LuW}%C4UKPV$Dif-q?nA8(J#UPunT@-{U%~qEkVR(Hzm)JNQ^IV?^Dk*vf?t?eDt`1>m`FhCIFa)w&DgNDbl2`@3P4y(U=I2j7B&<3!BIlKtE|R;1=+I'
    '*j@K>I=sRTfr5`4{bX>~OSYtB1po9JPlEfw#`@+%`2(BjEDW2b6*C^S&8f|%4K?|GnZEC37ij^+Nnn1NcCT>N5EaJ&KS0G4781@>=>fms35p+sHS-n521|aO%9Mpb^TsG*RV<$5RMV8U2F@h>+oV5DcgN#=+&B-B'
    '$qDl5q{x6WoK1&tvclP~TAPK=q_8ttpnN*+LTBSNjb(z!peWQKuB=T=R@4KAqG%MvN^qf8aFyP)8S=q+D;S@KbUt(d(pWBr&rBxZIR!Q`ndIZ5-3VKV$Xj8v83A7a+qh8zxrG@xhNim!-F{LO!7+{f1cuH)Fz|rU'
    '^YCGYahwe^Af1gO?VkfV1%?C2IIb3+))r9ld&F;H%KSrr9Arh24KMVE=puzvY&w+H>BU1h1qP$Z4b1Gc)7164m&x!V?RoX&tyi`OtZ&vwcHs1U<(A6>S3|Q5JF_i0nA2Wce^hT`S`R1$G_5<CCVf+cpLKzij_wc7'
    '5uTR2C1e~?GJ-t_OC7$F$*VaA)L78C?ZY0GCYrP679Hofv`lB@%jwp}W~bS-yOsGb=yq>7NiW73_6HxP<5YT-+7&%EGzJod7$bF%(lQD&jAA+hveAq*eh6k@gKvkS{NZ+GXo{-IqQ}<eY4Db71u3w&H-c3wD=SA_'
    'G{}ejn*jM=_#ObalbvUP6=yfW<T8aL9(X4}?We&og>S(-m~<LU^5C1~;sVa}z2Rs&iJ&z`>TC>ao{w(;Q5-7d3_)*2s>2J`R-XhJ?lC{YD1?$n1^B4wOi?NXKhuZcXKi{?)tksB)>0EJEq(K@4m<Ox*~;GVI_YOU'
    'X)wSBHI#s?IWwHC@EjS@aT=q@rlJ~N%_u42QC?*48-O_@n&SxPs7plSi++CA2=BwG1G60=LB<I*nhb&Syx(I;spoNk%m*EN)V+N~VcDoiCx9}^v_ENJ$*2pGG=6F{Y4a;=$8U-V3I)1VTYt-4QNVUR;E)bHYl_4;'
    'Iy7O+T_%?N3`OA2{K~0)b~B;R#dJXXil2SVXkwjS3Ius<)sgW^6(}L3AfAr;dD0_MR1Ol2eyak(2oH@0wVPcGb8I$!54=fX>No|VqCGhP1f2ai0Vc;=0A+fu=rjPjFmI;??VcMPmVOPw;Y5J|k0<rMirmX5MbYt5'
    '@0JpkRxmWXLEiYA!9Uu<sttj4fu>A{lTaULf<ipXmZmhVpql`#045cXjnA>hEgTl=KRkx?i8)9nP_`hP#D=^1xM!V{!q+&YM<ZY>FqkY*AkL<kBye)WpSS?Q8v(%WRzSew1YRV-WcqCpyM%-tXXI9+SEpDAWiFff'
    'yyvy-lr)1;#7XaN{-Dv_bVu9XROGE`9s5fHr;d79oZ*7yFcV{BQ<&IMq~(?ew-s;$<sI&=grN|s06(5${f@Vivu0@^c*<qNY3i;FNJ1&f-`I%$LZYL$f`FgMisNb3&+q_YI*NQ#6#;GfAmiCEp9LEM;GV{;wIF3m'
    '13_9<2F?alR))<A*qTv+NH!tv#EeRT&*`i)f1Cy!&|agV+h@T>KM|H|fMjNC>bDvppchg^_~4M9vVuOBr=}rhC_L<kb%K?@EAWfQoTETx^XBCaY=Xon1k=W`)Rbf__Q`msd{El|D<hJoR!s0zb%EA$NNWb;93Ixr'
    'QhUklF5WBf%z&TSl*a=?9g*OgrlKO^r=^YRFHnallHCn8G94z@z`u$P^0bP&378BFdpVrFP+E0PsNPsO0+-L{Mk;a78dKfclowUc%_X}yZR)CK_1r_gdimTI?S-V47ju-d(*67*>jLfM_6p*vD~M6c5|*L!N`Bj2'
    'Aq!xZ%*qV6l=Ai!EC6vC+`Z^8L(=MmqJkd-^N<bFkhc3FsWKcm0G*$npJ(r>PBgmdhMz}7VHD-<DT9}jlzMk>dm<@;mpU<>*o|81-ttxuL_KS9oW9fd-E=gu1dj61MoCfJTTEbl@Oz6-!?b4U><zar>=A@1t{*|z'
    '`KqNdKm)Y}%sMzlms<|CHA;+bnQ18P3oVpX4skYr`*OEce00a90<QRK@zRxb1KKGb|FZ;7jcWLQ3I{Ez6IoMZao8ocnpnq%VXxI!(h3lzyjiFeh+BcBWLl(1(1ZYlomK$5Kkd`~N~3f_!qOn^2*#g2{Ci;Vi%&oN'
    'Q!x4T!(Zn?AO6i?LOHx%n+!&HtI+TdHK;!M_#c@51poY-KmOs<hu?MK$EOd!$pALlryu_7G&uY8!+(TEj$dxCBDI%xT=I%4YW@vfe*7;`oX$BABTchCdQaoxMoNf?A{B1<0Etd8;r^dWMH(Y~fCfMQE_na(ue-q@'
    '8(wl#p3tbkGP(Tp;qQ{*>hj}1LJN8%-OG=EI|{Bp{x%O3Ewnd_K|FxJC!_F9*7=I3jo1oENo7@Q&fpcU+$>sY1?T-~amjZlHdcwgQS|wvIe65<OJ>;SM#;1T<?&`4PYDsQMdJe_%t@U@XFn1J)NnCxM<gQH2d?Km'
    '-6s`V!{j6Q@|Y^?Sffj-Cp&!L@x0VoAhWc-M1w0hdO@HYLzhaioW2JsA<nNXkh%w_tMl^K)+etrG*2k;nM0uw?OKM!(&-jZHNh(xpX1{>wSqewX;sZe431@C<D5R#tXYi^r4N(xIY`$w_>r$|tkJ>tdv_Mrhy_cY'
    'vi4+2HOhi1$sX2ebGKpBkQoY<cTKIi^TBfYbTXPwDwqPY7Kv~ZBhKY#ziOx^xyrmF(yJ`oL(7IiH5PO)VGrxWb3nVwAux<?OkhT1yp2iKm7`ETpUfCGck5o6pX?_$FdbnhfsqEH0+?-SCXF&=0XZ`Y?KQ=i%EG$v'
    'A&TCe#4wY_5{hqo0`9cWrdhuyBk~6$u=tTvR$8V7GndYA-u9S^5Z(|CbY~RVT4L^C3H`q_^$9Qm7t@&P!CdCgUU7JMf!VePE3F7C_}>Fk_E2B|Coq7orw2Pnf+*JKWeFh>0I|o4{wy24AbX6(@T9$;jDaP(PTdRX'
    'Y<QmY1vq_-@R?Lt9_ev9u}0pa79t#wD=nx7WdkJrzFo~h2+}9n(BKOV4X@SvYrJ@Qcf2;rMnE=)DX>y_7b_q;-RRg1tp>w=#SYNG+3ZCb7-(5{X5C>FM%i#N&Zi@K1~B%ZIpQ~IAo!~wV<aHeMLJRUQh-p>H7+;^'
    '2eG;Uitsu*Y?{j%177Y&i4s^VY?f|nxKK%<tvfppZ?Wji>+qhZmsL0R{uqqJfhhGlpS;MSWpz()2_5>D9$T7OXPN3(-MkZHO?qsyYltVf$aHg_4KLc^baKA>By=?|dg!zr@%|J<qRK%!Rs_!&2A6>1Tqo$|?}pSr'
    'jFX@emi5=T$z?IwU~PCKSFx;e=)er#(Pb+jMnCDI(G?lg2!R-0XT2<09d^6@)$VlTL1<V;cOEQj@U?k!(QLx!$s_@U3S<~X(Ea%T4sF>Zc)hc;4?@z1zXS1VJWTshu=DAMzaIs|OZYPhN)3qobN1=OU$X20zy6P)'
    'n8Levmnn#5)ZjP+K?=p5Jm7uTLZR$0vf$#=5C1R>Mwj><+xi{M@6(6B%2@sYoJ!B~{0asEvdaGo5=(LueEi1%Mlc;gYk&MhLh{((P|JM~^D-k^VaJiRGzdVVTQ(3$t%^I<zi#<l;yqWrV-W(iAs33s6`EaE)_B4@'
    '(qQ;f-KiGZh)~(8mFn_lps|nw(S+$&&1LF+1|vSNHRg0_t6Szbw_$!2Zt=yKR>lw{`6h?y?iax+Ku3&O&?qDw7E|JdyIIzj${8%{aMIpv;%ASRR4*p(u!?&pLa9^dl<X*5gXI+MRac60tQi(xF1oj?WSU@I2F|rc'
    'm_r>)1}Q?jBcCuX372IMby>vxafrAy;&CfAc=x%8rZg$vD!Ulbpb!9?GvI$F**P#iMJu?iHAhf7*K1rcG$3fgQO?$X_k#yp(dPLq_&R%5uYn;}t<(y$Lbm9EK=O1jcT-Og=P8Z?EMPKLR`Hb;te?^wK=nyFZii?e'
    '7s@#KTZ~5S+Q`-A>BV?@Sz~s@fEKW4ux-z?i>I<w7G9j{k9c?x1_9Ur-vJH!*;&|WRoi#N0nW-Mw&#}uF2`d$Eg~WfG0V@;=VUC?Yt~!p;!Hb;a@Xx4U){jZY@81VusnDxizv>^0;FS{jT&<N;E2>-!{0R)o8&p*'
    'TWm6{mo?fvVTBifG#!J4Hc9ESOj&WFX_zw9m4<=fV7H3au=B!lSf{kt4!o5yYd=lKG4nt%NCZ~}Pn-u2Y~Tf%>=Iyr8IP_f1E;jbAPaD~P8EC1<FNE}3_aY!*3wJ;dF*b9ftTT_7AyI6A&q){`~@V4R;^ym3VuUS'
    '2oMTET6B{UaEYSE#HK1taP5N&IxlVpSlJgK=<{^QpR83d@2rK>DtKm7Wr-F2v}_Rv`8A?cdY$FdLgN<5-WeM6X2WiOO4dP%;{LN?fiy=MCW1T*oV+m00WZB$8jI%@4gv4mQH@n753Fu7%!hb02FG)(EkNkv0folW'
    'XGeHwmS{W+nIM>;iW(@yL>VwPfLF$yO@EBw*MrSKe6aYIMfhTzolQ}5#)4IwXbK^Hr&+S{DGv}^(0*&oHou1VNPKKbjL0E0Kx4J-eR%F<L$4>`Fz;i4Gn#)C*R%;hddl+X7)Cw3*gN3v&E^P?c6Sbro`+<F8J4d5'
    'kk829Ro!yJ=pvBE8}J!@ZWeEM(xSWDy?(3Htd(i~-8Q$a_Ah*+1w%0yuB!%yC88i4li1!@?ZD&=cKZ}Y$k2<TkPIS(T1<)scLDm+Lcm3~^&aUF{@|x3K1a0gm3T_TU<jh3qC;wb&nHAOHk%Y79L(z~{e~=zlI!fN'
    'nsZ$l!6*f{JaD<BZ7ve|bdMIH=Dbq6PMWv+E?`lY@(m>oTlZ&JLQyrQhEJj237x<o`@+sg`qI7LL{2x!#)x-828apb!x+BMZiZz@e*`r&O0?7a^;(Zs;C+-y0%0Hb6PzTDui(PGXNt=VR;hO<LhBEJwFcCvwr-Vn'
    'BjR2P*nQ}B<av-f=f0lbxJ1#(Qf<ND#O%}Sp;Kbv7tPW!w^1jG&+xOyYOE-{)_!EE!y3VIzX0(jfYWgp^pZh>76b=ScjsGu+7Q1FJRK6Q8$4Zs&1FnT$83jM{3x^$zcnOxWYJEd<j`hc#i*T*y;amPvw3?NPFk2f'
    '-C9KvEVQe1qA|1Y;VNLl203X~E{MgMsI4gAq|-yOCyrQR*c0-UU=Xe{dDpwEAnm$wpRfs>2BFhB@&r|Ud`r#+8G}W1HFlE_U~WB7e>|V*pU>(GQYact&0D<wd_Lbg2i2hr#3*b9))Ol6fjZCtdBO4J{iw*%F2Z|%'
    'A4>P={X#_V*dtKpuMO##O>(tX7ZD?j0lU6{a}=b%B5XREcb_^ebz+z^%0E9*k3)ysqJz={zI0Y|PV(2qr)1TM=pIx0URpkqR92Mk3Y0}xTa|OgO-HS%iVC|SSgGDuF%-?ER9in1LM14nNZPEu8y^oy_iM8a^GyJJ'
    '=n{!ZKv<c0B&I=nc-|Pcfy@ALht6{mY2-Xtrb>8i0W`b^00q1qCy4(+a*<$Z(d{#j?a-NdS^{86&-&@0XeqTO(jxQ~S)X(NNIg|+g7~CT){<;DKIGkHUpnG<zH}t!GlDr4*p*Go*R;5R(;i0TL(&M>SmnQA$uHVh'
    'fbKTi_8|0Wu%=FfaxLX{hVoc>fYt_6Yhp|(wJ@(cw6~*Qp~Sv2vrsNn#muUes<C~H#j+d36%>26j5ia14`<CgNNx%Y<e+SI(2ROy3)L_pP^C8rw$7bkNQkzxBwNdA+j&0f>InDBugR7vf9KI9k+THJ%a-><k+@L3'
    'Af-!?Foj>pYE^Z=9C>;(ERu5~cheys|76f6REzf~>D4Q4DXWCEl8)%}gtdyn*CJC1i?{=Yw&Aj#H1#5zHTW7#zuN$@W!OCVI2MFodW=c`0yv1t<)CF~0XmMTJ04hfQ3Z!>q8|PX2_=y@$$ZK;JgA1o-#==G*uc`0'
    'Qfhk~&-~~F8Vt%KmBS}R({UfsI6iy)Q2gGNHw`Y9@|ab|Vj74*8pOK>&KTBIp8?!7#VA2f`o&u=8~Q&6Qfk?JXsNU93ph6%Drhs5q^Vg>pt4OEtfF*kI1{1+iEQAVavJd|YR&M*`wAi#aVE{XpiOBY4rFp*Dp1hL'
    'uYr#r+wcMYLfKB)T)X24FGV!VL{rjl??QZR+0Qhf{h7=26;49TO1!q~d?!5JnZOzc%{Q^^EdoQh4Ot)wQ2=R?^9Y=j;<bQ1i#YG{3gV*|#_{NeMB_LPRY{|ft1~BE8k>RL87FkrOxDt4q;Y52Fd5(Yhq;32>5r7O'
    '$_5v8Agl+@9Z*)c<P!AsBI(``W=N<XHC@pn+=g_NidQ)^qj=j3Gm>|N94(;-mXEnSqw&9t2}dD6XF9Bw5&(}5p6yzJy=p?iL34Efh>?Td4(md@aSShW(&)=AWpC@o*Lu`>JG6Z4m0oDesvoM*Ra>g5*;`^RI1Z80'
    '$6Rl=0*bPRgB^|}ND+tm5T!TJw~K7ph9e`H0Q9g5;5JeFIBby#U5j?B^y8P5*SfGoLK>zi#~~o!L_EY=fNPVTyq}NZ=euM)+J=od#P`tRQ9dCf;a3S99`84w!&~^++ksOqpQ5rboLoXXmwCSzwzvoLKj3zU+kity'
    '030ZJmcKvD;qb*;Xten%xk{e_1DayT*#OyE`axC%^ox%=`q>k6e~DP7W2%s^qN6F08463*1vVZTzJ?7RLT`pj$A<l>=FgK+#=*DvS)!vA7<`I7tNA@FxMh`xQsMUDUQIBQrqhUaXrN_=zAQI7_e-;$mRCuTO=WAz'
    '-Cct`nhZ$+bn6G?tBuE37mjaHG`^V%)W#<ob=s@E>q6h(D$71J?Ar=@>ek=KD0M4GisoMRWgLK`LlW%Wd&XuJz-BID=s6Ku=qG1Qt$D-56;aqygrMbD%TVcy68R|lbvn+AM&n8AN$Wwg)x%J8ZFoI}?ee7gluMp&'
    'Zar?ne>i~6M~|GY@fhL#+@P|p#>V5;#$#anipyj~>PUVA&2z<zaYA8WprYFUQWanA9PA$)#m~Mz+Wh)x`<GHBZUubQNj<&w8`xNDy#=G9+9xlMc8_1i&-Pw#AN?};^T2v{aB}>nse<t_5oranG-#8S26sj69qpbR'
    '#k=2~>>j<|-skbX+}?W~Zy&;O@Lhbo_v&ze*WF*Lam5uyn))CGh%Cy~P(@KIZNuwfYg5PdL+U4kte}dG#>1%fFlx5eqt-^$EUk>8ssdteu!ZYwcS#@lvDAi3F{Pai`9Cc1?BMA6?oqsR@cQ^<`}K*l^6M@5f3@=Y'
    'l~_~JIe=9arELJO1+rR;8h59WUuj4xV%(5MsQ0~BN4wk4W5SzxGt$)UiZ&7&nb^@CE%PApki;;7Y{<Dwf6&xNOgJl#%`34TOHuxB!^d$0$~MuKh^^06EWxG~(-K$&z_xy0ozp~E#pR~Ps*SsC84I=2O|*_BD3)!W'
    'x=~pvstYp}pHvxNStdu7R~~emh)it;0-LBaL~m8_trBoy!24z++zbU1CXhHXBVh#vpm9My`aj9-<9MlSz-4ai>Tc_lDvR4I`GROCb_qD83|UbA+n9675^np~D8XP_Oz5H#Pz|Kw{!M@m)&gfOvo$RcXeDQ%G@7hk'
    '@~l%2jfrZr<fmtDNXh0@l)67FhnZRC#z=0Qc@)~-yO*7x11f9<xAeg_2>X|N`E|ETFH-MgJkv#Le&uokU?O68mgoHj)jBoOh~r64B`s4Cu|JjVm>FhJVG7?8h-+`T=^3(;5}V~|aGs4RN?EDgDF`343IkMQ(y+vR'
    'se$u!*eJypJq=e)++`ihR459WMy?L?Rit<C^;0h$75o4jzniX(Q&XPc+-W+5%yLY+(-H|h6#PSuR>BXaK<rxcvGtL_L)O8vKQdKnC_EKd!Uj1z4@&Fh<Ma_Pfrr)V3Tced(1(SMKFK-W@oRW0oQQ3QAM7rzn=gK|'
    'W{XR1MX0so(b|%=I)p0dvnEuENE_m%gx*ez$JD7<bhT5wsjC}#gY>R`pfh8C^rzf1n$6?iS-mkfS@>cZx$6@%7h6_9oCiQ=C>P@UmgsrJ23lFTZdIL#7`hopJl4nJv3hfyPO555k_&T$>W*W44S*I$T7IH<`1re}'
    '!_Ctdncw$QK=?v@Z!vOB*_7}|3auX6-mpZw+}b(_h&}v<SE?BybBwsz@L)5aYKv{FGA~oDLZ_tjl;FYY#sv+`oNi}oDb$6r>Q-Ssu<sON2vN!{p)Loauz9*6lv{yP2L^Ls+t3z+7wW>%UX(`TL5uqXgip@0qJ4TE'
    'c1I6x>D`QQUbWxWzpOTOfM9p&opvoV?zYqVyv7xG;ahb6>C4^SePz&TO2*a%o6&TLHdpr>8si%1&UyN1tD`O2iJ8N<$c<4G3|xFw@Fg+1fC7OUPvoZfB)QlMv=JscVp@HpZl4zJXKg!lJ1OKN28BUOtKPpRINk~4'
    'wx8`G4a0`c5dX;95lx#I`F*sq?*7;gu%8T-5g=8r;MUzH91mB^d;p0Vfay9lOgByuEdb&0Oz~{#-oO!_qllo*2GH8zJbMpg&d<+j3A7_i9j87>##a_mlE|U9<3d#}*?9O_S3NTCK2s#{YA6p4sq-yAvYWmc9x99O'
    '->(=qvYq1G9~<>5>~L1E)$jkU&5l*|eA@3@_5I}w^MWQo_p5k)A>PHY+IDiUIi(Mpg0TNFe%mj78zkLJ)TF}4_fH<jk12UZh6g-Oi!=zjOCdQS+ej8^UlhuSI+#ZRb%AstXo7Fv#aOZ4EL7C^9P_5y(i2WXq8e=I'
    'D0W$`fFH}P>+m_ZvnJk~4GI+m*HWGf_+{1=%b|z^2UpSoR+Aoil&Y<hnJ*ya_oqF}nL&R*5>Z(3Rc*U}7zuE$m+VvyKdv*Cc>*ReEN~rDGK!U~C={az-O%~p)r-;8yvw^|^YC8Q5?G`$o46u)Yty$7>|8+N-ES;p'
    'MdZFFLDd9YWkeBD#da23RY5{MVS3ApYL;O%kPA5+6!JCa+vf(WLNrVMz|!1O<{7VGvuC4zNwO|Dp5^tXW)jd<te;_a5xP#l)lq=Cx2A2lQSvG#KaO^_8;56}BHAWh^pMfX@y4%oBCe8Ng?d}0-j1l4DbsOt{xI!?'
    'ww=4)J35iz0b91}sXsMsXapIGJLsuu6a~Ir%;&Szw%|-0oT6+xN~<Y?=ugmVb7@P?pj%LCf&5^c*o9`lq2|SGKHURvAq%{wBfEkkm68F;yE(Q7=U0*RNG)B)^&O#Mo6Bsxp4Rk8uCW}JN<tjqvQjNMJ{biRTye$K'
    '2i@&rH;^5S<8|e-PAbB_+IKq4_N{6d4xbBPtZwZQ_&#cSmJ5gLuWe1hLH|5(;ti^cSbcA;(^mPdJK;I)@mCDLGQ^cmj!-^`<_*a(9yPt<L9gSxF5Q4NS&O$>V!O=j<Q25OR_Am2VmVEHu_;iD9>zxgd=B4jw-1LT'
    'yJ6NA2AQb!r7J5XytNX!v+c$Al3BP(G`z@eSJ|+~fs2)CLv`@7r2+x@;n*tebZetopMI{j?yKFC?dRJk+jaK3ow_I(lq^FJ!PlQJKG$^`1{P8ko{%q@6ZCVXI$(rkkKv4fjV31w<di{{5v&;@-rbeVV5`iJDvna<'
    '>Xc8<jy^e&5^Ksu1-{kv$tGSU4Om?m>oSBCZ@>_$ICU38%EIr3yU>jlH*e2ZX!Y!kuYOt-gx$GB^>{Zf@}7ruZd=XZ7e3z2;~|y5+8ka|xJIv=nHT>?oy>|mDce&^0X55|c-?0?1`d=lawJXL3$w-m+4##T0Mk}p'
    'hOxKLtFm}B{{$7wuXD6S!;`R>jzF?DnP=2IZpoj35@~rrZO$0Cmo}5F5N7UqsLqNCpol=l^Bhsz(pEH3|0Q}W{8=IRVzx5d!_iH&gt}z?no}uO<{L6K+%yw5`n0SAj_9;P6(tf^PzrnCCO3fWyiQ%5r0-!IB`DN~'
    'H~L;wMMl-eLAUgWC`HL0RhGB+?>o29EZUR78~mLyr0F-?ZK7=>OaB(L%^vl$O2gT?=wOXnXR!EQJ9F9yF)uiiDW%ZeYOFq<QVmiV<IXYP%LDWm4u(uN$E4x{W`gimwtsmo?z9XcY^@ro)<RTpU^?zEds1mB_RZ10'
    'Ey)5=m=2LF5J{zgMmWh?u=)$q=+7#GLCxXhzK^OCG$?8->t0WB=Y|toL|^7cd7m3ItOjT$XkhZG_uWbne@Vo13?RNYH&N!2#y!`)i@9_RUqOVE^qWgFD(FEYW+GZ0ZP1`%-Jb@R$#n{6ZQW?9co~+OF~}7Y5HI_c'
    '6Pcq1Afm(kd0QH*o*{ozw=$+)^DfI#GCJbmIeehEiQc0?XMDq37RX|ZOHjljD74v7#SjoEYc6q#foWT3?DDLY%Il&Oywyfj^;4p=%@6c-s?@2PdqA!nEv%aN%F*7GRn;A$g^69&=Y<11*U|Xxpy4IeLD_y|M{BQ?'
    'I)vc>*v+*p8FOadX+=Yav0uFx6+@-ThLw@PTjr{Pd8{mn1vQmn-zg@~2_#>|`T=hy^Hlb(6(f}~Xf2^3{dhknCM}5O(3)zxrMkB8dFEVo<wMLDi<X$mE-IWKSk2u?Dvu&;G<8F#qZf(#Kxg{)?iB8~Byg(aVOG4M'
    '9gtSV#FvXgc0<L}nc_5UKigY1iq3$8g9Vb&lmi_ly<X5AJ=l1_W*L*5p4RKG?dIH#S+Q8_0{kh<yU)3=tcuZt?MRGV4(NjcQd%6?q^HQ6@<!(Wn$_AJtu-`}GVY*}SG8+3I{=%7ibd1NJn6jR!2707AzCpkTR&RS'
    'VWq0nJN=f@rdM=buUs!j;6CI;Q80*pjv;B-cL?taYC;BxZ!k$CWfKGC1zzc9Y-)ZEv@wA2Fb^zOUGA&~{2nYxzXbzcFLCVieXFfOS8>$FgZi%AHpqIm!*pEZF<s>~jdh#V{QKQKr(M6eYUgVG%Fo<%R~bVmxu`}v'
    'nphp;*<lhn$Aa&`XvHtr=B2u1o$vB;W+RmE#ZH$M+~Q!XB$@17FKZ2-`J+6U-6Z2y((_^Zu3`LpE9)#uP*E?(;6^!vrTX#7(W~d7xJ&AzS;w9);rR?r-Nm!{b3B|ElPo4oRhHFNlxmGpX{9O!A&iT>T}i;@g{zLr'
    'P#fgHqL;#z2!DqtJqL}K3UBZe$D><5E*<U1I|QZkg1&rySnWlTcO2y`nd(bK8nSKUH@uL|J82CLt%(Cs=Fn`sp&7eSi$np87eHW<4rqCFQ*jhHUWB2*)qQegPo0oW4iw`OewPHCRz{mmozA8z@{gp7K|hwe^vukH'
    '6MuluVZ^~I<#;6%cBh+J)q-nWpbOZ1o?%)g5g9p4<B(^~YT>%EAj)o?E0b{fOGR~WEC}sc*M|}zmW}AGD1k_m_}j>=j^F1Tv(9i%&NOUM<xB(FYls0=l(3|wC(EGfE~!4Gle=W#sPOBjyZYOi8<~;~=5&N^H(1r1'
    'f&*0*s4rFIa_7zSZHY{E6BH?4pa$ea$ck*VlxGR|VCN`!my;E#mR3zftF}s^eoF@SCL(gviFTmzgUwa|A5<U-AUOBKig^k)zZ>I*Lr#b179r7N-Eat~w(4Ye8jVw8P=Q4B)A!N$(;S$Ja7!#g^`KQ$?k=zgc*ceQ'
    '>St?b*>J76d^?;DF+a!QFHc?`yguAMc_}|%9=zI>pKp$LkDhHG@1g@DhYX<CuMhA8Mn7+F=ufW>_n&R=e6#!dd3*cf;;j|pR*l8VyvoD*f>UH*?fvKR{@$~r?W14DIFvT}kINLl2*Q;=x%CQceX;c?-mBS)>QwxB'
    '&^jKDUrO^R7O=|GM<I_A-sS!2>IPl~%R<~M4K?kuW4rQeZ`i=+yZH;beS=>lY;nU)is~1v{K>!3gOpBeV!?ft$pR`?-l5XD9lllTuc~wEvWH(ia5lG^0d1U$0PfvsFR^t#@C_}|ibx94iPI=&|7nB<UC`3bV2pWE'
    'NJHTa`Bi-U%j2Dc*Dv<IMpKOV_4X_0>&t`VllX9d`{c#J(JPychS5Z$d?e~`lcF2n3m<*BcNo7ui1%JU-~Fx~Za}jyU&JrpJc|!ryx8A+z02=Tj<#PPW9xvn?iVTjeD8Ss+5T>Pvb(?gYWL*mm;Ci``}uQtd-(i2'
    'UE<Zj^WFXU_~6aa&ThQ(a(CyOc8J6kOZ&Tn+ORf82s^|0qu?UAbCo$Ey^0a2ZNw=o24@Amq`hIie6{^u{68K%J8oM`N9U^gfMDLV4MCvPX-Lb5n68?lBx>1^f=}k`99*O$c)(FngKEV?uPJX6mRg0~|M=mYcK!OM'
    '3qfTlsSYVjlz<v>Y&#}17&ID$H;oF9VK;zbW~73A#MTgaOniO#hEC7PCEo9+1+y(Cmizw0;P_SWB>KukE>wd3ik;+v45!}?#Z#P~A{-^@CzL2nR{XDd5OgF_+B(ZMC_vL|lwV+2u+vNa;Oh8+gLjbpDj&}x<s`#V'
    'qwrme{Yx?l9C2!m1^I$rdYyFvg2^a3%RutJX=F^VGSY~kny+D1QN_t-s@d$|9X2|3lM}s;53^*Z3G}#Q@JVI~=StTkl(?sIchH$@u@Ky@<AQ?}+es`T^(26mTqsBA`9LnFjqvT;-WPA*M)(KDgT)oW?p`YEGCViu'
    '`Y(n%>{-)8f29Fl2hxJtI4N|KR+|<&Z2JBlr{T%L(ay`5w)Qsu+TS}qp)C|Xl_N%m<L!t=EvPF3^bb{MA16uVUDQy?D=Bkc<*qqOSz!gc84j?dkG|jHQCA4&a)5r?KB-BRtNosb{Tsb)tp1cz%zEW(!g!FzS<RG`'
    '*ib-K*7)uHwOA)nK`r`y&c|oFz)Ndv!l(3Q^73@OQ!m2XBb@7cCD_0gZ09$c@f4<<p~XGoWwB{i`>{$6Cp{>+Qb>`~B*lcS|CY>RCtfKk&M8*!qb4iq$7oQ|81hw8(^ynuN6}f{yP@!)1$o>KyJ-VxA=cgox_B{c'
    'n)Gy5kqH+;<3TPki82bYD@i#K72<0cJ<}*MU>HUcr?{!y<aAmMRJ3=spAFL)Xhn0o{DF@Seu*c>n)12U>uy#086Qa1iwO_~$PCFmHtO<00m4?q8kdG?h8VcU|Il9h^FR6X)3<MntxnTA3N^@YuhN?>)oKKkQPhV)'
    'yj2>7_zyKbpwr71mKFhsUYI=Q{77+n2o{cik-Srk#7^5G7CSpYT#yl69bx1ej5ZbQeD|GK=)3RArS_j6zuZ3BeeM=jZ&3-b{~Rh-7t$ZF+_ObZ2?@|v2Xg8)VDbtGX7&g>{zcisaj_LH5t6`5%N4U^%I4p~`$&tF'
    '7$p%pilY(wLL~}2TQss5@l~nR-etYXWqV^|ov`ICX)|i|2y|1A*Aa9>Xv>qxH`$jJ5`<pTwWR}!5PrYvUV_YfvB`NC0aC)vB0(&VzOWFGhI_Af_TN0;J^W?pp0ZMW9>G+EwxODaNijW)F75oII;gE(5T?dyv7Dmw'
    'iSVIIbT8?Ot8l!HR7e3j*)np!mQH7(7t7C@N?5gZL#Zeq<*CW0X)#|PVF)|{<nDJT@$s7%FZRA8g&Ax_^9^?;zJq;dz5I?#3d+WS$Q&w=9%ofKnJ0PPhws_Y&t6t@!XJ(?Wx(VzVQ*UuhY<(~M9fs@7CGxOubcQM'
    '8%4Ayr&D#%vm{AGyc7VT+ieBi_wOm!5Gyv{U1r@&5$`B8kt=UBo1XM}(M2XQy!he^>x`%4ysHoP_wVPkZo+{pJ1&}JK{4|JXr87E?LRf3zyqM4U0hCJrE{|QJxkBgZAj&TmyknYNjL@ii*rU23&XOgsI52XcAE~h'
    '4OTfZO6by5r^@^HwRK*9yR#GT>^$3h&3$SSQG?c#4MA=N8Cr?A`1@NFUqLPx>2^blwr^fP103GP;lNv)SDTI#@i&{pbXshmN|+oeGg26xQ8&PD#pow`1u%vv@Z39=mKdPc0)zng*e?Mt9XofrHMCP|v@=g$QT<Uk'
    '7t3%^WED5*oh^h|;oDhgtLnMRx`?BfI1z<RK6CmUnv%=j;%E_-+sDHMHPu$5Q8%<oItSptQdIcoidg^RE#mhi>%yi#IQtbyH^Kd6SYZ4>^P))m=PkRy7#Z$cMNbcJ_VeLI{9>FB@JHj}X7gz<vl@PtLTA0_DMc=Z'
    'U&Sf;WZbTBPb<J>(htDr-IJH`>x1p1ua8^73QcDPeZgP9+2053r+?S^$87dgcL79t3Il_lD0VCLM8B{b>uvlJgScT*M`+!L+luO7b)%&QihV%+gXdT$e(~n@4s^0N%qFp4yK$XoJqIY<YrF$n7<To)<^eHeYFP(O'
    'a;*h!lvWtI78eQab4d<!iim>*Dm3IHTs=SB+o2x^o01x`+?G20I!dC)f;nB@$mlxOmd1V-YGPSGptLF=(>)DToVo}oY&xDa)>|eNZHHn)QdxiQPgTUe650#eHrwron<^b=xMP8o3a?4KhVCDH6E;^=WS^!rvxeSc'
    '`tV(J31q=>iMt?`c#_DEQ(497K&?hR7<8{?ro0~R)his%+oMtc22N5q1K+_9keKlx6H<YJHjGkilnpU+&<U!6g3V}SeT|eG8xI&fWmOo*Nwcm*4f9@+dai}S8L71COmBpgqgw)AuNISDyZgl#8xNhQFGHgCIL)<u'
    'RIGK2aTgT;IK;JL1c14YDJn<AaSUKN^KpbRL^1GvH|Zzi6MC?|^6QujRi?z``7KC_S;J_TQ{Ay0*&-3%?gQXN(s{{_%okhmA05h7L3d$lYFQ=!!q`DPDZ{uXc1IkSM!D;G4xZIaC+!^s6F^BvZoo}OdQldGov00E'
    'c^FyBl4CS8>sMP=Tg)<Eft;+RL{~(v?V-^pJ3~Ome%W(eWeW59F`abgH;&?bWs^KYwRk8?)CCv>)cv`<{zcd<SCNQ{UJZWxi?DgGs$+Yt_1tb%ifReDu~mw_NFhFyO+d*7*|7+y@}(4H;3((#tEG6J0ywe(I`%xU'
    '!pNYEg-3xr!WR=)M!k&W6)D;K*;yC9Igzd~rB)YtMTQG^XQ5~)&WBqz-kO*Uo*=SAM<fbq@JH(|=(o{!V2Ky|+g}r>y>obIet-9!{=NU)yjJ%<_+a>W{iF-NdbRUy{HJ@s2pt^Ki%KnUb2gJhf=r`sVzCV{ai#R>'
    '%TYGB76v=wSXtZvuA7c`B$=^8>r&+DyeM!ORtTRflVSrx%bhy=J4x9_v(s#b_gtH;bkg{Yq}O*$aqY^QDou>-dD8F)T?HwQ1jbHYrw=*{^cf<jvU4;>^Xla(+rZOJhROP-1E;BgrjZlLi!oiq-Poq#^`yHp0<R(A'
    '!c#mW{Xh;>j_tDm&CYU|MnA`eB{dUgBuuo0-~cUP(1$==x4IHmP4N`k3B3K2oHqB^y8Onn9u?Cx3*l7TqR>N4iwJpQZrN-`m+5<DZF#!2De|7U@{Lnc^oFOaan$*O_?EC$riwb~x*NvVzMDEtR)4*|`c+5%u6FK+'
    'js;G2<X#nMKrWk<<sik?5`K)9qDIi1I5<&+@SY~NhJZ8Gmrs@6P1)9jdtE~-FnZ-?K2|R)_@|N!W07!*KWnF}>I-i}EYGiofm$m#IHn)X>Yi?yS8y%!ypL!@$nM=lrSRjH<vU#w_?RHl;-aTgx8tyU^oi9p3N1NM'
    'h29vbpW;2^=09n;0h2Mwp=nFS&MGBuvn<VsP)O846B@ZX;I3_Ig5~5Aix97;5KZ_I#DKgzj)lk)3@VTeeVla*KGHO+(4+z@=Cx3RDGs7ku9|A|GZyY=*1{=6eN#|&A`7QQtOCJ?MknqqG=C^X#$LS86~~LB8X96~'
    '{5G`C9CB;rZ?<{uT~jYAqjQN`G65R;YYeVg?a(zpQqy=knwSB5m}=d|EywQY^1T$}rzr)dB3tGty;ot5d#<P5vg^Az!3$-7*2`5D$Gn=nAJ#1p8vNcU3C-c}8Skq#5Y)S+d8odrvSme=N@(!gLs8mthGB6<Q7~X>'
    'tf|+L5+Ng2NqvXw*))B7@+t-x#7DdT)0@4c-FRnv=jAR_$8aO`Pe{Z|Ct@A4RJj#?xOyhU!LeOygck>hqv=_mUj^$42Pb#u+RF-*nR-!JQoh=EGoqF8xfgvzDwiIn2<9G}SqzvUsy<ecRR~KSvLp`3WsP({;&AG;'
    'h3D#rs#XQrw|cEKehqcGWw`<<i`E?2vN0T9#oO{(G|EOmzAr+@_?dZ2E(ekXOk7_FtYJb2sDSRb`9;+XINM-4U?O#rqRf$uZ=OrqyqgB!>9?7ZBew>VXwza45u2zQ7EhvHDahy-Rc>5LQcz|5i&EyO>byhBG=QN|'
    'Oh}LlRQkW0l4A%Tu5+S5dO^@Ba_8!RLUv{&j$@BY(L(s*)AjW=zU32dQ3QN=ARJj`Wv@gLrS~+jRT#K=gV~ZLay~S;N+$SWtHe48S4stWxwsV#Ti2+p9A{scKT9;+p~SX_{8S&_!rp$Pynl0%4<Ce8dE`iqd?+Pk'
    'b<E{eFH0_ld4ZP@4IRZ6QTRN``e-Ny2O$u_1f;WGb88O6H8;9P)xz3}Rnd36pM>gm3q`fC;|WgPcfEF<bIUS-tVu4I6h#Gwdnp0zEl<D!sv5kocfPgjpDI#!g=0!47}pA)E2E1r4k>dovn!Qd{=3V*%>|0x7}u(b'
    '{I+=b0HNNrtGqX)C5Z+FvT-&<`FA1&Y#HW?;=BF#VX!LX0PNi!=NyiSOzIr2$-Gu7%M!yhpJll<EcI4kq;;Kd(P~cRZPc2}3Lb2>{Jo3Km5?$;?)qa6aBR+VCy|SeS<s&6StCJ`r9(s0P39Q9*yBR+=;p!3gSb20'
    'c(5MYVG)=!LR($M2tA0P1lf%_b(Nu^Nt~eiW7BUjrwlD4jw-E~t-BB}m9l%_y!$>><f0?RlvK@h??=baHFP6H9`-;!oviYKq{r#pyOkgy@w{#6kOE++;1X>WDF9#l_A!1U8E(u3)sU=I;(sI_Uvz#=N0P_9(1EL+'
    '%<LJarb4c6$bLO@T$n2K4dwb&;ofoyY~0#+Nn~Rm$J*P=K8VN82RqWq8sl6vfrG|bF&#LgI&O&_0F_p<({fi(2QKl$dbBm*>#>6OgWN?fwHk}svQ}aVT)bQQGqqrJ=ldC%i}Zq%7Q<ZzWpd(pF9f`3I!mksayNNm'
    '@dZd;E%Wo+eeLZaD-Im49Mip+9M_O2(QOP@LZ=21u>@gyq2C6(8&tU-1iwI|4iw<%E}B1*tj4XQy9BpUXfKCGcrlu8tz#Ox_jL10-*YtCIo)gpn`lCCV?GR$$za-Vyl(~Xn-$I%w~AhGsR~wxSp+sZrEtfEDN{cq'
    'G-Df0V@wOa1uS2lOd2;4JT*9D00#_Iz81vY>2;;+3oFf#NS+*rj&j2AAvMN`L*sN@I;K2RM|Dg{A_Bb1;Jz>;^L=SoA6PLCn^D84thNFc{Ztt{5>JnA0rAJoJw$8Ax3$bEgt*TZXZpAd`5h5?<%h~$XlwWqJrm>F'
    'aC9^3ooCa*=IWzp<59G}>ac}1yKY&M%7+hKI^h6YdR!ijlohe3!Rx_qnjQcr`S9Xm>Vr?I7mB;FBivY6bHMs6Mk(>2J0@mT$|KXnno*Wghj=PnvqJniMjA0GJTK$sV$FYwDSFu|-%bnSWCh_atgzd&Ioz0lh#0E%'
    'a1KAkZ`TZ4^&nCe7V%C>ZKSM?xpNIjnbWCsv8ohC9g9<HF0*^iFb?cwoQx<DHehwTz4<75pv6YCXDZt5C+jfsXjAI|+U@n|;S*SYysoApwA+9V(FPS6X2V}4?e<0lD6*+DhCs{FX7o7hRHb3Fbq_xkgK}R_&bLDQ'
    'ksa45I2LnEbwyZcUC322;xSicO^KW=T>=PiHa4RtVTJ21olwJTHld~m{AWBdkKJzHVOuaNq-2S)WfFNGqhX+*IfTCey{nD_ukcsmVM;Lgmg&l3bZ`+7J4ftZkCV{F@QfUz=8@g$k^7*hjxWShEA=wG7ehuzGBfUc'
    '*J;pCZa^TZ2u^L?2cQcTcLsuPi&loVWD!?rLu>aYx%FDMnZ`ADO)Z>nne2#UzT2%PFnmR5eYL(>;WPZ|R>%{oqyw%oyqVrOm%pRj(tWG505Jg|Sf*lJV3l_PPRQX<SP|jiH(yxG#xjLO4P60h(tK;B{9K&0bnKl{'
    'I$DG-X^*%+<{7AQtgyNfeHm?PmcbQ;YlzN|)2S{&a%^oWjk;w+1Axs@l}q+STLGE;hvHiQ?9JZ(^LX$0V1FCaH1X$|fv#e$mI2I9LWr^PTEb2kDE7E0jpFSG%3Fg$KRH8PXlITzME0cI0skB2qXK{RQUcx>vz`OS'
    'AP3@kd#mJY#Od3~Yls@7HBqV9+u>RMp7;~~#&)TV)6+*$E5QFY<-hf)gXcSm4{hPZ-VO`kSSo5qUsaR^0p%)9@vK*~iiU$C=K1s*G>%rlEw+N_QB%(jt3_~(r~L*e%%~KNID^=h45cg>=}-3jTYRep-)K;7tzw`I'
    '_!fGwlRckICB|uCMl!(CgKt7#gmTlrmP9DcfNMt25{ZoYn0!T4Q{oc*w7zIT7yw-%BWV^H&fLp<pm}^G{*@g5LT-A?WvAz{`3_SZmN{kLY)M&gdKqgzpK%r~=D%?E8IN0vu4DPBg>(ixM*_Z??{0PiQ~!EO{$-SW'
    'J`iL2Xgxq0b(KwY24Lk%|4njnfsXVP8%vvT+1YT6lcty<E1%$oe`#)PN-?N*)ko!Nr#xvpet%!64so*9ZKwEd6{i~=xwZ?}rp0&-y({}`OnPcQtda~V+?sH)ET4;gyma9NR<ee~#zOFVO*>*zyPm&jD{%7T_{rM!'
    'wbC4)Fr2f`b$OvG&<;!j%LlA}mq7ixedP+P*sJ;>FUXp`tN7C-(0Q^;_|yWZ0lo*f*!(P9n&rH+#+-fPl;cau=3%}+sw}}zrf|Qzvi)DG3f&g=K0}3GPH=j6Rr>OZ@_Rp}ew&W^xRNW=PF-a_!YHMxxU8qa`!>HN'
    '&2M9~wb^XB{E=>()v%5C|K@$uRmEZ#Poi~YtzKqxmS9q9HM!QT9pQdXydw*Q67z#cGXDs-^WnLt`RV<N40NsFK4Xzd8cWeg+uA)I{nd4>N6rqb-$;-Se%n;ZWlP`rZPW0HY9+&ji&K=T>L@oY%k2)`H?E*@$NR@P'
    '<%tIn(&{Tz2DpCpvzZYkV{I(p_^vILsjr+n?`Jx9p5(jnmWB#c*>>61sGnoO?`$&19A4+|v}S8(vS8!LX4!1WVC`oq?O>tP)p2UdQ%BAX2G7jj|6*Cw3J90b24Xk@87xcBK!o`9B|87LFt?eOS)>BRPkcQc9HOm&'
    'ybA`?f)b*OkpmfuVZ2C5C=yV46+8lOmC)Wx+LOPH;V4SR<KzZJwe_e4(pl?t)BN4Uzw7iD*-G>;Y(ovAEsq~IjcApe6%A}DO&ZPBhfy>5d9abLex<*|l#pY|ew|{EzfQ+Fldsl;rq%8g2-1^gtMRY}ZKHR?h%A=y'
    '844<CP0fEz;Nd|}a%_kpB*VQ}13J#WrUiMx&bf@TSI!lveoB3itIJ`2u~Imx*rfK}G`r8WtZX_;jU{%SHK2;E_hZy6-kz=7t2h8#tMXHC(MnZ|>f}zUmN~<dYwDj&FLPF@QF(+rvc>zfN}KZ2C{kKQ>$9~dWv)Wb'
    '#0u4kXiiFsld>M9N)58$CV8$JMBfZAp$4f^aNJc(Qhk$bzB?M>EoZ09?%lJ@9Mtc;P``}Och9&b-a9-xc(#kk_jV|6`42jep)#PQjV31OP}agqZ-?pdA{(Z>DNy}iO!9%mJwfqO$1T`;*g;bT=Glprx||MgrU`=g'
    'R4@T2H%^8Hx~<|V)G@|lSV(^Amw9&#ML6E0`d>I2;p?S~%jIwV7?iS({`TlDJ5(DFf8QhnA}~M#X>Y^f6`Er=?95IXLNNP5RupK)>|X7h-Zu%9%w;?i%6)42``&=*xVs(^3R^MSQ#M9Tm9ZMSVV6VV<6tGv1MZ(l'
    'zM3rqL4p#>dPZ#bj+o4!)zu0W)4D8xBHE!-HgF0#Xvy$_%urEkd#9|`RM&kusIz63$%;9kN9mMZW&~VP^+wiarU$bW;ky*$UKDG5^R)&L29f}$U^YBY$I)m)_@N5nTqIl<UCnp4T;%cqIGQ)mQIpn+yj=N<G^QMo'
    '6=EP8$UpgeFYitVlweX`M~A)h=kk(LhRCEx=o{JA<?&OD?;1~9Po&6MHW|>xIKGeARHK36BGUGTTBrdx!$qS>9s?;;T?p}~G7+b{0fx-9&mX$|yZ{h9Rc+|!k@Hm#RNinCjNE#8dCFo?V<D4;^P0?93y$Vq_AXaY'
    '-kwf!vFgspraT&-yGvW!!prIKm(bg{`!DkGPCo2q$VMjpuhXHhl(>Gc@?P3E4T^pF9<{*M<V*I3<E6%+w><>z3Gjv|IsRMGk7BQZ#i4Kq&TRRnRDYwYO;gu{y8t!jpz(1s%FNy};LnVeRoD{BJoDFKX|ub!Y3M@U'
    '6s*ei6A`nms(CmxLdHhYR^32Au1_X-++S}3KSBab*y22l?#P^i-RsFt-p|LXC=i&vp;mHY3g+#@y>Ln7twDxX9cD|_Ijp4?Lh*nz6)Qp%As>X1YK^kF>5p0+0g*5W(mfhL35l+)p$-jI2iUP#y^DbbB-&EHHkHM_'
    'Wtq1h<^b15F2(m)Gjb0rREg6%yT^|Ne$yde)TI#KY`C;QA8C9<#c~*hEpGobl=pOEr9+e^PeTmXj195`I?90)kNKXz#i>49^E;8_bICra&~ojge{^XE#HgKmtl()lK=7+(=@!Cjwds*e31#GP4Sa7@c}A;<Mkc4)'
    'mTE$s-tbhasu#-Z-9G)X;#z3)3G?^>LWD0+82&;f+h}@2CJI{-hUtz?d4y0F#3h8HLqP}B!=v37`+Hx%Jc*BXkKgQ{P^Jn;nv?EBJ&;yl)TO@r0}H<#*%N<=q^6>Ka{RHFmb_^2AGsqM=kGivCA+?K&nfW)msNUC'
    'J0_yajd2PkBble%!$KqO06V36cp1oIBgsiE(`#gLxy}n-^G>}}qIDfrn}wpH?5UlMqQf;M!vM!lKfPa?B7Sd`BK&w+Ey89w*QPR~yS3#hI?0Xc?5yl&<qX$kg5VhMm<^(Caw>%A-LhHSEnmf(<K6h=MLS%V-!JyJ'
    'zr!a#&O<TndvX3ziy#<*cdR44l#nY^@c~9v7`$k2Y_6Ar{g;5tT|}&Rz>q6tK<aD*Qnr`?r%N}h<^_}~)$5^WB`VP+d4MFxBTM0p$E&cAIDoV8?fdn0isK<2u?LmPzrq=x+o3qHp=BH;+$%dt;<hx2vuGE><`%Cs'
    'G+D+g5=rJ9^5DpW4S$@jK3rd4-|FD3&cpP5Hfi6MKr>UeR}1r;QFKxVVz~!^CuSl}KvSogL4x}S@wJ-%d6DUE@XXihS(0x#$0N6cX&z5E`2MLc%<&l01{Z0MN*CgLL!D+rDg?|ReKOe|jJ*eI3Y|n#Gz1Tcmsu7%'
    'Ot8UIO~gC)D&baV%Ho%zz#Q^YXnki^jz1$YXIl1FI@Uksn;oQP4>h$Mr>4?H`SY+%<tv}L3BjPPd-q-+oa{b3IQS+$-aUD97$5I_{d)W4%@HcZ!UvmQJ=xeiU+-<MZ$AEV{n3Nu(UY_D$6sxv$;0ypUq0$SJm2hY'
    'KKv@}K6vsdUElaJ?LB$0-dq3b(U)gm_MUXZd-vkK*T*N@`}^_1o0G#gC(tT#fs(Fbis>#c3Wq*RqU!tXnrx*k;xl>W=*{b`m6kf5;$F&0lrSb2#h@*_Q+s5LY|KFZMtwTw1A#itiu5GdJNze^Y?;Va`8xA1YkTv='
    '_<u;R+V;YqJco~n>^@M4_Hec%#~4iAMQHP`hE7?dUs(w}ro(_t`GfQ6u*=st&>Fg#2gzA}oj$dblAD16%^){iOx&HI$F&a5g|?kZ-yg0O-$90h(Hzh`7(+d*-0Gpt@coJ%<zeNR#%C3IYPon<<YPAqRq08E(Ba-z'
    ')T6~_N~}_p;#@i*Wh&fAR){vPuuYT{D<`G1{FY-CF|>MADuhT(%Msggi8MI!acm-ImE&B+b#X(Tu{wpTN6f*{l;o}PXN9k{@H%|O&a|Zj@Seolb4E(M(~4faVuh?t*do7N&gy~gLU1HBGHh4weE=I-Gkm+kY!)=R'
    '!rax$j8{WC!xmOxQ>@7B3<|m9j#SdnZW;?qCA~CFBvK;|N-TOlGHq{S#2>|z$yfK|shS`PN-HW2mcj@LXyLVoxy@GaIOa+=;Qg2*f`;T-$x)meD}+JOWH4H(s8bjIht(NX4)6+<D%w(x`=V*Kr2+vntX9MI1wm5_'
    'Ow~~rg}|PjgSZRlxXe6nZG79}%qcdiZmGN?dJ1)M0ZL%028}%hH>pjmdDkk+=}1FlkANhXl2)(0Td5d_$zqLo%A$A7$4-OaH{B0#?x+v$3X~6HF6OcSbc2q$cPOluhte%EQ>A&g%Ll-UT%4nSOw2`@@RfuBZ)=Z6'
    'D;23r%toaWt+FqvKfDT>+?>-lig*_7K?*aQKMq~kiw*Hp3`9uZas$zYJ@?Qh24Y-BRgigNSa<1<XzLy3Q-rFOx5E|5IGWEpO0@*FBHLx9>8Ei+!N&?PVuI#Wu@$vgHP?K0W#{8>GyA9r`ky}h`|N4(&%gQjyP)^!'
    '!{5Q#Ga3K!51&5#RTsmpMVQCDJ9|6a*G$RV;m&0S((R`o{_7;@e*C|0g5kyV(}%ynTry_RQ0x{#G+Xha(UdY4m-nDYHkbuWfPrHhOuFe&YbqNWW6cSS_+U{7v+^Qdv5&Hn%G#}b>Qw9q4~o}hsHK#mjr+>zr)a4_'
    'YUC0o{CYNop<B<Q+>|F|hScvt@;>J0jTT*fftZc)r?Nq&n157SL*J^Ye0*iKob;|S&bA{P$RGij#~D(0%uZH}kV-f0?~%+v2b(RNulZ;qCY7oWl=6g+vNjQp4GJNbN=%r%k(+4dPvoUjRm^`?Iy?)Ak|Wr>Fd~7|'
    'LNcA?F<pP_wwzir1gDN(8P##Pc18XMYxKpqH<*HzM|f>}`bE=irFsf#HC1i>44~E{!<Rr!*3My-SQ&~Oet~Ak%7sO@$B2emeC1bp%?RDd?|Fj_3iySyVD!qVr4iNaml2yQX!?XWi`k8=qEduDHGRy{LW0Vn;9{N|'
    'B~*f^tpP8zMw!P6HYQ}C+q$Ba^IDcN)T#$*C2Fdpgrfd=T_~AKpsS4TVhM{J4uDC{FpXN4Se{6%`?pD!Gp2Nidgs`!%W77>LJ`eeMb7a%H_3Nf=p|fPPplSuad(`LC{IMep;<x534USw6W2_g>QxNXV3ZEbP`^mz'
    ';{;2h8wy+H&9mkX!CNS;!PUD@R6bHbg2ZDuSUh5A%NF%VOcijUi%HHm`!T#=M^ysWyjs^;F-<T&I_zi)n7idiP`u-UE{dc0a$`NX`S>5ELHE-Se=`ksU$ct*&Hl+={O$J9-uCMg3^7U?hv53-Z-Z_!7$w=oFc@BD'
    'pFaH0)8GtGest6N^x@w!-_ZZ#AEuF;xWhjj$oaZ<I*i@7v!;X(#%t9WbD4r*yf~7tKc~Z^-9yYi5bwR({(AQq_?b^1{<mNPEKL?jqde}*{Nrz7BA<Tv{R9dDs~DUK=kw`@-wiJ#*U`<JQtd)%bg#*aQXiak=kkw#'
    'NPwmKPZm=POqJj8EDCl2`2TJg{OQ5bH}UhmBdX(UGnm__AO3SUIQ#U&-%o<0H?MI~qf!bCAF-Vi&pqwRVCUif^KTF>h5_X@$FY3+@PBDIZfpUb{P7RL;M0fScZ0*{F9god?Pa+5^us?4Yk;i>q~jYb6Xn(b((T;>'
    'rfyidhgmjx5~f39G~sckZHLsIS1bmA`~aeZu`Z0S`_=<rkn5Ow(&QKIz_&<iHSL9vqg#21IHqe%|LvP|a|7r0XGgzKeQ&ubTP@+j?z5!Gz(t`7E;4&J&FA5t?&8(R@hL@$_}O^Y0bI7w#81HE-`dy_N8M=?384dU'
    'L7E}aL|Pa9)-42j%c>kiMm^}<ZhFh&Lv|+N&!h#yNgq($&jLtsEDa7&d^xkC4vE#@LOc&oE|C;ZHJ{fdJU{#N;V+*CR{-vROXfiV-G3g_-XwyKBAc}@_@3O`wr_`_gvDmzRtD7+S%^4p<E&VBP6|(6$vta+-7Q=n'
    'O4tgEBzwz`hvcOLu*>sE*A`XW^8ABjh~<_-CyA|zFjLuq!J*RoxdEQa!FU}o&>o$a=C+<d9&Yb^gJ&Zj9<o6{{s349n88SO>+}O>&OiU}Oz17t>B2Gew}}a_<={~Hv=KN~(Rp8vTmIZ}rc!_Q$WW`+hjzuBg-{b2'
    'a7c02<OCSFY=YMK4VA|eL*BFkTob&=#bou<0h)ZK=$Za5>0g!PO}r6y*ooU(Jer3uv*71^MZtZ0<c9~Z$d3rzIATu)hr>VJ>X=ztp@i&OksZz3pAkC7W6+^rw1dZ6OEoWW#2Sy**S&o>PLnG;pS?`R4{B8DFn;p!'
    '{}~{&`or&UI25s}Jkwl=+SCTe>yavQ!{bV0j%&TX%O#<e79*yc=!Vw2wfOqOP)tYlt6d%Z{bN*H^CA(F3jp!&NCsK1Dpk}T-XSp9KIQc5*wOg71_!ASOZ2rk&M4id=LT@EdgybVU*-?qb;nt5@V1kV8T>*Y26q)N'
    ')1*JSys5u#R@saeth%i@XqMKO*c&^W*ow|NhKu&}PIl5ygZg-&=o0Ql1EpY~ju*He6|}}Be5DqDnNzB8VXL$-ej1#$7dfr@Uh74>?6Fqk**?!(?R2B#_^7Rv@1^Q&z)oGwv4I0{PYO>vUc$9%b1@ZCqa2XCY<*Q>'
    'U3Po_zOfrbmrG-6x5b8bmN`zN!cHNiAm*`Wt>ETybquM5L4-e!McTZio669}H|avCuisl_D0ISe-hDqpL8fl|IC<ff7Egu2aamOl3QMP#_SM(b>~Ni~bJ=d7)eB5)myuDZm%Xid@4qeciQxxEqk4|iv-fbAMf+Tz'
    'mm$`gpfW7L>L%&jqSD4B<m_gGw`e86sgmA(hmKpDj#Y%as!C|Sa8oG1Q?zo=Pq%<jdLv|u61xE_SzZOk#9Toqmz!_W6IPh?gv;CBgmY|f!b)xSQWiow(Co4nLU%LjahhMks0Xzh0B+8>Y#AtvU2>;NC+UPlr_4_6'
    'YV{;Xn^D&cEn-iaDw>5h?+MM~hRVFh*DTRq^|dlQZtWvTBf-W^8LJaxz`>Dico=nD3ai39UUck+qig?(_&lSz(0SgZLzT+S%vEcHX|lG0cL_tk5FPWiBp`YTg!d+|sJsG}+#VC(nE*+%gO~m&#8<OUhBu8y9q-4+'
    'R4&_BjR4L0k*`8|($e~;4}TLB$y7@w-H*S`qTsu9K$eJ{_y**enrr>AS)w4dsYfil7BeCn3%AS^mPsq3thlO8vurpm(U{N1NIbSWqN!LJJ}m4dc8(7fFGRVh7oUFk|3HBG_}k${q*BUqh&1K=n2fTr60G9RK<|ty'
    'D&scEOtT_3TUB~{B_k~7#M)Zyk*b0_g)ey!=x<1IPb&CfWp-X-hkf^+|7MKu6JdNy-ZGYP9)grY4CGgwYWpnpxlZJRh@I61_>td3|Mq)$oZYL=1m!ZPTXkb7nWAf8D=~@N36k?kIu2l#fGvQky&{~|+ZkJk?;;TU'
    '`B{fOS}9d;J}TK#k<5~!n`Lbo?ZS;H=uS<X;zT<Ov@(0zNt)))L~U{jd+ZV@**^|1KtcslMg~Xo$rx`(9Vtv4#oXO!>{z^==cZBUW!;G}-e}v_tHz{*&2Rkg>K?kOOnV$l3uV^hSxvBok7GLlM>gto&=Sb;+G_jz'
    'YAE*+?R9QER{P@d`Z|WU>tPVK%l76&wlBilX;jIzAE2m#u$L{@8)fZHtb(PAte1jB`9vPk;%Sm!0Y+e}WF>Whz5qll+9B#}Lpu|N;+w1Qc3n<bZXGC+ngr^A1>Ips+57iZmI(GAZ)-<0s@4t<4_Be|_2YPu_JA|t'
    'lrMDgFGcky*Z|3xqWgv>@XCVDq>%271DHaM)#C|(`hnA(2v-GvY1)BG=1xelsA@(^akKjE{^~~bI8-@lRA`%ac+Ib^nX3E7jmOlniK8;FXLFS)T~E+7d^lN?%?Nd!UZniSBUK|t^E@8X!!I8{e3*8-56`<#9z9IY'
    'lZWSz)*o#=g8zT=<iW=IgU4Tf`5;XnhD|13moO{%fB291P!}0EB)%up+83|2Zih`H4R_BkV3@$B4yFUbV*Ei(HK6m&$D11uVM{#R`0C+<^@mMV+r!)S^)J`q?Z*1UCy%~-jBh9Sv}wI_74+hQ4E@BT>^<cSmx!m;'
    'xbIzAo?k?M0~k+<M_ttqY{nngk8S|_VW8;9tv2~sy8TIHSA$mJq+$)uWkiwL&<N=495B<r6m+h()$&>`U-Fn+>W4-*!Wf(8azm%O5sdR0T36r1tDa|RzsCsRUftvQ7qhQA3isR>8iV2byBGUZ6zzjAhM8a3MU1LR'
    '!Z7@-<D*FSE_z#?(hz-r2{k*Y-I+Hf+veaJfz>Td#j!+yV&b%4Irq<+&gaa=siaO0l(TzzH_peII%nc;KY=#RZxl14%1npvlJN*x5jKls9@TZZt{z3TRRuuTLHd#%fF<)$YFn=>6l&*nWbEzcl~VQ_WiwO7Vl1mZ'
    'tW+&o_Hz0%1s%pM4soHBc3Y16um#6>ZD<L0w{&z4dh5Eg?pU$4(HSLV#^BbC0D&*85D-f=PjMuiwr8`23BVo2bv0hu&OjqCl0X}Gx&O+&Eb#8)>XlR#sv7?q&*cxBLWxVB|EPrW24O3GI-P1udZ&<^OmH%PlGXQi'
    'RimBQ;S=UIW3KYN$PYR2N@L(*i_!_k_@cjywP7hX1U5zVt7QVp>q_O~7B#p`3frI1VQGY7H<!vm`(}7G%-;<u*L{&r(C}5|-HC3gotbAuRk*H<Rb9(;Tozp7YuZNAp7pQ(&g4;hMjsv>yx7~{#oKWaz*x7eTq8W`'
    '%LwS771BXUt8fk{yZgJZc2ACe`O^n{s59|34k^e!WR96F+x<#Za{3wD#eI|J({f{-vN^NuK2jDzOGkrl8!y%d<_0KK$FLe?v(9L>WYk(?rL7veCkBL-_R_5NX6;msMK^AJ#d&bXweP2x)i~)}hH%(|u%ivT1jzzp'
    'yuIm*ydO5_84T@yx4ZKOL(39o=m5y3MT%YEw|H-kDf2+IN2PJt7V)DIYyCBtaKX`F1AXlqB_YO<a5P`ss)nF6V@uYoKEH+;aq<yTTCstMdg~h1k>WZ-iyPFW>q4xb&NzxN%^@P<f%?(r5?R2%K713u+CFUGO6{34'
    'JuNUR16xd;aW>Qf9q=a{cKTKiz@~4aki^WXqyR+q17?VqwgFO4f%J}hzzU{Yl&?U|;wRys8dZFg15#JB?6XR`SiKHSaT$wltf^3)6Wl$mX{^*qd(reg9a#;cdn(L=9rN6HOzx3m;KsS@k=Zz3xgF>4BxYCGL3>wv'
    '7X4>uw!)FC^LLalgnLrPG+2;hcyt_NF>Zb#E^13byek=e{Krg`hyN#R%kE{4ma@N*I8h~oSMH)&m6V#{tn&&vV06PeywCCQe#l)s4Q?ADDpzXRybwqF@4A-tf7qNog%j@U-Q)JHqdffC7#P_7y;plD<^w4ZXVFTt'
    '*1@G_<Xe*OMjSnyi?EVc!!lQ5XoRSmv-mH$vLv%TEaG9$e49@Nl3;ZC>4)ECWZlm;tM)p2=6Pxm+L()IaHWoT-Uj$=SFMHTsoM3bayK83rvAQal@EHHhSZ_x$QG~GuHx_NW`M#p83#z=LSJudXGPmVwZapSt%lh7'
    '+!Szcw(KIR1E{H;%G%3&pK)Xgi^pl#T-m#f5T<?_XBX&uyr5>OSZJNEY03Ry&YFq~-24^&m%d1!2koX$QI?^Zx2inK?Gso58ax~k=(+=pj{lhmA%32G{OfLT{prK+=@#^2jbI!!E<SzuiveNTTG?l<OCD4(L7~ou'
    'Y@%seK?b@PY0USJsCm4d5wA6mD_n-xZ7yIN>-xuj7?omH<d;cvi?aoKRJDS5DCV()zT?RVjkv4VRedEcvVb-0P5T(_OR%HHuCG`~Pi+;yJ6tb7iXSIYsjM-szg<~Iy;T*DN{vdc7b-wm0ZG;;;7wd+Y)>2rhMi_P'
    '3SF5#{MbQwu^$&jrLdywXzbB_-iGu7Q&9pjr2uGAOt(CgaB=_xz@#81JQ}=YiM>a$;oddm8fV_bDCY-G2&vTtd@br`MmOxqWQrO+8Ckh`Ar9JGiYCTUD>yX8)8Hce^x+>YrU@gIO>bbQ1AYKG;Qh@3D@K^=CXYM?'
    '3iIyB^px2hbFprNRM|+{@7z#wvJ>QXdcdqb-6+rpkNHTiGYmq80y9ZvRlD&@`+w6#087PlI1u%%7$ua#t&55*Twn94(r4LxE;%P#Sp#7{l4cWUrp7NQO@Y_dl<1671?@BOp3ax8?%u0r)BZfp3#BZ3`g5D90Fi1%'
    'eUfU`&G_|IX%gQjk^pUf6KW2n;-Q!n#24dyI;yE{6$s7LtCqKYXCL+wL_M@<-^IV(G-mfT7czFi6g3e}9h$DfKAZcsC3aOHYzZP}1zs!>$5U6wFIXS2f!oo1qp4WMrfH^DJki^q;SjGwhnM%6ADcyaEy(_>AjLqg'
    'aSJj7nSgXp;U42|+V9saNJT*y?2~o+AFb?lhL$6j(EDZH@6|1<TPnmX&+Ct<eu82pKonTfy2@4&vQ;Nr6fuR-kx^>o3a5-=b=~*#k+Qlh?RSpLRH?bGcD1)z$Fje>9ShUZ-cyVlu?A8vQODB<N-7`lvY#j<J(+{p'
    '4&P_q(RzG8O&zCjtDDoRcj0%RKKy;q|M*Xm$uEH;pR{ywKJ4Fw9m$DYjQWWQoMtr;jnOZ)@zAlogyY_hNn#JJIb$ZDVm-h)i>{mb$G@ID4V)?-AfQ(F(}%yD1Xmyb2_r%LM+}Vc`w6Lr{({XqnQIzPF}-BcC(Fh`'
    'qXC4q-)M|05vR$4b!3mC4+x+n#)x5!md&3sz)wSrFsm6yJy+AMp4`X(6!c(((*T2R{4Z1GmAlse^uvE1t@Zx+hv53-e}RG$=AlS35^nyMC&zjuHlgErKernL(EWbIJw>j$gs9r(I0Fa_lJQN*jG7!v>ZNNaAa&jO'
    'dR787T{2*(F!>b-p?yG=4Web?^DY?=QL$bDUisu?q!bPE{vZD^4L<%8>WHP({3{<el5@y}uhLnLZAAOCq&rE+rY+4qR%}OCZ6}{ZeZ||rX^@fJ7{}jdW<er@A;MZg#&pf1jb3Be8d}y~5b3h?F?Jv^Ta4qwq$V<m'
    'uX+9S!(ZQM<8(yCD+(C*{q)lhztcvXMqQ`8>)d&}(b4&|#Zd|uZ6hJ38^wc8O`<mPjsl}9$orUrb;Ej&8w#16j%;BM(H%RJtEuYpi(=Ls(I{xRB#o&c36uaGek)~%<2)xos_yX=2i5WM-s%T&AInswQ%C)~rH6Ov'
    '6(t)-w9)~>xa2AEot#7xe~AMw<(KkHj&!=ok<PcG+f>mZ_#W;e_22PDKt7$M)u)W|a8W5tb!tp^R$lsaczZXx-g1s%-IxBU&}e?YG#S%dhaPwzlLecTu${cCOm&YJpv}phd&+8lSuLHYs8n{DlKA0Ap8y_>vxI3`'
    'Pc}+Rx1`X)e#6Vo<VQiW?b4!>7Kys5qH=fb(4u$lqtHJ5r|N=7w$3+~Xo${Xge;Z=X_*|meK>3rQ6&afT>J{2{%WKx)8!ng5dDc*vnPZTHnQ8F_EI8G>T<FR>@J)%zs^STLKPypfJ-e;uIFhZ9FCBOj^sffhNQ3Z'
    'Dn)8p%v@~RYs*wzBJ^kJJUP=fKTDH#AAefGip}Dyg*~LX@D*Cwe6(({CCPY_odbH=e2GBN*_*1I)f}HfRoqgzBXnEUI<;za{A=>_m0MGew+7y~ns-KHj`-ZNKZz!;Gka>kl@o?J1gOVMCq+jC?s95HPO~ZrwkWri'
    '&Q`Q>J}aV?(yji>lUMuh<R~w1d^s8P?b%sSI}HIBB=u7z!2WFS&~4I`uVk=fiEJw-wA#6UUu>_6Qb1Kd4y?Eow{KUpPw~oDP?tqQyAID#^wyo+^;%)2bSJb@auusG*Sd@GgR!eqdG7(W?FNslE#fS`xM~REB<|&R'
    '*2_|5M|SW32hK(+d;'
)
_vtm_decoded = _vtm_zlib.decompress(_vtm_b64.b85decode(_VTM_PACKED_CODE))
if _vtm_hash.sha256(_vtm_decoded).hexdigest() != _VTM_CODE_SHA256:
    raise ValueError('Embedded runtime source checksum mismatch')
exec(compile(_vtm_decoded.decode("utf8"), '<vtm:offline-setup>', "exec"), globals())
del _vtm_decoded, _VTM_PACKED_CODE, _VTM_CODE_SHA256


### Worker chạy riêng

Chỉ ô này dùng `%%writefile`: nội dung là một worker độc lập, không import mã dự án.
Mỗi model chạy trong một tiến trình rồi thoát để nhả GPU. Không chạy ô này là chưa có worker.


In [ ]:
%%writefile {NOTEBOOK_WORKER_FILE}
from __future__ import annotations

import dataclasses
import hashlib
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
HUNYUAN_PROMPT = '提取文档图片中正文的所有信息用markdown格式表示，其中页眉、页脚部分忽略，表格用html格式表达，文档中公式用latex格式表示，按照阅读顺序组织进行解析。'
LAYOUT_PROMPT = '按照阅读顺序解析图中的版式信息。'

def _json_default(o: Any):
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)

def config_hash(cfg: Any, exclude: set[str]) -> str:
    data = {k: v for k, v in dataclasses.asdict(cfg).items() if k not in exclude}
    raw = json.dumps(data, ensure_ascii=False, sort_keys=True, default=_json_default)
    return hashlib.sha1(raw.encode('utf-8')).hexdigest()

@dataclass
class OCRConfig:
    """Cấu hình bước OCR. Mọi trường đều có thể sửa từ notebook."""
    engine: str = 'hunyuan'
    input_dir: str = 'data'
    books: list[str] = field(default_factory=list)
    workdir: str = 'work'
    run_name: str = ''
    pages: str = ''
    dpi: int = 200
    max_side: int = 1280
    segmentation: str = 'none'
    reading_direction: str = 'vertical_rtl'
    segmentation_args: dict = field(default_factory=dict)
    region_max_pixels: int = 0
    region_max_side: int = 0
    region_upscale: float = 1.0
    resolution_strict: bool = False
    model_min_pixels: int | None = None
    model_max_pixels: int | None = None
    attention_backend: str = 'auto'
    image_dataset: str = ''
    image_dataset_key: str = ''
    image_variant: str = 'original'
    image_crop_type: str = 'group'
    image_dataset_fallback: bool = False
    image_content_key: str = ''
    allow_generated_variants: bool = False
    max_new_tokens: int = 4096
    dtype: str = 'fp16'
    model: str = 'tencent/HunyuanOCR'
    model_revision: str = ''
    ocr_algorithm_version: str = 'prepared-ocr-v1'
    prompt: str = ''
    repetition_penalty: float = 1.08
    device: str = 'auto'
    paddle_version: str = 'v1.6'
    paddle_args: dict = field(default_factory=dict)
    tonggu_repo: str = ''
    tonggu_config: str = ''
    limit: int | None = None
    page_timeout: int = 300
    n_gpu: int = 0
    worker_python: str = ''
    worker_pythonpath: list[str] = field(default_factory=list)
    budget_h: float = 10.5
    stall_min: int = 15
    max_restarts: int = 5
    n_layout: int = 0
    layout_budget_h: float = 1.0
    layout_prompt: str = LAYOUT_PROMPT
    export: bool = True
    fresh: bool = False
    resume_from: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        if self.attention_backend not in {'auto', 'sdpa', 'flash_attention_2', 'eager'}:
            raise ValueError('Unsupported attention_backend')
        if self.engine not in ('hunyuan', 'paddle', 'ppocr', 'tonggu', 'dummy'):
            raise ValueError(f'engine không hợp lệ: {self.engine}')
        if self.dtype not in ('fp32', 'fp16', 'bf16'):
            raise ValueError(f'dtype không hợp lệ: {self.dtype}')
        if self.dpi <= 0 or self.max_side <= 0 or self.max_new_tokens <= 0:
            raise ValueError('dpi, max_side, max_new_tokens phải > 0')
        if self.segmentation not in {'none', 'auto', 'columns'}:
            raise ValueError('segmentation must be none, auto or columns')
        if self.reading_direction not in {'vertical_rtl', 'horizontal_ltr'}:
            raise ValueError('reading_direction must be vertical_rtl or horizontal_ltr')
        if self.region_max_pixels < 0 or self.region_max_side < 0 or (not 1 <= self.region_upscale <= 4):
            raise ValueError('region caps must be >= 0; region_upscale must be in [1, 4]')
        if any((v is not None and v <= 0 for v in (self.model_min_pixels, self.model_max_pixels))):
            raise ValueError('model pixel budgets must be positive or None')
        if self.model_min_pixels is not None and self.model_max_pixels is not None and (self.model_min_pixels > self.model_max_pixels):
            raise ValueError('model_min_pixels must not exceed model_max_pixels')
        if not isinstance(self.segmentation_args, dict):
            raise ValueError('segmentation_args must be a dictionary')
        if self.image_crop_type not in {'page', 'group', 'column', 'native'}:
            raise ValueError('image_crop_type must be page, group, column or native')
        if self.image_variant not in OCR_IMAGE_VARIANTS:
            raise ValueError('Unsupported image_variant')
        if self.image_variant in {'difftsr', 'unihir_restoration'} and (not self.allow_generated_variants):
            raise ValueError('Generated-content variants require allow_generated_variants=True')
        if not self.image_dataset and (self.image_variant != 'original' or self.image_dataset_key):
            raise ValueError('Image variants/snapshot identity require image_dataset')

    def cache_key(self) -> str:
        exclude = {'input_dir', 'workdir', 'run_name', 'limit', 'page_timeout', 'n_gpu', 'worker_python', 'worker_pythonpath', 'budget_h', 'stall_min', 'max_restarts', 'n_layout', 'layout_budget_h', 'export', 'fresh', 'resume_from'}
        if self.attention_backend == 'auto' or self.engine != 'hunyuan':
            exclude.add('attention_backend')
        if not self.image_content_key:
            exclude.add('image_content_key')
        else:
            exclude.update({'image_dataset', 'image_dataset_key'})
        if self.ocr_algorithm_version == 'prepared-ocr-v1':
            exclude.add('ocr_algorithm_version')
        if not self.allow_generated_variants or self.image_variant not in {'difftsr', 'unihir_restoration'}:
            exclude.add('allow_generated_variants')
        if not self.image_dataset:
            exclude.update({'image_dataset', 'image_dataset_key', 'image_variant', 'image_crop_type', 'image_dataset_fallback'})
        return config_hash(self, exclude)

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)


import unicodedata

def comparison_text(text: str, mode: str='visible') -> str:
    if mode not in {'han', 'visible'}:
        raise ValueError('mode must be han or visible')
    if mode == 'visible':
        return ''.join((c for c in text if not c.isspace()))
    return ''.join((c for c in text if ord(c) >= 13312 and unicodedata.name(c, '').startswith(('CJK UNIFIED IDEOGRAPH-', 'CJK COMPATIBILITY IDEOGRAPH-'))))


from bisect import bisect_right
from math import isfinite
from statistics import median

def line_rectangle(polygon):
    if not isinstance(polygon, (list, tuple)):
        return None
    if len(polygon) == 4 and all((isinstance(v, (int, float)) for v in polygon)):
        x1, y1, x2, y2 = polygon
    else:
        if len(polygon) < 4 or any((not isinstance(p, (list, tuple)) or len(p) != 2 for p in polygon)):
            return None
        if any((not isinstance(v, (int, float)) or not isfinite(v) for p in polygon for v in p)):
            return None
        x1, y1 = (min((p[0] for p in polygon)), min((p[1] for p in polygon)))
        x2, y2 = (max((p[0] for p in polygon)), max((p[1] for p in polygon)))
    if not all((isfinite(v) for v in (x1, y1, x2, y2))) or x1 >= x2 or y1 >= y2:
        return None
    return (x1, y1, x2, y2)

def order_ppocr_columns(raw, *, mode='auto'):
    """Return comparison geometry; the original checkpoint remains untouched.

    Auto requires at least two tall detections and 70% of detected characters
    in tall detections. Unknown/missing geometry keeps detection order. Within
    a shared column, fragments are read top to bottom. Every character offset
    is moved with its original box, even when the same glyph occurs repeatedly.
    """
    if mode not in {'auto', 'vertical_rtl', 'detection'}:
        raise ValueError('reading_order must be auto, vertical_rtl or detection')
    info = {'mode': 'detection', 'changed': False}
    lines = raw.get('text_lines', [])
    if mode == 'detection' or not lines:
        return (raw, info)
    texts = [comparison_text(line.get('text', '')) for line in lines]
    if ''.join(texts) != comparison_text(raw.get('text', '')):
        return (raw, {**info, 'reason': 'line_text_mismatch'})
    boxes = [line_rectangle(line.get('polygon')) for line in lines]
    if any((box is None for box in boxes)):
        return (raw, {**info, 'reason': 'incomplete_line_geometry'})
    vertical = [i for i, (x1, y1, x2, y2) in enumerate(boxes) if y2 - y1 >= 1.5 * (x2 - x1)]
    total = sum(map(len, texts))
    if mode == 'auto' and (len(vertical) < 2 or not total or sum((len(texts[i]) for i in vertical)) < 0.7 * total):
        return (raw, info)
    tolerance = 0.45 * median((box[2] - box[0] for box in boxes))
    groups = []
    for index in sorted(range(len(lines)), key=lambda i: -(boxes[i][0] + boxes[i][2]) / 2):
        center = (boxes[index][0] + boxes[index][2]) / 2
        if groups and abs(groups[-1][0] - center) <= tolerance:
            groups[-1][1].append(index)
        else:
            groups.append((center, [index]))
    order = [index for _, group in groups for index in sorted(group, key=lambda i: boxes[i][1])]
    starts, offset = ([], 0)
    for text in texts:
        starts.append(offset)
        offset += len(text)
    new_starts, offset = ({}, 0)
    for index in order:
        new_starts[index] = offset
        offset += len(texts[index])
    characters = []
    for entry in raw.get('character_boxes', []):
        old = entry.get('offset')
        if not isinstance(old, int) or old < 0 or old >= total:
            continue
        index = bisect_right(starts, old) - 1
        local = old - starts[index]
        if texts[index][local:local + 1] == entry.get('text'):
            characters.append({**entry, 'offset': new_starts[index] + local})
    geometry = {**raw, 'text': '\n'.join((lines[i]['text'] for i in order)), 'text_lines': [lines[i] for i in order], 'character_boxes': sorted(characters, key=lambda entry: entry['offset'])}
    return (geometry, {'mode': 'vertical_rtl', 'changed': order != list(range(len(lines))), 'line_order': order})


from collections import Counter
import re
from statistics import median

def ocr_quality_issues(text, raw):
    issues = list(raw.get('quality_issues', []))
    if raw.get('hit_max'):
        issues.append('token_limit')
    visible = comparison_text(text)
    if re.search('(.{1,40}?)\\1{11,}', visible):
        match = max((m.end() - m.start() for m in re.finditer('(.{1,40}?)\\1{11,}', visible)), default=0)
        if match >= 80:
            issues.append('repetitive_output')
    if len(visible) >= 80:
        lines = [comparison_text(line) for line in text.splitlines()]
        signatures = [re.sub('[0-9０-９]+', '0', line) for line in lines if len(line) >= 12]
        if signatures and max(Counter(signatures).values()) >= 6:
            issues.append('repetitive_output')
        symbols = sum((not c.isalnum() for c in visible))
        if symbols >= 80 and symbols / len(visible) > 0.5:
            issues.append('symbol_dominated_output')
    return sorted(set(issues))

def page_quality_issues(records):
    issues = {name: list(record.get('quality_issues', [])) for name, record in records.items() if record is not None}
    lengths = {name: record['visible'] for name, record in records.items() if record is not None and (not issues[name]) and (record['visible'] > 0)}
    for name, length in lengths.items():
        peers = [value for other, value in lengths.items() if other != name]
        if len(peers) < 2 or max(peers) > 1.6 * min(peers):
            continue
        typical = median(peers)
        if typical >= 80 and length < 0.35 * typical:
            issues[name].append('suspected_incomplete_output')
        elif length > max(3 * typical, typical + 200):
            issues[name].append('length_outlier')
    return {name: sorted(set(reasons)) for name, reasons in issues.items() if reasons}


from math import floor, isfinite, sqrt
from pathlib import Path

def image_budget(cfg):
    defaults = {'hunyuan': 4194304, 'paddle': 1048576, 'tonggu': 1048576, 'ppocr': 4194304, 'dummy': 4194304}
    pixels = cfg.region_max_pixels or defaults[cfg.engine]
    if cfg.model_max_pixels and cfg.engine == 'hunyuan':
        pixels = min(pixels, cfg.model_max_pixels)
    return (pixels, cfg.region_max_side)

def projection_mask(image, options):
    """Analysis-only ink mask; never use these pixels for canonical/OCR crops."""
    from PIL import Image, ImageChops, ImageFilter, ImageOps
    gray = image.convert('L')
    gray.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    radius = max(2, min(gray.size) / 80)
    mask = ImageChops.subtract(gray.filter(ImageFilter.GaussianBlur(radius)), gray)
    return ImageOps.autocontrast(mask).point(lambda v: 255 if v > options.get('ink_threshold', 40) else 0)

def projection_gaps(image, axis, options):
    """Find internal low-ink gutters; suppress long rules in analysis only."""
    mask = projection_mask(image, options)
    w, h = mask.size
    data = mask.tobytes()
    left, right = (max(1, int(w * 0.03)), min(w - 1, int(w * 0.97)))
    top, bottom = (max(1, int(h * 0.03)), min(h - 1, int(h * 0.97)))
    if left >= right or top >= bottom:
        return []
    rows = [sum((v > 0 for v in data[y * w + left:y * w + right])) for y in range(top, bottom)]
    valid_rows = [y for y, count in zip(range(top, bottom), rows) if count < 0.7 * (right - left)]
    if not valid_rows:
        return []
    cols = [sum((data[y * w + x] > 0 for y in valid_rows)) for x in range(left, right)]
    valid_cols = [x for x, count in zip(range(left, right), cols) if count < 0.7 * len(valid_rows)]
    if not valid_cols:
        return []
    if axis == 'x':
        values = [sum((data[y * w + x] > 0 for y in valid_rows)) / len(valid_rows) for x in range(w)]
        extent, length = (image.width, w)
    else:
        values = [sum((data[y * w + x] > 0 for x in valid_cols)) / len(valid_cols) for y in range(h)]
        extent, length = (image.height, h)
    values = [0.0 if value >= 0.7 else value for value in values]
    threshold = options.get('gap_ink_fraction', 0.006)
    minimum = max(2, round(length * options.get('min_gap_fraction', 0.006)))
    edge = max(minimum, round(length * 0.06))
    gaps, start = ([], None)
    for i in range(edge, length - edge + 1):
        blank = i < length - edge and values[i] <= threshold
        if blank and start is None:
            start = i
        if not blank and start is not None:
            if i - start >= minimum:
                cut = round((start + i) / 2 * extent / length)
                span = max(minimum * 3, round(length * 0.035))
                before = values[max(edge, start - span):start]
                after = values[i:min(length - edge, i + span)]
                support = max(3, round(length * 0.006))
                if sum((v > threshold * 2 for v in before)) >= support and sum((v > threshold * 2 for v in after)) >= support:
                    gaps.append(cut)
            start = None
    return gaps

def _partition_box(box, cuts, axis, direction):
    x1, y1, x2, y2 = box
    ends = [0, *cuts, x2 - x1 if axis == 'x' else y2 - y1]
    result = [(x1 + a, y1, x1 + b, y2) for a, b in zip(ends, ends[1:])] if axis == 'x' else [(x1, y1 + a, x2, y1 + b) for a, b in zip(ends, ends[1:])]
    return result[::-1] if axis == 'x' and direction == 'vertical_rtl' else result

def plan_regions(image, cfg, *, page=0):
    """Return a disjoint, complete partition of the rendered page in read order."""
    cfg.__post_init__()
    full = (0, 0, image.width, image.height)
    info = {'method': 'whole_page', 'warnings': [], 'coverage': 1.0, 'overlap': False}
    if cfg.segmentation == 'none':
        return ([full], info)
    options = cfg.segmentation_args
    allowed = {'column_cuts', 'page_column_cuts', 'columns_per_region', 'max_regions', 'ink_threshold', 'gap_ink_fraction', 'min_gap_fraction'}
    if set(options) - allowed:
        raise ValueError(f'Unknown segmentation options: {sorted(set(options) - allowed)}')
    cuts = options.get('page_column_cuts', {}).get(str(page + 1), options.get('column_cuts'))
    native = cfg.segmentation == 'auto' and cfg.engine in {'paddle', 'tonggu'} and (cuts is None)
    if native:
        return ([full], {**info, 'method': 'native_layout', 'warnings': []})
    max_regions = int(options.get('max_regions', 64))
    group = int(options.get('columns_per_region', 3))
    if max_regions < 1 or group < 1:
        raise ValueError('max_regions and columns_per_region must be >= 1')
    boxes = [full]
    if cuts is not None:
        if not isinstance(cuts, list) or any((not isinstance(v, (int, float)) or not isfinite(v) or (not 0 < v < 1) for v in cuts)) or cuts != sorted(set(cuts)):
            raise ValueError('column_cuts must be unique increasing fractions in (0, 1)')
        pixel_cuts = [round(x * image.width) for x in cuts]
        if len(set([0, *pixel_cuts, image.width])) != len(cuts) + 2:
            raise ValueError('column_cuts collapse at this image resolution')
        boxes = _partition_box(full, pixel_cuts, 'x', cfg.reading_direction)
        info['method'] = 'manual_columns'
    elif cfg.segmentation == 'columns' or (cfg.engine == 'hunyuan' and cfg.reading_direction == 'vertical_rtl'):
        found = projection_gaps(image, 'x', options)
        if cfg.reading_direction == 'vertical_rtl':
            selected = list(reversed(list(reversed(found))[group - 1::group]))
        else:
            selected = found[group - 1::group]
        if cfg.segmentation == 'auto' and len(found) < 2 * group - 1:
            selected = []
            info['warnings'].append('too_few_gutters_for_context_groups; kept whole page')
        boxes = _partition_box(full, selected, 'x', cfg.reading_direction)
        info.update(method='whitespace_columns', detected_gutters=found)
        info['warnings'].append('heuristic_layout; preview cuts on text/annotation pages before OCR')
        if not found:
            info['warnings'].append('no_confident_column_gutter; kept whole page')
    pixels, side = image_budget(cfg)

    def too_big(box):
        w, h = (box[2] - box[0], box[3] - box[1])
        return w * h > pixels or (side > 0 and max(w, h) > side)
    pending, boxes = (boxes, [])
    while pending:
        box = pending.pop(0)
        if too_big(box):
            crop = image.crop(box)
            axes = (['x', 'y'] if crop.height >= 4 * crop.width else ['x']) if cfg.reading_direction == 'vertical_rtl' else ['y', 'x']
            children = None
            for axis in axes:
                candidates = projection_gaps(crop, axis, options)
                size = crop.width if axis == 'x' else crop.height
                candidates = [c for c in candidates if 0.15 * size <= c <= 0.85 * size]
                if candidates:
                    cut = min(candidates, key=lambda x: abs(x - size / 2))
                    children = _partition_box(box, [cut], axis, cfg.reading_direction)
                    break
            if children and len(boxes) + len(pending) + 2 <= max_regions:
                pending[0:0] = children
                continue
            info['warnings'].append('unsplittable_region_exceeds_budget')
        boxes.append(box)
    if len(boxes) > max_regions:
        raise ValueError('Region plan exceeds max_regions; increase columns_per_region')
    info.update(regions=len(boxes), coverage=1.0, overlap=False)
    return (boxes, info)

def prepare_region(image, box, cfg, *, native=False):
    from PIL import Image
    crop = image.crop(box)
    original = crop.size
    pixels, side = image_budget(cfg)
    if native:
        return (crop, {'crop_size': list(original), 'input_size': list(original), 'resized': False, 'scale_xy': [1.0, 1.0], 'budget_scope': 'native_blocks'})
    scale = min(cfg.region_upscale, sqrt(pixels / (crop.width * crop.height)))
    if side:
        scale = min(scale, side / max(crop.size))
    if cfg.resolution_strict and scale < 1:
        raise ValueError(f'Cannot preserve region {box}: {crop.size} exceeds pixel/side budget')
    target = (max(1, floor(crop.width * scale)), max(1, floor(crop.height * scale)))
    if target != original:
        crop = crop.resize(target, Image.Resampling.LANCZOS)
    return (crop, {'crop_size': list(original), 'input_size': list(target), 'resized': target != original, 'downsampled': target[0] < original[0] or target[1] < original[1], 'scale_xy': [target[0] / original[0], target[1] / original[1]], 'max_pixels': pixels, 'max_side': side, 'budget_scope': 'backend_input'})

def _page_geometry(extra, box, input_size, offset, region_index):
    """Map genuine crop detections back to the page; preserve their provenance."""
    from copy import deepcopy
    extra = deepcopy(extra)
    if 'blocks' in extra:
        extra['blocks'] = ordered_page_blocks(extra['blocks'])
    sx, sy = ((box[2] - box[0]) / input_size[0], (box[3] - box[1]) / input_size[1])

    def transform(points):
        if not isinstance(points, (list, tuple)):
            return points
        if len(points) == 4 and all((isinstance(v, (int, float)) for v in points)):
            return [points[0] * sx + box[0], points[1] * sy + box[1], points[2] * sx + box[0], points[3] * sy + box[1]]
        return [[x * sx + box[0], y * sy + box[1]] for x, y in points]
    for entry in extra.get('character_boxes', []):
        entry['polygon'] = transform(entry.get('polygon'))
        entry['offset'] += offset
        entry['region_index'] = region_index
    for entry in extra.get('text_lines', []):
        entry['polygon'] = transform(entry.get('polygon'))
        entry['region_index'] = region_index
    for field in ('blocks', 'boxes', 'spatial_regions'):
        for entry in extra.get(field, []):
            entry['box'] = transform(entry.get('box'))
            entry['region_index'] = region_index
    return extra

def processor_identity(backend, engine):
    """Observe software and processor config without importing/loading models."""
    from importlib import metadata
    packages = {}
    names = ('paddleocr', 'paddlepaddle-gpu', 'paddlex') if engine in {'paddle', 'ppocr'} else ('torch', 'transformers', 'mmcv-full', 'mmdet', 'mmrotate') if engine == 'tonggu' else ('torch', 'transformers')
    for name in names:
        try:
            packages[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            continue
    proc = getattr(backend, 'proc', None)
    info = {'backend': type(backend).__module__ + '.' + type(backend).__qualname__, 'packages': packages, 'observation': 'native_processor_internals_unobserved' if proc is None else 'observed'}
    if proc is not None:
        info['processor'] = type(proc).__module__ + '.' + type(proc).__qualname__
        image_processor = getattr(proc, 'image_processor', None)
        try:
            import json
            info['image_processor'] = json.loads(json.dumps(image_processor.to_dict(), default=str))
        except (AttributeError, TypeError, ValueError):
            info['observation'] = 'processor_config_unobserved'
    return info

def run_prepared_page(backend, image, image_path, cfg, *, page=0, original_record=None):
    """Run all regions transactionally; partial region text never becomes a page."""
    import tempfile
    if cfg.segmentation == 'none' and (not image.info.get('prepared_dataset')):
        return backend.run(str(image_path))
    with tempfile.TemporaryDirectory(prefix='vtm-regions-') as directory:
        return _run_prepared_regions(backend, image, cfg, page=page, directory=Path(directory), original_record=original_record)

def _run_prepared_regions(backend, image, cfg, *, page, directory, original_record=None):
    prepared = image.info.get('prepared_dataset')
    if prepared:
        boxes = [tuple(r['source_box']) for r in prepared['regions']]
        layout = {**prepared['layout'], 'warnings': list(prepared['layout']['warnings']), 'method': 'prepared_' + prepared['crop_type']}
    else:
        boxes, layout = plan_regions(image, cfg, page=page)
    if image.info.get('render', {}).get('downsampled'):
        layout['warnings'].append('page_render_downsampled_before_segmentation; raise max_side')
        if cfg.resolution_strict:
            raise ValueError('Page render was downsampled before segmentation; raise max_side')
    records, texts, merged, offset = ([], [], {}, 0)
    native = prepared['crop_type'] == 'native' and cfg.engine in {'paddle', 'tonggu'} if prepared else layout['method'] == 'native_layout'
    original_preparation = (original_record or {}).get('image_preparation', {})
    original_regions = {r.get('canonical_region_id'): r for r in original_preparation.get('regions', [])}
    processor_matches = original_preparation.get('processor_identity') == processor_identity(backend, cfg.engine)
    for i, box in enumerate(boxes):
        if prepared:
            entry = prepared['regions'][i]['images'][prepared['variant']]
            variant = dataset_load_image(prepared['root'], entry)
            crop, meta = prepare_region(variant, (0, 0, variant.width, variant.height), cfg, native=native)
            meta.update(variant_scale_xy=meta['scale_xy'], variant_downsampled=meta.get('downsampled', False), resized_from_variant=meta['resized'])
            meta.update(crop_size=[box[2] - box[0], box[3] - box[1]], variant_size=list(variant.size), scale_xy=[crop.width / (box[2] - box[0]), crop.height / (box[3] - box[1])], downsampled=crop.width < box[2] - box[0] or crop.height < box[3] - box[1], resized=crop.size != (box[2] - box[0], box[3] - box[1]), variant=prepared['variant'], variant_sha256=entry['sha256'], operation=entry['operation'], canonical_region_id=prepared['regions'][i]['id'], provenance=entry.get('provenance', 'source_observation' if prepared['variant'] == 'original' else 'deterministic_transformation'), warnings=entry.get('warnings', []), model_identity=entry.get('model_identity'), coordinate_mapping_evidence=entry.get('coordinate_mapping_evidence', 'crop_and_scale_geometry'), input_to_source=[(box[2] - box[0]) / crop.width, 0, 0, (box[3] - box[1]) / crop.height, box[0], box[1]])
        else:
            crop, meta = prepare_region(image, box, cfg, native=native)
        path = directory / f'region_{i:03d}.png'
        crop.save(path, format='PNG')
        import time
        ocr_started = time.monotonic()
        previous = original_regions.get(meta.get('canonical_region_id')) if processor_matches else None
        reuse = bool(prepared and meta.get('operation', {}).get('selective_original_fallback') and previous and (previous.get('variant_sha256') == meta.get('variant_sha256')) and (previous.get('page_box') == list(box)) and (previous.get('input_size') == meta.get('input_size')))
        try:
            if reuse:
                import copy
                text, extra = (previous['text'], copy.deepcopy(previous['raw']))
                meta['ocr_reused_from_original'] = True
                meta['original_ocr_seconds'] = previous.get('ocr_seconds')
            else:
                text, extra = backend.run(str(path))
        finally:
            path.unlink(missing_ok=True)
        meta['ocr_seconds'] = round(time.monotonic() - ocr_started, 6)
        observed = extra.get('model_input', {}).get('processed_sizes_wh', [])
        if len(observed) == 1:
            actual_w, actual_h = observed[0]
            meta['processor_size'] = [actual_w, actual_h]
            meta['processor_scale_xy'] = [actual_w / crop.width, actual_h / crop.height]
            meta['processor_downsampled'] = actual_w < crop.width or actual_h < crop.height
            if meta['processor_downsampled']:
                layout['warnings'].append(f'region_{i}: processor_downsampled (includes patch-grid rounding)')
                if cfg.resolution_strict:
                    raise ValueError(f'Processor downsampled region {i}: {crop.size} -> {observed[0]}')
        if cfg.engine == 'ppocr' and (not reuse):
            ordered, order_info = order_ppocr_columns({'text': text, **extra}, mode=('auto' if cfg.segmentation == 'auto' else 'vertical_rtl') if cfg.reading_direction == 'vertical_rtl' else 'detection')
            text = ordered.pop('text')
            extra = {**ordered, 'region_reading_order': order_info}
        records.append({'index': i, 'page_box': list(box), **meta, 'text': text, 'raw': extra})
        mapped = _page_geometry(extra, box, crop.size, offset, i)
        for field in ('character_boxes', 'text_lines', 'blocks', 'boxes', 'spatial_regions'):
            if field in mapped:
                merged.setdefault(field, []).extend(mapped[field])
        offset += len(comparison_text(text))
        texts.append(text)
    for i, block in enumerate(merged.get('blocks', [])):
        block['order'] = i
    issues = sorted({issue for rec in records for issue in rec['raw'].get('quality_issues', [])})
    hit_flags = [rec['raw'].get('hit_max') for rec in records]
    hit_max = True if any((v is True for v in hit_flags)) else None if any((v is None for v in hit_flags)) else False
    if any((rec.get('downsampled') for rec in records)):
        layout['warnings'].append('backend_input_downsampled; inspect regions or enable resolution_strict')
    if any((rec.get('variant_downsampled') for rec in records)):
        layout['warnings'].append('backend_variant_downsampled; inspect selected variant and input budget')
    observation = {'page_size': list(image.size), 'render': image.info.get('render', {}), 'processor_identity': processor_identity(backend, cfg.engine), 'layout': layout, 'regions': records, 'model_min_pixels': cfg.model_min_pixels, 'model_max_pixels': cfg.model_max_pixels, 'processor_observation': 'see region raw.model_input; native internal tensors may be unavailable'}
    if prepared:
        observation['dataset'] = {k: prepared[k] for k in ('snapshot_id', 'source_sha256', 'source', 'variant', 'crop_type', 'acquisition')}
    return ('\n'.join(texts), {**merged, 'hit_max': hit_max, 'quality_issues': issues, 'coordinate_frame': 'input_image', 'image_preparation': observation, 'reading_direction': cfg.reading_direction, 'region_order_applied': True})

def preview_preparation(src, page, cfg, *, output=None):
    """Model-free preview using 1-based PDF page numbers; never starts OCR."""
    from PIL import ImageDraw
    image = render(str(src), page - 1, cfg.dpi, cfg.max_side)
    boxes, layout = plan_regions(image, cfg, page=page - 1)
    preview = image.copy()
    draw = ImageDraw.Draw(preview)
    rows = []
    for i, box in enumerate(boxes):
        _, meta = prepare_region(image, box, cfg, native=layout['method'] == 'native_layout')
        draw.rectangle((box[0], box[1], box[2] - 1, box[3] - 1), outline='red', width=max(1, image.width // 600))
        draw.text((box[0] + 3, box[1] + 3), str(i + 1), fill='red')
        rows.append({'region': i + 1, 'page_box': list(box), **meta})
    if output:
        Path(output).parent.mkdir(parents=True, exist_ok=True)
        preview.save(output)
    return (preview, {'source': str(src), 'page': page, 'page_size': list(image.size), 'render': image.info.get('render', {}), 'engine': cfg.engine, 'layout': layout, 'regions': rows})


import math
PDF_IMAGE_VERSION = 'pdfium-visible-v1'

def pdf_pixel_to_page(size, bbox, rotation):
    """Six affine coefficients: x'=a*x+c*y+e, y'=b*x+d*y+f.

    Input pixels have a top-left origin; output is unrotated PDF canvas units.
    PDFium does not expose /UserUnit: do not label these physical DPI values.
    """
    w, h = size
    l, b, r, t = bbox
    if rotation == 90:
        return [0, (t - b) / w, (r - l) / h, 0, l, b]
    if rotation == 180:
        return [-(r - l) / w, 0, 0, (t - b) / h, r, b]
    if rotation == 270:
        return [0, -(t - b) / w, -(r - l) / h, 0, r, t]
    return [(r - l) / w, 0, 0, -(t - b) / h, l, t]

def pdf_visible_render(page, scale, max_pixels):
    w, h = page.get_size()
    size = [math.ceil(w * scale), math.ceil(h * scale)]
    if min(size) < 1 or size[0] * size[1] > max_pixels:
        raise ValueError(f'Page render {size} exceeds max_source_pixels={max_pixels}; review acquisition settings (source is never silently reduced)')
    bitmap = page.render(scale=scale, draw_annots=True)
    try:
        return bitmap.to_pil().convert('RGB')
    finally:
        bitmap.close()

def acquire_pdf_image(page, *, dpi=200, max_source_pixels=100000000, strategy='auto'):
    """Return owned RGB pixels and evidence for this page's acquisition decision.

    `dpi` is the rendering fallback in pixels per 72 PDF canvas units. For a
    raster-only full-page scan use its native density, including when raw decode
    fails validation. Mixed/vector pages use at least the requested resolution.
    """
    import pypdfium2 as pdfium
    from PIL import ImageChops
    if strategy not in {'auto', 'render'} or dpi <= 0 or max_source_pixels <= 0:
        raise ValueError('Invalid PDF acquisition options')
    width, height = page.get_size()
    bbox, rotation = (list(page.get_bbox()), page.get_rotation())
    objects = list(page.get_objects())
    annotations = pdfium.raw.FPDFPage_GetAnnotCount(page)
    images, density = ([], 0.0)
    for obj in objects:
        if obj.type != pdfium.raw.FPDF_PAGEOBJ_IMAGE:
            continue
        pixels = obj.get_px_size()
        matrix = obj.get_matrix()
        container = obj.container
        while container is not None:
            matrix = matrix.multiply(container.get_matrix())
            container = container.container
        axes = [math.hypot(matrix.a, matrix.b), math.hypot(matrix.c, matrix.d)]
        corners = [matrix.on_point(x, y) for x, y in ((0, 0), (1, 0), (1, 1), (0, 1))]
        bounds = [min((c[0] for c in corners)), min((c[1] for c in corners)), max((c[0] for c in corners)), max((c[1] for c in corners))]
        full = max(0, min(bbox[2], bounds[2]) - max(bbox[0], bounds[0])) * max(0, min(bbox[3], bounds[3]) - max(bbox[1], bounds[1])) >= 0.8 * (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
        native = max(pixels[0] / axes[0], pixels[1] / axes[1]) if min(axes) > 0 else 0
        density = max(density, native)
        images.append({'native_size': list(pixels), 'bounds_pdf': bounds, 'matrix_pdf': [matrix.a, matrix.b, matrix.c, matrix.d, matrix.e, matrix.f], 'level': obj.level, 'full_page_scan': full, 'pixels_per_72_canvas_units': 72 * native})
    reasons = []
    simple = len(objects) == 1 and objects[0].type == pdfium.raw.FPDF_PAGEOBJ_IMAGE
    obj = objects[0] if simple else None
    if not simple:
        reasons.append('not_one_top_level_image; preserve_composition')
    if rotation:
        reasons.append('page_rotation')
    if annotations != 0:
        reasons.append('page_annotations_or_unknown')
    if simple:
        m = obj.get_matrix()
        l, b, r, t = bbox
        if obj.level or abs(m.b) > 1e-06 or abs(m.c) > 1e-06 or (m.a <= 0) or (m.d <= 0) or any((abs(a - v) > 0.001 for a, v in zip(obj.get_bounds(), bbox))) or any((abs(a - v) > 0.001 for a, v in zip([m.e, m.f, m.a, m.d], [l, b, r - l, t - b]))):
            reasons.append('placement_crop_rotation_or_reflection')
        clip = pdfium.raw.FPDFPageObj_GetClipPath(obj)
        if clip and pdfium.raw.FPDFClipPath_CountPaths(clip) > 0:
            reasons.append('image_clip_path')
    image, method, validation = (None, 'pdf_render', None)
    if strategy == 'auto' and (not reasons):
        nw, nh = obj.get_px_size()
        if nw * nh > max_source_pixels:
            raise ValueError('Native image exceeds max_source_pixels')
        scale = math.nextafter(min(nw / width, nh / height), 0)
        if abs(nw / width - nh / height) > 0.0001 * max(nw / width, nh / height):
            reasons.append('anisotropic_native_geometry')
        else:
            bitmap = obj.get_bitmap(render=False)
            try:
                candidate = bitmap.to_pil().convert('RGB')
            finally:
                if hasattr(bitmap, 'warn_on_close'):
                    bitmap.warn_on_close = False
                bitmap.close()
            visible = pdf_visible_render(page, scale, max_source_pixels)
            equal = candidate.size == visible.size and ImageChops.difference(candidate, visible).getbbox() is None
            validation = {'method': 'full_pixel_equality_to_visible_render', 'equal': equal, 'render_size': list(visible.size), 'scale': scale}
            if equal:
                image, method = (candidate, 'native_validated')
            else:
                reasons.append('native_pixels_differ_from_visible_page')
    if image is None:
        raster_only = bool(images) and all((o.type in {pdfium.raw.FPDF_PAGEOBJ_IMAGE, pdfium.raw.FPDF_PAGEOBJ_FORM} for o in objects)) and (annotations == 0)
        scale = density if raster_only and density else max(dpi / 72, density)
        image = pdf_visible_render(page, scale, max_source_pixels)
    else:
        scale = validation['scale']
    meta = {'version': PDF_IMAGE_VERSION, 'method': method, 'size': list(image.size), 'page_size_pdf': [width, height], 'page_bbox_pdf': bbox, 'media_box_pdf': list(page.get_mediabox()), 'crop_box_pdf': list(page.get_cropbox()), 'rotation': rotation, 'object_types': [o.type for o in objects], 'annotations': annotations, 'embedded_images': images, 'requested_dpi': dpi, 'render_scale': scale, 'native_validation': validation, 'fallback_reasons': reasons, 'source_downsampled': False, 'source_to_pdf': pdf_pixel_to_page(image.size, bbox, rotation), 'resolution_unit': 'pixels_per_72_pdf_canvas_units; physical_UserUnit_unavailable', 'warnings': ['low_native_resolution; interpolation_cannot_recover_detail'] if images and max((max(i['native_size']) for i in images)) <= 750 else []}
    return (image, meta)


VARIANT_SPECS = {'docentr_binarization': (1, 'neural_restoration', 'neural_restoration', 'DocEnTR released base/8 H-DIBCO2018 ViT reconstruction', 'learned_prior_sensitivity'), 'degan_binarization': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 1024; Keras Torch backend; threshold .95', 'learned_prior_sensitivity'), 'degan_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 1024 deblur; Keras Torch backend', 'learned_prior_sensitivity'), 'degan_watermark_removal': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 512 watermark removal; explicit region review', 'learned_prior_sensitivity'), 'original': (1, 'source_observation', 'source', 'canonical RGB pixels', 'reference'), 'clahe': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CLAHE Lab L', 'conservative_probe'), 'bicubic_x2': (2, 'deterministic_transformation', 'geometric_scale_interpolation', 'Pillow bicubic x2', 'conservative_probe'), 'bicubic_x4': (4, 'deterministic_transformation', 'geometric_scale_interpolation', 'Pillow bicubic x4', 'conservative_probe'), 'denoised': (1, 'deterministic_transformation', 'denoising_sharpening', 'NLM Lab L', 'glyph_sensitivity'), 'grayscale_luma': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB luminance projection', 'glyph_sensitivity'), 'grayscale_red': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB R projection', 'selective_information'), 'grayscale_green': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB G projection', 'selective_information'), 'grayscale_blue': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB B projection', 'selective_information'), 'lab_l': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab L projection', 'glyph_sensitivity'), 'lab_a': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab a chroma diagnostic', 'selective_information'), 'lab_b': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab b chroma diagnostic', 'selective_information'), 'red_ink': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB red dominance mask', 'selective_information'), 'background_normalized': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'Gaussian background division Lab L', 'glyph_sensitivity'), 'sauvola': (1, 'deterministic_transformation', 'binarization', 'Sauvola reflected local moments', 'glyph_sensitivity'), 'niblack': (1, 'deterministic_transformation', 'binarization', 'Niblack mean-minus-k-std; equality is background', 'glyph_sensitivity'), 'bilateral': (1, 'deterministic_transformation', 'denoising_sharpening', 'bilateral Lab L', 'glyph_sensitivity'), 'unsharp': (1, 'deterministic_transformation', 'denoising_sharpening', 'thresholded unsharp Lab L', 'glyph_sensitivity'), 'retinex_msr': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'three-scale log Retinex intensity; percentile mapping', 'glyph_sensitivity'), 'retinex_msrcp': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'three-scale MSRCP; common RGB gain with saturation bound', 'glyph_sensitivity'), 'wolf': (1, 'deterministic_transformation', 'binarization', 'Wolf-Jolion global-normalized local threshold', 'glyph_sensitivity'), 'nick': (1, 'deterministic_transformation', 'binarization', 'NICK mean+k*sqrt(E[I^2]+variance)', 'glyph_sensitivity'), 'color_pca': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB principal-component projection; luminance sign', 'selective_information'), 'color_gmm': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'diagonal Lab GMM EM; brightest component is paper', 'selective_information'), 'mean_shift': (1, 'deterministic_transformation', 'denoising_sharpening', 'OpenCV spatial-color mean shift, no pyramid by default', 'glyph_sensitivity'), 'anisotropic_diffusion': (1, 'deterministic_transformation', 'denoising_sharpening', 'Perona-Malik exp conductivity; 4 neighbors; no-flux Lab L', 'glyph_sensitivity'), 'background_morphological': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'grayscale closing surface; divide original Lab L', 'glyph_sensitivity'), 'background_guided': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'closing surface self-guided filter; divide original Lab L', 'glyph_sensitivity'), 'background_otsu': (1, 'deterministic_transformation', 'binarization', 'hypothesis: closing surface -> division -> Otsu', 'glyph_sensitivity'), 'realesrgan_x4': (4, 'neural_restoration', 'neural_restoration', 'Real-ESRGAN RRDBNet x4plus', 'learned_prior_sensitivity'), 'tadisr_x4': (4, 'neural_restoration', 'neural_restoration', 'TADiSR CogView4 RealCE joint segmentation decoder', 'learned_prior_sensitivity'), 'difftsr': (None, 'neural_restoration', 'neural_restoration', 'DiffTSR IDM/TDM/MoM and TransOCR text prior', 'learned_prior_sensitivity'), 'unihir_restoration': (1, 'generative_reconstruction', 'generative_reconstruction', 'UniHIR BAGEL MLLM reconstruction', 'generative_sensitivity'), 'docres_appearance': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt appearance', 'learned_prior_sensitivity'), 'docres_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt deblurring', 'learned_prior_sensitivity'), 'docres_deshadowing': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt deshadowing', 'learned_prior_sensitivity'), 'docres_binarization': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt binarization', 'learned_prior_sensitivity'), 'docdiff_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DocDiff coarse predictor and linear-100 residual diffusion', 'learned_prior_sensitivity'), 'restormer_motion_deblurring': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN motion checkpoint', 'learned_prior_sensitivity'), 'restormer_defocus_deblurring': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN single-image defocus checkpoint', 'learned_prior_sensitivity'), 'restormer_real_denoising': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN SIDD checkpoint', 'learned_prior_sensitivity'), 'restormer_gaussian_denoising': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN blind Gaussian checkpoint', 'learned_prior_sensitivity')}
SURVEY_SOURCE_GROUPS = {'39d1b637eed99a6e1981c326e876537ccf96baa897e16f1dca4f6aec01e32800': 'G1', '3e9a2be8a104ad044cd054dee7c0a98422aa57d2c9a4ceaea985b8d9a56a0d0f': 'G1', 'ba8129a51d43a81a2ab566153596ebe482c72a570a0c9f319a69d79c95dde7b5': 'G1', '68b6accd0c037854e41e65ed45fe1ebcd27889999a75ae3df2bfc597275107e7': 'G1', '66fd3918224508aa4dffa5cecc5abe741094c563569a61ea9e976f893811b38e': 'G2', '4b26a2be1db3c95fbd4d4387fd72c417525a813f5ed2c6a6787e02a08163121a': 'G3'}
UNAVAILABLE_RESTORATION_METHODS = {'lwtformer': {'state': 'not_integrated', 'reason': 'Official repository has architecture/test code but no verified released pretrained checkpoint', 'upstream': 'https://github.com/INWLY/LWTformer'}, 'gladunet': {'state': 'not_integrated', 'reason': 'No accessible official inference implementation and pretrained checkpoint verified', 'upstream': 'https://www.nature.com/articles/s40494-025-01965-8'}, 'gatos_energy': {'state': 'not_integrated', 'reason': 'No pinned reliable complete upstream implementation; background_otsu is a separate implemented hypothesis'}, 'single_image_bleed_reduction': {'state': 'not_integrated', 'reason': 'No verified dedicated implementation and no established bleed-through labels in the surveyed corpus'}}

def variant_spec(name):
    if name == 'tadisr_text_mask':
        return dict(method_family='neural_restoration', algorithm='TADiSR predicted segmentation mask', perturbation_class='diagnostic_only', provenance='neural_restoration', scale=4)
    scale, provenance, group, algorithm, interpretation = VARIANT_SPECS[name]
    return dict(method_family=group, algorithm=algorithm, perturbation_class=interpretation, provenance=provenance, scale=scale)

def preparation_applicability(name, book, page, region, config):
    """Explicit analyst conditions only; never invent a degradation detector."""
    rule = config.get('applicability', {}).get(name, {})
    stem = str(book).removesuffix('.pdf')
    if rule.get('books') is not None and (not {str(book), stem}.intersection(rule['books'])):
        return ('not_applicable', 'book outside explicit method applicability')
    if rule.get('crop_types') is not None and region.get('crop_type') not in rule['crop_types']:
        return ('not_applicable', 'crop type outside explicit method applicability')
    review = preparation_region_review(book, page, region['id'], config)
    if rule.get('require_review') and name not in review.get('approved_variants', []):
        return ('not_applicable', 'requires explicit region approval; degradation unverified')
    return ('ready', None)

def preparation_region_review(book, page, region_id, config):
    reviews = config.get('reviewed_regions', {})
    return reviews.get(f'{book}:{page + 1}:{region_id}', reviews.get(f"{str(book).removesuffix('.pdf')}:{page + 1}:{region_id}", {}))

def validate_applicability(config):
    for name, rule in config.get('applicability', {}).items():
        if name not in VARIANT_SPECS or not isinstance(rule, dict) or set(rule) - {'books', 'crop_types', 'require_review'}:
            raise ValueError('Unknown method/applicability option')
        for key in ('books', 'crop_types'):
            if key in rule and (not isinstance(rule[key], list) or any((not isinstance(v, str) for v in rule[key]))):
                raise ValueError('Applicability selections must be lists of strings')
        if set(rule.get('crop_types', [])) - {'page', 'group', 'column'}:
            raise ValueError('Invalid applicability crop_types')
        if 'require_review' in rule and type(rule['require_review']) is not bool:
            raise ValueError('require_review must be boolean')


from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
import hashlib
import io
import json
import math
from pathlib import Path, PurePosixPath
import sys
import time
EXTRA_CLASSICAL = ()
IMAGE_DATASET_FORMAT = 'vtm-image-dataset'
IMAGE_DATASET_VERSION = 1
IMAGE_DATASET_CODE_SIGNATURE = ''

@dataclass
class ImageDatasetConfig:
    dpi: int = 200
    acquisition: str = 'auto'
    max_source_pixels: int = 100000000
    analysis_max_side: int = 1200
    reading_direction: str = 'vertical_rtl'
    columns_per_group: int = 3
    max_columns: int = 64
    column_cuts: list[float] | None = None
    page_column_cuts: dict = field(default_factory=dict)
    ink_threshold: int = 40
    gap_ink_fraction: float = 0.006
    min_gap_fraction: float = 0.006
    variants: list[str] = field(default_factory=lambda: ['original'])
    variant_options: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.dpi <= 0 or self.max_source_pixels < 1 or (not 32 <= self.analysis_max_side <= 1600) or (self.columns_per_group < 1) or (self.max_columns < 1):
            raise ValueError('Invalid preparation dimensions/group limits')
        if self.acquisition not in {'auto', 'render'}:
            raise ValueError('acquisition must be auto or render')
        if self.reading_direction not in {'vertical_rtl', 'horizontal_ltr'}:
            raise ValueError('Invalid reading_direction')
        if not 0 < self.ink_threshold < 255 or not 0 <= self.gap_ink_fraction < 1 or (not 0 < self.min_gap_fraction < 0.5):
            raise ValueError('Invalid projection thresholds')
        if not isinstance(self.variants, list) or 'original' not in self.variants or len(set(self.variants)) != len(self.variants) or set(self.variants) - {'original'}:
            raise ValueError('variants require original and supported classical names')
        if self.variant_options:
            raise ValueError('Source baseline has no transformation options')
        dataset_validate_cuts(self.column_cuts)
        for page, cuts in self.page_column_cuts.items():
            if not str(page).isdigit() or int(page) < 1:
                raise ValueError('page_column_cuts uses 1-based page keys')
            dataset_validate_cuts(cuts)

def dataset_json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False) + '\n').encode('utf-8')

def dataset_hash(value):
    return hashlib.sha256(dataset_json_bytes(value)).hexdigest()

def dataset_file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def dataset_code_signature():
    if IMAGE_DATASET_CODE_SIGNATURE:
        return IMAGE_DATASET_CODE_SIGNATURE
    root = Path(__file__).parent
    return dataset_hash({name: dataset_file_hash(root / name) for name in ('image_dataset.py', 'image_preparation.py', 'pdf_images.py', 'restoration_registry.py')})

def dataset_environment():
    import PIL
    import pypdfium2 as pdfium
    result = {'pillow': PIL.__version__, 'pypdfium2': str(pdfium.PYPDFIUM_INFO), 'pdfium': str(pdfium.PDFIUM_INFO), 'python': sys.version.split()[0]}
    return result

def dataset_path(root, relative):
    relative = str(relative)
    if '\\' in relative or ':' in relative:
        raise ValueError(f'Dataset requires portable relative paths: {relative}')
    parts = PurePosixPath(relative)
    if parts.is_absolute() or '..' in parts.parts or (not parts.parts):
        raise ValueError(f'Invalid dataset path: {relative}')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError(f'Dataset path escapes root: {relative}')
    return path

def dataset_write(root, relative, data):
    path = dataset_path(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_bytes(data)
    temporary.replace(path)
    return {'path': str(relative), 'sha256': hashlib.sha256(data).hexdigest()}

def dataset_read(root, reference):
    data = dataset_path(root, reference['path']).read_bytes()
    if hashlib.sha256(data).hexdigest() != reference['sha256']:
        raise ValueError(f"Dataset checksum mismatch: {reference['path']}")
    return data

def dataset_store_image(root, image):
    image = image.convert('RGB')
    output = io.BytesIO()
    image.save(output, format='PNG', compress_level=6)
    data = output.getvalue()
    sha = hashlib.sha256(data).hexdigest()
    relative = f'objects/{sha[:2]}/{sha}.png'
    path = dataset_path(root, relative)
    if not path.exists() or dataset_file_hash(path) != sha:
        dataset_write(root, relative, data)
    return {'path': relative, 'sha256': sha, 'size': list(image.size), 'pixel_sha256': hashlib.sha256(image.tobytes()).hexdigest()}

def dataset_load_image(root, reference):
    from PIL import Image
    with Image.open(io.BytesIO(dataset_read(root, reference))) as loaded:
        image = loaded.convert('RGB')
    if list(image.size) != reference['size'] or hashlib.sha256(image.tobytes()).hexdigest() != reference['pixel_sha256']:
        raise ValueError(f"Dataset image dimensions/pixels changed: {reference['path']}")
    return image

@contextmanager
def dataset_lock(root):
    """OS lock releases after interruption; no stale PID/lockfile cleanup needed."""
    import os
    Path(root).mkdir(parents=True, exist_ok=True)
    with (Path(root) / 'prepare.lock').open('a+b') as stream:
        if stream.tell() == 0:
            stream.write(b'0')
            stream.flush()
        stream.seek(0)
        import fcntl
        fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        try:
            yield
        finally:
            stream.seek(0)
            fcntl.flock(stream, fcntl.LOCK_UN)

def dataset_validate_cuts(cuts):
    if cuts is not None and (not isinstance(cuts, list) or any((isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (not 0 < v < 1) for v in cuts)) or cuts != sorted(set(cuts))):
        raise ValueError('column_cuts must be unique increasing fractions in (0,1)')

def dataset_partition_qc(regions, size, direction):
    """Exact geometric partition check, independent of OCR/text completeness."""
    w, h = size
    expected = 0 if direction == 'horizontal_ltr' else w
    for index, region in enumerate(regions):
        x1, y1, x2, y2 = region['source_box']
        if not all((isinstance(v, int) for v in (x1, y1, x2, y2))) or not 0 <= x1 < x2 <= w or (y1, y2) != (0, h) or (region['reading_order'] != index):
            raise ValueError('Invalid canonical crop bounds/order')
        start, end = (x1, x2) if direction == 'horizontal_ltr' else (x2, x1)
        if start != expected:
            raise ValueError('Canonical partition has a gap, overlap or inconsistent reading order')
        expected = end
    if not regions or expected != (w if direction == 'horizontal_ltr' else 0):
        raise ValueError('Canonical partition omits page content')

def dataset_plan_layout(image, config, *, page=0):
    """Projection baseline, no model/pixel budget; all coordinates in source pixels."""
    from PIL import Image, ImageDraw
    analysis = image.copy()
    analysis.thumbnail((config.analysis_max_side, config.analysis_max_side), Image.Resampling.LANCZOS)
    options = {key: getattr(config, key) for key in ('ink_threshold', 'gap_ink_fraction', 'min_gap_fraction')}
    mask = projection_mask(analysis, options)
    w, h = image.size
    aw, ah = analysis.size
    manual = config.page_column_cuts.get(str(page + 1), config.column_cuts)
    if manual is not None:
        cuts = [round(v * w) for v in manual]
        method = 'manual_columns'
    elif config.reading_direction == 'vertical_rtl':
        cuts = sorted(set((round(v * w / aw) for v in projection_gaps(analysis, 'x', options))))
        method = 'whitespace_projection'
    else:
        cuts, method = ([], 'whole_page_horizontal')
    if len(set([0, *cuts, w])) != len(cuts) + 2:
        raise ValueError('Column cuts collapse in source space')
    warnings = ['heuristic_layout; geometric_coverage_is_not_text_coverage']
    if len(cuts) + 1 > config.max_columns:
        if manual is not None:
            raise ValueError('Manual layout exceeds max_columns')
        warnings.append('too_many_gutters; retained_whole_page')
        cuts = []
    if not cuts:
        warnings.append('no_partition; whole_page_retained')
    thresholds = sorted(set([max(1, config.ink_threshold - 8), config.ink_threshold, min(254, config.ink_threshold + 8)]))
    found = [projection_gaps(analysis, 'x', {**options, 'ink_threshold': t}) for t in thresholds]
    ink = mask.tobytes()
    top, bottom = (int(0.03 * ah), max(1, int(0.97 * ah)))
    projection = [sum((ink[y * aw + x] > 0 for y in range(top, bottom))) / (bottom - top) for x in range(aw)]
    evidence = []
    for cut in cuts:
        x = min(aw - 1, max(0, round(cut * aw / w)))
        lo, hi = (x, x)
        while lo > 0 and projection[lo - 1] <= config.gap_ink_fraction:
            lo -= 1
        while hi < aw - 1 and projection[hi + 1] <= config.gap_ink_fraction:
            hi += 1
        tolerance = max(2, round(aw * 0.006))
        stability = sum((any((abs(c - x) <= tolerance for c in cs)) for cs in found)) / len(found)
        radius = max(2, round(aw * 0.035))
        left, right = (projection[max(0, x - radius):x], projection[x + 1:min(aw, x + radius + 1)])
        risks = []
        if projection[x] > 0.02:
            risks.append('cut_crosses_analysis_ink; rules_or_strokes_need_review')
        if stability < 2 / 3:
            risks.append('unstable_across_ink_thresholds')
        if not left or not right or min(max(left), max(right)) <= config.gap_ink_fraction * 2:
            risks.append('weak_text_support_on_one_side')
        evidence.append({'source_x': cut, 'analysis_x': x, 'gutter_width_analysis': hi - lo + 1, 'ink_fraction': projection[x], 'threshold_stability': stability, 'warnings': risks})
    full = (0, 0, w, h)
    columns = _partition_box(full, cuts, 'x', config.reading_direction)
    group = config.columns_per_group
    groups = []
    for start in range(0, len(columns), group):
        chunk = columns[start:start + group]
        groups.append((min((b[0] for b in chunk)), 0, max((b[2] for b in chunk)), h))
    regions = []
    for kind, boxes in (('page', [full]), ('group', groups), ('column', columns)):
        for index, box in enumerate(boxes):
            regions.append({'id': f'{kind}_{index:03d}', 'crop_type': kind, 'reading_order': index, 'source_box': list(box), 'overlap': False})
        dataset_partition_qc([r for r in regions if r['crop_type'] == kind], image.size, config.reading_direction)
    preview = analysis.copy()
    draw = ImageDraw.Draw(preview)
    for region in [r for r in regions if r['crop_type'] == 'column']:
        box = region['source_box']
        x1, x2 = (round(box[0] * aw / w), round(box[2] * aw / w))
        draw.rectangle((x1, 0, max(x1, x2 - 1), ah - 1), outline='red', width=1)
        draw.text((x1 + 2, 3), str(region['reading_order'] + 1), fill='red')
    for item in evidence:
        if item['warnings']:
            draw.line((item['analysis_x'], 0, item['analysis_x'], ah - 1), fill='orange', width=2)
    draw.text((3, ah - 14), 'RTL <-' if config.reading_direction == 'vertical_rtl' else 'LTR ->', fill='red')
    layout = {'method': method, 'source_size': [w, h], 'analysis_size': [aw, ah], 'analysis_to_source': [w / aw, 0, 0, h / ah, 0, 0], 'reading_direction': config.reading_direction, 'regions': regions, 'cuts': evidence, 'excluded_regions': [], 'uncertain_regions': [r['id'] for r in regions if r['crop_type'] == 'column'], 'geometric_coverage': 1.0, 'text_coverage': 'unverified', 'warnings': warnings, 'thresholds_tested': thresholds, 'review_status': 'needs_visual_review'}
    return (layout, mask.convert('RGB'), preview)

def dataset_variant(image, name, options):
    if name != 'original':
        raise ValueError(f'Removed image transformation: {name}; use original')
    return (image, {'method': 'identity', 'parameters': {}, 'warnings': []})

def dataset_validate_page(root, record, *, check_pixels=True):
    source = dataset_load_image(root, record['source']) if check_pixels else None
    acquisition = json.loads(dataset_read(root, record['acquisition']))
    layout = json.loads(dataset_read(root, record['layout']))
    if record['source']['size'] != layout['source_size'] or acquisition['size'] != layout['source_size']:
        raise ValueError('Source, acquisition and layout size mismatch')
    dataset_read(root, layout['analysis_mask'])
    dataset_read(root, layout['preview'])
    for kind in ('page', 'group', 'column'):
        regions = [r for r in layout['regions'] if r['crop_type'] == kind]
        dataset_partition_qc(regions, layout['source_size'], layout['reading_direction'])
    if [r['id'] for r in record['regions']] != [r['id'] for r in layout['regions']]:
        raise ValueError('Image dataset omits canonical regions')
    for region, geometry in zip(record['regions'], layout['regions']):
        if any((region[k] != geometry[k] for k in ('source_box', 'crop_type', 'reading_order'))):
            raise ValueError('Region geometry differs from canonical layout')
        if 'original' not in region['images']:
            raise ValueError('Region original variant is missing')
        x1, y1, x2, y2 = region['source_box']
        for name, entry in region['images'].items():
            expected = [(x2 - x1) * (2 if name == 'bicubic_x2' else 1), (y2 - y1) * (2 if name == 'bicubic_x2' else 1)]
            if entry['size'] != expected or entry['image_to_source'] != [(x2 - x1) / expected[0], 0, 0, (y2 - y1) / expected[1], x1, y1]:
                raise ValueError('Variant geometry/transform mismatch')
            if check_pixels:
                image = dataset_load_image(root, entry)
                if name == 'original' and image.tobytes() != source.crop(region['source_box']).tobytes():
                    raise ValueError('Original crop is not pixel-identical to source')
    return layout

def dataset_open(directory):
    path = runtime_path(directory)
    descriptor = path / 'dataset.json' if path.is_dir() else path
    root = descriptor.parent.parent if descriptor.parent.name == 'snapshots' else descriptor.parent
    data = json.loads(descriptor.read_text(encoding='utf-8'))
    if data.get('format') != IMAGE_DATASET_FORMAT or data.get('version') != IMAGE_DATASET_VERSION:
        raise ValueError('Unsupported image dataset format/version')
    body = {k: v for k, v in data.items() if k != 'snapshot_id'}
    if dataset_hash(body) != data['snapshot_id']:
        raise ValueError('Dataset snapshot identity mismatch')
    return (root, data)

def dataset_page(root, descriptor, source_sha, page):
    entries = [p for p in descriptor['pages'] if p['source_sha256'] == source_sha and p['page'] == page]
    if not entries:
        return None
    records = [json.loads(dataset_read(root, e['record'])) for e in entries]
    if len({r['input_key'] for r in records}) != 1:
        raise ValueError('Ambiguous prepared layouts for the same source/page')
    record = records[0]
    if record['source_sha256'] != source_sha or record['page'] != page:
        raise ValueError('Dataset page identity mismatch')
    return record

def bind_image_dataset(cfg, source_sha, pages, *, allow_unavailable=False):
    """Bind an immutable snapshot before looking up OCR checkpoints."""
    if not cfg.image_dataset:
        return
    root, descriptor = dataset_open(cfg.image_dataset)
    snapshot_path = root / f"snapshots/{descriptor['snapshot_id']}.json"
    if not snapshot_path.is_file():
        raise ValueError('Dataset immutable snapshot file is missing')
    if dataset_open(snapshot_path)[1] != descriptor:
        raise ValueError('Immutable dataset snapshot differs from descriptor')
    cfg.image_dataset = str(runtime_path(snapshot_path).resolve())
    cfg.image_dataset_key = descriptor['snapshot_id']
    if descriptor.get('restoration'):
        restoration_index(root, descriptor)
    needs_extension = cfg.image_variant not in {'original', 'clahe', 'bicubic_x2'}
    for page in pages:
        rec = dataset_page(root, descriptor, source_sha, page)
        kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
        if rec and any((cfg.image_variant not in r['images'] for r in rec['regions'] if r['crop_type'] == kind)):
            needs_extension = True
    if needs_extension:
        return bind_restoration_variant(cfg, root, descriptor, source_sha, pages, allow_unavailable=allow_unavailable)
    states = {}
    for page in pages:
        record = dataset_page(root, descriptor, source_sha, page)
        if record is None:
            if allow_unavailable:
                states[page] = {'state': 'missing_page'}
                continue
            if cfg.image_dataset_fallback:
                continue
            raise ValueError(f'Prepared dataset missing source {source_sha}, page {page + 1}')
        layout = dataset_validate_page(root, record)
        if layout['reading_direction'] != cfg.reading_direction:
            raise ValueError('Dataset and OCR reading directions differ; choose a matching preparation profile')
        kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
        for region in record['regions']:
            if region['crop_type'] == kind and cfg.image_variant not in region['images']:
                raise ValueError(f"Missing variant {cfg.image_variant} for {region['id']}")
        states[page] = {'state': 'ok'}
    return bind_restoration_variant(cfg, root, descriptor, source_sha, pages, allow_unavailable=allow_unavailable or cfg.image_dataset_fallback)

def dataset_ocr_page(cfg, source_sha, page):
    """Load prepared source with inference selection attached; never render a PDF."""
    root, descriptor = dataset_open(cfg.image_dataset)
    if cfg.image_dataset_key and descriptor['snapshot_id'] != cfg.image_dataset_key:
        raise ValueError('Prepared input snapshot changed since OCR plan')
    record = dataset_page(root, descriptor, source_sha, page)
    if record is None:
        if cfg.image_dataset_fallback:
            if cfg.image_variant != 'original':
                raise ValueError('Missing prepared variant cannot fall back to original PDF pixels')
            return None
        raise ValueError(f'Prepared dataset missing page {page + 1}')
    layout = dataset_validate_page(root, record, check_pixels=False)
    record = resolve_variant_page(root, descriptor, record, cfg)
    image = dataset_load_image(root, record['source'])
    kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
    regions = [r for r in record['regions'] if r['crop_type'] == kind]
    if not regions:
        raise ValueError(f'Prepared crop type missing: {kind}')
    for region in regions:
        dataset_load_image(root, region['images'][cfg.image_variant])
    image.info['prepared_dataset'] = {'root': str(root), 'snapshot_id': descriptor['snapshot_id'], 'source_sha256': source_sha, 'source': record['source'], 'layout': layout, 'regions': regions, 'variant': cfg.image_variant, 'crop_type': cfg.image_crop_type, 'acquisition': json.loads(dataset_read(root, record['acquisition']))}
    return (image, dataset_path(root, record['source']['path']))

def prepare_image_dataset(input_dir, output_dir, *, config=None, books=None, pages='1-2', limit=2, max_books=1, profile_dir=None, publish=True):
    """Prepare a bounded selection; repeat the same command to resume.

    A completed source/layout/variant is independently reusable. Only after QC
    succeeds is a new dataset.json published. Existing snapshots remain usable.
    """
    config = config or ImageDatasetConfig()
    config.__post_init__()
    if limit is not None and limit < 1 or (max_books is not None and max_books < 1):
        raise ValueError('limit/max_books must be positive or None')
    entries = select_entries(list_pdfs(input_dir), books or [])
    if max_books is not None:
        entries = entries[:max_books]
    selected, remaining = ([], limit)
    for entry in entries:
        wanted = parse_pages(pages, entry['n_pages'])
        if remaining is not None:
            wanted = wanted[:remaining]
            remaining -= len(wanted)
        if wanted:
            selected.append((entry, wanted))
    if not selected:
        raise ValueError('No pages selected')
    root = Path(output_dir).resolve()
    stats = {'pages': 0, 'source_reused': 0, 'layout_reused': 0, 'variants_reused': 0}
    started, environment, code = (time.monotonic(), dataset_environment(), dataset_code_signature())
    page_entries, manifest_rows, configurations = ([], [], {})
    with dataset_lock(root):
        marker = root / 'dataset.json'
        if marker.exists():
            dataset_open(root)
        for entry, wanted in selected:
            import pypdfium2 as pdfium
            cfg = ImageDatasetConfig(**asdict(config))
            if profile_dir:
                profile = json.loads((Path(profile_dir) / (entry['stem'] + '.json')).read_text(encoding='utf-8-sig'))
                for key in ('dpi', 'reading_direction'):
                    if key in profile:
                        setattr(cfg, key, profile[key])
            cfg.__post_init__()
            configurations[entry['name']] = asdict(cfg)
            source_sha = dataset_file_hash(entry['path'])
            doc = pdfium.PdfDocument(entry['path'])
            try:
                doc.init_forms()
                for page in wanted:
                    source_key = dataset_hash({'source_sha256': source_sha, 'page': page, 'acquisition': cfg.acquisition, 'dpi': cfg.dpi, 'max_source_pixels': cfg.max_source_pixels, 'code': code, 'environment': environment})
                    source_receipt = f'sources/{source_key}.json'
                    try:
                        cached = json.loads(dataset_path(root, source_receipt).read_bytes())
                        if cached['key'] != source_key:
                            raise ValueError('Source cache key mismatch')
                        image = dataset_load_image(root, cached['source'])
                        acquisition = json.loads(dataset_read(root, cached['acquisition']))
                        stats['source_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        pdf_page = doc[page]
                        try:
                            image, acquisition = acquire_pdf_image(pdf_page, dpi=cfg.dpi, max_source_pixels=cfg.max_source_pixels, strategy=cfg.acquisition)
                        finally:
                            pdf_page.close()
                        source_asset = dataset_store_image(root, image)
                        acquisition_ref = dataset_write(root, f'sources/{source_key}.acquisition.json', dataset_json_bytes(acquisition))
                        cached = {'key': source_key, 'source': source_asset, 'acquisition': acquisition_ref}
                        dataset_write(root, source_receipt, dataset_json_bytes(cached))
                    layout_options = {k: v for k, v in asdict(cfg).items() if k not in {'dpi', 'acquisition', 'max_source_pixels', 'variants', 'variant_options'}}
                    layout_options['page_column_cuts'] = {str(page + 1): cfg.page_column_cuts.get(str(page + 1), cfg.column_cuts)}
                    layout_key = dataset_hash({'source': cached['source']['sha256'], 'options': layout_options, 'code': code})
                    layout_relative = f'layouts/{layout_key}.json'
                    try:
                        layout_data = dataset_path(root, layout_relative).read_bytes()
                        layout = json.loads(layout_data)
                        if layout['key'] != layout_key:
                            raise ValueError('Layout key mismatch')
                        dataset_read(root, layout['analysis_mask'])
                        dataset_read(root, layout['preview'])
                        stats['layout_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        layout, mask, preview = dataset_plan_layout(image, cfg, page=page)
                        layout.update(key=layout_key, analysis_mask=dataset_store_image(root, mask), preview=dataset_store_image(root, preview))
                        layout_data = dataset_json_bytes(layout)
                        dataset_write(root, layout_relative, layout_data)
                    layout_ref = {'path': layout_relative, 'sha256': hashlib.sha256(layout_data).hexdigest()}
                    regions = []
                    for geometry in layout['regions']:
                        crop = image.crop(geometry['source_box'])
                        variants = {}
                        for name in cfg.variants:
                            options = cfg.variant_options.get(name, {})
                            variant_environment = dict(environment)
                            if name in {'clahe', 'denoised', *EXTRA_CLASSICAL}:
                                import cv2
                                variant_environment['opencv'] = cv2.__version__
                            key = dataset_hash({'source': cached['source']['sha256'], 'box': geometry['source_box'], 'variant': name, 'options': options, 'code': code, 'environment': variant_environment})
                            receipt = f'variants/{key}.json'
                            try:
                                variant = json.loads(dataset_path(root, receipt).read_bytes())
                                if variant['key'] != key:
                                    raise ValueError('Variant cache key mismatch')
                                dataset_load_image(root, variant)
                                stats['variants_reused'] += 1
                            except (OSError, ValueError, KeyError):
                                prepared, operation = dataset_variant(crop, name, options)
                                x1, y1, x2, y2 = geometry['source_box']
                                variant = {**dataset_store_image(root, prepared), 'key': key, 'operation': operation, 'image_to_source': [(x2 - x1) / prepared.width, 0, 0, (y2 - y1) / prepared.height, x1, y1]}
                                dataset_write(root, receipt, dataset_json_bytes(variant))
                            variants[name] = variant
                            manifest_rows.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page + 1, 'page_index': page, 'acquisition_method': acquisition['method'], 'source_size': cached['source']['size'], 'source_image': cached['source'], 'analysis_size': layout['analysis_size'], 'analysis_to_source': layout['analysis_to_source'], 'source_to_pdf': acquisition['source_to_pdf'], 'region_id': geometry['id'], 'crop_type': geometry['crop_type'], 'source_box': geometry['source_box'], 'reading_order': geometry['reading_order'], 'reading_direction': cfg.reading_direction, 'variant': name, **variant, 'qc_warnings': acquisition['warnings'] + layout['warnings'] + [w for c in layout['cuts'] for w in c['warnings']], 'layout': layout_ref, 'acquisition': cached['acquisition'], 'code_sha256': code, 'processing_version': IMAGE_DATASET_VERSION})
                        regions.append({**geometry, 'images': variants})
                    record = {'input_key': dataset_hash({'source': source_key, 'layout': layout_key, 'variants': cfg.variants, 'options': cfg.variant_options}), 'source_sha256': source_sha, 'page': page, 'source': cached['source'], 'acquisition': cached['acquisition'], 'layout': layout_ref, 'regions': regions}
                    dataset_validate_page(root, record)
                    record_bytes = dataset_json_bytes(record)
                    record_key = hashlib.sha256(record_bytes).hexdigest()
                    record_ref = dataset_write(root, f'pages/{record_key}.json', record_bytes)
                    page_entries.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page, 'record': record_ref})
                    stats['pages'] += 1
            finally:
                doc.close()
        manifest_data = b''.join((dataset_json_bytes(row) for row in manifest_rows))
        manifest_sha = hashlib.sha256(manifest_data).hexdigest()
        manifest_ref = dataset_write(root, f'manifests/{manifest_sha}.jsonl', manifest_data)
        descriptor = {'format': IMAGE_DATASET_FORMAT, 'version': IMAGE_DATASET_VERSION, 'pages': page_entries, 'expected_pages': [[e['book'], e['source_sha256'], e['page']] for e in page_entries], 'manifest': manifest_ref, 'configurations': configurations, 'environment': environment, 'code_sha256': code, 'geometry': 'top_left_source_pixels; half_open_boxes', 'ocr_text_coverage': 'unverified'}
        descriptor['snapshot_id'] = dataset_hash(descriptor)
        data = dataset_json_bytes(descriptor)
        snapshot = f"snapshots/{descriptor['snapshot_id']}.json"
        dataset_write(root, snapshot, data)
        if publish:
            dataset_write(root, 'dataset.json', data)
    stats.update(snapshot_id=descriptor['snapshot_id'], dataset=str(root / snapshot), seconds=round(time.monotonic() - started, 3), manifest_rows=len(manifest_rows))
    return stats

def validate_image_dataset(directory):
    root, descriptor = dataset_open(directory)
    snapshot = root / f"snapshots/{descriptor['snapshot_id']}.json"
    if not snapshot.is_file() or dataset_open(snapshot)[1] != descriptor:
        raise ValueError('Immutable dataset snapshot missing or mismatched')
    dataset_read(root, descriptor['manifest'])
    actual = [[e['book'], e['source_sha256'], e['page']] for e in descriptor['pages']]
    if actual != descriptor['expected_pages'] or len(set(map(tuple, actual))) != len(actual):
        raise ValueError('Dataset expected-page coverage mismatch')
    warnings = []
    for entry in descriptor['pages']:
        record = dataset_page(root, descriptor, entry['source_sha256'], entry['page'])
        layout = dataset_validate_page(root, record)
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}: {w}" for w in layout['warnings']))
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}/x{c['source_x']}: {w}" for c in layout['cuts'] for w in c['warnings']))
    if descriptor.get('restoration'):
        validate_restoration_dataset(root, descriptor)
    return {'pages': len(actual), 'snapshot_id': descriptor['snapshot_id'], 'warnings': warnings, 'partial': bool(descriptor.get('restoration', {}).get('partial')), 'geometric_coverage': 1.0, 'ocr_text_coverage': 'unverified'}

def validate_restoration_dataset(root, descriptor):
    """Check every extension asset, including rows not selected for OCR."""
    index, report = restoration_index(root, descriptor)
    for page in descriptor['pages']:
        record = dataset_page(root, descriptor, page['source_sha256'], page['page'])
        for region in record['regions']:
            for variant in report['selected_variants']:
                row = index[page['source_sha256'], page['page'], region['id'], variant]
                if row['state'] != 'ok':
                    continue
                if variant == 'tadisr_text_mask':
                    dataset_load_image(root, row['image']).close()
                else:
                    variant_region_entry(root, descriptor, record, region, variant, index=index, report=report, allow_generated=True)
                for extra in row.get('extras', {}).values():
                    dataset_read(root, extra)
    return (index, report)

def package_image_dataset(directory, archive):
    """ZIP_STORED PNGs; include only this validated snapshot's referenced files."""
    import zipfile
    root, descriptor = dataset_open(directory)
    validate_image_dataset(directory)
    archive = Path(archive).resolve()
    if archive.exists():
        raise FileExistsError(f'Archive already exists: {archive}')
    references = {descriptor['manifest']['path']}
    for entry in descriptor['pages']:
        references.add(entry['record']['path'])
        record = json.loads(dataset_read(root, entry['record']))
        references.update((record[k]['path'] for k in ('source', 'acquisition', 'layout')))
        layout = json.loads(dataset_read(root, record['layout']))
        references.update((layout[k]['path'] for k in ('analysis_mask', 'preview')))
        references.update((v['path'] for r in record['regions'] for v in r['images'].values()))
    if descriptor.get('restoration'):
        index, report = validate_restoration_dataset(root, descriptor)
        references.update((descriptor['restoration']['report']['path'], report['manifest']['path']))
        for row in index.values():
            if row.get('image'):
                references.add(row['image']['path'])
            references.update((ref['path'] for ref in row.get('extras', {}).values()))
    archive.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as output:
        for name in ['dataset.json', f"snapshots/{descriptor['snapshot_id']}.json", *sorted(references)]:
            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
            info.external_attr = 33188 << 16
            data = dataset_json_bytes(descriptor) if name == 'dataset.json' or name.startswith('snapshots/') else dataset_path(root, name).read_bytes()
            output.writestr(info, data)
    return {'archive': str(archive), 'files': len(references) + 2, 'bytes': archive.stat().st_size, 'snapshot_id': descriptor['snapshot_id']}

def add_image_dataset_commands(parser):
    actions = parser.add_subparsers(dest='image_action', required=True)
    prepare = actions.add_parser('prepare', help='prepare once; repeat to resume (default: one PDF/two pages)')
    prepare.add_argument('--input-dir', default='data')
    prepare.add_argument('--output-dir', required=True)
    prepare.add_argument('--config', help='JSON ImageDatasetConfig; no OCR/model settings')
    prepare.add_argument('--books', nargs='+', default=[])
    prepare.add_argument('--pages', default='1-2', help='1-based selection; empty string selects all pages')
    prepare.add_argument('--limit', type=int, default=2, help='total page limit; 0 means unlimited')
    prepare.add_argument('--max-books', type=int, default=1, help='0 means all selected PDFs')
    prepare.add_argument('--profile-dir', help='reuse only dpi/reading_direction from OCR book profiles')
    validate = actions.add_parser('validate', help='verify hashes, pixels, geometry, pages and references')
    validate.add_argument('dataset')
    package = actions.add_parser('package', help='package one validated snapshot for offline Kaggle input')
    package.add_argument('dataset')
    package.add_argument('archive')

def run_image_dataset_command(args):
    if args.image_action == 'prepare':
        config = ImageDatasetConfig(**json.loads(Path(args.config).read_text(encoding='utf-8-sig'))) if args.config else ImageDatasetConfig()
        result = prepare_image_dataset(args.input_dir, args.output_dir, config=config, books=args.books, pages=args.pages, limit=None if args.limit == 0 else args.limit, max_books=None if args.max_books == 0 else args.max_books, profile_dir=args.profile_dir)
    elif args.image_action == 'validate':
        result = validate_image_dataset(args.dataset)
    else:
        result = package_image_dataset(args.dataset, args.archive)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


import json
from contextvars import ContextVar
from functools import wraps
from math import isclose, isfinite
from pathlib import Path
OCR_IMAGE_VARIANTS = {name: (spec[0], spec[1]) for name, spec in VARIANT_SPECS.items()}
_PREPARED_INDEX_CACHE = ContextVar('prepared_index_cache', default=None)

def cache_prepared_index(function):
    """Validate a restoration manifest once per plan, never across notebook calls."""

    @wraps(function)
    def cached(*args, **kwargs):
        if _PREPARED_INDEX_CACHE.get() is not None:
            return function(*args, **kwargs)
        token = _PREPARED_INDEX_CACHE.set({})
        try:
            return function(*args, **kwargs)
        finally:
            _PREPARED_INDEX_CACHE.reset(token)
    return cached

def select_prepared_ocr_variants(dataset, selected='all', *, allow_generated=False):
    """Read method names from Notebook 1, retaining unavailable method states."""
    if selected != 'all':
        if not isinstance(selected, list) or not selected or len(set(selected)) != len(selected):
            raise ValueError("OCR_VARIANTS must be 'all' or a nonempty list of distinct methods")
        names = list(selected)
    else:
        root, descriptor = dataset_open(dataset)
        names = ['original']
        for config in descriptor.get('configurations', {}).values():
            names.extend(config.get('variants', []))
        if descriptor.get('restoration'):
            report = json.loads(dataset_read(root, descriptor['restoration']['report']))
            names.extend(report['selected_variants'])
        names = list(dict.fromkeys((n for n in names if n != 'tadisr_text_mask')))
    if 'original' not in names or set(names) - OCR_IMAGE_VARIANTS.keys():
        raise ValueError('OCR_VARIANTS requires original and supported OCR image methods; masks are excluded')
    generated = set(names) & {'unihir_restoration', 'difftsr'}
    if generated and (not allow_generated):
        raise ValueError(f'{sorted(generated)} require ALLOW_GENERATED_VARIANTS=True, or an explicit OCR_VARIANTS list excluding them')
    return names
RESTORATION_FAILURE_STATES = {'failed', 'missing_assets', 'runtime_incompatible', 'inference_failed', 'timeout', 'resource_exceeded'}

class VariantUnavailable(ValueError):

    def __init__(self, states):
        self.states = states
        super().__init__('Prepared variant unavailable: ' + json.dumps(states, ensure_ascii=False))

def restoration_index(root, descriptor):
    """Validate extension identity and index rows; pixels are verified when consumed."""
    cache = _PREPARED_INDEX_CACHE.get()
    key = (str(Path(root).resolve()), descriptor['snapshot_id'])
    if cache is not None and key in cache:
        return cache[key]
    ext = descriptor.get('restoration')
    if not ext:
        return ({}, None)
    if ext.get('format') != 'vtm-image-restoration' or ext.get('version') != 1:
        raise ValueError('Unsupported restoration extension format/version')
    report = json.loads(dataset_read(root, ext['report']))
    if report.get('format') != ext['format'] or report.get('version') != 1:
        raise ValueError('Restoration report identity mismatch')
    canonical = {k: v for k, v in descriptor.items() if k not in {'restoration', 'snapshot_id'}}
    if dataset_hash(canonical) != report.get('canonical_snapshot'):
        raise ValueError('Restoration canonical snapshot mismatch')
    rows, counts, aliases = ({}, {}, {})
    allowed_aliases = {(p['book'], p['source_sha256'], p['page']) for p in descriptor['pages']}
    for line in dataset_read(root, report['manifest']).splitlines():
        row = json.loads(line)
        key = (row['source_sha256'], row['page_index'], row['region_id'], row['variant'])
        if row.get('page') != row['page_index'] + 1 or (row['book'], row['source_sha256'], row['page_index']) not in allowed_aliases:
            raise ValueError('Duplicate/invalid restoration region identity')
        if row['book'] in aliases.get(key, set()):
            raise ValueError('Duplicate restoration row for the same book alias')
        if key in rows and any((row.get(k) != rows[key].get(k) for k in ('state', 'image', 'canonical_image', 'source_image', 'source_box', 'crop_type', 'reading_order', 'operation', 'parameters', 'provenance'))):
            raise ValueError('Conflicting restoration results for identical source aliases; select one alias per source PDF')
        aliases.setdefault(key, set()).add(row['book'])
        rows.setdefault(key, row)
        state = row.get('state')
        if state not in {'ok', 'not_applicable', 'skipped_by_configuration'} | RESTORATION_FAILURE_STATES:
            raise ValueError('Invalid restoration variant state')
        counts[state] = counts.get(state, 0) + 1
    expected = set()
    for page in descriptor['pages']:
        rec = dataset_page(root, descriptor, page['source_sha256'], page['page'])
        for region in rec['regions']:
            for variant in report['selected_variants']:
                key = (page['source_sha256'], page['page'], region['id'], variant)
                expected.add(key)
                if key not in rows:
                    raise ValueError('Restoration manifest silently omits selected variant/region')
                row = rows[key]
                if page['book'] not in aliases[key] or row.get('source_image_hash') != rec['source']['sha256'] or row.get('source_image') != rec['source'] or (row.get('canonical_image') != region['images']['original']) or any((row.get(k) != region[k] for k in ('source_box', 'crop_type', 'reading_order'))):
                    raise ValueError('Restoration source/crop identity mismatch')
                expected_provenance = 'neural_restoration' if variant == 'tadisr_text_mask' else OCR_IMAGE_VARIANTS.get(variant, (None, None))[1]
                if row.get('provenance') != expected_provenance or expected_provenance is None:
                    raise ValueError('Restoration provenance mismatch')
                if variant == 'original' and row['state'] == 'ok' and any((row['image'].get(k) != region['images']['original'][k] for k in ('sha256', 'pixel_sha256', 'size'))):
                    raise ValueError('Restoration original row differs from canonical pixels')
    if set(rows) != expected or counts != report['states']:
        raise ValueError('Restoration manifest inventory mismatch')
    if any((counts.get(s, 0) for s in RESTORATION_FAILURE_STATES)) and (not report.get('partial')):
        raise ValueError('Failed restoration falsely labelled complete')
    if bool(ext.get('partial')) != bool(report.get('partial')):
        raise ValueError('Restoration extension partial status mismatch')
    if cache is not None:
        cache[key] = (rows, report)
    return (rows, report)

def variant_region_entry(root, descriptor, record, region, variant, *, index=None, report=None, allow_generated=False):
    if variant not in OCR_IMAGE_VARIANTS:
        raise ValueError(f'Unsupported OCR variant: {variant}; diagnostic masks are not OCR images')
    if variant in {'difftsr', 'unihir_restoration'} and (not allow_generated):
        raise ValueError('Generated-content comparison requires explicit opt-in')
    if variant in region['images']:
        entry = region['images'][variant]
        image = dataset_load_image(root, entry)
        return (entry, {'state': 'ok', 'provenance': OCR_IMAGE_VARIANTS[variant][1]})
    if index is None:
        index, report = restoration_index(root, descriptor)
    row = index.get((record['source_sha256'], record['page'], region['id'], variant))
    if row is None:
        raise VariantUnavailable([{'region': region['id'], 'variant': variant, 'state': 'missing_variant'}])
    if row['state'] != 'ok':
        raise VariantUnavailable([{k: row.get(k) for k in ('region_id', 'variant', 'state', 'reason')}])
    if variant == 'degan_watermark_removal' and (not preparation_region_review(row['book'], row['page_index'], row['region_id'], report['configuration']).get('watermark_removal_approved')):
        raise ValueError('DE-GAN watermark-removal region approval is unverified')
    if variant == 'difftsr':
        review = report['configuration'].get('reviewed_regions', {}).get(f"{row['book']}:{row['page']}:{row['region_id']}", {})
        box = region['source_box']
        ratio = (box[2] - box[0]) / (box[3] - box[1])
        if not (review.get('upright_horizontal_single_line') and review.get('allow_simplified_text_prior') and (1 <= review.get('character_count', 0) <= 24) and (abs(ratio / 4 - 1) <= 0.05)):
            raise ValueError('DiffTSR applicability metadata/input contract unverified')
    image = dataset_load_image(root, row['image'])
    box = region['source_box']
    scale = OCR_IMAGE_VARIANTS[variant][0]
    if scale is not None and list(image.size) != [(box[2] - box[0]) * scale, (box[3] - box[1]) * scale]:
        raise ValueError('Restoration scale/dimensions mismatch')
    if variant == 'difftsr' and image.size != (512, 128):
        raise ValueError('DiffTSR processor canvas must be 512x128')
    mapping = [(box[2] - box[0]) / image.width, 0, 0, (box[3] - box[1]) / image.height, box[0], box[1]]
    affine = row['image'].get('image_to_source', [])
    if len(affine) != 6 or any((not isinstance(a, (int, float)) or not isfinite(a) or (not isclose(a, b, abs_tol=1e-09)) for a, b in zip(affine, mapping))):
        raise ValueError('Restoration coordinate mapping mismatch')
    for ref in row.get('extras', {}).values():
        dataset_read(root, ref)
    return ({**row['image'], 'operation': row.get('operation', {}), 'key': row['key'], 'provenance': row['provenance'], 'model_identity': row.get('model_identity'), 'coordinate_mapping_evidence': row.get('coordinate_mapping_evidence'), 'warnings': row.get('warnings', [])}, row)

def resolve_variant_page(root, descriptor, record, cfg, *, index=None, report=None):
    from copy import deepcopy
    record = deepcopy(record)
    kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
    states = []
    selective_originals, restored = (0, 0)
    for region in record['regions']:
        if region['crop_type'] != kind:
            continue
        try:
            entry, _ = variant_region_entry(root, descriptor, record, region, cfg.image_variant, index=index, report=report, allow_generated=cfg.allow_generated_variants)
            region['images'][cfg.image_variant] = entry
            restored += 1
        except VariantUnavailable as exc:
            if cfg.image_variant in {'unihir_restoration', 'tadisr_x4', 'difftsr'} and exc.states and all((s.get('state') == 'not_applicable' and str(s.get('reason', '')).startswith(('selective_unihir:', 'selective_restoration:')) for s in exc.states)):
                region['images'][cfg.image_variant] = {**region['images']['original'], 'provenance': 'source_observation', 'operation': {**region['images']['original'].get('operation', {}), 'selective_original_fallback': True, 'target_variant': cfg.image_variant}, 'warnings': ['selective_restoration: original retained; this region was not restored']}
                selective_originals += 1
                continue
            states.extend(exc.states)
    if states:
        raise VariantUnavailable(states)
    if selective_originals and (not restored):
        raise VariantUnavailable([{'state': 'not_applicable', 'variant': cfg.image_variant, 'reason': 'selective_restoration: no selected region on this page; skip duplicate OCR'}])
    return record

def ocr_input_asset_identity(entry):
    """Keep the existing pixel/config identity independent of new report annotations."""
    result = {k: v for k, v in entry.items() if k in {'sha256', 'pixel_sha256', 'size', 'mode', 'image_to_source', 'provenance', 'model_identity', 'operation', 'coordinate_mapping_evidence'}}
    if 'operation' in result:
        result['operation'] = {k: v for k, v in result['operation'].items() if k not in {'method_family', 'algorithm', 'perturbation_class', 'provenance', 'scale', 'resources', 'threshold_tie_epsilon'}}
    return result

def bind_restoration_variant(cfg, root, descriptor, source_sha, pages, *, allow_unavailable=False):
    """Do not put other selected variants into this variant's cache identity."""
    index, report = restoration_index(root, descriptor)
    states, identity = ({}, [])
    seen_pages = set()
    for page in descriptor['pages']:
        if page['source_sha256'] != source_sha:
            continue
        if page['page'] in seen_pages:
            continue
        seen_pages.add(page['page'])
        record = dataset_page(root, descriptor, source_sha, page['page'])
        layout = dataset_validate_page(root, record)
        if layout['reading_direction'] != cfg.reading_direction:
            raise ValueError('Dataset and OCR reading directions differ')
        try:
            resolved = resolve_variant_page(root, descriptor, record, cfg, index=index, report=report)
            kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
            assets = [{'region': r['id'], 'box': r['source_box'], 'order': r['reading_order'], 'image': r['images'][cfg.image_variant]} for r in resolved['regions'] if r['crop_type'] == kind]
            if not assets:
                raise ValueError('Prepared crop type contains no regions')
            for asset in assets:
                asset['image'] = ocr_input_asset_identity(asset['image'])
            identity.append({'page': page['page'], 'source': record['source'], 'assets': assets})
            states[page['page']] = {'state': 'ok'}
        except VariantUnavailable as exc:
            states[page['page']] = {'state': 'unavailable', 'regions': exc.states}
    selected = {p: states.get(p, {'state': 'missing_page'}) for p in pages}
    if not allow_unavailable and any((s['state'] != 'ok' for s in selected.values())):
        raise VariantUnavailable(selected)
    cfg.image_content_key = dataset_hash({'reader': 'restoration-ocr-v1', 'source_pdf': source_sha, 'variant': cfg.image_variant, 'inputs': identity})
    return selected


IMAGE_DATASET_CODE_SIGNATURE = '3a454bd5aa9586106fc34026efd24aeb0f3d97e2cfc5d2a444b91abcf88be492'

import argparse
import json
import os
import re
import signal
import tempfile
import threading
import time
from contextlib import contextmanager
from pathlib import Path
PADDLE_IGNORE = {'header', 'header_image', 'footer', 'footer_image', 'number', 'footnote', 'aside_text', 'image', 'chart', 'seal'}

def clean_degenerate(s: str, max_unit: int=40, min_chars: int=60, min_cycles: int=4) -> str:
    """Cắt đuôi lặp vô hạn (VLM kẹt vào 1 chu kỳ ngắn rồi lặp tới hết token)."""
    st = s.rstrip()
    for u in range(1, max_unit + 1):
        unit = st[-u:]
        if len(st) < min_chars or not unit.strip():
            continue
        n, i = (0, len(st))
        while i >= u and st[i - u:i] == unit:
            n, i = (n + 1, i - u)
        if n * u >= min_chars and n >= min_cycles:
            return st[:i].rstrip()
    return s

def md_to_lines(md: str) -> list[str]:
    """Markdown/HTML của OCR -> các dòng text thuần (bỏ #, **, ảnh, thẻ bảng, toạ độ hình)."""
    md = re.sub('</t[dh]>', ' ', md)
    md = re.sub('</tr>|<br\\s*/?>', '\n', md)
    md = re.sub('<[^>]+>', '', md)
    md = re.sub('!\\[[^\\]]*\\]\\([^)]*\\)', '', md)
    out = []
    for ln in md.splitlines():
        ln = re.sub('^\\s{0,3}#{1,6}\\s*', '', ln)
        ln = re.sub('^\\s*[-*+]\\s+(?=\\S)', '', ln)
        ln = re.sub('(?:[ \\t,，]*\\(\\d{1,4},\\d{1,4}\\),\\(\\d{1,4},\\d{1,4}\\))+\\s*$', '', ln)
        ln = ln.replace('**', '').replace('__', '').strip()
        if ln.startswith('|') and ln.endswith('|'):
            ln = ' '.join((c.strip() for c in ln.strip('|').split('|') if c.strip()))
        if ln and (not re.fullmatch('[-|:\\s]+', ln)):
            out.append(ln)
    return out

def page_lines(rec: dict) -> list[str]:
    """1 trang OCR -> các dòng chữ theo thứ tự đọc."""
    if 'blocks' not in rec:
        return md_to_lines(rec['text'])
    return [ln for b in ordered_page_blocks(rec['blocks']) for ln in md_to_lines(b['text'])]

def ordered_page_blocks(blocks):
    """Use the same block sequence for exported text and segmented geometry."""
    ordered = sorted((b for b in blocks if b['order'] is not None), key=lambda b: b['order'])
    center = lambda b: ((b['box'][0] + b['box'][2]) / 2, (b['box'][1] + b['box'][3]) / 2)
    for b in (b for b in blocks if b['order'] is None):
        if not ordered:
            ordered.append(b)
            continue
        cx, cy = center(b)
        k = min(range(len(ordered)), key=lambda i: (center(ordered[i])[0] - cx) ** 2 + (center(ordered[i])[1] - cy) ** 2)
        ordered.insert(k + (cy > center(ordered[k])[1]), b)
    return ordered
_DOCS = {}

def render(src: str, page: int, dpi: int=200, max_side: int=1600):
    """Trang PDF -> ảnh PIL (cạnh dài tối đa max_side)."""
    import pypdfium2 as pdfium
    if src not in _DOCS:
        _DOCS[src] = pdfium.PdfDocument(src)
    pdf_page = _DOCS[src][page]
    bitmap = None
    try:
        bitmap = pdf_page.render(scale=dpi / 72)
        img = bitmap.to_pil().convert('RGB')
    finally:
        if bitmap is not None:
            bitmap.close()
        pdf_page.close()
    original_size = img.size
    if max(img.size) > max_side:
        s = max_side / max(img.size)
        from PIL import Image
        img = img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))), Image.Resampling.LANCZOS)
    img.info['render'] = {'dpi': dpi, 'uncapped_size': list(original_size), 'size': list(img.size), 'downsampled': img.size != original_size, 'max_side': max_side}
    return img

class Dummy:
    """Engine không cần model, dùng để test toàn bộ pipeline trên CPU.

    Trả về đủ chữ Hán để bước dịch không lọc mất file.
    """

    def __init__(self, a):
        self.a = a

    def run(self, img_path):
        from PIL import Image
        with Image.open(img_path) as im:
            w, h = im.size
        text = f'測試頁面 {w}x{h}\n' + '黃帝內經素問' * 10
        return (text, {'hit_max': False, 'dummy': True})

class Hunyuan:

    def __init__(self, a):
        started = time.monotonic()
        print('[model] Hunyuan: bắt đầu import torch/transformers', flush=True)
        import torch
        from transformers import AutoProcessor
        try:
            from transformers import HunYuanVLForConditionalGeneration as M
        except ImportError:
            from transformers import AutoModelForImageTextToText as M
        self.torch, self.max_new, self.rep = (torch, a.max_new, a.rep_penalty)
        self.prompt = a.prompt or HUNYUAN_PROMPT
        self.model_min_pixels = getattr(a, 'model_min_pixels', None)
        self.model_max_pixels = getattr(a, 'model_max_pixels', None)
        self.dev = self._device(a.device)
        dtype = {'fp32': torch.float32, 'fp16': torch.float16, 'bf16': torch.bfloat16}[a.dtype]
        print(f'[model] Hunyuan: bắt đầu tải/đọc model {a.model} (dùng cache nếu có)', flush=True)
        revision = {'revision': a.model_revision} if getattr(a, 'model_revision', '') else {}
        self.proc = AutoProcessor.from_pretrained(a.model, trust_remote_code=True, use_fast=False, **revision)
        self.processor_default_size = dict(self.proc.image_processor.size)
        attention = getattr(a, 'attention_backend', 'auto')
        load_options = {} if attention == 'auto' else {'attn_implementation': attention}
        self.model = M.from_pretrained(a.model, dtype=dtype, trust_remote_code=True, **load_options, **revision).to(self.dev).eval()
        config = self.model.config
        self.attention_backend = {'requested': attention, 'effective': getattr(config, '_attn_implementation', None), 'text': getattr(getattr(config, 'text_config', None), '_attn_implementation', None), 'vision': getattr(getattr(config, 'vision_config', None), '_attn_implementation', None)}
        print('[model] Hunyuan attention:', self.attention_backend, flush=True)
        print(f'[model] Hunyuan: model sẵn sàng trên {self.dev}, sau {time.monotonic() - started:.1f}s', flush=True)

    @staticmethod
    def _device(requested: str) -> str:
        import torch
        if requested == 'auto':
            return 'cuda' if torch.cuda.is_available() else 'cpu'
        return requested

    def run(self, img_path):
        torch = self.torch
        processor = self.proc.image_processor
        size = dict(self.processor_default_size)
        if self.model_min_pixels is not None:
            size['shortest_edge'] = self.model_min_pixels
        if self.model_max_pixels is not None:
            size['longest_edge'] = self.model_max_pixels
        processor.size = size
        msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': img_path}, {'type': 'text', 'text': self.prompt}]}]
        inputs = self.proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt').to(self.dev)
        grid = inputs.get('image_grid_thw')
        grids = grid.detach().cpu().tolist() if grid is not None else []
        patch = getattr(processor, 'patch_size', 16)
        model_input = {'processor': type(processor).__name__, 'effective_size': size, 'attention_backend': getattr(self, 'attention_backend', {'requested': 'auto', 'effective': None}), 'image_grid_thw': grids, 'processed_sizes_wh': [[int(row[2]) * patch, int(row[1]) * patch] for row in grids], 'pixel_values_shape': list(inputs['pixel_values'].shape) if 'pixel_values' in inputs else None, 'observation': 'processor_tensor' if grids else 'unavailable'}
        for k, v in list(inputs.items()):
            if torch.is_tensor(v) and v.is_floating_point():
                inputs[k] = v.to(self.model.dtype)
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new, do_sample=False, repetition_penalty=self.rep)
        gen = out[:, inputs['input_ids'].shape[1]:]
        text = self.proc.batch_decode(gen, skip_special_tokens=True)[0]
        hit_max = int(gen.shape[1]) >= self.max_new
        return (text, {'hit_max': hit_max, 'model_input': model_input, 'quality_issues': ocr_quality_issues(text, {'hit_max': hit_max})})

class Paddle:

    def __init__(self, a):
        started = time.monotonic()
        print('[model] Paddle: bắt đầu import paddleocr', flush=True)
        from paddleocr import PaddleOCRVL
        print(f'[model] Paddle: bắt đầu tải/khởi tạo PaddleOCR-VL {a.paddle_version} (dùng cache nếu có)', flush=True)
        options = json.loads(a.paddle_args)
        self.spatial_frame_safe = all((options.get(k) is False for k in ('use_doc_orientation_classify', 'use_doc_unwarping')))
        self.pipe = PaddleOCRVL(pipeline_version=a.paddle_version, **options)
        self.max_new = a.max_new
        self.model_min_pixels = getattr(a, 'model_min_pixels', None)
        self.model_max_pixels = getattr(a, 'model_max_pixels', None)
        print(f'[model] Paddle: model sẵn sàng, sau {time.monotonic() - started:.1f}s', flush=True)

    def run(self, img_path):
        kwargs = {'max_new_tokens': self.max_new}
        for key, value in (('min_pixels', self.model_min_pixels), ('max_pixels', self.model_max_pixels)):
            if value is not None:
                kwargs[key] = value
        res = list(self.pipe.predict(img_path, **kwargs))[0]
        text, extra = self.parse(res.json)
        return (text, {**extra, 'coordinate_frame': 'input_image' if self.spatial_frame_safe else 'unverified_native_transform', 'model_input': {'requested': kwargs, 'observation': 'native_internal_unavailable'}})

    @staticmethod
    def parse(j):
        """res.json của PaddleOCR-VL -> (text, {"blocks": vùng đã đọc chữ, "boxes": mọi khung layout})."""
        j = j.get('res', j) if isinstance(j, dict) else {}
        blocks = [{'label': b.get('block_label'), 'box': [round(x) for x in b.get('block_bbox', [0, 0, 0, 0])], 'order': b.get('block_order'), 'text': b.get('block_content') or ''} for b in j.get('parsing_res_list', []) if b.get('block_label') not in PADDLE_IGNORE]
        blocks = [b for b in blocks if b['text']]
        text = '\n'.join(page_lines({'blocks': blocks}))
        boxes = [{'label': b['label'], 'box': [round(x) for x in b['coordinate']], 'order': b.get('order'), 'score': round(b.get('score', 0), 2)} for b in (j.get('layout_det_res') or {}).get('boxes', [])]
        return (text, {'blocks': blocks, 'boxes': boxes, 'quality_issues': ocr_quality_issues(text, {})})

class PPOCR:
    """PP-OCRv6 medium, retaining genuine single-character word boxes only."""

    def __init__(self, a):
        from paddleocr import PaddleOCR
        options = dict(ocr_version='PP-OCRv6', text_detection_model_name='PP-OCRv6_medium_det', text_recognition_model_name='PP-OCRv6_medium_rec', use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)
        options.update(json.loads(a.paddle_args))
        for key in ('use_doc_orientation_classify', 'use_doc_unwarping', 'use_textline_orientation'):
            if options.get(key):
                raise ValueError(f'{key} must be False for source-image character coordinates')
        self.pipe = PaddleOCR(**options)

    def run(self, img_path):
        kwargs = {'return_word_box': True}
        if getattr(self, 'preserve_input_resolution', False):
            from PIL import Image
            with Image.open(img_path) as image:
                kwargs.update(text_det_limit_side_len=(max(image.size) + 31) // 32 * 32, text_det_limit_type='max')
        results = list(self.pipe.predict(img_path, **kwargs))
        if len(results) != 1:
            raise ValueError('Expected exactly one PP-OCR page result')
        text, extra = self.parse(results[0].json)
        return (text, {**extra, 'model_input': {'requested': kwargs, 'observation': 'native_internal_unavailable'}})

    @staticmethod
    def parse(j):
        j = j.get('res', j)
        texts = j.get('rec_texts', [])
        polygons = j.get('rec_polys', [])
        words = j.get('text_word', [])
        regions = j.get('text_word_region', [])
        rectangles = j.get('text_word_boxes', [])
        characters, lines, offset = ([], [], 0)
        for index, text in enumerate(texts):
            filtered = comparison_text(text)
            line_words = words[index] if index < len(words) else []
            line_boxes = regions[index] if index < len(regions) else rectangles[index] if index < len(rectangles) else []
            if len(line_words) == len(line_boxes) and comparison_text(''.join(line_words)) == filtered:
                local_offset = 0
                for word, polygon in zip(line_words, line_boxes):
                    word = comparison_text(word)
                    if len(word) == 1:
                        characters.append({'offset': offset + local_offset, 'text': word, 'polygon': polygon, 'granularity': 'character', 'method': 'ppocr_word_box', 'estimated': True})
                    local_offset += len(word)
            lines.append({'text': text, 'polygon': polygons[index] if index < len(polygons) else None})
            offset += len(filtered)
        return ('\n'.join(texts), {'character_boxes': characters, 'text_lines': lines, 'coordinate_frame': 'input_image', 'hit_max': False})

class TongGu:
    """Use the official TongGu layout/block pipeline in its configured environment."""

    def __init__(self, a):
        import sys
        root = Path(a.tonggu_repo).expanduser().resolve() if a.tonggu_repo else None
        if root is None or not (root / 'tongguocr' / 'pipeline.py').is_file():
            raise ValueError('Set tonggu_repo to the official TongGuOCR checkout with installed dependencies/weights')
        sys.path.insert(0, str(root))
        from tongguocr.config import load_config
        from tongguocr.pipeline import run_pipeline_on_image
        self.predict = run_pipeline_on_image
        self.config = load_config(a.tonggu_config or None)
        self.default_image_limits = {key: self.config['ocr']['inference'].get(key, default) for key, default in (('image_min_pixels', 1024), ('image_max_pixels', 1048576))}
        self.model_min_pixels = getattr(a, 'model_min_pixels', None)
        self.model_max_pixels = getattr(a, 'model_max_pixels', None)
        for group, key in (('layout_inference', 'detector_checkpoint_path'), ('layout_inference', 'detector_config_path')):
            path = Path(self.config[group][key])
            if not path.is_absolute():
                self.config[group][key] = str(root / path)
        model_path = Path(self.config['ocr']['transformers']['model_path'])
        if not model_path.is_absolute():
            self.config['ocr']['transformers']['model_path'] = str(root / model_path)
        self.max_new = a.max_new
        self.temporary = tempfile.TemporaryDirectory(prefix='vtm-tonggu-')
        self.prepare_models()

    def prepare_models(self):
        """Fill the pinned upstream caches before the per-page deadline starts."""
        from tongguocr.layout.extractor import LineBBoxExtractor, _EXTRACTOR_CACHE
        started = time.monotonic()
        print('[model] TongGu: bắt đầu nạp detector layout', flush=True)
        layout = self.config['layout_inference']
        key = '|'.join((str(layout[field]) for field in ('detector_config_path', 'detector_checkpoint_path', 'device', 'detector_score_threshold')))
        if key not in _EXTRACTOR_CACHE:
            _EXTRACTOR_CACHE[key] = LineBBoxExtractor(self.config)
        if self.config['ocr']['backend'] == 'transformers':
            from tongguocr.ocr.transformers_backend import _load_transformers_runtime
            print('[model] TongGu: bắt đầu nạp model OCR', flush=True)
            backend = self.config['ocr']['transformers']
            _load_transformers_runtime(backend['model_path'], backend.get('device', 'cuda:0'))
        print(f'[model] TongGu: model sẵn sàng, sau {time.monotonic() - started:.1f}s; thời gian nạp không tính vào timeout trang.', flush=True)

    def run(self, img_path):
        self.config['ocr']['inference']['max_tokens'] = self.max_new
        for key, value in (('image_min_pixels', getattr(self, 'model_min_pixels', None)), ('image_max_pixels', getattr(self, 'model_max_pixels', None))):
            if value is not None or hasattr(self, 'default_image_limits'):
                self.config['ocr']['inference'][key] = value if value is not None else self.default_image_limits[key]
        output = Path(self.temporary.name) / str(time.time_ns())
        result = self.predict(img_path, output, self.config, visualize=False, crop_blocks=True)
        if result.status != 'ok':
            raise RuntimeError(f'TongGuOCR returned status={result.status}')
        recognized = {b.block_id: b.ocr_text for b in getattr(result, 'ocr_blocks', [])}
        spatial = [{'box': b['bbox'], 'text': recognized[b['block_id']], 'granularity': 'block', 'method': 'tonggu_recognized_block', 'sparse_canvas_applied': b.get('sparse_canvas_applied', False)} for b in result.blocks if b.get('block_id') in recognized and recognized[b['block_id']]]
        return (result.ocr_text, {'tonggu_layout': result.line_bboxes, 'tonggu_blocks': result.blocks, 'spatial_regions': spatial, 'coordinate_frame': 'input_image', 'hit_max': None, 'character_boxes': [], 'model_input': {'requested': {key: self.config['ocr']['inference'].get(key) for key in ('image_min_pixels', 'image_max_pixels')}, 'observation': 'native_internal_unavailable'}, 'quality_issues': ocr_quality_issues(result.ocr_text, {})})
ENGINES = {'hunyuan': Hunyuan, 'paddle': Paddle, 'ppocr': PPOCR, 'tonggu': TongGu, 'dummy': Dummy}
PAGE_TIMEOUT_EXIT = 124

@contextmanager
def page_deadline(seconds: int, on_hard_timeout):
    """Unix: interrupt the page; Windows: stop the worker so CUDA cannot keep running."""
    if seconds <= 0:
        yield
        return
    if hasattr(signal, 'SIGALRM'):

        def expire(signum, frame):
            raise TimeoutError(f'quá {seconds}s')
        previous = signal.signal(signal.SIGALRM, expire)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, previous)
    else:

        def expire():
            try:
                on_hard_timeout()
            finally:
                os._exit(PAGE_TIMEOUT_EXIT)
        timer = threading.Timer(seconds, expire)
        timer.daemon = True
        timer.start()
        try:
            yield
        finally:
            timer.cancel()
            timer.join()

def result_file(out: Path, book: str, name: str, shard: int) -> Path:
    return out / book / f'{name}_shard{shard}.jsonl'

def load_results(out: Path, book: str, name: str) -> dict[int, dict]:
    """{trang: bản ghi} của 1 sách, gộp mọi shard (dòng hỏng thì bỏ)."""
    recs = {}
    for f in sorted((out / book).glob(f'{name}_shard*.jsonl')):
        with open(f, encoding='utf-8') as stream:
            for line in stream:
                try:
                    r = json.loads(line)
                    recs[r['page']] = r
                except (json.JSONDecodeError, KeyError):
                    pass
    return recs


worker_ENGINES = ENGINES
worker__DOCS = _DOCS
worker_load_results = load_results
worker_page_deadline = page_deadline
worker_page_lines = page_lines
worker_render = render
worker_result_file = result_file

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from fnmatch import fnmatch
from datetime import datetime, timezone
from pathlib import Path
from tqdm.auto import tqdm

def export_ocr(run_dir: str | Path, books: dict, name: str) -> dict:
    run_dir = Path(run_dir)
    raw_dir, txt_dir = (run_dir / 'ocr_raw', run_dir / 'txt')
    txt_dir.mkdir(parents=True, exist_ok=True)
    total_ok = total = total_missing = 0
    for b, (_, sel) in books.items():
        recs = worker_load_results(raw_dir, b, name)
        pages = ['\n'.join(worker_page_lines(recs[p])) for p in sel if p in recs]
        (txt_dir / f'{b}.txt').write_text('\n\n'.join((p for p in pages if p)) + '\n', encoding='utf-8')
        miss = len(sel) - len(pages)
        total_ok += len(pages)
        total += len(sel)
        total_missing += miss
        print(f'{b:<50} {len(pages)}/{len(sel)} trang' + (f'  -- THIẾU {miss} trang' if miss else ''))
    return {'pages': total, 'done': total_ok, 'missing': total_missing}
from contextlib import contextmanager
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
import uuid
from threading import Lock, Event
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.' + uuid.uuid4().hex + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)
import os
from pathlib import Path, PureWindowsPath
import re


def runtime_path(value: str | Path) -> Path:
    return Path(value)
import json
from pathlib import Path
import sqlite3
from contextlib import contextmanager

class GPUPageQueue:

    def __init__(self, path):
        self.path = str(path)

    @contextmanager
    def connect(self):
        db = sqlite3.connect(self.path, timeout=30)
        db.execute('PRAGMA busy_timeout=30000')
        try:
            with db:
                yield db
        finally:
            db.close()

    def initialize(self, jobs, totals):
        """Called once per notebook attempt, using only pages without checkpoints."""
        Path(self.path).parent.mkdir(parents=True, exist_ok=True)
        with self.connect() as db:
            db.execute('PRAGMA journal_mode=WAL')
            db.execute('CREATE TABLE jobs (id INTEGER PRIMARY KEY, engine TEXT, book TEXT, page INTEGER, payload TEXT, state TEXT, owner TEXT, seconds REAL, UNIQUE(engine,book,page))')
            db.execute('CREATE INDEX claim_job ON jobs(engine,book,state,id)')
            db.execute('CREATE TABLE models (engine TEXT PRIMARY KEY, total INTEGER, cached INTEGER, load_seconds REAL DEFAULT 0)')
            for engine, (total, cached) in totals.items():
                db.execute('INSERT INTO models(engine,total,cached) VALUES (?,?,?)', (engine, total, cached))
            db.executemany('INSERT INTO jobs(engine,book,page,payload,state) VALUES (?,?,?,?,?)', ((engine, book, task['page'], json.dumps(task), 'pending') for engine, book, task in jobs))

    def claim(self, engine, book, owner):
        with self.connect() as db:
            db.execute('BEGIN IMMEDIATE')
            row = db.execute("SELECT id,payload FROM jobs WHERE engine=? AND book=? AND (state='pending' OR (state='reserved' AND owner=?)) ORDER BY CASE state WHEN 'reserved' THEN 0 ELSE 1 END,id LIMIT 1", (engine, book, owner)).fetchone()
            if row:
                db.execute("UPDATE jobs SET state='running',owner=? WHERE id=?", (owner, row[0]))
            db.commit()
            return json.loads(row[1]) if row else None

    def reserve(self, engine, owner):
        """Keep one page for an incoming helper while it imports/loads the model."""
        with self.connect() as db:
            db.execute('BEGIN IMMEDIATE')
            row = db.execute("SELECT id FROM jobs WHERE engine=? AND state='pending' ORDER BY id LIMIT 1", (engine,)).fetchone()
            if row:
                db.execute("UPDATE jobs SET state='reserved',owner=? WHERE id=?", (owner, row[0]))
            return row is not None

    def pages(self, engine, book, owner):
        while True:
            task = self.claim(engine, book, owner)
            if task is None:
                return
            yield task

    def finish(self, engine, book, page, owner, *, success, seconds=0):
        with self.connect() as db:
            db.execute("UPDATE jobs SET state=?,seconds=? WHERE engine=? AND book=? AND page=? AND owner=? AND state='running'", ('done' if success else 'failed', seconds, engine, book, page, owner))

    def abandon(self, owner):
        with self.connect() as db:
            db.execute("UPDATE jobs SET state='failed' WHERE owner=? AND state IN ('running','reserved')", (owner,))

    def loaded(self, engine, seconds):
        with self.connect() as db:
            db.execute('UPDATE models SET load_seconds=? WHERE engine=?', (seconds, engine))

    def snapshot(self):
        with self.connect() as db:
            models = {n: dict(pages=total, done=cached, missing=total - cached, pending=0, running=0, failed=0, load_seconds=load, page_seconds=0) for n, total, cached, load in db.execute('SELECT * FROM models')}
            for engine, state, count, seconds in db.execute('SELECT engine,state,COUNT(*),AVG(seconds) FROM jobs GROUP BY engine,state'):
                if state == 'done':
                    models[engine]['done'] += count
                    models[engine]['missing'] -= count
                    models[engine]['page_seconds'] = seconds or 0
                else:
                    key = 'running' if state == 'reserved' else state
                    models[engine][key] += count
            return models
import argparse
import gc
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import time
import traceback
from types import SimpleNamespace
import uuid
from contextlib import contextmanager, nullcontext

def create_engine(engine: str, cfg: OCRConfig):
    args = SimpleNamespace(model=cfg.model, dtype=cfg.dtype, device=cfg.device, attention_backend=cfg.attention_backend, model_revision=cfg.model_revision, tonggu_repo=cfg.tonggu_repo, tonggu_config=cfg.tonggu_config, max_new=cfg.max_new_tokens, rep_penalty=cfg.repetition_penalty, prompt=cfg.prompt, paddle_version=cfg.paddle_version, model_min_pixels=cfg.model_min_pixels, model_max_pixels=cfg.model_max_pixels, paddle_args=json.dumps(cfg.paddle_args))
    return worker_ENGINES[engine](args)

def append_record(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size:
        with path.open('r+b') as stream:
            stream.seek(-1, os.SEEK_END)
            if stream.read(1) != b'\n':
                stream.seek(0)
                contents = stream.read()
                boundary = contents.rfind(b'\n') + 1
                tail = contents[boundary:]
                try:
                    json.loads(tail)
                except (json.JSONDecodeError, UnicodeDecodeError):
                    path.with_suffix(f'.interrupted-{time.time_ns()}.log').write_bytes(tail)
                    stream.truncate(boundary)
                else:
                    stream.seek(0, os.SEEK_END)
                    stream.write(b'\n')
    with path.open('ab') as stream:
        stream.write((json.dumps(record, ensure_ascii=False) + '\n').encode('utf-8'))
        stream.flush()
        os.fsync(stream.fileno())

def corpus_book_plan(pair, engine, attempt):
    directory = runtime_path(pair[engine])
    manifest_file = directory / 'run_manifest.json'
    manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
    cfg = OCRConfig(**manifest['config'])
    name = cfg.run_name or engine
    raw = directory / 'ocr_raw'
    raw_book = raw / pair['book']
    raw_book.mkdir(parents=True, exist_ok=True)
    tasks = [json.loads(line) for line in (directory / 'tasks.jsonl').read_text(encoding='utf-8').splitlines() if line]
    done = worker_load_results(raw, pair['book'], name)
    failed = set()
    errors = raw_book / f'{name}_errors.jsonl'
    if errors.exists():
        for line in errors.read_text(encoding='utf-8').splitlines():
            try:
                error = json.loads(line)
                if error.get('attempt') == attempt:
                    failed.add(int(error['page']))
            except (json.JSONDecodeError, KeyError, ValueError):
                pass
    todo = [task for task in tasks if task['page'] not in done and task['page'] not in failed and (task.get('prepared_input', {}).get('state', 'ok') == 'ok')]
    if pair.get('original_checkpoint_readonly'):
        todo = []
    return dict(pair=pair, directory=directory, manifest_file=manifest_file, manifest=manifest, cfg=cfg, name=name, raw=raw, errors=errors, tasks=tasks, done=done, todo=todo)

@contextmanager
def corpus_render_lock(path):
    """Serialize cache misses for a page so models share a single PDF render."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a+b') as stream:
        if stream.tell() == 0:
            stream.write(b'0')
            stream.flush()
        stream.seek(0)
        import fcntl
        fcntl.flock(stream, fcntl.LOCK_EX)
        try:
            yield
        finally:
            stream.seek(0)
            fcntl.flock(stream, fcntl.LOCK_UN)

def corpus_page_image(campaign, manifest, book, task, cfg, render, temporary):
    """Share lossless renders across models; a damaged cache is rendered again."""
    from PIL import Image
    source_sha = manifest.get('books', {}).get(book, {}).get('sha256')
    if cfg.image_dataset:
        prepared = dataset_ocr_page(cfg, source_sha, task['page'])
        if prepared is not None:
            image, path = prepared
            return (image, path, True)
    key = {'source_sha256': source_sha, 'page': task['page'], 'dpi': cfg.dpi, 'max_side': cfg.max_side, 'renderer': 'pdfium-rgb-lanczos-v2'}
    cached = bool(campaign.get('cache_renders') and source_sha)
    if cached:
        cache_root = runtime_path(campaign['render_cache']).resolve()
        if not cache_root.is_relative_to(runtime_path(cfg.workdir).resolve()):
            raise ValueError('Render cache must stay within the campaign workdir')
        path = cache_root / source_sha / f'{cfg.dpi}_{cfg.max_side}' / f"p{task['page'] + 1:05d}.png"
    else:
        path = temporary
    with corpus_render_lock(path.with_suffix('.lock')) if cached else nullcontext():
        if cached:
            metadata = path.with_suffix('.json')
            try:
                if path.is_symlink() or metadata.is_symlink():
                    raise ValueError('Cache symlink')
                info = json.loads(metadata.read_text(encoding='utf-8'))
                data = path.read_bytes()
                if info['key'] != key or hashlib.sha256(data).hexdigest() != info['sha256']:
                    raise ValueError('Render cache mismatch')
                with Image.open(io.BytesIO(data)) as loaded:
                    image = loaded.convert('RGB')
                if list(image.size) != info['size']:
                    raise ValueError('Render cache size mismatch')
                image.info['render'] = info.get('render', {})
                return (image, path, True)
            except (OSError, ValueError, KeyError):
                pass
        image = render(str(runtime_path(task['src'])), task['page'], cfg.dpi, cfg.max_side)
        path.parent.mkdir(parents=True, exist_ok=True)
        destination = path.with_suffix('.' + uuid.uuid4().hex + '.png.tmp')
        image.save(destination, format='PNG', compress_level=1)
        destination.replace(path)
        if cached:
            write_json(metadata, {'key': key, 'size': list(image.size), 'render': image.info.get('render', {}), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
        return (image, path, False)

def process_corpus(manifest_path: str | Path, engine: str, *, engine_factory=None, renderer=None, attempt_id=None):
    factory = engine_factory or create_engine
    render = renderer or worker_render
    campaign = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
    attempt = attempt_id or os.environ.get('VTM_OCR_ATTEMPT') or uuid.uuid4().hex
    queue_path = os.environ.get('VTM_OCR_PAGE_QUEUE')
    queue = GPUPageQueue(queue_path) if queue_path else None
    owner = os.environ.get('VTM_OCR_QUEUE_OWNER', attempt)
    plans = [corpus_book_plan(pair, engine, attempt) for pair in campaign['pairs']]
    completed = sum((len({task['page'] for task in plan['tasks']} & plan['done'].keys()) for plan in plans))
    progress_file = runtime_path(Path(manifest_path).parent) / f'{engine}_progress.json'

    def update_progress():
        if queue is not None:
            return
        write_json(progress_file, {'engine': engine, 'attempt': attempt, 'pages': campaign['total_pages'], 'done': completed, 'missing': campaign['total_pages'] - completed, 'updated': now()})
    update_progress()
    backend = None
    try:
        with tempfile.TemporaryDirectory(prefix='vtm-corpus-') as temporary:
            image_path = Path(temporary) / 'page.png'
            for plan in plans:
                pair, directory, manifest_file, manifest, cfg, name, raw, errors, tasks, done, todo = (plan[key] for key in ('pair', 'directory', 'manifest_file', 'manifest', 'cfg', 'name', 'raw', 'errors', 'tasks', 'done', 'todo'))
                if pair.get('original_checkpoint_readonly'):
                    continue
                original_done = {}
                if pair.get('variant', 'original') != 'original':
                    original_pair = next((p for p in campaign['pairs'] if p.get('variant') == 'original' and p['book'] == pair['book'] and (p.get('crop_type') == pair.get('crop_type'))), None)
                    if original_pair:
                        original_directory = runtime_path(original_pair[engine])
                        original_manifest = json.loads((original_directory / 'run_manifest.json').read_text(encoding='utf8'))
                        original_name = original_manifest['config'].get('run_name') or engine
                        original_done = worker_load_results(original_directory / 'ocr_raw', pair['book'], original_name)
                if todo and backend is None:
                    load_started = time.monotonic()
                    backend = factory(engine, cfg)
                    if queue is not None:
                        queue.loaded(engine, time.monotonic() - load_started)
                    print(f'[corpus] {engine}: model đã tải; dùng chung cho toàn bộ sách đã chọn.', flush=True)
                if todo:
                    expected_sha = manifest.get('books', {}).get(pair['book'], {}).get('sha256')
                    if expected_sha and expected_sha != dataset_file_hash(runtime_path(todo[0]['src'])):
                        raise ValueError('Source PDF hash changed since OCR planning')
                if backend is not None:
                    backend.max_new = cfg.max_new_tokens
                    backend.model_min_pixels, backend.model_max_pixels = (cfg.model_min_pixels, cfg.model_max_pixels)
                    backend.preserve_input_resolution = cfg.segmentation != 'none' or bool(cfg.image_dataset)
                    if engine == 'hunyuan':
                        backend.rep, backend.prompt = (cfg.repetition_penalty, cfg.prompt or HUNYUAN_PROMPT)
                if queue is None:
                    manifest.update(status='running', updated=now())
                    write_json(manifest_file, manifest)
                else:
                    errors = errors.with_name(f'{name}_errors_{owner}.jsonl')
                queue_key = str(directory.resolve())
                print(f"[corpus] {engine} / {pair['book']}: {len(tasks)} trang được chọn, còn {len(todo)} trang cần OCR.", flush=True)
                for task in queue.pages(engine, queue_key, owner) if queue is not None else todo:
                    page, started = (task['page'], time.monotonic())
                    print(f"[corpus] {engine} / {pair['book']} / trang {page + 1}: bắt đầu (timeout={cfg.page_timeout}s).", flush=True)

                    def on_timeout():
                        append_record(errors, {'page': page, 'error': 'TimeoutError', 'message': f'quá {cfg.page_timeout}s; worker phải khởi động lại', 'attempt': attempt, 'timeout_seconds': cfg.page_timeout, 'seconds': round(time.monotonic() - started, 1), 'updated': now()})
                        if queue is not None:
                            queue.finish(engine, queue_key, page, owner, success=False, seconds=time.monotonic() - started)
                    try:
                        with worker_page_deadline(cfg.page_timeout, on_timeout):
                            image, rendered_path, cache_hit = corpus_page_image(campaign, manifest, pair['book'], task, cfg, render, image_path)
                            rendered_at = time.monotonic()
                            text, extra = run_prepared_page(backend, image, rendered_path, cfg, page=page, original_record=original_done.get(page))
                        record = {'page': page, 'text': text, 'w': image.width, 'h': image.height, 'sec': round(time.monotonic() - started, 1), **extra, 'render_cache_hit': cache_hit, 'render_sec': round(rendered_at - started, 3), 'inference_sec': round(time.monotonic() - rendered_at, 3), 'image_input': 'prepared' if image.info.get('prepared_dataset') else 'legacy_missing_page_fallback' if cfg.image_dataset else 'legacy'}
                        preparation = extra.get('image_preparation')
                        if preparation:
                            regions = preparation['regions']
                            dimensions = ', '.join(('x'.join(map(str, r['input_size'])) for r in regions))
                            observed = [r.get('processor_size') for r in regions]
                            processor = ', '.join(('x'.join(map(str, size)) for size in observed)) if all(observed) else 'unobserved native tensors'
                            print(f'[image] {image.width}x{image.height} -> {len(regions)} input(s): {dimensions}; processor: {processor}', flush=True)
                        append_record(worker_result_file(raw, pair['book'], name, owner if queue is not None else 0), record)
                        if queue is not None:
                            queue.finish(engine, queue_key, page, owner, success=True, seconds=time.monotonic() - started)
                        done[page] = record
                        completed += 1
                        update_progress()
                        print(f"[corpus] {engine} / {pair['book']} / trang {page + 1}: xong sau {record['sec']}s.", flush=True)
                    except Exception as exc:
                        if queue is not None:
                            queue.finish(engine, queue_key, page, owner, success=False, seconds=time.monotonic() - started)
                        append_record(errors, {'page': page, 'error': type(exc).__name__, 'message': str(exc), 'traceback': traceback.format_exc(), 'attempt': attempt, 'timeout_seconds': cfg.page_timeout, 'seconds': round(time.monotonic() - started, 1), 'updated': now()})
                        detail = str(exc).replace('\n', ' ')[:300]
                        print(f"[corpus] {engine} / {pair['book']} / trang {page + 1}: lỗi {type(exc).__name__}: {detail}; log: {errors}", flush=True)
                wanted = {t['page'] for t in tasks}
                got = len(wanted & done.keys())
                counts = {'pages': len(tasks), 'done': got, 'missing': len(tasks) - got}
                if cfg.export and queue is None:
                    export_ocr(directory, {pair['book']: (tasks[0]['src'], sorted(wanted))}, name)
                if queue is None:
                    manifest.update(status='done' if got == len(tasks) else 'partial', counts=counts, updated=now())
                    write_json(manifest_file, manifest)
                print(f"[corpus] {engine} / {pair['book']}: xong {got}/{len(tasks)} trang.", flush=True)
    finally:
        if backend is not None:
            del backend
        for document in worker__DOCS.values():
            document.close()
        worker__DOCS.clear()
        gc.collect()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="One isolated OCR model per process")
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--engine", choices=["ppocr", "tonggu", "paddle", "hunyuan"], required=True)
    parser.add_argument("--token")
    parser.add_argument("--pid-file")
    parser.add_argument("--attempt")
    args = parser.parse_args()
    if args.pid_file:
        if os.getpgrp() != os.getpid():
            os.setsid()
        marker = Path(args.pid_file)
        write_json(marker, {"token": args.token, "pgid": os.getpgrp()})
    process_corpus(args.manifest, args.engine, attempt_id=args.attempt)


## Setup bốn OCR offline

Dùng các model/package OCR đang có; không setup model restoration.


In [ ]:
if RUN_OCR and not MODELS_DIR:
    raise ValueError('Điền MODELS_DIR cho bốn OCR trước khi chạy notebook 1a.')
SESSION_SETTINGS = dict(packages_dir=PACKAGES_DIR, input_dir=DATA_DIR, weights=WEIGHTS, books=BOOKS, max_books=MAX_BOOKS, pages=PAGES, limit=LIMIT, run_ocr=RUN_OCR, work_dir=WORK_DIR, advanced={**ADVANCED, 'prepared_crop_types': PREPARED_CROP_TYPES, 'ocr': {**IMAGE_PREPARATION, 'image_dataset': str(PREPARED_IMAGES), 'image_variant': PREPARED_VARIANT, 'image_crop_type': PREPARED_CROP_TYPES[0], 'image_dataset_fallback': PREPARED_MISSING_FALLBACK, 'allow_generated_variants': ALLOW_GENERATED_VARIANTS, **ADVANCED.get('ocr', {})}, 'enable_multi_variant_ocr': ENABLE_MULTI_VARIANT_OCR, 'ocr_variants': ['original'] if RUN_OCR else OCR_VARIANTS, 'enable_spatial_coverage': ENABLE_SPATIAL_COVERAGE, 'spatial_coverage_options': SPATIAL_COVERAGE_OPTIONS, 'variant_stability_options': VARIANT_STABILITY_OPTIONS, 'enable_restoration_ablation': ENABLE_RESTORATION_ABLATION, 'enable_visual_fidelity': ENABLE_VISUAL_FIDELITY, 'restoration_evaluation_options': RESTORATION_EVALUATION_OPTIONS})
SESSION_SETTINGS['advanced']['ocr']['export'] = False
session = setup_ocr(**SESSION_SETTINGS)


## OCR gốc và xuất gói chuyển tiếp

Bốn model cùng nhận ảnh gốc đã chuẩn bị. Gói handoff lưu ngay sau OCR, trước phân tích báo cáo.


In [ ]:
# Chỉ OCR ảnh original, không dùng/ngầm bật bất kỳ restoration nào.
stage_results = run_ocr_models(session)
if RUN_OCR:
    HANDOFF_ARCHIVE = package_original_ocr_handoff(session.manifest, PREPARED_IMAGES,
        WORK_DIR / 'original_ocr_handoff.zip')
    print('Gói để đặt vào DATA_DIR của notebook tiếp tục:', HANDOFF_ARCHIVE)
    print('Bốn model đã có output:', {e:engine_progress(session.manifest,e) for e in session.models})
    display(FileLink(str(WORK_DIR / 'original_ocr_handoff.zip')))


## Kết quả lượt OCR gốc

Raw OCR nằm trong gói handoff; notebook 1 sẽ đếm/đối chiếu và tạo báo cáo.


In [ ]:
# Đối chiếu ký tự/coverage và báo cáo được làm ở notebook 1.


## Chuyển sang notebook 1

Chỉ tải `original_ocr_handoff.zip` và đặt cùng PDF trong DATA_DIR. Ảnh/raw tại WORK_DIR được giữ để resume.


In [ ]:
# Gói chuyển tiếp duy nhất đã được xuất ngay sau OCR: original_ocr_handoff.zip.


Giữ thư mục ảnh/raw khi cần resume. Chuyển sang notebook 1 bằng gói handoff duy nhất.
Raw của cả bốn model được giữ nguyên kể cả thiếu/lỗi; hoàn tất OCR không chứng minh accuracy.
